<!-- Cell No: 001 -->
# Parallel Motion as an Exact Counterfactual
### How much geometry feed-forward 3D reconstruction models use for moving objects (the convoy ambiguity)

A point moving by a fraction `s` of the camera baseline, parallel to it, gives the same two images as a static point at depth `Z/(1−s)`. The slope `β_s` of a model's log-depth error on `x = −log(1−s)` is the share of cross-view geometry it uses for the object (0: single-image prior, 1: triangulation). This notebook measures that share for DUSt3R, MASt3R, MonST3R, VGGT, π³ and MapAnything on CONVOY-3D, CONVOY-REAL, E5/E5-REAL, Virtual KITTI 2, Argoverse 2 and Bonn RGB-D Dynamic, tests the registered hypotheses (P1–P4, H5–H8, H11–H14) and writes every number, table and figure of the paper into `outputs/`. The idea lock, the amendments with every registered prediction, and all decisions are in Section 0 (code cells) and in `outputs/manifest/`.

**Version 6.6.** H14 (registered): the static-twin test with optical flow and a LiDAR road plane instead of labelled boxes. Bonn ground truth corrected (relative poses re-estimated from static correspondences, photometric check of every moving region, quality gate). Figure and caption corrections; explanatory text cells removed.

**How to run (Kaggle).** Accelerator: GPU T4, Internet: on. Attach the `convoy_checkpoints.zip` of the v6.5 run as a dataset (Section 0.5 finds it) and run all cells. Every v6.5 stage is restored unchanged; only the Bonn ground truth, the six models on Bonn and the flow-based twin stage are computed: about 1.5–2 h, of which restoring the archive takes about 45–50 min. Without an archive a fresh run takes about 13 h and needs a second session (attach the first session's archive and run again). Optional: KITTI tracking as a private dataset (Section 3.10); the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` to copy the archives to a private dataset.

**Outputs.** `convoy_checkpoints.zip` (every finished stage, a valid resume point at any time) and `convoy_paper_outputs.zip` (figures as PDF/PNG with data and LaTeX captions, tables as CSV/JSON/LaTeX, results, manifests). For a run that must survive a closed browser use *Save Version → Save & Run All*.

In [ ]:
# Cell No: 002
# =============================================================================
# 0.1  IDEA LOCK AND V6 AMENDMENT.
#      IDEA_LOCK: the scientific content locked before the first experiment
#      (question, hypothesis, falsification conditions, propositions,
#      assumptions, predictions P1-P4, metrics, claim scope), unchanged. Only the
#      two "actually_*" inventory fields record what THIS notebook version runs.
#      AMENDMENT_V6: the reframing and the hypotheses H5-H10, registered after
#      the v5 results were read and before any v6 experiment was generated.
#      AMENDMENT_V6_1: written after the first complete v6 run and before any
#      v6.1 data: H11a-b (contact ablation on real images) registered, H10
#      withdrawn, H9 moved to exploratory. H5-H8 and P1-P4 are unchanged.
#      AMENDMENT_V6_2 to AMENDMENT_V6_4: exploratory analyses only, each written
#      after reading the previous complete run and before its own data existed.
#      AMENDMENT_V6_5: two designed experiments with registered predictions
#      (H12 baseline sweep, H13 contact dose-response) and exploratory analyses
#      (static twin, cue model, Bonn RGB-D Dynamic), written after the complete
#      v6.4 run and before any of their data existed.
#      AMENDMENT_V6_6: H14 (the twin test with optical flow) registered, the Bonn
#      ground truth corrected, written after the complete v6.5 run.
#      All of them are written to outputs/manifest/ in Section 13.
# =============================================================================
IDEA_LOCK = {
    "title": "The Convoy Ambiguity: Feed-Forward 3D and 4D Reconstruction Models "
             "Inherit the Epipolar Degeneracy for Motion Parallel to Camera Translation",

    "contribution_type": "Scientific discovery (primary); Evaluation (motion-decomposed "
        "protocol for dynamic-scene reconstruction); an elementary, exact geometric law "
        "fixes the falsifiable prediction (not a general theory paper).",

    "research_question": (
        "Do frozen feed-forward 3D/4D reconstruction models (DUSt3R, VGGT, MonST3R, ...) "
        "reconstruct an object moving parallel to the camera's translation at the exact "
        "depth a static object at that location would have (Z*(s)=Z/(1-s)), and do "
        "training-free dynamic-mask methods built on epipolar-inconsistency fail to flag "
        "exactly this motion?"
    ),

    "central_hypothesis": (
        "For static-trained feed-forward models, the predicted depth of an object moving "
        "with parallel displacement fraction s and perpendicular pixel offset eps~=0 "
        "satisfies Zhat/Z = 1/(1-s) to within the model's static-scene error, with no "
        "increase in any dynamic-mask signal; as eps grows, the prediction departs from "
        "the epipolar solution toward the model's monocular estimate; dynamic-trained "
        "models differ from static-trained ones only in the eps>0 regime and, at eps~=0, "
        "deviate from 1/(1-s) only for object classes with strong size priors."
    ),

    "falsification_conditions": (
        "The hypothesis is FALSIFIED if: (a) Zhat/Z is insensitive to s (priors dominate "
        "everywhere -> beta_s CI excludes [0.5,1.5] or R^2<0.5 at eps~=0); "
        "(b) dynamic-mask methods flag parallel motion (|eps|<1px) as reliably as "
        "perpendicular motion (eps>=3px) -> recall-difference CI includes 0; or "
        "(c) dynamic-trained models beat Z*(s) at eps~=0 for ALL object classes, i.e. no "
        "class-dependent prior takeover is observed."
    ),

    "propositions_exact_and_verified": {
        "Proposition 1 (parallel-motion degeneracy)": (
            "delta = s*b (pure parallel motion) makes the moving point's camera-2 "
            "coordinates identical, up to a positive scalar, to the static point "
            "X* = X/(1-s)'s camera-2 coordinates -> both pixels are indistinguishable "
            "from a static point at depth Z*(s) = Z/(1-s). Verified numerically in "
            "Section 2 to floating-point precision, for arbitrary camera rotation R."
        ),
        "Proposition 2 (multi-view convoy persistence)": (
            "If the camera's TRANSLATION per frame is constant (b each frame -> a "
            "straight path) and the object's own world-frame velocity is s*b (constant, "
            "parallel to that translation), the degeneracy persists for any number of "
            "frames and for ANY per-frame camera ROTATION R_k. It requires a CHANGE in "
            "the camera's translation direction (a curved path / true turning) to break -- "
            "rotation of the camera's heading alone, with a straight translation path, "
            "does NOT break it. This distinction is verified numerically in Section 2."
        ),
        "Proposition 3 (perpendicular motion is detectable)": (
            "If the perpendicular component delta_perp != 0, no static point reproduces "
            "both observations; the second observation lies eps pixels off the true "
            "epipolar line -- the only first-order geometric evidence of motion available "
            "to a two-view method."
        ),
    },

    "assumptions": {
        "A1 (correspondence-based depth)": (
            "The model's two-view depth for textured object pixels is driven by cross-view "
            "correspondence, not monocular cues alone. DIRECTLY TESTABLE per model via the "
            "static-scene sensitivity test (mask the corresponding patch in view 2, observe "
            "the depth change). If it fails for a model, w~=1 is EXPECTED for that model and "
            "is reported as such, not treated as a hypothesis failure."
        ),
        "A2 (small-displacement first-order geometry)": (
            "eps computed to first order (eps ~= f*||delta_perp||/Z) is adequate. "
            "DIRECTLY TESTABLE by comparing the first-order formula against the exact "
            "epipolar-line perpendicular distance on synthetic data with known geometry. "
            "Where it is violated, the EXACT offset is used instead for regime "
            "classification -- see Section 2.3 for the measured divergence."
        ),
        "A3 (ground-truth adequacy)": (
            "Synthetic data supplies exact depth/velocity by construction; any real data "
            "(Sintel, TUM) supplies GT depth/flow/poses with bounded, documented sensor "
            "noise. STRUCTURAL for the synthetic testbed (true by construction); "
            "EMPIRICALLY PROXY-TESTABLE for any real dataset that loads, via GT "
            "self-consistency checks."
        ),
    },

    "falsifiable_numerical_predictions": {
        "Prediction 1 (degeneracy)": "beta_s in [0.8,1.2] (95% cluster-bootstrap CI) for "
            "static-trained models in the parallel regime (|eps|<1px), R^2>=0.5.",
        "Prediction 2 (masks)": "Dynamic-mask recall for |eps|<1px is lower than for "
            "eps>=3px by >=0.3 absolute, at matched |s| and object size.",
        "Prediction 3 (prior takeover)": "Prior weight w increases with eps (Spearman "
            "rho>0.5) and, at eps~=0, is larger for size-prior-bearing classes than for "
            "texture-only synthetic objects.",
        "Prediction 4 (convoy persistence)": "beta_s stays in [0.8,1.2] over 2-8 frames on "
            "straight (constant-translation) segments and drops toward 0 on segments with "
            "a genuinely curved translation path.",
    },

    "method_under_test": "Feed-forward 3D/4D reconstruction models, probed by the "
        "measurement protocol in Section 4 (not a newly proposed architecture).",

    "models_in_scope_per_idea": [
        "DUSt3R", "MASt3R", "VGGT-1B", "pi3", "MapAnything",
        "Depth Anything 3", "MonST3R", "CUT3R",
    ],
    "models_actually_registered_this_notebook": [
        "DUSt3R", "MASt3R", "MonST3R", "VGGT-1B", "pi3", "MapAnything",
    ],

    "datasets_in_scope_per_idea": [
        "Kubric MOVi-F", "MPI-Sintel", "KITTI tracking",
        "Bonn RGB-D Dynamic", "TUM RGB-D dynamic",
    ],
    "datasets_actually_used_this_notebook": [
        "CONVOY-3D (ray-cast synthetic street scenes; plays the role of Kubric MOVi-F, Section 3.1), "
        "including the v6 ground-contact ablation (E5)",
        "LEGACY-FLAT (negative control for the validity gate, Section 3.4)",
        "TUM RGB-D fr3 static sequences (real backgrounds for the CONVOY-REAL composites, Section 3.5, and "
        "for the v6.1 contact ablation on real images, E5-REAL, Section 3.5.1)",
        "TUM RGB-D fr3 walking_xyz / walking_halfsphere (prevalence of motion regimes only, Section 3.7)",
        "Virtual KITTI 2, clone variation, Camera_0 (photo-realistic driving with exact depth and poses, Section 3.8)",
        "Argoverse 2 sensor dataset, validation logs, front-centre camera (real driving with LiDAR; exploratory, Sections 3.11-3.11c)",
        "KITTI tracking (real driving with LiDAR; only when attached; exploratory, Section 3.10)",
        "CONVOY-3D designed experiments of v6.5: baseline sweep (BSW) and contact dose-response (WDR), Section 3.2b",
        "Bonn RGB-D Dynamic, four sequences with balloons and boxes moved by people (real objects with measured depth; "
        "exploratory, Section 3.12)",
    ],

    "baselines_per_idea": [
        "Classical two-view triangulation with GT poses (positive control -- MUST follow "
        "1/(1-s) exactly)",
        "Monocular model / oracle (negative control -- MUST be s-invariant)",
        "Flow-based dynamic mask (MonST3R-style)",
        "Attention-based dynamic mask (Easi3R-style / VGGT4D-style)",
    ],

    "main_metrics": ["beta_s (regression slope)", "w(s,eps,class,texture) (prior weight)",
                     "dynamic-mask recall by regime", "depth AbsRel by motion angle"],

    "sota_position": "Scientific discovery + evaluation advance. The paper predicts a "
        "FAILURE of existing dynamic-mask methods rather than beating a leaderboard. "
        "No SOTA is claimed anywhere in this notebook.",

    "claim_scope": (
        "Findings are scoped to: (i) the specific models whose checkpoints actually "
        "loaded in THIS execution (see MODEL_STATUS in Section 4); (ii) the datasets that "
        "actually became available (see DATASET_STATUS in Section 3); (iii) the parameter "
        "ranges swept (depth 5-60 m, |s| up to 1.4, frame gaps and baselines documented in "
        "Section 0.2). No claim in this notebook generalizes beyond what is listed here."
    ),
}

# =============================================================================
# V6 AMENDMENT (registered before any v6 data were generated; the v5 run had shown
# beta_s between 0 and 0.4 for every model, i.e. P1 falsified as locked)
# =============================================================================
AMENDMENT_V6 = {
    "status": "registered after reading the v5 results and before generating any v6 data; "
              "IDEA_LOCK and the verdict rules of P1-P4 are not changed",

    "reframing": (
        "Parallel motion is used as an exact counterfactual. For a mover with delta = s*b the two "
        "views are pixel-identical to a static object at Z/(1-s) (Proposition 1), so the geometric "
        "answer and the true depth differ by the known amount x = -log(1-s) while nothing else in the "
        "images changes. The slope beta_s of the log depth error on x therefore measures how much of "
        "the cross-view geometry a model uses for the object: 1 = pure triangulation, 0 = pure single-"
        "image prior. Through the slope-weight identity (Section 2.4) beta_s maps to a prior weight, "
        "and Section 6.0b checks that the estimator recovers planted weights. The question becomes: "
        "how much geometry do feed-forward models use for moving objects, what controls it, and what "
        "does that imply for dynamic masks built on geometric residuals?"
    ),

    "registered_hypotheses": {
        "H5 (support cue)": "Ground-contact ablation E5 (same object, same s, eps = 0): beta(floating, no "
            "shadow) - beta(grounded, no shadow) > 0, paired cluster-bootstrap 95% CI > 0, per model, car and "
            "crate pooled. Rationale: without a ground contact the single-image depth cue of the object is "
            "missing, so a model that combines cues should lean more on geometry.",
        "H6 (contact visibility)": "beta(grounded, contact hidden by a wall) - beta(grounded) > 0, CI > 0.",
        "H7 (the wall acts through the contact)": "(beta_CH - beta_G) - (beta_FH - beta_F) > 0, CI > 0: hiding "
            "the contact matters more for an object that has one.",
        "H8 (single-image signature)": "median rho_mono(floating) - median rho_mono(grounded, no shadow) > 0, "
            "CI > 0: a lifted object is placed farther away by the single-image prediction (its lowest "
            "pixels sit higher in the image, as for a farther grounded object).",
        "H9 (driving)": "Virtual KITTI 2 moving vehicles in the parallel regime (|eps| < 1 px), measurable and "
            "identifiable, x_geo = log(Z_geo/Z) inside the CONVOY-3D design range [-0.47, 1.90]: the slope "
            "beta_geo of rho on x_geo is below 0.3. SUPPORTED if the estimate is < 0.3 and the CI upper end "
            "< 0.5; FALSIFIED if the CI lower end > 0.3; PARTIAL otherwise. Tested only with >= 20 movers "
            "from >= 5 pairs. Models trained on Virtual KITTI (MASt3R, VGGT) are flagged, not removed.",
        "H10 (attention masks on long clips)": "Easi3R on 10-frame CONVOY-3D clips (window-5 scene graph, "
            "the dynamic map of Eq. 9 refined by k-means clustering of encoder features as in the Easi3R "
            "paper), evaluated on the middle frame: recall(perpendicular) - recall(parallel) >= 0.3 with "
            "CI > 0, using the P2 rule unchanged (recall at 5% static FPR, stratified by |s| and size). "
            "Backbones DUSt3R and MonST3R. Replicated on Virtual KITTI 2 clips when each regime has >= 20 movers.",
    },

    "exploratory_without_prediction": [
        "E5 shadow effect: beta(grounded, no shadow) - beta(grounded).",
        "E5 class effect: floating car versus floating crate.",
        "E6 prevalence: share of moving-vehicle pixels in Virtual KITTI 2 within 1 px of the epipolar line, and "
        "share of moving vehicles whose track over K = 2, 4, 8 frames a static point fits within 1 px.",
        "E7 secondary outputs: the raw Eq. 9 map and the Otsu-thresholded binary Easi3R mask.",
    ],

    "removed_from_v5": [
        "Real-people slope analysis (v5 Section 6.7): the regressor log(Z_tri/Z) is built from noisy flow and "
        "sensor depth, which attenuates the slope by an unknown factor. Only the prevalence of motion "
        "regimes in TUM walking is kept.",
        "M3 Easi3R on a 3-frame complete graph (v5 E2): far shorter than the windows the method was designed "
        "for. Replaced by H10 on 10-frame clips.",
    ],
}

# =============================================================================
# V6.1 AMENDMENT (written after the first complete v6 run, before any v6.1 data)
# =============================================================================
AMENDMENT_V6_1 = {
    "status": "written after reading the first complete v6 run and before generating any v6.1 data "
              "(the real-image contact ablation did not exist in v6). H5-H8 and P1-P4 keep their rules.",

    "registered_hypotheses": {
        "H11a (support cue, real images)": "E5-REAL: the same textured crate inserted into real TUM RGB-D frame "
            "pairs, once resting on the real support plane with a contact shadow (G) and once lifted 0.25-0.50 m "
            "without a shadow (F); purely parallel motion delta = s*b. beta(F) - beta(G) > 0, paired cluster-"
            "bootstrap 95% CI > 0 (cluster = image pair), Holm across models. Same estimator as H5.",
        "H11b (single-image signature, real images)": "median rho_mono(F) - median rho_mono(G) > 0, CI > 0, on the "
            "same crates. Same estimator as H8.",
        "sample": "crates measurable and identifiable in both conditions, with >= 5 contact pixels (within "
            "3 cm of the plane) in both views when grounded and none when lifted; at least 8 crates from 5 pairs, "
            "otherwise NOT TESTABLE.",
    },

    "withdrawn": {
        "H10 (Easi3R on 10-frame clips)": "Outcome of the first complete v6 run, kept on record: FALSIFIED by its "
            "rule for both backbones (recall(perp) - recall(par) = -0.02 [-0.15, 0.10] for DUSt3R and -0.08 "
            "[-0.18, 0.02] for MonST3R, 52 parallel and 43 perpendicular movers). The rule could not have come out "
            "otherwise: the cluster-refined map found perpendicular movers with recall 0.17 and 0.07 at a 5% static "
            "false-positive rate, and the binary mask flagged 40-45% of static pixels, so the map detects movers in "
            "neither regime and cannot show a regime difference. The test is withdrawn from v6.1 and the attention-"
            "based mask of the locked baseline list is not evaluated (Section 0.2, D33); the paper reports this in "
            "its limitations, not as evidence for or against the probe.",
    },

    "moved_to_exploratory": {
        "H9 (driving)": "The v6.0 measurability rule for Virtual KITTI 2 (whole box inside the crop, truncation "
            "<= 5%, >= 85% visible in view 2) left 2 vehicles, so H9 could not be tested. v6.1 applies the "
            "CONVOY-3D pixel rule (>= 40 eroded visible pixels, >= 85% of the silhouette unoccluded). Because "
            "the rule was changed after the v6.0 sample size was seen, H9 is reported as exploratory, with "
            "its registered thresholds and the full attrition table (Section 3.8), not as a confirmatory test.",
    },
}

# =============================================================================
# V6.2 AMENDMENT (written after the complete v6.1 run, before any v6.2 analysis existed)
# =============================================================================
AMENDMENT_V6_2 = {
    "status": "written after reading the complete v6.1 run and before any of the analyses below existed. Nothing in it "
              "is a registered test: every item is exploratory and none changes a registered verdict.",
    "exploratory_analyses": {
        "masks by support (6.7c)": "M2 recall (exact correspondences) on the same parallel E5 movers in each condition; "
            "expected direction: lower when the contact is hidden (G -> CH) or the mover lifted (NS -> F).",
        "ground contact in traffic (6.8b)": "Virtual KITTI 2 parallel vehicles (>= 40 eroded px, identifiable; the D34 "
            "occupancy rule left 3 vehicles and is dropped here): per-object geometry share g for vehicles whose wheel "
            "contact points are hidden vs visible; expected direction: higher when hidden (as H6).",
        "the convoy case (6.8b)": "vehicles keeping pace with the camera (0.85 <= s < 1.15): g for s < 1 and, for s >= 1, "
            "the share placed farther than the model's own single-image depth; Virtual KITTI 2 and KITTI tracking. "
            "Identifiability is not required (keeping pace means almost no parallax by construction).",
        "KITTI tracking (3.10, 6.8b)": "real vehicles with LiDAR depth when the dataset is attached: regime shares, the slope "
            "beta_geo on parallel vehicles, and the convoy case. Correspondences come from labelled boxes; the parallel "
            "threshold is the label-noise floor measured on parked vehicles.",
        "single-image protocol (6.7d)": "for the multi-view models, H11b and the CONVOY-REAL single-image slope with the image "
            "given once and twice; a model whose single-image results depend on the protocol makes no single-image claim.",
    },
    "per_object_share": "g = (e^rho - e^rho_mono) / (e^x - e^rho_mono), from the oracle's mixing model Zhat = w Zhat_mono + "
                        "(1 - w) Z_geo; computed only when |x - rho_mono| >= 0.3.",
}

# v6.3 (after reading the complete v6.2 run, before any Argoverse 2 data or any of the analyses below existed)
AMENDMENT_V6_3 = {
    "status": "written after reading the complete v6.2 run and before any Argoverse 2 data, any additional Virtual KITTI 2 "
              "pair or any of the analyses below existed. Nothing in it is a registered test; every item is exploratory "
              "and no registered verdict depends on it.",
    "exploratory_analyses": {
        "Argoverse 2 (3.11, 6.8b)": "real vehicles with LiDAR depth from the public Argoverse 2 sensor dataset (validation "
            "split, front-centre camera), downloaded automatically: regime shares, the convoy case and the slope beta_geo, "
            "with the KITTI rules of v6.2 unchanged (moving >= 0.5 m, parked <= 0.15 m, >= 40 LiDAR pixels, parallel "
            "threshold = 90th percentile of |eps| over parked vehicles, at least 1 px).",
        "matched contact comparison (6.8b)": "every Virtual KITTI 2 vehicle with hidden wheel contact is matched to the "
            "nearest vehicle with visible contact on log depth, log image area and x_geo (within 0.5 pooled SD on each, "
            "with replacement); the median paired difference of g is reported with covariate balance.",
        "more Virtual KITTI 2 pairs": "at most 150 pairs per scene instead of 60 (the short scenes stay limited by their "
            "length); same selection rule and ground truth.",
        "cluster unit (6.8b)": "intervals of the driving analyses resample vehicle tracks instead of image pairs, because "
            "the denser sampling shows the same vehicle in several pairs.",
    },
    "ground_truth_changes": "projected LiDAR (KITTI, Argoverse 2) keeps a pixel only where its return is the nearest surface "
                            "in a 5 x 5 window (no see-through between sparse returns); a moving vehicle's Argoverse 2 "
                            "returns are carried to the image time along its labelled track.",
}

# v6.4 (after reading the complete v6.3 run, before any of the data or analyses below existed)
AMENDMENT_V6_4 = {
    "status": "written after reading the complete v6.3 run and before the additional Argoverse 2 logs, the ground-contact "
              "label of any Argoverse 2 vehicle or any of the synthesis tables below existed. Nothing in it is a registered "
              "test; every item is exploratory and no registered verdict depends on it. The model outputs for the 16 Argoverse 2 "
              "logs of v6.3 existed and had been read, but never split by ground contact.",
    "exploratory_analyses": {
        "more Argoverse 2 logs (3.11b)": "48 further validation logs, taken in the same fixed random order after the logs the "
            "v6.3 stage tried, with every v6.3 rule unchanged; the two stages are pooled (64 logs) in every Argoverse 2 analysis.",
        "ground contact of real vehicles (3.11b, 6.8b)": "the four wheel contact points of every labelled Argoverse 2 vehicle "
            "as for Virtual KITTI 2 (0.10 m above the box bottom at +-0.35 length, +-0.40 width); without instance masks a "
            "point counts as visible if it lies in the image, the vehicle's own body does not hide it (the camera lies beyond "
            "the side or end face next to that wheel, and the line of sight does not cross the vehicle's own box more than "
            "0.3 m above its bottom, as a bumper overhanging the wheel would), no other labelled object's box (shrunk by 10 cm) crosses the line of "
            "sight at least 0.3 m in front of it, and no static LiDAR return in the window of columns u +- 2, rows v - 8 ... v "
            "is nearer than it by more than max(10%, 1 m). Labels, sample, contrast and matching as for Virtual KITTI 2; the "
            "image size used for matching is the projected box area times (1 - the vehicle's LiDAR occlusion). Expected "
            "direction: g higher when the contact is hidden (as H6). Check of the rule: the same rule, with the exact depth of "
            "static pixels in place of the LiDAR, is applied to the Virtual KITTI 2 vehicles and compared with their "
            "instance-mask labels (D37); agreement and Cohen's kappa are reported (T15d), with no threshold.",
        "support across testbeds (6.9)": "a descriptive synthesis of results that already exist, with no new test: the "
            "geometry share by ground support in every testbed (CONVOY-3D cars and crates against floating crates; "
            "CONVOY-REAL, whose inserted movers are all floating crates by design; the E5 and E5-REAL conditions; Virtual "
            "KITTI 2 and Argoverse 2 vehicles by contact), and the contrast of unsupported or hidden-contact objects against "
            "grounded ones in each testbed. Only the CONVOY-3D contrast (floating crates minus cars and crates, unpaired, "
            "cluster bootstrap over scenes) is new; the others are those of H5, H6, H11a and 6.8b.",
        "implied depth error (6.9)": "for each slope beta, exp(beta x) at s = 0.85 (x = -log 0.15): the depth overestimate "
            "the slope implies for an object keeping pace with the camera, with the interval mapped through the same function.",
    },
    "why": "In the v6.3 run the geometry share on CONVOY-REAL was higher than on CONVOY-3D and, for four of the six models, "
           "close to that of the CONVOY-3D floating crates; CONVOY-REAL inserts only floating crates, and the pooled CONVOY-3D "
           "slope mixes grounded and floating movers. Reporting every testbed by support was decided after seeing this pattern, which is why the "
           "synthesis is descriptive and adds no verdict.",
}

# v6.5 (after reading the complete v6.4 run, before any of the data below existed)
AMENDMENT_V6_5 = {
    "status": "written after reading the complete v6.4 run and before any of the following existed: the baseline-sweep and "
              "dose-response renders, any Bonn pair, the camera height of any Argoverse 2 vehicle. H12 and H13 are registered "
              "tests; every other item is exploratory and no registered verdict depends on it. The analyses that re-use v6.4 "
              "model outputs (the cue model, the parallax check of CONVOY-REAL, the mask trade-off) had not been run on them "
              "before this amendment, but those outputs had been read.",
    "registered_hypotheses": {
        "H12 (parallax channel: baseline sweep)": "BSW, 64 new base scenes with three purely parallel movers (cars and crates on "
            "the street, floating crates), each seen with the camera baseline scaled by 0.5, 1, 2 and 3: same first image, same "
            "rotation of the second camera, the mover's displacement scaled with the baseline, so s, eps = 0 and the geometric "
            "answer are unchanged and only the parallax changes. For DUSt3R and MASt3R: beta(x3) - beta(x0.5) > 0, paired "
            "cluster-bootstrap 95% CI > 0 (cluster = base scene), Holm across the two models, on movers measurable at every "
            "baseline and identifiable at x1. Rationale: in v6.4 these two models had beta_s of 0.55-0.61 for movers with more "
            "than 40 px of parallax and 0.07-0.17 below 40 px, and the other four showed no such rise; E1 cannot say whether "
            "parallax causes it. The same contrast is reported for the other models without a prediction.",
        "H13 (the contact junction suffices: dose-response)": "WDR, 64 new base scenes (cars and crates, purely parallel), each "
            "mover rendered without a wall and behind walls that hide at least its lowest 15, 35 and 70 cm in both views (the E5 "
            "wall, designed by ray casting; a wall must cover the whole visible base in both views, so it hides more of the "
            "mover's near end, and the share of the mover it hides is measured on the renders). For every model: beta(15 cm) - beta(no wall) > 0, CI > 0, Holm across models, on movers "
            "whose contact is visible without the wall (>= 5 contact pixels in both views) and hidden by the 15 cm wall (none). "
            "Rationale: H6 hid 70 cm of a car and 35% of a crate; the lowest wall that still hides the contact in both views "
            "tests whether hiding the contact, rather than a large part of the body, raises the geometry share.",
        "rules": "NOT TESTABLE below 8 movers or 5 base scenes; verdicts as for H5-H8 (SUPPORTED: interval > 0; CONTRADICTED: "
                 "interval < 0; NOT SUPPORTED otherwise). The static-scene gate (Section 6.0) applies to both new testbeds.",
    },
    "exploratory_analyses": {
        "shape of the dose-response and of the sweep (6.10)": "beta(70 cm) - beta(no wall), beta(70 cm) - beta(15 cm) and "
            "the share of the 70 cm effect reached at 15 cm, reported only where the 70 cm effect is clearly positive (a step "
            "points to the contact junction, a graded rise to the extent of the hidden lower body); "
            "beta by baseline for grounded and floating movers separately; M2 recall by baseline and by wall height.",
        "the static twin (Proposition 4, 3.11c, 6.13)": "on Argoverse 2 (all 64 logs) the camera height H above each vehicle's "
            "contact plane comes from its labelled box; the static twin of a vehicle then lies H (1 - e^x_geo) above that plane, "
            "below the road for a vehicle moving with the camera. Twin test: |twin height| above its 95th percentile over parked "
            "vehicles; epipolar test: |eps| above its 95th percentile over parked vehicles (5% false positives each). Recall of "
            "each test and of either, by motion regime and by s, with the detection curve the parked distribution implies. "
            "Correspondences come from labelled boxes: this measures what each test could see, not what a pipeline would.",
        "two channels: a cue model (6.11)": "per model, rho - rho_mono = w (x - rho_mono) with w = c_support d^2 / (d^2 + kappa^2) "
            "(d: translational parallax of the static-equivalent point in px; support: contact in view, contact hidden, "
            "unsupported), fitted by least squares on CONVOY-3D and E5 and compared by 10-fold cross-validation over scenes "
            "with the same model without parallax (kappa = 0) and without support (one c). Without refitting, the model then "
            "predicts the slope on BSW, WDR, CONVOY-REAL, E5-REAL and Argoverse 2; predicted and observed slopes are reported "
            "side by side. Also: binned rho against x on CONVOY-3D with the model's curve; MonST3R minus DUSt3R on the same "
            "movers (slope above 40 px of parallax, and the H6 contrast); the change of M2 recall against the change of "
            "beta_s between E5 conditions.",
        "CONVOY-REAL and parallax (6.11)": "correction of the v6.4 reading (D57): the CONVOY-REAL movers have far more parallax "
            "(median about 69 px) than the CONVOY-3D floating crates (median 13-17 px), so their higher slope may come from "
            "parallax, not from real images; the comparison is repeated within parallax bins and against the cue model.",
        "Bonn RGB-D Dynamic (3.12, 6.12)": "real balloons and boxes moved by people (four sequences), measured depth and motion-"
            "capture poses, A3 as for TUM. Moving regions as for TUM walking (Section 3.7); people masked by an off-the-shelf "
            "instance segmenter; per object: its regime (parallel threshold: the flow-noise floor of static patches, at least "
            "1 px), x_geo from the flow correspondence after a per-pair scale check on static pixels, and the height of its "
            "lowest point above the floor. All six models are evaluated on the pairs; reported are the regime shares and the "
            "per-object geometry share g of parallel objects by kind (balloon, box, person) and by floor contact. Because x_geo "
            "is built from flow and sensor depth (D24), no slope is estimated; only per-object shares with |x - rho_mono| >= "
            "0.3 enter.",
    },
    "why": "The v6.4 run showed that ground support goes with the geometry share in every testbed, and that the rise of beta_s "
           "with parallax is specific to DUSt3R and MASt3R. Two readings remained open: that parallax rather than support "
           "drives part of the effect (the CONVOY-REAL movers have far more parallax than the CONVOY-3D floating crates), and "
           "that the contact effect needs a large occluder. The two designed experiments test these with registered "
           "predictions; the twin, the cue model and Bonn connect the result to geometry and to real objects without "
           "changing any verdict.",
}

# v6.6 (after reading the complete v6.5 run, before any flow-based twin quantity or corrected Bonn pair existed)
AMENDMENT_V6_6 = {
    "status": "written after reading the complete v6.5 run and before any of the following existed: the optical flow of any "
              "Argoverse 2 pair, the LiDAR road plane of any pair, any Bonn pair with a re-estimated pose. The Argoverse 2 images, "
              "LiDAR and labels were already used in v6.5 (exact correspondences only), so H14 is registered on data that had "
              "been seen but on quantities that had not been computed. Every other item is exploratory or a correction.",
    "registered_hypotheses": {
        "H14 (the twin test survives optical flow)": "Argoverse 2, all logs, moving vehicles in the parallel regime (regime from "
            "the labelled correspondences, as in v6.5) whose flow correspondences have a static solution (at most half of "
            "their LiDAR pixels triangulate behind the camera). Both tests from the same forward-backward checked flow at the "
            "vehicle's LiDAR pixels: epipolar test, median |eps| above its 95th percentile over parked vehicles; twin test, the "
            "10th percentile of the heights of the static reconstruction above the road plane fitted by RANSAC to the LiDAR "
            "returns of the image (no labelled box) outside its 2.5-97.5% range over parked vehicles. Prediction: "
            "det_twin - det_epipolar > 0, paired cluster-bootstrap 95% CI > 0 (cluster = vehicle track). Rationale: with exact "
            "correspondences the twin test flagged parallel movers the epipolar test cannot see; a deployed system has flow, "
            "not labels, and noise in the flow could erase the twin's displacement for slow vehicles.",
        "rules": "NOT TESTABLE below 20 vehicles or 10 tracks, or below 20 parked vehicles with flow (no thresholds); verdicts "
                 "as for H5-H13. No model output enters H14.",
    },
    "corrections": {
        "Bonn ground truth (D62)": "in v6.5 the motion-capture poses (calibration and synchronisation error) made static pixels "
            "miss their rigid flow by about 2 px, so large parts of the static scene were taken for moving objects. v6.6 "
            "re-estimates the relative pose of every pair from its static correspondences (RANSAC PnP, pairs whose fit fails "
            "are left out), does not test pixels hidden in the second image, keeps a moving region only if the static "
            "prediction fails photometrically on it and if it is not larger than 20% of the image, and takes the floor as the "
            "lowest well-supported horizontal level. If the median pair still flags more than 15% of its static scene, Bonn is "
            "not interpreted. Bonn stays exploratory.",
    },
    "exploratory_analyses": {
        "contact gates parallax (6.10)": "the rise of beta_s from the x0.5 to the x3 baseline for floating minus grounded "
            "movers (the H12-interaction row that v6.5 already reported), now listed as claim C33; chosen after the v6.5 "
            "results, so descriptive.",
        "how much of the hump the cue model explains (6.11)": "the fall of the median rho from its observed peak to the bin "
            "of largest x, observed and as each fit has it.",
    },
    "why": "Two weaknesses of v6.5 were in the measurements, not in the models: the twin test used labelled correspondences, "
           "which no deployed system has, and the Bonn objects were partly static scenery. H14 asks whether the twin test "
           "still works with the correspondences a pipeline would have; the Bonn correction asks whether the real floating "
           "objects say anything once the ground truth is right.",
}

import textwrap as _tw

def _print_block(title, body):
    print("=" * 88)
    print(title)
    print("=" * 88)
    if isinstance(body, dict):
        for k, v in body.items():
            print(_tw.fill(f"{k}: {v}", width=88, subsequent_indent="    "))
    elif isinstance(body, (list, tuple)):
        for v in body:
            print(_tw.fill(f"- {v}", width=88, subsequent_indent="  "))
    else:
        print(_tw.fill(str(body), width=88))
    print()

_LOCKED_FIELDS = [
    ("LOCKED RESEARCH QUESTION", "research_question"),
    ("LOCKED HYPOTHESIS", "central_hypothesis"),
    ("LOCKED FALSIFICATION CRITERIA", "falsification_conditions"),
    ("LOCKED PROPOSITIONS", "propositions_exact_and_verified"),
    ("LOCKED ASSUMPTIONS", "assumptions"),
    ("LOCKED NUMERICAL PREDICTIONS", "falsifiable_numerical_predictions"),
    ("LOCKED BASELINES", "baselines_per_idea"),
    ("LOCKED METRICS", "main_metrics"),
    ("LOCKED CONTRIBUTION TYPE / SOTA POSITION", None),
    ("LOCKED CLAIM SCOPE", "claim_scope"),
]
for _title, _key in _LOCKED_FIELDS:
    if _key is None:
        _print_block(_title, {"contribution_type": IDEA_LOCK["contribution_type"],
                              "sota_position": IDEA_LOCK["sota_position"]})
    else:
        _print_block(_title, IDEA_LOCK[_key])
_print_block("V6 AMENDMENT: STATUS AND REFRAMING", {"status": AMENDMENT_V6["status"],
                                                   "reframing": AMENDMENT_V6["reframing"]})
_print_block("V6 AMENDMENT: REGISTERED HYPOTHESES H5-H10", AMENDMENT_V6["registered_hypotheses"])
_print_block("V6 AMENDMENT: EXPLORATORY (no prediction)", AMENDMENT_V6["exploratory_without_prediction"])
_print_block("V6 AMENDMENT: REMOVED FROM V5", AMENDMENT_V6["removed_from_v5"])
_print_block("V6.1 AMENDMENT: STATUS", AMENDMENT_V6_1["status"])
_print_block("V6.1 AMENDMENT: REGISTERED HYPOTHESES H11a-b", AMENDMENT_V6_1["registered_hypotheses"])
_print_block("V6.1 AMENDMENT: WITHDRAWN", AMENDMENT_V6_1["withdrawn"])
_print_block("V6.1 AMENDMENT: MOVED TO EXPLORATORY", AMENDMENT_V6_1["moved_to_exploratory"])
_print_block("V6.2 AMENDMENT: STATUS", AMENDMENT_V6_2["status"])
_print_block("V6.2 AMENDMENT: EXPLORATORY ANALYSES", AMENDMENT_V6_2["exploratory_analyses"])
_print_block("V6.3 AMENDMENT: STATUS", AMENDMENT_V6_3["status"])
_print_block("V6.3 AMENDMENT: EXPLORATORY ANALYSES", AMENDMENT_V6_3["exploratory_analyses"])
_print_block("V6.4 AMENDMENT: STATUS", AMENDMENT_V6_4["status"])
_print_block("V6.4 AMENDMENT: EXPLORATORY ANALYSES", AMENDMENT_V6_4["exploratory_analyses"])
_print_block("V6.4 AMENDMENT: WHY THE SYNTHESIS IS DESCRIPTIVE", AMENDMENT_V6_4["why"])
_print_block("V6.5 AMENDMENT: STATUS", AMENDMENT_V6_5["status"])
_print_block("V6.5 AMENDMENT: REGISTERED HYPOTHESES H12-H13", AMENDMENT_V6_5["registered_hypotheses"])
_print_block("V6.5 AMENDMENT: EXPLORATORY ANALYSES", AMENDMENT_V6_5["exploratory_analyses"])
_print_block("V6.5 AMENDMENT: WHY", AMENDMENT_V6_5["why"])
_print_block("V6.6 AMENDMENT: STATUS", AMENDMENT_V6_6["status"])
_print_block("V6.6 AMENDMENT: REGISTERED HYPOTHESIS H14", AMENDMENT_V6_6["registered_hypotheses"])
_print_block("V6.6 AMENDMENT: CORRECTIONS", AMENDMENT_V6_6["corrections"])
_print_block("V6.6 AMENDMENT: EXPLORATORY ANALYSES", AMENDMENT_V6_6["exploratory_analyses"])
_print_block("V6.6 AMENDMENT: WHY", AMENDMENT_V6_6["why"])
_print_block("INVENTORY (record of this notebook version, not part of the lock)",
             {"models": ", ".join(IDEA_LOCK["models_actually_registered_this_notebook"]),
              "datasets": " | ".join(IDEA_LOCK["datasets_actually_used_this_notebook"])})
print(f"{len(_LOCKED_FIELDS)} locked fields, {len(AMENDMENT_V6['registered_hypotheses'])} v6 hypotheses (H10 withdrawn and "
      f"H9 exploratory in v6.1), {len(AMENDMENT_V6_1['registered_hypotheses']) - 1} v6.1 hypotheses and "
      f"{len(AMENDMENT_V6_5['registered_hypotheses']) - 1} v6.5 hypotheses printed.")

In [ ]:
# Cell No: 003
# 0.2 Implementation decisions, corrections and deviations (written to outputs/manifest/ in Section 13). Each was fixed before
#     any model output of the version that introduced it was seen, except D51, D57 and D59 (re-analyses of outputs already
#     read, hence descriptive or exploratory) and the v6.6 corrections D62-D63, made after reading the v6.5 run.
DECISIONS = {
    "D1_testbed": "CONVOY-3D ray-cast street scenes replace the 2-D compositing testbed (correction).",
    "D2_validity_gate": "static-pixel log-depth slope in [0.7, 1.3] and AbsRel <= 0.25; LEGACY-FLAT negative control.",
    "D3_scale": "median ratio over all static pixels of the image.",
    "D4_mover_criteria": "measurable: >= 40 eroded visible px and >= 85% visible; identifiable: parallax >= 5 px.",
    "D5_masks": "M1 epipolar residual, M2 rigid-flow residual, M3 Easi3R Eq. 9 (3-frame graph; v5 only, replaced by D23), "
                "M4 confidence; recall at 5% static FPR and AUROC (correction).",
    "D6_precision": "fp16 if compute capability < 8 else bf16; MapAnything and CroCo models fp32 (correction).",
    "D7_input_sizes": {"CroCo": [512, 384], "VGGT": [504, 378], "pi3": [560, 420], "MapAnything": [504, 378]},
    "D8_depth_outputs": {"CroCo": "pts3d[...,2] of view 1", "VGGT": "depth head", "pi3": "local_points[...,2]",
                         "MapAnything": "depth_z (apply_mask=False)"},
    "D9_single_image_reference": "one image (multi-view models) or the image paired with itself (CroCo).",
    "D10_datasets": ["TUM fr3 structure_texture_far", "TUM fr3 long_office_household", "TUM fr3 walking_xyz",
                     "TUM fr3 walking_halfsphere"],
    "D11_dependencies": "repositories at pinned commits, no requirements.txt, pip under constraints (correction).",
    "D12_evaluation": "model-major, cached, resumable; figures/tables read cached results only.",
    "D13_statistics": "cluster bootstrap (clip / image pair), 10,000 resamples, Holm-Bonferroni.",
    "D14_verdicts": "three levels, thresholds from the locked predictions (Section 1 table).",
    "D15_e4": "straight/curved twins, multi-view models only.",
    "D16_figures": "5.5 x 1.8 in and 2.8 x 2.0 in, exact size, embedded fonts, layout QA.",
    "D17_extensions": "exploratory, no locked prediction: M5 (two-view vs single-image depth) as a remedy; "
                      "convoy corollary of Proposition 2 (object copying the camera displacement), verified numerically.",
    "E1_erratum_prop3": "in-plane perpendicular motion is also undetectable; regimes use the exact offset.",
    "E2_erratum_A2": "locked first-order offset fails; corrected first-order formula reported; exact offset used.",
    # ---- v6 ----
    "D18_reframing": "beta_s read as the geometry share (oracle-validated); P1-P4 re-evaluated under their locked rules; "
                     "H5-H10 registered before any v6 data (AMENDMENT_V6).",
    "D19_environment": "Kaggle detected before Colab; outputs always under /kaggle/working on Kaggle (correction).",
    "D20_checkpoints": "per-stage archives checkpoints/<stage>.zip with fingerprints; final zip = outputs/ + checkpoints/; "
                       "attached archives restore matching stages exactly.",
    "D21_ablation": "E5: 72 base scenes x 3 parallel movers x 5 conditions (G, NS, CH, F, FH); wall by ray casting; contact "
                    "visibility measured on the renders; paired by mover, cluster = base scene.",
    "D22_vkitti": "Virtual KITTI 2 clone/Camera_0, 500x375 centre crop, frames by HTTP range requests, conventions from data (A3), "
                  "movers = vehicles flagged moving; MASt3R and VGGT trained on Virtual KITTI (flagged).",
    "D23_easi3r_long": "(v6.0 only; withdrawn in v6.1, D33) 10-frame clips, window-5 symmetric graph, Eq. 9 map, k-means (k=64) cluster refinement on encoder "
                       "tokens, Otsu binary mask; first pass only; middle frame evaluated.",
    "D24_removed": "real-people slope analysis and TUM walking model evaluation removed; walking kept for prevalence only.",
    "D25_figures": "no bold text anywhere; layout QA includes legend-data overlap; colour-blind-safe palettes.",
    "D26_v5_sets": "MAIN, knockout, E4, LEGACY-FLAT and CONVOY-REAL regenerated from identical specifications and seeds; "
                   "MAIN renders frames 0-1 only.",
    "D27_m1_resolution": "M1 from exact correspondences compared at 1e-6 px; float64 noise no longer separates a parallel "
                         "mover from the static scene (correction).",
    "D28_gate_everywhere": "validity gate applied to every per-model verdict (E1-E4, A1, CONVOY-REAL, E5, E6); failing "
                           "combinations listed in T2, left out of paper tables and figures, named in captions (correction).",
    "D29_long_clips": "E7 used 64 ten-frame clips (v6.0 only; withdrawn in v6.1, D33).",
    "D30_m1_vkitti": "exact-correspondence M1 not used on Virtual KITTI 2 (0 px threshold, pose precision).",
    "D31_rebuild_outputs": "figures, tables and analysis results rebuilt each run; leftovers of earlier runs deleted first.",
    # ---- v6.1 ----
    "D32_memory": "the v6 run was killed by the host (out of RAM) at Figure 3 after 6.5 h; v6.1 returns freed heap to the "
                  "system (glibc arena limits, malloc_trim), releases memory after every stage and figure, logs RAM at "
                  "each milestone (manifest/memory_log.csv), shows saved figure PNGs instead of re-rendering them, and "
                  "defers a model whose load would leave < 5 GiB of RAM (correction).",
    "D33_easi3r_withdrawn": "E7 / H10 withdrawn: the attention map detected movers in neither regime in the v6 run "
                            "(AMENDMENT_V6_1); no attention-based mask is evaluated in v6.1.",
    "D34_vkitti_measurable": "Virtual KITTI 2 measurability = the CONVOY-3D pixel rule (>= 40 eroded visible px, occupancy "
                             ">= 0.85); the v6.0 rule left 2 vehicles. H9 reported as exploratory; attrition table printed.",
    "D35_e5_real": "E5-REAL: the same crate grounded with a contact shadow (G) or lifted 0.25-0.50 m without one (F) in real "
                   "TUM pairs; RANSAC support plane on measured depth referenced to the motion-capture vertical; "
                   "baseline within 6 deg of the plane; delta = s b; H11a-b registered before the data (AMENDMENT_V6_1).",
    # ---- v6.2 ----
    "D37_driving_mechanism": "exploratory: VKITTI 2 wheel-contact visibility from 3-D boxes (convention decided by IoU with the "
                             "2-D boxes), per-object geometry share g, the convoy case; the D34 occupancy rule is dropped there.",
    "D38_kitti": "KITTI tracking attached as a Kaggle dataset (free registration): OXTS poses checked by A3 on LiDAR, LiDAR "
                 "depth, correspondences from labelled boxes, parallel threshold = label-noise floor of parked cars; exploratory.",
    "D39_masks_by_condition": "exploratory: M2 recall per E5 condition on the same parallel movers.",
    "D40_protocol_check": "multi-view single-image reference also computed with the image given twice; protocol-dependent "
                          "models make no single-image claim.",
    "D41_archives": "two archives: convoy_checkpoints.zip (resume) and convoy_paper_outputs.zip (figures, tables, results, "
                    "manifests); download links in the output, optional copy to a private Kaggle dataset via the API; model "
                    "weights deleted after each model; page cache counted as reclaimable in the RAM guard (correction).",
    "D36_load_once": "each model is loaded once per session: load check and every testbed on the same copy, then unloaded "
                     "and memory released; each (model, testbed) result is appended to convoy_checkpoints.zip the moment "
                     "it is written, so an interrupted session loses at most the stage that was running.",
    "D42_argoverse2": "Argoverse 2 validation logs from the public bucket (16 logs, <= 12 front-centre pairs each, 640x480), "
                      "A3 on static depth, static grey values and moving-vehicle grey values; KITTI rules unchanged (exploratory).",
    "D43_lidar_time": "Argoverse 2 vehicle returns carried to the image time along the labelled track (relative motion only).",
    "D44_zbuffer": "projected LiDAR keeps a pixel only if its return is the nearest surface in a 5x5 window (correction).",
    "D45_vk_sample": "Virtual KITTI 2: <= 150 pairs per scene; driving intervals resample vehicle tracks; matched "
                     "hidden/visible contact comparison on log depth, log area and x_geo.",
    "D46_checkpoint_messages": "every stage prints RESTORED / RUNNING / SAVED; attached data searched automatically.",
    "D47_captions": "bold lead phrase per caption panel; figures/captions.tex with LaTeX figure environments; graphics in "
                    "regular weight.",
    "D48_fig3": "Figure 3(c) without legend (colours as in (a)); A1 caption corrected (correction).",
    # ---- v6.4 ----
    "D49_av2_extension": "48 further Argoverse 2 validation logs in a second stage (same order and rules); the v6.3 stage is "
                         "kept, so a resumed run adds only the new logs; new pairs evaluated as eval:<model>:AV2X, pooled as AV2.",
    "D50_av2_contact": "exploratory: wheel contact of real vehicles from geometry (image frame, own body, other labelled boxes, "
                       "static LiDAR just above the point); labels, contrast and matching as in D37/D45; the rule is checked "
                       "against the instance-mask labels on Virtual KITTI 2 (T15d).",
    "D51_support_synthesis": "descriptive: every testbed reported by ground support (Section 6.9, Fig. 10, T18), with each "
                             "testbed's contrast and the depth overestimate exp(beta x) implied at s = 0.85; no new verdict.",
    "D52_convoy_real_support": "CONVOY-REAL movers are all floating crates; compared with the CONVOY-3D floating crates, not the "
                               "pooled slope (correction).",
    "D53_claim_references": "claim audit: C7 tables only, C8 Fig. 7b and real Argoverse 2 share, C12 Fig. S3 (correction).",
    "D54_fig9": "Figure 9 driving row uses the driving pixel rule; figure height follows the rows (correction).",
    # ---- v6.5 ----
    "D55_designed_experiments": "registered H12 (baseline sweep, BSW: baselines x0.5, x1, x2, x3 of the same scene, rotation, "
                                "first image, s and eps fixed) and H13 (contact dose-response, WDR: walls hiding the lowest 15, 35 "
                                "and 70 cm); 64 new base scenes each; estimators, samples and verdicts as E5.",
    "D56_bonn": "exploratory: Bonn RGB-D Dynamic, four sequences; undistortion; pose T_ROS^-1 T T_ROS T_m chosen by A3; moving "
                "regions as TUM walking; people by Mask R-CNN; objects with >= 150 px; regimes with a static-patch noise floor; "
                "x_geo with a per-pair static scale check; floor height by RANSAC; per-object g only (no slope, D24).",
    "D57_convoy_real_parallax": "the CONVOY-REAL vs floating-crate comparison is confounded by parallax (about 69 vs 13-17 px); "
                                "repeated within parallax bins and against the cue model (correction of the D52 reading).",
    "D58_static_twin": "exploratory: Proposition 4; stage av2_twin (camera height above each vehicle's contact plane, box size, "
                       "contact points); twin test vs epipolar test at 5% false positives on parked vehicles.",
    "D59_cue_model": "exploratory: rho - rho_mono = w (x - rho_mono), w = c_support d^2 / (d^2 + kappa^2); fitted on CONVOY-3D and "
                     "E5, 10-fold CV over scenes against kappa = 0 and a single c; out-of-sample slope predictions.",
    "D60_figures": "every panel titled '(a) Title' in regular weight; new Figures 11-14 and S4-S5.",
    "D61_resume_v64": "v6.5 work only in new stages and families; every v6.4 fingerprint unchanged.",
    # ---- v6.6 ----
    "D62_bonn_ground_truth": "correction: relative pose of every Bonn pair re-estimated from static correspondences (RANSAC PnP "
                             "of measured 3-D points against their flow positions; pairs whose fit fails are left out); pixels "
                             "hidden in the second image not tested; a moving region is an object only if the static "
                             "prediction fails photometrically (ZNCC), it contains a 5 px disc and covers <= 20% of the image; "
                             "floor = lowest well-supported horizontal level; quality gate: Bonn not interpreted if the median "
                             "pair flags > 15% of its static scene.",
    "D63_twin_flow": "registered H14: the twin and epipolar tests with forward-backward checked optical flow at each vehicle's "
                     "LiDAR pixels and a RANSAC road plane from the LiDAR (no labelled box), thresholds at 5% false "
                     "positives on parked vehicles; stage av2_flowtwin.",
    "D64_compact_notebook": "explanatory text cells removed (the notebook stays below 1 MB); the record of every decision, "
                            "amendment and registered prediction is kept in code (Sections 0.1-0.4) and written to the manifest.",
}
print(f"{len(DECISIONS)} decisions recorded.")

In [ ]:
# Cell No: 004
# 0.3.1 Environment detection and canonical paths (single source of truth).
import os, sys, platform, json, time, subprocess, importlib, importlib.util, shutil, gc, math
import warnings, hashlib, glob, re, io, contextlib, itertools, textwrap

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")      # keep the outputs readable
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)        # deprecation notices from model code
warnings.filterwarnings("ignore", message=".*xFormers.*")
warnings.filterwarnings("ignore", message="Mean of empty slice")                  # nan-aware summaries of empty groups
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="An input array is constant")          # Spearman on a constant prior weight


def _detect_env():
    # Kaggle is checked first: recent Kaggle images also ship the google.colab package, and the
    # previous version of this notebook took a Kaggle session for Colab and wrote its outputs to
    # /content, which Kaggle does not keep (correction D19).
    if os.path.isdir("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"


ENV_NAME = _detect_env()
_ROOT_OVERRIDE = os.environ.get("CONVOY_ROOT")
if _ROOT_OVERRIDE:
    ROOT_WORK = os.path.abspath(_ROOT_OVERRIDE)
elif ENV_NAME == "kaggle":
    ROOT_WORK = "/kaggle/working"
elif ENV_NAME == "colab":
    ROOT_WORK = "/content"
else:
    ROOT_WORK = os.path.abspath(".")

OUTPUT_DIR   = os.path.join(ROOT_WORK, "outputs")              # results, figures, tables, logs, manifests
CKPT_DIR     = os.path.join(ROOT_WORK, "checkpoints")          # one archive per finished stage (resume points)
ZIP_PATH     = os.path.join(ROOT_WORK, "convoy_checkpoints.zip")    # resume archive: every finished stage, nothing else
PAPER_ZIP    = os.path.join(ROOT_WORK, "convoy_paper_outputs.zip")  # figures, tables, results, manifests for the paper
LOG_DIR      = os.path.join(OUTPUT_DIR, "logs")
FIGURE_DIR   = os.path.join(OUTPUT_DIR, "figures")
TABLE_DIR    = os.path.join(OUTPUT_DIR, "tables")               # formatted paper tables (csv/json/tex)
RESULTS_DIR  = os.path.join(OUTPUT_DIR, "results")              # unformatted numeric results of every analysis
MANIFEST_DIR = os.path.join(OUTPUT_DIR, "manifest")
CACHE_DIR    = os.path.join(OUTPUT_DIR, "cache")               # compact per-model measurements

if _ROOT_OVERRIDE:
    SCRATCH_DIR = os.path.join(ROOT_WORK, "scratch")
elif ENV_NAME in ("kaggle", "colab"):
    SCRATCH_DIR = "/tmp/convoy_scratch"                         # large working data; archived per stage in checkpoints/
else:
    SCRATCH_DIR = os.path.join(ROOT_WORK, "convoy_scratch")
DATA_DIR    = os.path.join(SCRATCH_DIR, "data")                 # downloaded datasets
RENDER_DIR  = os.path.join(SCRATCH_DIR, "renders")              # CONVOY-3D renders and CONVOY-REAL composites
FLOW_DIR    = os.path.join(SCRATCH_DIR, "flow")                 # optical-flow cache
VK_DIR      = os.path.join(SCRATCH_DIR, "vkitti")               # Virtual KITTI 2 frames used by this run
REPO_DIR    = os.path.join(SCRATCH_DIR, "repos")                # cloned model code

for _p in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, FIGURE_DIR, TABLE_DIR, RESULTS_DIR, MANIFEST_DIR, CACHE_DIR,
           SCRATCH_DIR, DATA_DIR, RENDER_DIR, FLOW_DIR, VK_DIR, REPO_DIR]:
    os.makedirs(_p, exist_ok=True)
# Figures, tables and analysis results are rebuilt by every run from the stage outputs, so files left
# by an earlier run in the same folder (e.g. a figure this run does not draw) are removed first.
_n_old = 0
for _p in [FIGURE_DIR, TABLE_DIR, RESULTS_DIR]:
    for _fn in os.listdir(_p):
        if os.path.isfile(os.path.join(_p, _fn)):
            os.remove(os.path.join(_p, _fn)); _n_old += 1
if _n_old:
    print(f"Removed {_n_old} figure/table/result files of an earlier run in this folder (they are rebuilt below).")

for _k in ["ENV_NAME", "ROOT_WORK", "OUTPUT_DIR", "CKPT_DIR", "ZIP_PATH", "PAPER_ZIP", "LOG_DIR", "FIGURE_DIR", "TABLE_DIR",
           "RESULTS_DIR", "MANIFEST_DIR", "CACHE_DIR", "SCRATCH_DIR", "DATA_DIR", "RENDER_DIR", "FLOW_DIR",
           "VK_DIR", "REPO_DIR"]:
    print(f"{_k:<13}= {globals()[_k]}")
_du = shutil.disk_usage(SCRATCH_DIR)
print(f"\nFree disk at scratch location: {_du.free / 2**30:.1f} GiB")
if ENV_NAME == "colab":
    print("Colab: files in /content disappear when the runtime ends. Download convoy_checkpoints.zip at the end "
          "(or copy checkpoints/ to Google Drive) to resume later.")
# ---- memory (D32): allocator settings, explicit release, and a report at every milestone ----
# Large numpy/torch buffers are mapped from the OS and handed back when freed (instead of being kept
# in fragmented heaps), the number of allocator arenas is capped for the rendering threads, and
# release_memory() returns everything that is free to the OS. mem_report() prints this process's
# resident memory, the memory still available to it (container limit, page cache excluded) and the
# GPU memory in use; MEMORY_LOG is written to outputs/manifest/memory_log.csv.
import ctypes
import numpy as np
try:
    _LIBC = ctypes.CDLL("libc.so.6")
    _LIBC.mallopt(-3, 8 << 20)                 # M_MMAP_THRESHOLD: blocks >= 8 MiB are mmapped
    _LIBC.mallopt(-1, 64 << 20)                # M_TRIM_THRESHOLD
    _LIBC.mallopt(-8, 4)                       # M_ARENA_MAX
except (OSError, AttributeError):
    _LIBC = None
MEMORY_LOG = []


def _read_first(paths):
    for p in paths:
        try:
            with open(p) as f:
                return f.read().strip()
        except OSError:
            continue
    return None


def ram_status():
    # GiB: process resident set, memory available to this process, and the limit it lives under
    info = {}
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                k, v = line.split(":", 1)
                info[k] = int(v.split()[0]) / 2**20
    except OSError:
        pass
    rss = np.nan
    try:
        with open("/proc/self/status") as f:
            rss = next(int(l.split()[1]) / 2**20 for l in f if l.startswith("VmRSS"))
    except (OSError, StopIteration):
        pass
    total, avail = info.get("MemTotal", np.nan), info.get("MemAvailable", np.nan)
    lim = _read_first(["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"])
    use = _read_first(["/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"])
    stat = _read_first(["/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"]) or ""
    if lim and lim != "max" and use and int(lim) < (1 << 50):
        # file-backed page cache (downloaded weights, datasets) is counted in the cgroup's usage but is
        # reclaimed on demand; only shared memory (tmpfs) is not (D32, corrected in v6.2)
        kv = {l.split()[0]: int(l.split()[1]) for l in stat.splitlines() if len(l.split()) == 2 and l.split()[1].isdigit()}
        cache = kv.get("file", kv.get("total_cache", kv.get("inactive_file", kv.get("total_inactive_file", 0))))
        shmem = kv.get("shmem", kv.get("total_shmem", 0))
        reclaim = max(cache - shmem, 0)
        total = int(lim) / 2**30
        avail = total - (int(use) - reclaim) / 2**30
    return dict(rss=rss, available=avail, total=total)


def release_memory():
    gc.collect(); gc.collect()
    if "torch" in sys.modules:
        _t = sys.modules["torch"]
        try:
            if _t.cuda.is_available() and _t.cuda.is_initialized():
                _t.cuda.synchronize(); _t.cuda.empty_cache(); _t.cuda.ipc_collect()
            if hasattr(_t._C, "_host_emptyCache"):
                _t._C._host_emptyCache()               # pinned host memory cached by torch
        except Exception:
            pass
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass


def mem_report(tag, quiet=False):
    r = ram_status()
    gpu = (np.nan, np.nan)
    if "torch" in sys.modules:
        _t = sys.modules["torch"]
        try:
            if _t.cuda.is_available() and _t.cuda.is_initialized():
                gpu = (_t.cuda.memory_allocated() / 2**30, _t.cuda.memory_reserved() / 2**30)
        except Exception:
            pass
    MEMORY_LOG.append(dict(time_utc=time.strftime("%H:%M:%S", time.gmtime()), milestone=tag, process_rss_gib=r["rss"],
                           available_gib=r["available"], total_gib=r["total"], gpu_allocated_gib=gpu[0],
                           gpu_reserved_gib=gpu[1]))
    if not quiet:
        g = f" | GPU {gpu[0]:.1f} GiB in use ({gpu[1]:.1f} reserved)" if np.isfinite(gpu[0]) else ""
        print(f"[memory] {tag}: this process {r['rss']:.1f} GiB | available {r['available']:.1f} of {r['total']:.1f} GiB{g}")
    return r


release_memory(); _ = mem_report("start")

In [ ]:
# Cell No: 005
# 0.3.2 Minimal dependency installation under a constraints file.
#       Freezes the preinstalled numerical stack so no install below can replace it.
from importlib import metadata as _md

_FREEZE_DISTS = ["torch", "torchvision", "torchaudio", "numpy", "opencv-python",
                 "opencv-python-headless", "opencv-contrib-python", "pillow", "scipy",
                 "pandas", "matplotlib"]
_constraint_lines = []
for _d in _FREEZE_DISTS:
    try:
        _constraint_lines.append(f"{_d}=={_md.version(_d)}")
    except _md.PackageNotFoundError:
        pass
CONSTRAINTS_FILE = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
with open(CONSTRAINTS_FILE, "w") as _f:
    _f.write("\n".join(_constraint_lines) + "\n")

# (import name, pip requirement, needed by)
REQUIRED_PACKAGES = [
    ("numpy", "numpy", "everything"),
    ("scipy", "scipy", "statistics, root finding"),
    ("pandas", "pandas", "result tables"),
    ("pyarrow", "pyarrow", "Argoverse 2 tables and LiDAR sweeps (feather)"),
    ("matplotlib", "matplotlib", "figures"),
    ("PIL", "pillow", "image I/O"),
    ("cv2", "opencv-python-headless", "DIS optical flow, PnP, image ops"),
    ("torch", "torch", "models"),
    ("torchvision", "torchvision", "RAFT optical flow"),
    ("huggingface_hub", "huggingface_hub", "checkpoint download"),
    ("safetensors", "safetensors", "checkpoint loading"),
    ("einops", "einops", "DUSt3R family, VGGT"),
    ("roma", "roma", "DUSt3R family"),
    ("omegaconf", "omegaconf", "MapAnything"),
    ("hydra", "hydra-core", "MapAnything"),
    ("box", "python-box", "MapAnything"),
    ("orjson", "orjson", "MapAnything"),
    ("natsort", "natsort", "MapAnything"),
    ("jaxtyping", "jaxtyping", "MapAnything (uniception)"),
    ("timm", "timm", "MapAnything (uniception)"),
    # installed WITHOUT dependencies: its requirement list includes torchaudio and other
    # packages that the model code never imports; the ones it does import are listed above
    ("uniception", "uniception==0.1.7 --no-deps", "MapAnything (encoders and heads)"),
]

INSTALL_LOG = []
for _mod, _req, _why in REQUIRED_PACKAGES:
    _status = "preinstalled"
    if importlib.util.find_spec(_mod) is None:
        _cmd = [sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS_FILE] + _req.split()
        _rc = subprocess.run(_cmd, capture_output=True, text=True, timeout=1800)
        importlib.invalidate_caches()
        if _rc.returncode == 0 and importlib.util.find_spec(_mod) is not None:
            _status = "installed now"
        else:
            _status = "INSTALL FAILED"
            print(f"[install] {_req} failed:\n{_rc.stderr[-1500:]}")
    _dist = _req.split()[0].split("==")[0]
    try:
        _ver = _md.version(_dist)
    except _md.PackageNotFoundError:
        _ver = "-"
    INSTALL_LOG.append(dict(package=_dist, import_name=_mod, version=_ver, status=_status, needed_by=_why))

print(f"Constraints file ({len(_constraint_lines)} frozen distributions): {CONSTRAINTS_FILE}")
for _l in _constraint_lines:
    print("   ", _l)
print()
print(f"{'package':<26}{'version':<14}{'status':<16}needed by")
for _r in INSTALL_LOG:
    print(f"{_r['package']:<26}{_r['version']:<14}{_r['status']:<16}{_r['needed_by']}")
_n_fail = sum(r["status"] == "INSTALL FAILED" for r in INSTALL_LOG)
print(f"\n{len(INSTALL_LOG) - _n_fail}/{len(INSTALL_LOG)} required packages available.")
with open(os.path.join(LOG_DIR, "install_log.json"), "w") as _f:
    json.dump(INSTALL_LOG, _f, indent=2)

In [ ]:
# Cell No: 006
# 0.3.3 GPU requirement and numerical precision policy.
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. This notebook requires one GPU (T4-class). "
        "Kaggle: Settings -> Accelerator -> GPU T4. Colab: Runtime -> Change runtime type -> GPU.")
torch.cuda.set_device(0)
device = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
GPU_TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30

# Precision policy. bfloat16 needs compute capability >= 8.0 (Ampere). On a T4 (7.5)
# bf16 is either emulated (slow) or, for code that forces FlashAttention when it sees
# bf16 inputs (pi3), fails with "No available kernel" -- the failure seen in the
# previous version of this notebook. fp16 is the documented fallback of VGGT and pi3.
AMP_DTYPE = torch.bfloat16 if GPU_CAPABILITY[0] >= 8 else torch.float16

print(f"GPU            : {GPU_NAME}  (compute capability {GPU_CAPABILITY[0]}.{GPU_CAPABILITY[1]}, "
      f"{GPU_TOTAL_GIB:.1f} GiB)")
print(f"CUDA / torch   : {torch.version.cuda} / {torch.__version__}")
print(f"Mixed precision: {AMP_DTYPE} (bf16 only when compute capability >= 8.0)")
if torch.cuda.device_count() > 1:
    print(f"{torch.cuda.device_count()} GPUs visible; only cuda:0 is used.")
if "T4" not in GPU_NAME:
    print(f"Note: GPU is '{GPU_NAME}', not a T4. Sample sizes are unchanged; runtimes will differ.")

In [ ]:
# Cell No: 007
# 0.3.4 Seeds, deterministic settings and the environment record.
import random
import pandas as pd
import scipy
import matplotlib
import PIL
import cv2

GLOBAL_SEED = 20270918
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# Full torch.use_deterministic_algorithms(True) is NOT enabled: several attention and
# upsampling kernels used by the evaluated models have no deterministic CUDA
# implementation and would raise. Inference-only forward passes are reproducible to
# within fp16 rounding; Section 0.3.5 measures this directly.
cv2.setNumThreads(max(1, (os.cpu_count() or 2)))

def seed_for(*keys):
    # Stable 32-bit seed derived from GLOBAL_SEED and any hashable keys.
    h = hashlib.sha256(("|".join(map(str, (GLOBAL_SEED,) + keys))).encode()).hexdigest()
    return int(h[:8], 16)

ENV_LOG = {
    "env_name": ENV_NAME, "python": sys.version.split()[0], "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__, "pillow": PIL.__version__, "opencv": cv2.__version__,
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": GPU_NAME,
    "gpu_capability": f"{GPU_CAPABILITY[0]}.{GPU_CAPABILITY[1]}", "amp_dtype": str(AMP_DTYPE),
    "cpu_count": os.cpu_count(), "global_seed": GLOBAL_SEED,
    "start_time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(os.path.join(LOG_DIR, "env_log.json"), "w") as _f:
    json.dump(ENV_LOG, _f, indent=2)
for _k, _v in ENV_LOG.items():
    print(f"{_k:<16}{_v}")
RUN_T0 = time.time()
STAGE_TIMES = {}

@contextlib.contextmanager
def stage(name):
    # Wall-clock timer for each expensive stage; reported in Section 13.
    t0 = time.time()
    try:
        yield
    finally:
        STAGE_TIMES[name] = STAGE_TIMES.get(name, 0.0) + time.time() - t0

In [ ]:
# Cell No: 008
# 0.3.5 Determinism self-check: identical inputs must give identical outputs.
_rng_a = np.random.default_rng(seed_for("check")).standard_normal(5)
_rng_b = np.random.default_rng(seed_for("check")).standard_normal(5)
assert np.array_equal(_rng_a, _rng_b), "NumPy generator is not reproducible"

_g = torch.Generator(device="cpu").manual_seed(GLOBAL_SEED)
_x = torch.randn(2, 3, 64, 64, generator=_g).to(device)
_conv = torch.nn.Conv2d(3, 8, 3, padding=1).to(device)
with torch.no_grad():
    _y1 = _conv(_x)
    _y2 = _conv(_x)
    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE):
        _y3 = _conv(_x).float()
        _y4 = _conv(_x).float()
_det_fp32 = float((_y1 - _y2).abs().max())
_det_amp = float((_y3 - _y4).abs().max())
print(f"NumPy generator reproducible           : True")
print(f"Repeated fp32 convolution, max |diff|  : {_det_fp32:.2e}")
print(f"Repeated {str(AMP_DTYPE).split('.')[-1]} convolution, max |diff|: {_det_amp:.2e}")
assert _det_fp32 == 0.0 and _det_amp == 0.0, "repeated forward passes differ"
ENV_LOG["determinism_check"] = {"fp32_max_abs_diff": _det_fp32, "amp_max_abs_diff": _det_amp}
del _x, _conv, _y1, _y2, _y3, _y4

In [ ]:
# Cell No: 009
# 0.4 Run configuration and pre-registered constants.
#     Every threshold used by a verdict is fixed here, before any model is run. The v5 entries
#     are unchanged (so the v5 testbeds are regenerated identically); the v6 entries below them
#     were added for the experiments registered in the v6 amendment (Section 0.1), and each later
#     version only adds entries (v6.5: the designed experiments, Bonn, the twin test and the cue
#     model, AMENDMENT_V6_5), so every stage of an earlier version keeps its fingerprint. v6.6 adds
#     H14 and FLOWTWIN_DESIGN and extends BONN_DESIGN (D62), so only the Bonn stage is recomputed.
RUN_CONFIG = {
    # image format shared by CONVOY-3D and TUM (640x480, 4:3)
    "W": 640, "H": 480,
    # CONVOY-3D main set (scene specifications have 3 frames as in v5; frames 0 and 1 are rendered)
    "n_scenes_main": 240, "movers_per_scene": 3,
    # A1 correspondence knockout: frame 1 re-rendered without movers, first n scenes
    "n_scenes_knockout": 96,
    # LEGACY-FLAT negative control for the validity gate
    "n_scenes_flat": 24,
    # E4 multi-frame clips per camera path type (straight / curved)
    "n_clips_e4_per_path": 48, "e4_n_frames": 8, "e4_window_sizes": (2, 4, 8),
    # CONVOY-REAL: TUM static sequences, frame pairs per sequence, inserted movers per pair
    "n_real_pairs_per_seq": 60, "real_movers_per_pair": 2,
    # TUM walking: candidate pairs scanned for the prevalence of motion regimes
    "n_dyn_candidate_pairs_per_seq": 150,
    # statistics
    "n_boot": 10000, "alpha": 0.05,
    # parallel use of CPU cores for rendering
    "render_workers": max(1, min(4, os.cpu_count() or 1)),
    # ---- v6 ----
    # E5 causal ablation: base scenes (3 parallel movers each), rendered once per condition
    "n_scenes_ablation": 72,
    # E6 Virtual KITTI 2: scenes and two-view pairs per scene (v6.3: at most 150 per scene, was 60; the
    # three short scenes are limited by their length, so only Scene18 and Scene20 gain pairs)
    "n_vk_scenes": 5, "vk_pairs_per_scene": 150,
    # ---- v6.1 ----
    # E5-REAL contact ablation on real TUM backgrounds: frame pairs per static sequence, base movers per pair
    "n_rabl_pairs_per_seq": 40, "rabl_movers_per_pair": 2,
    # KITTI tracking (only if attached): pairs per sequence and sequences used
    "n_kt_pairs_per_seq": 20, "n_kt_sequences": 12,
    # archives: copy them to a private Kaggle dataset when Kaggle API secrets are attached (Section 0.5)
    "dataset_backup": True, "dataset_backup_slug": "convoy-checkpoints",
    # ---- v6.3 ----
    # Argoverse 2 (real driving with LiDAR, downloaded automatically): validation logs and pairs per log
    "use_av2": True, "n_av2_logs": 16, "n_av2_pairs_per_log": 12,
    # ---- v6.4 ----
    # Argoverse 2 extension: further validation logs, taken in the same fixed order after the 16 logs of the v6.3
    # stage, which is kept unchanged (a session resumed from a v6.3 run restores it and adds only the new logs)
    "n_av2_logs_ext": 48,
    # ---- v6.5 (new stages only; every stage of v6.4 keeps its fingerprint) ----
    # designed experiments: base scenes of the baseline sweep and of the contact dose-response (3 parallel movers each)
    "n_scenes_bsw": 64, "n_scenes_wdr": 64,
    # Bonn RGB-D Dynamic (real balloons and boxes moved by people, measured depth, motion-capture poses)
    "use_bonn": True, "n_bonn_pairs_per_seq": 50,
}

PREREG = {
    # motion regimes (exact epipolar offset at the mover's centre, pixels)
    "eps_parallel_max": 1.0, "eps_perp_min": 3.0,
    # identifiability: translational parallax of the static-equivalent point (pixels)
    "parallax_min_px": 5.0,
    # static-scene validity gate (per model x testbed, medians over images)
    "gate_slope_range": (0.7, 1.3), "gate_absrel_max": 0.25,
    # Prediction 1
    "beta_support_range": (0.8, 1.2), "beta_ci_band": (0.5, 1.5), "r2_min": 0.5,
    # Prediction 2
    "recall_diff_min": 0.3, "mask_fpr": 0.05,
    # Prediction 3
    "spearman_min": 0.5, "w_separability_min": 0.10,
    # Prediction 4: straight-minus-curved beta_s difference at the longest window
    "e4_drop_min": 0.3,
    # object measurement
    "min_object_pixels": 40, "erosion_px": 2,
    # depth range of static pixels used for scale alignment / gate (metres)
    "static_depth_range_synth": (2.0, 80.0), "static_depth_range_real": (0.3, 4.5),
    # ---- v6 (registered in the amendment, before any v6 data was generated) ----
    "static_depth_range_vk": (2.0, 80.0),
    "h9_beta_max": 0.3, "h9_ci_hi_max": 0.5,           # H9: driving beta_geo below 0.3, CI below 0.5
    "h9_x_range": (-0.47, 1.90),                       # H9 sample: x_geo inside the CONVOY-3D design range of x
    "h9_min_movers": 20, "h9_min_pairs": 5,
    "min_movers_per_regime": 20,                       # a regime comparison is tested only with >= 20 movers each
    "contact_band_m": 0.12,                            # GT 'ground contact' pixels: <= 12 cm above the ground
    # ---- v6.1 (fixed after the first full v6 run, before any data of the analyses they govern) ----
    "vk_min_occupancy": 0.85,                          # Virtual KITTI 2 measurability: the CONVOY-3D pixel rule
                                                       # (>= 40 eroded visible px, >= 85% of the silhouette visible)
    "rabl_contact_band_m": 0.03,                       # E5-REAL: crate pixels within 3 cm of the support plane
    # ---- v6.2, exploratory analyses declared before their data existed (AMENDMENT_V6_2) ----
    "x6_min_eroded_px": 40,                            # driving sample: the pixel rule without the occupancy rule
    "x6_min_lever": 0.3,                               # per-object geometry share needs |x - rho_mono| >= 0.3
    "x6_pace_s": (0.85, 1.15),                         # "keeping pace with the camera": s in this range
    "x6_wheel_height_m": 0.10, "x6_wheel_depth_tol_m": 0.6,   # a wheel contact point is visible if the vehicle's own
                                                       # pixel is there at the point's depth (within 0.6 m)
    "kt_min_lidar_px": 40,                             # KITTI: LiDAR pixels on a vehicle (instead of eroded pixels)
    "kt_moving_disp_m": 0.5, "kt_parked_disp_m": 0.15, # world displacement of the labelled box between the two frames
    "kt_eps_noise_quantile": 0.9,                      # KITTI parallel regime: |eps| below this quantile of parked cars
    # ---- v6.3, exploratory analyses declared before their data existed (AMENDMENT_V6_3) ----
    # Argoverse 2 uses the KITTI rules above (kt_*) unchanged.
    "x6_match_caliper_sd": 0.5,                        # hidden/visible contact matched on log Z, log area and x_geo:
                                                       # nearest visible vehicle within 0.5 SD on every covariate
    # ---- v6.4, exploratory analyses declared before their data existed (AMENDMENT_V6_4) ----
    # Argoverse 2 ground contact uses the Virtual KITTI 2 wheel points and labels (x6_wheel_height_m, x6_match_caliper_sd)
    # with the visibility tests of AV2X_DESIGN; the synthesis (Section 6.9) adds no test and no threshold except this one:
    "implied_factor_s": 0.85,                          # depth overestimate implied by a slope for an object keeping pace
                                                       # with the camera at this s: exp(beta * x), x = -log(1 - s)
    # ---- v6.5, registered in AMENDMENT_V6_5 before any v6.5 data existed ----
    "h12_contrast": ("B3", "B05"),                     # H12: beta(baseline x3) - beta(baseline x0.5) > 0
    "h12_models": ("DUSt3R", "MASt3R"),                # ... predicted for these two models (Holm across them)
    "h13_contrast": ("W15", "G"),                      # H13: beta(15 cm wall) - beta(no wall) > 0, every model (Holm)
    "h13_contact_px_min": 5,                           # H13 sample: >= 5 contact pixels in both views without the wall,
                                                       # none with the 15 cm wall (the E5 rule for CH)
    "dose_min_movers": 8, "dose_min_scenes": 5,        # H12 / H13: NOT TESTABLE below 8 movers or 5 base scenes
    # exploratory (v6.5): the twin test, the cue model and Bonn
    "twin_fpr": 0.05,                                  # twin and epipolar tests: thresholds at 5% false positives on
                                                       # parked Argoverse 2 vehicles
    "bonn_min_object_px": 150,                         # Bonn: >= 150 eroded pixels with valid depth per object
    "bonn_floor_contact_m": 0.08, "bonn_floor_clear_m": 0.20,   # lowest point <= 8 cm above the floor: on the floor;
                                                       # >= 20 cm: off the floor (held or floating); otherwise undetermined
    # ---- v6.6, registered in AMENDMENT_V6_6 before any flow-based twin quantity existed ----
    "h14_min_vehicles": 20, "h14_min_tracks": 10,      # H14: on moving Argoverse 2 vehicles in the parallel regime, the twin
                                                       # test with optical flow flags more than the epipolar test with the
                                                       # same flow (paired difference, CI > 0); NOT TESTABLE below these
}

# Memory guard: a model is loaded only if the machine still has this much free RAM after the previous
# model was released (otherwise the run stops cleanly; every finished stage is already archived).
MEMORY_GUARD_GIB = 5.0

# Mover design (CONVOY-3D). Probabilities are exact proportions, not random draws:
# the sampler cycles a shuffled deck so every condition is represented as designed.
MOVER_DESIGN = {
    "classes": {"car": 0.40, "crate": 0.30, "crate_floating": 0.30},
    "eps_targets_px": {0.0: 0.40, 0.5: 0.08, 1.5: 0.07, 2.0: 0.08, 3.0: 0.09,
                       5.0: 0.10, 8.0: 0.10, 12.0: 0.08},
    "s_range_main": (-0.6, 0.85),
    "s_range_beyond": (1.10, 1.40), "p_beyond": 0.08,   # no static solution exists (Z* < 0)
    "depth_range_m": (5.0, 45.0),
}
CAMERA_DESIGN = {
    # heading of the optical axis relative to the direction of travel
    "forward": {"p": 0.40, "yaw_deg": (-8, 8), "baseline_m": (1.2, 3.0)},
    "oblique": {"p": 0.30, "yaw_deg": (20, 45), "baseline_m": (0.8, 2.0)},
    "side":    {"p": 0.30, "yaw_deg": (60, 80), "baseline_m": (0.5, 1.5)},
    "focal_px": (520.0, 820.0), "height_m": (1.3, 1.8), "rot_jitter_deg": 1.0,
}
# ---- v6 designs ----
# E5: the SAME mover rendered in five conditions; everything else in the scene is identical.
ABLATION_DESIGN = {
    "classes": {"car": 0.5, "crate": 0.5},             # base objects; every one is also lifted (floating car / crate)
    "conditions": ["G", "NS", "CH", "F", "FH"],
    "labels": {"G": "grounded", "NS": "no shadow", "CH": "contact hidden", "F": "floating", "FH": "floating, wall"},
    "lift_m": (0.8, 1.6),                              # F / FH: gap between the object and the ground
    "contact_cover_car_m": 0.70,                       # CH: wall hides the wheels (and up to 0.70 m)
    "contact_cover_crate_frac": 0.35,                  # CH: wall hides the lowest 35% of a crate
    "wall_gap_m": 0.6, "wall_max_cover_frac": 0.65, "wall_thickness_m": 0.25,
}
# E5-REAL (v6.1): the same crate inserted into a real TUM image pair twice, resting on the support plane
# (floor or table top, fitted to the measured depth) with a contact shadow (G), and lifted off it without
# a shadow (F). Purely parallel motion (delta = s b) on pairs whose baseline is almost parallel to the
# plane, so the grounded crate stays on the plane in both views.
RABL_DESIGN = {
    "conditions": ["G", "F"], "labels": {"G": "grounded, contact shadow", "F": "lifted, no shadow"},
    "crate_size_m": (0.15, 0.32), "lift_m": (0.25, 0.50), "depth_m": (0.7, 3.5),
    "max_baseline_tilt_deg": 6.0,                     # angle between the baseline and the support plane
    "vertical_max_tilt_deg": 12.0,                    # plane normal vs the motion-capture vertical (when it is identified)
    "plane_max_tilt_deg": 35.0,                       # fallback: plane normal vs the camera's up direction
    "plane_inlier_m": 0.02, "plane_fit_max_depth_m": 3.0, "plane_min_inlier_frac": 0.08, "plane_ransac_iters": 400,
    "on_plane_sigma_k": 2.5,                          # on-plane tolerance: max(2 cm, k sigma_Kinect(z)), Nguyen et al. 2012
    "shadow_strength": 0.55, "shadow_softness": 0.30,  # darkening at the footprint centre; soft edge (x size)
    "attempts": 300, "margin_px": 16, "max_candidates_per_seq": 600,
}
# E6: Virtual KITTI 2 (photo-realistic driving with exact depth, camera and object poses; no login)
# KITTI tracking (real driving, LiDAR, labelled 3-D boxes, OXTS poses). Not downloadable without a
# (free) registration: attach it as a Kaggle dataset (Section 3.10). Used for exploratory analyses only.
KT_DESIGN = {
    "root_env": "CONVOY_KITTI_ROOT", "classes": ("Car", "Van", "Truck"),
    "pair_baseline_m": (1.0, 3.0), "max_rot_deg": 10.0, "max_gap_frames": 10,
    "max_truncation": 0.3, "max_occlusion": 1, "lidar_min_depth_m": 2.0, "lidar_max_depth_m": 80.0,
    "static_box_margin_m": 0.3, "crop_h_max": 375,
    "zbuffer_window_px": 5, "zbuffer_tol": (0.10, 1.0),   # v6.3: occlusion test of projected LiDAR (Section 3.10.0)
}
# Argoverse 2 sensor dataset (v6.3, exploratory): real driving with two 32-beam LiDARs, labelled 3-D boxes with
# track IDs and ego poses, read anonymously from the public AWS Open Data bucket (CC BY-NC-SA 4.0). Front-centre
# camera (portrait 1550 x 2048, undistorted); the 4:3 window of the full width is resized to 640 x 480.
AV2_DESIGN = {
    "base_url": os.environ.get("CONVOY_AV2_BASE", "https://argoverse.s3.amazonaws.com/"),
    "prefix": "datasets/av2/sensor/", "split": "val", "camera": "ring_front_center",
    "classes": ("REGULAR_VEHICLE", "LARGE_VEHICLE", "BUS", "BOX_TRUCK", "TRUCK", "TRUCK_CAB", "SCHOOL_BUS",
                "ARTICULATED_BUS", "VEHICULAR_TRAILER"),
    "out_wh": (640, 480),
    "pair_baseline_m": (1.0, 3.0), "max_rot_deg": 10.0, "max_gap_frames": 20, "min_start_gap_frames": 10,
    "lidar_min_depth_m": 2.0, "lidar_max_depth_m": 80.0, "static_box_margin_m": 0.3, "own_box_margin_m": 0.1,
    "ground_cut_m": 0.15, "zbuffer_window_px": 5, "zbuffer_tol": (0.10, 1.0),
    "max_truncation": 0.3, "max_occlusion": 0.5, "min_moving_speed_mps": 1.0,
    "min_pairs_per_log": 4, "max_logs_tried": 60,
}
# Argoverse 2, v6.4 (exploratory, AMENDMENT_V6_4): the extension stage and the ground contact of real vehicles. Kept apart
# from AV2_DESIGN so that the v6.3 stage keeps its fingerprint.
AV2X_DESIGN = {
    "max_logs_tried_total": 150,                       # both stages together; the validation split has 150 logs
    # A wheel contact point (Virtual KITTI 2 geometry: x6_wheel_height_m above the box bottom at +-0.35 length,
    # +-0.40 width) is visible in the first image of a pair if (i) it projects into the image in front of the camera,
    # (ii) the vehicle's own body does not hide it: the camera lies beyond the side face or the end face next to that
    # wheel, and the line of sight does not cross the vehicle's own box above body_clearance_m (a bumper overhanging the
    # wheel) at least occluder_gap_m in front of the point, (iii) no other labelled object's box, shrunk by
    # occluder_shrink_m on every side, crosses the line of sight at least occluder_gap_m in front of the point, and
    # (iv) no static LiDAR return in the window of columns u +- 2 and
    # rows v - 8 ... v (the point's row and the rows above it, which the near road cannot reach) is nearer than the
    # point by more than max(10%, 1 m). Contact labels as for Virtual KITTI 2: visible with >= 1 visible wheel point,
    # hidden with none when >= 2 lie in the image, otherwise undetermined.
    "body_clearance_m": 0.30, "occluder_shrink_m": 0.10, "occluder_gap_m": 0.30,
    "lidar_window_cols": 2, "lidar_window_rows_above": 8, "lidar_occluder_tol": (0.10, 1.0),
}
VK_DESIGN = {
    "base_url": os.environ.get("CONVOY_VKITTI_BASE", "https://download.europe.naverlabs.com/virtual_kitti_2.0.3/"),
    "scenes": ["Scene01", "Scene02", "Scene06", "Scene18", "Scene20"], "variation": "clone", "camera": 0,
    "crop_wh": (500, 375),                             # centred 4:3 crop of the 1242 x 375 frames, native pixels
    "pair_baseline_m": (1.0, 3.0), "max_rot_deg": 10.0, "max_gap_frames": 8,
    "min_vehicle_px": 300, "sky_depth_m": 600.0,
}

# ---- v6.5 designs (AMENDMENT_V6_5); new dictionaries, so no v6.4 stage changes its fingerprint ----
# Baseline sweep (BSW): the same scene and movers seen with the camera baseline scaled by each factor. The second camera
# keeps its rotation and moves along the same line, C = C0 + factor (C1 - C0), and every mover moves factor x its
# displacement, so s, eps = 0 and the geometric answer x = -log(1 - s) are unchanged: only the parallax changes.
# Scenes are sampled with 4 frames, so the sampler's constraints (movers in view, apart and in the street) hold up to
# three baselines.
SWEEP_DESIGN = {
    "classes": {"car": 0.35, "crate": 0.30, "crate_floating": 0.35},
    "levels": {"B05": 0.5, "B1": 1.0, "B2": 2.0, "B3": 3.0},
    "labels": {"B05": "x0.5", "B1": "x1", "B2": "x2", "B3": "x3"},
    "n_frames_sampled": 4,
}
# Contact dose-response (WDR): the same grounded mover with no wall (G) and behind static walls that hide its lowest
# 15, 35 and 70 cm in both views (the E5 wall, designed by ray casting, with the cover height as the dose; 70 cm is the
# E5 cover of a car). Base scenes as in E5 (cars and crates, purely parallel), new seeds.
WALL_DOSE_DESIGN = {
    "classes": {"car": 0.5, "crate": 0.5},
    "levels": {"G": 0.0, "W15": 0.15, "W35": 0.35, "W70": 0.70},
    "labels": {"G": "no wall", "W15": "15 cm", "W35": "35 cm", "W70": "70 cm"},
}
# Bonn RGB-D Dynamic Dataset (Palazzolo et al., IROS 2019): TUM format, Asus Xtion depth registered to the RGB images,
# motion-capture poses of a marker body. The pose of the RGB camera is T_ROS^-1 T(t) T_ROS T_m (dataset page); T_m is a
# calibrated similarity (scale about 1.06), so camera centres are expressed in the sensor's depth units. The
# convention is checked on the data (A3), as for TUM. Images and depth are undistorted with the published coefficients.
BONN_DESIGN = {
    "base_url": os.environ.get("CONVOY_BONN_BASE", "https://www.ipb.uni-bonn.de/html/projects/rgbd_dynamic2019/"),
    "sequences": {"balloon": dict(file="rgbd_bonn_balloon.zip", object="balloon"),
                  "balloon2": dict(file="rgbd_bonn_balloon2.zip", object="balloon"),
                  "moving_nonobstructing_box": dict(file="rgbd_bonn_moving_nonobstructing_box.zip", object="box"),
                  "moving_nonobstructing_box2": dict(file="rgbd_bonn_moving_nonobstructing_box2.zip", object="box")},
    "intrinsics": dict(fx=542.822841, fy=542.576870, cx=315.593520, cy=237.756098, W=640, H=480),
    "distortion": [0.039903, -0.099343, -0.000730, -0.000144, 0.0],
    "depth_scale": 5000.0,
    "T_ros": [[-1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
    "T_m": [[1.0157, 0.1828, -0.2389, 0.0113], [0.0009, -0.8431, -0.6413, -0.0098], [-0.3009, 0.6147, -0.8085, 0.0111],
            [0.0, 0.0, 0.0, 1.0]],
    "pair_baseline_m": (0.05, 0.25), "max_rot_deg": 20.0,
    "person_score_min": 0.6, "person_dilate_px": 5,       # people: torchvision Mask R-CNN (COCO person class)
    "min_component_px": 400,                             # moving regions: the TUM walking rule of Section 3.7
    "static_patch_px": 40, "n_static_patches": 30,       # flow-noise floor: |eps| of static patches (as parked cars)
    # v6.6 (D62): the relative pose of every pair from its static correspondences (RANSAC PnP of frame i's measured points
    # against their flow positions in frame j, started from the motion-capture pose); pairs whose fit fails are left out
    "pnp_points": 6000, "pnp_reproj_px": 1.5, "pnp_min_inlier_frac": 0.4, "pnp_max_med_err_px": 0.8,
    "pnp_max_rot_change_deg": 5.0, "pnp_max_trans_change_m": 0.10,
    "max_object_frac": 0.20, "max_object_extent": 0.70,  # an object covers <= 20% of the image and <= 70% of each side
    # a pixel hidden in frame j behind something nearer (all depth within occl_window_px of its predicted position is
    # nearer) is not tested; a moving region is an object only if the static prediction fails photometrically (median
    # ZNCC of 7 x 7 patches between frame i and frame j warped with the rigid flow <= photo_max_zncc_static on >= 30
    # textured pixels), the measured flow explains frame j at least as well, it contains a disc of radius 5 px and, if the
    # floor is found, 10% of its points are at least 5 cm above the floor (a moving shadow on the floor is not an object)
    "occl_window_px": 2, "photo_window_px": 3, "photo_min_texture": 4.0, "photo_min_px": 30,
    "photo_max_zncc_static": 0.85, "photo_min_support": 0.0, "min_object_radius_px": 5.0, "min_object_height_m": 0.05,
    "floor_cam_height_m": (0.9, 2.2), "floor_min_frac": 0.02, "floor_max_tilt_deg": 10.0,   # the floor: the lowest level
    "qc_max_object_frac": 0.15,                          # Bonn is not interpreted if the objects of the median pair cover
                                                         # more than 15% of the image (the static scene would be flagged)
}
# The static twin (Proposition 4) on Argoverse 2: the camera height above each vehicle's contact plane from its labelled
# box, and the twin test against the epipolar test (exploratory, Section 6.13).
TWIN_DESIGN = {
    "s_bins": [-1.0, -0.4, -0.15, -0.05, 0.05, 0.15, 0.4, 0.7, 1.0],
    "theory_s": [-0.9, 0.95],
}
# The twin test with optical flow (v6.6, D63, H14): correspondences from RAFT flow (forward-backward checked), the road
# plane fitted by RANSAC to all LiDAR returns of the image (no labels), and the 10th percentile of the heights of a
# vehicle's static reconstruction above that plane; a static solution behind the camera for most pixels counts as none.
FLOWTWIN_DESIGN = {"road_depth_m": (2.0, 40.0), "road_inlier_m": 0.08, "road_max_tilt_deg": 10.0, "road_cam_height_m": (0.8, 3.0),
                   "ransac_iters": 300, "fb_px": 1.0, "fb_rel": 0.05, "min_px": 10, "low_pct": 10, "behind_frac": 0.5}
# The cue model (exploratory, Section 6.11): w = c_support * d^2 / (d^2 + kappa^2), d = translational parallax (px)
CUE_DESIGN = {
    "kappa_grid_px": [0.0] + [float(v) for v in np.round(np.logspace(0, 3, 61), 3)],
    "n_folds": 10, "parallax_bins_px": [0, 5, 10, 20, 40, 80, 1e9],
    "x_bins": [-0.6, -0.25, 0.0, 0.25, 0.5, 0.8, 1.2, 2.0],
}

# Optical flow used for the flow-based variants of M1/M2 and for the real walking ground truth
FLOW_PREFERRED = "RAFT-large (torchvision C_T_SKHT_V2); DIS (OpenCV) if the weights cannot be downloaded"

# Version of each stage's code. A stage's checkpoint is reused only when its version, its slice of
# the configuration and the fingerprints of the stages it reads are all unchanged (Section 0.5).
STAGE_VERSIONS = {"render": "v6.0", "tum": "v6.0", "flow": "v6.0", "vk": "v6.2", "eval": "v6.2", "kitti": "v6.3", "av2": "v6.3",
                  "av2x": "v6.4", "render65": "v6.5", "bonn": "v6.6", "av2t": "v6.5", "av2ft": "v6.6"}


def _jsonable(d):
    return {k: ({str(a): b for a, b in v.items()} if isinstance(v, dict) else v) for k, v in d.items()}


def write_run_config():
    with open(os.path.join(MANIFEST_DIR, "run_config.json"), "w") as _f:
        json.dump({"RUN_CONFIG": RUN_CONFIG, "PREREG": PREREG, "MOVER_DESIGN": _jsonable(MOVER_DESIGN),
                   "CAMERA_DESIGN": CAMERA_DESIGN, "ABLATION_DESIGN": _jsonable(ABLATION_DESIGN),
                   "RABL_DESIGN": _jsonable(RABL_DESIGN), "VK_DESIGN": VK_DESIGN, "KT_DESIGN": KT_DESIGN, "AV2_DESIGN": AV2_DESIGN,
                   "AV2X_DESIGN": AV2X_DESIGN, "SWEEP_DESIGN": SWEEP_DESIGN, "WALL_DOSE_DESIGN": WALL_DOSE_DESIGN,
                   "BONN_DESIGN": BONN_DESIGN, "TWIN_DESIGN": TWIN_DESIGN, "CUE_DESIGN": CUE_DESIGN,
                   "FLOWTWIN_DESIGN": FLOWTWIN_DESIGN, "STAGE_VERSIONS": STAGE_VERSIONS},
                  _f, indent=2, default=str)


for _name, _d in [("RUN_CONFIG", RUN_CONFIG), ("PREREG", PREREG), ("MOVER_DESIGN", MOVER_DESIGN),
                  ("CAMERA_DESIGN", CAMERA_DESIGN), ("ABLATION_DESIGN", ABLATION_DESIGN),
                  ("RABL_DESIGN", RABL_DESIGN), ("VK_DESIGN", VK_DESIGN), ("KT_DESIGN", KT_DESIGN), ("AV2_DESIGN", AV2_DESIGN),
                  ("AV2X_DESIGN", AV2X_DESIGN), ("SWEEP_DESIGN", SWEEP_DESIGN), ("WALL_DOSE_DESIGN", WALL_DOSE_DESIGN),
                  ("BONN_DESIGN", BONN_DESIGN), ("TWIN_DESIGN", TWIN_DESIGN), ("CUE_DESIGN", CUE_DESIGN),
                  ("FLOWTWIN_DESIGN", FLOWTWIN_DESIGN)]:
    print(_name)
    for _k, _v in _d.items():
        print(f"    {_k:<30}{_v}")
write_run_config()

In [ ]:
# Cell No: 010
# 0.5 Stage checkpoints. Every expensive stage (rendering, real-data preparation, optical flow,
#     each model on each testbed) is archived as checkpoints/<stage>.zip the moment it finishes and
#     appended at once to convoy_checkpoints.zip, so that archive always holds every stage finished
#     so far, even if the session stops later. The paper's figures, tables and results go into a
#     second archive, convoy_paper_outputs.zip (Section 13). A later session that attaches the archive
#     (or this run's output) restores each stage whose fingerprint matches, byte for byte.
import zipfile, zlib

CKPT_FORMAT = "convoy-checkpoint/1"
_STORED_EXT = (".png", ".jpg", ".jpeg", ".npz", ".zip", ".gz")      # already compressed: store as is
_STAGE_REG_DIR = os.path.join(MANIFEST_DIR, "stages")
os.makedirs(_STAGE_REG_DIR, exist_ok=True)
STAGE_LOG = []
SEARCH_ROOTS = []
for _r in ["/kaggle/input", os.environ.get("CONVOY_RESUME_FROM", ""), "/content/drive/MyDrive/convoy_checkpoints"]:
    if _r and os.path.exists(_r) and os.path.abspath(_r) not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(os.path.abspath(_r))


def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=str))


def stage_fingerprint(name, family, config=None, upstream=()):
    rec = dict(format=CKPT_FORMAT, stage=name, version=STAGE_VERSIONS[family], seed=GLOBAL_SEED,
               config=canonical(config), upstream=sorted(str(u) for u in upstream))
    return hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest()[:20]


def _arcname(path):
    p = os.path.abspath(path)
    for tag, root in (("S", SCRATCH_DIR), ("O", OUTPUT_DIR)):
        r = os.path.abspath(root)
        if p.startswith(r + os.sep):
            return tag + "/" + os.path.relpath(p, r).replace(os.sep, "/")
    raise ValueError(f"{path} is outside the scratch and output folders")


def _destination(arc):
    tag, rel = arc.split("/", 1)
    root = os.path.abspath({"S": SCRATCH_DIR, "O": OUTPUT_DIR}[tag])
    dst = os.path.normpath(os.path.join(root, rel))
    if not dst.startswith(root + os.sep):
        raise ValueError("unsafe path in a checkpoint: " + arc)
    return dst


def _safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


def _read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def _zip_meta(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return json.loads(zf.read("stage.json"))
    except (OSError, KeyError, ValueError, zipfile.BadZipFile):
        return None


# ---- the progressive archive convoy_checkpoints.zip -----------------------------------------------
_ARCHIVE = dict(names=None, stale=set(), enabled=True, note="")
_KAGGLE_OUTPUT_CAP = 19.5 * 2**30                    # Kaggle keeps at most 20 GB of /kaggle/working


def _dir_bytes(root):
    tot = 0
    for dp, _, fns in os.walk(root):
        for fn in fns:
            try:
                tot += os.path.getsize(os.path.join(dp, fn))
            except OSError:
                pass
    return tot


def _crc_of(path):
    c = 0
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            c = zlib.crc32(b, c)
    return c & 0xFFFFFFFF


def _archive_names():
    if _ARCHIVE["names"] is None:
        _ARCHIVE["names"] = {}
        if os.path.exists(ZIP_PATH):
            try:
                with zipfile.ZipFile(ZIP_PATH) as zf:
                    _ARCHIVE["names"] = {i.filename: (i.file_size, i.CRC) for i in zf.infolist()}
            except (zipfile.BadZipFile, OSError):
                os.remove(ZIP_PATH)                  # an interrupted append: start the archive again
    return _ARCHIVE["names"]


def archive_add(path, arcname, compress=False):
    # append one file to convoy_checkpoints.zip unless an identical copy is already in it
    if not _ARCHIVE["enabled"] or not os.path.exists(path):
        return
    names = _archive_names()
    size = os.path.getsize(path)
    if arcname in names:
        if names[arcname][0] != size or names[arcname][1] != _crc_of(path):
            _ARCHIVE["stale"].add(arcname)            # superseded in this folder: the archive is rebuilt at the end
        return
    if ENV_NAME == "kaggle" and _dir_bytes(ROOT_WORK) + size > _KAGGLE_OUTPUT_CAP:
        _ARCHIVE.update(enabled=False, note="stopped: the duplicate copy would exceed Kaggle's 20 GB output limit "
                                            "(checkpoints/ alone is the resume point)")
        print("[archive] " + _ARCHIVE["note"])
        return
    with zipfile.ZipFile(ZIP_PATH, "a", allowZip64=True) as zf:
        zf.write(path, arcname, compress_type=zipfile.ZIP_DEFLATED if compress else zipfile.ZIP_STORED)
        names[arcname] = (size, zf.getinfo(arcname).CRC)


def archive_stage(zip_path):
    archive_add(zip_path, "checkpoints/" + os.path.basename(zip_path))


def _extract(zf):
    for info in zf.infolist():
        if info.filename == "stage.json" or info.is_dir():
            continue
        dst = _destination(info.filename)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with zf.open(info) as src, open(dst + ".part", "wb") as out:
            shutil.copyfileobj(src, out, 1 << 22)
        os.replace(dst + ".part", dst)


class _PreviousRuns:
    # index of the stage archives found in attached data (lazy: built on first use)
    def __init__(self):
        self.entries = None

    def _add(self, meta, loc):
        if isinstance(meta, dict) and meta.get("format") == CKPT_FORMAT:
            self.entries.setdefault(meta["stage"], []).append((loc, meta))

    def _scan_zip(self, path):
        try:
            with zipfile.ZipFile(path) as zf:
                names = zf.namelist()
                if "stage.json" in names:
                    self._add(json.loads(zf.read("stage.json")), ("zip", path, None))
                    return
                for n in names:
                    if re.search(r"(^|/)checkpoints/[^/]+\.zip$", n):
                        with zf.open(n) as fh, zipfile.ZipFile(fh) as inner:
                            if "stage.json" in inner.namelist():
                                self._add(json.loads(inner.read("stage.json")), ("nested", path, n))
        except (OSError, ValueError, KeyError, zipfile.BadZipFile, RuntimeError):
            pass

    def build(self):
        self.entries = {}
        for root in SEARCH_ROOTS:
            for dirpath, dirnames, filenames in os.walk(root):
                if os.path.abspath(dirpath).startswith(os.path.abspath(CKPT_DIR)) or \
                   dirpath[len(root):].count(os.sep) >= 7:
                    dirnames[:] = []
                    continue
                if "stage.json" in filenames:
                    self._add(_read_json(os.path.join(dirpath, "stage.json")), ("dir", dirpath, None))
                    dirnames[:] = []                     # an extracted stage archive: nothing further below
                for fn in filenames:
                    if fn.lower().endswith(".zip"):
                        self._scan_zip(os.path.join(dirpath, fn))

    def find(self, name, fp):
        if self.entries is None:
            self.build()
        for loc, meta in self.entries.get(name, []):
            if meta.get("fingerprint") == fp:
                return loc, meta
        return None

    def summary(self):
        if self.entries is None:
            self.build()
        return {k: len(v) for k, v in sorted(self.entries.items())}


PREVIOUS = _PreviousRuns()


def _fmt_seconds(sec):
    sec = float(sec or 0)
    return f"{sec:.0f} s" if sec < 90 else f"{sec / 60:.1f} min" if sec < 5400 else f"{sec / 3600:.1f} h"


def _where(kind, path, member=None):
    # where a checkpoint came from, in words; only the archive or folder name is shown (an attached
    # dataset's mount path can contain the account name, which this notebook never prints)
    if kind == "checkpoints":
        return "this run's checkpoints/ folder"
    if kind == "session":
        return "this session"
    return f"the attached data ({os.path.basename(path.rstrip(os.sep))})"


_SESSION_SOURCE = {}                                  # stage -> where it came from in this session


def _original_seconds(meta):
    ex = meta.get("extra") or {}
    return ex.get("seconds") if isinstance(ex.get("seconds"), (int, float)) else meta.get("compute_seconds", 0.0)


class Stage:
    """One resumable unit of work: restore it if a matching checkpoint exists, else compute and commit.

    Every call of done() and commit() prints one line, so each cell states plainly whether its work
    was restored from a checkpoint (nothing is re-run) or is being computed in this session:
        [checkpoint] RESTORED  <stage>  from <where>: no need to re-run (...)
        [checkpoint] RUNNING   <stage>  no matching checkpoint ...: running it now
        [checkpoint] SAVED     <stage>  computed in ...; archived in checkpoints/ and convoy_checkpoints.zip
    """

    def __init__(self, name, family, config=None, upstream=()):
        self.name, self.family = name, family
        self.fp = stage_fingerprint(name, family, config, upstream)
        self.zip_path = os.path.join(CKPT_DIR, _safe(name) + ".zip")
        self.reg_path = os.path.join(_STAGE_REG_DIR, _safe(name) + ".json")
        self.t0 = time.time()

    def _present(self, meta):
        try:
            return all(os.path.exists(_destination(a)) and os.path.getsize(_destination(a)) == s for a, s in meta["files"])
        except (KeyError, ValueError):
            return False

    def _register(self, meta):
        with open(self.reg_path, "w") as f:
            json.dump(meta, f, indent=1)

    def _say(self, word, text):
        print(f"[checkpoint] {word:<9} {self.name:<22} {text}", flush=True)

    def _log(self, status, source, meta):
        mib = round(sum(s for _, s in meta.get("files", [])) / 2**20, 1)
        took = float(_original_seconds(meta) or 0.0) if status == "computed" else time.time() - self.t0
        STAGE_LOG.append(dict(stage=self.name, status=status, source=source, fingerprint=self.fp,
                              files=len(meta.get("files", [])), MiB=mib, seconds=round(took, 1),
                              original_seconds=round(float(_original_seconds(meta) or 0.0), 1)))
        prev = _SESSION_SOURCE.get(self.name)
        if status == "kept":
            if prev and prev != "computed":
                self._say("RESTORED", f"restored earlier in this session from {prev}: no need to re-run")
            else:
                self._say("COMPLETE", ("computed earlier in this session" if prev else "already complete in this folder")
                                      + ": no need to re-run")
            return
        _SESSION_SOURCE[self.name] = source if status == "restored" else "computed"
        if status == "restored":
            self._say("RESTORED", f"from {source}: no need to re-run ({len(meta.get('files', []))} files, {mib:,.0f} MiB, "
                                  f"{_fmt_seconds(time.time() - self.t0)} to restore; computing it took "
                                  f"{_fmt_seconds(_original_seconds(meta))})")
        else:
            self._say("SAVED", f"computed in {_fmt_seconds(took)}; archived in checkpoints/{os.path.basename(self.zip_path)}"
                               + (" and convoy_checkpoints.zip" if _ARCHIVE["enabled"] else ""))

    def _not_found(self):
        if PREVIOUS.entries is None:
            PREVIOUS.build()
        others = [m_ for _, m_ in PREVIOUS.entries.get(self.name, []) if m_.get("fingerprint") != self.fp]
        old = _zip_meta(self.zip_path)
        if old and old.get("fingerprint") != self.fp:
            others.append(old)
        why = (f"; {len(others)} archived version(s) of this stage were found but were made with other settings or "
               f"code, so they are not used" if others else "")
        self._say("RUNNING", f"no matching checkpoint in checkpoints/ or the attached data{why}: running it now")

    def _write_zip(self, meta):
        tmp = self.zip_path + ".part"
        with zipfile.ZipFile(tmp, "w", allowZip64=True) as zf:
            zf.writestr("stage.json", json.dumps(meta, indent=1))
            for arc, _ in meta["files"]:
                ct = zipfile.ZIP_STORED if arc.lower().endswith(_STORED_EXT) else zipfile.ZIP_DEFLATED
                zf.write(_destination(arc), arc, compress_type=ct)
        os.replace(tmp, self.zip_path)
        archive_stage(self.zip_path)

    def available(self):
        # where this stage could be taken from, without extracting anything (None: must be computed)
        meta = _read_json(self.reg_path)
        if meta and meta.get("fingerprint") == self.fp and self._present(meta):
            return "this session"
        meta = _zip_meta(self.zip_path)
        if meta and meta.get("fingerprint") == self.fp:
            return _where("checkpoints", self.zip_path)
        hit = PREVIOUS.find(self.name, self.fp)
        return _where(*hit[0]) if hit else None

    def done(self):
        # True if the stage's outputs are in place (kept or restored); False if it must be computed.
        # Order: this session -> this run's checkpoints/ -> any attached data (Kaggle inputs, CONVOY_RESUME_FROM)
        self.t0 = time.time()
        meta = _read_json(self.reg_path)
        if meta and meta.get("fingerprint") == self.fp and self._present(meta):
            if _zip_meta(self.zip_path) is None:
                self._write_zip(meta)
            else:
                archive_stage(self.zip_path)
            self._log("kept", _where("session", ""), meta)
            return True
        meta = _zip_meta(self.zip_path)
        if meta and meta.get("fingerprint") == self.fp:
            with zipfile.ZipFile(self.zip_path) as zf:
                _extract(zf)
            if self._present(meta):
                self._register(meta); self._log("restored", _where("checkpoints", self.zip_path), meta)
                archive_stage(self.zip_path)
                return True
        hit = PREVIOUS.find(self.name, self.fp)
        if hit:
            (kind, path, member), meta = hit
            if kind == "zip":
                with zipfile.ZipFile(path) as zf:
                    _extract(zf)
                if os.path.abspath(path) != os.path.abspath(self.zip_path):
                    shutil.copyfile(path, self.zip_path + ".part"); os.replace(self.zip_path + ".part", self.zip_path)
            elif kind == "nested":
                with zipfile.ZipFile(path) as outer:
                    with outer.open(member) as fh, zipfile.ZipFile(fh) as zf:
                        _extract(zf)
                    with outer.open(member) as fh, open(self.zip_path + ".part", "wb") as out:
                        shutil.copyfileobj(fh, out, 1 << 22)
                os.replace(self.zip_path + ".part", self.zip_path)
            else:
                for arc, _ in meta["files"]:
                    dst = _destination(arc)
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copyfile(os.path.join(path, *arc.split("/")), dst)
                self._write_zip(meta)
            if self._present(meta):
                self._register(meta); self._log("restored", _where(kind, path, member), meta)
                archive_stage(self.zip_path)
                return True
        self._not_found()
        return False

    def commit(self, paths, extra=None):
        files = []
        for p in paths if isinstance(paths, (list, tuple)) else [paths]:
            if os.path.isdir(p):
                for dp, _, fns in os.walk(p):
                    files += [os.path.join(dp, fn) for fn in sorted(fns) if not fn.endswith(".part")]
            elif os.path.exists(p):
                files.append(p)
        files = sorted(set(files))
        meta = dict(format=CKPT_FORMAT, stage=self.name, family=self.family, fingerprint=self.fp,
                    created_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    files=[[_arcname(f), os.path.getsize(f)] for f in files], extra=extra or {},
                    compute_seconds=round(time.time() - self.t0, 1))
        self._write_zip(meta)
        self._register(meta)
        self._log("computed", "this session", meta)
        return meta


def stage_table(prefix=None):
    df = pd.DataFrame(STAGE_LOG)
    if prefix and len(df):
        df = df[df["stage"].str.startswith(prefix)]
    return df


# ---- the second archive, download links and an optional off-session copy (v6.2) --------------------
_PAPER_SUBDIRS = ("figures", "tables", "results", "manifest", "logs")
PAPER_README = (
    "convoy_paper_outputs.zip: everything needed to check the paper's numbers, figures and tables.\n"
    "  figures/   every figure as vector PDF and 300-dpi PNG, with the plotted data (*_data.json)\n"
    "  tables/    formatted tables as CSV, JSON and booktabs LaTeX (T*.csv / .json / .tex)\n"
    "  results/   unformatted results of every analysis (estimates, intervals, bins, claim audit)\n"
    "  manifest/  idea lock, amendments, decisions, run configuration, model and data status, stage log,\n"
    "             figure captions (figures.json), table captions (tables.json), RAM log, file checksums\n"
    "  logs/      environment, installation, testbed self-tests, ground-truth validation, model status\n"
    "The resume point is the other archive, convoy_checkpoints.zip (attach it as a Kaggle dataset).\n")


def build_paper_zip(note=""):
    tmp, n = PAPER_ZIP + ".part", 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for sub in _PAPER_SUBDIRS:
            for dp, _, fns in os.walk(os.path.join(OUTPUT_DIR, sub)):
                for fn in sorted(fns):
                    p = os.path.join(dp, fn)
                    zf.write(p, "paper_outputs/" + os.path.relpath(p, OUTPUT_DIR).replace(os.sep, "/"),
                             compress_type=zipfile.ZIP_STORED if fn.lower().endswith(_STORED_EXT) else zipfile.ZIP_DEFLATED)
                    n += 1
        zf.writestr("paper_outputs/README.txt", PAPER_README + note)
    os.replace(tmp, PAPER_ZIP)
    return n


def offer_downloads(paths, auto=False):
    # Download links in the cell output. A browser can only receive a file while it is connected to
    # the session; auto=True also asks it to start the downloads (best effort). For a run that must
    # survive a closed browser, use Save Version -> Save & Run All (the files then stay in the
    # version's Output tab) and/or the dataset copy below.
    items = [p for p in paths if p and os.path.exists(p)]
    if not items:
        return
    try:
        from IPython.display import display as _disp, HTML as _HTML, Javascript as _JS
    except ImportError:
        return
    rows = []
    for p in items:
        rel = os.path.relpath(p, os.getcwd())
        href = rel if not rel.startswith("..") else p
        rows.append(f'<li><a href="{href}" download="{os.path.basename(p)}" target="_blank">{os.path.basename(p)}</a> '
                    f'({os.path.getsize(p) / 2**20:,.0f} MiB)</li>')
    _disp(_HTML("<p>Download (right-click and save if the browser does not start it):</p><ul>" + "".join(rows) + "</ul>"))
    if auto:
        names = [os.path.basename(p) for p in items]
        _disp(_JS("setTimeout(function(){var n=%s;document.querySelectorAll('a').forEach(function(a){"
                  "if(n.indexOf(a.textContent.trim())>=0){try{a.click();}catch(e){}}});},1500);" % json.dumps(names)))


def _kaggle_api_ready():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
        return bool(os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"])
    except Exception:
        return False


def backup_to_dataset(paths, message):
    # Copies the archives into a PRIVATE Kaggle dataset <user>/<dataset_backup_slug> with the Kaggle API,
    # so they survive a closed browser or a stopped session and can be attached directly next time.
    # Active only on Kaggle with the secrets KAGGLE_USERNAME and KAGGLE_KEY attached (Add-ons -> Secrets).
    if not RUN_CONFIG.get("dataset_backup") or ENV_NAME != "kaggle" or not _kaggle_api_ready():
        return "not configured (attach the secrets KAGGLE_USERNAME and KAGGLE_KEY to enable it)"
    import subprocess
    user, slug = os.environ["KAGGLE_USERNAME"], RUN_CONFIG["dataset_backup_slug"]
    d = os.path.join(SCRATCH_DIR, "dataset_backup")
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    for p in paths:
        if os.path.exists(p):
            try:
                os.link(p, os.path.join(d, os.path.basename(p)))
            except OSError:
                shutil.copy2(p, d)
    with open(os.path.join(d, "dataset-metadata.json"), "w") as f:
        json.dump(dict(title="Convoy ambiguity checkpoints", id=f"{user}/{slug}", licenses=[{"name": "CC0-1.0"}]), f)
    cli = shutil.which("kaggle")
    if cli is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], capture_output=True, timeout=600)
        cli = shutil.which("kaggle") or "kaggle"
    note = ""
    for cmd in ([cli, "datasets", "version", "-p", d, "-m", message[:200], "-q"], [cli, "datasets", "create", "-p", d, "-q"]):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
        except Exception as e:                        # never stop the run for a backup
            note = f"{type(e).__name__}: {e}"
            continue
        if r.returncode == 0:
            shutil.rmtree(d, ignore_errors=True)
            return f"copied to your private Kaggle dataset '{slug}' ({cmd[2]})"
        note = (r.stderr or r.stdout).strip()[-300:]
    shutil.rmtree(d, ignore_errors=True)
    return "dataset copy failed: " + note.replace(user, "<account>")         # the account name is never printed


print(f"Checkpoint folder: {CKPT_DIR}")
print("=" * 100)
if SEARCH_ROOTS:
    _found = PREVIOUS.summary()
    _labels = ["/kaggle/input" if r == "/kaggle/input" else "CONVOY_RESUME_FROM" if r == os.path.abspath(os.environ.get("CONVOY_RESUME_FROM", "-"))
               else "Google Drive" for r in SEARCH_ROOTS]
    print(f"RESUME CHECK: searched {', '.join(_labels)} for checkpoints of a previous run.")
    if _found:
        def _src_label(kind, path):
            # nested: the outer archive; loose or extracted stage archives: the folder that holds them
            path = path.rstrip(os.sep)
            if kind != "nested" and os.path.basename(os.path.dirname(path)) == "checkpoints":
                return os.path.basename(os.path.dirname(os.path.dirname(path))) + "/checkpoints/"
            return os.path.basename(path)
        _src = {}
        for _lst in PREVIOUS.entries.values():
            for (_kind, _path, _m), _ in _lst:
                _lab = _src_label(_kind, _path)
                _src[_lab] = _src.get(_lab, 0) + 1
        print(f"  found {sum(_found.values())} stage checkpoint(s) for {len(_found)} stage(s) in: "
              + "; ".join(f"{k} ({v})" for k, v in sorted(_src.items())))
        print("  Each stage is restored when its cell is reached, if its fingerprint (settings, code version and inputs)\n"
              "  matches this run; the cell then prints 'RESTORED ... no need to re-run'. A stage without a match prints\n"
              "  'RUNNING ... running it now' and is computed and saved.")
        print(textwrap.fill(", ".join(_found), width=110, initial_indent="  stages found: ", subsequent_indent="    "))
    else:
        print("  no stage checkpoints in the attached data: every stage runs in this session (and is saved as it finishes).")
else:
    print("RESUME CHECK: nothing attached; every stage runs in this session (and is saved as it finishes).\n"
          "  To resume later, attach convoy_checkpoints.zip from this run's output as a Kaggle dataset (Section 0.5).")
print("=" * 100)

In [ ]:
# Cell No: 011
# 2.1 Geometry core (vectorized, float64).
from scipy.optimize import brentq


class PinholeCamera:
    def __init__(self, fx, fy, cx, cy, W, H):
        self.fx, self.fy, self.cx, self.cy = float(fx), float(fy), float(cx), float(cy)
        self.W, self.H = int(W), int(H)

    @property
    def K(self):
        return np.array([[self.fx, 0.0, self.cx], [0.0, self.fy, self.cy], [0.0, 0.0, 1.0]])

    def rays(self, u, v):
        # camera-frame ray directions scaled so that their z-component is 1
        u = np.asarray(u, np.float64); v = np.asarray(v, np.float64)
        return np.stack([(u - self.cx) / self.fx, (v - self.cy) / self.fy, np.ones_like(u)], -1)

    def project(self, Xc):
        Xc = np.asarray(Xc, np.float64)
        z = Xc[..., 2]
        return np.stack([self.cx + self.fx * Xc[..., 0] / z, self.cy + self.fy * Xc[..., 1] / z], -1)

    def scaled(self, sx, sy):
        # intrinsics of the same camera after resizing the image by (sx, sy)
        return PinholeCamera(self.fx * sx, self.fy * sy, (self.cx + 0.5) * sx - 0.5,
                             (self.cy + 0.5) * sy - 0.5, round(self.W * sx), round(self.H * sy))

    def to_dict(self):
        return dict(fx=self.fx, fy=self.fy, cx=self.cx, cy=self.cy, W=self.W, H=self.H)

    @staticmethod
    def from_dict(d):
        return PinholeCamera(d["fx"], d["fy"], d["cx"], d["cy"], d["W"], d["H"])


def rot_x(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]], np.float64)

def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], np.float64)

def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float64)

def rot_ypr(yaw, pitch=0.0, roll=0.0):
    # camera-to-world rotation: yaw about world y (down axis), then pitch about the camera
    # x axis, then roll about the optical axis
    return rot_y(yaw) @ rot_x(pitch) @ rot_z(roll)

def random_rotation(rng):
    q = rng.standard_normal(4); q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])

def world_to_cam(Xw, R, C):
    return (np.asarray(Xw, np.float64) - C) @ R          # = R^T (Xw - C), row-vector form

def cam_to_world(Xc, R, C):
    return np.asarray(Xc, np.float64) @ R.T + C

def relative_pose(R1, C1, R2, C2):
    # X2 = R21 X1 + t21  (camera-1 coordinates -> camera-2 coordinates)
    R21 = R2.T @ R1
    t21 = R2.T @ (np.asarray(C1, np.float64) - np.asarray(C2, np.float64))
    return R21, t21

def skew(t):
    return np.array([[0, -t[2], t[1]], [t[2], 0, -t[0]], [-t[1], t[0], 0]], np.float64)

def fundamental_matrix(K1, K2, R21, t21):
    return np.linalg.inv(K2).T @ skew(t21) @ R21 @ np.linalg.inv(K1)

def _hom(p):
    p = np.asarray(p, np.float64)
    return np.concatenate([p, np.ones(p.shape[:-1] + (1,))], -1)

def epipolar_offset(p1, p2, F):
    # signed distance (px) of p2 from the epipolar line F p1; vectorized over leading dims
    l = _hom(p1) @ F.T
    den = np.sqrt(l[..., 0] ** 2 + l[..., 1] ** 2)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.sum(l * _hom(p2), -1) / den

def triangulate_depth(p1, p2, K1, K2, R21, t21):
    # Closest-point triangulation; returns (Z in camera 1, ray gap in metres).
    # Z < 0 means the rays meet behind camera 1 (no valid static explanation).
    d1 = _hom(p1) @ np.linalg.inv(K1).T                      # z = 1 in camera 1
    d2 = (_hom(p2) @ np.linalg.inv(K2).T) @ R21              # camera-2 rays in camera-1 frame
    o2 = -R21.T @ t21                                        # camera-2 centre in camera-1 frame
    w0 = -o2
    a = np.sum(d1 * d1, -1); b = np.sum(d1 * d2, -1); c = np.sum(d2 * d2, -1)
    d = np.sum(d1 * w0, -1); e = np.sum(d2 * w0, -1)
    den = a * c - b * b
    with np.errstate(invalid="ignore", divide="ignore"):
        t1 = (b * e - c * d) / den
        t2 = (a * e - b * d) / den
    gap = np.linalg.norm(t1[..., None] * d1 - (o2 + t2[..., None] * d2), axis=-1)
    return t1, gap

def infinite_homography_map(p1, K1, K2, R21):
    q = _hom(p1) @ (K2 @ R21 @ np.linalg.inv(K1)).T
    return q[..., :2] / q[..., 2:3]

def translational_parallax(p1, p2, K1, K2, R21):
    # image displacement after removing the rotation-induced part (pixels)
    return np.linalg.norm(np.asarray(p2, np.float64) - infinite_homography_map(p1, K1, K2, R21), axis=-1)

def triangulate_multiview(pixels, cams, Rs, Cs):
    # least-squares intersection of K rays (world frame); returns (point, rms reprojection px)
    A = np.zeros((3, 3)); bb = np.zeros(3)
    for p, cam, R, C in zip(pixels, cams, Rs, Cs):
        d = R @ cam.rays(p[0], p[1]); d = d / np.linalg.norm(d)
        P = np.eye(3) - np.outer(d, d)
        A += P; bb += P @ C
    X = np.linalg.solve(A, bb)
    errs = [np.linalg.norm(cam.project(world_to_cam(X, R, C)) - np.asarray(p))
            for p, cam, R, C in zip(pixels, cams, Rs, Cs)]
    return X, float(np.sqrt(np.mean(np.square(errs))))

def perp_unit(b, prefer=None, rng=None):
    # unit vector perpendicular to b: 'horizontal' (in the ground plane) or random
    b = np.asarray(b, np.float64); bh = b / np.linalg.norm(b)
    if prefer == "horizontal":
        n = np.cross(bh, np.array([0.0, 1.0, 0.0]))
        if np.linalg.norm(n) < 1e-8:
            n = np.cross(bh, np.array([1.0, 0.0, 0.0]))
    else:
        r = (rng or np.random.default_rng(0)).standard_normal(3)
        n = r - (r @ bh) * bh
    return n / np.linalg.norm(n)

def solve_perp_for_eps(X, delta_par, n_hat, target_eps, cam0, R0, C0, cam1, R1, C1, max_m=50.0):
    # Find the magnitude m of the perpendicular motion m*n_hat such that the moved point's
    # camera-1 image lies EXACTLY |target_eps| px from the epipolar line of its camera-0 image.
    # Both signs are tried; the smaller |m| is returned. Returns (None, nan) if impossible.
    R21, t21 = relative_pose(R0, C0, R1, C1)
    F = fundamental_matrix(cam0.K, cam1.K, R21, t21)
    p0 = cam0.project(world_to_cam(X, R0, C0))
    def eps_of(m):
        Xm = X + delta_par + m * n_hat
        Xc = world_to_cam(Xm, R1, C1)
        if Xc[2] <= 0.05:
            return np.nan
        return float(epipolar_offset(p0, cam1.project(Xc), F))
    if target_eps == 0.0:
        return 0.0, abs(eps_of(0.0))
    best = None
    for sgn in (+1.0, -1.0):
        g = lambda m: eps_of(m) - sgn * target_eps
        lo, hi = 0.0, 0.05
        g_lo = g(lo)
        if not np.isfinite(g_lo):
            continue
        found = False
        while abs(hi) <= max_m:
            for h in (hi, -hi):
                gh = g(h)
                if np.isfinite(gh) and g_lo * gh <= 0:
                    try:
                        m = brentq(g, min(0.0, h), max(0.0, h), xtol=1e-12, rtol=1e-12, maxiter=200)
                    except ValueError:
                        continue
                    if best is None or abs(m) < abs(best):
                        best = m
                    found = True
                    break
            if found:
                break
            hi *= 1.6
    if best is None:
        return None, np.nan
    return float(best), abs(eps_of(best))

print("Geometry core ready: PinholeCamera, rotations, relative_pose, fundamental_matrix, "
      "epipolar_offset, triangulate_depth, translational_parallax, triangulate_multiview, "
      "perp_unit, solve_perp_for_eps.")

In [ ]:
# Cell No: 012
# 2.2 Exact verification of Propositions 1-4 with arbitrary rotations and baselines.
_rng = np.random.default_rng(seed_for("prop-checks"))
_cam = PinholeCamera(700.0, 690.0, 319.5, 239.5, 640, 480)
PROP_CHECKS = {}

# ---- Proposition 1: moved point (delta = s b) == static point X/(1-s) in both views ----
_err_px, _err_rel_depth, _behind = [], [], []
for _ in range(20000):
    R1, R2 = random_rotation(_rng), random_rotation(_rng)
    C1 = _rng.uniform(-5, 5, 3)
    b = _rng.standard_normal(3) * _rng.uniform(0.05, 5.0)
    C2 = C1 + b
    Xc1 = np.array([_rng.uniform(-8, 8), _rng.uniform(-6, 6), _rng.uniform(2, 80)])
    X = cam_to_world(Xc1, R1, C1)
    s = _rng.uniform(-2.0, 0.95)
    Xs = C1 + (X - C1) / (1.0 - s)                         # static equivalent
    q_moved = world_to_cam(X + s * b, R2, C2)
    q_static = world_to_cam(Xs, R2, C2)
    if q_moved[2] <= 0.1 or q_static[2] <= 0.1:
        continue
    p1 = _cam.project(Xc1)
    p2 = _cam.project(q_moved)
    _err_px.append(max(np.abs(_cam.project(world_to_cam(Xs, R1, C1)) - p1).max(),
                       np.abs(_cam.project(q_static) - p2).max()))
    R21, t21 = relative_pose(R1, C1, R2, C2)
    Z_tri, _gap = triangulate_depth(p1, p2, _cam.K, _cam.K, R21, t21)
    _err_rel_depth.append(abs(Z_tri / (Xc1[2] / (1 - s)) - 1.0))
# s > 1: the rays meet behind camera 1
for _ in range(2000):
    R1, R2 = random_rotation(_rng), random_rotation(_rng)
    b = _rng.standard_normal(3); C1 = np.zeros(3); C2 = b
    Xc1 = np.array([_rng.uniform(-4, 4), _rng.uniform(-3, 3), _rng.uniform(4, 40)])
    X = cam_to_world(Xc1, R1, C1); s = _rng.uniform(1.05, 2.0)
    q = world_to_cam(X + s * b, R2, C2)
    if q[2] <= 0.1:
        continue
    R21, t21 = relative_pose(R1, C1, R2, C2)
    Z_tri, _ = triangulate_depth(_cam.project(Xc1), _cam.project(q), _cam.K, _cam.K, R21, t21)
    _behind.append(Z_tri < 0)
PROP_CHECKS["P1_max_reprojection_diff_px"] = float(np.max(_err_px))
PROP_CHECKS["P1_max_rel_error_triangulated_vs_Z_over_1_minus_s"] = float(np.max(_err_rel_depth))
PROP_CHECKS["P1_n_trials"] = len(_err_px)
PROP_CHECKS["P1_fraction_s_gt_1_triangulating_behind_camera"] = float(np.mean(_behind))

# ---- Proposition 2: K views, straight path (any rotations) vs curved path vs varying ratio ----
def _multi_view_residual(K, path, rng):
    s = rng.uniform(-0.5, 0.8)
    b0 = rng.standard_normal(3); b0 *= rng.uniform(0.5, 2.0) / np.linalg.norm(b0)
    X = np.array([rng.uniform(-6, 6), rng.uniform(-3, 3), rng.uniform(8, 40)])
    Cs, Rs, pix = [], [], []
    C = np.zeros(3)
    for k in range(K):
        if path == "straight":
            Ck = k * b0
            Xk = X + k * s * b0
        elif path in ("curved", "curved_convoy"):              # translation direction turns
            Ck = C.copy()
            ang = np.deg2rad(6.0)
            b_k = rot_y(ang * k) @ b0
            C = C + b_k
            # curved: the object keeps moving on a straight line; curved_convoy: the object
            # copies the camera's displacement, X_k = X_0 + s (C_k - C_0)
            Xk = X + k * s * b0 if path == "curved" else X + s * Ck
        else:                                                   # straight path, ratio changes
            Ck = k * b0
            Xk = X + k * (s + 0.25 * k) * b0
        # arbitrary per-frame heading/pitch/roll changes (up to ~17 deg yaw)
        Rk = rot_ypr(rng.uniform(-0.3, 0.3), rng.uniform(-0.1, 0.1), rng.uniform(-0.1, 0.1)) if k > 0 else np.eye(3)
        q = world_to_cam(Xk, Rk, Ck)
        if q[2] <= 0.5:
            return np.nan
        Cs.append(Ck); Rs.append(Rk); pix.append(_cam.project(q))
    _, rms = triangulate_multiview(pix, [_cam] * K, Rs, Cs)
    return rms

for _path in ["straight", "curved", "varying_ratio", "curved_convoy"]:
    for _K in [2, 3, 4, 8]:
        _vals = [_multi_view_residual(_K, _path, _rng) for _ in range(400)]
        _vals = np.array([v for v in _vals if np.isfinite(v)])
        PROP_CHECKS[f"P2_{_path}_K{_K}_median_static_fit_rms_px"] = float(np.median(_vals))

# ---- Proposition 3: which perpendicular motions leave the epipolar line ----
# The epipolar plane of X contains C1, C2 and X. Motion with a component along the plane
# normal n = (X - C1) x b moves the observation off the epipolar line; perpendicular motion
# INSIDE the plane (u = in-plane direction orthogonal to b) does not. The second case is a
# correction to the locked wording of Proposition 3 (see Section 0.2, erratum E-1).
_eps_vals = []
for _ in range(3000):
    R1 = np.eye(3); C1 = np.zeros(3)
    b = _rng.standard_normal(3); b *= _rng.uniform(0.5, 3.0) / np.linalg.norm(b)
    R2 = rot_ypr(_rng.uniform(-0.2, 0.2), _rng.uniform(-0.05, 0.05)); C2 = C1 + b
    X = np.array([_rng.uniform(-6, 6), _rng.uniform(-2, 2), _rng.uniform(8, 40)])
    n_plane = np.cross(X - C1, b); n_plane /= np.linalg.norm(n_plane)
    u_plane = np.cross(b, n_plane); u_plane /= np.linalg.norm(u_plane)       # in-plane, orthogonal to b
    m = _rng.uniform(0.05, 1.0); s = _rng.uniform(-0.5, 0.8)
    R21, t21 = relative_pose(R1, C1, R2, C2)
    F = fundamental_matrix(_cam.K, _cam.K, R21, t21)
    p1 = _cam.project(world_to_cam(X, R1, C1))
    q_par = world_to_cam(X + s * b, R2, C2)
    q_in = world_to_cam(X + s * b + m * u_plane, R2, C2)
    q_out = world_to_cam(X + s * b + m * n_plane, R2, C2)
    if min(q_par[2], q_in[2], q_out[2]) <= 0.5:
        continue
    e_par = abs(float(epipolar_offset(p1, _cam.project(q_par), F)))
    e_in = abs(float(epipolar_offset(p1, _cam.project(q_in), F)))
    e_out = abs(float(epipolar_offset(p1, _cam.project(q_out), F)))
    # the in-plane mover is explained exactly by the static point at the ray intersection
    Z_tri, gap = triangulate_depth(p1, _cam.project(q_in), _cam.K, _cam.K, R21, t21)
    _eps_vals.append((e_par, e_in, e_out, float(gap)))
_eps_vals = np.array(_eps_vals)
PROP_CHECKS["P3_max_eps_parallel_px"] = float(_eps_vals[:, 0].max())
PROP_CHECKS["P3_max_eps_inplane_perpendicular_px"] = float(_eps_vals[:, 1].max())
PROP_CHECKS["P3_max_ray_gap_inplane_m"] = float(_eps_vals[:, 3].max())
PROP_CHECKS["P3_min_eps_out_of_plane_px"] = float(_eps_vals[:, 2].min())
PROP_CHECKS["P3_median_eps_out_of_plane_px"] = float(np.median(_eps_vals[:, 2]))

# ---- Proposition 4 (v6.5): the static twin of a grounded parallel mover ----
# A road plane with a random upward normal, camera 1 at height H above the mover's contact point, a baseline in the road
# plane, arbitrary rotations: the twin's contact (i) lies H (1 - e^x) above the road, (ii) scales every length by e^x and
# (iii) for 0 < s < 1 is reached from camera 1 only through the contact point itself, i.e. it lies below the road.
_rng4 = np.random.default_rng(seed_for("prop-checks-4"))
_h_err, _len_err, _behind_road, _same_px = [], [], [], []
for _ in range(20000):
    n = _rng4.standard_normal(3); n /= np.linalg.norm(n)
    Hc = _rng4.uniform(0.8, 3.0)
    C1 = _rng4.uniform(-5, 5, 3)
    t1 = np.cross(n, _rng4.standard_normal(3)); t1 /= np.linalg.norm(t1)       # direction inside the road plane
    t2 = np.cross(n, t1)
    Xc = C1 - Hc * n + _rng4.uniform(5, 40) * t1 + _rng4.uniform(-6, 6) * t2      # contact point on the road
    X2 = Xc + _rng4.uniform(0.2, 2.0) * n + _rng4.uniform(-1, 1) * t1             # a second point of the object
    a_ = _rng4.uniform(0, 2 * np.pi)
    b = _rng4.uniform(0.3, 3.0) * (np.cos(a_) * t1 + np.sin(a_) * t2)
    s = _rng4.uniform(-2.0, 0.95)
    xs = -np.log(1.0 - s)
    tw_c, tw_2 = C1 + (Xc - C1) / (1 - s), C1 + (X2 - C1) / (1 - s)
    _h_err.append(abs((tw_c - Xc) @ n - Hc * (1.0 - np.exp(xs))))
    _len_err.append(abs(np.linalg.norm(tw_2 - tw_c) / np.linalg.norm(X2 - Xc) - np.exp(xs)))
    R1, R2 = random_rotation(_rng4), random_rotation(_rng4)
    q1, q2 = world_to_cam(Xc + s * b, R2, C1 + b), world_to_cam(tw_c, R2, C1 + b)
    if q1[2] > 0.1 and q2[2] > 0.1 and world_to_cam(Xc, R1, C1)[2] > 0.1:
        _same_px.append(float(np.abs(_cam.project(q1) - _cam.project(q2)).max()))
    if 0 < s < 1:
        _behind_road.append(bool(((tw_c - C1) @ n) < ((Xc - C1) @ n) - 1e-12))     # below the road plane
PROP_CHECKS["P4_max_twin_height_error_m"] = float(np.max(_h_err))
PROP_CHECKS["P4_max_twin_scale_error"] = float(np.max(_len_err))
PROP_CHECKS["P4_max_reprojection_diff_px"] = float(np.max(_same_px)) if _same_px else float("nan")
PROP_CHECKS["P4_fraction_twin_below_road_for_0_lt_s_lt_1"] = float(np.mean(_behind_road)) if _behind_road else float("nan")
PROP_CHECKS["P4_n_trials"] = len(_h_err)

for _k, _v in PROP_CHECKS.items():
    print(f"{_k:<60}{_v:.3e}" if isinstance(_v, float) else f"{_k:<60}{_v}")

assert PROP_CHECKS["P1_max_reprojection_diff_px"] < 1e-6
assert PROP_CHECKS["P1_max_rel_error_triangulated_vs_Z_over_1_minus_s"] < 1e-6
assert PROP_CHECKS["P1_fraction_s_gt_1_triangulating_behind_camera"] == 1.0
for _K in [2, 3, 4, 8]:
    assert PROP_CHECKS[f"P2_straight_K{_K}_median_static_fit_rms_px"] < 1e-6
    assert PROP_CHECKS[f"P2_curved_convoy_K{_K}_median_static_fit_rms_px"] < 1e-6
for _K in [3, 4, 8]:
    assert PROP_CHECKS[f"P2_curved_K{_K}_median_static_fit_rms_px"] > 0.1
    assert PROP_CHECKS[f"P2_varying_ratio_K{_K}_median_static_fit_rms_px"] > 0.1
assert PROP_CHECKS["P3_max_eps_parallel_px"] < 1e-6
assert PROP_CHECKS["P3_max_eps_inplane_perpendicular_px"] < 1e-6
assert PROP_CHECKS["P3_max_ray_gap_inplane_m"] < 1e-5          # numerical zero at 8-40 m depth
assert PROP_CHECKS["P3_min_eps_out_of_plane_px"] > 0.0
assert PROP_CHECKS["P4_max_twin_height_error_m"] < 1e-9
assert PROP_CHECKS["P4_max_twin_scale_error"] < 1e-9
assert PROP_CHECKS["P4_max_reprojection_diff_px"] < 1e-6
assert PROP_CHECKS["P4_fraction_twin_below_road_for_0_lt_s_lt_1"] == 1.0
print("\nAll proposition checks passed: P1 exact for arbitrary rotations; P2 exact on straight "
      "paths for any K and any rotations, broken by curved paths and by a changing ratio "
      "(K>=3), and exact again on curved paths when the object copies the camera's "
      "displacement (convoy); P3 motion inside the epipolar plane (parallel OR perpendicular "
      "to b) has eps = 0 and is explained by a static point; only the out-of-plane component "
      "is visible; P4 the static twin of a grounded parallel mover lies H (1 - e^x) above the road (below it "
      "for 0 < s < 1) and is e^x times larger.")
with open(os.path.join(LOG_DIR, "proposition_checks.json"), "w") as _f:
    json.dump(PROP_CHECKS, _f, indent=2)

In [ ]:
# Cell No: 013
# 2.3 Assumption A2: locked first-order formula eps ~= f||delta_perp||/Z versus the exact
#     epipolar offset, plus the corrected first-order formula that uses only the component
#     of the motion normal to the epipolar plane (Proposition 3).
# 2.4 Slope-weight identity: OLS slope beta(w) of rho = log(w + (1-w) e^x) over the design range.
_rng = np.random.default_rng(seed_for("a2"))
_rows = []
for _ in range(6000):
    _kind = _rng.choice(["forward", "oblique", "side"], p=[0.4, 0.3, 0.3])
    _cd = CAMERA_DESIGN[_kind]
    _yaw = np.deg2rad(_rng.uniform(*_cd["yaw_deg"])) * _rng.choice([-1, 1])
    _f = _rng.uniform(*CAMERA_DESIGN["focal_px"])
    _cam = PinholeCamera(_f, _f, 319.5, 239.5, 640, 480)
    R0 = rot_ypr(_yaw); C0 = np.zeros(3)
    b = np.array([0.0, 0.0, 1.0]) * _rng.uniform(*_cd["baseline_m"])
    R1 = R0 @ rot_ypr(np.deg2rad(_rng.uniform(-1, 1))); C1 = C0 + b
    # mover centre: in front of camera 0 at the design depth, below the camera
    Zc = _rng.uniform(*MOVER_DESIGN["depth_range_m"])
    Xc = np.array([_rng.uniform(-0.45, 0.45) * Zc * 640 / _f, _rng.uniform(0.0, 0.3) * Zc * 480 / _f, Zc])
    X = cam_to_world(Xc, R0, C0)
    s = _rng.uniform(*MOVER_DESIGN["s_range_main"])
    n_h = perp_unit(b, "horizontal")
    m = _rng.uniform(0.05, 1.0) * _rng.choice([-1, 1])
    q0 = world_to_cam(X, R0, C0); q1 = world_to_cam(X + s * b + m * n_h, R1, C1)
    if q1[2] <= 1.0:
        continue
    p0, p1 = _cam.project(q0), _cam.project(q1)
    if not (0 <= p0[0] < 640 and 0 <= p0[1] < 480 and 0 <= p1[0] < 640 and 0 <= p1[1] < 480):
        continue
    R21, t21 = relative_pose(R0, C0, R1, C1)
    F = fundamental_matrix(_cam.K, _cam.K, R21, t21)
    eps_exact = abs(float(epipolar_offset(p0, p1, F)))
    n_plane = np.cross(X - C0, b); n_plane /= np.linalg.norm(n_plane)
    eps_locked = _f * abs(m) / Zc
    eps_corrected = _f * abs(m * (n_h @ n_plane)) / q1[2]
    _rows.append((_kind, Zc, eps_exact, eps_locked, eps_corrected, abs(n_h @ n_plane)))
A2_DIAG = pd.DataFrame(_rows, columns=["camera", "Z", "eps_exact", "eps_locked_first_order",
                                       "eps_corrected_first_order", "normal_fraction"])
A2_DIAG = A2_DIAG[A2_DIAG["eps_exact"] > 0.05].reset_index(drop=True)

def _ratio_stats(num, den):
    r = num / den
    return np.percentile(r, [5, 50, 95])

_rl = _ratio_stats(A2_DIAG["eps_locked_first_order"].values, A2_DIAG["eps_exact"].values)
_rc = _ratio_stats(A2_DIAG["eps_corrected_first_order"].values, A2_DIAG["eps_exact"].values)
A2_SUMMARY = {
    "n_configurations": int(len(A2_DIAG)),
    "locked_over_exact_p05_p50_p95": [float(v) for v in _rl],
    "corrected_over_exact_p05_p50_p95": [float(v) for v in _rc],
    "fraction_regime_flip_locked": float(np.mean(
        (A2_DIAG["eps_locked_first_order"] >= PREREG["eps_perp_min"]) & (A2_DIAG["eps_exact"] < PREREG["eps_parallel_max"]))),
    "median_normal_fraction_horizontal_motion": float(A2_DIAG["normal_fraction"].median()),
}
print("2.3  A2 diagnostic over", A2_SUMMARY["n_configurations"], "street-scene configurations "
      "(horizontal perpendicular motion, forward/oblique/side cameras)")
print(f"     locked first-order / exact      : p5={_rl[0]:.2f}  median={_rl[1]:.2f}  p95={_rl[2]:.2f}")
print(f"     corrected first-order / exact   : p5={_rc[0]:.2f}  median={_rc[1]:.2f}  p95={_rc[2]:.2f}")
print(f"     movers the locked formula calls 'perpendicular' (>= 3 px) that are exactly "
      f"'parallel' (< 1 px): {100 * A2_SUMMARY['fraction_regime_flip_locked']:.1f}%")
print(f"     median share of horizontal perpendicular motion that leaves the epipolar plane: "
      f"{A2_SUMMARY['median_normal_fraction_horizontal_motion']:.2f}")
print("     -> A2 in its locked form does not hold; every regime in this notebook is assigned "
      "from the exact offset, as the lock prescribes for this case.")
np.savez_compressed(os.path.join(LOG_DIR, "a2_diagnostic.npz"), **{c: A2_DIAG[c].values for c in A2_DIAG.columns})

# ---- 2.4 slope-weight identity over the design range of x = -log(1-s) ----
_s_design = np.linspace(MOVER_DESIGN["s_range_main"][0], MOVER_DESIGN["s_range_main"][1], 2001)
_x_design = -np.log(1.0 - _s_design)
SLOPE_WEIGHT_TABLE = []
for _w in np.round(np.linspace(0, 1, 11), 2):
    _rho = np.log(_w + (1 - _w) * np.exp(_x_design))
    _beta = np.polyfit(_x_design, _rho, 1)[0]
    SLOPE_WEIGHT_TABLE.append({"w": float(_w), "beta_ols_over_design": float(_beta),
                               "local_slope_at_x0": float(1 - _w)})
SLOPE_WEIGHT_TABLE = pd.DataFrame(SLOPE_WEIGHT_TABLE)
print("\n2.4  Slope-weight identity: beta(w) = OLS slope of log(w + (1-w) e^x) on x over the "
      "design range s in", MOVER_DESIGN["s_range_main"])
print(SLOPE_WEIGHT_TABLE.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))

def beta_to_effective_w(beta):
    # invert beta(w) by interpolation (monotone decreasing in w)
    tab = SLOPE_WEIGHT_TABLE.sort_values("beta_ols_over_design")
    return float(np.interp(beta, tab["beta_ols_over_design"].values, tab["w"].values))

assert np.all(np.diff(SLOPE_WEIGHT_TABLE["beta_ols_over_design"].values) < 0), "beta(w) must be monotone"
with open(os.path.join(LOG_DIR, "a2_and_identity.json"), "w") as _f:
    json.dump({"A2": A2_SUMMARY, "slope_weight": SLOPE_WEIGHT_TABLE.to_dict("records")}, _f, indent=2)

In [ ]:
# Cell No: 014
# 3.1.1 Procedural textures with mip-map pyramids and a trilinear, footprint-aware sampler.
#       Textures are generated once from a fixed seed, so every run sees identical images.

def _tileable_noise(rng, size, base=4, octaves=6, persistence=0.55, aspect=(1, 1)):
    # zero-mean, unit-std fractal value noise that tiles seamlessly (size = (H, W) or int)
    H, W = (size, size) if np.isscalar(size) else size
    out = np.zeros((H, W), np.float32); amp, tot = 1.0, 0.0
    for o in range(octaves):
        gh, gw = int(base * aspect[0] * 2 ** o), int(base * aspect[1] * 2 ** o)
        if gh > H or gw > W:
            break
        grid = rng.random((gh, gw)).astype(np.float32)
        big = np.tile(grid, (3, 3))
        up = cv2.resize(big, (3 * W, 3 * H), interpolation=cv2.INTER_CUBIC)
        out += amp * up[H:2 * H, W:2 * W]
        tot += amp; amp *= persistence
    out /= max(tot, 1e-6)
    return (out - out.mean()) / (out.std() + 1e-6)

def _mip_pyramid(img):
    pyr = [img.astype(np.float32)]
    while min(pyr[-1].shape[:2]) > 1:
        a = pyr[-1]
        h2, w2 = a.shape[0] // 2, a.shape[1] // 2
        a = a[:2 * h2, :2 * w2]
        pyr.append(0.25 * (a[0::2, 0::2] + a[1::2, 0::2] + a[0::2, 1::2] + a[1::2, 1::2]))
    return pyr

TEXTURES = {}

def _register(name, img, tile_m, wrap=(True, True)):
    # tile_m: physical width (metres) of the texture image
    img = np.clip(img, 0.0, 1.0).astype(np.float32)
    TEXTURES[name] = dict(pyr=_mip_pyramid(img), texel=float(tile_m) / img.shape[1], wrap=wrap,
                          shape=img.shape)

def _bilinear(img, x, y, wrap):
    H, W = img.shape[:2]
    x = x - 0.5; y = y - 0.5
    x0 = np.floor(x); y0 = np.floor(y)
    fx = (x - x0).astype(np.float32)[:, None]; fy = (y - y0).astype(np.float32)[:, None]
    x0 = x0.astype(np.int64); y0 = y0.astype(np.int64)
    if wrap[0]:
        x0 %= W; x1 = (x0 + 1) % W
    else:
        x1 = np.clip(x0 + 1, 0, W - 1); x0 = np.clip(x0, 0, W - 1)
    if wrap[1]:
        y0 %= H; y1 = (y0 + 1) % H
    else:
        y1 = np.clip(y0 + 1, 0, H - 1); y0 = np.clip(y0, 0, H - 1)
    return (img[y0, x0] * (1 - fx) * (1 - fy) + img[y0, x1] * fx * (1 - fy)
            + img[y1, x0] * (1 - fx) * fy + img[y1, x1] * fx * fy)

def tex_sample(name, u_m, v_m, footprint_m):
    # trilinear lookup; u_m, v_m texture coordinates in metres, footprint_m metres per pixel
    T = TEXTURES[name]; pyr = T["pyr"]; texel = T["texel"]
    n_lev = len(pyr)
    lev = np.log2(np.maximum(np.asarray(footprint_m, np.float64) / texel, 1.0))
    lev = np.minimum(lev, n_lev - 1)
    l0 = np.floor(lev).astype(np.int32)
    fr = (lev - l0).astype(np.float32)
    out = np.empty((u_m.shape[0], pyr[0].shape[2]), np.float32)
    for L in np.unique(l0):
        sel = np.nonzero(l0 == L)[0]
        sc = texel * (2 ** L)
        c0 = _bilinear(pyr[L], u_m[sel] / sc, v_m[sel] / sc, T["wrap"])
        if L + 1 < n_lev:
            c1 = _bilinear(pyr[L + 1], u_m[sel] / (2 * sc), v_m[sel] / (2 * sc), T["wrap"])
            f = fr[sel, None]
            out[sel] = c0 * (1 - f) + c1 * f
        else:
            out[sel] = c0
    return out

def _build_textures(seed):
    rng = np.random.default_rng(seed)
    N = 512
    # asphalt: fine aggregate + patches
    n1, n2, n3 = _tileable_noise(rng, N, base=32), _tileable_noise(rng, N, base=4), _tileable_noise(rng, N, base=64)
    n4 = _tileable_noise(rng, N, base=12)
    g = 0.30 + 0.034 * n1 + 0.010 * n2 + 0.014 * n4 + 0.05 * (n3 > 2.2) - 0.04 * (n3 < -2.3)
    _register("asphalt", np.stack([g * 1.02, g * 1.0, g * 0.97], -1), tile_m=4.0)
    # paving slabs 0.5 m with grout
    yy, xx = np.mgrid[0:N, 0:N] / N * 3.0
    grout = ((np.mod(xx, 0.5) < 0.012) | (np.mod(yy, 0.5) < 0.012)).astype(np.float32)
    tile_id = (np.floor(xx / 0.5) * 7 + np.floor(yy / 0.5) * 13).astype(np.int64)
    tile_var = rng.uniform(-0.05, 0.05, tile_id.max() + 1)[tile_id]
    p = 0.60 + tile_var + 0.025 * _tileable_noise(rng, N, base=24)
    p = p * (1 - grout) + 0.36 * grout
    _register("paving", np.stack([p * 1.0, p * 0.98, p * 0.94], -1), tile_m=3.0)
    # facades: 8 m x 8 m tiles, window grid 4 x 3
    facade_styles = [
        dict(wall=(0.78, 0.70, 0.58), brick=False, win=(0.16, 0.20, 0.25), frame=(0.92, 0.92, 0.90)),
        dict(wall=(0.62, 0.30, 0.22), brick=True,  win=(0.14, 0.17, 0.22), frame=(0.88, 0.86, 0.82)),
        dict(wall=(0.58, 0.58, 0.56), brick=False, win=(0.20, 0.25, 0.30), frame=(0.30, 0.30, 0.32)),
        dict(wall=(0.70, 0.76, 0.80), brick=False, win=(0.12, 0.16, 0.22), frame=(0.95, 0.95, 0.95)),
        dict(wall=(0.84, 0.78, 0.64), brick=False, win=(0.18, 0.18, 0.20), frame=(0.45, 0.32, 0.22)),
        dict(wall=(0.42, 0.26, 0.20), brick=True,  win=(0.16, 0.18, 0.21), frame=(0.80, 0.78, 0.70)),
    ]
    yy, xx = np.mgrid[0:N, 0:N] / N * 8.0           # metres; xx along facade, yy downward
    for i, st in enumerate(facade_styles):
        wall = np.ones((N, N, 3), np.float32) * np.array(st["wall"], np.float32)
        wall *= (1 + 0.06 * _tileable_noise(rng, N, base=16))[..., None]
        if st["brick"]:
            row = np.floor(yy / 0.075)
            off = np.where(np.mod(row, 2) == 0, 0.0, 0.12)
            mortar = (np.mod(yy, 0.075) < 0.010) | (np.mod(xx + off, 0.24) < 0.010)
            bid = (np.floor((xx + off) / 0.24) * 31 + row * 17).astype(np.int64)
            bvar = rng.uniform(-0.08, 0.08, bid.max() + 1)[bid]
            wall = wall * (1 + bvar)[..., None]
            wall[mortar] = np.array([0.72, 0.70, 0.66], np.float32)
        cx_ = np.mod(xx, 2.0); cy_ = np.mod(yy, 8.0 / 3.0)
        win = (np.abs(cx_ - 1.0) < 0.55) & (np.abs(cy_ - 1.25) < 0.75)
        frame = (np.abs(cx_ - 1.0) < 0.62) & (np.abs(cy_ - 1.25) < 0.82) & ~win
        sill = (np.abs(cx_ - 1.0) < 0.70) & (np.abs(cy_ - 2.10) < 0.05)
        glass = np.array(st["win"], np.float32) * (1 + 0.5 * np.clip(1.3 - cy_, 0, 1))[..., None]
        glass = glass * (1 + 0.15 * _tileable_noise(rng, N, base=8))[..., None]
        img = wall.copy()
        img[frame] = np.array(st["frame"], np.float32)
        img[sill] = np.array(st["frame"], np.float32) * 0.9
        img[win] = glass[win]
        _register(f"facade{i}", img, tile_m=8.0)
    # wood crate planks 0.15 m
    N2 = 256
    yy, xx = np.mgrid[0:N2, 0:N2] / N2 * 1.2
    plank = np.floor(yy / 0.15).astype(np.int64)
    pc = rng.uniform(0.85, 1.15, plank.max() + 1)[plank]
    grain = _tileable_noise(rng, (N2, N2), base=4, aspect=(4, 1))
    w = (0.55 + 0.06 * grain) * pc
    gap = np.mod(yy, 0.15) < 0.008
    img = np.stack([w * 1.0, w * 0.74, w * 0.46], -1); img[gap] = 0.12
    _register("wood", img, tile_m=1.2)
    # corrugated container (grey, tinted at render time)
    yy, xx = np.mgrid[0:N2, 0:N2] / N2 * 2.4
    corr = 0.82 + 0.13 * np.sin(2 * np.pi * xx / 0.3)
    rust = _tileable_noise(rng, N2, base=8)
    c = corr * (1 - 0.10 * (rust > 1.4))
    img = np.stack([c, c, c], -1) * (1 + 0.04 * _tileable_noise(rng, N2, base=32))[..., None]
    _register("container", img, tile_m=2.4)
    # patterned crate: colour patchwork + stripes (texture-only object)
    yy, xx = np.mgrid[0:N2, 0:N2]
    cell = (yy // 32) * 8 + (xx // 32)
    cols = rng.uniform(0.15, 0.95, (64, 3))
    img = cols[cell].astype(np.float32)
    stripes = (np.mod(xx + yy, 24) < 6)
    img[stripes] *= 0.55
    img *= (1 + 0.08 * _tileable_noise(rng, N2, base=16))[..., None]
    _register("pattern", img, tile_m=1.0)
    # car paint micro-variation, glass, tyre, metal, foliage, bark
    _register("paint", np.repeat((1 + 0.04 * _tileable_noise(rng, 128, base=8))[..., None], 3, -1), tile_m=1.0)
    yy, xx = np.mgrid[0:128, 0:128] / 128.0
    gl = 0.11 + 0.10 * np.clip(1 - 1.6 * yy, 0, 1) + 0.008 * _tileable_noise(rng, 128, base=4)
    _register("glass", np.stack([gl * 0.92, gl * 1.0, gl * 1.12], -1), tile_m=0.6)
    # car body side profile (0.8 m tall): shoulder highlight, trim line, darker sill
    prof = np.ones(128, np.float32)
    yb = np.arange(128) / 128.0 * 0.8
    prof[yb < 0.04] = 1.18; prof[(yb >= 0.52) & (yb < 0.58)] = 0.45; prof[yb >= 0.58] = 0.78
    body = prof[:, None] * (1 + 0.02 * _tileable_noise(rng, 128, base=8))
    _register("carbody", np.repeat(body[..., None], 3, -1), tile_m=0.8)
    t = 0.10 + 0.03 * _tileable_noise(rng, 64, base=8)
    _register("tyre", np.stack([t, t, t], -1), tile_m=0.5)
    m = 0.55 + 0.04 * _tileable_noise(rng, 64, base=8)
    _register("metal", np.stack([m, m * 1.01, m * 1.03], -1), tile_m=1.0)
    f1, f2 = _tileable_noise(rng, N2, base=16), _tileable_noise(rng, N2, base=64)
    leaf = np.clip(0.30 + 0.10 * f1 + 0.08 * f2, 0.05, 0.6)
    img = np.stack([leaf * 0.45, leaf * 0.95, leaf * 0.30], -1)
    img[(f2 < -1.2)] *= 0.35
    _register("foliage", img, tile_m=2.0)
    bk = 0.30 + 0.06 * _tileable_noise(rng, (128, 128), base=4, aspect=(1, 8))
    _register("bark", np.stack([bk * 1.0, bk * 0.80, bk * 0.62], -1), tile_m=1.0)
    # distant skyline with alpha (1200 m wide, 300 m tall), wraps horizontally
    SW, SH = 1024, 256
    img = np.zeros((SH, SW, 4), np.float32)
    x = 0
    while x < SW:
        bw = int(rng.integers(12, 60)); bh = int(rng.integers(30, 200))
        tone = rng.uniform(0.50, 0.68)
        img[SH - bh:, x:x + bw, :3] = np.array([tone * 0.92, tone * 0.96, tone * 1.04])
        img[SH - bh:, x:x + bw, 3] = 1.0
        win = (np.mod(np.arange(SH)[:, None], 6) < 2) & (np.mod(np.arange(x, x + bw)[None, :], 5) < 2)
        reg = img[SH - bh:, x:x + bw, :3]
        reg[win[SH - bh:, :reg.shape[1]]] *= 0.9
        x += bw + int(rng.integers(0, 6))
    _register("skyline", img, tile_m=1200.0, wrap=(True, False))
    # clouds (sampled by ray direction, no parallax)
    cl = _tileable_noise(rng, (256, 1024), base=4, aspect=(1, 4))
    _register("clouds", np.repeat(np.clip((cl - 0.3) / 1.6, 0, 1)[..., None], 3, -1), tile_m=2 * np.pi)

with stage("textures"):
    _build_textures(seed_for("textures"))
print(f"{len(TEXTURES)} textures built:", ", ".join(f"{k}{list(v['shape'][:2])}" for k, v in TEXTURES.items()))

In [ ]:
# Cell No: 015
# 3.1.2 Scene description: street layout, static furniture, cars and crates as oriented boxes.
#       A scene is a JSON-serialisable dict; rendering (3.1.3) is a pure function of it.
SEM = {"sky": 0, "road": 1, "sidewalk": 2, "facade": 3, "backdrop": 4, "static_box": 5, "mover": 6}
SEM_NAMES = {v: k for k, v in SEM.items()}
CAR_COLORS = [(0.70, 0.07, 0.07), (0.10, 0.20, 0.52), (0.86, 0.86, 0.84), (0.13, 0.13, 0.15),
              (0.55, 0.57, 0.60), (0.12, 0.34, 0.18), (0.80, 0.58, 0.10), (0.35, 0.22, 0.45)]
CONTAINER_TINTS = [(0.85, 0.40, 0.12), (0.15, 0.32, 0.62), (0.20, 0.50, 0.25), (0.70, 0.12, 0.10),
                   (0.80, 0.78, 0.72)]


def _box(center, half, yaw, mat, kind="static", mover=-1, part=""):
    return dict(center=[float(v) for v in center], half=[float(v) for v in half], yaw=float(yaw),
                mat=mat, kind=kind, mover=int(mover), part=part)

def _mat(tex, tint=(1, 1, 1), x=None, y=None, z=None):
    # material per face axis (box-local x / y / z faces); default = same texture everywhere
    base = [tex, [float(t) for t in tint]]
    return {"x": x or base, "y": y or base, "z": z or base}

def car_boxes(origin, yaw, L, Wd, color, kind="static", mover=-1):
    # origin: ground-contact centre (world, y down). Returns the boxes of a low-poly car.
    Ry = rot_y(yaw)
    def place(local_c, half, mat, part):
        return _box(np.asarray(origin) + Ry @ np.asarray(local_c, np.float64), half, yaw, mat, kind, mover, part)
    paint = ["paint", list(color)]
    glass = ["glass", [1, 1, 1]]
    boxes = [
        place((0, -0.72, 0), (Wd / 2, 0.40, L / 2),
              {"x": ["carbody", list(color)], "y": paint, "z": ["carbody", list(color)]}, "body"),
        place((0, -1.40, -0.15 * L), (Wd / 2 * 0.90, 0.30, L / 2 * 0.52),
              {"x": glass, "y": paint, "z": glass}, "cabin"),
    ]
    for sx in (-1, 1):
        for sz in (-1, 1):
            boxes.append(place((sx * (Wd / 2 - 0.13), -0.33, sz * (L / 2 - 0.80)), (0.13, 0.33, 0.33),
                               _mat("tyre"), "wheel"))
        boxes.append(place((sx * (Wd / 2 - 0.30), -0.86, L / 2 + 0.012), (0.20, 0.07, 0.015),
                           _mat("metal", (1.6, 1.6, 1.45)), "headlight"))
        boxes.append(place((sx * (Wd / 2 - 0.30), -0.88, -L / 2 - 0.012), (0.20, 0.06, 0.015),
                           _mat("metal", (1.5, 0.15, 0.12)), "taillight"))
    return boxes

def crate_boxes(center, half, yaw, style, tint, kind="mover", mover=-1):
    tex = {"wood": "wood", "container": "container", "pattern": "pattern"}[style]
    return [_box(center, half, yaw, _mat(tex, tint), kind, mover, "crate")]

STREET_HALF_WIDTH = {"forward": (6.0, 9.0), "oblique": (9.0, 12.0), "side": (11.0, 14.0)}

def lane_centres(road_half):
    c = [1.75 + 3.5 * k for k in range(8) if 1.75 + 3.5 * k <= road_half - 1.2]
    return np.array(sorted([-v for v in c] + c))

def sample_street(rng, kind="forward", z_min=-60.0, z_max=420.0):
    # wider streets for oblique / side-looking cameras so that whole vehicles fit in view
    g = float(rng.uniform(*CAMERA_DESIGN["height_m"]))           # camera height = ground y
    wR = float(rng.uniform(*STREET_HALF_WIDTH[kind]))              # road half width
    side = float(rng.uniform(2.0, 4.0))                            # sidewalk width
    street = dict(ground_y=g, road_half=wR, facade_x_left=-(wR + side), facade_x_right=wR + side,
                  backdrop_z=450.0, skyline_shift=float(rng.uniform(0, 1200)))
    for sd in ("left", "right"):
        blocks, z = [], z_min
        while z < z_max:
            L = float(rng.uniform(10, 30))
            blocks.append([z, z + L, float(rng.uniform(9, 30)), int(rng.integers(0, 6)), float(rng.uniform(0, 8))])
            z += L
        street["blocks_" + sd] = blocks
    return street

def sample_static_furniture(rng, street, z_from=-25.0, z_to=140.0):
    g, wR = street["ground_y"], street["road_half"]
    boxes, shadows = [], []
    for sgn in (-1, 1):
        z = z_from + rng.uniform(0, 10)
        while z < z_to:                                           # street lights
            x = sgn * (wR + 0.6)
            boxes.append(_box((x, g - 3.5, z), (0.07, 3.5, 0.07), 0, _mat("metal"), part="pole"))
            shadows.append(shadow_fp(x, z, 0.07, 0.07, 0.0, 0.35))
            boxes.append(_box((x - sgn * 0.7, g - 6.95, z), (0.7, 0.05, 0.05), 0, _mat("metal"), part="arm"))
            boxes.append(_box((x - sgn * 1.35, g - 6.85, z), (0.28, 0.08, 0.16), 0,
                              _mat("metal", (1.5, 1.45, 1.2)), part="lamp"))
            z += rng.uniform(22, 32)
        z = z_from + rng.uniform(0, 8)
        while z < z_to:                                           # trees
            x = sgn * (wR + 1.4 + rng.uniform(0, 0.6))
            boxes.append(_box((x, g - 1.6, z), (0.16, 1.6, 0.16), 0, _mat("bark"), part="trunk"))
            shadows.append(shadow_fp(x, z, 0.9, 0.9, 0.0, 0.30))
            hc = rng.uniform(1.0, 1.5)
            boxes.append(_box((x, g - 3.2 - hc, z), (rng.uniform(1.1, 1.6), hc, rng.uniform(1.1, 1.6)),
                              rng.uniform(0, np.pi), _mat("foliage", (rng.uniform(0.8, 1.2),) * 3), part="canopy"))
            z += rng.uniform(9, 18)
        z = z_from + rng.uniform(0, 6)
        while z < z_to:                                           # parked cars
            if rng.random() < 0.35:
                L, Wd = rng.uniform(4.1, 4.8), rng.uniform(1.70, 1.85)
                col = CAR_COLORS[int(rng.integers(len(CAR_COLORS)))]
                yaw_p = 0.0 if rng.random() < 0.5 else np.pi
                boxes += car_boxes((sgn * (wR - 1.15), g, z), yaw_p, L, Wd, col)
                shadows.append(shadow_fp(sgn * (wR - 1.15), z, Wd / 2, L / 2, yaw_p, 0.60))
            z += rng.uniform(6.0, 7.5)
        z = z_from + rng.uniform(0, 12)
        while z < z_to:                                           # bins / boxes on the sidewalk
            if rng.random() < 0.5:
                tint = CONTAINER_TINTS[int(rng.integers(len(CONTAINER_TINTS)))]
                boxes.append(_box((sgn * (wR + 0.9), g - 0.55, z), (0.32, 0.55, 0.32), 0,
                                  _mat("container", tint), part="bin"))
                shadows.append(shadow_fp(sgn * (wR + 0.9), z, 0.32, 0.32, 0.0, 0.45))
            z += rng.uniform(8, 16)
    return boxes, shadows

def shadow_fp(cx, cz, hx, hz, yaw, strength, mover=-1):
    # soft contact shadow on the ground (world x-z footprint); movers' shadows move with them
    return dict(c=[float(cx), float(cz)], h=[float(hx), float(hz)], yaw=float(yaw), k=float(strength), mover=int(mover))

def box_corners(b, offset=np.zeros(3)):
    c = np.asarray(b["center"]) + offset; h = np.asarray(b["half"])
    sg = np.array(list(itertools.product((-1, 1), repeat=3)), np.float64)
    return c + (sg * h) @ rot_y(b["yaw"]).T

print("Scene description helpers ready (street, furniture, cars, crates).")

In [ ]:
# Cell No: 016
# 3.1.3 Analytic ray casting and shading. Ray directions are scaled so that their
#       camera-z component is 1; the ray parameter t is therefore the camera depth Z.
SUN_DIR = np.array([-0.35, -1.0, 0.45]) / np.linalg.norm([-0.35, -1.0, 0.45])   # towards the sun (y down)
SKY_HORIZON = np.array([0.80, 0.85, 0.90], np.float32)
SKY_ZENITH = np.array([0.40, 0.58, 0.84], np.float32)
_TEX_IDS = None


def _scene_box_arrays(scene):
    # stacked per-box arrays (cached on the scene dict); textures mapped to integer ids
    global _TEX_IDS
    if _TEX_IDS is None:
        _TEX_IDS = {name: i for i, name in enumerate(sorted(TEXTURES))}
    if "_arr" in scene:
        return scene["_arr"]
    B = scene["boxes"]
    arr = dict(
        center=np.array([b["center"] for b in B], np.float64).reshape(-1, 3),
        half=np.array([b["half"] for b in B], np.float64).reshape(-1, 3),
        yaw=np.array([b["yaw"] for b in B], np.float64),
        mover=np.array([b["mover"] for b in B], np.int32),
        tex=np.array([[_TEX_IDS[b["mat"][a][0]] for a in "xyz"] for b in B], np.int32).reshape(-1, 3),
        tint=np.array([[b["mat"][a][1] for a in "xyz"] for b in B], np.float32).reshape(-1, 3, 3),
    )
    scene["_arr"] = arr
    return arr

def _mover_offsets(scene, frame_k):
    deltas = np.array([m["delta"] for m in scene["movers"]], np.float64).reshape(-1, 3)
    return deltas * float(frame_k)

def cast_rays(scene, C, D, frame_k, include_movers=True, grid=None, pix=None, cam=None, R=None,
              only_movers=None):
    # C: (3,) ray origin; D: (N,3) world directions (camera z = 1).
    # grid=(W,H) when D is the full row-major pixel grid (fast culling); otherwise pix=(u,v)
    # arrays give each ray's pixel position for box culling.
    N = D.shape[0]
    t = np.full(N, np.inf); sem = np.zeros(N, np.int8)
    bid = np.full(N, -1, np.int32); aux = np.zeros(N, np.int32)
    st = scene["street"]; g = st["ground_y"]
    with np.errstate(divide="ignore", invalid="ignore"):
        if only_movers is None:
            tg = (g - C[1]) / D[:, 1]
            ok = (D[:, 1] > 1e-12) & (tg > 1e-6)
            t[ok] = tg[ok]
            xg = C[0] + tg * D[:, 0]
            sem[ok] = np.where(np.abs(xg[ok]) < st["road_half"], SEM["road"], SEM["sidewalk"])
            for code, side in ((1, "left"), (2, "right")):
                xp = st["facade_x_" + side]
                tf = (xp - C[0]) / D[:, 0]
                dir_ok = (D[:, 0] < 0) if side == "left" else (D[:, 0] > 0)
                ok = dir_ok & (tf > 1e-6) & (tf < t)
                if not ok.any():
                    continue
                idx_ok = np.nonzero(ok)[0]
                y = C[1] + tf[idx_ok] * D[idx_ok, 1]; z = C[2] + tf[idx_ok] * D[idx_ok, 2]
                blocks = np.asarray(st["blocks_" + side], np.float64)
                bi = np.searchsorted(blocks[:, 0], z, side="right") - 1
                inb = (bi >= 0) & (bi < len(blocks))
                bi_c = np.clip(bi, 0, len(blocks) - 1)
                inb &= z < blocks[bi_c, 1]
                inb &= (y >= g - blocks[bi_c, 2]) & (y <= g)
                sel = idx_ok[inb]
                t[sel] = tf[sel]; sem[sel] = SEM["facade"]; aux[sel] = code * 10000 + bi_c[inb]
            tb = (st["backdrop_z"] - C[2]) / D[:, 2]
            ok = (D[:, 2] > 1e-9) & (tb > 1e-6) & (tb < t)
            if ok.any():
                idx_ok = np.nonzero(ok)[0]
                x = C[0] + tb[idx_ok] * D[idx_ok, 0]; y = C[1] + tb[idx_ok] * D[idx_ok, 1]
                fp = tb[idx_ok] * np.linalg.norm(D[idx_ok], axis=1) / 600.0
                a = tex_sample("skyline", x + st["skyline_shift"], y - (g - 300.0), fp)[:, 3]
                sel = idx_ok[a > 0.5]
                t[sel] = tb[sel]; sem[sel] = SEM["backdrop"]
        arr = _scene_box_arrays(scene)
        offs = _mover_offsets(scene, frame_k)
        nb = arr["center"].shape[0]
        if grid is not None:
            Wg, Hg = grid
        for i in range(nb):
            mv = arr["mover"][i]
            if mv >= 0 and not include_movers:
                continue
            if only_movers is not None and mv not in only_movers:
                continue
            ctr = arr["center"][i] + (offs[mv] if mv >= 0 else 0.0)
            hb = arr["half"][i]
            # ---- cull with the projected bounding box ----
            if cam is not None and R is not None:
                cn = box_corners({"center": ctr, "half": hb, "yaw": arr["yaw"][i]})
                Xc = (cn - C) @ R
                if np.all(Xc[:, 2] < 0.05):
                    continue
                if np.all(Xc[:, 2] > 0.05):
                    uv = np.stack([cam.cx + cam.fx * Xc[:, 0] / Xc[:, 2], cam.cy + cam.fy * Xc[:, 1] / Xc[:, 2]], -1)
                    u0, v0 = np.floor(uv.min(0)) - 1; u1, v1 = np.ceil(uv.max(0)) + 1
                    if grid is not None:
                        u0, u1 = int(max(u0, 0)), int(min(u1, Wg - 1)); v0, v1 = int(max(v0, 0)), int(min(v1, Hg - 1))
                        if u1 < u0 or v1 < v0:
                            continue
                        sel = (np.arange(v0, v1 + 1)[:, None] * Wg + np.arange(u0, u1 + 1)[None, :]).ravel()
                    else:
                        sel = np.nonzero((pix[0] >= u0) & (pix[0] <= u1) & (pix[1] >= v0) & (pix[1] <= v1))[0]
                        if sel.size == 0:
                            continue
                else:
                    sel = np.arange(N)
            else:
                sel = np.arange(N)
            cy_, sy_ = np.cos(arr["yaw"][i]), np.sin(arr["yaw"][i])
            o = C - ctr
            ol = np.array([cy_ * o[0] - sy_ * o[2], o[1], sy_ * o[0] + cy_ * o[2]])
            Ds = D[sel]
            dl = np.stack([cy_ * Ds[:, 0] - sy_ * Ds[:, 2], Ds[:, 1], sy_ * Ds[:, 0] + cy_ * Ds[:, 2]], -1)
            dl = np.where(np.abs(dl) < 1e-12, 1e-12, dl)
            t1 = (-hb - ol) / dl; t2 = (hb - ol) / dl
            tn = np.minimum(t1, t2); tx = np.maximum(t1, t2)
            tmin = tn.max(1); tmax = tx.min(1)
            hit = (tmax >= tmin) & (tmin > 1e-6) & (tmin < t[sel])
            if not hit.any():
                continue
            hs = sel[hit]
            ax = tn[hit].argmax(1)
            sgn = (-np.sign(dl[hit, :][np.arange(hit.sum()), ax])).astype(np.int32)
            t[hs] = tmin[hit]
            sem[hs] = SEM["mover"] if mv >= 0 else SEM["static_box"]
            bid[hs] = i
            aux[hs] = ax * 2 + (sgn > 0)
    return dict(t=t, sem=sem, bid=bid, aux=aux)

def shade_rays(scene, C, D, hit, frame_k, f_mean):
    N = D.shape[0]
    t, sem, bid, aux = hit["t"], hit["sem"], hit["bid"], hit["aux"]
    col = np.zeros((N, 3), np.float32)
    nrm = np.zeros((N, 3), np.float64)
    dn = np.linalg.norm(D, axis=1)
    fin = np.isfinite(t)
    P = C + np.where(fin, t, 0.0)[:, None] * D
    fp_minor = np.where(fin, t, 0.0) * dn / f_mean
    st = scene["street"]; g = st["ground_y"]
    # ---- ground ----
    for code in (SEM["road"], SEM["sidewalk"]):
        s_ = np.nonzero(sem == code)[0]
        if s_.size == 0:
            continue
        x, z = P[s_, 0], P[s_, 2]
        cos_t = np.abs(D[s_, 1]) / dn[s_]
        fp = fp_minor[s_] / np.sqrt(np.maximum(cos_t, 0.02))
        if code == SEM["road"]:
            alb = tex_sample("asphalt", x, z, fp)
            fx_ = fp_minor[s_]; fz_ = fp_minor[s_] / np.maximum(cos_t, 0.02)
            paint = np.zeros(s_.size, np.float32)
            wR = st["road_half"]
            lines = [(1.75, True), (-1.75, True), (5.25, True), (-5.25, True), (wR - 0.25, False), (-(wR - 0.25), False)]
            for xl, dashed in lines:
                if abs(xl) > wR - 0.2 and dashed:
                    continue
                cov_x = np.clip((0.07 - np.abs(x - xl)) / np.maximum(fx_, 1e-4) + 0.5, 0, 1)
                if dashed:
                    ph = np.mod(z + 3.0 * (xl > 0), 9.0)
                    cz = np.clip(np.minimum(ph, 3.0 - ph) / np.maximum(fz_, 1e-4) + 0.5, 0, 1) * (ph < 3.0 + fz_)
                    cz = np.where(fz_ > 3.0, 1.0 / 3.0, cz)
                    cov_x = cov_x * cz
                paint = np.maximum(paint, cov_x.astype(np.float32))
            alb = alb * (1 - paint[:, None]) + np.array([0.84, 0.83, 0.78], np.float32) * paint[:, None]
        else:
            alb = tex_sample("paving", x, z, fp)
            curb = np.abs(np.abs(x) - st["road_half"]) < 0.15
            alb[curb] = alb[curb] * 0.72
        col[s_] = alb
        nrm[s_] = (0.0, -1.0, 0.0)
    # ---- soft contact shadows (furniture, parked cars, ground movers) ----
    gi = np.nonzero((sem == SEM["road"]) | (sem == SEM["sidewalk"]))[0]
    shadow_mover = np.zeros(N, bool)
    if gi.size and scene.get("shadows"):
        gx, gz = P[gi, 0], P[gi, 2]
        fac = np.ones(gi.size, np.float32)
        offs_s = _mover_offsets(scene, frame_k)
        for sh in scene["shadows"]:
            cx_, cz_ = sh["c"]; hx, hz = sh["h"]
            if sh["mover"] >= 0:
                cx_ += offs_s[sh["mover"]][0]; cz_ += offs_s[sh["mover"]][2]
            r = np.hypot(hx, hz) + 0.7
            q = np.nonzero((np.abs(gx - cx_) < r) & (np.abs(gz - cz_) < r))[0]
            if q.size == 0:
                continue
            c_, s_y = np.cos(sh["yaw"]), np.sin(sh["yaw"])
            dx, dz = gx[q] - cx_, gz[q] - cz_
            lx, lz = c_ * dx - s_y * dz, s_y * dx + c_ * dz
            ox, oz = np.maximum(np.abs(lx) - hx, 0), np.maximum(np.abs(lz) - hz, 0)
            dist = np.hypot(ox, oz)
            inner = np.clip(np.minimum(hx - np.abs(lx), hz - np.abs(lz)) / 0.3, 0, 1)
            amt = sh["k"] * np.clip(1 - dist / 0.6, 0, 1) ** 2 * (0.75 + 0.25 * inner)
            fac[q] *= (1 - amt).astype(np.float32)
            if sh["mover"] >= 0:
                shadow_mover[gi[q[amt > 0.02]]] = True
        col[gi] *= fac[:, None]
    # ---- facades ----
    s_ = np.nonzero(sem == SEM["facade"])[0]
    if s_.size:
        side = aux[s_] // 10000; bi = aux[s_] % 10000
        cos_t = np.abs(D[s_, 0]) / dn[s_]
        fp = fp_minor[s_] / np.sqrt(np.maximum(cos_t, 0.02))
        for sd_code, sd in ((1, "left"), (2, "right")):
            blocks = np.asarray(st["blocks_" + sd], np.float64)
            m = side == sd_code
            if not m.any():
                continue
            ii = s_[m]; b_i = bi[m]
            styles = blocks[b_i, 3].astype(np.int32); shift = blocks[b_i, 4]
            for sty in np.unique(styles):
                q = styles == sty
                jj = ii[q]
                col[jj] = tex_sample(f"facade{sty}", P[jj, 2] + shift[q], P[jj, 1] - g, fp[m][q])
            plinth = P[ii, 1] > g - 0.45
            col[ii[plinth]] *= 0.62
            nrm[ii] = (1.0, 0.0, 0.0) if sd == "left" else (-1.0, 0.0, 0.0)
    # ---- backdrop ----
    s_ = np.nonzero(sem == SEM["backdrop"])[0]
    if s_.size:
        fp = fp_minor[s_]
        col[s_] = tex_sample("skyline", P[s_, 0] + st["skyline_shift"], P[s_, 1] - (g - 300.0), fp)[:, :3]
        nrm[s_] = (0.0, -0.3, -1.0)
    # ---- boxes (static furniture and movers) ----
    s_ = np.nonzero(bid >= 0)[0]
    if s_.size:
        arr = _scene_box_arrays(scene)
        offs = _mover_offsets(scene, frame_k)
        b = bid[s_]
        mv = arr["mover"][b]
        ctr = arr["center"][b] + np.where(mv[:, None] >= 0, offs[np.maximum(mv, 0)] if len(offs) else 0.0, 0.0)
        hb = arr["half"][b]; yaw = arr["yaw"][b]
        cy_, sy_ = np.cos(yaw), np.sin(yaw)
        d = P[s_] - ctr
        lp = np.stack([cy_ * d[:, 0] - sy_ * d[:, 2], d[:, 1], sy_ * d[:, 0] + cy_ * d[:, 2]], -1) + hb
        ax = aux[s_] // 2; sg = np.where(aux[s_] % 2 == 1, 1.0, -1.0)
        u = np.where(ax == 0, lp[:, 2], lp[:, 0])
        v = np.where(ax == 1, lp[:, 2], lp[:, 1])
        nl = np.zeros((s_.size, 3)); nl[np.arange(s_.size), ax] = sg
        nw = np.stack([cy_ * nl[:, 0] + sy_ * nl[:, 2], nl[:, 1], -sy_ * nl[:, 0] + cy_ * nl[:, 2]], -1)
        cos_t = np.abs(np.sum(nw * D[s_], 1)) / dn[s_]
        fp = fp_minor[s_] / np.sqrt(np.maximum(cos_t, 0.02))
        tid = arr["tex"][b, ax]; tint = arr["tint"][b, ax]
        inv_tex = {v_: k for k, v_ in _TEX_IDS.items()}
        for tk in np.unique(tid):
            q = tid == tk
            col[s_[q]] = tex_sample(inv_tex[int(tk)], u[q], v[q], fp[q])[:, :3] * tint[q]
        nrm[s_] = nw
    # ---- lighting, haze, sky ----
    lam = np.clip(nrm @ SUN_DIR, 0, None).astype(np.float32)
    lit = col * (0.55 + 0.55 * lam)[:, None]
    back = sem == SEM["backdrop"]
    lit[back] = col[back]
    haze = (1 - np.exp(-np.where(fin, t, 0.0) / 900.0)).astype(np.float32)
    lit = lit * (1 - haze[:, None]) + SKY_HORIZON * haze[:, None]
    sk = np.nonzero(~fin)[0]
    if sk.size:
        Dn = D[sk] / dn[sk, None]
        elev = np.arcsin(np.clip(-Dn[:, 1], -1, 1))
        grad = np.clip(elev / 0.6, 0, 1) ** 0.7
        sky = SKY_HORIZON * (1 - grad[:, None]) + SKY_ZENITH * grad[:, None]
        az = np.arctan2(Dn[:, 0], Dn[:, 2]) + np.pi
        cl = tex_sample("clouds", az, (np.pi / 2 - elev) * 0.5, np.full(sk.size, 1.0 / f_mean))[:, 0]
        cl = cl * np.clip(elev / 0.05, 0, 1)
        lit[sk] = (sky * (1 - 0.8 * cl[:, None]) + 0.97 * 0.8 * cl[:, None]).astype(np.float32)
    return lit, shadow_mover

def render_frame(scene, frame_k, include_movers=True, aa=True, add_noise=True):
    camd = scene["cams"][frame_k]
    cam = PinholeCamera.from_dict(camd)
    R = np.asarray(camd["R"], np.float64); C = np.asarray(camd["C"], np.float64)
    W, H = cam.W, cam.H
    vv, uu = np.mgrid[0:H, 0:W]
    D = cam.rays(uu.ravel().astype(np.float64), vv.ravel().astype(np.float64)) @ R.T
    hit = cast_rays(scene, C, D, frame_k, include_movers, grid=(W, H), cam=cam, R=R)
    f_mean = 0.5 * (cam.fx + cam.fy)
    rgb, shadow_mover = shade_rays(scene, C, D, hit, frame_k, f_mean)
    t = hit["t"].reshape(H, W); sem = hit["sem"].reshape(H, W); bid = hit["bid"].reshape(H, W)
    if aa:
        key = sem.astype(np.int64) * 1000003 + bid
        e = np.zeros((H, W), bool)
        dh = key[:, 1:] != key[:, :-1]; dv = key[1:, :] != key[:-1, :]
        tt = np.where(np.isfinite(t), t, 1e9)
        dh |= np.abs(tt[:, 1:] / tt[:, :-1] - 1) > 0.03; dv |= np.abs(tt[1:, :] / tt[:-1, :] - 1) > 0.03
        e[:, 1:] |= dh; e[:, :-1] |= dh; e[1:, :] |= dv; e[:-1, :] |= dv
        ei = np.nonzero(e.ravel())[0]
        if ei.size:
            acc = rgb[ei].copy()
            for du, dv_ in ((-0.25, -0.25), (0.25, -0.25), (-0.25, 0.25), (0.25, 0.25)):
                pu = uu.ravel()[ei] + du; pv = vv.ravel()[ei] + dv_
                Ds = cam.rays(pu, pv) @ R.T
                h2 = cast_rays(scene, C, Ds, frame_k, include_movers, pix=(pu, pv), cam=cam, R=R)
                acc += shade_rays(scene, C, Ds, h2, frame_k, f_mean)[0]
            rgb[ei] = acc / 5.0
    rgb = rgb.reshape(H, W, 3)
    li = scene["light"]
    rgb = rgb * np.asarray(li["white_balance"], np.float32) * li["exposure"]
    if add_noise:
        nrng = np.random.default_rng(scene["seed"] * 31 + frame_k * 7 + (0 if include_movers else 1))
        rgb = rgb + nrng.normal(0, li["noise_sigma"], rgb.shape).astype(np.float32)
    rgb8 = (np.clip(rgb, 0, 1) * 255 + 0.5).astype(np.uint8)
    arr = _scene_box_arrays(scene)
    mover_id = np.where(bid >= 0, arr["mover"][np.maximum(bid, 0)], -1).astype(np.int16)
    mover_id[sem != SEM["mover"]] = -1
    return dict(rgb=rgb8, depth=t.astype(np.float32), sem=sem, mover_id=mover_id, box_id=bid.astype(np.int32),
                mover_shadow=shadow_mover.reshape(H, W))

print("Ray caster ready: cast_rays, shade_rays, render_frame.")

In [ ]:
# Cell No: 017
# 3.1.4 Clip sampler: cameras on a straight or curved path, and movers with an exactly
#       controlled parallel fraction s and exact epipolar offset eps (frames 0 -> 1).


def scene_to_json(scene):
    return {k: v for k, v in scene.items() if not k.startswith("_")}

def make_design_deck(n_items, seed, classes=None, eps_targets=None, p_beyond=None):
    # exact proportions (rounded), shuffled with a fixed seed
    rng = np.random.default_rng(seed)
    classes = classes or MOVER_DESIGN["classes"]
    eps_targets = eps_targets or MOVER_DESIGN["eps_targets_px"]
    p_beyond = MOVER_DESIGN["p_beyond"] if p_beyond is None else p_beyond
    def _alloc(d):
        keys = list(d); w = np.array([d[k] for k in keys], np.float64); w /= w.sum()
        cnt = np.floor(w * n_items).astype(int)
        for j in np.argsort(-(w * n_items - cnt))[: n_items - cnt.sum()]:
            cnt[j] += 1
        out = [k for k, c in zip(keys, cnt) for _ in range(c)]
        rng.shuffle(out)
        return out
    cls = _alloc(classes); eps = _alloc(eps_targets)
    n_b = int(round(p_beyond * n_items))
    beyond = np.array([True] * n_b + [False] * (n_items - n_b)); rng.shuffle(beyond)
    return [dict(cls=c, eps=float(e), beyond=bool(b)) for c, e, b in zip(cls, eps, beyond)]

def _camera_params(rng, kind, street, n_frames, e4=False):
    cd = CAMERA_DESIGN[kind]
    psi = np.deg2rad(rng.uniform(*cd["yaw_deg"])) * (1 if kind == "forward" else rng.choice([-1, 1]))
    lanes = lane_centres(street["road_half"])
    jit = np.deg2rad(CAMERA_DESIGN["rot_jitter_deg"])
    return dict(kind=kind, psi=psi, pitch0=np.deg2rad(rng.uniform(-5.0, 1.0)),
                f=float(rng.uniform(*CAMERA_DESIGN["focal_px"])),
                mag=float(rng.uniform(*cd["baseline_m"])) * (0.75 if e4 else 1.0),
                eta=np.deg2rad(rng.uniform(-2, 2)),
                omega=np.deg2rad(rng.uniform(5.0, 8.0)) * rng.choice([-1, 1]),
                x_cam=float(rng.choice([-1.75, 1.75])) if kind == "forward"
                      else float(-np.sign(psi) * (lanes.max() if kind == "side" else 1.75)),
                jitter=rng.uniform(-jit, jit, size=(n_frames, 3)))

def _cams_from_params(p, n_frames, path):
    # Straight: constant translation b per frame. Curved: the translation direction turns by
    # omega per frame (frames 0 and 1 are identical in both cases). The per-frame rotations
    # (small independent jitter around a fixed heading) are identical for both paths, so a
    # straight/curved twin differs ONLY in the translation path (Proposition 2).
    omega = p["omega"] if path == "curved" else 0.0
    cams, C = [], np.array([p["x_cam"], 0.0, 0.0])
    for k in range(n_frames):
        R = rot_ypr(p["psi"], p["pitch0"], 0.0) if k == 0 else \
            rot_ypr(p["psi"] + p["jitter"][k, 0], p["pitch0"] + p["jitter"][k, 1], p["jitter"][k, 2])
        cams.append(dict(fx=p["f"], fy=p["f"], cx=319.5, cy=239.5, W=RUN_CONFIG["W"], H=RUN_CONFIG["H"],
                         R=R.tolist(), C=C.tolist()))
        heading = p["eta"] + k * omega
        C = C + p["mag"] * np.array([np.sin(heading), 0.0, np.cos(heading)])
    meta = dict(kind=p["kind"], psi_deg=float(np.rad2deg(p["psi"])), focal=p["f"], baseline=p["mag"],
                omega_deg=float(np.rad2deg(omega)), path=path)
    return cams, meta

def _object_boxes(cls, ref, dims, yaw, style, tint, mover_idx):
    if cls == "car":
        return car_boxes(ref, yaw, dims[0], dims[1], tint, kind="mover", mover=mover_idx)
    return crate_boxes(ref, dims, yaw, style, tint, kind="mover", mover=mover_idx)

def _project_boxes(boxes, offset, cam, R, C, znear=2.0):
    # image coordinates of all box corners; None if any corner is closer than znear (metres)
    pts = np.concatenate([box_corners(b, offset) for b in boxes])
    Xc = (pts - C) @ R
    if np.any(Xc[:, 2] < znear):
        return None
    return cam.project(Xc)

def _place_mover(rng, item, scene, cam_sets, mover_idx, placed_sets, n_frames):
    # cam_sets: one or more camera paths sharing frames 0 and 1; every constraint (inside
    # the image with a margin, no overlap with other movers, inside the street, clear of
    # furniture) must hold in every frame of every path.
    cams = cam_sets[0]
    st = scene["street"]; g = st["ground_y"]; wR = st["road_half"]; xf = st["facade_x_right"]
    cam0 = PinholeCamera.from_dict(cams[0]); R0 = np.asarray(cams[0]["R"]); C0 = np.asarray(cams[0]["C"])
    cam1 = PinholeCamera.from_dict(cams[1]); R1 = np.asarray(cams[1]["R"]); C1 = np.asarray(cams[1]["C"])
    b = C1 - C0
    furn = [bx for bx in scene["boxes"] if bx["kind"] == "static"]
    furn_c = np.array([bx["center"] for bx in furn]); furn_r = np.array([np.linalg.norm(bx["half"]) for bx in furn])
    W, H = cam0.W, cam0.H
    for attempt in range(300):
        cls = item["cls"]
        Zc = rng.uniform(*MOVER_DESIGN["depth_range_m"])
        u = rng.uniform(0.08 * W, 0.92 * W)
        a = (u - cam0.cx) / cam0.fx
        if cls == "crate_floating":
            hover = rng.uniform(1.5, 5.0)
            y_target = g - hover
        else:
            y_target = g
        bv = ((y_target - C0[1]) / Zc - R0[1, 0] * a - R0[1, 2]) / R0[1, 1]
        P = C0 + Zc * (R0 @ np.array([a, bv, 1.0]))
        if cls == "car":
            L, Wd = rng.uniform(4.1, 4.8), rng.uniform(1.70, 1.85)
            lanes = lane_centres(wR)
            lane = lanes[np.argmin(np.abs(lanes - P[0]))]
            if abs(lane - P[0]) > 2.5:
                continue
            P[0] = lane + rng.uniform(-0.3, 0.3)
            dims = (L, Wd); style = "car"; tint = CAR_COLORS[int(rng.integers(len(CAR_COLORS)))]
            ref = np.array([P[0], g, P[2]])
            centroid = ref + np.array([0, -0.8, 0])
            ext = np.array([Wd / 2, 0.85, L / 2])
        else:
            a_ = rng.uniform(0.9, 2.0)
            half = np.array([a_ / 2, a_ / 2 * rng.uniform(0.8, 1.0), a_ / 2 * rng.uniform(0.8, 1.2)])
            style = ["wood", "container", "pattern"][int(rng.integers(3))]
            tint = CONTAINER_TINTS[int(rng.integers(len(CONTAINER_TINTS)))] if style == "container" else (1, 1, 1)
            ref = np.array([P[0], (g - half[1]) if cls == "crate" else (y_target - half[1]), P[2]])
            centroid = ref.copy(); dims = half; ext = half
        s = rng.uniform(*(MOVER_DESIGN["s_range_beyond"] if item["beyond"] else MOVER_DESIGN["s_range_main"]))
        yaw = (0.0 if s >= 0 else np.pi) + np.deg2rad(rng.uniform(-3, 3)) if cls == "car" else rng.uniform(0, np.pi / 2)
        delta = s * b
        motion_type = "parallel"
        if item["eps"] > 0:
            if cls == "crate_floating":
                n_dir = np.cross(centroid - C0, b); n_dir /= np.linalg.norm(n_dir)
            else:
                n_dir = perp_unit(b, "horizontal")
            m, ach = solve_perp_for_eps(centroid, s * b, n_dir, item["eps"], cam0, R0, C0, cam1, R1, C1, max_m=4.0)
            if m is None or abs(m) > 3.5 or not np.isfinite(ach) or abs(ach - item["eps"]) > 1e-3:
                continue
            delta = s * b + m * n_dir
            motion_type = "perpendicular_component"
        boxes = _object_boxes(cls, ref, dims, yaw, style, tint, mover_idx)
        ok = True; bbs_sets = []
        for cset, placed_bboxes in zip(cam_sets, placed_sets):
            bbs = []
            for k in range(n_frames):
                off = k * delta
                cR = np.asarray(cset[k]["R"]); cC = np.asarray(cset[k]["C"]); ck = PinholeCamera.from_dict(cset[k])
                uv = _project_boxes(boxes, off, ck, cR, cC)
                if uv is None:
                    ok = False; break
                u0, v0 = uv.min(0); u1, v1 = uv.max(0)
                if u0 < 6 or v0 < 6 or u1 > W - 7 or v1 > H - 7:
                    ok = False; break
                if k == 0 and (u1 - u0) * (v1 - v0) < 900:
                    ok = False; break
                for other in placed_bboxes:
                    ou0, ov0, ou1, ov1 = other[k]
                    if not (u1 + 8 < ou0 or ou1 + 8 < u0 or v1 + 8 < ov0 or ov1 + 8 < v0):
                        ok = False; break
                if not ok:
                    break
                ck_c = centroid + off
                if abs(ck_c[0]) + ext.max() > xf - 0.3 or (cls == "car" and abs(ck_c[0]) > wR - 0.9):
                    ok = False; break
                if len(furn_c) and np.any(np.linalg.norm(furn_c - ck_c, axis=1) < furn_r + np.linalg.norm(ext) + 0.2):
                    ok = False; break
                bbs.append((u0, v0, u1, v1))
            if not ok:
                break
            bbs_sets.append(bbs)
        if not ok:
            continue
        # exact centre quantities between frames 0 and 1
        p0 = cam0.project(world_to_cam(centroid, R0, C0)); p1 = cam1.project(world_to_cam(centroid + delta, R1, C1))
        R21, t21 = relative_pose(R0, C0, R1, C1)
        F = fundamental_matrix(cam0.K, cam1.K, R21, t21)
        eps_c = abs(float(epipolar_offset(p0, p1, F)))
        Zgeo, gap = triangulate_depth(p0, p1, cam0.K, cam1.K, R21, t21)
        par = float(translational_parallax(p0, p1, cam0.K, cam1.K, R21))
        Zc0 = float(world_to_cam(centroid, R0, C0)[2])
        mover = dict(idx=mover_idx, cls=cls, style=style, tint=[float(v) for v in tint], s=float(s),
                     eps_target=float(item["eps"]), eps_center=eps_c, beyond=bool(item["beyond"]),
                     motion_type=motion_type, delta=[float(v) for v in delta], centroid0=centroid.tolist(),
                     Z_center=Zc0, Zgeo_center=float(Zgeo), parallax_center=par,
                     dims=[float(v) for v in np.ravel(dims)], yaw=float(yaw),
                     bbox0=[float(v) for v in bbs_sets[0][0]])
        return mover, boxes, bbs_sets
    return None, None, None

def sample_clip(seed, n_frames=3, path="straight", kind=None, items=None, e4=False, return_failed=False):
    # path: "straight", "curved" or "twin" (returns a straight and a curved scene that share
    # the street, the movers and frames 0-1, and differ only in the later translation path)
    rng = np.random.default_rng(seed)
    if kind is None:
        kinds = ["forward", "oblique", "side"]
        kind = kinds[int(rng.choice(3, p=[CAMERA_DESIGN[k]["p"] for k in kinds]))]
    street = sample_street(rng, kind)
    furn_boxes, furn_shadows = sample_static_furniture(rng, street)
    scene = dict(seed=int(seed), street=street, boxes=furn_boxes, shadows=furn_shadows, movers=[],
                 light=dict(exposure=float(rng.uniform(0.92, 1.08)),
                            white_balance=[float(v) for v in rng.uniform(0.97, 1.03, 3)],
                            noise_sigma=1.2 / 255))
    params = _camera_params(rng, kind, street, n_frames, e4=e4)
    paths = ["straight", "curved"] if path == "twin" else [path]
    cam_sets, metas = zip(*[_cams_from_params(params, n_frames, p) for p in paths])
    scene["cams"] = cam_sets[0]; scene["camera"] = metas[0]
    placed_sets, failed = [[] for _ in paths], []
    items = items if items is not None else [dict(cls="car", eps=0.0, beyond=False)]
    for item in items:
        mv, boxes, bbs_sets = _place_mover(rng, item, scene, list(cam_sets), len(scene["movers"]), placed_sets, n_frames)
        if mv is None:
            failed.append(item)
            continue
        mv["box_ids"] = list(range(len(scene["boxes"]), len(scene["boxes"]) + len(boxes)))
        scene["boxes"] += boxes
        if mv["cls"] == "car":
            c0 = mv["centroid0"]
            scene["shadows"].append(shadow_fp(c0[0], c0[2], mv["dims"][1] / 2, mv["dims"][0] / 2, mv["yaw"], 0.60, mv["idx"]))
        elif mv["cls"] == "crate":
            c0 = mv["centroid0"]
            scene["shadows"].append(shadow_fp(c0[0], c0[2], mv["dims"][0], mv["dims"][2], mv["yaw"], 0.50, mv["idx"]))
        scene["movers"].append(mv)
        for ps, bb in zip(placed_sets, bbs_sets):
            ps.append(bb)
    scene.pop("_arr", None)
    if path == "twin":
        out = []
        for cams, meta in zip(cam_sets, metas):
            sc = json.loads(json.dumps(scene_to_json(scene)))
            sc["cams"], sc["camera"] = cams, meta
            out.append(sc)
        return (out, failed) if return_failed else out
    return (scene, failed) if return_failed else scene

def sample_clip_sequence(seeds, decks, n_frames, path, e4, max_movers=4):
    # Sequential sampling with carry-over: an item that cannot be placed in clip i is offered
    # again in clip i+1, so the realised design matches the deck as closely as possible.
    scenes, carry, unplaced = [], [], []
    for seed, deck_items in zip(seeds, decks):
        items = (carry + list(deck_items))[:max_movers]
        overflow = (carry + list(deck_items))[max_movers:]
        sc, failed = sample_clip(seed, n_frames=n_frames, path=path, items=items, e4=e4, return_failed=True)
        carry = failed + overflow
        scenes.append(sc)
    unplaced = carry
    return scenes, unplaced

print("Clip sampler ready: make_design_deck, sample_clip.")

In [ ]:
# Cell No: 018
# 3.1.7 E5 causal ablation of ground contact. A base scene with three purely parallel movers is
#       rendered in five conditions that change ONE thing about the movers and nothing else:
#         G   grounded, with its soft contact shadow (the CONVOY-3D default)
#         NS  grounded, no contact shadow
#         CH  grounded and shadowed, behind a static wall that hides its ground contact in both views
#         F   lifted off the ground by 0.8-1.6 m (a floating car or crate), no shadow
#         FH  lifted, behind the same wall as in CH (control for the wall itself)
#       The motion (s, delta = s b, eps = 0), the camera, the street and every static object are
#       identical across conditions. The wall is designed by ray casting: it hides every visible
#       pixel of the mover up to the cover height (wheels of a car, lowest 35% of a crate) in both
#       frames; whether the contact really is hidden is then measured on the renders (Section 3.3).

def _rect_poly_xz(center, half_x, half_z, yaw):
    R = rot_y(yaw)
    return np.array([(np.asarray(center) + R @ np.array([sx * half_x, 0.0, sz * half_z]))[[0, 2]]
                     for sx, sz in ((-1, -1), (1, -1), (1, 1), (-1, 1))])

def _sat_overlap(a, b):
    # separating-axis test for two convex quadrilaterals in the ground plane
    for poly in (a, b):
        for i in range(4):
            e = poly[(i + 1) % 4] - poly[i]
            n = np.array([-e[1], e[0]])
            if np.linalg.norm(n) == 0:
                continue
            pa, pb = a @ n, b @ n
            if pa.max() < pb.min() or pb.max() < pa.min():
                return False
    return True

def _mover_footprints(scene, j, k):
    m = scene["movers"][j]
    off = np.asarray(m["delta"]) * k
    return [_rect_poly_xz(np.asarray(scene["boxes"][i]["center"]) + off, scene["boxes"][i]["half"][0],
                          scene["boxes"][i]["half"][2], scene["boxes"][i]["yaw"]) for i in m["box_ids"]]

def _mover_height(scene, j):
    bx = [scene["boxes"][i] for i in scene["movers"][j]["box_ids"]]
    top = min(b["center"][1] - b["half"][1] for b in bx); bot = max(b["center"][1] + b["half"][1] for b in bx)
    return bot - top

def _band_rays(scene, j, k, cover_m):
    # camera centre and world directions of the visible pixels of mover j (rendered alone) whose 3-D
    # point lies at most cover_m above the ground, in frame k
    cam, R, C = frame_camera(scene, k)
    m = scene["movers"][j]
    off = np.asarray(m["delta"]) * k
    uv = _project_boxes([scene["boxes"][i] for i in m["box_ids"]], off, cam, R, C, znear=0.05)
    if uv is None:
        return C, np.zeros((0, 3))
    u0, v0 = np.maximum(np.floor(uv.min(0)).astype(int) - 1, 0)
    u1, v1 = np.minimum(np.ceil(uv.max(0)).astype(int) + 1, [cam.W - 1, cam.H - 1])
    vv, uu = np.mgrid[v0:v1 + 1, u0:u1 + 1]
    D = cam.rays(uu.ravel().astype(np.float64), vv.ravel().astype(np.float64)) @ R.T
    h = cast_rays(scene, C, D, k, True, pix=(uu.ravel(), vv.ravel()), cam=cam, R=R, only_movers=[j])
    hit = np.isfinite(h["t"])
    P = C + h["t"][hit, None] * D[hit]
    above = scene["street"]["ground_y"] - P[:, 1]
    return C, D[hit][above <= cover_m]

def design_wall(scene, j, cover_m=None):
    # static wall hiding the ground contact of mover j (grounded geometry) in frames 0 and 1, or None. cover_m (v6.5,
    # contact dose-response) sets the hidden height directly; by default the E5 cover (wheels of a car, 35% of a crate)
    D_ = ABLATION_DESIGN
    m = scene["movers"][j]
    g = scene["street"]["ground_y"]
    if cover_m is not None:
        cover = float(cover_m)
    else:
        cover = D_["contact_cover_car_m"] if m["cls"] == "car" else D_["contact_cover_crate_frac"] * _mover_height(scene, j)
    C0 = np.asarray(scene["cams"][0]["C"]); C1 = np.asarray(scene["cams"][1]["C"])
    c0, c1 = C0[[0, 2]], C1[[0, 2]]
    fps = [np.concatenate(_mover_footprints(scene, j, k)) for k in (0, 1)]
    u = fps[0].mean(0) - c0; u /= np.linalg.norm(u); v = np.array([-u[1], u[0]])
    d_near = min(((f - c0) @ u).min() for f in fps)
    Dw = d_near - D_["wall_gap_m"] - D_["wall_thickness_m"]
    if Dw < 2.0 or (c1 - c0) @ u > Dw - 1.0:
        return None
    lat, hts = [], []
    for k in (0, 1):
        C, rays = _band_rays(scene, j, k, cover)
        if len(rays) == 0:
            return None
        den = rays[:, [0, 2]] @ u
        t = (Dw - (C[[0, 2]] - c0) @ u) / den
        if np.any(den <= 0) or np.any(t <= 0):
            return None
        X = C + t[:, None] * rays
        lat.append((X[:, [0, 2]] - c0) @ v)
        hts.append(g - X[:, 1])
    lat, hts = np.concatenate(lat), np.concatenate(hts)
    h_wall = float(hts.max() + 0.03)
    if h_wall > D_["wall_max_cover_frac"] * max(_mover_height(scene, j), 1.0) + cover or h_wall > 2.5:
        return None
    half_w = float((lat.max() - lat.min()) / 2 + 0.3); lat_c = float((lat.max() + lat.min()) / 2)
    cxz = c0 + u * (Dw + D_["wall_thickness_m"] / 2) + v * lat_c
    yaw = float(np.arctan2(-v[1], v[0]))
    poly = _rect_poly_xz((cxz[0], 0.0, cxz[1]), half_w, D_["wall_thickness_m"] / 2, yaw)
    # the wall may not touch any mover (either frame) or static object
    for jj in range(len(scene["movers"])):
        for k in (0, 1):
            if any(_sat_overlap(poly, f) for f in _mover_footprints(scene, jj, k)):
                return None
    for b in scene["boxes"]:
        if b["kind"] == "static" and _sat_overlap(poly, _rect_poly_xz(b["center"], b["half"][0], b["half"][2], b["yaw"])):
            return None
    return dict(mover=j, center=[float(cxz[0]), float(g - h_wall / 2), float(cxz[1])],
                half=[half_w, h_wall / 2, D_["wall_thickness_m"] / 2], yaw=yaw, height=h_wall, cover_m=float(cover))

def ablation_variant(base, cond, walls, lifts):
    # one condition of a base scene (a new scene dict; the base is not modified)
    sc = json.loads(json.dumps(scene_to_json(base)))
    sc["condition"] = cond
    if cond in ("NS", "F", "FH"):
        sc["shadows"] = [s_ for s_ in sc["shadows"] if s_["mover"] < 0]
    if cond in ("F", "FH"):
        for j, mv in enumerate(sc["movers"]):
            h = lifts[j]
            for i in mv["box_ids"]:
                sc["boxes"][i]["center"][1] -= h
            mv["centroid0"][1] -= h
            mv["lift"] = float(h)
    for mv in sc["movers"]:
        mv["support"] = "lifted" if cond in ("F", "FH") else "grounded"
    if cond in ("CH", "FH"):
        for w in walls:
            sc["boxes"].append(_box(w["center"], w["half"], w["yaw"], _mat("paving", (0.86, 0.86, 0.84)), part="wall"))
            sc["shadows"].append(shadow_fp(w["center"][0], w["center"][2], w["half"][0], w["half"][2] + 0.08, w["yaw"], 0.45))
    sc.pop("_arr", None)
    return sc

def contact_pixels(scene, render, k, j):
    # number of visible pixels of mover j within PREREG['contact_band_m'] of the ground (frame k)
    cam, R, C = frame_camera(scene, k)
    m = render["mover_id"] == j
    if not m.any():
        return 0
    vv, uu = np.nonzero(m)
    D = cam.rays(uu.astype(np.float64), vv.astype(np.float64)) @ R.T
    P = C + render["depth"][vv, uu].astype(np.float64)[:, None] * D
    return int(np.sum(scene["street"]["ground_y"] - P[:, 1] <= PREREG["contact_band_m"]))

print("Ablation builders ready: design_wall, ablation_variant, contact_pixels.")

In [ ]:
# Cell No: 019
# 3.1.5 Exact ground truth from a render: back-projection, correspondences between frames,
#       per-pixel epipolar offset and static-hypothesis depth, and per-mover statistics.

def frame_camera(scene, k):
    cd = scene["cams"][k]
    return PinholeCamera.from_dict(cd), np.asarray(cd["R"], np.float64), np.asarray(cd["C"], np.float64)

def backproject(depth, cam, R, C):
    H, W = depth.shape
    vv, uu = np.mgrid[0:H, 0:W]
    D = cam.rays(uu.astype(np.float64), vv.astype(np.float64)) @ R.T
    with np.errstate(invalid="ignore"):
        return C + np.where(np.isfinite(depth), depth, np.nan)[..., None].astype(np.float64) * D

def gt_correspondence(scene, rA, a, b, rB=None, tol_rel=0.01, tol_abs=0.02):
    # For every pixel of frame a: its exact position in frame b (static points stay fixed,
    # mover points translate by (b-a) * delta). 'visible' also checks occlusion in frame b.
    camA, RA, CA = frame_camera(scene, a)
    camB, RB, CB = frame_camera(scene, b)
    P = backproject(rA["depth"], camA, RA, CA)
    mid = rA["mover_id"].astype(np.int64)
    if scene["movers"]:
        deltas = np.array([m["delta"] for m in scene["movers"]], np.float64)
        P = P + np.where(mid[..., None] >= 0, deltas[np.maximum(mid, 0)] * (b - a), 0.0)
    Xc = (P - CB) @ RB
    with np.errstate(invalid="ignore", divide="ignore"):
        pB = camB.project(Xc)
    zB = Xc[..., 2]
    valid = np.isfinite(pB).all(-1) & (zB > 0.1)
    valid &= (pB[..., 0] >= 0) & (pB[..., 0] <= camB.W - 1) & (pB[..., 1] >= 0) & (pB[..., 1] <= camB.H - 1)
    visible = valid.copy()
    if rB is not None:
        ui = np.clip(np.round(np.nan_to_num(pB[..., 0])).astype(int), 0, camB.W - 1)
        vi = np.clip(np.round(np.nan_to_num(pB[..., 1])).astype(int), 0, camB.H - 1)
        zb_obs = rB["depth"][vi, ui]
        visible &= np.abs(zb_obs - zB) <= tol_rel * zB + tol_abs
        visible &= rB["mover_id"][vi, ui] == rA["mover_id"]
    return pB, zB, valid, visible

def pixel_epipolar_quantities(scene, a, b, pA_grid, pB):
    # per-pixel exact epipolar offset, static-hypothesis depth, translational parallax
    camA, RA, CA = frame_camera(scene, a)
    camB, RB, CB = frame_camera(scene, b)
    R21, t21 = relative_pose(RA, CA, RB, CB)
    F = fundamental_matrix(camA.K, camB.K, R21, t21)
    eps = np.abs(epipolar_offset(pA_grid, pB, F))
    zgeo, _ = triangulate_depth(pA_grid, pB, camA.K, camB.K, R21, t21)
    par = translational_parallax(pA_grid, pB, camA.K, camB.K, R21)
    return eps, zgeo, par

def erode_mask(mask, px):
    if px <= 0:
        return mask
    k = np.ones((2 * px + 1, 2 * px + 1), np.uint8)
    return cv2.erode(mask.astype(np.uint8), k, borderValue=0).astype(bool)

def mover_silhouette_area(scene, frame_k, j):
    # pixel count of mover j rendered alone (no occluders) -> visible fraction
    cam, R, C = frame_camera(scene, frame_k)
    m = scene["movers"][j]
    off = np.asarray(m["delta"]) * frame_k
    boxes = [scene["boxes"][i] for i in m["box_ids"]]
    uv = _project_boxes(boxes, off, cam, R, C, znear=0.05)
    if uv is None:
        return 0
    u0, v0 = np.maximum(np.floor(uv.min(0)).astype(int) - 1, 0)
    u1, v1 = np.minimum(np.ceil(uv.max(0)).astype(int) + 1, [cam.W - 1, cam.H - 1])
    vv, uu = np.mgrid[v0:v1 + 1, u0:u1 + 1]
    D = cam.rays(uu.ravel().astype(np.float64), vv.ravel().astype(np.float64)) @ R.T
    h = cast_rays(scene, C, D, frame_k, True, pix=(uu.ravel(), vv.ravel()), cam=cam, R=R, only_movers=[j])
    return int(np.isfinite(h["t"]).sum())

def mover_gt_stats(scene, renders, a=0, b=1):
    # Per-mover ground truth between frames a and b, measured on the mover's visible,
    # eroded pixels in frame a (the pixels every model-based statistic is read from).
    rA, rB = renders[a], renders[b]
    camA, _, _ = frame_camera(scene, a)
    H, W = rA["depth"].shape
    vv, uu = np.mgrid[0:H, 0:W]
    pA = np.stack([uu, vv], -1).astype(np.float64)
    pB, zB, valid, visible = gt_correspondence(scene, rA, a, b, rB)
    eps, zgeo, par = pixel_epipolar_quantities(scene, a, b, pA, pB)
    out = []
    for j, m in enumerate(scene["movers"]):
        mask = rA["mover_id"] == j
        area = int(mask.sum())
        sil = mover_silhouette_area(scene, a, j)
        em = erode_mask(mask, PREREG["erosion_px"]) & visible
        n = int(em.sum())
        rec = dict(mover=j, area_px=area, visible_frac=float(area / sil) if sil else 0.0, n_eroded=n)
        if n > 0:
            Zt = rA["depth"][em].astype(np.float64)
            rec.update(Z_true=float(np.median(Zt)), eps_med=float(np.median(eps[em])),
                       eps_max=float(np.max(eps[em])), Zgeo_med=float(np.median(zgeo[em])),
                       x_geo=float(np.median(np.log(np.where(zgeo[em] > 0, zgeo[em], np.nan) / Zt)))
                       if np.all(zgeo[em] > 0) else float("nan"),
                       parallax_med=float(np.median(par[em])),
                       vis_in_b=float(visible[mask].mean()) if area else 0.0)
        out.append(rec)
    return out

print("Ground-truth utilities ready.")

In [ ]:
# Cell No: 020
# 3.1.6 LEGACY-FLAT control: the compositing testbed of the previous notebook version,
#       reproduced exactly (smooth background pasted identically into both views, seven
#       static marker discs at their true projections, one 2-D object sprite). It is kept
#       only as a negative control: the static-scene validity gate (Section 6.0) must reject it.
from PIL import Image, ImageDraw

LEGACY_CAM = PinholeCamera(700.0, 700.0, 320.0, 240.0, 640, 480)

def _legacy_background(H, W, rng):
    lo = rng.integers(2, 6)
    base = rng.uniform(40, 200, size=(lo, lo, 3)).astype(np.float32)
    img = Image.fromarray(base.astype(np.uint8)).resize((W, H), Image.BICUBIC)
    arr = np.asarray(img).astype(np.float32) + rng.normal(0, 2, size=(H, W, 3))
    return np.clip(arr, 0, 255).astype(np.uint8)

def _legacy_anchor(draw, uv, color, marker, r=6):
    c = tuple(int(v) for v in color); x, y = uv
    if marker == "dot":
        draw.ellipse([x - r, y - r, x + r, y + r], fill=c)
    elif marker == "cross":
        draw.line([x - r, y, x + r, y], fill=c, width=3); draw.line([x, y - r, x, y + r], fill=c, width=3)
    else:
        draw.ellipse([x - r, y - r, x + r, y + r], outline=c, width=3)

def render_legacy_flat(seed):
    rng = np.random.default_rng(seed)
    cam = LEGACY_CAM
    yaw = rng.uniform(-0.03, 0.03)
    R1 = rot_y(yaw); C1 = np.array([0.0, 0.0, rng.uniform(0.05, 0.30)])
    bg = _legacy_background(cam.H, cam.W, rng)
    im0, im1 = Image.fromarray(bg.copy()), Image.fromarray(bg.copy())
    d0, d1 = ImageDraw.Draw(im0), ImageDraw.Draw(im1)
    anchors = []
    for _ in range(7):
        X = np.array([rng.uniform(-18, 18) * rng.uniform(0.3, 1.0), rng.uniform(-4, 4), rng.uniform(4, 90)])
        col = rng.uniform(30, 230, 3); mk = rng.choice(["dot", "cross", "ring"])
        uv0 = cam.project(X); uv1 = cam.project(world_to_cam(X, R1, C1))
        in0 = 6 <= uv0[0] < cam.W - 6 and 6 <= uv0[1] < cam.H - 6
        in1 = 6 <= uv1[0] < cam.W - 6 and 6 <= uv1[1] < cam.H - 6
        if in0:
            _legacy_anchor(d0, uv0, col, mk)
        if in1:
            _legacy_anchor(d1, uv1, col, mk)
        if in0 and in1:
            anchors.append(dict(u=float(uv0[0]), v=float(uv0[1]), Z=float(X[2])))
    # one 2-D sprite moving parallel to the baseline (as in the old testbed)
    Zo = rng.uniform(6, 55); s = rng.uniform(-0.5, 0.9)
    Xo = np.array([rng.uniform(-0.3, 0.3) * Zo, rng.uniform(-0.1, 0.25) * Zo, Zo])
    for im, (R, C, off) in ((d0, (np.eye(3), np.zeros(3), 0.0)), (d1, (R1, C1, 1.0))):
        uv = cam.project(world_to_cam(Xo + off * s * (C1 - 0), R, C))
        half = max(4.0, cam.fx * 0.9 / Zo)
        im.rounded_rectangle([uv[0] - half, uv[1] - 0.62 * half, uv[0] + half, uv[1] + 0.62 * half],
                             radius=0.15 * half, fill=tuple(int(v) for v in rng.uniform(60, 220, 3)))
    return dict(rgb=[np.asarray(im0), np.asarray(im1)], anchors=anchors,
                cams=[dict(**cam.to_dict(), R=np.eye(3).tolist(), C=[0.0, 0.0, 0.0]),
                      dict(**cam.to_dict(), R=R1.tolist(), C=C1.tolist())])

print("LEGACY-FLAT generator ready (reproduces the previous testbed for the gate control).")

In [ ]:
# Cell No: 021
# 3.2 Build the CONVOY-3D manifests and render everything. Each set is one checkpointed stage
#     (Section 0.5): it is restored from a previous run when its fingerprint matches, otherwise
#     sampled and rendered here and archived the moment it is complete.
#     MAIN      : the 240 v5 clips (specifications identical to v5); frames 0 and 1 are rendered
#     KNOCKOUT  : frame 1 of the first n MAIN clips re-rendered with every mover removed (A1)
#     E4        : 8-frame clips, straight vs curved camera path, purely parallel movers
#     FLAT      : LEGACY-FLAT control pairs
#     ABL       : E5 causal ablation, 72 base scenes x 5 conditions (Section 3.1.7)
#     Memory is released and reported after every set (renders are large, transient numpy arrays).
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

def _save_frame(dirpath, k, r, tag=""):
    Image.fromarray(r["rgb"]).save(os.path.join(dirpath, f"frame{k}{tag}.png"))
    np.savez_compressed(os.path.join(dirpath, f"gt{k}{tag}.npz"), depth=r["depth"], sem=r["sem"],
                        mover_id=r["mover_id"].astype(np.int8), mover_shadow=r["mover_shadow"])

def load_frame(dirpath, k, tag=""):
    rgb = np.asarray(Image.open(os.path.join(dirpath, f"frame{k}{tag}.png")).convert("RGB"))
    z = np.load(os.path.join(dirpath, f"gt{k}{tag}.npz"))
    return dict(rgb=rgb, depth=z["depth"], sem=z["sem"], mover_id=z["mover_id"].astype(np.int16),
                mover_shadow=z["mover_shadow"])

def load_rgb(dirpath, k, tag=""):
    return np.asarray(Image.open(os.path.join(dirpath, f"frame{k}{tag}.png")).convert("RGB"))

def _specs(set_name, seeds, decks, n_frames, path, e4):
    # sample (or reload) the scene specifications of one set (deterministic)
    spec_file = os.path.join(RENDER_DIR, f"specs_{set_name}_{path}.json")
    if os.path.exists(spec_file):
        with open(spec_file) as f:
            blob = json.load(f)
        if blob.get("n") == len(seeds):
            return blob["scenes"], blob["unplaced"]
    scenes, unplaced = sample_clip_sequence(seeds, decks, n_frames, path, e4)
    scenes = [[scene_to_json(t) for t in s] if isinstance(s, list) else scene_to_json(s) for s in scenes]
    with open(spec_file, "w") as f:
        json.dump({"n": len(seeds), "scenes": scenes, "unplaced": unplaced}, f)
    return scenes, unplaced

def _render_job(job):
    d = job["dir"]; meta_path = os.path.join(d, "scene.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if meta.get("complete"):
            return meta
    os.makedirs(d, exist_ok=True)
    scene = json.loads(json.dumps(job["scene"]))
    frames = job.get("frames") or list(range(len(scene["cams"])))
    renders = {k: render_frame(scene, k) for k in frames}
    for k, r in renders.items():
        _save_frame(d, k, r)
    if job.get("knockout"):
        _save_frame(d, 1, render_frame(scene, 1, include_movers=False), tag="_ko")
    gt01 = mover_gt_stats(scene, renders, 0, 1)
    last = max(frames)
    gtlast = mover_gt_stats(scene, renders, 0, last) if last > 1 else gt01
    meta = dict(scene=scene_to_json(scene), gt01=gt01, gt0last=gtlast, set=job["set"], index=job["index"],
                frames=frames, dir=d, complete=True)
    if job.get("eval_frame") is not None:
        m = job["eval_frame"]
        meta["eval_frame"] = m
        meta["gt_eval"] = mover_gt_stats(scene, renders, m, m + 1)
    if scene.get("condition"):
        meta["condition"] = scene["condition"]
        meta["contact_px"] = [[contact_pixels(scene, renders[k], k, j) for k in (0, 1)]
                              for j in range(len(scene["movers"]))]
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    return meta

def _render_flat_job(job):
    d = job["dir"]; meta_path = os.path.join(d, "scene.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            return json.load(f)
    os.makedirs(d, exist_ok=True)
    out = render_legacy_flat(job["seed"])
    for k in range(2):
        Image.fromarray(out["rgb"][k]).save(os.path.join(d, f"frame{k}.png"))
    meta = dict(anchors=out["anchors"], cams=out["cams"], set="FLAT", index=job["index"], dir=d, complete=True)
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    return meta

def _abl_job(job):
    # one condition of one ablation base scene
    base = job["scene"]
    walls = []
    if job["cond"] in ("CH", "FH"):
        walls = [w for w in (design_wall(base, j) for j in range(len(base["movers"]))) if w is not None]
    lifts = [float(np.random.default_rng(seed_for("abl-lift", job["index"], j)).uniform(*ABLATION_DESIGN["lift_m"]))
             for j in range(len(base["movers"]))]
    sc = ablation_variant(base, job["cond"], walls, lifts)
    sc["walls"] = walls
    return _render_job(dict(set="ABL", index=job["index"], scene=sc, frames=[0, 1], dir=job["dir"]))

def _load_metas(root):
    out = []
    for d in sorted(glob.glob(os.path.join(root, "*", "scene.json"))):
        with open(d) as f:
            m = json.load(f)
        m["dir"] = os.path.dirname(d)                # restored checkpoints may come from another path
        out.append(m)
    return out

def _run_jobs(fn, jobs):
    with ThreadPoolExecutor(max_workers=RUN_CONFIG["render_workers"]) as ex:
        return list(ex.map(fn, jobs))

N_MAIN = RUN_CONFIG["n_scenes_main"]; M_PER = RUN_CONFIG["movers_per_scene"]
N_E4 = RUN_CONFIG["n_clips_e4_per_path"]; K_E4 = RUN_CONFIG["e4_n_frames"]
N_ABL = RUN_CONFIG["n_scenes_ablation"]
_base_cfg = dict(W=RUN_CONFIG["W"], H=RUN_CONFIG["H"], movers_per_scene=M_PER, mover=MOVER_DESIGN, camera=CAMERA_DESIGN)
RENDER_STAGES = {
    "MAIN": Stage("render:MAIN", "render", dict(_base_cfg, n=N_MAIN, ko=RUN_CONFIG["n_scenes_knockout"], frames=[0, 1])),
    "E4": Stage("render:E4", "render", dict(_base_cfg, n=N_E4, k=K_E4)),
    "FLAT": Stage("render:FLAT", "render", dict(n=RUN_CONFIG["n_scenes_flat"])),
    "ABL": Stage("render:ABL", "render", dict(_base_cfg, n=N_ABL, design=ABLATION_DESIGN, band=PREREG["contact_band_m"])),
}
RENDER_FP = {k: v.fp for k, v in RENDER_STAGES.items()}
_ = _scene_box_arrays(dict(boxes=[]))                    # initialise the texture-id table
_render_log = {}

with stage("render"):
    st = RENDER_STAGES["MAIN"]
    if not st.done():
        deck = make_design_deck(N_MAIN * M_PER, seed_for("deck-main"))
        specs, unplaced = _specs("MAIN", [seed_for("main", i) for i in range(N_MAIN)],
                                 [deck[i * M_PER:(i + 1) * M_PER] for i in range(N_MAIN)], 3, "straight", False)
        _run_jobs(_render_job, [dict(set="MAIN", index=i, scene=specs[i], frames=[0, 1],
                                     knockout=i < RUN_CONFIG["n_scenes_knockout"],
                                     dir=os.path.join(RENDER_DIR, "main", f"{i:04d}")) for i in range(N_MAIN)])
        st.commit([os.path.join(RENDER_DIR, "main"), os.path.join(RENDER_DIR, "specs_MAIN_straight.json")],
                  extra=dict(unplaced=len(unplaced)))
    release_memory(); mem_report("render:MAIN")
    st = RENDER_STAGES["E4"]
    if not st.done():
        deck = make_design_deck(N_E4 * M_PER, seed_for("deck-e4"), eps_targets={0.0: 1.0}, p_beyond=0.0)
        # straight/curved TWINS: same street, movers and frames 0-1; only the later path differs
        specs, unplaced = _specs("E4", [seed_for("e4", i) for i in range(N_E4)],
                                 [deck[i * M_PER:(i + 1) * M_PER] for i in range(N_E4)], K_E4, "twin", True)
        _run_jobs(_render_job, [dict(set="E4", index=pi * N_E4 + i, scene=specs[i][pi],
                                     dir=os.path.join(RENDER_DIR, "e4", f"{path}_{i:04d}"))
                                for pi, path in enumerate(["straight", "curved"]) for i in range(N_E4)])
        st.commit([os.path.join(RENDER_DIR, "e4"), os.path.join(RENDER_DIR, "specs_E4_twin.json")],
                  extra=dict(unplaced=len(unplaced)))
    release_memory(); mem_report("render:E4")
    st = RENDER_STAGES["FLAT"]
    if not st.done():
        _run_jobs(_render_flat_job, [dict(set="FLAT", index=i, seed=seed_for("flat", i),
                                          dir=os.path.join(RENDER_DIR, "flat", f"{i:04d}"))
                                     for i in range(RUN_CONFIG["n_scenes_flat"])])
        st.commit([os.path.join(RENDER_DIR, "flat")])
    st = RENDER_STAGES["ABL"]
    if not st.done():
        deck = make_design_deck(N_ABL * M_PER, seed_for("deck-abl"), classes=ABLATION_DESIGN["classes"],
                                eps_targets={0.0: 1.0}, p_beyond=0.0)
        specs, unplaced = _specs("ABL", [seed_for("abl", i) for i in range(N_ABL)],
                                 [deck[i * M_PER:(i + 1) * M_PER] for i in range(N_ABL)], 2, "straight", False)
        _run_jobs(_abl_job, [dict(index=i, scene=specs[i], cond=c, dir=os.path.join(RENDER_DIR, "abl", f"{i:04d}_{c}"))
                             for i in range(N_ABL) for c in ABLATION_DESIGN["conditions"]])
        st.commit([os.path.join(RENDER_DIR, "abl"), os.path.join(RENDER_DIR, "specs_ABL_straight.json")],
                  extra=dict(unplaced=len(unplaced)))
    release_memory(); mem_report("render:ABL")

META_MAIN = sorted(_load_metas(os.path.join(RENDER_DIR, "main")), key=lambda m: m["index"])
META_E4 = sorted(_load_metas(os.path.join(RENDER_DIR, "e4")), key=lambda m: m["index"])
META_FLAT = sorted(_load_metas(os.path.join(RENDER_DIR, "flat")), key=lambda m: m["index"])
META_ABL = sorted(_load_metas(os.path.join(RENDER_DIR, "abl")),
                  key=lambda m: (m["index"], ABLATION_DESIGN["conditions"].index(m["condition"])))
for _k, _l in [("MAIN", META_MAIN), ("E4", META_E4), ("FLAT", META_FLAT), ("ABL", META_ABL)]:
    assert len(_l) > 0, f"no {_k} renders"

def _mover_table(metas):
    rows = []
    for meta in metas:
        sc = meta["scene"]
        gev = meta.get("gt_eval") or [{}] * len(sc["movers"])
        for j, (mv, g01, gl) in enumerate(zip(sc["movers"], meta["gt01"], meta["gt0last"])):
            r = {k: v for k, v in mv.items() if k not in ("delta", "centroid0", "dims", "tint", "box_ids", "bbox0")}
            r.update({f"gt_{k}": v for k, v in g01.items() if k != "mover"})
            r.update({f"gtlast_{k}": v for k, v in gl.items() if k != "mover"})
            r.update({f"gtev_{k}": v for k, v in gev[j].items() if k != "mover"})
            r.update(set=meta["set"], scene=meta["index"], scene_seed=sc["seed"], cam_kind=sc["camera"]["kind"],
                     path=sc["camera"]["path"], focal=sc["camera"]["focal"], baseline=sc["camera"]["baseline"],
                     cam_psi_deg=sc["camera"]["psi_deg"], omega_deg=sc["camera"]["omega_deg"],
                     uid=f"{meta['set']}-{meta['index']:04d}-{mv['idx']}")
            if meta.get("condition"):
                cp = meta["contact_px"][j]
                r.update(condition=meta["condition"], contact_px0=cp[0], contact_px1=cp[1],
                         has_wall=any(w["mover"] == j for w in sc.get("walls", [])))
            rows.append(r)
    return pd.DataFrame(rows)

def _regimes(df, prefix="gt_"):
    df["x_par"] = np.where(df["s"] < 1, -np.log(np.clip(1 - df["s"], 1e-9, None)), np.nan)
    df["regime"] = np.where(df[prefix + "eps_med"] < PREREG["eps_parallel_max"], "parallel",
                            np.where(df[prefix + "eps_med"] >= PREREG["eps_perp_min"], "perpendicular", "intermediate"))
    df["measurable"] = (df[prefix + "n_eroded"] >= PREREG["min_object_pixels"]) & (df[prefix + "visible_frac"] >= 0.85)
    df["identifiable"] = df[prefix + "parallax_med"] >= PREREG["parallax_min_px"]
    return df

MOVERS_MAIN = _regimes(_mover_table(META_MAIN))
MOVERS_E4 = _regimes(_mover_table(META_E4))
MOVERS_E4["twin"] = MOVERS_E4["scene"] % N_E4
MOVERS_ABL = _regimes(_mover_table(META_ABL))
for _n, _df in [("main", MOVERS_MAIN), ("e4", MOVERS_E4), ("ablation", MOVERS_ABL)]:
    _df.to_csv(os.path.join(MANIFEST_DIR, f"convoy3d_{_n}_movers.csv"), index=False)

print(stage_table("render:").to_string(index=False))
print(f"\nMAIN: {len(MOVERS_MAIN)} movers in {len(META_MAIN)} clips (design deck {N_MAIN * M_PER})")
print(MOVERS_MAIN.groupby(["cls", "regime"]).size().unstack(fill_value=0).to_string())
print("\nby camera kind:", MOVERS_MAIN.groupby("cam_kind").size().to_dict(),
      "| motion types:", MOVERS_MAIN.groupby("motion_type").size().to_dict(),
      "| s > 1 (no static solution):", int((MOVERS_MAIN["s"] > 1).sum()))
print(f"measurable (>= {PREREG['min_object_pixels']} eroded px, >= 85% visible): "
      f"{int(MOVERS_MAIN['measurable'].sum())}/{len(MOVERS_MAIN)};  identifiable "
      f"(parallax >= {PREREG['parallax_min_px']} px): {int(MOVERS_MAIN['identifiable'].sum())}/{len(MOVERS_MAIN)}")
print(f"E4: {len(MOVERS_E4)} movers ({MOVERS_E4.groupby('path').size().to_dict()})")
_abl_g = MOVERS_ABL[MOVERS_ABL["condition"] == "G"]
print(f"ABL: {len(_abl_g)} base movers ({_abl_g.groupby('cls').size().to_dict()}) in {N_ABL} scenes x "
      f"{len(ABLATION_DESIGN['conditions'])} conditions; walls designed for "
      f"{int(MOVERS_ABL[MOVERS_ABL['condition'] == 'CH']['has_wall'].sum())} of them")

In [ ]:
# Cell No: 022
# 3.2b Designed experiments of v6.5 (D55; H12 and H13 registered in AMENDMENT_V6_5). Two checkpointed render stages:
#      BSW  baseline sweep: 64 base scenes x 4 baselines (x0.5, x1, x2, x3); only the parallax changes
#      WDR  contact dose-response: 64 base scenes x 4 wall heights (none, 15, 35, 70 cm)
#      Both use the CONVOY-3D sampler, renderer and the E5 wall design; frames 0 and 1 are rendered.
N_BSW, N_WDR = RUN_CONFIG["n_scenes_bsw"], RUN_CONFIG["n_scenes_wdr"]
BSW_LEVELS, WDR_LEVELS = list(SWEEP_DESIGN["levels"]), list(WALL_DOSE_DESIGN["levels"])

def _center_quantities(sc, mv):
    # exact epipolar offset, static-hypothesis depth and translational parallax at the mover's centre (frames 0 -> 1)
    cam0, cam1 = PinholeCamera.from_dict(sc["cams"][0]), PinholeCamera.from_dict(sc["cams"][1])
    R0, C0 = np.asarray(sc["cams"][0]["R"]), np.asarray(sc["cams"][0]["C"])
    R1, C1 = np.asarray(sc["cams"][1]["R"]), np.asarray(sc["cams"][1]["C"])
    c, d = np.asarray(mv["centroid0"], float), np.asarray(mv["delta"], float)
    p0 = cam0.project(world_to_cam(c, R0, C0)); p1 = cam1.project(world_to_cam(c + d, R1, C1))
    R21, t21 = relative_pose(R0, C0, R1, C1)
    eps = abs(float(epipolar_offset(p0, p1, fundamental_matrix(cam0.K, cam1.K, R21, t21))))
    zgeo, _ = triangulate_depth(p0, p1, cam0.K, cam1.K, R21, t21)
    return eps, float(zgeo), float(translational_parallax(p0, p1, cam0.K, cam1.K, R21))

def sweep_variant(base, level):
    # the base scene seen with the baseline scaled by m: camera 2 keeps the rotation of frame 1 and moves along the same
    # line, C = C0 + m (C1 - C0), and every mover is displaced by m delta, so s, eps = 0 and x = -log(1-s) are unchanged
    m = float(SWEEP_DESIGN["levels"][level])
    sc = json.loads(json.dumps(scene_to_json(base)))
    c0, c1 = sc["cams"][0], dict(sc["cams"][1])
    C0, C1 = np.asarray(c0["C"], float), np.asarray(c1["C"], float)
    c1["C"] = (C0 + m * (C1 - C0)).tolist()
    sc["cams"] = [c0, c1]
    sc["camera"] = dict(sc["camera"], baseline=float(sc["camera"]["baseline"] * m), level=level, factor=m)
    for mv in sc["movers"]:
        mv["delta"] = [float(m * v) for v in mv["delta"]]
        mv["eps_center"], mv["Zgeo_center"], mv["parallax_center"] = _center_quantities(sc, mv)
    sc["condition"] = level
    return sc

def _wdr_job(job):
    # one wall height of one base scene: no wall (G) or static walls hiding the lowest cover_m of every mover
    base, level = job["scene"], job["level"]
    cover = float(WALL_DOSE_DESIGN["levels"][level])
    walls = []
    if cover > 0:
        walls = [w for w in (design_wall(base, j, cover_m=cover) for j in range(len(base["movers"]))) if w is not None]
    sc = ablation_variant(base, "CH" if cover > 0 else "G", walls, [0.0] * len(base["movers"]))
    sc["walls"], sc["condition"] = walls, level
    return _render_job(dict(set="WDR", index=job["index"], scene=sc, frames=[0, 1], dir=job["dir"]))

DESIGN_STAGES = {
    "BSW": Stage("render:BSW", "render65", dict(_base_cfg, n=N_BSW, design=_jsonable(SWEEP_DESIGN))),
    "WDR": Stage("render:WDR", "render65", dict(_base_cfg, n=N_WDR, design=_jsonable(WALL_DOSE_DESIGN), wall=ABLATION_DESIGN,
                                                band=PREREG["contact_band_m"])),
}
RENDER_FP.update({k: v.fp for k, v in DESIGN_STAGES.items()})

with stage("render_designed"):
    st = DESIGN_STAGES["BSW"]
    if not st.done():
        deck = make_design_deck(N_BSW * M_PER, seed_for("deck-bsw"), classes=SWEEP_DESIGN["classes"],
                                eps_targets={0.0: 1.0}, p_beyond=0.0)
        specs, unplaced = _specs("BSW", [seed_for("bsw", i) for i in range(N_BSW)],
                                 [deck[i * M_PER:(i + 1) * M_PER] for i in range(N_BSW)],
                                 SWEEP_DESIGN["n_frames_sampled"], "straight", False)
        _run_jobs(_render_job, [dict(set="BSW", index=i, scene=sweep_variant(specs[i], lv), frames=[0, 1],
                                     dir=os.path.join(RENDER_DIR, "bsw", f"{i:04d}_{lv}"))
                                for i in range(N_BSW) for lv in BSW_LEVELS])
        st.commit([os.path.join(RENDER_DIR, "bsw"), os.path.join(RENDER_DIR, "specs_BSW_straight.json")],
                  extra=dict(unplaced=len(unplaced)))
    release_memory(); mem_report("render:BSW")
    st = DESIGN_STAGES["WDR"]
    if not st.done():
        deck = make_design_deck(N_WDR * M_PER, seed_for("deck-wdr"), classes=WALL_DOSE_DESIGN["classes"],
                                eps_targets={0.0: 1.0}, p_beyond=0.0)
        specs, unplaced = _specs("WDR", [seed_for("wdr", i) for i in range(N_WDR)],
                                 [deck[i * M_PER:(i + 1) * M_PER] for i in range(N_WDR)], 2, "straight", False)
        _run_jobs(_wdr_job, [dict(index=i, scene=specs[i], level=lv, dir=os.path.join(RENDER_DIR, "wdr", f"{i:04d}_{lv}"))
                             for i in range(N_WDR) for lv in WDR_LEVELS])
        st.commit([os.path.join(RENDER_DIR, "wdr"), os.path.join(RENDER_DIR, "specs_WDR_straight.json")],
                  extra=dict(unplaced=len(unplaced)))
    release_memory(); mem_report("render:WDR")

META_BSW = sorted(_load_metas(os.path.join(RENDER_DIR, "bsw")), key=lambda m: (m["index"], BSW_LEVELS.index(m["condition"])))
META_WDR = sorted(_load_metas(os.path.join(RENDER_DIR, "wdr")), key=lambda m: (m["index"], WDR_LEVELS.index(m["condition"])))
for _k, _l in [("BSW", META_BSW), ("WDR", META_WDR)]:
    assert len(_l) > 0, f"no {_k} renders"
MOVERS_BSW = _regimes(_mover_table(META_BSW))
MOVERS_BSW["factor"] = MOVERS_BSW["condition"].map(SWEEP_DESIGN["levels"]).astype(float)
MOVERS_WDR = _regimes(_mover_table(META_WDR))
MOVERS_WDR["cover_m"] = MOVERS_WDR["condition"].map(WALL_DOSE_DESIGN["levels"]).astype(float)
for _n, _df in [("baseline_sweep", MOVERS_BSW), ("dose_response", MOVERS_WDR)]:
    _df.to_csv(os.path.join(MANIFEST_DIR, f"convoy3d_{_n}_movers.csv"), index=False)

# ---- self-tests of the designed sets ----------------------------------------------------------------------------
DESIGN_CHECKS = {}
_b = MOVERS_BSW[MOVERS_BSW["gt_n_eroded"] > 0]
DESIGN_CHECKS["bsw_n_base_movers"] = int(MOVERS_BSW["uid"].nunique())
DESIGN_CHECKS["bsw_s_identical_across_levels"] = bool((MOVERS_BSW.groupby("uid")["s"].nunique() == 1).all())
DESIGN_CHECKS["bsw_max_abs_xgeo_minus_xpar"] = float(np.nanmax(np.abs(_b["gt_x_geo"] - _b["x_par"]))) if len(_b) else np.nan
DESIGN_CHECKS["bsw_max_pixel_eps_px"] = float(np.nanmax(_b["gt_eps_max"])) if len(_b) else np.nan
_pm = MOVERS_BSW.pivot_table(index="uid", columns="condition", values="gt_parallax_med", aggfunc="first")
for _lv in BSW_LEVELS:
    if _lv in _pm and "B1" in _pm:
        _r = (_pm[_lv] / _pm["B1"]).replace([np.inf, -np.inf], np.nan)
        DESIGN_CHECKS[f"bsw_parallax_ratio_{_lv}_to_B1_median"] = float(np.nanmedian(_r))
_mm = MOVERS_BSW.pivot_table(index="uid", columns="condition", values="measurable", aggfunc="first").astype(float)
DESIGN_CHECKS["bsw_measurable_at_every_level"] = int((_mm[BSW_LEVELS].sum(1) == len(BSW_LEVELS)).sum()) if len(_mm) else 0
_w = MOVERS_WDR
DESIGN_CHECKS["wdr_n_base_movers"] = int(_w["uid"].nunique())
for _lv in WDR_LEVELS[1:]:
    _q = _w[_w["condition"] == _lv]
    DESIGN_CHECKS[f"wdr_{_lv}_wall_built"] = int(_q["has_wall"].astype(bool).sum())
    DESIGN_CHECKS[f"wdr_{_lv}_contact_hidden_both_views"] = int(((_q["contact_px0"] == 0) & (_q["contact_px1"] == 0)
                                                                 & _q["has_wall"].astype(bool)).sum())
_g = _w[_w["condition"] == "G"]
DESIGN_CHECKS["wdr_G_contact_visible_both_views"] = int(((_g["contact_px0"] >= PREREG["h13_contact_px_min"])
                                                         & (_g["contact_px1"] >= PREREG["h13_contact_px_min"])).sum())
# the dose as built: a wall must cover the whole visible base in both views, so it hides more than its nominal band at
# the mover's near end. Share of each mover's silhouette that is hidden (view 1) and the physical height of the walls.
for _lv in WDR_LEVELS:
    _q = _w[(_w["condition"] == _lv) & ((_w["has_wall"].astype(bool)) if _lv != "G" else True)]
    DESIGN_CHECKS[f"wdr_{_lv}_median_hidden_share"] = float(np.nanmedian(1.0 - _q["gt_visible_frac"])) if len(_q) else np.nan
    _hw = [w_["height"] for m_ in META_WDR if m_["condition"] == _lv for w_ in m_.get("scene", {}).get("walls", [])]
    DESIGN_CHECKS[f"wdr_{_lv}_median_wall_height_m"] = float(np.median(_hw)) if _hw else np.nan
TESTBED_CHECKS_V65 = DESIGN_CHECKS
with open(os.path.join(LOG_DIR, "design_checks.json"), "w") as f:
    json.dump(DESIGN_CHECKS, f, indent=2)
print(stage_table("render:").to_string(index=False))
print(f"\nBSW: {DESIGN_CHECKS['bsw_n_base_movers']} base movers in {N_BSW} scenes x {len(BSW_LEVELS)} baselines "
      f"({MOVERS_BSW[MOVERS_BSW['condition'] == 'B1'].groupby('cls').size().to_dict()}); s identical across levels: "
      f"{DESIGN_CHECKS['bsw_s_identical_across_levels']}; Proposition 1 on the renders: max |x_geo - x| = "
      f"{DESIGN_CHECKS['bsw_max_abs_xgeo_minus_xpar']:.1e}, max pixel |eps| = {DESIGN_CHECKS['bsw_max_pixel_eps_px']:.1e} px")
print("     median parallax relative to x1: " + ", ".join(f"{SWEEP_DESIGN['labels'][lv]} {DESIGN_CHECKS.get(f'bsw_parallax_ratio_{lv}_to_B1_median', np.nan):.2f}"
                                                        for lv in BSW_LEVELS)
      + f"; measurable at every baseline: {DESIGN_CHECKS['bsw_measurable_at_every_level']}")
print(f"WDR: {DESIGN_CHECKS['wdr_n_base_movers']} base movers in {N_WDR} scenes x {len(WDR_LEVELS)} wall heights; contact visible "
      f"without a wall (>= {PREREG['h13_contact_px_min']} px in both views): {DESIGN_CHECKS['wdr_G_contact_visible_both_views']}")
for _lv in WDR_LEVELS[1:]:
    print(f"     {WALL_DOSE_DESIGN['labels'][_lv]:>6} wall: built for {DESIGN_CHECKS[f'wdr_{_lv}_wall_built']}, contact hidden in both "
          f"views for {DESIGN_CHECKS[f'wdr_{_lv}_contact_hidden_both_views']}; median wall height "
          f"{DESIGN_CHECKS[f'wdr_{_lv}_median_wall_height_m']:.2f} m, median share of the mover hidden in view 1 "
          f"{DESIGN_CHECKS[f'wdr_{_lv}_median_hidden_share']:.2f} (no wall: {DESIGN_CHECKS['wdr_G_median_hidden_share']:.2f})")
assert DESIGN_CHECKS["bsw_s_identical_across_levels"]
assert not np.isfinite(DESIGN_CHECKS["bsw_max_abs_xgeo_minus_xpar"]) or DESIGN_CHECKS["bsw_max_abs_xgeo_minus_xpar"] < 1e-3

In [ ]:
# Cell No: 023
# 3.3 Testbed self-tests on the rendered images (exactness and photometric consistency).
TESTBED_CHECKS = {}

# (a) Proposition 1 on rendered pixels: for parallel movers the triangulated depth of every
#     visible pixel equals Z/(1-s); x_geo is the median log(Z_geo/Z) over the mover's pixels.
_par = MOVERS_MAIN[(MOVERS_MAIN["motion_type"] == "parallel") & (MOVERS_MAIN["s"] < 1)
                   & (MOVERS_MAIN["gt_n_eroded"] > 0)]
_dev = np.abs(_par["gt_x_geo"] - _par["x_par"])
TESTBED_CHECKS["prop1_rendered_max_abs_dev_log_depth"] = float(_dev.max())
TESTBED_CHECKS["prop1_rendered_n_movers"] = int(len(_par))
# parallel motion leaves EVERY pixel of an extended mover on its epipolar line
TESTBED_CHECKS["parallel_movers_max_pixel_eps_px"] = float(_par["gt_eps_max"].max())
# (b) achieved epipolar offset vs design target (exact solver at the mover centre)
TESTBED_CHECKS["eps_center_max_abs_error_px"] = float(np.abs(MOVERS_MAIN["eps_center"] - MOVERS_MAIN["eps_target"]).max())

# (c) static pixels: eps ~ 0 and photometric consistency of the exact warp, versus a
#     control warp in which movers are (wrongly) treated as static
_static_eps, _photo_static, _photo_mover, _photo_mover_wrong = [], [], [], []
for meta in META_MAIN[: min(12, len(META_MAIN))]:
    sc = meta["scene"]; d = os.path.join(RENDER_DIR, "main", f"{meta['index']:04d}")
    r0, r1 = load_frame(d, 0), load_frame(d, 1)
    H, W = r0["depth"].shape
    vv, uu = np.mgrid[0:H, 0:W]
    pA = np.stack([uu, vv], -1).astype(np.float64)
    pB, zB, valid, vis = gt_correspondence(sc, r0, 0, 1, r1)
    eps, _, _ = pixel_epipolar_quantities(sc, 0, 1, pA, pB)
    stat = vis & (r0["mover_id"] < 0) & np.isfinite(r0["depth"]) & ~r0["mover_shadow"] & ~r1["mover_shadow"]
    stat &= erode_mask(stat, 1)
    _static_eps.append(eps[stat])
    mapx, mapy = pB[..., 0].astype(np.float32), pB[..., 1].astype(np.float32)
    warped = cv2.remap(r1["rgb"].astype(np.float32), np.nan_to_num(mapx), np.nan_to_num(mapy), cv2.INTER_LINEAR)
    err = np.abs(warped - r0["rgb"].astype(np.float32)).mean(-1)
    _photo_static.append(err[stat])
    mv = erode_mask(r0["mover_id"] >= 0, 2) & vis
    _photo_mover.append(err[mv])
    # wrong hypothesis: movers static -> use the static warp for mover pixels
    P = backproject(r0["depth"], *frame_camera(sc, 0))
    cam1, R1, C1 = frame_camera(sc, 1)
    pW = cam1.project((P - C1) @ R1)
    w2 = cv2.remap(r1["rgb"].astype(np.float32), np.nan_to_num(pW[..., 0]).astype(np.float32),
                   np.nan_to_num(pW[..., 1]).astype(np.float32), cv2.INTER_LINEAR)
    _photo_mover_wrong.append(np.abs(w2 - r0["rgb"].astype(np.float32)).mean(-1)[mv])
_static_eps = np.concatenate(_static_eps)
TESTBED_CHECKS["static_pixels_eps_median_px"] = float(np.median(_static_eps))
TESTBED_CHECKS["static_pixels_eps_p999_px"] = float(np.percentile(_static_eps, 99.9))
TESTBED_CHECKS["photometric_err_static_exact_warp"] = float(np.mean(np.concatenate(_photo_static)))
TESTBED_CHECKS["photometric_err_movers_exact_warp"] = float(np.mean(np.concatenate(_photo_mover)))
TESTBED_CHECKS["photometric_err_movers_if_assumed_static"] = float(np.mean(np.concatenate(_photo_mover_wrong)))

# (d) E5 ablation: identical static content across conditions, ground contact visible when grounded
#     and hidden behind the wall in CH (measured on the renders, not assumed from the design)
_ab = MOVERS_ABL.pivot_table(index="uid", columns="condition", values=["contact_px0", "contact_px1"], aggfunc="first")
_vis_g = (_ab[("contact_px0", "G")] >= 5) & (_ab[("contact_px1", "G")] >= 5)
_walled = MOVERS_ABL[(MOVERS_ABL["condition"] == "CH") & MOVERS_ABL["has_wall"]]["uid"]
_hid_ch = (_ab[("contact_px0", "CH")] == 0) & (_ab[("contact_px1", "CH")] == 0)
TESTBED_CHECKS["abl_contact_visible_when_grounded"] = float(_vis_g.mean())
TESTBED_CHECKS["abl_walls_designed_frac"] = float(len(_walled) / max(1, len(_ab)))
TESTBED_CHECKS["abl_contact_hidden_given_wall"] = float(_hid_ch.loc[_walled].mean()) if len(_walled) else float("nan")
TESTBED_CHECKS["abl_lifted_contact_px_max"] = int(MOVERS_ABL[MOVERS_ABL["condition"].isin(["F", "FH"])][["contact_px0", "contact_px1"]].max().max())
_diff_n, _diff_bad = 0, 0
for meta_ns in [m for m in META_ABL if m["condition"] == "NS"][:8]:
    meta_f = next(m for m in META_ABL if m["index"] == meta_ns["index"] and m["condition"] == "F")
    for k in (0, 1):
        a, b = load_frame(meta_ns["dir"], k), load_frame(meta_f["dir"], k)
        # pixels more than 3 px from either mover (anti-aliasing samples may touch a mover's edge)
        away = ~cv2.dilate(((a["mover_id"] >= 0) | (b["mover_id"] >= 0)).astype(np.uint8), np.ones((7, 7), np.uint8)).astype(bool)
        _diff_n += int(away.sum())
        _diff_bad += int((np.abs(a["rgb"].astype(int) - b["rgb"].astype(int)).max(-1)[away] > 1).sum())
TESTBED_CHECKS["abl_share_of_pixels_away_from_movers_differing_NS_vs_F"] = float(_diff_bad / max(_diff_n, 1))

for k, v in TESTBED_CHECKS.items():
    print(f"{k:<52}{v:.4g}" if isinstance(v, float) else f"{k:<52}{v}")
assert TESTBED_CHECKS["prop1_rendered_max_abs_dev_log_depth"] < 1e-3
assert TESTBED_CHECKS["eps_center_max_abs_error_px"] < 1e-2
assert TESTBED_CHECKS["parallel_movers_max_pixel_eps_px"] < 1e-3
assert TESTBED_CHECKS["static_pixels_eps_p999_px"] < 0.05
assert TESTBED_CHECKS["photometric_err_static_exact_warp"] < 6.0
assert TESTBED_CHECKS["photometric_err_movers_if_assumed_static"] > 2 * TESTBED_CHECKS["photometric_err_movers_exact_warp"]
assert TESTBED_CHECKS["abl_lifted_contact_px_max"] == 0
assert TESTBED_CHECKS["abl_share_of_pixels_away_from_movers_differing_NS_vs_F"] < 1e-4
print("\nTestbed self-tests passed: rendered parallel movers sit exactly at Z/(1-s) under "
      "static triangulation, static pixels have zero epipolar offset, the exact warp "
      "reproduces the second view up to sensor noise and resampling, and the ablation conditions "
      "differ only at the movers (lifted movers never touch the ground).")
with open(os.path.join(LOG_DIR, "testbed_checks.json"), "w") as f:
    json.dump(TESTBED_CHECKS, f, indent=2)

release_memory(); _ = mem_report("CONVOY-3D renders checked")

In [ ]:
# Cell No: 024
# 3.4.1 TUM RGB-D: download and extract the four sequences (resumable, retried, size-checked).
#       The TUM data are needed only to build CONVOY-REAL and the walking ground truth; when both
#       of those stages are restored from a previous run's checkpoints, nothing is downloaded.
import urllib.request, tarfile

TUM_BASE = "https://cvg.cit.tum.de/rgbd/dataset/freiburg3/"
TUM_SEQUENCES = {
    "fr3_structure_texture_far":  dict(file="rgbd_dataset_freiburg3_structure_texture_far.tgz", role="static"),
    "fr3_long_office_household":  dict(file="rgbd_dataset_freiburg3_long_office_household.tgz", role="static"),
    "fr3_walking_xyz":            dict(file="rgbd_dataset_freiburg3_walking_xyz.tgz", role="dynamic"),
    "fr3_walking_halfsphere":     dict(file="rgbd_dataset_freiburg3_walking_halfsphere.tgz", role="dynamic"),
}
TUM_FR3_INTRINSICS = dict(fx=535.4, fy=539.2, cx=320.1, cy=247.6, W=640, H=480)
TUM_DEPTH_SCALE = 5000.0

def download_file(url, dest, retries=4, timeout=120):
    # streaming download with HTTP range resume; returns (ok, message)
    tmp = dest + ".part"
    msg = "not attempted"
    for attempt in range(1, retries + 1):
        try:
            have = os.path.getsize(tmp) if os.path.exists(tmp) else 0
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            if have:
                req.add_header("Range", f"bytes={have}-")
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                total = resp.headers.get("Content-Length")
                total = int(total) + have if total is not None else None
                if have and resp.status != 206:          # server ignored the range: restart
                    have = 0
                mode = "ab" if have else "wb"
                with open(tmp, mode) as out:
                    while True:
                        chunk = resp.read(1 << 22)
                        if not chunk:
                            break
                        out.write(chunk)
            size = os.path.getsize(tmp)
            if total is not None and size != total:
                raise IOError(f"incomplete: {size} of {total} bytes")
            os.replace(tmp, dest)
            return True, f"{size / 2**20:.0f} MiB"
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            time.sleep(min(30, 5 * attempt))
    return False, msg

# stages built from the TUM data (Sections 3.5 and 3.7); identity of the data = sequence names and URLs
TUM_STAGE_CFG = dict(base=TUM_BASE, sequences={k: v["file"] for k, v in TUM_SEQUENCES.items()},
                     intrinsics=TUM_FR3_INTRINSICS, depth_scale=TUM_DEPTH_SCALE)
TUM_STAGES = {
    "REAL": Stage("tum:REAL", "tum", dict(TUM_STAGE_CFG, n_pairs=RUN_CONFIG["n_real_pairs_per_seq"],
                                          movers=RUN_CONFIG["real_movers_per_pair"], s_range=MOVER_DESIGN["s_range_main"])),
    "DYN": Stage("tum:DYN", "tum", dict(TUM_STAGE_CFG, n_pairs=RUN_CONFIG["n_dyn_candidate_pairs_per_seq"],
                                        regimes=[PREREG["eps_parallel_max"], PREREG["eps_perp_min"]], flow=FLOW_PREFERRED)),
    "RABL": Stage("tum:RABL", "tum", dict(TUM_STAGE_CFG, n_pairs=RUN_CONFIG["n_rabl_pairs_per_seq"],
                                          movers=RUN_CONFIG["rabl_movers_per_pair"], design=_jsonable(RABL_DESIGN),
                                          s_range=MOVER_DESIGN["s_range_main"], band=PREREG["rabl_contact_band_m"])),
}
TUM_NEEDED = any(st.available() is None for st in TUM_STAGES.values())

DATASET_STATUS = {}
with stage("download_tum"):
    for name, spec in TUM_SEQUENCES.items():
        seq_dir = os.path.join(DATA_DIR, name)
        done_flag = os.path.join(seq_dir, ".extracted")
        t0 = time.time()
        if not TUM_NEEDED:
            DATASET_STATUS[name] = dict(role=spec["role"], downloaded=False, restored=True, n_rgb=0, n_depth=0,
                                        groundtruth=False, note="every stage built from it is restored from a checkpoint; "
                                        "not downloaded", seconds=0.0, path=seq_dir)
            continue
        if not os.path.exists(done_flag):
            tgz = os.path.join(DATA_DIR, spec["file"])
            ok, msg = (True, "cached") if os.path.exists(tgz) else download_file(TUM_BASE + spec["file"], tgz)
            if ok:
                try:
                    os.makedirs(seq_dir, exist_ok=True)
                    with tarfile.open(tgz) as tf:
                        members = [m for m in tf.getmembers() if m.isfile() and not os.path.isabs(m.name)
                                   and ".." not in m.name.split("/")]
                        for m in members:
                            parts = m.name.split("/", 1)
                            if len(parts) < 2:
                                continue
                            m.name = parts[1]                   # strip the top-level folder
                            try:
                                tf.extract(m, seq_dir, filter="data")
                            except TypeError:                   # Python without extraction filters
                                tf.extract(m, seq_dir)
                    open(done_flag, "w").write("ok")
                    os.remove(tgz)                              # keep disk usage low
                except Exception as e:
                    ok, msg = False, f"extraction failed: {type(e).__name__}: {e}"
        else:
            ok, msg = True, "already extracted"
        n_rgb = len(glob.glob(os.path.join(seq_dir, "rgb", "*.png"))) if ok else 0
        n_dep = len(glob.glob(os.path.join(seq_dir, "depth", "*.png"))) if ok else 0
        has_gt = os.path.exists(os.path.join(seq_dir, "groundtruth.txt"))
        ok = ok and n_rgb > 100 and n_dep > 100 and has_gt
        DATASET_STATUS[name] = dict(role=spec["role"], downloaded=bool(ok), restored=False, n_rgb=n_rgb, n_depth=n_dep,
                                    groundtruth=has_gt, note=msg, seconds=round(time.time() - t0, 1), path=seq_dir)

print(f"{'sequence':<30}{'role':<9}{'status':<10}{'rgb':>6}{'depth':>7}  note")
for k, v in DATASET_STATUS.items():
    status = "RESTORED" if v["restored"] else ("PASS" if v["downloaded"] else "FAIL")
    print(f"{k:<30}{v['role']:<9}{status:<10}{v['n_rgb']:>6}{v['n_depth']:>7}  {v['note']}")

In [ ]:
# Cell No: 025
# 3.4.2 TUM loader: timestamp association (|dt| <= 20 ms), pose interpolation from the
#       100 Hz motion-capture trajectory, and the ground-truth self-consistency test (A3).

def _read_tum_list(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            p = line.split()
            rows.append((float(p[0]), p[1]))
    return rows

def _read_tum_gt(path):
    a = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            a.append([float(v) for v in line.split()[:8]])
    return np.asarray(a, np.float64)

def quat_to_R(qx, qy, qz, qw):
    n = np.sqrt(qx * qx + qy * qy + qz * qz + qw * qw); qx, qy, qz, qw = qx / n, qy / n, qz / n, qw / n
    return np.array([[1 - 2 * (qy * qy + qz * qz), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
                     [2 * (qx * qy + qz * qw), 1 - 2 * (qx * qx + qz * qz), 2 * (qy * qz - qx * qw)],
                     [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx * qx + qy * qy)]])

def _slerp(q0, q1, a):
    d = float(np.dot(q0, q1))
    if d < 0:
        q1, d = -q1, -d
    if d > 0.9995:
        q = q0 + a * (q1 - q0)
        return q / np.linalg.norm(q)
    th = np.arccos(d)
    return (np.sin((1 - a) * th) * q0 + np.sin(a * th) * q1) / np.sin(th)

class TUMSequence:
    def __init__(self, name, root, max_dt=0.02, max_gt_gap=0.05):
        self.name, self.root = name, root
        rgb = _read_tum_list(os.path.join(root, "rgb.txt"))
        dep = _read_tum_list(os.path.join(root, "depth.txt"))
        gt = _read_tum_gt(os.path.join(root, "groundtruth.txt"))
        self.cam = PinholeCamera(**TUM_FR3_INTRINSICS)
        dt = np.array([d[0] for d in dep]); gtt = gt[:, 0]
        self.frames = []
        for t, f in rgb:
            j = int(np.argmin(np.abs(dt - t)))
            if abs(dt[j] - t) > max_dt:
                continue
            k = int(np.searchsorted(gtt, t))
            if k == 0 or k >= len(gtt) or gtt[k] - gtt[k - 1] > max_gt_gap:
                continue
            a = (t - gtt[k - 1]) / (gtt[k] - gtt[k - 1])
            C = (1 - a) * gt[k - 1, 1:4] + a * gt[k, 1:4]
            q = _slerp(gt[k - 1, [4, 5, 6, 7]], gt[k, [4, 5, 6, 7]], a)
            self.frames.append(dict(t=t, rgb=os.path.join(root, f), depth=os.path.join(root, dep[j][1]),
                                    R=quat_to_R(*q), C=C))
        self.pose_inverted = False

    def __len__(self):
        return len(self.frames)

    def rgb(self, i):
        return np.asarray(Image.open(self.frames[i]["rgb"]).convert("RGB"))

    def depth(self, i):
        d = np.asarray(Image.open(self.frames[i]["depth"])).astype(np.float32) / TUM_DEPTH_SCALE
        d[d <= 0] = np.nan
        return d

    def pose(self, i):
        R, C = self.frames[i]["R"], self.frames[i]["C"]
        if self.pose_inverted:                      # only if the convention test demands it
            return R.T, -R.T @ C
        return R, C

def depth_warp_error(seq, i, j, stride=4, inverted=None):
    # median relative disagreement between frame j's measured depth and frame i's depth
    # warped into frame j with the ground-truth relative pose (static pixels dominate)
    if inverted is not None:
        old = seq.pose_inverted; seq.pose_inverted = inverted
    cam = seq.cam
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    if inverted is not None:
        seq.pose_inverted = old
    di, dj = seq.depth(i), seq.depth(j)
    vv, uu = np.mgrid[0:cam.H:stride, 0:cam.W:stride]
    z = di[vv, uu]; ok = np.isfinite(z)
    Xc = cam.rays(uu[ok], vv[ok]) * z[ok][:, None]
    P = Xc @ Ri.T + Ci
    Xj = (P - Cj) @ Rj
    front = Xj[:, 2] > 0.1
    p = cam.project(Xj[front])
    inside = (p[:, 0] >= 0) & (p[:, 0] <= cam.W - 1) & (p[:, 1] >= 0) & (p[:, 1] <= cam.H - 1)
    zj = Xj[front][inside, 2]
    zo = dj[np.round(p[inside, 1]).astype(int), np.round(p[inside, 0]).astype(int)]
    m = np.isfinite(zo)
    if m.sum() < 200:
        return np.nan
    return float(np.median(np.abs(zo[m] - zj[m]) / zo[m]))

def baseline_and_angle(seq, i, j):
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    ang = np.rad2deg(np.arccos(np.clip((np.trace(Ri.T @ Rj) - 1) / 2, -1, 1)))
    return float(np.linalg.norm(Cj - Ci)), float(ang)

TUM = {}
A3_VALIDATION = {}
A3_PATH = os.path.join(LOG_DIR, "a3_validation.json")
if not TUM_NEEDED:
    for _st in TUM_STAGES.values():                  # restores the composites, walking GT and the A3 record
        assert _st.done(), f"{_st.name}: checkpoint vanished"
    with open(A3_PATH) as f:
        A3_VALIDATION = json.load(f)
for name, st in DATASET_STATUS.items():
    if not st["downloaded"]:
        continue
    seq = TUMSequence(name, st["path"])
    rng_v = np.random.default_rng(seed_for("a3", name))
    pairs = []
    for _ in range(400):
        i = int(rng_v.integers(0, len(seq) - 1))
        for j in range(i + 1, min(len(seq), i + 90)):
            b, ang = baseline_and_angle(seq, i, j)
            if b >= 0.08:
                if ang < 20:
                    pairs.append((i, j))
                break
        if len(pairs) >= 16:
            break
    errA = np.nanmedian([depth_warp_error(seq, i, j, inverted=False) for i, j in pairs]) if pairs else np.nan
    errB = np.nanmedian([depth_warp_error(seq, i, j, inverted=True) for i, j in pairs]) if pairs else np.nan
    seq.pose_inverted = bool(np.isfinite(errB) and errB < 0.5 * errA)
    err = errB if seq.pose_inverted else errA
    passed = bool(np.isfinite(err) and err <= 0.05)
    A3_VALIDATION[name] = dict(n_frames=len(seq), n_test_pairs=len(pairs), warp_err_documented=float(errA),
                               warp_err_inverted=float(errB), convention="inverted" if seq.pose_inverted else "documented",
                               median_rel_depth_err=float(err), passed=passed)
    st["validated"] = passed
    if passed:
        TUM[name] = seq

print(f"{'sequence':<28}{'frames':>7}{'pairs':>6}{'warp err (doc.)':>17}{'(inverted)':>12}  A3")
for k, v in A3_VALIDATION.items():
    print(f"{k:<28}{v['n_frames']:>7}{v['n_test_pairs']:>6}{v['warp_err_documented']:>17.3f}"
          f"{v['warp_err_inverted']:>12.3f}  {'PASS' if v['passed'] else 'FAIL'} ({v['convention']} convention)")
print("\nA3 criterion (pre-registered): median relative disagreement <= 5% between measured depth and "
      "depth warped with the motion-capture pose. Dynamic sequences pass on their static majority.")
if TUM_NEEDED:
    with open(A3_PATH, "w") as f:
        json.dump(A3_VALIDATION, f, indent=2)
else:
    print("(A3 record restored from the checkpoint of the stages built from these sequences)")

In [ ]:
# Cell No: 026
# 3.5 CONVOY-REAL: real TUM image pairs (real background, real camera motion, motion-capture
#     poses) with inserted rigid movers whose motion (s, eps) is controlled exactly. Movers
#     are ray-cast with the real intrinsics and poses, depth-tested against the measured
#     depth so that they are never inside or behind real surfaces, and alpha-blended at
#     their silhouettes. Positions are proposed only where the measured depth leaves free
#     space in front of the camera. A pair with no room for any mover (e.g. the camera close
#     to a desk or wall) is replaced by the next candidate pair, and a mover that does not fit
#     one pair is carried over to the next, so the designed number of movers and the designed
#     mix of epipolar offsets are kept.
from collections import Counter

REAL_EPS_TARGETS = {0.0: 0.45, 0.5: 0.10, 1.5: 0.05, 2.0: 0.05, 3.0: 0.10, 5.0: 0.10, 8.0: 0.10, 12.0: 0.05}
REAL_PLACEMENT = dict(attempts=300, z_min=0.7, z_max=3.0, clearance_window_px=41, min_clearance_m=1.1,
                      min_valid_frac=0.7, margin_px=70, max_pairs_per_mover=6)
_DUMMY_STREET = dict(ground_y=1e6, road_half=1.0, facade_x_left=-1e6, facade_x_right=1e6, backdrop_z=1e7,
                     skyline_shift=0.0, blocks_left=[[0, 1, 1, 0, 0]], blocks_right=[[0, 1, 1, 0, 0]])

def iter_static_pairs(seq, rng, b_range=(0.10, 0.30), max_angle=20.0, max_warp=0.06, max_starts=2000):
    # candidate frame pairs in random order: baseline in b_range, rotation <= max_angle, enough
    # valid depth in both frames, and depth/pose self-consistency (warp error) <= max_warp
    n = len(seq)
    starts = np.unique(np.linspace(0, n - 2, num=max(2, min(max_starts, n - 1))).astype(int))
    rng.shuffle(starts)
    seen = set()
    for i in starts:
        target = rng.uniform(*b_range)
        for j in range(i + 1, min(n, i + 150)):
            b, ang = baseline_and_angle(seq, i, j)
            if b >= target:
                if ang <= max_angle and b <= b_range[1] + 0.05 and (i, j) not in seen:
                    di, dj = seq.depth(i), seq.depth(j)
                    if np.isfinite(di).mean() >= 0.5 and np.isfinite(dj).mean() >= 0.5:
                        e = depth_warp_error(seq, i, j)
                        if np.isfinite(e) and e <= max_warp:
                            seen.add((i, j))
                            yield int(i), int(j), float(b), float(ang), float(e)
                break

def _free_space_candidates(depth):
    # pixels with at least min_clearance_m of free space in front of the nearest real surface
    # within the window, enough valid depth around them, and away from the image border
    P = REAL_PLACEMENT
    valid = np.isfinite(depth)
    k = np.ones((P["clearance_window_px"], P["clearance_window_px"]), np.uint8)
    clearance = cv2.erode(np.where(valid, depth, 1e3).astype(np.float32), k)
    vfrac = cv2.blur(valid.astype(np.float32), (P["clearance_window_px"], P["clearance_window_px"]))
    ok = valid & (clearance >= P["min_clearance_m"]) & (vfrac >= P["min_valid_frac"])
    m = P["margin_px"]
    ok[:m] = False; ok[-m:] = False; ok[:, :m] = False; ok[:, -m:] = False
    return clearance, np.argwhere(ok)

def render_movers_only(scene, frame_k):
    # colour, coverage (alpha), depth and mover id of the movers alone (anti-aliased edges)
    cam, R, C = frame_camera(scene, frame_k)
    W, H = cam.W, cam.H
    if not scene["movers"]:
        return (np.zeros((H, W, 3), np.float32), np.zeros((H, W), np.float32),
                np.full((H, W), np.inf), np.full((H, W), -1, np.int32))
    vv, uu = np.mgrid[0:H, 0:W]
    D = cam.rays(uu.ravel().astype(np.float64), vv.ravel().astype(np.float64)) @ R.T
    ids = list(range(len(scene["movers"])))
    hit = cast_rays(scene, C, D, frame_k, True, grid=(W, H), cam=cam, R=R, only_movers=ids)
    f_mean = 0.5 * (cam.fx + cam.fy)
    hitm = np.isfinite(hit["t"])
    col = np.zeros((H * W, 3), np.float32)
    if hitm.any():
        hs = np.nonzero(hitm)[0]
        sub = {k: v[hs] for k, v in hit.items()}
        col[hs] = shade_rays(scene, C, D[hs], sub, frame_k, f_mean)[0]
    alpha = hitm.astype(np.float32)
    key = np.where(hitm, hit["bid"], -1).reshape(H, W)
    e = np.zeros((H, W), bool)
    dh = key[:, 1:] != key[:, :-1]; dv = key[1:, :] != key[:-1, :]
    e[:, 1:] |= dh; e[:, :-1] |= dh; e[1:, :] |= dv; e[:-1, :] |= dv
    ei = np.nonzero(e.ravel())[0]
    if ei.size:
        acc_c = col[ei] * alpha[ei, None]; acc_a = alpha[ei].copy()
        for du, dv_ in ((-0.25, -0.25), (0.25, -0.25), (-0.25, 0.25), (0.25, 0.25)):
            pu = uu.ravel()[ei] + du; pv = vv.ravel()[ei] + dv_
            Ds = cam.rays(pu, pv) @ R.T
            h2 = cast_rays(scene, C, Ds, frame_k, True, pix=(pu, pv), cam=cam, R=R, only_movers=ids)
            m2 = np.isfinite(h2["t"])
            if m2.any():
                q = np.nonzero(m2)[0]
                c2 = shade_rays(scene, C, Ds[q], {k: v[q] for k, v in h2.items()}, frame_k, f_mean)[0]
                acc_c[q] += c2
            acc_a += m2
        alpha[ei] = acc_a / 5.0
        col[ei] = np.where(acc_a[:, None] > 0, acc_c / np.maximum(acc_a[:, None], 1e-6), 0)
    arr = _scene_box_arrays(scene)
    mid = np.full(H * W, -1, np.int32)
    mid[hitm] = arr["mover"][hit["bid"][hitm]]                  # only pixels that hit a mover box
    return col.reshape(H, W, 3), alpha.reshape(H, W), hit["t"].reshape(H, W), mid.reshape(H, W)

def build_composite(seq, i, j, items, seed):
    # returns (scene, renders or None if no mover fits, placed flag per item, failure counts)
    P = REAL_PLACEMENT
    rng = np.random.default_rng(seed)
    cam = seq.cam
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    di, dj = seq.depth(i), seq.depth(j)
    cams = [dict(**cam.to_dict(), R=Ri.tolist(), C=Ci.tolist()), dict(**cam.to_dict(), R=Rj.tolist(), C=Cj.tolist())]
    b = Cj - Ci
    scene = dict(seed=int(seed), street=_DUMMY_STREET, boxes=[], shadows=[], movers=[], cams=cams,
                 camera=dict(kind="tum", path="straight", focal=0.5 * (cam.fx + cam.fy), baseline=float(np.linalg.norm(b)),
                             psi_deg=0.0, omega_deg=0.0),
                 light=dict(exposure=1.0, white_balance=[1.0, 1.0, 1.0], noise_sigma=1.2 / 255))
    real_d = [di, dj]
    placed, flags, fails = [], [], Counter()
    clearance, cand = _free_space_candidates(di)
    if len(cand) == 0:
        fails["no free space in view 1"] += 1
        return scene, None, [False] * len(items), fails
    for item in items:
        ok_item = False
        for attempt in range(P["attempts"]):
            v_, u_ = cand[int(rng.integers(len(cand)))]
            u, v = u_ + rng.uniform(-0.5, 0.5), v_ + rng.uniform(-0.5, 0.5)
            z_hi = min(P["z_max"], float(clearance[v_, u_]) - 0.35)
            Zc = rng.uniform(P["z_min"], z_hi)
            a_ = rng.uniform(0.12, 0.30)
            half = np.array([a_ / 2, a_ / 2 * rng.uniform(0.8, 1.0), a_ / 2 * rng.uniform(0.8, 1.2)])
            yaw = rng.uniform(0, np.pi / 2)
            centroid = Ci + Ri @ (cam.rays(u, v) * Zc)
            s = rng.uniform(*MOVER_DESIGN["s_range_main"])
            delta = s * b; motion_type = "parallel"
            if item["eps"] > 0:
                n_dir = np.cross(centroid - Ci, b); n_dir /= np.linalg.norm(n_dir)
                m, ach = solve_perp_for_eps(centroid, s * b, n_dir, item["eps"], cam, Ri, Ci, cam, Rj, Cj, max_m=1.0)
                if m is None or abs(ach - item["eps"]) > 1e-3:
                    fails["offset not reachable"] += 1
                    continue
                delta = s * b + m * n_dir; motion_type = "perpendicular_component"
            style = ["wood", "container", "pattern"][int(rng.integers(3))]
            tint = CONTAINER_TINTS[int(rng.integers(len(CONTAINER_TINTS)))] if style == "container" else (1, 1, 1)
            boxes = crate_boxes(centroid, half, yaw, style, tint, kind="mover", mover=len(scene["movers"]))
            ok, bbs, why = True, [], None
            for k, (R_, C_) in enumerate(((Ri, Ci), (Rj, Cj))):
                uv = _project_boxes(boxes, k * delta, cam, R_, C_, znear=0.3)
                if uv is None:
                    ok, why = False, "outside view"; break
                u0, v0 = uv.min(0); u1, v1 = uv.max(0)
                if u0 < 8 or v0 < 8 or u1 > cam.W - 9 or v1 > cam.H - 9 or (k == 0 and (u1 - u0) * (v1 - v0) < 600):
                    ok, why = False, "outside view"; break
                zmax = ((np.concatenate([box_corners(bx, k * delta) for bx in boxes]) - C_) @ R_)[:, 2].max()
                reg = real_d[k][int(v0):int(v1) + 1, int(u0):int(u1) + 1]
                fin = np.isfinite(reg)
                if fin.mean() < 0.6 or np.nanmin(reg) < zmax + 0.15:
                    ok, why = False, "behind a real surface"; break
                for other in placed:
                    ou0, ov0, ou1, ov1 = other[k]
                    if not (u1 + 8 < ou0 or ou1 + 8 < u0 or v1 + 8 < ov0 or ov1 + 8 < v0):
                        ok, why = False, "overlaps another mover"; break
                if not ok:
                    break
                bbs.append((u0, v0, u1, v1))
            if not ok:
                fails[why] += 1
                continue
            p0 = cam.project(world_to_cam(centroid, Ri, Ci)); p1 = cam.project(world_to_cam(centroid + delta, Rj, Cj))
            R21, t21 = relative_pose(Ri, Ci, Rj, Cj)
            F = fundamental_matrix(cam.K, cam.K, R21, t21)
            Zg, _ = triangulate_depth(p0, p1, cam.K, cam.K, R21, t21)
            mv = dict(idx=len(scene["movers"]), cls="crate_floating", style=style, tint=[float(t) for t in tint],
                      s=float(s), eps_target=float(item["eps"]), eps_center=abs(float(epipolar_offset(p0, p1, F))),
                      beyond=False, motion_type=motion_type, delta=[float(x) for x in delta],
                      centroid0=centroid.tolist(), Z_center=float(world_to_cam(centroid, Ri, Ci)[2]),
                      Zgeo_center=float(Zg), parallax_center=float(translational_parallax(p0, p1, cam.K, cam.K, R21)),
                      dims=[float(x) for x in half], yaw=float(yaw), bbox0=[float(x) for x in bbs[0]])
            mv["box_ids"] = list(range(len(scene["boxes"]), len(scene["boxes"]) + len(boxes)))
            scene["boxes"] += boxes; scene["movers"].append(mv); placed.append(bbs)
            scene.pop("_arr", None)
            ok_item = True
            break
        flags.append(ok_item)
    if not scene["movers"]:
        return scene, None, flags, fails
    renders = []
    for k, (fi, dk) in enumerate(((i, di), (j, dj))):
        col, alpha, zt, mid = render_movers_only(scene, k)
        real = seq.rgb(fi).astype(np.float32) / 255.0
        nz = np.random.default_rng(seed * 7 + k).normal(0, 1.2 / 255, col.shape).astype(np.float32)
        comp = alpha[..., None] * (col + nz) + (1 - alpha[..., None]) * real
        mover_px = alpha >= 0.5
        depth = np.where(mover_px, zt, dk).astype(np.float32)
        sem = np.where(mover_px, SEM["mover"], np.where(np.isfinite(dk), SEM["static_box"], 0)).astype(np.int8)
        renders.append(dict(rgb=(np.clip(comp, 0, 1) * 255 + 0.5).astype(np.uint8), depth=depth, sem=sem,
                            mover_id=np.where(mover_px, mid, -1).astype(np.int16),
                            mover_shadow=np.zeros(depth.shape, bool)))
    return scene, renders, flags, fails

REAL_BUILD_LOG_PATH = os.path.join(LOG_DIR, "convoy_real_build.json")
with stage("convoy_real"):
    _st = TUM_STAGES["REAL"]
    if not _st.done():
        # built from scratch: earlier composites (and their optical flows) are removed first
        shutil.rmtree(os.path.join(RENDER_DIR, "real"), ignore_errors=True)
        shutil.rmtree(os.path.join(FLOW_DIR, "real"), ignore_errors=True)
        _log_out = {}
        _n_meta = 0
        for name, seq in TUM.items():
            if DATASET_STATUS[name]["role"] != "static":
                continue
            n_target, n_m = RUN_CONFIG["n_real_pairs_per_seq"], RUN_CONFIG["real_movers_per_pair"]
            deck = make_design_deck(n_target * n_m, seed_for("deck-real", name), classes={"crate_floating": 1.0},
                                    eps_targets=REAL_EPS_TARGETS, p_beyond=0.0)
            queue = [dict(item=it, tries=0) for it in deck]
            used, log = [], dict(designed=len(deck), designed_eps=Counter(it["eps"] for it in deck), candidates=0,
                                 no_room=0, carried=0, dropped=0, fails=Counter())
            for pk, (i, j, b, ang, werr) in enumerate(iter_static_pairs(seq, np.random.default_rng(seed_for("real-pairs", name)))):
                if not queue:
                    break
                log["candidates"] += 1
                take, queue = queue[:n_m], queue[n_m:]
                scene, renders, flags, fails = build_composite(seq, i, j, [t["item"] for t in take], seed_for("real", name, pk))
                log["fails"].update(fails)
                back = []
                for t, ok in zip(take, flags):
                    if not ok:
                        t["tries"] += 1
                        if t["tries"] < REAL_PLACEMENT["max_pairs_per_mover"]:
                            back.append(t); log["carried"] += 1
                        else:
                            log["dropped"] += 1
                queue = back + queue                                 # unplaced movers go first on the next pair
                if renders is None:
                    log["no_room"] += 1
                    continue
                d = os.path.join(RENDER_DIR, "real", f"{name}_{len(used):03d}")
                os.makedirs(d, exist_ok=True)
                for k, r in enumerate(renders):
                    _save_frame(d, k, r)
                gt01 = mover_gt_stats(scene, renders, 0, 1)
                meta = dict(scene=scene_to_json(scene), gt01=gt01, gt0last=gt01, set="REAL", index=_n_meta,
                            seq=name, frames=[i, j], baseline=b, rot_deg=ang, warp_err=werr, dir=d, complete=True)
                with open(os.path.join(d, "scene.json"), "w") as f:
                    json.dump(meta, f)
                _n_meta += 1
                used.append((i, j, b, ang, werr))
            log["dropped"] += len(queue)                             # candidate pairs exhausted
            log = {k: (dict(v) if isinstance(v, Counter) else v) for k, v in log.items()}
            log["designed_eps"] = {str(k): v for k, v in log["designed_eps"].items()}
            _log_out[name] = dict(pairs=used, log=log)
        with open(REAL_BUILD_LOG_PATH, "w") as f:
            json.dump(_log_out, f, indent=1)
        if all(DATASET_STATUS[n]["downloaded"] for n, v in TUM_SEQUENCES.items() if v["role"] == "static"):
            _st.commit([os.path.join(RENDER_DIR, "real"), REAL_BUILD_LOG_PATH, A3_PATH])
        else:
            print("CONVOY-REAL is incomplete (a static TUM sequence is missing); it is not checkpointed.")
META_REAL = sorted(_load_metas(os.path.join(RENDER_DIR, "real")), key=lambda m: m["index"]) \
    if os.path.isdir(os.path.join(RENDER_DIR, "real")) else []
_blog = {}
if os.path.exists(REAL_BUILD_LOG_PATH):
    with open(REAL_BUILD_LOG_PATH) as f:
        _blog = json.load(f)
REAL_PAIRS = {k: v["pairs"] for k, v in _blog.items()}
REAL_BUILD_LOG = {k: v["log"] for k, v in _blog.items()}

MOVERS_REAL = _mover_table(META_REAL) if META_REAL else pd.DataFrame()
if len(MOVERS_REAL):
    MOVERS_REAL["seq"] = [META_REAL[i]["seq"] for i in MOVERS_REAL["scene"]]
    MOVERS_REAL["x_par"] = -np.log(1 - MOVERS_REAL["s"])
    MOVERS_REAL["regime"] = np.where(MOVERS_REAL["gt_eps_med"] < PREREG["eps_parallel_max"], "parallel",
                                     np.where(MOVERS_REAL["gt_eps_med"] >= PREREG["eps_perp_min"], "perpendicular", "intermediate"))
    MOVERS_REAL["measurable"] = (MOVERS_REAL["gt_n_eroded"] >= PREREG["min_object_pixels"]) & (MOVERS_REAL["gt_visible_frac"] >= 0.85)
    MOVERS_REAL["identifiable"] = MOVERS_REAL["gt_parallax_med"] >= PREREG["parallax_min_px"]
    MOVERS_REAL.to_csv(os.path.join(MANIFEST_DIR, "convoy_real_movers.csv"), index=False)
for name, pairs in REAL_PAIRS.items():
    lg = REAL_BUILD_LOG[name]
    n_placed = int((MOVERS_REAL["seq"] == name).sum()) if len(MOVERS_REAL) else 0
    print(f"{name}: {len(pairs)} pairs from {lg['candidates']} candidates ({lg['no_room']} without room for a mover); "
          f"{n_placed}/{lg['designed']} designed movers placed ({lg['carried']} moved to another pair, {lg['dropped']} not placed)")
    if pairs:
        bs = [p[2] for p in pairs]
        print(f"    baseline {np.min(bs):.2f}-{np.max(bs):.2f} m, median warp error {np.median([p[4] for p in pairs]):.3f}; "
              f"rejected placement attempts: {dict(lg['fails'])}")
if len(MOVERS_REAL):
    print(f"CONVOY-REAL: {len(MOVERS_REAL)} movers in {len(META_REAL)} pairs;",
          MOVERS_REAL.groupby("regime").size().to_dict(), "| measurable:", int(MOVERS_REAL["measurable"].sum()),
          "| identifiable:", int(MOVERS_REAL["identifiable"].sum()))
    _designed = sum((Counter({float(k): v for k, v in l["designed_eps"].items()}) for l in REAL_BUILD_LOG.values()), Counter())
    print("placed / designed movers per target offset (px):",
          {e: f"{int((MOVERS_REAL['eps_target'] == e).sum())}/{_designed[e]}" for e in sorted(_designed)})

release_memory(); _ = mem_report("CONVOY-REAL composites")

In [ ]:
# Cell No: 027
# 3.5.1 E5-REAL (v6.1): the ground-contact ablation on real images. In real TUM frame pairs (static
#     sequences, motion-capture poses, measured depth) the same textured crate is inserted twice:
#       G  resting on the support plane (floor or table top) with a soft contact shadow,
#       F  lifted 0.25-0.50 m off the plane, without a shadow;
#     everything else (pair, crate, texture, s, motion) is identical. The support plane is fitted
#     to the measured depth of view 1 (RANSAC). It must be horizontal: its normal within 12 deg of the
#     vertical of the motion-capture frame when the trajectory identifies one (cameras held upright
#     along one world axis), otherwise within 35 deg of the camera's up direction; the camera must be
#     at least 0.3 m above it. On-plane tests allow for the depth noise of the sensor at range z;
#     the crate's footprint must lie on that plane in the real depth, the crate may neither intersect
#     nor be hidden by real surfaces, and the motion is purely parallel (delta = s b) on pairs whose
#     baseline is within 6 deg of the plane, so the grounded crate stays on the plane in both views.
#     Each composite is rendered in a frame aligned with the plane (y down, plane at y = 0), so the
#     crate stands on one of its faces; depth, correspondences and every model input are unaffected
#     by this choice of world frame. One checkpointed stage (Section 0.5).
RABL_STAGE = TUM_STAGES["RABL"]

def _ray_box_t(Cc, Dd, center, half, yaw):
    # depth of the first intersection of camera rays (origin Cc, directions Dd with unit camera z)
    # with an oriented box (np.inf where the ray misses)
    Rb = rot_y(yaw)
    o = (Cc - np.asarray(center)) @ Rb
    d = Dd @ Rb
    h = np.asarray(half)
    with np.errstate(divide="ignore", invalid="ignore"):
        t1 = (-h - o) / d; t2 = (h - o) / d
    tmin = np.nanmax(np.minimum(t1, t2), axis=-1); tmax = np.nanmin(np.maximum(t1, t2), axis=-1)
    hit = (tmax >= np.maximum(tmin, 0)) & np.isfinite(tmin)
    return np.where(hit, np.maximum(tmin, 0), np.inf)

def sequence_vertical(seq):
    # the world axis along which the cameras are held upright (motion-capture frames are levelled
    # to the floor); None if no axis is consistent (mean cosine < 0.6), then the camera's up is used
    ups = np.stack([-seq.pose(k)[0][:, 1] for k in range(0, len(seq), 5)])
    m = ups.mean(0)
    a = int(np.argmax(np.abs(m)))
    if abs(m[a]) < 0.6:
        return None
    g = np.zeros(3); g[a] = np.sign(m[a])
    return g

def kinect_sigma(z):
    # axial depth noise of a structured-light RGB-D sensor (Nguyen, Izadi and Lovell, 3DIMPVT 2012)
    return 0.0012 + 0.0019 * (np.asarray(z, np.float64) - 0.4) ** 2

def _plane_tol(z):
    return np.maximum(0.02, RABL_DESIGN["on_plane_sigma_k"] * kinect_sigma(z))

def fit_support_plane(seq, i, rng, vertical=None):
    D = RABL_DESIGN
    d = seq.depth(i); cam = seq.cam; R, C = seq.pose(i)
    H, W = d.shape
    vv, uu = np.mgrid[0:H:4, 0:W:4]
    z = d[vv, uu]
    ok = np.isfinite(z) & (z > 0.3) & (z < D["plane_fit_max_depth_m"])
    if ok.sum() < 400:
        return None
    Xw = C + (cam.rays(uu[ok].astype(np.float64), vv[ok].astype(np.float64)) * z[ok][:, None]) @ R.T
    up = vertical if vertical is not None else -R[:, 1]   # motion-capture vertical, else the image's up
    max_tilt = D["vertical_max_tilt_deg"] if vertical is not None else D["plane_max_tilt_deg"]
    best = None
    for _ in range(D["plane_ransac_iters"]):
        p = Xw[rng.choice(len(Xw), 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0]); nn = np.linalg.norm(n)
        if nn < 1e-9:
            continue
        n /= nn
        n = n if n @ up >= 0 else -n
        if np.degrees(np.arccos(np.clip(n @ up, -1, 1))) > max_tilt or (C - p[0]) @ n < 0.3:
            continue                                     # not horizontal, or not below the camera (a ceiling)
        inl = np.abs((Xw - p[0]) @ n) < D["plane_inlier_m"]
        if best is None or inl.sum() > best.sum():
            best = inl
    if best is None or best.sum() < D["plane_min_inlier_frac"] * len(Xw):
        return None
    P = Xw[best]; c0 = P.mean(0)
    n = np.linalg.svd(P - c0, full_matrices=False)[2][-1]
    n = n if n @ up >= 0 else -n
    h = float((C - c0) @ n)
    if h < 0.3:
        return None
    tilt_v = float(np.degrees(np.arccos(np.clip(n @ vertical, -1, 1)))) if vertical is not None else np.nan
    return dict(n=n, origin=c0, height=h, inlier_frac=float(best.mean()), tilt_to_vertical_deg=tilt_v,
                reference="motion-capture vertical" if vertical is not None else "camera up")

def _plane_frame(n, R_cam):
    # rotation A (rows) from the world frame to a frame with y = -n (down) and x along the camera's right
    e_y = -n
    x0 = R_cam[:, 0] - (R_cam[:, 0] @ e_y) * e_y
    e_x = x0 / np.linalg.norm(x0)
    return np.stack([e_x, e_y, np.cross(e_x, e_y)])

def _on_plane(depth, cam, R, C):
    # mask of real pixels whose measured 3-D point lies on the plane y = 0 (plane frame), within the
    # sensor's noise at that range
    H, W = depth.shape
    vv, uu = np.mgrid[0:H, 0:W]
    fin = np.isfinite(depth)
    Y = np.full((H, W), np.nan)
    D_ = cam.rays(uu[fin].astype(np.float64), vv[fin].astype(np.float64)) @ R.T
    Y[fin] = (C + D_ * depth[fin][:, None])[:, 1]
    tol = np.full((H, W), np.nan); tol[fin] = _plane_tol(depth[fin])
    with np.errstate(invalid="ignore"):
        return np.abs(Y) < tol, Y

def _crate_ok(center, half, yaw, k_off, cams_L, depths, cam, margin):
    # bounding box per view; rejects crates outside the image, colliding with or hidden by real surfaces
    bbs = []
    for k, (R_, C_) in enumerate(cams_L):
        c = np.asarray(center) + k_off[k]
        corners = box_corners(dict(center=c, half=half, yaw=yaw))
        Xc = (corners - C_) @ R_
        if np.any(Xc[:, 2] < 0.3):
            return None
        uv = cam.project(Xc)
        u0, v0 = uv.min(0); u1, v1 = uv.max(0)
        if u0 < margin or v0 < margin or u1 > cam.W - 1 - margin or v1 > cam.H - 1 - margin or \
           (k == 0 and (u1 - u0) * (v1 - v0) < 600):
            return None
        vv, uu = np.mgrid[int(v0):int(v1) + 1, int(u0):int(u1) + 1]
        Dd = cam.rays(uu.ravel().astype(np.float64), vv.ravel().astype(np.float64)) @ R_.T
        t = _ray_box_t(C_, Dd, c, half, yaw)
        hit = np.isfinite(t)
        real = depths[k][vv.ravel(), uu.ravel()]
        if hit.sum() < 300 or np.mean(np.isfinite(real[hit])) < 0.7:
            return None
        rh, th = real[hit], t[hit]
        with np.errstate(invalid="ignore"):
            Py = C_[1] + Dd[hit][:, 1] * rh                     # plane-frame height of the real point
            front = (rh < th - np.maximum(0.03, _plane_tol(rh))) & ~(np.abs(Py) < _plane_tol(rh))
        if np.mean(front) > 0.02:                                # a real surface in front of (or inside) the crate;
            return None                                          # the support plane itself cannot hide it
        bbs.append((u0, v0, u1, v1))
    return bbs

def _footprint_on_plane(center, half, yaw, R_, C_, cam, depth):
    # the real depth under the crate's footprint (its corners and edge midpoints) lies on the plane
    sg = np.array([(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0)], np.float64)
    pts = np.column_stack([sg[:, 0] * half[0], np.zeros(len(sg)), sg[:, 1] * half[2]]) @ rot_y(yaw).T
    pts = pts + np.array([center[0], 0.0, center[2]])
    Xc = (pts - C_) @ R_
    if np.any(Xc[:, 2] < 0.3):
        return False
    uv = np.round(cam.project(Xc)).astype(int)
    if np.any((uv[:, 0] < 0) | (uv[:, 0] >= cam.W) | (uv[:, 1] < 0) | (uv[:, 1] >= cam.H)):
        return False
    z = depth[uv[:, 1], uv[:, 0]]
    if not np.all(np.isfinite(z)):
        return False
    P = C_ + (cam.rays(uv[:, 0].astype(np.float64), uv[:, 1].astype(np.float64)) * z[:, None]) @ R_.T
    return bool(np.all(np.abs(P[:, 1]) < _plane_tol(z)))

def _contact_shadow(rgb, depth, cam, R_, C_, movers_k, onplane):
    # soft darkening of the real support-plane pixels around each crate's footprint (view k)
    D = RABL_DESIGN
    H, W = depth.shape
    vv, uu = np.mgrid[0:H, 0:W]
    fin = onplane & np.isfinite(depth)
    w = np.zeros((H, W), np.float32)
    if fin.any():
        P = C_ + (cam.rays(uu[fin].astype(np.float64), vv[fin].astype(np.float64)) * depth[fin][:, None]) @ R_.T
        for c, half, yaw in movers_k:
            q = (P - np.asarray(c)) @ rot_y(yaw)
            r = np.maximum(np.abs(q[:, 0]) / half[0], np.abs(q[:, 2]) / half[2])
            wk = np.clip((1 + D["shadow_softness"] - r) / D["shadow_softness"], 0, 1) ** 1.5
            w[fin] = np.maximum(w[fin], wk)
    out = rgb.astype(np.float32) * (1 - D["shadow_strength"] * w[..., None])
    return out, w

def build_rabl_pair(seq, i, j, n_items, seed, vertical=None):
    # -> ((scenes {cond: scene}, renders {cond: [r0, r1]}, plane), None) or (None, reason)
    D = RABL_DESIGN
    rng = np.random.default_rng(seed)
    cam = seq.cam
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    plane = fit_support_plane(seq, i, rng, vertical)
    if plane is None:
        return None, "no support plane"
    b = Cj - Ci
    tilt = np.degrees(np.arcsin(min(1.0, abs(plane["n"] @ b) / max(np.linalg.norm(b), 1e-9))))
    if tilt > D["max_baseline_tilt_deg"]:
        return None, "baseline not parallel to the plane"
    A = _plane_frame(plane["n"], Ri); O = plane["origin"]
    cams_L = [(A @ Ri, A @ (Ci - O)), (A @ Rj, A @ (Cj - O))]
    bL = cams_L[1][1] - cams_L[0][1]
    di, dj = seq.depth(i), seq.depth(j)
    onp0, _ = _on_plane(di, cam, *cams_L[0])
    cand = np.argwhere(onp0 & (di >= D["depth_m"][0]) & (di <= D["depth_m"][1]))
    m = D["margin_px"]
    cand = cand[(cand[:, 0] > m) & (cand[:, 0] < cam.H - m) & (cand[:, 1] > m) & (cand[:, 1] < cam.W - m)]
    if len(cand) == 0:
        return None, "no free plane in view"
    placed, bbs_all = [], []
    for item in range(n_items):
        for attempt in range(D["attempts"]):
            v_, u_ = cand[int(rng.integers(len(cand)))]
            P = cams_L[0][1] + cams_L[0][0] @ (cam.rays(float(u_), float(v_)) * float(di[v_, u_]))
            a_ = rng.uniform(*D["crate_size_m"])
            half = np.array([a_ / 2, a_ / 2 * rng.uniform(0.8, 1.0), a_ / 2 * rng.uniform(0.8, 1.2)])
            yaw = rng.uniform(0, np.pi / 2)
            s = rng.uniform(*MOVER_DESIGN["s_range_main"])
            delta = s * bL
            lift = rng.uniform(*D["lift_m"])
            cG = np.array([P[0], -half[1], P[2]]); cF = cG + np.array([0.0, -lift, 0.0])
            if not _footprint_on_plane(cG, half, yaw, *cams_L[0], cam, di):
                continue
            offs = [np.zeros(3), delta]
            bG = _crate_ok(cG, half, yaw, offs, cams_L, (di, dj), cam, 8)
            bF = _crate_ok(cF, half, yaw, offs, cams_L, (di, dj), cam, 8) if bG else None
            if not bG or not bF:
                continue
            boxes_all = bG + bF
            clash = any(not (u1 + 8 < ou0 or ou1 + 8 < u0 or v1 + 8 < ov0 or ov1 + 8 < v0)
                        for (u0, v0, u1, v1) in boxes_all for ob in bbs_all for (ou0, ov0, ou1, ov1) in ob)
            if clash:
                continue
            style = ["wood", "container", "pattern"][int(rng.integers(3))]
            tint = CONTAINER_TINTS[int(rng.integers(len(CONTAINER_TINTS)))] if style == "container" else (1, 1, 1)
            placed.append(dict(cG=cG, cF=cF, half=half, yaw=yaw, s=s, delta=delta, lift=lift, style=style, tint=tint,
                               bbG=bG, bbF=bF))
            bbs_all.append(boxes_all)
            break
    if not placed:
        return None, "no crate fits"
    scenes, renders = {}, {}
    R21, t21 = relative_pose(*cams_L[0], *cams_L[1])
    F_ = fundamental_matrix(cam.K, cam.K, R21, t21)
    for cond in RABL_DESIGN["conditions"]:
        sc = dict(seed=int(seed), street=_DUMMY_STREET, boxes=[], shadows=[], movers=[], condition=cond,
                  cams=[dict(**cam.to_dict(), R=Rk.tolist(), C=Ck.tolist()) for Rk, Ck in cams_L],
                  camera=dict(kind="tum", path="straight", focal=0.5 * (cam.fx + cam.fy), baseline=float(np.linalg.norm(b)),
                              psi_deg=0.0, omega_deg=0.0),
                  light=dict(exposure=1.0, white_balance=[1.0, 1.0, 1.0], noise_sigma=1.2 / 255),
                  plane=dict(normal_world=plane["n"].tolist(), origin_world=O.tolist(), camera_height_m=plane["height"],
                             inlier_frac=plane["inlier_frac"], baseline_tilt_deg=float(tilt), reference=plane["reference"],
                             tilt_to_vertical_deg=plane["tilt_to_vertical_deg"]))
        for p in placed:
            c = p["cG"] if cond == "G" else p["cF"]
            idx = len(sc["movers"])
            boxes = crate_boxes(c, p["half"], p["yaw"], p["style"], p["tint"], kind="mover", mover=idx)
            p0 = cam.project(world_to_cam(c, *cams_L[0])); p1 = cam.project(world_to_cam(c + p["delta"], *cams_L[1]))
            Zg, _ = triangulate_depth(p0, p1, cam.K, cam.K, R21, t21)
            mv = dict(idx=idx, cls="crate", style=p["style"], tint=[float(t) for t in p["tint"]], s=float(p["s"]),
                      eps_target=0.0, eps_center=abs(float(epipolar_offset(p0, p1, F_))), beyond=False, motion_type="parallel",
                      delta=[float(x) for x in p["delta"]], centroid0=c.tolist(), Z_center=float(world_to_cam(c, *cams_L[0])[2]),
                      Zgeo_center=float(Zg), parallax_center=float(translational_parallax(p0, p1, cam.K, cam.K, R21)),
                      dims=[float(x) for x in p["half"]], yaw=float(p["yaw"]), lift=float(p["lift"]) if cond == "F" else 0.0,
                      support="plane" if cond == "G" else "none", bbox0=[float(x) for x in (p["bbG"] if cond == "G" else p["bbF"])[0]])
            mv["box_ids"] = list(range(len(sc["boxes"]), len(sc["boxes"]) + len(boxes)))
            sc["boxes"] += boxes; sc["movers"].append(mv)
        sc.pop("_arr", None)
        rr = []
        for k, (fi, dk) in enumerate(((i, di), (j, dj))):
            R_, C_ = cams_L[k]
            col, alpha, zt, mid = render_movers_only(sc, k)
            real = seq.rgb(fi).astype(np.float32) / 255.0
            onp, _ = _on_plane(dk, cam, R_, C_)
            if cond == "G":
                real, w_sh = _contact_shadow(real, dk, cam, R_, C_, [(np.asarray(p["cG"]) + k * p["delta"], p["half"], p["yaw"])
                                                                     for p in placed], onp)
            else:
                w_sh = np.zeros(dk.shape, np.float32)
            nz = np.random.default_rng(seed * 7 + k).normal(0, 1.2 / 255, col.shape).astype(np.float32)
            # real surfaces in front hide the crate; the support plane cannot (the camera is above it), so
            # sensor noise on the plane never eats into the contact line
            with np.errstate(invalid="ignore"):
                occl = np.isfinite(dk) & np.isfinite(zt) & (dk < zt - np.maximum(0.02, _plane_tol(dk))) & ~onp
            vis = (alpha > 0) & ~occl
            a_ = np.where(vis, alpha, 0.0)
            comp = a_[..., None] * (col + nz) + (1 - a_[..., None]) * real
            mover_px = a_ >= 0.5
            depth = np.where(mover_px, zt, dk).astype(np.float32)
            sem = np.where(mover_px, SEM["mover"], np.where(np.isfinite(dk) & (w_sh < 0.1), SEM["static_box"], 0)).astype(np.int8)
            rr.append(dict(rgb=(np.clip(comp, 0, 1) * 255 + 0.5).astype(np.uint8), depth=depth, sem=sem,
                           mover_id=np.where(mover_px, mid, -1).astype(np.int16), mover_shadow=w_sh >= 0.1))
        scenes[cond], renders[cond] = sc, rr
    return (scenes, renders, plane), None

def rabl_contact_pixels(scene, render, k, j):
    # visible pixels of crate j within rabl_contact_band_m of the support plane (plane frame: y = 0)
    cam, R, C = frame_camera(scene, k)
    m_ = render["mover_id"] == j
    if not m_.any():
        return 0
    vv, uu = np.nonzero(m_)
    P = C + (cam.rays(uu.astype(np.float64), vv.astype(np.float64)) * render["depth"][vv, uu][:, None].astype(np.float64)) @ R.T
    return int(np.sum(-P[:, 1] <= PREREG["rabl_contact_band_m"]))

RABL_BUILD_LOG_PATH = os.path.join(LOG_DIR, "convoy_real_contact_build.json")
with stage("convoy_real_contact"):
    if not RABL_STAGE.done():
        shutil.rmtree(os.path.join(RENDER_DIR, "rabl"), ignore_errors=True)
        _log_out, _n_base = {}, 0
        for name, seq in TUM.items():
            if DATASET_STATUS[name]["role"] != "static":
                continue
            vert = sequence_vertical(seq)
            log = dict(pairs=0, candidates=0, reasons=Counter(), crates=0,
                       plane_reference="motion-capture vertical" if vert is not None else "camera up",
                       vertical_axis=None if vert is None else vert.tolist(), camera_height_m=[], tilt_to_vertical_deg=[])
            for pk, (i, j, b, ang, werr) in enumerate(iter_static_pairs(seq, np.random.default_rng(seed_for("rabl-pairs", name)))):
                if log["pairs"] >= RUN_CONFIG["n_rabl_pairs_per_seq"] or log["candidates"] >= RABL_DESIGN["max_candidates_per_seq"]:
                    break
                log["candidates"] += 1
                res, why = build_rabl_pair(seq, i, j, RUN_CONFIG["rabl_movers_per_pair"], seed_for("rabl", name, pk), vert)
                if res is None:
                    log["reasons"][why] += 1
                    continue
                scenes, renders, plane = res
                log["camera_height_m"].append(round(plane["height"], 3))
                log["tilt_to_vertical_deg"].append(round(plane["tilt_to_vertical_deg"], 2) if np.isfinite(plane["tilt_to_vertical_deg"]) else None)
                for cond in RABL_DESIGN["conditions"]:
                    d = os.path.join(RENDER_DIR, "rabl", f"{name}_{log['pairs']:03d}_{cond}")
                    os.makedirs(d, exist_ok=True)
                    for k, r in enumerate(renders[cond]):
                        _save_frame(d, k, r)
                    sc = scenes[cond]
                    gt01 = mover_gt_stats(sc, renders[cond], 0, 1)
                    meta = dict(scene=scene_to_json(sc), gt01=gt01, gt0last=gt01, set="RABL", index=_n_base, condition=cond,
                                seq=name, frames=[i, j], baseline=b, rot_deg=ang, warp_err=werr, dir=d, complete=True,
                                contact_px=[[rabl_contact_pixels(sc, renders[cond][k], k, jj) for k in (0, 1)]
                                            for jj in range(len(sc["movers"]))])
                    with open(os.path.join(d, "scene.json"), "w") as f:
                        json.dump(meta, f)
                log["crates"] += len(scenes["G"]["movers"])
                log["pairs"] += 1; _n_base += 1
            log["reasons"] = dict(log["reasons"])
            _log_out[name] = log
        with open(RABL_BUILD_LOG_PATH, "w") as f:
            json.dump(_log_out, f, indent=1)
        if all(DATASET_STATUS[n]["downloaded"] for n, v in TUM_SEQUENCES.items() if v["role"] == "static"):
            RABL_STAGE.commit([os.path.join(RENDER_DIR, "rabl"), RABL_BUILD_LOG_PATH])
        else:
            print("E5-REAL is incomplete (a static TUM sequence is missing); it is not checkpointed.")
META_RABL = sorted(_load_metas(os.path.join(RENDER_DIR, "rabl")), key=lambda m: (m["index"], m["condition"])) \
    if os.path.isdir(os.path.join(RENDER_DIR, "rabl")) else []
MOVERS_RABL = _regimes(_mover_table(META_RABL)) if META_RABL else pd.DataFrame()
if len(MOVERS_RABL):
    MOVERS_RABL["seq"] = MOVERS_RABL["scene"].map({m["index"]: m["seq"] for m in META_RABL})
_rlog = json.load(open(RABL_BUILD_LOG_PATH)) if os.path.exists(RABL_BUILD_LOG_PATH) else {}
for k, v in _rlog.items():
    _h = [x for x in v.get("camera_height_m", []) if x is not None]
    print(f"{k}: {v['pairs']} pairs with {v['crates']} crates from {v['candidates']} candidate pairs; skipped: {v['reasons']}")
    if _h:
        print(f"    support plane referenced to the {v['plane_reference']}; camera {np.min(_h):.2f}-{np.max(_h):.2f} m above it "
              f"(median {np.median(_h):.2f} m)")
if len(MOVERS_RABL):
    _g = MOVERS_RABL[MOVERS_RABL["condition"] == "G"]
    print(f"E5-REAL: {len(_g)} crates x {len(RABL_DESIGN['conditions'])} conditions in {MOVERS_RABL['scene'].nunique()} pairs; "
          f"contact visible when grounded: {int((_g['contact_px0'] >= 5).sum())}/{len(_g)}; "
          f"measurable and identifiable in both conditions: "
          f"{int(MOVERS_RABL.groupby('uid').apply(lambda d: bool((d['measurable'] & d['identifiable']).all())).sum())}")
else:
    print("E5-REAL: no composite could be built (no TUM static sequence or no support plane); it is left out of the run.")
release_memory(); _ = mem_report("E5-REAL composites")

In [ ]:
# Cell No: 028
# 3.6 Optical flow used by (i) the classical triangulation positive control, (ii) the
#     flow-based dynamic masks of Section 4, and (iii) the ground truth of real walking people
#     (Section 3.7). RAFT-large (torchvision weights) is the estimator; if its weights cannot be
#     fetched, OpenCV's DIS flow is used. RAFT is loaded only when some flow must be computed;
#     flows restored from a checkpoint keep the estimator they were computed with.
FLOW = {"method": None, "model": None, "transforms": None, "preferred": FLOW_PREFERRED}

def _ensure_flow_model():
    if FLOW["method"] is not None:
        return
    try:
        from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
        _w = Raft_Large_Weights.DEFAULT
        FLOW["model"] = raft_large(weights=_w, progress=False).eval()
        FLOW["transforms"] = _w.transforms()
        FLOW["method"] = f"RAFT-large (torchvision {_w.name})"
    except Exception as e:
        FLOW["method"] = "DIS (OpenCV, preset MEDIUM)"
        FLOW["fallback_reason"] = f"{type(e).__name__}: {str(e)[:160]}"
    print("Optical flow estimator loaded:", FLOW["method"])

def _flow_dis(a, b):
    ga, gb = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
    return dis.calc(ga, gb, None).astype(np.float32)

@torch.no_grad()
def _flow_raft(pairs):
    model = FLOW["model"].to(device)
    out = []
    for a, b in pairs:
        H, W = a.shape[:2]
        H8, W8 = (H + 7) // 8 * 8, (W + 7) // 8 * 8           # RAFT needs sizes divisible by 8
        pad = ((0, H8 - H), (0, W8 - W), (0, 0))
        ta = torch.from_numpy(np.pad(a, pad, mode="edge").copy()).permute(2, 0, 1)[None]
        tb = torch.from_numpy(np.pad(b, pad, mode="edge").copy()).permute(2, 0, 1)[None]
        ta, tb = FLOW["transforms"](ta, tb)
        flows = model(ta.to(device), tb.to(device), num_flow_updates=12)
        out.append(flows[-1][0].permute(1, 2, 0).float().cpu().numpy()[:H, :W])
    return out

def compute_flows(pairs, cache_paths):
    # pairs: list of (img_a, img_b) uint8; returns list of (H,W,2) flows a->b (cached as fp16)
    res, todo = [None] * len(pairs), []
    for n, p in enumerate(cache_paths):
        if os.path.exists(p):
            res[n] = np.load(p)["flow"].astype(np.float32)
        else:
            todo.append(n)
    if todo:
        _ensure_flow_model()
        if FLOW["model"] is not None:
            for start in range(0, len(todo), 16):
                chunk = todo[start:start + 16]
                for n, f in zip(chunk, _flow_raft([pairs[n] for n in chunk])):
                    res[n] = f
            FLOW["model"].to("cpu")
            release_memory() if "release_memory" in globals() else torch.cuda.empty_cache()
        else:
            for n in todo:
                res[n] = _flow_dis(*pairs[n])
        for n in todo:
            os.makedirs(os.path.dirname(cache_paths[n]), exist_ok=True)
            np.savez_compressed(cache_paths[n], flow=res[n].astype(np.float16))
    return res

def sample_map(arr, pts):
    # bilinear lookup of an (H,W[,C]) map at float pixel positions (N,2); NaN outside
    H, W = arr.shape[:2]
    x, y = pts[:, 0], pts[:, 1]
    ok = (x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)
    xc, yc = np.clip(x, 0, W - 1), np.clip(y, 0, H - 1)
    x0, y0 = np.floor(xc).astype(int), np.floor(yc).astype(int)
    x1, y1 = np.minimum(x0 + 1, W - 1), np.minimum(y0 + 1, H - 1)
    fx, fy = xc - x0, yc - y0
    if arr.ndim == 3:
        fx, fy = fx[:, None], fy[:, None]
    v = (arr[y0, x0] * (1 - fx) * (1 - fy) + arr[y0, x1] * fx * (1 - fy)
         + arr[y1, x0] * (1 - fx) * fy + arr[y1, x1] * fx * fy)
    v = v.astype(np.float64)
    v[~ok] = np.nan
    return v

FLOW_FP = {}

def flow_stage(name, metas, upstream):
    # flows of frames 0 -> 1 for every meta (one checkpointed stage); returns {meta dir: flow file}
    st = Stage(f"flow:{name}", "flow", dict(preferred=FLOW_PREFERRED), upstream=upstream)
    FLOW_FP[name] = st.fp
    paths = {m["dir"]: os.path.join(FLOW_DIR, name.lower(), f"{os.path.basename(m['dir'])}_01.npz") for m in metas}
    method_file = os.path.join(FLOW_DIR, f"method_{name.lower()}.json")
    if not st.done():
        pairs = [(load_rgb(m["dir"], 0), load_rgb(m["dir"], 1)) for m in metas]
        compute_flows(pairs, [paths[m["dir"]] for m in metas])
        with open(method_file, "w") as f:
            json.dump({"method": FLOW["method"], "fallback_reason": FLOW.get("fallback_reason")}, f)
        st.commit(list(paths.values()) + [method_file])
    with open(method_file) as f:
        return paths, json.load(f)["method"]

with stage("flow"):
    _t0 = time.time()
    FLOW_CACHE, _m1 = flow_stage("MAIN", META_MAIN, [RENDER_FP["MAIN"]])
    FLOW_METHOD = {"MAIN": _m1}
    if META_REAL:
        _p, _m2 = flow_stage("REAL", META_REAL, [TUM_STAGES["REAL"].fp])
        FLOW_CACHE.update(_p); FLOW_METHOD["REAL"] = _m2
FLOW["method_used"] = FLOW_METHOD
print(f"flows ready for {len(FLOW_CACHE)} pairs ({time.time() - _t0:.0f}s); estimator per set: {FLOW_METHOD}")
FLOW_NAME = sorted(set(FLOW_METHOD.values()))[0].split(" (")[0] if FLOW_METHOD else "flow"

def load_flow(meta):
    return np.load(FLOW_CACHE[meta["dir"]])["flow"].astype(np.float32)

# flow accuracy on CONVOY-3D static pixels (end-point error against the exact correspondence)
_epe = []
for meta in META_MAIN[: min(24, len(META_MAIN))]:
    sc = meta["scene"]; r0 = load_frame(meta["dir"], 0); r1 = load_frame(meta["dir"], 1)
    pB, _, _, vis = gt_correspondence(sc, r0, 0, 1, r1)
    H, W = r0["depth"].shape
    vv, uu = np.mgrid[0:H, 0:W]
    fl = load_flow(meta)
    gtf = pB - np.stack([uu, vv], -1)
    m = vis & (r0["mover_id"] < 0) & np.isfinite(r0["depth"])
    _epe.append(np.linalg.norm(fl[m] - gtf[m], axis=-1))
_epe = np.concatenate(_epe)
FLOW["epe_static_convoy3d_median"] = float(np.median(_epe))
FLOW["epe_static_convoy3d_p90"] = float(np.percentile(_epe, 90))
print(f"{FLOW_METHOD['MAIN']}: end-point error on CONVOY-3D static pixels: median "
      f"{FLOW['epe_static_convoy3d_median']:.2f} px, 90th percentile {FLOW['epe_static_convoy3d_p90']:.2f} px")

release_memory(); _ = mem_report("optical flow")

In [ ]:
# Cell No: 029
# 3.7 Real moving people (TUM fr3 walking): how often does real motion stay on its epipolar line?
#     A pixel is MOVING if its measured depth or its flow disagrees with the rigid motion implied
#     by the camera pose (|dZ| > max(0.10 m, 8%) or flow residual > 3 px), with forward-backward
#     flow consistency, away from depth edges, inside a connected region of >= 400 px. For every
#     moving pixel: eps (exact epipolar offset of its flow correspondence under the true pose)
#     and Z_tri (static triangulation of that correspondence). Only the prevalence of the motion
#     regimes is reported; the models are not evaluated on these pairs (Section 0.2, D24).

def select_dynamic_pairs(seq, n_pairs, rng, b_range=(0.05, 0.25), max_angle=20.0):
    pairs, n = [], len(seq)
    starts = np.linspace(0, n - 2, num=max(3 * n_pairs, 6)).astype(int)
    rng.shuffle(starts)
    for i in starts:
        target = rng.uniform(*b_range)
        for j in range(i + 1, min(n, i + 120)):
            b, ang = baseline_and_angle(seq, i, j)
            if b >= target:
                if ang <= max_angle and b <= b_range[1] + 0.05:
                    pairs.append((int(i), int(j), float(b), float(ang)))
                break
        if len(pairs) >= n_pairs:
            break
    return sorted(pairs)

def real_dynamic_ground_truth(seq, i, j, f_ij, f_ji):
    cam = seq.cam; H, W = cam.H, cam.W
    di, dj = seq.depth(i), seq.depth(j)
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    vv, uu = np.mgrid[0:H, 0:W]
    p = np.stack([uu, vv], -1).reshape(-1, 2).astype(np.float64)
    z = di.reshape(-1).astype(np.float64)
    valid = np.isfinite(z)
    X = cam.rays(p[:, 0], p[:, 1]) * np.where(valid, z, 1.0)[:, None]
    Xj = (X @ Ri.T + Ci - Cj) @ Rj
    with np.errstate(divide="ignore", invalid="ignore"):
        p_rig = cam.project(Xj)
    p_obs = p + f_ij.reshape(-1, 2)
    zj_obs = sample_map(np.nan_to_num(dj, nan=0.0), p_rig)
    zj_obs[zj_obs <= 0] = np.nan
    back = sample_map(f_ji, p_obs)
    fb_ok = np.linalg.norm(f_ij.reshape(-1, 2) + back, axis=1) < 1.0 + 0.05 * np.linalg.norm(f_ij.reshape(-1, 2), axis=1)
    flow_res = np.linalg.norm(p_obs - p_rig, axis=1)
    dz = np.abs(zj_obs - Xj[:, 2])
    depth_bad = dz > np.maximum(0.10, 0.08 * Xj[:, 2])
    vimg = np.isfinite(di).astype(np.float32)
    nvalid = cv2.boxFilter(vimg, -1, (3, 3), normalize=False)
    gy, gx = np.gradient(np.nan_to_num(di, nan=0.0))
    edge = (((np.hypot(gx, gy) > 0.08) & (nvalid >= 9)) | (nvalid < 7)).reshape(-1)
    inside = np.isfinite(p_obs).all(1) & (p_obs[:, 0] >= 0) & (p_obs[:, 0] <= W - 1) & (p_obs[:, 1] >= 0) & (p_obs[:, 1] <= H - 1)
    base = valid & fb_ok & ~edge & inside & np.isfinite(zj_obs)
    dyn = base & (depth_bad | (flow_res > 3.0))
    dyn_img = cv2.morphologyEx(dyn.reshape(H, W).astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n_cc, lab, stats, _ = cv2.connectedComponentsWithStats(dyn_img, connectivity=8)
    keep = np.zeros(n_cc, bool); keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= 400
    dyn = keep[lab].reshape(-1) & base
    R21, t21 = relative_pose(Ri, Ci, Rj, Cj)
    F = fundamental_matrix(cam.K, cam.K, R21, t21)
    idx = np.nonzero(dyn)[0]
    eps = np.abs(epipolar_offset(p[idx], p_obs[idx], F))
    ztri, _ = triangulate_depth(p[idx], p_obs[idx], cam.K, cam.K, R21, t21)
    par = translational_parallax(p[idx], p_obs[idx], cam.K, cam.K, R21)
    return dict(eps=eps.astype(np.float32), z_tri=ztri.astype(np.float32), parallax=par.astype(np.float32))

DYN_DIR = os.path.join(RENDER_DIR, "realdyn")
PREV_TUM_PATH = os.path.join(DYN_DIR, "prevalence_tum.csv")
DYN_PAIRS_PATH = os.path.join(DYN_DIR, "tum_dynamic_pairs.csv")
with stage("real_dynamic_gt"):
    _st = TUM_STAGES["DYN"]
    if not _st.done():
        os.makedirs(DYN_DIR, exist_ok=True)
        _rows, _prev = [], []
        for name, seq in TUM.items():
            if DATASET_STATUS[name]["role"] != "dynamic":
                continue
            rng_d = np.random.default_rng(seed_for("dyn-pairs", name))
            pairs = select_dynamic_pairs(seq, RUN_CONFIG["n_dyn_candidate_pairs_per_seq"], rng_d)
            imgs = {k: seq.rgb(k) for k in sorted({x for p_ in pairs for x in p_[:2]})}
            fw = compute_flows([(imgs[i], imgs[j]) for i, j, _, _ in pairs],
                               [os.path.join(FLOW_DIR, "dyn", f"{name}_{i}_{j}.npz") for i, j, _, _ in pairs])
            bw = compute_flows([(imgs[j], imgs[i]) for i, j, _, _ in pairs],
                               [os.path.join(FLOW_DIR, "dyn", f"{name}_{j}_{i}.npz") for i, j, _, _ in pairs])
            eps_all, zt_all = [], []
            for pk, ((i, j, b, ang), f1, f2) in enumerate(zip(pairs, fw, bw)):
                g = real_dynamic_ground_truth(seq, i, j, f1, f2)
                eps_all.append(g["eps"]); zt_all.append(g["z_tri"])
                _rows.append(dict(seq=name, index=pk, i=i, j=j, baseline=b, rot_deg=ang, n_dyn=int(len(g["eps"])),
                                  n_parallel=int(np.sum(g["eps"] < PREREG["eps_parallel_max"])),
                                  n_perp=int(np.sum(g["eps"] >= PREREG["eps_perp_min"]))))
            e, zt = np.concatenate(eps_all), np.concatenate(zt_all)
            _prev.append(dict(dataset="TUM fr3 walking", seq=name, n_pairs=len(pairs), n_moving_pixels=int(len(e)),
                              frac_parallel=float(np.mean(e < PREREG["eps_parallel_max"])),
                              frac_intermediate=float(np.mean((e >= PREREG["eps_parallel_max"]) & (e < PREREG["eps_perp_min"]))),
                              frac_perpendicular=float(np.mean(e >= PREREG["eps_perp_min"])),
                              frac_parallel_no_static_solution=float(np.mean(zt[e < 1] <= 0)) if np.any(e < 1) else float("nan"),
                              median_eps=float(np.median(e))))
            Image.fromarray(seq.rgb(len(seq) // 2)).save(os.path.join(DYN_DIR, f"thumb_{name}.png"))
            del imgs
        pd.DataFrame(_rows).to_csv(DYN_PAIRS_PATH, index=False)
        pd.DataFrame(_prev).to_csv(PREV_TUM_PATH, index=False)
        if all(DATASET_STATUS[n]["downloaded"] for n, v in TUM_SEQUENCES.items() if v["role"] == "dynamic"):
            _st.commit([DYN_DIR], extra=dict(flow=FLOW.get("method")))
        else:
            print("The walking ground truth is incomplete (a dynamic TUM sequence is missing); it is not checkpointed.")

PREVALENCE = pd.read_csv(PREV_TUM_PATH) if os.path.exists(PREV_TUM_PATH) else pd.DataFrame()
DYN_PAIRS = pd.read_csv(DYN_PAIRS_PATH) if os.path.exists(DYN_PAIRS_PATH) else pd.DataFrame()
TUM_THUMBS = {os.path.basename(p)[len("thumb_"):-4]: p for p in sorted(glob.glob(os.path.join(DYN_DIR, "thumb_*.png")))}
if len(PREVALENCE):
    print(PREVALENCE.drop(columns=["dataset"]).to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
    PREVALENCE.to_csv(os.path.join(RESULTS_DIR, "prevalence_tum.csv"), index=False)

FLOW["model"] = None                                 # RAFT is not used after this cell
release_memory(); _ = mem_report("TUM walking prevalence")

In [ ]:
# Cell No: 030
# 3.8 Virtual KITTI 2 (Cabon et al., 2020): photo-realistic driving with exact depth, camera
#     extrinsics and per-vehicle 3-D poses, downloadable without login. The image archives are
#     several gigabytes each, so only the frames this run uses are fetched: the compact text
#     ground truth (vehicle and camera poses) is read first, the frame pairs are chosen
#     from it, and the matching image, depth and instance files are read from the remote tar
#     archives with HTTP range requests (one streamed pass over an archive is the fallback when
#     the server ignores ranges). The stage is one checkpoint. If any part fails, Virtual KITTI 2
#     is removed from every experiment, table and figure.
import tarfile as _tarfile
import urllib.error

VK_FILES = {"textgt": "vkitti_2.0.3_textgt.tar.gz", "rgb": "vkitti_2.0.3_rgb.tar", "depth": "vkitti_2.0.3_depth.tar",
            "instance": "vkitti_2.0.3_instanceSegmentation.tar"}
VK_SCENES = VK_DESIGN["scenes"][:RUN_CONFIG["n_vk_scenes"]]
VK_RAW = os.path.join(VK_DIR, "raw")
_VK_MEMBER = re.compile(r"(?:^|/)(Scene\d\d)/([A-Za-z0-9_-]+)/frames/([A-Za-z]+)/Camera_(\d)/[A-Za-z]+_(\d{5})\.(jpg|png)$")


class RangeUnsupported(Exception):
    pass


class RemoteTar:
    """Random access to an uncompressed tar archive over HTTP (range requests)."""

    def __init__(self, url, timeout=90, retries=4):
        self.url, self.timeout, self.retries = url, timeout, retries
        self.n_requests, self.n_bytes = 0, 0
        data, total = self._get(0, 511, want_total=True)
        self.size = total

    def _get(self, start, end, want_total=False):
        err = None
        for attempt in range(self.retries):
            try:
                req = urllib.request.Request(self.url, headers={"User-Agent": "Mozilla/5.0", "Range": f"bytes={start}-{end}"})
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeUnsupported(f"HTTP {r.status} to a range request")
                    data = r.read()
                    cr = r.headers.get("Content-Range", "")
                self.n_requests += 1; self.n_bytes += len(data)
                if want_total:
                    return data, int(cr.rsplit("/", 1)[1])
                return data
            except RangeUnsupported:
                raise
            except Exception as e:
                err = e
                time.sleep(2 + 3 * attempt)
        raise IOError(f"range request failed: {type(err).__name__}: {err}")

    @staticmethod
    def parse_header(block):
        if len(block) < 512 or block[257:262] != b"ustar":
            return None
        chk = int(block[148:156].split(b"\0", 1)[0].strip() or b"0", 8)
        if chk != sum(block[:148]) + 8 * 32 + sum(block[156:512]):
            return None
        name = block[0:100].split(b"\0", 1)[0].decode("utf-8", "replace")
        prefix = block[345:500].split(b"\0", 1)[0].decode("utf-8", "replace")
        if prefix:
            name = prefix + "/" + name
        sz = block[124:136]
        size = int.from_bytes(sz[1:], "big") if sz[0] & 0x80 else int(sz.split(b"\0", 1)[0].strip() or b"0", 8)
        return dict(name=name, size=size, type=chr(block[156]) if block[156] else "0")

    def header_at_or_after(self, offset, window=1 << 18, max_windows=24):
        o = max(0, offset - offset % 512)
        for _ in range(max_windows):
            if o >= self.size:
                return None
            buf = self._get(o, min(o + window, self.size) - 1)
            for b in range(0, len(buf) - 511, 512):
                h = self.parse_header(buf[b:b + 512])
                if h and h["type"] in "0\0":
                    return o + b, h
            o += window
        return None

    def next_header(self, off, h):
        return off + 512 + (h["size"] + 511) // 512 * 512

    def iter_members(self, start, end):
        # stream [start, end) and yield (name, bytes) of the regular files in it
        req = urllib.request.Request(self.url, headers={"User-Agent": "Mozilla/5.0", "Range": f"bytes={start}-{end - 1}"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            if r.status != 206:
                raise RangeUnsupported(f"HTTP {r.status} to a range request")
            yield from _iter_tar_stream(r)


def _read_exact(f, n):
    out = bytearray()
    while len(out) < n:
        chunk = f.read(n - len(out))
        if not chunk:
            break
        out += chunk
    return bytes(out)


def _iter_tar_stream(f):
    long_name = None
    while True:
        block = _read_exact(f, 512)
        if len(block) < 512 or block == b"\0" * 512:
            return
        h = RemoteTar.parse_header(block)
        if h is None:
            return
        data = _read_exact(f, h["size"])
        _read_exact(f, (512 - h["size"] % 512) % 512)
        if h["type"] == "L":
            long_name = data.split(b"\0", 1)[0].decode("utf-8", "replace"); continue
        if h["type"] in ("x", "g"):
            for rec in data.decode("utf-8", "replace").split("\n"):
                if " path=" in rec:
                    long_name = rec.split(" path=", 1)[1]
            continue
        name = long_name or h["name"]; long_name = None
        if h["type"] in "0\0":
            yield name, data


def _member_key(name):
    m = _VK_MEMBER.search(name)
    if not m:
        return None
    scene, variation, folder, cam, frame, _ = m.groups()
    return scene, variation, folder, int(cam), int(frame)


def vk_fetch_frames(url, wanted, label, log):
    # wanted: {(scene, frame)} for variation clone / camera 0 -> {(scene, frame): bytes}
    out = {}
    need_scenes = sorted({s for s, _ in wanted})
    try:
        rt = RemoteTar(url)
        # coarse probes: which directory lives where in the archive
        n_probe = int(np.clip(rt.size // (24 << 20), 64, 480))
        probes = []
        for off in np.linspace(0, max(rt.size - 1024, 0), n_probe).astype(np.int64):
            hit = rt.header_at_or_after(int(off))
            if hit:
                probes.append((hit[0], _member_key(hit[1]["name"]), hit[1]))
        probes = sorted({p[0]: p for p in probes}.values())

        def in_dir(key, scene):
            return key is not None and key[0] == scene and key[1] == VK_DESIGN["variation"] and key[3] == VK_DESIGN["camera"]

        for scene in need_scenes:
            idx = [i for i, p in enumerate(probes) if in_dir(p[1], scene)]
            if not idx:
                raise RangeUnsupported(f"{scene} not found by the probes")
            # start: bisect between the last probe before the directory and its first probe
            a = probes[idx[0] - 1][0] if idx[0] > 0 else 0
            b = probes[idx[0]][0]
            while b - a > (1 << 18):
                mid = (a + b) // 2
                hit = rt.header_at_or_after(mid)
                if hit is None or hit[0] >= b:
                    a = mid
                elif in_dir(_member_key(hit[1]["name"]), scene):
                    b = hit[0]
                else:
                    a = hit[0]
            off, h = rt.header_at_or_after(a)
            while not in_dir(_member_key(h["name"]), scene):
                off = rt.next_header(off, h)
                h = rt.parse_header(rt._get(off, off + 511)) if off + 512 <= rt.size else None
                if h is None:
                    raise IOError(f"{scene}: tar walk ended before the directory")
            start = off
            # end: bisect between the last probe inside and the first probe after the directory
            a = probes[idx[-1]][0]
            b = probes[idx[-1] + 1][0] if idx[-1] + 1 < len(probes) else rt.size
            while b - a > (1 << 18):
                mid = (a + b) // 2
                hit = rt.header_at_or_after(mid)
                if hit is None or hit[0] >= b:
                    b = mid
                elif in_dir(_member_key(hit[1]["name"]), scene):
                    a = hit[0]
                else:
                    b = hit[0]
            off, h = rt.header_at_or_after(a)
            while h is not None and in_dir(_member_key(h["name"]), scene):
                off = rt.next_header(off, h)
                h = rt.parse_header(rt._get(off, off + 511)) if off + 512 <= rt.size else None
            end = off
            todo = {f for s_, f in wanted if s_ == scene}
            for name, data in rt.iter_members(start, end):
                key = _member_key(name)
                if in_dir(key, scene) and key[4] in todo:
                    out[(scene, key[4])] = data
                    todo.discard(key[4])
                    if not todo:
                        break                               # every wanted frame of this scene is in hand
        log[label] = dict(mode="range requests", requests=rt.n_requests, MiB=round(rt.n_bytes / 2**20, 1))
    except (RangeUnsupported, IOError, ValueError, TypeError) as e:
        # fallback: one streamed pass over the whole archive, stopping when everything is found
        log[label] = dict(mode=f"streamed ({type(e).__name__}: {str(e)[:80]})")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=120) as r:
            for name, data in _iter_tar_stream(r):
                key = _member_key(name)
                if key and key[1] == VK_DESIGN["variation"] and key[3] == VK_DESIGN["camera"] and (key[0], key[4]) in wanted:
                    out[(key[0], key[4])] = data
                    if len(out) == len(wanted):
                        break
    return out


# ---------------- text ground truth ----------------
VK_COLUMNS = {   # documented column order of the Virtual KITTI 2 text ground truth (header row first)
    "extrinsic": ["frame", "cameraID"] + [f"m{r}{c}" for r in range(4) for c in range(4)],
    "intrinsic": ["frame", "cameraID", "fx", "fy", "cx", "cy"],
    "pose": ["frame", "cameraID", "trackID", "alpha", "width", "height", "length", "world_space_X", "world_space_Y",
             "world_space_Z", "rotation_world_space_y", "rotation_world_space_x", "rotation_world_space_z",
             "camera_space_X", "camera_space_Y", "camera_space_Z", "rotation_camera_space_y",
             "rotation_camera_space_x", "rotation_camera_space_z"],
    "bbox": ["frame", "cameraID", "trackID", "left", "right", "top", "bottom", "number_pixels", "truncation_ratio",
             "occupancy_ratio", "isMoving"],
    "info": ["trackID", "label", "model", "color"],
}

def _read_vk_table(text, kind):
    lines = [l.split() for l in text.splitlines() if l.strip()]
    head, rows = lines[0], lines[1:]
    cols = VK_COLUMNS[kind]
    if head[0] not in ("frame", "trackID") or any(len(r) < len(cols) for r in rows):
        raise ValueError(f"{kind}.txt: unexpected layout (header {head[:4]})")
    df = pd.DataFrame([r[:len(cols)] for r in rows], columns=cols)
    for c in cols:
        if c not in ("label", "model", "color", "isMoving"):
            df[c] = pd.to_numeric(df[c])
    return df


def vk_parse_textgt(scene_dir):
    tg = {}
    for k in ("extrinsic", "intrinsic", "pose", "bbox", "info"):
        with open(os.path.join(scene_dir, f"{k}.txt")) as f:
            tg[k] = _read_vk_table(f.read(), k)
    for k in ("extrinsic", "intrinsic", "pose", "bbox"):
        tg[k] = tg[k][tg[k]["cameraID"].astype(int) == VK_DESIGN["camera"]].reset_index(drop=True)
    tg["bbox"]["isMoving"] = tg["bbox"]["isMoving"].astype(str).str.lower().isin(["true", "1"])
    return tg


def vk_camera_centres(tg, inverted=False):
    # camera centres C_k (world) for camera 0, assuming extrinsic = world-to-camera (or its inverse)
    E = tg["extrinsic"].sort_values("frame")
    M = E[[f"m{r}{c}" for r in range(4) for c in range(4)]].to_numpy(np.float64).reshape(-1, 4, 4)
    Rs, Cs = [], []
    for T in M:
        if inverted:
            T = np.linalg.inv(T)
        Rwc = T[:3, :3]; t = T[:3, 3]
        Rs.append(Rwc.T); Cs.append(-Rwc.T @ t)
    return E["frame"].to_numpy(int), np.array(Rs), np.array(Cs)


def vk_select(tg, scene, rng):
    # frame pairs chosen from the text ground truth alone (before any image is read)
    frames, Rs, Cs = vk_camera_centres(tg)
    fidx = {f: n for n, f in enumerate(frames)}
    W_full = int(tg["intrinsic"]["cx"].iloc[0] * 2 + 1)
    x0 = int(round(tg["intrinsic"]["cx"].iloc[0] - (VK_DESIGN["crop_wh"][0] - 1) / 2))
    bb = tg["bbox"]
    bb = bb.assign(cl=np.clip(bb["left"], x0, x0 + VK_DESIGN["crop_wh"][0] - 1),
                   cr=np.clip(bb["right"], x0, x0 + VK_DESIGN["crop_wh"][0] - 1))
    bb["area_crop"] = (bb["cr"] - bb["cl"]).clip(lower=0) * (bb["bottom"] - bb["top"]).clip(lower=0)
    mov = bb[bb["isMoving"] & (bb["area_crop"] >= VK_DESIGN["min_vehicle_px"])]
    frames_with_movers = set(mov["frame"].astype(int))
    def rot_deg(i, j):
        R = Rs[fidx[i]].T @ Rs[fidx[j]]
        return float(np.rad2deg(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1))))
    def base(i, j):
        return float(np.linalg.norm(Cs[fidx[j]] - Cs[fidx[i]]))
    starts = [int(f) for f in frames if int(f) in frames_with_movers]
    rng.shuffle(starts)
    pairs, used = [], set()
    for i in starts:
        target = rng.uniform(*VK_DESIGN["pair_baseline_m"])
        for j in range(i + 1, i + VK_DESIGN["max_gap_frames"] + 1):
            if j not in fidx:
                break
            if base(i, j) >= target:
                if base(i, j) <= VK_DESIGN["pair_baseline_m"][1] + 0.5 and rot_deg(i, j) <= VK_DESIGN["max_rot_deg"] \
                   and not any(abs(i - u) < 3 for u in used):
                    pairs.append((int(i), int(j), base(i, j), rot_deg(i, j))); used.add(i)
                break
        if len(pairs) >= RUN_CONFIG["vk_pairs_per_scene"]:
            break
    return sorted(pairs), dict(x0=x0, width_full=W_full)


VK_STAGE = Stage("vk:data", "vk", dict(files=VK_FILES, scenes=VK_SCENES, design={k: v for k, v in VK_DESIGN.items() if k != "base_url"},
                                       pairs=RUN_CONFIG["vk_pairs_per_scene"]))
VK_SELECTION_PATH = os.path.join(VK_RAW, "selection.json")
VK_STATUS = dict(available=False, note="", download_log={})
with stage("download_vkitti"):
    if VK_STAGE.done():
        VK_STATUS.update(available=True, note=f"restored from checkpoint ({STAGE_LOG[-1]['source']})")
    else:
        try:
            shutil.rmtree(VK_RAW, ignore_errors=True); os.makedirs(VK_RAW, exist_ok=True)
            tgz = os.path.join(DATA_DIR, VK_FILES["textgt"])
            ok, msg = (True, "cached") if os.path.exists(tgz) else download_file(VK_DESIGN["base_url"] + VK_FILES["textgt"], tgz)
            if not ok:
                raise IOError(f"text ground truth: {msg}")
            pat = re.compile(r"(?:^|/)(Scene\d\d)/" + VK_DESIGN["variation"] + r"/(extrinsic|intrinsic|pose|bbox|info)\.txt$")
            with _tarfile.open(tgz) as tf:
                for m in tf.getmembers():
                    mm = pat.search(m.name)
                    if m.isfile() and mm and mm.group(1) in VK_SCENES:
                        d = os.path.join(VK_RAW, mm.group(1)); os.makedirs(d, exist_ok=True)
                        with tf.extractfile(m) as src, open(os.path.join(d, mm.group(2) + ".txt"), "wb") as out:
                            out.write(src.read())
            selection, wanted = {}, set()
            for scene in VK_SCENES:
                sd = os.path.join(VK_RAW, scene)
                if not all(os.path.exists(os.path.join(sd, f"{k}.txt")) for k in ("extrinsic", "intrinsic", "pose", "bbox", "info")):
                    raise IOError(f"{scene}: text ground truth incomplete")
                tg = vk_parse_textgt(sd)
                pairs, geo = vk_select(tg, scene, np.random.default_rng(seed_for("vk-select", scene)))
                selection[scene] = dict(pairs=pairs, **geo)
                wanted |= {(scene, f) for p in pairs for f in p[:2]}
            log = {}
            for kind, prefix, ext in (("rgb", "rgb", "jpg"), ("depth", "depth", "png"), ("instance", "inst", "png")):
                got = vk_fetch_frames(VK_DESIGN["base_url"] + VK_FILES[kind], wanted, kind, log)
                missing = wanted - set(got)
                if missing:
                    raise IOError(f"{kind}: {len(missing)} of {len(wanted)} frames not found in the archive")
                for (scene, f), data in got.items():
                    with open(os.path.join(VK_RAW, scene, f"{prefix}_{f:05d}.{ext}"), "wb") as out:
                        out.write(data)
            with open(VK_SELECTION_PATH, "w") as f:
                json.dump(dict(selection=selection, download_log=log), f, indent=1,
                          default=lambda o: o.item() if hasattr(o, "item") else str(o))
            VK_STAGE.commit([VK_RAW])
            VK_STATUS.update(available=True, note="downloaded", download_log=log)
        except Exception as e:
            VK_STATUS.update(available=False, note=f"{type(e).__name__}: {str(e)[:300]}")
            shutil.rmtree(VK_RAW, ignore_errors=True)

if VK_STATUS["available"]:
    with open(VK_SELECTION_PATH) as f:
        _sel = json.load(f)
    VK_SELECTION = _sel["selection"]
    VK_STATUS["download_log"] = _sel.get("download_log", {})
    print(f"Virtual KITTI 2: PASS ({VK_STATUS['note']})")
    for k, v in VK_STATUS["download_log"].items():
        print(f"    {k:<9} {v}")
    for sc_, v in VK_SELECTION.items():
        print(f"    {sc_}: {len(v['pairs'])} pairs")
else:
    VK_SELECTION = {}
    print(f"Virtual KITTI 2: FAIL -> removed from every experiment, table and figure ({VK_STATUS['note']})")

In [ ]:
# Cell No: 031
# 3.9 Virtual KITTI 2 ground truth. Frames are cropped to the centred 500 x 375 window (4:3, native
#     pixels, principal point at the crop centre). Conventions that the files do not state are
#     decided from the data, as for TUM (assumption A3): (i) whether extrinsic.txt maps world to
#     camera or the reverse (static depth warped between frames must agree with the measured
#     depth), (ii) the axis and sign of the vehicle yaw (the rigid motion must carry a vehicle's
#     pixels onto its pixels in the other frame), (iii) how instance-mask values map to track IDs
#     (overlap with bbox.txt). Every pixel then gets its exact correspondence: static points stay,
#     moving vehicles move rigidly with their pose. Movers are the vehicles flagged as moving.
#     Two prevalence statistics need no model: the share of moving-vehicle pixels within 1 px of
#     their epipolar line (two views), and the share of moving vehicles whose track over K = 2, 4, 8
#     frames is fitted by a static point to within 1 px (no K-view geometric test can see them).

VK_CROP_W, VK_CROP_H = VK_DESIGN["crop_wh"]

def vk_measurable(df):
    # the CONVOY-3D pixel rule (D34): >= 40 eroded pixels visible in both views, inside the crop, and
    # >= 85% of the vehicle's silhouette not occluded (occupancy ratio of the released ground truth)
    return (df["gt_n_eroded"] >= PREREG["min_object_pixels"]) & (df["gt_occupancy"] >= PREREG["vk_min_occupancy"])

def _rot_axis(axis, a):
    return {"y": rot_y, "z": rot_z, "x": rot_x}[axis](a)

class VKScene:
    def __init__(self, name):
        self.name, self.dir = name, os.path.join(VK_RAW, name)
        self.tg = vk_parse_textgt(self.dir)
        self.sel = VK_SELECTION[name]
        self.x0 = int(self.sel["x0"])
        I = self.tg["intrinsic"].iloc[0]
        self.cam = PinholeCamera(I["fx"], I["fy"], I["cx"] - self.x0, I["cy"], VK_CROP_W, VK_CROP_H)
        self.inverted = False
        self.yaw_axis, self.yaw_sign = "y", 1.0
        self.inst_mode = "track+1"
        self._set_poses()
        P = self.tg["pose"]
        self.pose = {(int(r.frame), int(r.trackID)): r for r in P.itertuples()}
        B = self.tg["bbox"]
        self.bbox = {(int(r.frame), int(r.trackID)): r for r in B.itertuples()}
        self.labels = {int(r.trackID): str(r.label) for r in self.tg["info"].itertuples()}
        self._cache = {}

    def _set_poses(self):
        fr, Rs, Cs = vk_camera_centres(self.tg, inverted=self.inverted)
        self.R = {int(f): R for f, R in zip(fr, Rs)}
        self.C = {int(f): C for f, C in zip(fr, Cs)}

    def cam_dict(self, f):
        return dict(**self.cam.to_dict(), R=self.R[f].tolist(), C=self.C[f].tolist())

    def rgb(self, f):
        return np.asarray(Image.open(os.path.join(self.dir, f"rgb_{f:05d}.jpg")).convert("RGB"))[:, self.x0:self.x0 + VK_CROP_W].copy()

    def depth(self, f):
        d = cv2.imread(os.path.join(self.dir, f"depth_{f:05d}.png"), cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH)
        d = d.astype(np.float32)[:, self.x0:self.x0 + VK_CROP_W] / 100.0
        d[(d <= 0) | (d >= VK_DESIGN["sky_depth_m"])] = np.inf
        return d

    def _inst_raw(self, f):
        a = np.asarray(Image.open(os.path.join(self.dir, f"inst_{f:05d}.png")))
        if a.ndim == 3:
            a = a[..., 0].astype(np.int64) * 65536 + a[..., 1].astype(np.int64) * 256 + a[..., 2]
        return a.astype(np.int64)

    def _match_values(self, f, raw):
        # map instance values to track IDs by overlap with the bbox.txt boxes of frame f
        out = {}
        for v in np.unique(raw):
            if v == 0:
                continue
            vv, uu = np.nonzero(raw == v)
            box = (uu.min(), uu.max(), vv.min(), vv.max())
            best, best_iou = None, 0.2
            for (ff, tid), r in self.bbox.items():
                if ff != f:
                    continue
                ix = min(box[1], r.right) - max(box[0], r.left); iy = min(box[3], r.bottom) - max(box[2], r.top)
                if ix <= 0 or iy <= 0:
                    continue
                inter = ix * iy
                iou = inter / ((box[1] - box[0]) * (box[3] - box[2]) + (r.right - r.left) * (r.bottom - r.top) - inter + 1e-9)
                if iou > best_iou:
                    best, best_iou = tid, iou
            if best is not None:
                out[int(v)] = int(best)
        return out

    def inst(self, f):
        # per-pixel track ID (-1: no vehicle), cropped
        raw = self._inst_raw(f)
        if self.inst_mode == "track+1":
            tid = np.where(raw > 0, raw - 1, -1)
        else:
            m = self._match_values(f, raw)
            tid = np.full(raw.shape, -1, np.int64)
            for v, t in m.items():
                tid[raw == v] = t
        return tid[:, self.x0:self.x0 + VK_CROP_W]

    def obj_transform(self, f, tid):
        r = self.pose.get((f, tid))
        if r is None:
            return None
        Rw = _rot_axis(self.yaw_axis, self.yaw_sign * r.rotation_world_space_y)
        return Rw, np.array([r.world_space_X, r.world_space_Y, r.world_space_Z], np.float64)

    def moving(self, f, tid):
        r = self.bbox.get((f, tid))
        return bool(r.isMoving) if r is not None else False

    def frame_gt(self, f):
        if ("f", f) not in self._cache:
            self._cache[("f", f)] = dict(rgb=self.rgb(f), depth=self.depth(f), inst=self.inst(f))
            if sum(1 for k in self._cache if k[0] == "f") > 24:          # bounded memory
                del self._cache[next(k for k in self._cache if k[0] == "f")]
        return self._cache[("f", f)]

    def pair_gt(self, i, j, tol_rel=0.02, tol_abs=0.05):
        # exact correspondences i -> j (static pixels fixed, moving vehicles rigid), movers, masks
        key = ("p", i, j)
        if key in self._cache:
            return self._cache[key]
        A, B = self.frame_gt(i), self.frame_gt(j)
        H, W = A["depth"].shape
        vv, uu = np.mgrid[0:H, 0:W]
        Ri, Ci, Rj, Cj = self.R[i], self.C[i], self.R[j], self.C[j]
        D = self.cam.rays(uu.astype(np.float64), vv.astype(np.float64)) @ Ri.T
        P = Ci + np.where(np.isfinite(A["depth"]), A["depth"], np.nan)[..., None] * D
        tids = [int(t) for t in np.unique(A["inst"]) if t >= 0 and self.moving(i, int(t))]
        movers, mover_id = [], np.full((H, W), -1, np.int16)
        for t in tids:
            Ti, Tj = self.obj_transform(i, t), self.obj_transform(j, t)
            if Ti is None or Tj is None:
                continue
            m = A["inst"] == t
            Pm = P[m]
            P[m] = (Pm - Ti[1]) @ Ti[0] @ Tj[0].T + Tj[1]          # rigid motion of the vehicle
            mover_id[m] = len(movers)
            movers.append(dict(track=t, label=self.labels.get(t, "?"), pos_i=Ti[1], pos_j=Tj[1]))
        Xc = (P - Cj) @ Rj
        with np.errstate(invalid="ignore", divide="ignore"):
            pB = self.cam.project(Xc)
        zB = Xc[..., 2]
        valid = np.isfinite(pB).all(-1) & (zB > 0.1) & (pB[..., 0] >= 0) & (pB[..., 0] <= W - 1) & (pB[..., 1] >= 0) & (pB[..., 1] <= H - 1)
        ui = np.clip(np.round(np.nan_to_num(pB[..., 0])).astype(int), 0, W - 1)
        vi = np.clip(np.round(np.nan_to_num(pB[..., 1])).astype(int), 0, H - 1)
        visible = valid & (np.abs(B["depth"][vi, ui] - zB) <= tol_rel * zB + tol_abs) & (B["inst"][vi, ui] == A["inst"])
        sc = dict(cams=[self.cam_dict(i), self.cam_dict(j)])
        eps, zgeo, par = pixel_epipolar_quantities(sc, 0, 1, np.stack([uu, vv], -1).astype(np.float64), pB)
        out = dict(pB=pB, zB=zB, valid=valid, visible=visible, eps=eps, zgeo=zgeo, parallax=par,
                   mover_id=mover_id, movers=movers, scene=sc, depth=A["depth"], inst=A["inst"])
        self._cache[key] = out
        if sum(1 for k in self._cache if k[0] == "p") > 12:            # bounded memory
            del self._cache[next(k for k in self._cache if k[0] == "p")]
        return out

    # ---- ground contact of a vehicle (v6.2, exploratory driving analysis) ----------------------
    # The 3-D box of pose.txt is read under four candidate conventions (location at the bottom face
    # or at the box centre; length along the first or the second horizontal axis of the object);
    # the one whose projection best matches the released 2-D boxes is used (decided from the data,
    # like the other conventions here). A wheel contact point is 0.10 m above the bottom face at
    # (+-0.35 length, +-0.40 width); it counts as visible if the vehicle's own pixel is there at the
    # point's depth (within 0.6 m), i.e. neither another object nor the vehicle's own body hides it.
    def _box_geometry(self, f, tid, conv):
        r = self.pose.get((f, tid)); T = self.obj_transform(f, tid)
        if r is None or T is None:
            return None
        Rw, loc = T
        v = {"x": 0, "y": 1, "z": 2}[self.yaw_axis]
        hz = [a for a in range(3) if a != v]
        e_len, e_wid = (Rw[:, hz[0]], Rw[:, hz[1]]) if conv[1] == 0 else (Rw[:, hz[1]], Rw[:, hz[0]])
        down = Rw[:, v] * np.sign(Rw[:, v] @ self.R[f][:, 1] or 1.0)
        w, h, l = float(r.width), float(r.height), float(r.length)
        c = loc - 0.5 * h * down if conv[0] == "bottom" else loc
        return dict(c=c, e_len=e_len, e_wid=e_wid, down=down, w=w, h=h, l=l)

    def _project_world(self, f, X):
        Xc = (np.atleast_2d(X) - self.C[f]) @ self.R[f]
        return self.cam.project(Xc), Xc[:, 2]

    def box_convention(self):
        if hasattr(self, "_box_conv"):
            return self._box_conv
        cands = [(a, b) for a in ("bottom", "centre") for b in (0, 1)]
        keys = [k for k, r in self.bbox.items() if k in self.pose and (r.right - r.left) * (r.bottom - r.top) >= 400
                and r.truncation_ratio < 0.1][:120]
        Wf = 2.0 * float(self.tg["intrinsic"].iloc[0]["cx"])
        best = (None, 0.0)
        for conv in cands:
            ious = []
            for (f, t) in keys:
                if f not in self.R:
                    continue
                g = self._box_geometry(f, t, conv)
                if g is None:
                    continue
                sg = np.array([(a, b, d) for a in (-1, 1) for b in (-1, 1) for d in (-1, 1)], float)
                X = g["c"] + sg[:, :1] * 0.5 * g["l"] * g["e_len"] + sg[:, 1:2] * 0.5 * g["w"] * g["e_wid"] + sg[:, 2:] * 0.5 * g["h"] * g["down"]
                uv, z = self._project_world(f, X)
                if np.any(z < 0.5):
                    continue
                u0, u1 = np.clip([uv[:, 0].min() + self.x0, uv[:, 0].max() + self.x0], 0, Wf - 1)
                v0, v1 = np.clip([uv[:, 1].min(), uv[:, 1].max()], 0, VK_CROP_H - 1)
                r = self.bbox[(f, t)]
                iw = max(0.0, min(u1, r.right) - max(u0, r.left)); ih = max(0.0, min(v1, r.bottom) - max(v0, r.top))
                inter = iw * ih; uni = (u1 - u0) * (v1 - v0) + (r.right - r.left) * (r.bottom - r.top) - inter
                ious.append(inter / uni if uni > 0 else 0.0)
            m = float(np.median(ious)) if len(ious) >= 5 else 0.0
            if m > best[1]:
                best = (conv, m)
        self._box_conv = best[0] if best[1] >= 0.5 else None
        self.box_iou = best[1]
        return self._box_conv

    def wheel_contact(self, f, tid, A):
        conv = self.box_convention()
        g = self._box_geometry(f, tid, conv) if conv is not None else None
        if g is None:
            return np.nan, np.nan
        hgt, tol = PREREG["x6_wheel_height_m"], PREREG["x6_wheel_depth_tol_m"]
        n_vis = n_in = 0
        H, W = A["depth"].shape
        for a in (-1, 1):
            for b in (-1, 1):
                X = g["c"] + a * 0.35 * g["l"] * g["e_len"] + b * 0.40 * g["w"] * g["e_wid"] + (0.5 * g["h"] - hgt) * g["down"]
                uv, z = self._project_world(f, X)
                u, v = int(round(uv[0, 0])), int(round(uv[0, 1]))
                if z[0] <= 0.5 or not (0 <= u < W and 0 <= v < H):
                    continue
                n_in += 1
                win = (slice(max(v - 1, 0), v + 2), slice(max(u - 1, 0), u + 2))
                if np.any((A["inst"][win] == tid) & (np.abs(A["depth"][win] - z[0]) <= tol)):
                    n_vis += 1
        return n_vis, n_in

    def static_fit_residual(self, frames, tid):
        # rms reprojection error (px) of the best static point for the vehicle's reference point
        pix, Rs, Cs = [], [], []
        for f in frames:
            T = self.obj_transform(f, tid)
            if T is None or f not in self.R:
                return np.nan
            q = world_to_cam(T[1], self.R[f], self.C[f])
            if q[2] <= 1.0:
                return np.nan
            p = self.cam.project(q)
            if not (0 <= p[0] < VK_CROP_W and 0 <= p[1] < VK_CROP_H):
                return np.nan
            pix.append(p); Rs.append(self.R[f]); Cs.append(self.C[f])
        return triangulate_multiview(pix, [self.cam] * len(frames), Rs, Cs)[1]


def _vk_warp_error(sc, i, j, stride=4):
    A, B = sc.frame_gt(i), sc.frame_gt(j)
    H, W = A["depth"].shape
    vv, uu = np.mgrid[0:H:stride, 0:W:stride]
    z = A["depth"][vv, uu]; st_ = (A["inst"][vv, uu] < 0) & np.isfinite(z) & (z < 80)
    X = sc.C[i] + (sc.cam.rays(uu[st_].astype(float), vv[st_].astype(float)) * z[st_][:, None]) @ sc.R[i].T
    Xj = (X - sc.C[j]) @ sc.R[j]
    ok = Xj[:, 2] > 0.5
    p = sc.cam.project(Xj[ok])
    ins = (p[:, 0] >= 0) & (p[:, 0] <= W - 1) & (p[:, 1] >= 0) & (p[:, 1] <= H - 1)
    zo = B["depth"][np.round(p[ins, 1]).astype(int), np.round(p[ins, 0]).astype(int)]
    zp = Xj[ok][ins, 2]
    m = np.isfinite(zo)
    return float(np.median(np.abs(zo[m] - zp[m]) / zo[m])) if m.sum() > 200 else np.nan

def _vk_object_error(sc, pairs):
    # median relative depth disagreement of moving-vehicle pixels warped rigidly from i to j
    errs = []
    for i, j in pairs:
        g = sc.pair_gt(i, j)
        B = sc.frame_gt(j)
        m = (g["mover_id"] >= 0) & g["valid"]
        if m.sum() < 50:
            continue
        ui = np.round(g["pB"][m][:, 0]).astype(int); vi = np.round(g["pB"][m][:, 1]).astype(int)
        zo = B["depth"][vi, ui]; ok = np.isfinite(zo) & (B["inst"][vi, ui] >= 0)
        if ok.sum() > 20:
            errs.append(np.median(np.abs(zo[ok] - g["zB"][m][ok]) / zo[ok]))
    return float(np.median(errs)) if errs else np.nan

def _vk_inst_mode(sc, frames):
    # does instance value v mean track v-1? (bbox overlap must agree for >= 80% of the instances)
    agree, total = 0, 0
    for f in frames:
        raw = sc._inst_raw(f)
        m = sc._match_values(f, raw)
        for v, t in m.items():
            total += 1; agree += int(t == v - 1)
    return "track+1" if total and agree / total >= 0.8 else "overlap"

VK_SCENE_OBJS, VK_A3, VK_PAIR_META, _vk_rows, _vk_prev_rows, _vk_kview = {}, {}, [], [], [], []
VK_GT_STAGE = Stage("vk:gt", "vk", dict(regimes=[PREREG["eps_parallel_max"], PREREG["eps_perp_min"]], erosion=PREREG["erosion_px"],
                                        min_px=PREREG["min_object_pixels"]), upstream=[VK_STAGE.fp])
VK_GT_DIR = os.path.join(VK_DIR, "gt")
_VK_RESTORED_A3 = None
_VK_GT_READY = bool(VK_STATUS["available"] and VK_GT_STAGE.available() is not None and VK_GT_STAGE.done())
if _VK_GT_READY:
    with open(os.path.join(VK_GT_DIR, "a3.json")) as f:
        _VK_RESTORED_A3 = json.load(f)["a3"]
if VK_STATUS["available"] and _VK_RESTORED_A3 is not None:
    # conventions were decided when the stage was computed; reuse them exactly
    for name, v in _VK_RESTORED_A3.items():
        if not v["passed"]:
            continue
        sc = VKScene(name)
        sc.inverted, sc.yaw_axis, sc.yaw_sign, sc.inst_mode = v["convention"] == "inverted", v["yaw_axis"], v["yaw_sign"], v["instance_mode"]
        sc._set_poses()
        VK_SCENE_OBJS[name] = sc
    VK_A3 = _VK_RESTORED_A3
elif VK_STATUS["available"]:
    with stage("vkitti_gt"):
        for name in VK_SELECTION:
            sc = VKScene(name)
            pairs = [tuple(p[:2]) for p in VK_SELECTION[name]["pairs"]]
            test = pairs[: min(12, len(pairs))]
            sc.inst_mode = _vk_inst_mode(sc, sorted({f for p in test for f in p})[:6])
            eA = np.nanmedian([_vk_warp_error(sc, i, j) for i, j in test])
            sc.inverted = True; sc._set_poses(); sc._cache.clear()
            eB = np.nanmedian([_vk_warp_error(sc, i, j) for i, j in test])
            sc.inverted = bool(np.isfinite(eB) and (not np.isfinite(eA) or eB < 0.5 * eA)); sc._set_poses(); sc._cache.clear()
            werr = eB if sc.inverted else eA
            obj = {}
            for axis in ("y", "z"):
                for sign in (1.0, -1.0):
                    sc.yaw_axis, sc.yaw_sign = axis, sign; sc._cache.clear()
                    obj[(axis, sign)] = _vk_object_error(sc, test)
            (sc.yaw_axis, sc.yaw_sign), oerr = min(obj.items(), key=lambda kv: kv[1] if np.isfinite(kv[1]) else 1e9)
            sc._cache.clear()
            VK_A3[name] = dict(n_frames=len(sc.R), n_test_pairs=len(test), warp_err_documented=float(eA), warp_err_inverted=float(eB),
                               convention="inverted" if sc.inverted else "documented", static_warp_err=float(werr),
                               yaw_axis=sc.yaw_axis, yaw_sign=sc.yaw_sign, vehicle_warp_err=float(oerr), instance_mode=sc.inst_mode,
                               passed=bool(np.isfinite(werr) and werr <= 0.05 and np.isfinite(oerr) and oerr <= 0.05))
            if not VK_A3[name]["passed"]:
                continue
            VK_SCENE_OBJS[name] = sc
        VK_STATUS["available"] = len(VK_SCENE_OBJS) > 0
        if not VK_STATUS["available"]:
            VK_STATUS["note"] = "no scene passed the ground-truth self-consistency test (A3)"
VK_STATUS["available"] = bool(VK_STATUS["available"] and VK_SCENE_OBJS)

if VK_STATUS["available"]:
    def vk_meta_pair(name, n, i, j):
        sc = VK_SCENE_OBJS[name]
        return dict(set="VK", index=n, seq=name, frames=[int(i), int(j)], dir=sc.dir,
                    baseline=float(np.linalg.norm(sc.C[j] - sc.C[i])))

    _n = 0
    for name, sc in VK_SCENE_OBJS.items():
        for (i, j, b, rot) in VK_SELECTION[name]["pairs"]:
            VK_PAIR_META.append(vk_meta_pair(name, _n, i, j)); _n += 1

    def vk_mover_stats(sc, i, j):
        g = sc.pair_gt(i, j)
        b = sc.C[j] - sc.C[i]
        out = []
        for k, mv in enumerate(g["movers"]):
            mask = g["mover_id"] == k
            em = erode_mask(mask, PREREG["erosion_px"]) & g["visible"]
            n = int(em.sum())
            bb = sc.bbox.get((i, mv["track"]))
            inside = bb is not None and bb.left >= sc.x0 and bb.right <= sc.x0 + VK_CROP_W - 1
            d = mv["pos_j"] - mv["pos_i"]
            rec = dict(mover=k, track=mv["track"], label=mv["label"], s=float(d @ b / (b @ b)),
                       speed_ratio=float(np.linalg.norm(d) / max(np.linalg.norm(b), 1e-9)),
                       area_px=int(mask.sum()), n_eroded=n, occupancy=float(bb.occupancy_ratio) if bb is not None else np.nan,
                       truncation=float(bb.truncation_ratio) if bb is not None else np.nan, inside_crop=bool(inside),
                       vis_in_b=float(g["visible"][mask].mean()) if mask.any() else 0.0)
            n_vis, n_in = sc.wheel_contact(i, mv["track"], sc.frame_gt(i))
            rec.update(wheels_visible=n_vis, wheels_in_view=n_in,
                       contact=("visible" if n_vis >= 1 else "hidden" if n_in >= 2 else "undetermined") if np.isfinite(n_vis) else "undetermined")
            if n > 0:
                Zt = g["depth"][em].astype(np.float64); zg = g["zgeo"][em]
                rec.update(Z_true=float(np.median(Zt)), eps_med=float(np.median(g["eps"][em])), eps_max=float(np.max(g["eps"][em])),
                           x_geo=float(np.median(np.log(np.where(zg > 0, zg, np.nan) / Zt))) if np.all(zg > 0) else float("nan"),
                           parallax_med=float(np.median(g["parallax"][em])))
            out.append(rec)
        return out

    def _vk_table(metas):
        rows = []
        for meta in metas:
            sc = VK_SCENE_OBJS[meta["seq"]]
            i, j = meta["frames"]
            for r in vk_mover_stats(sc, i, j):
                r = {("gt_" + k if k not in ("mover", "track", "label", "s", "speed_ratio") else k): v for k, v in r.items()}
                r.update(set=meta["set"], scene=meta["index"], seq=meta["seq"], uid=f"{meta['set']}-{meta['index']:04d}-{r['mover']}",
                         cls="vehicle", cam_kind="driving", motion_type="vehicle")
                rows.append(r)
        df = pd.DataFrame(rows)
        if not len(df):
            return df
        df["x_par"] = np.where(df["s"] < 1, -np.log(np.clip(1 - df["s"], 1e-9, None)), np.nan)
        df["regime"] = np.where(df["gt_eps_med"] < PREREG["eps_parallel_max"], "parallel",
                                np.where(df["gt_eps_med"] >= PREREG["eps_perp_min"], "perpendicular", "intermediate"))
        df["measurable"] = vk_measurable(df)
        df["identifiable"] = df["gt_parallax_med"] >= PREREG["parallax_min_px"]
        return df

    with stage("vkitti_gt"):
        if not _VK_GT_READY and not VK_GT_STAGE.done():
            os.makedirs(VK_GT_DIR, exist_ok=True)
            _vk_table(VK_PAIR_META).to_csv(os.path.join(VK_GT_DIR, "movers_pairs.csv"), index=False)
            # prevalence 1: moving-vehicle pixels in the two-view pairs
            for name, sc in VK_SCENE_OBJS.items():
                e_all = []
                for meta in [m for m in VK_PAIR_META if m["seq"] == name]:
                    g = sc.pair_gt(*meta["frames"])
                    m = (g["mover_id"] >= 0) & g["visible"]
                    e_all.append(g["eps"][m])
                e = np.concatenate(e_all) if e_all else np.zeros(0)
                if len(e):
                    _vk_prev_rows.append(dict(dataset="Virtual KITTI 2", seq=name, n_pairs=len(e_all), n_moving_pixels=int(len(e)),
                                              frac_parallel=float(np.mean(e < PREREG["eps_parallel_max"])),
                                              frac_intermediate=float(np.mean((e >= PREREG["eps_parallel_max"]) & (e < PREREG["eps_perp_min"]))),
                                              frac_perpendicular=float(np.mean(e >= PREREG["eps_perp_min"])),
                                              frac_parallel_no_static_solution=float("nan"), median_eps=float(np.median(e))))
                # prevalence 2: every moving vehicle in every frame, static fit over K frames (text GT only)
                frames = sorted(sc.R)
                fidx = {f: n for n, f in enumerate(frames)}
                for (f, tid), r in sc.bbox.items():
                    if not r.isMoving or (r.right - r.left) * (r.bottom - r.top) < VK_DESIGN["min_vehicle_px"]:
                        continue
                    if not (r.left >= sc.x0 and r.right <= sc.x0 + VK_CROP_W - 1):
                        continue
                    gap = next((g_ for g_ in range(1, VK_DESIGN["max_gap_frames"] + 1)
                                if f + g_ in sc.C and np.linalg.norm(sc.C[f + g_] - sc.C[f]) >= VK_DESIGN["pair_baseline_m"][0]), None)
                    if gap is None:
                        continue
                    for K in (2, 4, 8):
                        res = sc.static_fit_residual([f + k * gap for k in range(K)], tid)
                        if np.isfinite(res):
                            _vk_kview.append(dict(seq=name, frame=f, track=tid, K=K, gap=gap, residual_px=float(res)))
            pd.DataFrame(_vk_prev_rows).to_csv(os.path.join(VK_GT_DIR, "prevalence_pixels.csv"), index=False)
            pd.DataFrame(_vk_kview).to_csv(os.path.join(VK_GT_DIR, "prevalence_kview.csv"), index=False)
            with open(os.path.join(VK_GT_DIR, "a3.json"), "w") as f:
                json.dump(dict(a3=VK_A3, pairs=VK_PAIR_META), f, indent=1, default=str)
            VK_GT_STAGE.commit([VK_GT_DIR])
    MOVERS_VK = pd.read_csv(os.path.join(VK_GT_DIR, "movers_pairs.csv"))
    # v6.1 measurability (D34): the CONVOY-3D pixel rule, recomputed from the stored columns so that a
    # restored v6.0 checkpoint gives the same table; the registered v6.0 rule is kept for the record
    MOVERS_VK["measurable_v6_0"] = ((MOVERS_VK["gt_n_eroded"] >= PREREG["min_object_pixels"]) & (MOVERS_VK["gt_occupancy"] >= 0.85)
                                    & MOVERS_VK["gt_inside_crop"].astype(bool) & (MOVERS_VK["gt_truncation"] <= 0.05)
                                    & (MOVERS_VK["gt_vis_in_b"] >= 0.85))
    MOVERS_VK["measurable"] = vk_measurable(MOVERS_VK)
    for _sc in VK_SCENE_OBJS.values():
        _sc._cache.clear()
    PREVALENCE_VK = pd.read_csv(os.path.join(VK_GT_DIR, "prevalence_pixels.csv"))
    KVIEW_VK = pd.read_csv(os.path.join(VK_GT_DIR, "prevalence_kview.csv"))
    with open(os.path.join(VK_GT_DIR, "a3.json")) as f:
        _a3 = json.load(f)
    VK_A3 = _a3["a3"]
    MOVERS_VK.to_csv(os.path.join(MANIFEST_DIR, "vkitti_movers.csv"), index=False)
    print(f"{'scene':<9}{'frames':>7}{'pairs':>6}  extrinsics   static warp  yaw   vehicle warp  instances  A3")
    for name, v in VK_A3.items():
        print(f"{name:<9}{v['n_frames']:>7}{sum(m['seq'] == name for m in VK_PAIR_META):>6}"
              f"  {v['convention']:<11}{v['static_warp_err']:>12.3f}  {('+' if v['yaw_sign'] > 0 else '-') + v['yaw_axis']:<5}"
              f"{v['vehicle_warp_err']:>12.3f}  {v['instance_mode']:<10} {'PASS' if v['passed'] else 'FAIL'}")
    print(f"\nVirtual KITTI 2: {len(MOVERS_VK)} moving vehicles in {len(VK_PAIR_META)} pairs; regimes "
          f"{MOVERS_VK.groupby('regime').size().to_dict()}; measurable {int(MOVERS_VK['measurable'].sum())}, "
          f"identifiable {int((MOVERS_VK['measurable'] & MOVERS_VK['identifiable']).sum())}")
    _lo, _hi = PREREG["h9_x_range"]
    _par = MOVERS_VK["regime"] == "parallel"
    _inx = pd.to_numeric(MOVERS_VK["gt_x_geo"], errors="coerce").between(_lo, _hi)
    _att = [("moving vehicles in the pairs", np.ones(len(MOVERS_VK), bool)), ("parallel regime (|eps| < 1 px)", _par),
            (f"... and >= {PREREG['min_object_pixels']} eroded visible px", _par & (MOVERS_VK["gt_n_eroded"] >= PREREG["min_object_pixels"])),
            (f"... and >= {100 * PREREG['vk_min_occupancy']:.0f}% of the silhouette visible (measurable)", _par & MOVERS_VK["measurable"]),
            (f"... and parallax >= {PREREG['parallax_min_px']:g} px (identifiable)", _par & MOVERS_VK["measurable"] & MOVERS_VK["identifiable"]),
            (f"... and x_geo in [{_lo}, {_hi}] (the H9 sample)", _par & MOVERS_VK["measurable"] & MOVERS_VK["identifiable"] & _inx),
            ("H9 sample under the v6.0 rule (whole box in the crop, truncation <= 5%, visible in view 2 >= 85%)",
             _par & MOVERS_VK["measurable_v6_0"] & MOVERS_VK["identifiable"] & _inx)]
    VK_ATTRITION = pd.DataFrame([dict(step=k, vehicles=int(np.sum(v)), pairs=int(MOVERS_VK.loc[np.asarray(v, bool), "scene"].nunique()))
                                 for k, v in _att])
    print("\nVirtual KITTI 2 sample, step by step:\n" + VK_ATTRITION.to_string(index=False))
    if "gt_contact" in MOVERS_VK:
        _pm = MOVERS_VK[_par & (MOVERS_VK["gt_n_eroded"] >= PREREG["x6_min_eroded_px"])]
        print("ground contact of the parallel vehicles with >= 40 eroded px (wheel contact points visible / hidden / "
              "undetermined):", _pm["gt_contact"].value_counts().to_dict())
    print(PREVALENCE_VK.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    _kv = KVIEW_VK.assign(invisible=KVIEW_VK["residual_px"] < 1.0).groupby("K").agg(
        n=("invisible", "size"), share_static_fit_below_1px=("invisible", "mean"), median_residual_px=("residual_px", "median"))
    print("\nmoving vehicles whose K-frame track a static point fits within 1 px:\n" + _kv.to_string(float_format=lambda v: f"{v:.3f}"))
else:
    MOVERS_VK = PREVALENCE_VK = KVIEW_VK = VK_ATTRITION = pd.DataFrame()
    print("Virtual KITTI 2 is not used in this run:", VK_STATUS["note"])

release_memory(); _ = mem_report("Virtual KITTI 2 ground truth")

In [ ]:
# Cell No: 032
# 3.10.0 LiDAR ground truth shared by the real-driving data (KITTI tracking, Argoverse 2; v6.3).
#        LiDAR returns are sparse, so a far surface can show through the gaps between the returns of
#        a near one. A pixel is therefore kept only if its nearest return is also the nearest surface
#        in a small window around it (within max(10%, 1 m) of the window minimum); pixels that fail
#        this test carry no ground truth. This is the usual occlusion test for projected LiDAR.
from scipy.ndimage import minimum_filter as _minimum_filter


def lidar_zbuffer(uv, z, H, W, window=5, tol_rel=0.10, tol_abs=1.0):
    """Nearest return per pixel after the occlusion test.
    uv (N, 2) pixel coordinates, z (N,) depth. Returns D (H, W) float32 (NaN: no ground truth) and
    idx (H, W) int64, the index of the point that gives the pixel its depth (-1: none)."""
    uv = np.asarray(uv, np.float64); z = np.asarray(z, np.float64)
    u = np.round(uv[:, 0]).astype(np.int64) if len(uv) else np.zeros(0, np.int64)
    v = np.round(uv[:, 1]).astype(np.int64) if len(uv) else np.zeros(0, np.int64)
    ok = np.isfinite(z) & (z > 0) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    ids = np.nonzero(ok)[0]
    D = np.full(H * W, np.inf, np.float64)
    idx = np.full(H * W, -1, np.int64)
    if len(ids):
        pix = v[ids] * W + u[ids]
        order = np.lexsort((z[ids], pix))
        ps = pix[order]
        first = np.ones(len(order), bool)
        first[1:] = ps[1:] != ps[:-1]
        D[ps[first]] = z[ids[order[first]]]
        idx[ps[first]] = ids[order[first]]
    D = D.reshape(H, W); idx = idx.reshape(H, W)
    if window and window > 1:
        Dmin = _minimum_filter(D, size=int(window), mode="constant", cval=np.inf)
        hidden = np.isfinite(D) & (D > Dmin + np.maximum(tol_rel * Dmin, tol_abs))
        D[hidden] = np.inf
        idx[hidden] = -1
    D[~np.isfinite(D)] = np.nan
    return D.astype(np.float32), idx


def sample_gray(img, uv):
    """Bilinear grey value of an RGB uint8 image at sub-pixel positions uv (N, 2); NaN outside."""
    g = img.astype(np.float32).mean(-1) if img.ndim == 3 else img.astype(np.float32)
    uv = np.asarray(uv, np.float32).reshape(-1, 1, 2)
    out = cv2.remap(g, uv[..., 0], uv[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=np.nan)
    return out.reshape(-1)


def lidar_vehicle_stats(sc, uv_i, uv_j, Z, in_j):
    """Epipolar offset, static-hypothesis depth and parallax of a vehicle's ground-truth pixels."""
    eps, zgeo, par = pixel_epipolar_quantities(sc, 0, 1, uv_i, uv_j)
    ok = in_j & np.isfinite(eps)
    rec = dict(n_lidar_px=int(ok.sum()))
    if ok.sum() >= 3:
        zg = zgeo[ok]
        rec.update(Z_true=float(np.median(Z[ok])), eps_med=float(np.median(eps[ok])), parallax_med=float(np.median(par[ok])),
                   x_geo=float(np.median(np.log(zg / Z[ok]))) if np.all(zg > 0) else float("nan"))
    return rec, ok


def lidar_mover_table(metas, set_):
    # one row per labelled vehicle of every pair (moving and parked), in the column layout of Virtual KITTI 2
    rows = []
    for m_ in metas:
        for k, v in enumerate(m_["vehicles"]):
            r = {("gt_" + a if a not in ("track", "label", "s", "speed_ratio", "state") else a): b
                 for a, b in v.items() if a != "pixels"}
            r.update(set=set_, scene=m_["index"], seq=m_["seq"], mover=k, uid=f"{set_}-{m_['index']:04d}-{k}", cls="vehicle",
                     cam_kind="driving", motion_type="vehicle")
            rows.append(r)
    return pd.DataFrame(rows)


def lidar_regimes(df):
    # Motion regimes on real data. Labelled boxes carry a few centimetres of annotation noise, which
    # enters the correspondence; the epipolar offset of parked vehicles (zero without noise) measures it.
    # Parallel regime: |eps| below the 90th percentile over parked vehicles, and at least 1 px.
    pk = df[df["state"] == "parked"]
    noise = (float(np.nanquantile(pk["gt_eps_med"], PREREG["kt_eps_noise_quantile"]))
             if "gt_eps_med" in pk and pk["gt_eps_med"].notna().sum() >= 10 else np.nan)
    tau = max(PREREG["eps_parallel_max"], noise if np.isfinite(noise) else PREREG["eps_parallel_max"])
    df["regime"] = np.where(df["gt_eps_med"] < tau, "parallel",
                            np.where(df["gt_eps_med"] >= max(PREREG["eps_perp_min"], 2 * tau), "perpendicular", "intermediate"))
    df["x_par"] = np.where(df["s"] < 1, -np.log(np.clip(1 - df["s"], 1e-9, None)), np.nan)
    df["measurable"] = df["gt_n_lidar_px"] >= PREREG["kt_min_lidar_px"]
    df["identifiable"] = df["gt_parallax_med"] >= PREREG["parallax_min_px"]
    return df, noise, tau, len(pk)


print("LiDAR ground-truth helpers ready (nearest return per pixel with a 5 x 5 occlusion test).")

In [ ]:
# Cell No: 033
# 3.10 KITTI tracking (exploratory, v6.2): find an attached copy, read calibration, OXTS poses and
#      labels, check the poses against the LiDAR (A3), choose frame pairs and build the LiDAR ground
#      truth of every vehicle. One checkpointed stage ("kitti_gt"); its archive holds the cropped
#      images, so the evaluation of a later session does not need the dataset attached again.
KT_REQUIRED = ("image_02", "velodyne", "label_02", "calib", "oxts")
KT_STATUS = dict(available=False, root=None, note="", sequences=[])
KT_DIR = os.path.join(SCRATCH_DIR, "kitti")
KT_PAIR_META, KT_A3 = [], {}

def kt_find_root():
    for base in [os.environ.get(KT_DESIGN["root_env"], ""), "/kaggle/input"]:
        if not base or not os.path.isdir(base):
            continue
        for dp, dns, _ in os.walk(base):
            if dp[len(base):].count(os.sep) >= 6:
                dns[:] = []
                continue
            if all(k in dns for k in KT_REQUIRED):
                return dp
    return None

def _kt_rx(a):
    c, s = np.cos(a), np.sin(a); return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
def _kt_ry(a):
    c, s = np.cos(a), np.sin(a); return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
def _kt_rz(a):
    c, s = np.cos(a), np.sin(a); return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
def _h4(M):
    T = np.eye(4); T[:M.shape[0], :M.shape[1]] = M; return T

def kt_calib(path):
    d = {}
    with open(path) as f:
        for line in f:
            p = line.replace(":", " ").split()
            if len(p) > 1:
                d[p[0]] = np.array([float(v) for v in p[1:]])
    alias = lambda *ks: next(d[k] for k in ks if k in d)
    P2 = alias("P2").reshape(3, 4)
    R_rect = alias("R_rect", "R0_rect").reshape(3, 3)
    Tvc = alias("Tr_velo_cam", "Tr_velo_to_cam").reshape(3, 4)
    Tiv = alias("Tr_imu_velo", "Tr_imu_to_velo").reshape(3, 4)
    K2 = P2[:, :3].copy()
    t2 = np.linalg.solve(K2, P2[:, 3])                   # camera 2 = rectified camera 0 shifted by t2
    T_rect_velo = _h4(R_rect) @ _h4(Tvc)
    return dict(K2=K2, t2=t2, T_rect_velo=T_rect_velo, T_rect_imu=T_rect_velo @ _h4(Tiv))

def kt_oxts_poses(path):
    a = np.loadtxt(path, ndmin=2)
    er = 6378137.0
    scale = np.cos(a[0, 0] * np.pi / 180.0)
    t = np.stack([scale * a[:, 1] * np.pi * er / 180.0, scale * er * np.log(np.tan((90.0 + a[:, 0]) * np.pi / 360.0)), a[:, 2]], 1)
    t = t - t[0]
    out = []
    for k in range(len(a)):
        T = np.eye(4); T[:3, :3] = _kt_rz(a[k, 5]) @ _kt_ry(a[k, 4]) @ _kt_rx(a[k, 3]); T[:3, 3] = t[k]
        out.append(T)
    return out

def kt_labels(path):
    lab = {}
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) < 17:
                continue
            fr = int(p[0])
            lab.setdefault(fr, []).append(dict(track=int(p[1]), type=p[2], trunc=float(p[3]), occ=int(float(p[4])),
                                               bbox=[float(v) for v in p[6:10]], h=float(p[10]), w=float(p[11]), l=float(p[12]),
                                               loc=np.array([float(v) for v in p[13:16]]), ry=float(p[16])))
    return lab

class KTSequence:
    def __init__(self, root, name):
        self.root, self.name = root, name
        self.calib = kt_calib(os.path.join(root, "calib", name + ".txt"))
        self.imu = kt_oxts_poses(os.path.join(root, "oxts", name + ".txt"))
        self.labels = kt_labels(os.path.join(root, "label_02", name + ".txt"))
        self.frames = sorted(int(os.path.splitext(f)[0]) for f in os.listdir(os.path.join(root, "image_02", name)) if f.endswith(".png"))
        self.frames = [f for f in self.frames if f < len(self.imu)
                       and os.path.exists(os.path.join(root, "velodyne", name, f"{f:06d}.bin"))]
        self.convention = "documented"
        im = self.rgb(self.frames[0]) if self.frames else np.zeros((375, 1242, 3), np.uint8)
        self.H, self.W = im.shape[:2]

    def rgb(self, f):
        return np.asarray(Image.open(os.path.join(self.root, "image_02", self.name, f"{f:06d}.png")).convert("RGB"))

    def pose(self, f):
        # camera 2: rotation camera -> world (columns = camera axes) and centre, in the OXTS world
        Tri = self.calib["T_rect_imu"]
        T_w_rect = self.imu[f] @ (np.linalg.inv(Tri) if self.convention == "documented" else Tri)
        R = T_w_rect[:3, :3]
        return R, T_w_rect[:3, :3] @ (-self.calib["t2"]) + T_w_rect[:3, 3]

    def lidar_cam(self, f):
        # LiDAR points in camera-2 coordinates (N, 3), in front of the camera within the depth range
        X = np.fromfile(os.path.join(self.root, "velodyne", self.name, f"{f:06d}.bin"), np.float32).reshape(-1, 4)
        Xr = (self.calib["T_rect_velo"] @ np.c_[X[:, :3], np.ones(len(X))].T).T[:, :3] + self.calib["t2"]
        z = Xr[:, 2]
        return Xr[(z >= KT_DESIGN["lidar_min_depth_m"]) & (z <= KT_DESIGN["lidar_max_depth_m"])]

    def project(self, Xc):
        uvw = Xc @ self.calib["K2"].T
        return uvw[:, :2] / uvw[:, 2:3]

    def box_world(self, f, ob):
        # labelled box (bottom centre, KITTI rotation about the rectified camera's y axis) in the world
        R, C = self.pose(f)
        loc_c = ob["loc"] + self.calib["t2"]                 # rectified camera 0 -> camera 2 coordinates
        return R @ loc_c + C, R @ _kt_ry(ob["ry"])

def kt_in_box(Xc, ob, t2, margin, ground_cut=None):
    # points (camera-2 coordinates) inside a labelled box, optionally without its lowest ground_cut metres
    q = (Xc - t2 - ob["loc"]) @ _kt_ry(ob["ry"])           # object frame: x length, y down (0 = bottom), z width
    ins = (np.abs(q[:, 0]) <= ob["l"] / 2 + margin) & (np.abs(q[:, 2]) <= ob["w"] / 2 + margin) & \
          (q[:, 1] >= -ob["h"] - margin) & (q[:, 1] <= margin)
    if ground_cut is not None:
        ins &= q[:, 1] <= -ground_cut
    return ins

def _kt_depth_image(uv, z, H, W):
    u, v = np.round(uv[:, 0]).astype(int), np.round(uv[:, 1]).astype(int)
    ok = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    u, v, z = u[ok], v[ok], z[ok]
    D = np.full((H, W), np.inf, np.float32)
    np.minimum.at(D, (v, u), z.astype(np.float32))
    D[~np.isfinite(D)] = np.nan
    return D

def kt_a3(seq, pairs):
    # median relative disagreement between frame j's LiDAR and frame i's static LiDAR warped with the poses
    errs = []
    for i, j in pairs:
        Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
        X = seq.lidar_cam(i)
        stat = np.ones(len(X), bool)
        for ob in seq.labels.get(i, []):
            if ob["type"] != "DontCare":
                stat &= ~kt_in_box(X, ob, seq.calib["t2"], KT_DESIGN["static_box_margin_m"])
        Xw = X[stat] @ Ri.T + Ci
        Xj = (Xw - Cj) @ Rj
        Xj = Xj[Xj[:, 2] > 1.0]
        Dj = _kt_depth_image(seq.project(seq.lidar_cam(j)), seq.lidar_cam(j)[:, 2], seq.H, seq.W)
        uv = np.round(seq.project(Xj)).astype(int)
        ok = (uv[:, 0] >= 0) & (uv[:, 0] < seq.W) & (uv[:, 1] >= 0) & (uv[:, 1] < seq.H)
        zo = Dj[uv[ok, 1], uv[ok, 0]]; zp = Xj[ok, 2]
        m = np.isfinite(zo)
        if m.sum() >= 200:
            errs.append(float(np.median(np.abs(zo[m] - zp[m]) / zo[m])))
    return float(np.median(errs)) if errs else np.nan

def _kt_cam_dict(seq, f, x0, y0, w, h):
    R, C = seq.pose(f); K = seq.calib["K2"]
    return dict(fx=float(K[0, 0]), fy=float(K[1, 1]), cx=float(K[0, 2] - x0), cy=float(K[1, 2] - y0), W=int(w), H=int(h),
                R=R.tolist(), C=C.tolist())

def kt_pair_gt(seq, i, j, x0, y0, w, h):
    # LiDAR ground truth of one pair in the crop: sparse depth (nearest return per pixel after the
    # occlusion test of Section 3.10.0), static mask, per-vehicle pixels and statistics
    t2 = seq.calib["t2"]
    Ri, Ci = seq.pose(i); Rj, Cj = seq.pose(j)
    X = seq.lidar_cam(i)
    uv = seq.project(X) - np.array([x0, y0])
    inside = (uv[:, 0] >= -0.5) & (uv[:, 0] < w - 0.5) & (uv[:, 1] >= -0.5) & (uv[:, 1] < h - 0.5)
    X, uv = X[inside], uv[inside]
    D, idx = lidar_zbuffer(uv, X[:, 2], h, w, KT_DESIGN["zbuffer_window_px"], *KT_DESIGN["zbuffer_tol"])
    has = idx >= 0
    win = np.zeros(len(X), bool)
    win[idx[has]] = True                                   # points that give a pixel its depth
    labs_i = {o["track"]: o for o in seq.labels.get(i, []) if o["type"] != "DontCare"}
    labs_j = {o["track"]: o for o in seq.labels.get(j, []) if o["type"] != "DontCare"}
    stat = np.ones(len(X), bool)
    for ob in labs_i.values():
        stat &= ~kt_in_box(X, ob, t2, KT_DESIGN["static_box_margin_m"])
    sm = np.zeros((h, w), bool)
    sm[has] = stat[idx[has]]
    for o in seq.labels.get(i, []):
        if o["type"] == "DontCare":
            l_, t_, r_, b_ = o["bbox"]
            sm[max(int(t_ - y0), 0):max(int(b_ - y0) + 1, 0), max(int(l_ - x0), 0):max(int(r_ - x0) + 1, 0)] = False
    b = Cj - Ci
    sc = dict(cams=[_kt_cam_dict(seq, i, x0, y0, w, h), _kt_cam_dict(seq, j, x0, y0, w, h)])
    vehicles = []
    for tid, oi in labs_i.items():
        oj = labs_j.get(tid)
        if oj is None or oi["type"] not in KT_DESIGN["classes"]:
            continue
        Pi, Rwi = seq.box_world(i, oi); Pj, Rwj = seq.box_world(j, oj)
        disp = float(np.linalg.norm(Pj - Pi))
        if KT_DESIGN["max_truncation"] < max(oi["trunc"], oj["trunc"]) or max(oi["occ"], oj["occ"]) > KT_DESIGN["max_occlusion"]:
            continue
        state = "moving" if disp >= PREREG["kt_moving_disp_m"] else "parked" if disp <= PREREG["kt_parked_disp_m"] else None
        if state is None:
            continue
        own = kt_in_box(X, oi, t2, 0.1, ground_cut=0.15) & win
        if own.sum() < 3:
            continue
        Xw = X[own] @ Ri.T + Ci
        Xw2 = (Xw - Pi) @ Rwi @ Rwj.T + Pj                       # rigid motion of the labelled box
        Xj = (Xw2 - Cj) @ Rj
        uvj = seq.project(Xj) - np.array([x0, y0])
        okj = (Xj[:, 2] > 0.5) & (uvj[:, 0] >= 0) & (uvj[:, 0] <= w - 1) & (uvj[:, 1] >= 0) & (uvj[:, 1] <= h - 1)
        uva = uv[own]
        st_, ok = lidar_vehicle_stats(sc, uva, uvj, X[own, 2], okj)
        px = np.unique(np.round(uva[ok]).astype(int), axis=0) if ok.any() else np.zeros((0, 2), int)
        rec = dict(track=int(tid), label=oi["type"], state=state, disp_m=disp, s=float((Pj - Pi) @ b / (b @ b)),
                   speed_ratio=float(disp / max(np.linalg.norm(b), 1e-9)), occlusion=int(max(oi["occ"], oj["occ"])),
                   truncation=float(max(oi["trunc"], oj["trunc"])), pixels=px.tolist(), **st_)
        rec["n_lidar_px"] = int(len(px))
        vehicles.append(rec)
    return dict(depth=D, static=sm, vehicles=vehicles, scene=sc, baseline=float(np.linalg.norm(b)))

def kt_select_pairs(seq, rng, n_max):
    lo, hi = KT_DESIGN["pair_baseline_m"]
    out, last = [], -10**9
    frames = seq.frames
    starts = list(range(0, len(frames), 3)); rng.shuffle(starts)
    for si in starts:
        i = frames[si]
        if len(out) >= n_max:
            break
        if any(abs(i - p[0]) < 5 for p in out):
            continue
        Ri, Ci = seq.pose(i)
        for f in frames[si + 1: si + 1 + KT_DESIGN["max_gap_frames"]]:
            Rj, Cj = seq.pose(f)
            bl = float(np.linalg.norm(Cj - Ci))
            if bl >= lo:
                ang = float(np.degrees(np.arccos(np.clip((np.trace(Ri.T @ Rj) - 1) / 2, -1, 1))))
                if bl <= hi and ang <= KT_DESIGN["max_rot_deg"]:
                    out.append((i, f, bl, ang))
                break
    return out

def kt_crop(seq, i, j):
    h = min(seq.H, KT_DESIGN["crop_h_max"]); h -= h % 3
    w = h * 4 // 3
    y0 = (seq.H - h) // 2
    cand = [o for o in seq.labels.get(i, []) if o["type"] in KT_DESIGN["classes"] and o["occ"] <= KT_DESIGN["max_occlusion"]]
    best, best_x = -1, (seq.W - w) // 2
    for x0 in range(0, seq.W - w + 1, 10):
        n = sum(1 for o in cand if o["bbox"][0] >= x0 and o["bbox"][2] <= x0 + w - 1)
        if n > best or (n == best and abs(x0 - (seq.W - w) / 2) < abs(best_x - (seq.W - w) / 2)):
            best, best_x = n, x0
    return best_x, y0, w, h

_kt_root = kt_find_root()
KT_STAGE = Stage("kitti_gt", "kitti", dict(design=KT_DESIGN, n_pairs=RUN_CONFIG["n_kt_pairs_per_seq"],
                                           n_seq=RUN_CONFIG["n_kt_sequences"],
                                           prereg={k: v for k, v in PREREG.items() if k.startswith("kt_")}))
KT_GT_DIR = os.path.join(KT_DIR, "pairs")
with stage("kitti_gt"):
    if _kt_root is None and KT_STAGE.available() is None:
        KT_STATUS["note"] = ("no KITTI tracking folder attached (needs image_02, velodyne, label_02, calib and oxts; "
                             "attach them as a private Kaggle dataset)")
    elif not KT_STAGE.done():
        shutil.rmtree(KT_DIR, ignore_errors=True); os.makedirs(KT_GT_DIR)
        names = sorted(d for d in os.listdir(os.path.join(_kt_root, "label_02")) if d.endswith(".txt"))
        names = [os.path.splitext(d)[0] for d in names
                 if os.path.isdir(os.path.join(_kt_root, "image_02", os.path.splitext(d)[0]))][:RUN_CONFIG["n_kt_sequences"]]
        n_pair = 0
        for name in names:
            seq = KTSequence(_kt_root, name)
            if len(seq.frames) < 10:
                continue
            pairs = kt_select_pairs(seq, np.random.default_rng(seed_for("kt-pairs", name)), RUN_CONFIG["n_kt_pairs_per_seq"])
            test = [(p[0], p[1]) for p in pairs[:8]]
            errs = {}
            for conv in ("documented", "inverted"):
                seq.convention = conv
                errs[conv] = kt_a3(seq, test)
            conv = min(errs, key=lambda k: errs[k] if np.isfinite(errs[k]) else np.inf)
            seq.convention = conv
            passed = bool(np.isfinite(errs[conv]) and errs[conv] <= 0.05)
            KT_A3[name] = dict(n_frames=len(seq.frames), n_pairs=len(pairs), convention=conv,
                               warp_err_documented=errs["documented"], warp_err_inverted=errs["inverted"], passed=passed)
            if not passed:
                continue
            for (i, j, bl, ang) in pairs:
                x0, y0, w, h = kt_crop(seq, i, j)
                g = kt_pair_gt(seq, i, j, x0, y0, w, h)
                if not any(v["state"] == "moving" for v in g["vehicles"]):
                    continue
                d = os.path.join(KT_GT_DIR, f"{name}_{i:06d}_{j:06d}")
                os.makedirs(d)
                for k_, f_ in enumerate((i, j)):
                    Image.fromarray(seq.rgb(f_)[y0:y0 + h, x0:x0 + w]).save(os.path.join(d, f"rgb{k_}.png"))
                np.savez_compressed(os.path.join(d, "gt.npz"), depth=g["depth"].astype(np.float32), static=g["static"])
                with open(os.path.join(d, "meta.json"), "w") as f:
                    json.dump(dict(set="KT", index=n_pair, seq=name, frames=[int(i), int(j)], crop=[int(x0), int(y0), int(w), int(h)],
                                   baseline=bl, rot_deg=ang, scene=g["scene"], vehicles=g["vehicles"], dir=d), f)
                n_pair += 1
        with open(os.path.join(KT_DIR, "a3.json"), "w") as f:
            json.dump(KT_A3, f, indent=1)
        KT_STAGE.commit([KT_DIR])
if os.path.exists(os.path.join(KT_DIR, "a3.json")):
    KT_A3 = json.load(open(os.path.join(KT_DIR, "a3.json")))
    for d in sorted(glob.glob(os.path.join(KT_GT_DIR, "*", "meta.json"))):
        m_ = json.load(open(d)); m_["dir"] = os.path.dirname(d)
        KT_PAIR_META.append(m_)
    KT_PAIR_META.sort(key=lambda m_: m_["index"])
    KT_STATUS.update(available=bool(KT_PAIR_META), root=_kt_root, sequences=sorted({m_["seq"] for m_ in KT_PAIR_META}),
                     note="" if KT_PAIR_META else "no sequence passed A3 or no pair had a moving vehicle")

MOVERS_KT_ALL = lidar_mover_table(KT_PAIR_META, "KT") if KT_PAIR_META else pd.DataFrame()
if len(MOVERS_KT_ALL):
    MOVERS_KT_ALL, KT_EPS_NOISE, KT_TAU_PAR, _n_pk = lidar_regimes(MOVERS_KT_ALL)
    MOVERS_KT = MOVERS_KT_ALL[MOVERS_KT_ALL["state"] == "moving"].reset_index(drop=True)
    print(f"KITTI tracking: {len(KT_PAIR_META)} pairs from {len(KT_STATUS['sequences'])} sequences "
          f"({', '.join(KT_STATUS['sequences'])}); A3: " + "; ".join(f"{k} {'PASS' if v['passed'] else 'FAIL'} "
                                                                  f"({v['warp_err_' + v['convention']]:.3f}, {v['convention']})"
                                                                  for k, v in KT_A3.items()))
    print(f"label-noise floor: 90th percentile of |eps| over {_n_pk} parked vehicles = {KT_EPS_NOISE:.2f} px; "
          f"parallel regime on KITTI: |eps| < {KT_TAU_PAR:.2f} px")
    print(f"moving vehicles: {len(MOVERS_KT)}; regimes {MOVERS_KT['regime'].value_counts().to_dict()}; "
          f"measurable (>= {PREREG['kt_min_lidar_px']} LiDAR px) {int(MOVERS_KT['measurable'].sum())}, identifiable "
          f"{int((MOVERS_KT['measurable'] & MOVERS_KT['identifiable']).sum())}")
else:
    MOVERS_KT = pd.DataFrame(); KT_EPS_NOISE = KT_TAU_PAR = np.nan
    print("KITTI tracking is not used in this run:", KT_STATUS["note"] or "no data")
release_memory(); mem_report("KITTI ground truth");

In [ ]:
# Cell No: 034
# 3.11 Argoverse 2 (exploratory, v6.3): real driving with LiDAR, read directly from the public AWS
#      Open Data bucket (anonymous access, no account or registration). A deterministic sample of
#      validation logs is drawn; for each log the compact tables (ego poses, calibration, labelled 3-D
#      boxes with track IDs) are read first, frame pairs of the front-centre camera are chosen from
#      them, and only the images and LiDAR sweeps of those pairs are then fetched. The ground truth is
#      built as for KITTI (Section 3.10) with the occlusion test of Section 3.10.0. One checkpointed
#      stage ("av2_gt") holds the cropped images and the ground truth, so a later session needs no
#      download. If the bucket cannot be reached or a log fails the checks (A3), that part is left out
#      of every experiment, table and figure with a note. Section 3.11b (v6.4) adds further logs in a
#      second stage, the ground contact of every vehicle and the vehicle table of all logs.
import urllib.parse, urllib.request, urllib.error
import xml.etree.ElementTree as _ET
from concurrent.futures import ThreadPoolExecutor as _AV2Pool
from scipy.spatial.transform import Rotation as _Rot

AV2_STATUS = dict(available=False, note="", logs=[], tried=0, download={})
AV2_DIR = os.path.join(SCRATCH_DIR, "av2")
AV2_RAW = os.path.join(AV2_DIR, "raw")
AV2_GT_DIR = os.path.join(AV2_DIR, "pairs")
AV2_PAIR_META, AV2_A3, AV2_VEHICLE_CHECK = [], {}, {}


class PublicS3:
    """Anonymous read access to a public S3 bucket over HTTPS: ListObjectsV2 (with continuation) and GET."""

    def __init__(self, base, timeout=60, retries=4):
        self.base = base.rstrip("/") + "/"
        self.timeout, self.retries = timeout, retries
        self.n_requests, self.n_bytes = 0, 0

    def _read(self, url):
        err = None
        for attempt in range(self.retries):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    data = r.read()
                self.n_requests += 1
                self.n_bytes += len(data)
                return data
            except urllib.error.HTTPError as e:
                if e.code in (400, 403, 404):
                    raise IOError(f"HTTP {e.code} for {url[len(self.base):][:120]}")
                err = e
            except Exception as e:
                err = e
            time.sleep(2 + 3 * attempt)
        raise IOError(f"the public bucket could not be read after {self.retries} attempts ({type(err).__name__}: {str(err)[:160]})")

    def list(self, prefix, delimiter="/"):
        keys, prefixes, token = [], [], None
        tag = lambda e: e.tag.rsplit("}", 1)[-1]
        while True:
            q = {"list-type": "2", "prefix": prefix}
            if delimiter:
                q["delimiter"] = delimiter
            if token:
                q["continuation-token"] = token
            root = _ET.fromstring(self._read(self.base + "?" + urllib.parse.urlencode(q)))
            token, truncated = None, False
            for e in root:
                t = tag(e)
                if t == "Contents":
                    keys += [c.text for c in e if tag(c) == "Key"]
                elif t == "CommonPrefixes":
                    prefixes += [c.text for c in e if tag(c) == "Prefix"]
                elif t == "IsTruncated":
                    truncated = (e.text or "").strip().lower() == "true"
                elif t == "NextContinuationToken":
                    token = e.text
            if not truncated or not token:
                return keys, prefixes

    def get(self, key, dest):
        if os.path.exists(dest):
            return dest
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        data = self._read(self.base + urllib.parse.quote(key))
        with open(dest + ".part", "wb") as f:
            f.write(data)
        os.replace(dest + ".part", dest)
        return dest


def _av2_se3(df):
    # (N, 4, 4) rigid transforms from the qw, qx, qy, qz, tx_m, ty_m, tz_m columns (scalar-first quaternions)
    q = df[["qx", "qy", "qz", "qw"]].to_numpy(np.float64)
    T = np.repeat(np.eye(4)[None], len(df), 0)
    if len(df):
        T[:, :3, :3] = _Rot.from_quat(q / np.linalg.norm(q, axis=1, keepdims=True)).as_matrix()
        T[:, :3, 3] = df[["tx_m", "ty_m", "tz_m"]].to_numpy(np.float64)
    return T


def _interp_se3(ts, T, t):
    # geodesic interpolation of rigid transforms sampled at sorted times ts (s), at times t (s); beyond
    # the ends, constant-velocity extrapolation from the two nearest samples
    t = np.atleast_1d(np.asarray(t, np.float64))
    if len(ts) == 1:
        return np.repeat(T[:1], len(t), 0)
    b = np.clip(np.searchsorted(ts, t), 1, len(ts) - 1)
    a = b - 1
    w = (t - ts[a]) / np.maximum(ts[b] - ts[a], 1e-6)
    Ra, Rb = T[a, :3, :3], T[b, :3, :3]
    rv = _Rot.from_matrix(np.einsum("nji,njk->nik", Ra, Rb)).as_rotvec()
    out = np.repeat(np.eye(4)[None], len(t), 0)
    out[:, :3, :3] = Ra @ _Rot.from_rotvec(rv * w[:, None]).as_matrix()
    out[:, :3, 3] = T[a, :3, 3] + w[:, None] * (T[b, :3, 3] - T[a, :3, 3])
    return out


def _rot_angle_deg(Ra, Rb):
    return float(np.degrees(np.arccos(np.clip((np.trace(Ra.T @ Rb) - 1) / 2, -1, 1))))


class AV2Log:
    """One Argoverse 2 sensor log: poses, calibration, labelled tracks, front-centre camera and LiDAR."""

    TABLES = ("annotations.feather", "city_SE3_egovehicle.feather", "calibration/egovehicle_SE3_sensor.feather",
              "calibration/intrinsics.feather")

    def __init__(self, s3, log_id, raw_root=None):
        self.s3, self.id = s3, log_id
        self.key = AV2_DESIGN["prefix"] + AV2_DESIGN["split"] + "/" + log_id + "/"
        self.dir = os.path.join(raw_root or AV2_RAW, log_id)
        for t in self.TABLES:
            s3.get(self.key + t, os.path.join(self.dir, *t.split("/")))
        cam = AV2_DESIGN["camera"]
        ck, _ = s3.list(self.key + f"sensors/cameras/{cam}/")
        lk, _ = s3.list(self.key + "sensors/lidar/")
        self.cam_ts = np.array(sorted(int(os.path.basename(k).split(".")[0]) for k in ck if k.endswith(".jpg")), np.int64)
        self.lidar_ts = np.array(sorted(int(os.path.basename(k).split(".")[0]) for k in lk if k.endswith(".feather")), np.int64)
        if len(self.cam_ts) < 10 or len(self.lidar_ts) < 5:
            raise IOError("too few camera images or LiDAR sweeps")
        P = pd.read_feather(os.path.join(self.dir, "city_SE3_egovehicle.feather")).sort_values("timestamp_ns")
        P = P.drop_duplicates("timestamp_ns")
        self.t0 = int(P["timestamp_ns"].iloc[0])
        self.pose_t = (P["timestamp_ns"].to_numpy(np.int64) - self.t0) * 1e-9
        self.pose_T = _av2_se3(P)
        self.origin = self.pose_T[0, :3, 3].copy()                 # local city origin (keeps coordinates small)
        self.pose_T[:, :3, 3] -= self.origin
        E = pd.read_feather(os.path.join(self.dir, "calibration", "egovehicle_SE3_sensor.feather"))
        I = pd.read_feather(os.path.join(self.dir, "calibration", "intrinsics.feather"))
        e, i_ = E[E["sensor_name"] == cam], I[I["sensor_name"] == cam]
        if not len(e) or not len(i_):
            raise IOError(f"no calibration for {cam}")
        self.ego_T_cam = _av2_se3(e)[0]
        r = i_.iloc[0]
        self.W_full, self.H_full = int(r["width_px"]), int(r["height_px"])
        # 4:3 window of the full image width (centred on the principal point's row), resized to out_wh
        Wo, Ho = AV2_DESIGN["out_wh"]
        wc = self.W_full - self.W_full % 4
        hc = wc * 3 // 4
        if hc > self.H_full:
            hc = self.H_full - self.H_full % 3; wc = hc * 4 // 3
        x0 = (self.W_full - wc) // 2
        y0 = int(np.clip(round(float(r["cy_px"]) - hc / 2), 0, self.H_full - hc))
        self.crop = (int(x0), int(y0), int(wc), int(hc))
        s = Wo / wc
        self.cam = PinholeCamera(float(r["fx_px"]) * s, float(r["fy_px"]) * s, (float(r["cx_px"]) - x0 + 0.5) * s - 0.5,
                                 (float(r["cy_px"]) - y0 + 0.5) * s - 0.5, Wo, Ho)
        A = pd.read_feather(os.path.join(self.dir, "annotations.feather"))
        A = A[A["timestamp_ns"].isin(set(self.lidar_ts.tolist()))].copy()
        A["Tego"] = list(_av2_se3(A))
        self.boxes = {int(ts): g for ts, g in A.groupby("timestamp_ns")}
        self.tracks = {}
        for uid, g in A.sort_values("timestamp_ns").groupby("track_uuid"):
            ts = (g["timestamp_ns"].to_numpy(np.int64) - self.t0) * 1e-9
            Te = self.ego_T(ts)
            Tw = np.einsum("nij,njk->nik", Te, np.stack(g["Tego"].to_list()))
            self.tracks[uid] = dict(t=ts, T=Tw, category=str(g["category"].iloc[0]),
                                    lwh=g[["length_m", "width_m", "height_m"]].median().to_numpy(np.float64))

    # ---- time and poses ---------------------------------------------------------------------------
    def sec(self, ts_ns):
        return (np.asarray(ts_ns, np.int64) - self.t0) * 1e-9

    def ego_T(self, t):
        # city_SE3_egovehicle maps ego coordinates into the city frame (documented; checked by A3 below)
        return _interp_se3(self.pose_t, self.pose_T, t)

    def cam_pose(self, t):
        # camera -> city rotation (columns: camera axes) and camera centre at time t (s)
        T = self.ego_T(t)[0] @ self.ego_T_cam
        return T[:3, :3], T[:3, 3]

    def cam_dict(self, t):
        R, C = self.cam_pose(t)
        return dict(**self.cam.to_dict(), R=R.tolist(), C=C.tolist())

    def track_T(self, uid, t):
        tr = self.tracks[uid]
        return _interp_se3(tr["t"], tr["T"], t)

    def nearest_sweep(self, ts_ns):
        return int(self.lidar_ts[np.argmin(np.abs(self.lidar_ts - int(ts_ns)))])

    # ---- pair selection from the tables alone ---------------------------------------------------------
    def _moving_vehicle_in_view(self, ts_ns):
        t = float(self.sec(ts_ns))
        R, C = self.cam_pose(t)
        for uid in self.boxes.get(self.nearest_sweep(ts_ns), pd.DataFrame(columns=["track_uuid"]))["track_uuid"]:
            tr = self.tracks.get(uid)
            if tr is None or tr["category"] not in AV2_DESIGN["classes"] or len(tr["t"]) < 2:
                continue
            if not (tr["t"][0] - 0.05 <= t <= tr["t"][-1] + 0.05):
                continue
            T2 = self.track_T(uid, [t - 0.1, t + 0.1])
            if np.linalg.norm(T2[1, :3, 3] - T2[0, :3, 3]) < 0.2 * AV2_DESIGN["min_moving_speed_mps"]:
                continue
            q = (T2[:, :3, 3].mean(0) - C) @ R
            if not (3.0 <= q[2] <= 60.0):
                continue
            u, v = self.cam.project(q)
            if 0 <= u < self.cam.W and 0 <= v < self.cam.H:
                return True
        return False

    def select_pairs(self, rng, n_max):
        lo, hi = AV2_DESIGN["pair_baseline_m"]
        t = self.sec(self.cam_ts)
        poses = [self.cam_pose(x) for x in t]
        good = [f for f in range(len(t)) if self._moving_vehicle_in_view(self.cam_ts[f])]
        out = []
        for f in rng.permutation(good):
            f = int(f)
            if len(out) >= n_max:
                break
            if any(abs(f - p[0]) < AV2_DESIGN["min_start_gap_frames"] for p in out):
                continue
            target = rng.uniform(lo, hi)
            for g in range(f + 1, min(f + 1 + AV2_DESIGN["max_gap_frames"], len(t))):
                bl = float(np.linalg.norm(poses[g][1] - poses[f][1]))
                if bl >= target:
                    ang = _rot_angle_deg(poses[f][0], poses[g][0])
                    if bl <= hi + 0.5 and ang <= AV2_DESIGN["max_rot_deg"]:
                        out.append((f, g, bl, ang))
                    break
        return [(int(self.cam_ts[f]), int(self.cam_ts[g]), bl, ang) for f, g, bl, ang in sorted(out)]

    # ---- sensor files ------------------------------------------------------------------------------
    def image_path(self, ts_ns):
        return os.path.join(self.dir, "cam", f"{int(ts_ns)}.jpg")

    def sweep_path(self, ts_ns):
        return os.path.join(self.dir, "lidar", f"{int(ts_ns)}.feather")

    def fetch(self, cam_ts, sweep_ts, workers=8):
        jobs = [(self.key + f"sensors/cameras/{AV2_DESIGN['camera']}/{int(x)}.jpg", self.image_path(x)) for x in cam_ts]
        jobs += [(self.key + f"sensors/lidar/{int(x)}.feather", self.sweep_path(x)) for x in sweep_ts]
        with _AV2Pool(workers) as ex:
            list(ex.map(lambda kd: self.s3.get(*kd), jobs))

    def image(self, ts_ns):
        x0, y0, wc, hc = self.crop
        im = Image.open(self.image_path(ts_ns)).convert("RGB").crop((x0, y0, x0 + wc, y0 + hc))
        return np.asarray(im.resize((self.cam.W, self.cam.H), Image.LANCZOS))

    def frame_points(self, ts_cam):
        """LiDAR returns of the sweep nearest to a camera image, in city coordinates at the image's time.
        Static returns are exact (ego-motion compensated by the dataset). A return on a labelled vehicle
        was captured at its own time (sweep time + offset_ns); it is carried to the image's time along
        the vehicle's labelled track, which uses only the track's relative motion."""
        ts_l = self.nearest_sweep(ts_cam)
        L = pd.read_feather(self.sweep_path(ts_l))
        X = L[["x", "y", "z"]].to_numpy(np.float64)
        tp = float(self.sec(ts_l)) + L["offset_ns"].to_numpy(np.float64) * 1e-9
        Te = self.ego_T(float(self.sec(ts_l)))[0]
        Xw = X @ Te[:3, :3].T + Te[:3, 3]
        t = float(self.sec(ts_cam))
        R, C = self.cam_pose(t)
        q = (Xw - C) @ R
        with np.errstate(divide="ignore", invalid="ignore"):
            uv = self.cam.project(q)
        pre = (q[:, 2] > 0.5) & (uv[:, 0] > -60) & (uv[:, 0] < self.cam.W + 60) & (uv[:, 1] > -60) & (uv[:, 1] < self.cam.H + 60)
        X, Xw, tp = X[pre], Xw[pre], tp[pre]
        static = np.ones(len(X), bool)
        owner = np.full(len(X), -1, np.int64)
        vehicles = []
        m_s, m_o, cut = AV2_DESIGN["static_box_margin_m"], AV2_DESIGN["own_box_margin_m"], AV2_DESIGN["ground_cut_m"]
        for r in self.boxes.get(ts_l, pd.DataFrame()).itertuples():
            Tb = r.Tego
            qb = (X - Tb[:3, 3]) @ Tb[:3, :3]
            half = np.array([r.length_m, r.width_m, r.height_m]) / 2
            static &= ~np.all(np.abs(qb) <= half + m_s, axis=1)
            if r.category in AV2_DESIGN["classes"] and r.track_uuid in self.tracks:
                ins = np.all(np.abs(qb[:, :2]) <= half[:2] + m_o, axis=1) & (qb[:, 2] <= half[2] + m_o) & (qb[:, 2] >= -half[2] + cut)
                ins &= owner < 0
                owner[ins] = len(vehicles)
                vehicles.append(dict(track=str(r.track_uuid), category=str(r.category), lwh=half * 2))
        Xt = Xw.copy()
        dt_ms = np.full(len(X), np.nan)
        for n, v in enumerate(vehicles):
            m = owner == n
            if not m.any():
                continue
            Tp = self.track_T(v["track"], tp[m])
            Ti = self.track_T(v["track"], [t])[0]
            Xo = np.einsum("nji,nj->ni", Tp[:, :3, :3], Xw[m] - Tp[:, :3, 3])      # object coordinates
            Xt[m] = Xo @ Ti[:3, :3].T + Ti[:3, 3]
            dt_ms[m] = (tp[m] - t) * 1e3
        return dict(Xw=Xt, static=static, owner=owner, vehicles=vehicles, t=t, sweep=ts_l, dt_ms=dt_ms)

    def project(self, P, t):
        R, C = self.cam_pose(t)
        q = (P["Xw"] - C) @ R
        with np.errstate(divide="ignore", invalid="ignore"):
            uv = self.cam.project(q)
        z = np.where((q[:, 2] >= AV2_DESIGN["lidar_min_depth_m"]) & (q[:, 2] <= AV2_DESIGN["lidar_max_depth_m"]), q[:, 2], np.nan)
        D, idx = lidar_zbuffer(uv, z, self.cam.H, self.cam.W, AV2_DESIGN["zbuffer_window_px"], *AV2_DESIGN["zbuffer_tol"])
        return uv, z, D, idx

    def _box_truncation(self, uid, t):
        # share of the projected box outside the image (1 if any corner is behind the camera)
        T = self.track_T(uid, [t])[0]
        l, w, h = self.tracks[uid]["lwh"] / 2
        corners = np.array([[a * l, b * w, c * h] for a in (-1, 1) for b in (-1, 1) for c in (-1, 1)])
        R, C = self.cam_pose(t)
        q = (corners @ T[:3, :3].T + T[:3, 3] - C) @ R
        if np.any(q[:, 2] <= 0.1):
            return 1.0
        uv = self.cam.project(q)
        u0, v0 = uv.min(0); u1, v1 = uv.max(0)
        area = max((u1 - u0) * (v1 - v0), 1e-9)
        iw = max(0.0, min(u1, self.cam.W - 1) - max(u0, 0.0)); ih = max(0.0, min(v1, self.cam.H - 1) - max(v0, 0.0))
        return float(1.0 - iw * ih / area)

    def pair_gt(self, ci, cj, img_i=None, img_j=None):
        ti, tj = float(self.sec(ci)), float(self.sec(cj))
        P = self.frame_points(ci)
        uv, z, D, idx = self.project(P, ti)
        has = idx >= 0
        win = np.zeros(len(z), bool)
        win[idx[has]] = True
        with np.errstate(invalid="ignore"):
            inimg = (uv[:, 0] >= -0.5) & (uv[:, 0] < self.cam.W - 0.5) & (uv[:, 1] >= -0.5) & (uv[:, 1] < self.cam.H - 0.5)
        sm = np.zeros(D.shape, bool)
        sm[has] = P["static"][idx[has]]
        Ri, Ci = self.cam_pose(ti)
        Rj, Cj = self.cam_pose(tj)
        b = Cj - Ci
        sc = dict(cams=[self.cam_dict(ti), self.cam_dict(tj)])
        vehicles, photo = [], []
        for n, v in enumerate(P["vehicles"]):
            uid = v["track"]
            tr = self.tracks[uid]
            if not (tr["t"][0] - 0.06 <= min(ti, tj) and max(ti, tj) <= tr["t"][-1] + 0.06):
                continue                                         # the track must cover both images
            own_all = (P["owner"] == n) & np.isfinite(z) & inimg
            own = own_all & win
            if own.sum() < 3:
                continue
            Ti, Tj = self.track_T(uid, [ti, tj])
            disp = float(np.linalg.norm(Tj[:3, 3] - Ti[:3, 3]))
            state = "moving" if disp >= PREREG["kt_moving_disp_m"] else "parked" if disp <= PREREG["kt_parked_disp_m"] else None
            if state is None:
                continue
            n_proj = len(np.unique(np.round(uv[own_all]).astype(int), axis=0))
            occlusion = 1.0 - len(np.unique(np.round(uv[own]).astype(int), axis=0)) / max(n_proj, 1)
            trunc = self._box_truncation(uid, ti)
            if trunc > AV2_DESIGN["max_truncation"] or occlusion > AV2_DESIGN["max_occlusion"]:
                continue
            Xi = P["Xw"][own]
            Xj = (Xi - Ti[:3, 3]) @ Ti[:3, :3] @ Tj[:3, :3].T + Tj[:3, 3]      # rigid motion along the track
            qj = (Xj - Cj) @ Rj
            uvj = self.cam.project(qj)
            okj = (qj[:, 2] > 0.5) & (uvj[:, 0] >= 0) & (uvj[:, 0] <= self.cam.W - 1) & (uvj[:, 1] >= 0) & (uvj[:, 1] <= self.cam.H - 1)
            st_, ok = lidar_vehicle_stats(sc, uv[own], uvj, z[own], okj)
            px = np.unique(np.round(uv[own][ok]).astype(int), axis=0) if ok.any() else np.zeros((0, 2), int)
            rec = dict(track=uid, label=v["category"], state=state, disp_m=disp, s=float((Tj[:3, 3] - Ti[:3, 3]) @ b / (b @ b)),
                       speed_ratio=float(disp / max(np.linalg.norm(b), 1e-9)), occlusion=float(occlusion), truncation=trunc,
                       dt_ms=float(np.nanmedian(np.abs(P["dt_ms"][own]))), pixels=px.tolist(), **st_)
            rec["n_lidar_px"] = int(len(px))
            vehicles.append(rec)
            if img_i is not None and state == "moving" and ok.sum() >= 10:
                # photometric check of the vehicle ground truth: grey value at the rigidly moved position
                # versus at the position a static point at the same depth would have
                qs = (Xi[ok] - Cj) @ Rj
                uvs = self.cam.project(qs)
                informative = np.linalg.norm(uvs - uvj[ok], axis=1) >= 3.0
                if informative.sum() >= 10:
                    g0 = sample_gray(img_i, uv[own][ok][informative])
                    photo.append((g0, sample_gray(img_j, uvj[ok][informative]), sample_gray(img_j, uvs[informative])))
        return dict(depth=D, static=sm, vehicles=vehicles, scene=sc, baseline=float(np.linalg.norm(b)), points=P,
                    uv=uv, z=z, idx=idx, photo=photo)

    def a3(self, pairs):
        # (i) static LiDAR of image i, warped with the poses, against the LiDAR of image j (relative depth
        # disagreement); (ii) grey-value error of static LiDAR pixels with the exact warp versus no warp;
        # (iii) the same for moving vehicles: exact rigid motion versus a static point at the same depth
        werr, pw, pid, veh = [], [], [], []
        for ci, cj in pairs:
            ti, tj = float(self.sec(ci)), float(self.sec(cj))
            Ii, Ij = self.image(ci), self.image(cj)
            g = self.pair_gt(ci, cj, Ii, Ij)
            Pj = self.frame_points(cj)
            if not len(Pj["static"]):
                continue
            _, zj, Dj, idxj = self.project(Pj, tj)
            Dj = np.where((idxj >= 0) & Pj["static"][np.maximum(idxj, 0)], Dj, np.nan)
            m = (g["idx"] >= 0) & g["static"]
            sel = g["idx"][m]
            Rj, Cj = self.cam_pose(tj)
            q = (g["points"]["Xw"][sel] - Cj) @ Rj
            ok = q[:, 2] > 1.0
            uvj = self.cam.project(q[ok])
            ins = (uvj[:, 0] >= 0) & (uvj[:, 0] <= self.cam.W - 1) & (uvj[:, 1] >= 0) & (uvj[:, 1] <= self.cam.H - 1)
            zo = Dj[np.round(uvj[ins, 1]).astype(int), np.round(uvj[ins, 0]).astype(int)]
            zp = q[ok][ins, 2]
            f = np.isfinite(zo)
            if f.sum() >= 200:
                werr.append(float(np.median(np.abs(zo[f] - zp[f]) / zo[f])))
            src = g["uv"][sel][ok][ins]                          # sub-pixel position of the return in image i
            if len(src) >= 200:
                g0 = sample_gray(Ii, src)
                pw.append(float(np.nanmedian(np.abs(g0 - sample_gray(Ij, uvj[ins])))))
                pid.append(float(np.nanmedian(np.abs(g0 - sample_gray(Ij, src)))))
            for g0, gr, gs in g["photo"]:
                veh.append((np.abs(g0 - gr), np.abs(g0 - gs)))
        out = dict(n_test_pairs=len(pairs), static_warp_err=float(np.median(werr)) if werr else np.nan,
                   photo_static_warp=float(np.median(pw)) if pw else np.nan, photo_static_nowarp=float(np.median(pid)) if pid else np.nan)
        er = np.concatenate([a for a, _ in veh]) if veh else np.zeros(0)
        es = np.concatenate([b_ for _, b_ in veh]) if veh else np.zeros(0)
        out.update(photo_vehicle_rigid=float(np.nanmedian(er)) if len(er) else np.nan,
                   photo_vehicle_static=float(np.nanmedian(es)) if len(es) else np.nan, n_vehicle_px=int(len(er)))
        return out, (er, es)


def _av2_a3_pass(r):
    # per log: static geometry (depth warp) and the camera model against the images (grey-value warp)
    ok = np.isfinite(r["static_warp_err"]) and r["static_warp_err"] <= 0.05
    ok &= np.isfinite(r["photo_static_warp"]) and r["photo_static_warp"] <= 0.8 * r["photo_static_nowarp"]
    return bool(ok)


def _av2_vehicle_check(err_rigid, err_static, min_px=300):
    # pooled over the accepted logs: grey values of moving vehicles agree better with their rigid motion along
    # the labelled track than with a static point at the same depth (a single log has too few pixels to decide)
    er, es = np.concatenate(err_rigid) if err_rigid else np.zeros(0), np.concatenate(err_static) if err_static else np.zeros(0)
    n = int(np.isfinite(er).sum())
    if n < min_px:
        return dict(n_px=n, rigid=float(np.nanmedian(er)) if n else np.nan, static=float(np.nanmedian(es)) if n else np.nan,
                    passed=True, note=f"not decided (only {n} informative vehicle pixels)")
    rg, st = float(np.nanmedian(er)), float(np.nanmedian(es))
    return dict(n_px=n, rigid=rg, static=st, passed=bool(rg <= 0.9 * st),
                note=f"median grey-value error {rg:.1f} (rigid motion) vs {st:.1f} (static point) over {n} pixels")


def av2_list_logs(s3):
    # validation log IDs in the bucket, sorted; the order in which logs are tried is a fixed permutation of this list
    _, prefixes = s3.list(AV2_DESIGN["prefix"] + AV2_DESIGN["split"] + "/")
    log_ids = sorted(p.rstrip("/").rsplit("/", 1)[-1] for p in prefixes)
    if not log_ids:
        raise IOError("no logs listed in the bucket")
    return log_ids, np.random.default_rng(seed_for("av2-logs")).permutation(len(log_ids))


def av2_take_log(s3, log_id, gt_dir, index0):
    """Select, download and check one log, and write its pairs (pair indices from index0) to gt_dir.
    Returns (A3 record or None, reason for leaving the log out or None, (grey errors rigid, static) or None).
    The raw files are deleted in every case; a log that fails half-way leaves no pair behind."""
    lg = AV2Log(s3, log_id)
    written = []
    try:
        pairs = lg.select_pairs(np.random.default_rng(seed_for("av2-pairs", log_id)), RUN_CONFIG["n_av2_pairs_per_log"])
        if len(pairs) < AV2_DESIGN["min_pairs_per_log"]:
            return None, f"{len(pairs)} usable pairs", None
        cams = sorted({c for p in pairs for c in p[:2]})
        lg.fetch(cams, sorted({lg.nearest_sweep(c) for c in cams}))
        r, err = lg.a3([(p[0], p[1]) for p in pairs[:6]])
        rec = dict(n_pairs=len(pairs), convention="documented (city_SE3_egovehicle: ego -> city)", **r, passed=_av2_a3_pass(r))
        if not rec["passed"]:
            return rec, "A3 failed", None
        for (ci, cj, bl, ang) in pairs:
            g = lg.pair_gt(ci, cj)
            if not any(v["state"] == "moving" for v in g["vehicles"]):
                continue
            d = os.path.join(gt_dir, f"{log_id[:8]}_{ci}_{cj}")
            os.makedirs(d)
            written.append(d)
            for k_, c_ in enumerate((ci, cj)):
                Image.fromarray(lg.image(c_)).save(os.path.join(d, f"rgb{k_}.png"))
            np.savez_compressed(os.path.join(d, "gt.npz"), depth=g["depth"].astype(np.float32), static=g["static"])
            with open(os.path.join(d, "meta.json"), "w") as f:
                json.dump(dict(set="AV2", index=index0 + len(written) - 1, seq=log_id[:8], log=log_id, frames=[int(ci), int(cj)],
                               crop=list(lg.crop), out_wh=list(AV2_DESIGN["out_wh"]), baseline=bl, rot_deg=ang,
                               dt_s=float(lg.sec(cj) - lg.sec(ci)), scene=g["scene"], vehicles=g["vehicles"], dir=d),
                          f, default=lambda o: o.item() if hasattr(o, "item") else str(o))
        rec["n_pairs_used"] = len(written)
        return rec, None, err
    except Exception:
        for d in written:
            shutil.rmtree(d, ignore_errors=True)
        raise
    finally:
        shutil.rmtree(lg.dir, ignore_errors=True)                # raw images and sweeps are no longer needed


def av2_run_logs(s3, log_ids, order, start, skip, n_target, max_tried, gt_dir, index0, a3):
    """Try logs in the fixed order from position start (skipping the IDs in skip) until n_target are accepted or
    max_tried have been tried in all; writes their pairs to gt_dir and the A3 records into a3."""
    n_pair, accepted, rejected, veh_err, tried = index0, [], {}, ([], []), 0
    for k in order[start:]:
        if len(accepted) >= n_target or start + tried >= max_tried:
            break
        log_id = log_ids[int(k)]
        if log_id in skip:
            continue
        tried += 1
        try:
            rec, why, err = av2_take_log(s3, log_id, gt_dir, n_pair)
        except Exception as e:                      # one unreadable log never stops the stage
            rejected[log_id] = f"{type(e).__name__}: {str(e)[:120]}"
            continue
        if rec is not None:
            a3[log_id] = rec
        if why:
            rejected[log_id] = why
            continue
        n_pair += rec["n_pairs_used"]
        accepted.append(log_id)
        veh_err[0].append(err[0]); veh_err[1].append(err[1])
    return accepted, rejected, tried, _av2_vehicle_check(*veh_err)


AV2_STAGE = Stage("av2_gt", "av2", dict(design={k: v for k, v in AV2_DESIGN.items() if k != "base_url"},
                                        n_logs=RUN_CONFIG["n_av2_logs"], n_pairs=RUN_CONFIG["n_av2_pairs_per_log"],
                                        prereg={k: v for k, v in PREREG.items() if k.startswith("kt_")}))
with stage("av2_gt"):
    if not RUN_CONFIG.get("use_av2", True):
        AV2_STATUS["note"] = "switched off in RUN_CONFIG"
    elif not AV2_STAGE.done():
        try:
            shutil.rmtree(AV2_DIR, ignore_errors=True); os.makedirs(AV2_GT_DIR)
            s3 = PublicS3(AV2_DESIGN["base_url"])
            log_ids, order = av2_list_logs(s3)
            accepted, rejected, AV2_STATUS["tried"], _vc = av2_run_logs(
                s3, log_ids, order, 0, set(), RUN_CONFIG["n_av2_logs"], AV2_DESIGN["max_logs_tried"], AV2_GT_DIR, 0, AV2_A3)
            AV2_STATUS["download"] = dict(requests=s3.n_requests, MiB=round(s3.n_bytes / 2**20, 1))
            with open(os.path.join(AV2_DIR, "a3.json"), "w") as f:
                json.dump(dict(a3=AV2_A3, vehicle_check=_vc, accepted=accepted, rejected=rejected, tried=AV2_STATUS["tried"],
                               download=AV2_STATUS["download"], n_logs_listed=len(log_ids)), f, indent=1, default=str)
            shutil.rmtree(AV2_RAW, ignore_errors=True)
            if not accepted:
                raise IOError(f"no log passed (tried {AV2_STATUS['tried']}: {list(rejected.values())[:3]})")
            if not _vc["passed"]:
                raise IOError("the moving-vehicle ground truth is not confirmed by the images: " + _vc["note"])
            AV2_STAGE.commit([AV2_DIR])
        except Exception as e:
            AV2_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(AV2_DIR, ignore_errors=True)

if os.path.exists(os.path.join(AV2_DIR, "a3.json")):
    _av2_rec = json.load(open(os.path.join(AV2_DIR, "a3.json")))
    AV2_A3 = _av2_rec["a3"]
    AV2_VEHICLE_CHECK = _av2_rec.get("vehicle_check", {})
    AV2_STATUS["download"] = _av2_rec.get("download", {})
    for d in sorted(glob.glob(os.path.join(AV2_GT_DIR, "*", "meta.json"))):
        m_ = json.load(open(d)); m_["dir"] = os.path.dirname(d)
        AV2_PAIR_META.append(m_)
    AV2_PAIR_META.sort(key=lambda m_: m_["index"])
    AV2_STATUS.update(available=bool(AV2_PAIR_META), logs=sorted({m_["log"] for m_ in AV2_PAIR_META}),
                      note=AV2_STATUS["note"] or ("" if AV2_PAIR_META else "no log passed A3 or no pair had a moving vehicle"))


def av2_a3_table(a3):
    return pd.DataFrame([dict(log=k[:8], pairs_used=v.get("n_pairs_used", 0), static_warp=v["static_warp_err"],
                              grey_static=f"{v['photo_static_warp']:.1f} vs {v['photo_static_nowarp']:.1f}",
                              grey_vehicles=(f"{v['photo_vehicle_rigid']:.1f} vs {v['photo_vehicle_static']:.1f} ({v['n_vehicle_px']} px)"
                                             if v.get("n_vehicle_px", 0) else "–"),
                              A3="PASS" if v["passed"] else "FAIL") for k, v in a3.items()])


if AV2_PAIR_META:
    print(f"Argoverse 2, first stage (v6.3): {len(AV2_PAIR_META)} pairs from {len(AV2_STATUS['logs'])} validation logs "
          f"(downloaded {AV2_STATUS['download'].get('MiB', 0):,.0f} MiB in {AV2_STATUS['download'].get('requests', 0)} requests)")
    print("A3 per log: static warp = median relative depth disagreement; grey-value error (0-255) of static LiDAR pixels with "
          "the exact warp vs without, and of moving vehicles with their rigid motion vs a static point at the same depth:")
    print(av2_a3_table(AV2_A3).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print("moving-vehicle ground truth against the images (pooled over the logs): "
          + (AV2_VEHICLE_CHECK.get("note", "–") + (", PASS" if AV2_VEHICLE_CHECK.get("passed") else ", FAIL")))
else:
    print("Argoverse 2 is not used in this run:", AV2_STATUS["note"] or "no data")
release_memory(); mem_report("Argoverse 2 ground truth (first stage)");

In [ ]:
# Cell No: 035
# 3.11b Argoverse 2, v6.4 (exploratory, AMENDMENT_V6_4, D49-D50).
#       (1) A second stage ("av2_gt_ext") takes further validation logs in the fixed order of Section 3.11,
#           after the logs the first stage tried, with every rule unchanged; the first stage is untouched,
#           so a session resumed from a v6.3 run restores it and downloads only the new logs.
#       (2) A third stage ("av2_contact") decides for every labelled vehicle of every pair whether its
#           ground contact is visible in the first image. Only the compact tables (poses, calibration,
#           labelled boxes) are read again; the LiDAR depth comes from the stored ground truth.
#       (3) The vehicle table of all logs, with motion regimes and contact labels, used by every
#           Argoverse 2 analysis.
AV2X_DIR = os.path.join(SCRATCH_DIR, "av2x")
AV2X_GT_DIR = os.path.join(AV2X_DIR, "pairs")
AV2C_DIR = os.path.join(SCRATCH_DIR, "av2c")
AV2X_PAIR_META, AV2X_A3, AV2X_VEHICLE_CHECK = [], {}, {}
AV2X_STATUS = dict(available=False, note="", tried=0, download={})
AV2C_STATUS = dict(available=False, note="", download={}, notes={})
AV2_CONTACT = pd.DataFrame()

# ---- (1) more logs -------------------------------------------------------------------------------------
AV2X_STAGE = Stage("av2_gt_ext", "av2x", dict(design={k: v for k, v in AV2_DESIGN.items() if k != "base_url"},
                                              n_logs=RUN_CONFIG["n_av2_logs_ext"], n_pairs=RUN_CONFIG["n_av2_pairs_per_log"],
                                              max_tried_total=AV2X_DESIGN["max_logs_tried_total"],
                                              prereg={k: v for k, v in PREREG.items() if k.startswith("kt_")}),
                   upstream=[AV2_STAGE.fp])
with stage("av2_gt_ext"):
    if not AV2_STATUS["available"]:
        AV2X_STATUS["note"] = "the first Argoverse 2 stage is not part of this run"
    elif RUN_CONFIG.get("n_av2_logs_ext", 0) <= 0:
        AV2X_STATUS["note"] = "switched off in RUN_CONFIG"
    elif not AV2X_STAGE.done():
        try:
            shutil.rmtree(AV2X_DIR, ignore_errors=True); os.makedirs(AV2X_GT_DIR)
            _first = json.load(open(os.path.join(AV2_DIR, "a3.json")))
            s3 = PublicS3(AV2_DESIGN["base_url"])
            log_ids, order = av2_list_logs(s3)
            _index0 = max(m_["index"] for m_ in AV2_PAIR_META) + 1
            accepted, rejected, AV2X_STATUS["tried"], _vc = av2_run_logs(
                s3, log_ids, order, int(_first["tried"]), set(_first["accepted"]) | set(_first["rejected"]),
                RUN_CONFIG["n_av2_logs_ext"], AV2X_DESIGN["max_logs_tried_total"], AV2X_GT_DIR, _index0, AV2X_A3)
            AV2X_STATUS["download"] = dict(requests=s3.n_requests, MiB=round(s3.n_bytes / 2**20, 1))
            with open(os.path.join(AV2X_DIR, "a3.json"), "w") as f:
                json.dump(dict(a3=AV2X_A3, vehicle_check=_vc, accepted=accepted, rejected=rejected, tried=AV2X_STATUS["tried"],
                               download=AV2X_STATUS["download"], n_logs_listed=len(log_ids), first_index=_index0),
                          f, indent=1, default=str)
            shutil.rmtree(AV2_RAW, ignore_errors=True)
            if not accepted:
                raise IOError(f"no further log passed (tried {AV2X_STATUS['tried']}: {list(rejected.values())[:3]})")
            if not _vc["passed"]:
                raise IOError("the moving-vehicle ground truth of the new logs is not confirmed by the images: " + _vc["note"])
            AV2X_STAGE.commit([AV2X_DIR])
        except Exception as e:
            AV2X_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(AV2X_DIR, ignore_errors=True)

if os.path.exists(os.path.join(AV2X_DIR, "a3.json")):
    _rec = json.load(open(os.path.join(AV2X_DIR, "a3.json")))
    AV2X_A3, AV2X_VEHICLE_CHECK, AV2X_STATUS["download"] = _rec["a3"], _rec.get("vehicle_check", {}), _rec.get("download", {})
    for d in sorted(glob.glob(os.path.join(AV2X_GT_DIR, "*", "meta.json"))):
        m_ = json.load(open(d)); m_["dir"] = os.path.dirname(d)
        AV2X_PAIR_META.append(m_)
    AV2X_PAIR_META.sort(key=lambda m_: m_["index"])
    AV2X_STATUS.update(available=bool(AV2X_PAIR_META), logs=sorted({m_["log"] for m_ in AV2X_PAIR_META}))
if AV2X_PAIR_META:
    print(f"Argoverse 2, second stage (v6.4): {len(AV2X_PAIR_META)} pairs from {len(AV2X_STATUS['logs'])} further validation logs "
          f"(downloaded {AV2X_STATUS['download'].get('MiB', 0):,.0f} MiB in {AV2X_STATUS['download'].get('requests', 0)} requests)")
    print(av2_a3_table(AV2X_A3).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print("moving-vehicle ground truth of the new logs against the images (pooled): "
          + AV2X_VEHICLE_CHECK.get("note", "–") + (", PASS" if AV2X_VEHICLE_CHECK.get("passed") else ", FAIL"))
elif AV2_STATUS["available"]:
    print("Argoverse 2, second stage: not part of this run (" + (AV2X_STATUS["note"] or "no data") + "); the analyses use the first stage")


# ---- (2) ground contact of every labelled vehicle (D50) ---------------------------------------------------
def _segment_blocked(C, X, T, half, gap):
    """True if the line of sight from the camera centre C to the point X crosses one of the boxes (poses T (n, 4, 4),
    half sizes half (n, 3)) at least `gap` metres in front of X (slab test in each box's frame)."""
    if not len(T):
        return False
    Rb, tb = T[:, :3, :3], T[:, :3, 3]
    o = np.einsum("nji,nj->ni", Rb, C[None] - tb)
    d = np.einsum("nji,j->ni", Rb, X - C)
    with np.errstate(divide="ignore", invalid="ignore"):
        t1, t2 = (-half - o) / d, (half - o) / d
    inside = np.abs(o) <= half
    lo = np.where(d == 0, np.where(inside, -np.inf, np.inf), np.minimum(t1, t2))
    hi = np.where(d == 0, np.where(inside, np.inf, -np.inf), np.maximum(t1, t2))
    t_near, t_far = lo.max(1), hi.min(1)
    t_lim = 1.0 - gap / max(float(np.linalg.norm(X - C)), 1e-9)
    return bool(np.any((t_near <= t_far) & (t_far >= 0) & (t_near <= t_lim)))


def _static_blocked(D_static, u, v, z):
    # a static LiDAR return in the columns u +- c and the rows v - r ... v that is nearer than the point
    c, r = AV2X_DESIGN["lidar_window_cols"], AV2X_DESIGN["lidar_window_rows_above"]
    win = D_static[max(v - r, 0):v + 1, max(u - c, 0):u + c + 1]
    tol = max(AV2X_DESIGN["lidar_occluder_tol"][0] * z, AV2X_DESIGN["lidar_occluder_tol"][1])
    with np.errstate(invalid="ignore"):
        return bool(np.any(win < z - tol))


def wheel_contact_rule(C, R, cam, T, lwh, T_others, half_others, D_static, points=None):
    """D50 for one vehicle: box pose T (4 x 4; x along the length, y along the width, z up, origin at the box centre)
    and size lwh (length, width, height), camera centre C and camera-to-world rotation R, the boxes of every other
    labelled object (T_others, half sizes half_others, already shrunk) and the depth of static points (NaN elsewhere).
    Each in-view wheel contact point is hidden by the vehicle's own body, by another object, by static structure, or
    visible; the label follows the Virtual KITTI 2 rule (D37). If a list is passed as points, (u, v, status) of every
    in-view point is appended to it (v6.5, for the gallery of Fig. S4)."""
    H, W = D_static.shape
    l, w, h = (float(a) for a in lwh)
    hgt, gap, cl = PREREG["x6_wheel_height_m"], AV2X_DESIGN["occluder_gap_m"], AV2X_DESIGN["body_clearance_m"]
    c_obj = T[:3, :3].T @ (C - T[:3, 3])                # camera centre in the vehicle's frame
    T_body = T.copy(); T_body[:3, 3] = T[:3, 3] + T[:3, 2] * cl / 2       # the body: the box above the ground clearance
    half_body = np.array([[l / 2, w / 2, max(h / 2 - cl / 2, 0.01)]])
    out = dict(wheels_in_view=0, wheels_visible=0, hidden_by_body=0, hidden_by_object=0, hidden_by_static=0)
    for a in (-1, 1):
        for b in (-1, 1):
            X = T[:3, :3] @ np.array([a * 0.35 * l, b * 0.40 * w, -h / 2 + hgt]) + T[:3, 3]
            q = (X - C) @ R
            if q[2] <= 0.5:
                continue
            uv = cam.project(q)
            u_, v_ = int(round(float(uv[0]))), int(round(float(uv[1])))
            if not (0 <= u_ < W and 0 <= v_ < H):
                continue
            out["wheels_in_view"] += 1
            if not (b * c_obj[1] > w / 2 or a * c_obj[0] > l / 2) or _segment_blocked(C, X, T_body[None], half_body, gap):
                out["hidden_by_body"] += 1; status = "body"
            elif _segment_blocked(C, X, T_others, half_others, gap):
                out["hidden_by_object"] += 1; status = "object"
            elif _static_blocked(D_static, u_, v_, float(q[2])):
                out["hidden_by_static"] += 1; status = "static"
            else:
                out["wheels_visible"] += 1; status = "visible"
            if points is not None:
                points.append((float(uv[0]), float(uv[1]), status))
    n_in, n_vis = out["wheels_in_view"], out["wheels_visible"]
    out["contact"] = "visible" if n_vis >= 1 else "hidden" if n_in >= 2 else "undetermined"
    return out


def av2_contact_rows(lg, meta):
    """Wheel contact visibility (D50) of every labelled vehicle of one pair, in the pair's first image."""
    ti = float(lg.sec(int(meta["frames"][0])))
    z = np.load(os.path.join(meta["dir"], "gt.npz"))
    D_static = np.where(z["static"], z["depth"], np.nan).astype(np.float64)
    H, W = D_static.shape
    R, C = lg.cam_pose(ti)
    uids, Tb, half = [], [], []                          # every labelled object at the image's time
    for uid, tr in lg.tracks.items():
        if tr["t"][0] - 0.06 <= ti <= tr["t"][-1] + 0.06:
            uids.append(uid); Tb.append(lg.track_T(uid, [ti])[0])
            half.append(np.maximum(tr["lwh"] / 2 - AV2X_DESIGN["occluder_shrink_m"], 0.01))
    uids = np.asarray(uids, dtype=object)
    Tb = np.stack(Tb) if Tb else np.zeros((0, 4, 4)); half = np.stack(half) if half else np.zeros((0, 3))
    rows = []
    for k, v in enumerate(meta["vehicles"]):
        rec = dict(scene=int(meta["index"]), mover=k, track=v["track"], contact="undetermined", wheels_visible=np.nan,
                   wheels_in_view=np.nan, hidden_by_body=0, hidden_by_object=0, hidden_by_static=0, box_area_px=np.nan,
                   visible_area_px=np.nan)
        if v["track"] not in lg.tracks:
            rows.append(rec); continue
        T = lg.track_T(v["track"], [ti])[0]
        l, w, h = lg.tracks[v["track"]]["lwh"]
        others = uids != v["track"]
        rec.update(wheel_contact_rule(C, R, lg.cam, T, (l, w, h), Tb[others], half[others], D_static))
        corners = np.array([[a * l / 2, b * w / 2, c * h / 2] for a in (-1, 1) for b in (-1, 1) for c in (-1, 1)])
        qc = (corners @ T[:3, :3].T + T[:3, 3] - C) @ R
        if np.all(qc[:, 2] > 0.1):
            uvc = lg.cam.project(qc)
            iw = max(0.0, min(uvc[:, 0].max(), W - 1) - max(uvc[:, 0].min(), 0.0))
            ih = max(0.0, min(uvc[:, 1].max(), H - 1) - max(uvc[:, 1].min(), 0.0))
            rec.update(box_area_px=float(iw * ih), visible_area_px=float(iw * ih * (1.0 - float(v.get("occlusion", 0.0)))))
        rows.append(rec)
    return rows


AV2_PAIRS_ALL = AV2_PAIR_META + AV2X_PAIR_META
AV2C_STAGE = Stage("av2_contact", "av2x", dict(rules={k: v for k, v in AV2X_DESIGN.items() if k != "max_logs_tried_total"},
                                               wheel_height=PREREG["x6_wheel_height_m"], wheel_position=[0.35, 0.40]),
                   upstream=[AV2_STAGE.fp] + ([AV2X_STAGE.fp] if AV2X_PAIR_META else []))
with stage("av2_contact"):
    if not AV2_PAIRS_ALL:
        AV2C_STATUS["note"] = "no Argoverse 2 pairs in this run"
    elif not AV2C_STAGE.done():
        try:
            shutil.rmtree(AV2C_DIR, ignore_errors=True); os.makedirs(AV2C_DIR)
            s3 = PublicS3(AV2_DESIGN["base_url"])
            _by_log, _rows = {}, []
            for m_ in AV2_PAIRS_ALL:
                _by_log.setdefault(m_["log"], []).append(m_)
            for log_id, metas in sorted(_by_log.items()):
                try:
                    lg = AV2Log(s3, log_id, raw_root=os.path.join(AV2C_DIR, "raw"))
                    for m_ in metas:
                        _rows += av2_contact_rows(lg, m_)
                except Exception as e:                  # the vehicles of an unreadable log keep no contact label
                    AV2C_STATUS["notes"][log_id] = f"{type(e).__name__}: {str(e)[:120]}"
                finally:
                    shutil.rmtree(os.path.join(AV2C_DIR, "raw"), ignore_errors=True)
            if not _rows:
                raise IOError("no contact label could be computed: " + "; ".join(list(AV2C_STATUS["notes"].values())[:3]))
            pd.DataFrame(_rows).to_csv(os.path.join(AV2C_DIR, "contact.csv"), index=False)
            AV2C_STATUS["download"] = dict(requests=s3.n_requests, MiB=round(s3.n_bytes / 2**20, 1))
            with open(os.path.join(AV2C_DIR, "contact.json"), "w") as f:
                json.dump(dict(download=AV2C_STATUS["download"], notes=AV2C_STATUS["notes"], n_logs=len(_by_log),
                               n_rows=len(_rows)), f, indent=1)
            AV2C_STAGE.commit([AV2C_DIR])
        except Exception as e:
            AV2C_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(AV2C_DIR, ignore_errors=True)
if os.path.exists(os.path.join(AV2C_DIR, "contact.csv")):
    AV2_CONTACT = pd.read_csv(os.path.join(AV2C_DIR, "contact.csv"))
    _cj = json.load(open(os.path.join(AV2C_DIR, "contact.json")))
    AV2C_STATUS.update(available=len(AV2_CONTACT) > 0, download=_cj.get("download", {}), notes=_cj.get("notes", {}))

# ---- (3) the vehicle table of all logs ------------------------------------------------------------------
MOVERS_AV2_ALL = lidar_mover_table(AV2_PAIRS_ALL, "AV2") if AV2_PAIRS_ALL else pd.DataFrame()
if len(MOVERS_AV2_ALL):
    MOVERS_AV2_ALL, AV2_EPS_NOISE, AV2_TAU_PAR, _n_pk = lidar_regimes(MOVERS_AV2_ALL)
    if len(AV2_CONTACT):
        MOVERS_AV2_ALL = MOVERS_AV2_ALL.merge(
            AV2_CONTACT.rename(columns={"contact": "gt_contact", "wheels_visible": "gt_wheels_visible",
                                        "wheels_in_view": "gt_wheels_in_view", "visible_area_px": "gt_area_px",
                                        "box_area_px": "gt_box_area_px"}).drop(columns=["track"]),
            on=["scene", "mover"], how="left")
    MOVERS_AV2 = MOVERS_AV2_ALL[MOVERS_AV2_ALL["state"] == "moving"].reset_index(drop=True)
    AV2_STATUS["logs"] = sorted({m_["log"] for m_ in AV2_PAIRS_ALL})
    _speed = np.array([m_["baseline"] / max(m_["dt_s"], 1e-6) for m_ in AV2_PAIRS_ALL])
    AV2_EGO_SPEED = dict(median=float(np.median(_speed)), p90=float(np.quantile(_speed, 0.9)), max=float(np.max(_speed)))
    print(f"\nArgoverse 2, all stages: {len(AV2_PAIRS_ALL)} pairs from {len(AV2_STATUS['logs'])} validation logs; ego speed "
          f"(camera baseline / time between the images) median {AV2_EGO_SPEED['median']:.1f} m/s, 90th percentile "
          f"{AV2_EGO_SPEED['p90']:.1f} m/s, maximum {AV2_EGO_SPEED['max']:.1f} m/s")
    print(f"label-noise floor: 90th percentile of |eps| over {_n_pk} parked vehicles = {AV2_EPS_NOISE:.2f} px; "
          f"parallel regime on Argoverse 2: |eps| < {AV2_TAU_PAR:.2f} px")
    print(f"moving vehicles: {len(MOVERS_AV2)}; regimes {MOVERS_AV2['regime'].value_counts().to_dict()}; "
          f"measurable (>= {PREREG['kt_min_lidar_px']} LiDAR px) {int(MOVERS_AV2['measurable'].sum())}, identifiable "
          f"{int((MOVERS_AV2['measurable'] & MOVERS_AV2['identifiable']).sum())}; median time between a vehicle's "
          f"LiDAR returns and its image {MOVERS_AV2['gt_dt_ms'].median():.0f} ms (compensated along the track)")
    if "gt_contact" in MOVERS_AV2:
        _pv = MOVERS_AV2[MOVERS_AV2["measurable"].astype(bool) & (MOVERS_AV2["regime"] == "parallel")]
        print(f"ground contact (D50) of the parallel, measurable moving vehicles ({len(_pv)}): "
              f"{_pv['gt_contact'].fillna('no label').value_counts().to_dict()}; in-view wheel points hidden by the vehicle's own "
              f"body {int(_pv['hidden_by_body'].sum())}, by another labelled object {int(_pv['hidden_by_object'].sum())}, by "
              f"static structure (LiDAR) {int(_pv['hidden_by_static'].sum())}"
              + (f"; logs without contact labels: {len(AV2C_STATUS['notes'])}" if AV2C_STATUS["notes"] else ""))
    elif AV2C_STATUS["note"]:
        print("ground contact of Argoverse 2 vehicles is not part of this run:", AV2C_STATUS["note"])
else:
    MOVERS_AV2 = pd.DataFrame(); AV2_EPS_NOISE = AV2_TAU_PAR = np.nan; AV2_EGO_SPEED = {}


# ---- (4) check of the rule on Virtual KITTI 2 (D50): the same rule, with the exact depth of static pixels in place of
#      the LiDAR, applied to the Virtual KITTI 2 vehicles, whose instance masks give the reference labels (D37) --------
def vk_contact_rule_rows():
    rows, by_index = [], {m_["index"]: m_ for m_ in VK_PAIR_META}
    for seq, d in MOVERS_VK.groupby("seq", sort=False):
        sc = VK_SCENE_OBJS.get(seq)
        conv = sc.box_convention() if sc is not None else None
        if conv is None:
            continue
        frame_tids = {}
        for (f, tid) in sc.pose:
            frame_tids.setdefault(int(f), []).append(int(tid))
        for scene, g in d.groupby("scene", sort=True):
            f = int(by_index[int(scene)]["frames"][0])
            A = sc.frame_gt(f)
            D_static = np.where(A["inst"] < 0, A["depth"], np.nan).astype(np.float64)
            boxes = {}
            for tid in frame_tids.get(f, []):
                gb = sc._box_geometry(f, tid, conv)
                if gb is not None:
                    T = np.eye(4); T[:3, 0], T[:3, 1], T[:3, 2], T[:3, 3] = gb["e_len"], gb["e_wid"], -gb["down"], gb["c"]
                    boxes[tid] = (T, np.array([gb["l"], gb["w"], gb["h"]], float))
            for r in g.itertuples():
                if int(r.track) not in boxes:
                    continue
                T, lwh = boxes[int(r.track)]
                oth = [t for t in boxes if t != int(r.track)]
                T_o = np.stack([boxes[t][0] for t in oth]) if oth else np.zeros((0, 4, 4))
                h_o = (np.stack([np.maximum(boxes[t][1] / 2 - AV2X_DESIGN["occluder_shrink_m"], 0.01) for t in oth])
                       if oth else np.zeros((0, 3)))
                res = wheel_contact_rule(sc.C[f], sc.R[f], sc.cam, T, lwh, T_o, h_o, D_static)
                rows.append(dict(seq=seq, scene=int(scene), mover=int(r.mover), track=int(r.track), regime=r.regime,
                                 n_eroded=int(r.gt_n_eroded), mask_rule=str(r.gt_contact), geometry_rule=res["contact"],
                                 **{k: v for k, v in res.items() if k != "contact"}))
        sc._cache.clear()
    return pd.DataFrame(rows)


def contact_rule_agreement(df, sample):
    # agreement of the geometry rule with the instance-mask rule where both decide
    d = df[df["mask_rule"].isin(["visible", "hidden"]) & df["geometry_rule"].isin(["visible", "hidden"])]
    n = len(d)
    if not n:
        return dict(sample=sample, n=0)
    hm, hg = d["mask_rule"] == "hidden", d["geometry_rule"] == "hidden"
    po = float(np.mean(hm == hg))
    pe = float(hm.mean() * hg.mean() + (1 - hm.mean()) * (1 - hg.mean()))
    return dict(sample=sample, n=n, agreement=po, kappa=(po - pe) / (1 - pe) if pe < 1 else np.nan,
                hidden_mask_rule=int(hm.sum()), hidden_geometry_rule=int(hg.sum()), hidden_both=int((hm & hg).sum()),
                share_of_mask_hidden_found=float((hm & hg).sum() / hm.sum()) if hm.sum() else np.nan,
                share_of_rule_hidden_confirmed=float((hm & hg).sum() / hg.sum()) if hg.sum() else np.nan,
                undetermined_by_rule=int((df["mask_rule"].isin(["visible", "hidden"]) & (df["geometry_rule"] == "undetermined")).sum()))


VK_CONTACT_CHECK, CONTACT_RULE_CHECK = pd.DataFrame(), pd.DataFrame()
if VK_STATUS["available"] and len(MOVERS_VK) and "gt_contact" in MOVERS_VK:
    try:
        VK_CONTACT_CHECK = vk_contact_rule_rows()
    except Exception as e:                                  # a check, never a reason to stop the run
        print(f"check of the contact rule on Virtual KITTI 2 not computed: {type(e).__name__}: {str(e)[:200]}")
if len(VK_CONTACT_CHECK):
    _x6s = (VK_CONTACT_CHECK["regime"] == "parallel") & (VK_CONTACT_CHECK["n_eroded"] >= PREREG["x6_min_eroded_px"])
    CONTACT_RULE_CHECK = pd.DataFrame([contact_rule_agreement(VK_CONTACT_CHECK, "moving vehicles"),
                                       contact_rule_agreement(VK_CONTACT_CHECK[_x6s], "parallel, >= 40 eroded px (6.8b sample)")])
    print("\nCheck of the contact rule (D50) on Virtual KITTI 2: the same rule, with the exact depth of static pixels in place "
          "of the LiDAR,\nagainst the labels from the instance masks (D37), on the vehicles where both decide:")
    print(CONTACT_RULE_CHECK.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print("rows: instance-mask rule; columns: geometry rule (moving vehicles):")
    print(pd.crosstab(VK_CONTACT_CHECK["mask_rule"], VK_CONTACT_CHECK["geometry_rule"]).to_string())
    VK_CONTACT_CHECK.to_csv(os.path.join(MANIFEST_DIR, "contact_rule_check_vkitti.csv"), index=False)
    CONTACT_RULE_CHECK.to_csv(os.path.join(RESULTS_DIR, "contact_rule_check.csv"), index=False)
elif VK_STATUS["available"]:
    print("check of the contact rule on Virtual KITTI 2: no vehicle with a 3-D box in this run")
release_memory(); mem_report("Argoverse 2 extension and ground contact");

In [ ]:
# Cell No: 036
# 3.11c The static twin of Argoverse 2 vehicles (v6.5, exploratory, D58). One stage ("av2_twin") reads the compact tables
#       of every log again and stores, for every labelled vehicle of every pair (moving and parked), the camera height
#       above its contact plane (the bottom face of its labelled box, along the box's vertical axis), the box size and
#       the image positions and visibility status of its wheel contact points (Section 3.11b rule).
AV2T_DIR = os.path.join(SCRATCH_DIR, "av2t")
AV2T_STATUS = dict(available=False, note="", download={}, notes={})
AV2_TWIN = pd.DataFrame()


def av2_twin_rows(lg, meta):
    ti = float(lg.sec(int(meta["frames"][0])))
    R, C = lg.cam_pose(ti)
    z = np.load(os.path.join(meta["dir"], "gt.npz"))
    D_static = np.where(z["static"], z["depth"], np.nan).astype(np.float64)
    uids, Tb, half = [], [], []
    for uid, tr in lg.tracks.items():
        if tr["t"][0] - 0.06 <= ti <= tr["t"][-1] + 0.06:
            uids.append(uid); Tb.append(lg.track_T(uid, [ti])[0])
            half.append(np.maximum(tr["lwh"] / 2 - AV2X_DESIGN["occluder_shrink_m"], 0.01))
    uids = np.asarray(uids, dtype=object)
    Tb = np.stack(Tb) if Tb else np.zeros((0, 4, 4)); half = np.stack(half) if half else np.zeros((0, 3))
    rows = []
    for k, v in enumerate(meta["vehicles"]):
        rec = dict(scene=int(meta["index"]), mover=k, track=v["track"], H_cam=np.nan, box_l=np.nan, box_w=np.nan, box_h=np.nan,
                   bottom_u=np.nan, bottom_v=np.nan, up_tilt_deg=np.nan, wheel_points="[]")
        if v["track"] not in lg.tracks:
            rows.append(rec); continue
        T = lg.track_T(v["track"], [ti])[0]
        l, w, h = (float(a) for a in lg.tracks[v["track"]]["lwh"])
        n_up = T[:3, 2] / np.linalg.norm(T[:3, 2])           # the box's vertical axis (up, normal to the road under it)
        X_b = T[:3, 3] - n_up * h / 2                         # centre of the bottom face: the vehicle's contact plane
        q = (X_b - C) @ R
        uv = lg.cam.project(q) if q[2] > 0.5 else np.array([np.nan, np.nan])
        pts = []
        others = uids != v["track"]
        wheel_contact_rule(C, R, lg.cam, T, (l, w, h), Tb[others], half[others], D_static, points=pts)
        rec.update(H_cam=float((C - X_b) @ n_up), box_l=l, box_w=w, box_h=h, bottom_u=float(uv[0]), bottom_v=float(uv[1]),
                   up_tilt_deg=float(np.degrees(np.arccos(np.clip(abs(n_up[2]), -1.0, 1.0)))), wheel_points=json.dumps(pts))
        rows.append(rec)
    return rows


AV2T_STAGE = Stage("av2_twin", "av2t", dict(what="camera height above each labelled vehicle's box bottom; wheel points",
                                            wheel_height=PREREG["x6_wheel_height_m"], wheel_position=[0.35, 0.40],
                                            rules={k: v for k, v in AV2X_DESIGN.items() if k != "max_logs_tried_total"}),
                   upstream=[AV2_STAGE.fp] + ([AV2X_STAGE.fp] if AV2X_PAIR_META else []))
with stage("av2_twin"):
    if not AV2_PAIRS_ALL:
        AV2T_STATUS["note"] = "no Argoverse 2 pairs in this run"
    elif not AV2T_STAGE.done():
        try:
            shutil.rmtree(AV2T_DIR, ignore_errors=True); os.makedirs(AV2T_DIR)
            s3 = PublicS3(AV2_DESIGN["base_url"])
            _by_log, _rows = {}, []
            for m_ in AV2_PAIRS_ALL:
                _by_log.setdefault(m_["log"], []).append(m_)
            for log_id, metas in sorted(_by_log.items()):
                try:
                    lg = AV2Log(s3, log_id, raw_root=os.path.join(AV2T_DIR, "raw"))
                    for m_ in metas:
                        _rows += av2_twin_rows(lg, m_)
                except Exception as e:                      # the vehicles of an unreadable log keep no camera height
                    AV2T_STATUS["notes"][log_id] = f"{type(e).__name__}: {str(e)[:120]}"
                finally:
                    shutil.rmtree(os.path.join(AV2T_DIR, "raw"), ignore_errors=True)
            if not _rows:
                raise IOError("no camera height could be computed: " + "; ".join(list(AV2T_STATUS["notes"].values())[:3]))
            pd.DataFrame(_rows).to_csv(os.path.join(AV2T_DIR, "twin.csv"), index=False)
            AV2T_STATUS["download"] = dict(requests=s3.n_requests, MiB=round(s3.n_bytes / 2**20, 1))
            with open(os.path.join(AV2T_DIR, "twin.json"), "w") as f:
                json.dump(dict(download=AV2T_STATUS["download"], notes=AV2T_STATUS["notes"], n_logs=len(_by_log), n_rows=len(_rows)),
                          f, indent=1)
            AV2T_STAGE.commit([AV2T_DIR])
        except Exception as e:
            AV2T_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(AV2T_DIR, ignore_errors=True)
if os.path.exists(os.path.join(AV2T_DIR, "twin.csv")):
    AV2_TWIN = pd.read_csv(os.path.join(AV2T_DIR, "twin.csv"))
    _tj = json.load(open(os.path.join(AV2T_DIR, "twin.json")))
    AV2T_STATUS.update(available=len(AV2_TWIN) > 0, download=_tj.get("download", {}), notes=_tj.get("notes", {}))
if len(AV2_TWIN) and len(MOVERS_AV2_ALL):
    _tw = MOVERS_AV2_ALL.merge(AV2_TWIN.drop(columns=["track"]), on=["scene", "mover"], how="left")
    _tw = _tw[_tw["measurable"].astype(bool)]
    print(f"Argoverse 2 camera heights above the vehicles' contact planes: {int(_tw['H_cam'].notna().sum())} measurable vehicles "
          f"(moving {int((_tw['state'] == 'moving').sum())}, parked {int((_tw['state'] == 'parked').sum())}); median H = "
          f"{_tw['H_cam'].median():.2f} m (5-95%: {_tw['H_cam'].quantile(0.05):.2f}-{_tw['H_cam'].quantile(0.95):.2f} m); box axis "
          f"tilt from the city vertical: median {_tw['up_tilt_deg'].median():.1f} deg")
    if AV2T_STATUS["notes"]:
        print("logs without camera heights:", len(AV2T_STATUS["notes"]))
else:
    print("The static twin of Argoverse 2 vehicles is not part of this run:", AV2T_STATUS["note"] or "no Argoverse 2 data")
release_memory(); mem_report("Argoverse 2 static twin")

In [ ]:
# Cell No: 037
# 3.11d The twin test with optical flow (v6.6, D63, H14). One stage ("av2_flowtwin") computes optical flow in both
#       directions for every Argoverse 2 pair, fits the road plane to all LiDAR returns of the first image (RANSAC over
#       horizontal planes, city z up; no labels) and stores, for every labelled vehicle (moving and parked), its epipolar
#       offset, static-hypothesis depth and the heights of its static reconstruction above the road, all from the flow at
#       its LiDAR pixels (forward-backward checked). The labels only say which pixels belong to a vehicle and whether it
#       moves; the correspondences, the plane and the camera height are what a deployed system would have.
AV2F_DIR = os.path.join(SCRATCH_DIR, "av2ft")
AV2F_STATUS = dict(available=False, note="", flow="", n_pairs=0, roads={})
AV2_FLOWTWIN = pd.DataFrame()


def av2_road_plane(depth, cam, R, C, rng):
    """Road plane of one image from all its LiDAR returns: the horizontal plane (normal within road_max_tilt_deg of the
    city z axis) with the most returns within road_inlier_m, sampled from returns below the camera, refined on its
    inliers; None if no such plane has a plausible camera height."""
    D = FLOWTWIN_DESIGN
    vv, uu = np.nonzero(np.isfinite(depth) & (depth >= D["road_depth_m"][0]) & (depth <= D["road_depth_m"][1]))
    if len(vv) < 200:
        return None
    X = C + (cam.rays(uu.astype(np.float64), vv.astype(np.float64)) * depth[vv, uu].astype(np.float64)[:, None]) @ R.T
    low = np.nonzero(X[:, 2] < C[2] - 0.5)[0]
    if len(low) < 100:
        return None
    cos_max = np.cos(np.radians(D["road_max_tilt_deg"]))
    best, n_best = None, 0
    for _ in range(D["ransac_iters"]):
        a = X[rng.choice(low, 3, replace=False)]
        n = np.cross(a[1] - a[0], a[2] - a[0]); nn = np.linalg.norm(n)
        if nn < 1e-9:
            continue
        n = n / nn * (1.0 if n[2] >= 0 else -1.0)
        if n[2] < cos_max or not (D["road_cam_height_m"][0] <= (C - a[0]) @ n <= D["road_cam_height_m"][1]):
            continue
        inl = np.abs((X - a[0]) @ n) < D["road_inlier_m"]
        if inl.sum() > n_best:
            best, n_best = inl, int(inl.sum())
    if best is None or n_best < 100:
        return None
    Q = X[best]; o = Q.mean(0)
    n = np.linalg.svd(Q - o, full_matrices=False)[2][-1]
    n = n if n[2] >= 0 else -n
    h_cam, tilt = float((C - o) @ n), float(np.degrees(np.arccos(np.clip(n[2], -1.0, 1.0))))
    if not (D["road_cam_height_m"][0] <= h_cam <= D["road_cam_height_m"][1]) or tilt > D["road_max_tilt_deg"]:
        return None
    return dict(n=n, o=o, cam_height=h_cam, tilt_deg=tilt, inlier_frac=float(n_best / len(X)), n_points=int(len(X)))


def av2_flow_vehicles(meta, f_ij, f_ji, depth, plane):
    """Flow-based quantities of every labelled vehicle of one pair (one row per vehicle, in the order of meta)."""
    D = FLOWTWIN_DESIGN
    cam, R0, C0 = frame_camera(meta["scene"], 0)
    _, R1, C1 = frame_camera(meta["scene"], 1)
    R21, t21 = relative_pose(R0, C0, R1, C1)
    F = fundamental_matrix(cam.K, cam.K, R21, t21)
    rows = []
    for k, v in enumerate(meta["vehicles"]):
        rec = dict(scene=int(meta["index"]), mover=k, n_lidar_px=0, n_flow_px=0, frac_fb_ok=np.nan, eps_flow=np.nan,
                   x_flow=np.nan, frac_behind=np.nan, h_low_flow=np.nan, h_low_true=np.nan, road=plane is not None)
        px = np.asarray(v.get("pixels", []), np.int64).reshape(-1, 2)
        rec["n_lidar_px"] = int(len(px))
        if len(px) < D["min_px"]:
            rows.append(rec); continue
        Z = depth[px[:, 1], px[:, 0]].astype(np.float64)
        p = px.astype(np.float64)
        f = f_ij[px[:, 1], px[:, 0]].astype(np.float64)
        q = p + f
        with np.errstate(invalid="ignore"):
            fb = np.linalg.norm(f + sample_map(f_ji, q), axis=1)
            ok = np.isfinite(Z) & np.isfinite(fb) & (fb < D["fb_px"] + D["fb_rel"] * np.linalg.norm(f, axis=1))
        rec.update(frac_fb_ok=float(ok.mean()), n_flow_px=int(ok.sum()))
        if ok.sum() < D["min_px"]:
            rows.append(rec); continue
        p, q, Z = p[ok], q[ok], Z[ok]
        zt, _ = triangulate_depth(p, q, cam.K, cam.K, R21, t21)
        pos = np.isfinite(zt) & (zt > 0)
        rec.update(eps_flow=float(np.median(np.abs(epipolar_offset(p, q, F)))), frac_behind=float(1.0 - pos.mean()))
        if rec["frac_behind"] <= D["behind_frac"]:
            rec["x_flow"] = float(np.median(np.log(zt[pos] / Z[pos])))
            if plane is not None:
                Xs = C0 + (cam.rays(p[pos, 0], p[pos, 1]) * zt[pos][:, None]) @ R0.T     # the static reconstruction
                Xt = C0 + (cam.rays(p[:, 0], p[:, 1]) * Z[:, None]) @ R0.T             # the LiDAR returns
                rec["h_low_flow"] = float(np.percentile((Xs - plane["o"]) @ plane["n"], D["low_pct"]))
                rec["h_low_true"] = float(np.percentile((Xt - plane["o"]) @ plane["n"], D["low_pct"]))
        rows.append(rec)
    return rows


AV2F_STAGE = Stage("av2_flowtwin", "av2ft", dict(design=_jsonable(FLOWTWIN_DESIGN), flow=FLOW_PREFERRED),
                   upstream=[AV2_STAGE.fp] + ([AV2X_STAGE.fp] if AV2X_PAIR_META else []))
with stage("av2_flowtwin"):
    if not AV2_PAIRS_ALL:
        AV2F_STATUS["note"] = "no Argoverse 2 pairs in this run"
    elif not AV2F_STAGE.done():
        try:
            shutil.rmtree(AV2F_DIR, ignore_errors=True); os.makedirs(AV2F_DIR)
            if FLOW.get("model") is None and str(FLOW.get("method") or "").startswith("RAFT"):
                FLOW["method"] = None                        # RAFT was released after an earlier stage: load it again
            _tmp = os.path.join(FLOW_DIR, "av2ft_tmp")
            _rows, _roads = [], []
            for _s in range(0, len(AV2_PAIRS_ALL), 16):     # flows are cached on disk one chunk at a time
                _chunk = AV2_PAIRS_ALL[_s:_s + 16]
                _ims = [tuple(np.asarray(Image.open(os.path.join(m_["dir"], f"rgb{k_}.png")).convert("RGB")) for k_ in (0, 1))
                        for m_ in _chunk]
                _tags = [os.path.basename(m_["dir"]) for m_ in _chunk]
                _fw = compute_flows(_ims, [os.path.join(_tmp, f"{t_}_f.npz") for t_ in _tags])
                _bw = compute_flows([(b_, a_) for a_, b_ in _ims], [os.path.join(_tmp, f"{t_}_b.npz") for t_ in _tags])
                shutil.rmtree(_tmp, ignore_errors=True)
                for m_, f1, f2 in zip(_chunk, _fw, _bw):
                    _dep = np.load(os.path.join(m_["dir"], "gt.npz"))["depth"]
                    _cam, _R0, _C0 = frame_camera(m_["scene"], 0)
                    _pl = av2_road_plane(_dep, _cam, _R0, _C0, np.random.default_rng(seed_for("av2-road", m_["index"])))
                    _roads.append(dict(scene=int(m_["index"]), log=m_["log"], found=_pl is not None,
                                       cam_height=_pl["cam_height"] if _pl else np.nan, tilt_deg=_pl["tilt_deg"] if _pl else np.nan,
                                       inlier_frac=_pl["inlier_frac"] if _pl else np.nan))
                    _rows += av2_flow_vehicles(m_, f1, f2, _dep, _pl)
                del _ims, _fw, _bw
            FLOW["model"] = None
            release_memory()
            if not _rows:
                raise IOError("no vehicle with flow")
            pd.DataFrame(_rows).to_csv(os.path.join(AV2F_DIR, "flowtwin.csv"), index=False)
            pd.DataFrame(_roads).to_csv(os.path.join(AV2F_DIR, "roads.csv"), index=False)
            with open(os.path.join(AV2F_DIR, "flowtwin.json"), "w") as f:
                json.dump(dict(flow=FLOW.get("method"), fallback_reason=FLOW.get("fallback_reason"), n_pairs=len(AV2_PAIRS_ALL),
                               n_rows=len(_rows)), f, indent=1)
            AV2F_STAGE.commit([AV2F_DIR])
        except Exception as e:
            AV2F_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(AV2F_DIR, ignore_errors=True)
            FLOW["model"] = None
if os.path.exists(os.path.join(AV2F_DIR, "flowtwin.csv")):
    AV2_FLOWTWIN = pd.read_csv(os.path.join(AV2F_DIR, "flowtwin.csv"))
    _fj = json.load(open(os.path.join(AV2F_DIR, "flowtwin.json")))
    _rd = pd.read_csv(os.path.join(AV2F_DIR, "roads.csv"))
    AV2F_STATUS.update(available=len(AV2_FLOWTWIN) > 0, flow=_fj.get("flow") or "", n_pairs=_fj.get("n_pairs", 0),
                       roads=dict(found_share=float(_rd["found"].mean()), cam_height_median=float(_rd["cam_height"].median()),
                                  tilt_median_deg=float(_rd["tilt_deg"].median())))
if len(AV2_FLOWTWIN) and len(MOVERS_AV2_ALL):
    _fw_ = MOVERS_AV2_ALL[["scene", "mover", "state", "measurable"]].merge(AV2_FLOWTWIN, on=["scene", "mover"], how="inner")
    _fw_ = _fw_[_fw_["measurable"].astype(bool)]
    _pkf = _fw_[_fw_["state"] == "parked"]
    print(f"Twin test with optical flow ({AV2F_STATUS['flow']}): {AV2F_STATUS['n_pairs']} Argoverse 2 pairs; road plane found in "
          f"{100 * AV2F_STATUS['roads']['found_share']:.0f}% of them (camera {AV2F_STATUS['roads']['cam_height_median']:.2f} m above "
          f"it, tilt {AV2F_STATUS['roads']['tilt_median_deg']:.1f} deg, medians)")
    print(f"measurable vehicles: {len(_fw_)}; with >= {FLOWTWIN_DESIGN['min_px']} forward-backward consistent LiDAR pixels: "
          f"{int((_fw_['n_flow_px'] >= FLOWTWIN_DESIGN['min_px']).sum())} (median share kept {_fw_['frac_fb_ok'].median():.2f}); "
          f"parked vehicles: median |x_flow| {np.nanmedian(np.abs(_pkf['x_flow'])):.3f}, median |eps_flow| "
          f"{np.nanmedian(_pkf['eps_flow']):.2f} px, lowest returns {np.nanmedian(_pkf['h_low_true']):.2f} m above the road "
          f"(static reconstruction from flow: {np.nanmedian(_pkf['h_low_flow']):.2f} m)")
else:
    print("The twin test with optical flow is not part of this run:", AV2F_STATUS["note"] or "no Argoverse 2 data")
release_memory(); mem_report("Argoverse 2 twin test with flow")

In [ ]:
# Cell No: 038
# 3.12 Bonn RGB-D Dynamic (v6.5, corrected in v6.6, exploratory, D56 and D62). One checkpointed stage ("bonn_gt") downloads
#      four sequences, checks the pose convention (A3), selects frame pairs, re-estimates the relative pose of every pair
#      from its static correspondences (v6.6), finds moving regions and people, and stores for every pair the undistorted
#      images, the measured depth, the static mask and the objects with their ground truth. The raw sequences are deleted.
#      v6.6 object rule: pixels hidden in frame j are not tested; a moving region counts only if the static prediction
#      fails photometrically on it (flow errors on repeated texture and occlusion bands are rejected), it is thicker than
#      a 5 px disc, smaller than 20% of the image and not flat on the floor (a moving shadow); the floor is the lowest
#      well-supported horizontal level.
import zipfile as _zipfile

BONN_DIR = os.path.join(SCRATCH_DIR, "bonn")
BONN_GT_DIR = os.path.join(BONN_DIR, "pairs")
BONN_RAW = os.path.join(DATA_DIR, "bonn_raw")
BONN_PAIR_META, BONN_A3 = [], {}
BONN_STATUS = dict(available=False, note="", sequences=[], download={}, person_model="", noise_floor_px=np.nan)
MOVERS_BONN = pd.DataFrame()
BONN_TAU_PAR = np.nan


def _se3(R, t):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = t
    return T


class BonnSequence(TUMSequence):
    """TUM-format sequence with the Bonn camera: undistorted images and depth, and the marker-to-camera calibration."""
    CONVENTIONS = ("documented", "inverse calibration", "raw file")

    def __init__(self, name, root):
        super().__init__(name, root)
        k = BONN_DESIGN["intrinsics"]
        self.cam = PinholeCamera(**k)
        K = np.array([[k["fx"], 0.0, k["cx"]], [0.0, k["fy"], k["cy"]], [0.0, 0.0, 1.0]])
        self._maps = cv2.initUndistortRectifyMap(K, np.asarray(BONN_DESIGN["distortion"], np.float64), None, K,
                                                 (k["W"], k["H"]), cv2.CV_32FC1)
        Tm = np.asarray(BONN_DESIGN["T_m"], np.float64)
        U, S, Vt = np.linalg.svd(Tm[:3, :3])                 # T_m = [scale * rotation | t]: split the two
        self._calib = dict(Tros=np.asarray(BONN_DESIGN["T_ros"], np.float64), Rm=U @ Vt, tm=Tm[:3, 3], scale=float(S.mean()))
        self._calib["Tros_inv"] = np.linalg.inv(self._calib["Tros"])
        self.convention = "documented"

    def rgb(self, i):
        im = np.asarray(Image.open(self.frames[i]["rgb"]).convert("RGB"))
        return cv2.remap(im, self._maps[0], self._maps[1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

    def raw_rgb_path(self, i):
        return self.frames[i]["rgb"]

    def depth(self, i):
        d = np.asarray(Image.open(self.frames[i]["depth"])).astype(np.float32) / BONN_DESIGN["depth_scale"]
        d[d <= 0] = np.nan
        return cv2.remap(d, self._maps[0], self._maps[1], cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=np.nan)

    def undistort_mask(self, m):
        return cv2.remap(m.astype(np.uint8), self._maps[0], self._maps[1], cv2.INTER_NEAREST,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0).astype(bool)

    def pose(self, i):
        # camera -> world rotation and camera centre; the world frame is the motion-capture frame divided by T_m's scale
        R, C = self.frames[i]["R"], self.frames[i]["C"]
        if self.convention == "raw file":
            return R, C
        c = self._calib
        A = c["Tros_inv"] @ _se3(R, C) @ c["Tros"]
        RA, tA = A[:3, :3], A[:3, 3]
        if self.convention == "documented":                  # world <- camera = A T_m
            return RA @ c["Rm"], (RA @ c["tm"] + tA) / c["scale"]
        return RA @ c["Rm"].T, c["scale"] * tA - RA @ c["Rm"].T @ c["tm"]     # world <- camera = A T_m^-1


def bonn_download(name, spec):
    """Download and extract one sequence into its own folder (paths checked; the top-level folder is stripped)."""
    seq_dir = os.path.join(BONN_RAW, name)
    flag = os.path.join(seq_dir, ".extracted")
    if os.path.exists(flag):
        return seq_dir, "already extracted", 0.0
    os.makedirs(BONN_RAW, exist_ok=True)
    zpath = os.path.join(BONN_RAW, spec["file"])
    ok, msg = (True, "cached") if os.path.exists(zpath) else download_file(BONN_DESIGN["base_url"] + spec["file"], zpath)
    if not ok:
        raise IOError(f"download failed: {msg}")
    mib = os.path.getsize(zpath) / 2**20
    shutil.rmtree(seq_dir, ignore_errors=True); os.makedirs(seq_dir)
    root = os.path.abspath(seq_dir)
    with _zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            parts = [p_ for p_ in info.filename.replace("\\", "/").split("/") if p_]
            if info.is_dir() or not parts or any(p_ in ("..", ".") for p_ in parts) or info.filename.startswith("/"):
                continue
            top_level = parts[0] not in ("rgb", "depth") and not (len(parts) == 1 and parts[0].endswith(".txt"))
            rel = parts[1:] if top_level else parts            # strip the top-level folder, if there is one
            if not rel:
                continue
            if rel[0] not in ("rgb", "depth") and not (len(rel) == 1 and rel[0].endswith(".txt")):
                continue                                       # only images, depth and the three lists are needed
            dst = os.path.normpath(os.path.join(root, *rel))
            if not dst.startswith(root + os.sep):
                continue
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with zf.open(info) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out, 1 << 20)
    os.remove(zpath)
    for f_ in ("rgb.txt", "depth.txt", "groundtruth.txt"):
        if not os.path.exists(os.path.join(seq_dir, f_)):
            raise IOError(f"{f_} missing in the archive")
    open(flag, "w").write("ok")
    return seq_dir, msg, mib


def bonn_a3(seq, rng):
    """Pose convention and the A3 test: median relative disagreement of measured depth and depth warped with the pose."""
    pairs = []
    for _ in range(400):
        i = int(rng.integers(0, len(seq) - 1))
        for j in range(i + 1, min(len(seq), i + 90)):
            b, ang = baseline_and_angle(seq, i, j)
            if b >= 0.08:
                if ang < 20:
                    pairs.append((i, j))
                break
        if len(pairs) >= 16:
            break
    errs = {}
    for conv in BonnSequence.CONVENTIONS:
        seq.convention = conv
        errs[conv] = float(np.nanmedian([depth_warp_error(seq, i, j) for i, j in pairs])) if pairs else np.nan
    best = min(errs, key=lambda k_: errs[k_] if np.isfinite(errs[k_]) else np.inf)
    seq.convention = best
    return dict(n_frames=len(seq), n_test_pairs=len(pairs), **{f"warp_err_{k_.replace(' ', '_')}": v for k_, v in errs.items()},
                convention=best, median_rel_depth_err=errs[best], passed=bool(np.isfinite(errs[best]) and errs[best] <= 0.05))


_PERSON = {"model": None, "note": ""}


def _person_model():
    if _PERSON["model"] is None and not _PERSON["note"].startswith("unavailable"):
        try:
            from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
            _w = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
            _PERSON["model"] = maskrcnn_resnet50_fpn_v2(weights=_w, progress=False).eval().to(device)
            _PERSON["note"] = f"Mask R-CNN ResNet-50 FPN v2 (torchvision {_w.name})"
        except Exception as e:
            _PERSON["note"] = f"unavailable: {type(e).__name__}: {str(e)[:160]}"
    return _PERSON["model"]


@torch.no_grad()
def bonn_person_masks(seq, frames):
    """People in the undistorted images of the given frames (COCO person, score >= person_score_min)."""
    model = _person_model()
    if model is None:
        raise RuntimeError("no person segmentation: " + _PERSON["note"])
    out = []
    for f_ in frames:
        t = torch.from_numpy(seq.rgb(f_).copy()).permute(2, 0, 1).float().div(255.0).to(device)
        r = model([t])[0]
        keep = (r["labels"] == 1) & (r["scores"] >= BONN_DESIGN["person_score_min"])
        m = (r["masks"][keep, 0] >= 0.5).any(0) if bool(keep.any()) else torch.zeros(t.shape[1:], dtype=torch.bool)
        out.append(m.cpu().numpy())
    return out


def _release_person_model():
    _PERSON["model"] = None
    release_memory()


def _angle_deg(Ra, Rb):
    return float(np.degrees(np.arccos(np.clip((np.trace(Ra.T @ Rb) - 1) / 2, -1, 1))))


def bonn_refine_pose(cam, di, Ri, Ci, Rj, Cj, f_ij, f_ji, excl, rng):
    """Relative pose i -> j from static correspondences (v6.6, D62): RANSAC PnP of frame i's measured 3-D points against
    their optical-flow positions in frame j, started from the motion-capture pose and polished on the inliers. Returns
    (Rj, Cj, record); record["refined"] is False when the fit is not trustworthy, and the pair is then left out (with the
    motion-capture pose alone, static pixels miss their rigid flow by about 2 px and are taken for motion)."""
    D = BONN_DESIGN; H, W = di.shape
    vv, uu = np.mgrid[0:H:2, 0:W:2]
    p = np.stack([uu.ravel(), vv.ravel()], 1).astype(np.float64)
    z = di[vv, uu].ravel().astype(np.float64)
    f = f_ij[vv, uu].reshape(-1, 2).astype(np.float64)
    q = p + f
    with np.errstate(invalid="ignore"):
        fb = np.linalg.norm(f + sample_map(f_ji, q), axis=1)
        ok = (np.isfinite(z) & (z > 0.3) & (z < 6.0) & np.isfinite(fb) & (fb < 0.5) & ~excl[vv, uu].ravel()
              & (q[:, 0] >= 0) & (q[:, 0] <= W - 1) & (q[:, 1] >= 0) & (q[:, 1] <= H - 1))
    rec = dict(refined=False, reason="", n_points=int(ok.sum()), inlier_frac=np.nan, med_err_px=np.nan,
               mocap_med_err_px=np.nan, rot_change_deg=np.nan, trans_change_m=np.nan)
    if ok.sum() < 300:
        rec["reason"] = "too few static correspondences"
        return Rj, Cj, rec
    idx = rng.choice(np.nonzero(ok)[0], min(D["pnp_points"], int(ok.sum())), replace=False)
    X = cam.rays(p[idx, 0], p[idx, 1]) * z[idx][:, None]
    img = q[idx]
    R21, t21 = relative_pose(Ri, Ci, Rj, Cj)

    def _err(Rr, tt):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.linalg.norm(cam.project(X @ Rr.T + tt) - img, axis=1)
    rec["mocap_med_err_px"] = float(np.nanmedian(_err(R21, t21)))
    try:
        okp, rvec, tvec, inl = cv2.solvePnPRansac(X, img, cam.K, None, rvec=cv2.Rodrigues(R21)[0].copy(),
                                                  tvec=t21.reshape(3, 1).copy(), useExtrinsicGuess=True,
                                                  iterationsCount=300, reprojectionError=D["pnp_reproj_px"],
                                                  confidence=0.999, flags=cv2.SOLVEPNP_ITERATIVE)
        if okp and inl is not None and len(inl) >= 100:
            rvec, tvec = cv2.solvePnPRefineLM(X[inl.ravel()], img[inl.ravel()], cam.K, None, rvec, tvec)
    except cv2.error as e:
        rec["reason"] = f"PnP failed: {str(e)[:80]}"
        return Rj, Cj, rec
    if not okp or inl is None or len(inl) < 100:
        rec["reason"] = "no consistent pose"
        return Rj, Cj, rec
    R21n, t21n = cv2.Rodrigues(rvec)[0], tvec.ravel()
    e = _err(R21n, t21n)
    good_px = e < D["pnp_reproj_px"]
    rec.update(inlier_frac=float(good_px.mean()), med_err_px=float(np.median(e[good_px])) if good_px.any() else np.nan,
               rot_change_deg=_angle_deg(R21, R21n), trans_change_m=float(np.linalg.norm(t21n - t21)))
    if not (rec["inlier_frac"] >= D["pnp_min_inlier_frac"] and rec["med_err_px"] <= D["pnp_max_med_err_px"]
            and rec["rot_change_deg"] <= D["pnp_max_rot_change_deg"] and rec["trans_change_m"] <= D["pnp_max_trans_change_m"]):
        rec["reason"] = "fit outside the limits"
        return Rj, Cj, rec
    rec["refined"] = True
    Rjn = Ri @ R21n.T
    return Rjn, Ci - Rjn @ t21n, rec


def bonn_floor_plane(depth, cam, R, C, up, rng):
    """The floor (v6.6): the lowest horizontal level under the camera with enough support, refined by RANSAC; a table top is
    not the floor. None if no level has the support or the camera height above it is implausible for a hand-held camera."""
    D = BONN_DESIGN
    if up is None:
        return None
    H, W = depth.shape
    vv, uu = np.mgrid[0:H:4, 0:W:4]
    z = depth[vv, uu]
    ok = np.isfinite(z) & (z > 0.3) & (z < 6.0)
    if ok.sum() < 400:
        return None
    Xw = C + (cam.rays(uu[ok].astype(np.float64), vv[ok].astype(np.float64)) * z[ok][:, None]) @ R.T
    h = (Xw - C) @ up
    lo, hi = -D["floor_cam_height_m"][1], -D["floor_cam_height_m"][0]
    need = max(80, D["floor_min_frac"] * len(Xw))
    edges = np.arange(lo, hi + 0.02, 0.02)
    cnt = np.convolve(np.histogram(h, edges)[0], np.ones(3), mode="same")
    levels = np.nonzero(cnt >= need)[0]
    if not len(levels):
        return None
    h0 = (edges[levels[0]] + edges[levels[0] + 1]) / 2          # the lowest well-supported level
    P = Xw[np.abs(h - h0) < 0.06]
    if len(P) < 3:
        return None
    best = None
    for _ in range(200):
        a = P[rng.choice(len(P), 3, replace=False)]
        n = np.cross(a[1] - a[0], a[2] - a[0]); nn = np.linalg.norm(n)
        if nn < 1e-9:
            continue
        n /= nn
        n = n if n @ up >= 0 else -n
        if np.degrees(np.arccos(np.clip(n @ up, -1, 1))) > D["floor_max_tilt_deg"]:
            continue
        inl = np.abs((Xw - a[0]) @ n) < 0.03
        if best is None or inl.sum() > best.sum():
            best = inl
    if best is None or best.sum() < need:
        return None
    Q = Xw[best]; o = Q.mean(0)
    n = np.linalg.svd(Q - o, full_matrices=False)[2][-1]
    n = n if n @ up >= 0 else -n
    ch = float((C - o) @ n)
    if not (D["floor_cam_height_m"][0] <= ch <= D["floor_cam_height_m"][1]):
        return None
    return dict(n=n, origin=o, height=ch, inlier_frac=float(best.mean()))


def _window_min_max(a, valid, r):
    """Minimum and maximum of a over the (2r+1) x (2r+1) window of every pixel, ignoring invalid pixels (+-inf if none)."""
    k_ = np.ones((2 * r + 1, 2 * r + 1), np.uint8)
    return (cv2.erode(np.where(valid, a, np.inf).astype(np.float32), k_),
            cv2.dilate(np.where(valid, a, -np.inf).astype(np.float32), k_))


def _warp_gray(g, q, H, W):
    """Grey image g sampled at the positions q (H*W, 2); a flag for the positions inside the image."""
    q = np.asarray(q, np.float32).reshape(H, W, 2)
    ok = np.isfinite(q).all(-1) & (q[..., 0] >= 0) & (q[..., 0] <= W - 1) & (q[..., 1] >= 0) & (q[..., 1] <= H - 1)
    q = np.where(ok[..., None], q, -1.0).astype(np.float32)
    return cv2.remap(g, q[..., 0], q[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0), ok


def _zncc(a, b, valid, r):
    """Zero-mean normalised cross-correlation of a and b over the (2r+1)^2 window of every pixel, the standard deviation of
    a there and a flag for windows that are at least 80% valid."""
    k_ = (2 * r + 1, 2 * r + 1)
    n = cv2.boxFilter(valid.astype(np.float32), -1, k_, normalize=False)
    a0 = np.where(valid, a, 0).astype(np.float32); b0 = np.where(valid, b, 0).astype(np.float32)
    nn = np.maximum(n, 1.0)
    ma = cv2.boxFilter(a0, -1, k_, normalize=False) / nn; mb = cv2.boxFilter(b0, -1, k_, normalize=False) / nn
    va = cv2.boxFilter(a0 * a0, -1, k_, normalize=False) / nn - ma ** 2
    vb = cv2.boxFilter(b0 * b0, -1, k_, normalize=False) / nn - mb ** 2
    cv_ = cv2.boxFilter(a0 * b0, -1, k_, normalize=False) / nn - ma * mb
    return (cv_ / np.sqrt(np.maximum(va, 1e-3) * np.maximum(vb, 1e-3)), np.sqrt(np.maximum(va, 0.0)),
            n >= 0.8 * k_[0] * k_[1])


def bonn_pair_gt(seq, i, j, Rj, Cj, f_ij, f_ji, person, plane, kind, rng, img_i, img_j):
    """Moving regions, people, objects and their ground truth for one pair (frames i -> j), the static mask, the static
    scale check and the epipolar offsets of static patches (the flow-noise floor)."""
    D = BONN_DESIGN
    cam = seq.cam; H, W = cam.H, cam.W
    di, dj = seq.depth(i), seq.depth(j)
    Ri, Ci = seq.pose(i)                                   # (Rj, Cj): the pose of frame j re-estimated from the pair
    vv, uu = np.mgrid[0:H, 0:W]
    p = np.stack([uu, vv], -1).reshape(-1, 2).astype(np.float64)
    z = di.reshape(-1).astype(np.float64)
    valid = np.isfinite(z)
    X = cam.rays(p[:, 0], p[:, 1]) * np.where(valid, z, 1.0)[:, None]
    Xj = (X @ Ri.T + Ci - Cj) @ Rj
    with np.errstate(divide="ignore", invalid="ignore"):
        p_rig = cam.project(Xj)
    p_obs = p + f_ij.reshape(-1, 2)
    zj_obs = sample_map(np.nan_to_num(dj, nan=0.0), p_rig)
    zj_obs[zj_obs <= 0] = np.nan
    back = sample_map(f_ji, p_obs)
    fb_ok = np.linalg.norm(f_ij.reshape(-1, 2) + back, axis=1) < 1.0 + 0.05 * np.linalg.norm(f_ij.reshape(-1, 2), axis=1)
    flow_res = np.linalg.norm(p_obs - p_rig, axis=1)
    with np.errstate(invalid="ignore"):
        depth_bad = np.abs(zj_obs - Xj[:, 2]) > np.maximum(0.10, 0.08 * Xj[:, 2])
    vimg = np.isfinite(di).astype(np.float32)
    nvalid = cv2.boxFilter(vimg, -1, (3, 3), normalize=False)
    gy, gx = np.gradient(np.nan_to_num(di, nan=0.0))
    edge = (((np.hypot(gx, gy) > 0.08) & (nvalid >= 9)) | (nvalid < 7)).reshape(-1)
    inside = np.isfinite(p_obs).all(1) & (p_obs[:, 0] >= 0) & (p_obs[:, 0] <= W - 1) & (p_obs[:, 1] >= 0) & (p_obs[:, 1] <= H - 1)
    base = valid & fb_ok & ~edge & inside
    # hidden in frame j: everything observed near the predicted position is nearer than the point (occluded, not moving)
    _, dmax = _window_min_max(np.nan_to_num(dj, nan=0.0), np.isfinite(dj), D["occl_window_px"])
    zmax_j = sample_map(np.where(np.isfinite(dmax), dmax, 0.0).astype(np.float32), p_rig)
    with np.errstate(invalid="ignore"):
        occluded = (zmax_j > 0) & (zmax_j < Xj[:, 2] - np.maximum(0.10, 0.08 * Xj[:, 2]))
    dyn = base & ~occluded & np.isfinite(zj_obs) & (depth_bad | (flow_res > 3.0))
    dyn_img = cv2.morphologyEx(dyn.reshape(H, W).astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n_cc, lab, stats, _ = cv2.connectedComponentsWithStats(dyn_img, connectivity=8)
    keep = np.zeros(n_cc, bool); keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= BONN_DESIGN["min_component_px"]
    dyn_m = keep[lab]
    # epipolar offset, static-hypothesis depth and translational parallax of every pixel with a correspondence
    R21, t21 = relative_pose(Ri, Ci, Rj, Cj)
    F = fundamental_matrix(cam.K, cam.K, R21, t21)
    eps = np.full(H * W, np.nan); ztri = np.full(H * W, np.nan); par = np.full(H * W, np.nan)
    idx = np.nonzero(base)[0]
    eps[idx] = np.abs(epipolar_offset(p[idx], p_obs[idx], F))
    ztri[idx], _ = triangulate_depth(p[idx], p_obs[idx], cam.K, cam.K, R21, t21)
    par[idx] = translational_parallax(p[idx], p_obs[idx], cam.K, cam.K, R21)
    eps, ztri, par = eps.reshape(H, W), ztri.reshape(H, W), par.reshape(H, W)
    # static pixels: valid correspondence, depth in the real-data range, away from moving regions and people
    lo, hi = PREREG["static_depth_range_real"]
    away = ~cv2.dilate((dyn_m | person).astype(np.uint8), np.ones((19, 19), np.uint8)).astype(bool)
    static = base.reshape(H, W) & away & (di >= lo) & (di <= hi)
    with np.errstate(divide="ignore", invalid="ignore"):
        lr = np.log(ztri / di)
    sv = lr[static & (ztri > 0) & np.isfinite(lr)]
    scale_log = float(np.median(sv)) if len(sv) >= 500 else np.nan
    # flow-noise floor: median |eps| of static 40 x 40 patches (>= 80% static pixels)
    P_ = BONN_DESIGN["static_patch_px"]
    patches = []
    for _ in range(4 * BONN_DESIGN["n_static_patches"]):
        a, b_ = int(rng.integers(0, W - P_)), int(rng.integers(0, H - P_))
        sm = static[b_:b_ + P_, a:a + P_]
        if sm.mean() >= 0.8:
            patches.append(float(np.median(eps[b_:b_ + P_, a:a + P_][sm])))
        if len(patches) >= BONN_DESIGN["n_static_patches"]:
            break
    # photometric check of the static prediction: frame j warped with the rigid flow and with the measured flow, compared
    # with frame i by 7 x 7 ZNCC on textured windows
    gi = img_i.astype(np.float32).mean(-1); gj = img_j.astype(np.float32).mean(-1)
    w_rig, ok_rig = _warp_gray(gj, p_rig, H, W); w_obs, ok_obs = _warp_gray(gj, p_obs, H, W)
    ok_rig &= valid.reshape(H, W)                          # the rigid prediction needs a measured depth
    z_rig, sd_i, full_r = _zncc(gi, w_rig, ok_rig, D["photo_window_px"])
    z_obs, _, full_o = _zncc(gi, w_obs, ok_obs, D["photo_window_px"])
    textured = (sd_i >= D["photo_min_texture"]) & full_r & full_o
    # objects: every moving region split into its part on people and its part off them; an object larger than
    # max_object_frac of the image or wider/taller than max_object_extent of it is not an object, nor is a region that the
    # static prediction explains photometrically, one thinner than a disc of min_object_radius_px, or one lying flat on
    # the floor (v6.6)
    person_d = cv2.dilate(person.astype(np.uint8), np.ones((2 * D["person_dilate_px"] + 1,) * 2, np.uint8)).astype(bool)
    flag_frac = float((dyn_m & ~person_d).mean())          # share of the image flagged as moving outside people (QC)
    objs, n_large, n_photo, n_thin, n_flat = [], 0, 0, 0, 0
    ring = np.ones((2 * D["photo_window_px"] + 1,) * 2, np.uint8)
    for comp in range(1, n_cc):
        if not keep[comp]:
            continue
        cm = lab == comp
        for part, knd in ((cm & ~person_d, kind), (cm & person_d, "person")):
            n2, lab2, st2, _ = cv2.connectedComponentsWithStats(part.astype(np.uint8), connectivity=8)
            for c2 in range(1, n2):
                reg = lab2 == c2
                if knd != "person" and (st2[c2, cv2.CC_STAT_AREA] > D["max_object_frac"] * H * W
                                        or st2[c2, cv2.CC_STAT_WIDTH] > D["max_object_extent"] * W
                                        or st2[c2, cv2.CC_STAT_HEIGHT] > D["max_object_extent"] * H):
                    n_large += 1
                    continue
                px_m = erode_mask(reg, PREREG["erosion_px"]) & np.isfinite(eps) & np.isfinite(di)
                if px_m.sum() < PREREG["bonn_min_object_px"]:
                    continue
                radius = float(cv2.distanceTransform(np.pad(reg, 1).astype(np.uint8), cv2.DIST_L2, 5).max())
                if radius < D["min_object_radius_px"]:
                    n_thin += 1
                    continue
                tx = cv2.dilate(reg.astype(np.uint8), ring).astype(bool) & textured
                zs = float(np.median(z_rig[tx])) if tx.sum() >= D["photo_min_px"] else np.nan
                sup = float(np.median(z_obs[tx] - z_rig[tx])) if tx.sum() >= D["photo_min_px"] else np.nan
                if not (np.isfinite(zs) and zs <= D["photo_max_zncc_static"] and sup >= D["photo_min_support"]):
                    n_photo += 1
                    continue
                vv2, uu2 = np.nonzero(reg & np.isfinite(di))
                h_reg = None
                if plane is not None and len(vv2):
                    Xw = Ci + (cam.rays(uu2.astype(np.float64), vv2.astype(np.float64)) * di[vv2, uu2][:, None]) @ Ri.T
                    h_reg = (Xw - plane["origin"]) @ plane["n"]
                    if knd != "person" and np.percentile(h_reg, 90) < D["min_object_height_m"]:
                        n_flat += 1                        # flat on the floor: a moving shadow or reflection
                        continue
                zz, ee, tt, pp = di[px_m].astype(np.float64), eps[px_m], ztri[px_m], par[px_m]
                pos = tt > 0
                with np.errstate(divide="ignore", invalid="ignore"):
                    xr = float(np.median(np.log(tt[pos] / zz[pos]))) if pos.mean() >= 0.5 else np.nan
                v_, u_ = np.nonzero(reg)
                rec = dict(kind=knd, n_px=int(px_m.sum()), area_px=int(reg.sum()), Z_true=float(np.median(zz)),
                           eps_med=float(np.median(ee)), parallax_med=float(np.median(pp)), x_geo_raw=xr,
                           x_geo=xr - scale_log if np.isfinite(xr) and np.isfinite(scale_log) else np.nan,
                           no_static_solution=bool(pos.mean() < 0.5),
                           bbox=[int(u_.min()), int(v_.min()), int(u_.max()), int(v_.max())],
                           touches_bottom=bool(v_.max() >= H - 3), height_above_floor=np.nan, floor="undetermined",
                           zncc_static=zs, photo_support=sup, radius_px=radius)
                if h_reg is not None and not rec["touches_bottom"]:
                    h_low = float(np.percentile(h_reg, 5))
                    rec["height_above_floor"] = h_low
                    rec["floor"] = ("on the floor" if h_low <= PREREG["bonn_floor_contact_m"]
                                    else "off the floor" if h_low >= PREREG["bonn_floor_clear_m"] else "undetermined")
                yy, xx = np.nonzero(px_m)
                rec["pixels"] = np.stack([xx, yy], 1).tolist()
                objs.append(rec)
    return dict(depth=di, static=static, objects=objs, scale_log=scale_log, static_patch_eps=patches,
                n_moving_px=int(dyn_m.sum()), n_person_px=int(person.sum()), flag_frac=flag_frac, n_large=n_large,
                n_photo=n_photo, n_thin=n_thin, n_flat=n_flat, occluded_frac=float(occluded.mean()))


def bonn_object_table(metas):
    rows = []
    for m_ in metas:
        for k, o in enumerate(m_["objects"]):
            r = {("gt_" + a if a not in ("kind", "floor") else a): b for a, b in o.items() if a not in ("pixels", "bbox")}
            r.update(set="BONN", scene=m_["index"], seq=m_["seq"], mover=k, uid=f"BONN-{m_['index']:04d}-{k}", cls=o["kind"],
                     cam_kind="hand-held", motion_type="object", s=np.nan, track=f"{m_['seq']}/{m_['frames'][0]}")
            rows.append(r)
    return pd.DataFrame(rows)


def bonn_regimes(df, metas):
    # the parallel threshold is the flow-noise floor: 90th percentile of the median |eps| of static patches, >= 1 px
    pe = np.concatenate([np.asarray(m_.get("static_patch_eps", []), float) for m_ in metas]) if metas else np.zeros(0)
    noise = float(np.nanquantile(pe, 0.9)) if np.isfinite(pe).sum() >= 20 else np.nan
    tau = max(PREREG["eps_parallel_max"], noise if np.isfinite(noise) else PREREG["eps_parallel_max"])
    df["regime"] = np.where(df["gt_eps_med"] < tau, "parallel",
                            np.where(df["gt_eps_med"] >= max(PREREG["eps_perp_min"], 2 * tau), "perpendicular", "intermediate"))
    df["x_par"] = np.nan
    df["measurable"] = df["gt_n_px"] >= PREREG["bonn_min_object_px"]
    df["identifiable"] = df["gt_parallax_med"] >= PREREG["parallax_min_px"]
    df["gt_n_eroded"] = df["gt_n_px"]
    return df, noise, tau


BONN_STAGE = Stage("bonn_gt", "bonn", dict(design={k: v for k, v in BONN_DESIGN.items() if k != "base_url"},
                                           n_pairs=RUN_CONFIG["n_bonn_pairs_per_seq"], flow=FLOW_PREFERRED,
                                           regimes=[PREREG["eps_parallel_max"], PREREG["eps_perp_min"]],
                                           erosion=PREREG["erosion_px"], static_range=PREREG["static_depth_range_real"],
                                           prereg={k: v for k, v in PREREG.items() if k.startswith("bonn_")},
                                           plane=_jsonable(RABL_DESIGN)))
with stage("bonn_gt"):
    if not RUN_CONFIG.get("use_bonn", True):
        BONN_STATUS["note"] = "switched off in RUN_CONFIG"
    elif not BONN_STAGE.done():
        try:
            shutil.rmtree(BONN_DIR, ignore_errors=True); os.makedirs(BONN_GT_DIR)
            if FLOW.get("model") is None and str(FLOW.get("method") or "").startswith("RAFT"):
                FLOW["method"] = None                    # RAFT was released after Section 3.7: load it again
            _a3, _dl, _notes, _n, _refine = {}, {}, {}, 0, []
            for name, spec in BONN_DESIGN["sequences"].items():
                try:
                    seq_dir, msg, mib = bonn_download(name, spec)
                    _dl[name] = dict(note=msg, MiB=round(mib, 1))
                    seq = BonnSequence(name, seq_dir)
                    rec = bonn_a3(seq, np.random.default_rng(seed_for("bonn-a3", name)))
                    _a3[name] = rec
                    if not rec["passed"]:
                        _notes[name] = f"A3 failed ({rec['median_rel_depth_err']:.3f} with the {rec['convention']} convention)"
                        continue
                    vert = sequence_vertical(seq)
                    pairs = select_dynamic_pairs(seq, RUN_CONFIG["n_bonn_pairs_per_seq"], np.random.default_rng(seed_for("bonn-pairs", name)),
                                                 b_range=BONN_DESIGN["pair_baseline_m"], max_angle=BONN_DESIGN["max_rot_deg"])
                    imgs = {k_: seq.rgb(k_) for k_ in sorted({x for p_ in pairs for x in p_[:2]})}
                    fw = compute_flows([(imgs[i], imgs[j]) for i, j, _, _ in pairs],
                                       [os.path.join(FLOW_DIR, "bonn", f"{name}_{i}_{j}.npz") for i, j, _, _ in pairs])
                    bw = compute_flows([(imgs[j], imgs[i]) for i, j, _, _ in pairs],
                                       [os.path.join(FLOW_DIR, "bonn", f"{name}_{j}_{i}.npz") for i, j, _, _ in pairs])
                    persons = bonn_person_masks(seq, [i for i, _, _, _ in pairs])
                    rng_b = np.random.default_rng(seed_for("bonn-gt", name))
                    for (i, j, bl, ang), f1, f2, pm in zip(pairs, fw, bw, persons):
                        Ri, Ci = seq.pose(i); Rj0, Cj0 = seq.pose(j)
                        di = seq.depth(i)
                        excl = cv2.dilate(pm.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
                        Rj, Cj, prec = bonn_refine_pose(seq.cam, di, Ri, Ci, Rj0, Cj0, f1, f2, excl,
                                                        np.random.default_rng(seed_for("bonn-pnp", name, i)))
                        _refine.append(dict(seq=name, i=int(i), j=int(j), **prec))
                        if not prec["refined"]:
                            continue                         # no trustworthy relative pose: the pair is left out
                        plane = bonn_floor_plane(di, seq.cam, Ri, Ci, vert, np.random.default_rng(seed_for("bonn-plane", name, i)))
                        g = bonn_pair_gt(seq, i, j, Rj, Cj, f1, f2, pm, plane, spec["object"], rng_b, imgs[i], imgs[j])
                        _refine[-1].update(flag_frac=g["flag_frac"], n_large=g["n_large"], n_photo=g["n_photo"],
                                           n_thin=g["n_thin"], n_flat=g["n_flat"], occluded_frac=g["occluded_frac"],
                                           n_objects=len(g["objects"]),
                                           floor_found=plane is not None)
                        if not g["objects"]:
                            continue
                        d = os.path.join(BONN_GT_DIR, f"{name}_{i:05d}_{j:05d}")
                        os.makedirs(d)
                        Image.fromarray(imgs[i]).save(os.path.join(d, "rgb0.png"))
                        Image.fromarray(imgs[j]).save(os.path.join(d, "rgb1.png"))
                        np.savez_compressed(os.path.join(d, "gt.npz"), depth=g["depth"].astype(np.float32), static=g["static"])
                        scene = dict(cams=[dict(**seq.cam.to_dict(), R=Ri.tolist(), C=Ci.tolist()),
                                           dict(**seq.cam.to_dict(), R=Rj.tolist(), C=Cj.tolist())])
                        with open(os.path.join(d, "meta.json"), "w") as f:
                            json.dump(dict(set="BONN", index=_n, seq=name, frames=[int(i), int(j)], baseline=bl, rot_deg=ang,
                                           scene=scene, objects=g["objects"], scale_log=g["scale_log"],
                                           static_patch_eps=g["static_patch_eps"], n_moving_px=g["n_moving_px"],
                                           n_person_px=g["n_person_px"], flag_frac=g["flag_frac"], pose_refine=prec,
                                           plane=None if plane is None else dict(height=plane["height"], inlier_frac=plane["inlier_frac"]),
                                           dir=d), f, default=lambda o: o.item() if hasattr(o, "item") else str(o))
                        _n += 1
                    Image.fromarray(seq.rgb(len(seq) // 2)).save(os.path.join(BONN_DIR, f"thumb_{name}.png"))
                    del imgs, fw, bw, persons
                except Exception as e:                      # one unreadable sequence never stops the stage
                    _notes[name] = f"{type(e).__name__}: {str(e)[:200]}"
                finally:
                    shutil.rmtree(os.path.join(BONN_RAW, name), ignore_errors=True)
                    shutil.rmtree(os.path.join(FLOW_DIR, "bonn"), ignore_errors=True)
            _release_person_model()
            FLOW["model"] = None
            pd.DataFrame(_refine).to_csv(os.path.join(BONN_DIR, "pose_refine.csv"), index=False)
            with open(os.path.join(BONN_DIR, "a3.json"), "w") as f:
                json.dump(dict(a3=_a3, download=_dl, notes=_notes, person_model=_PERSON["note"], flow=FLOW.get("method"),
                               n_pairs=_n), f, indent=1, default=str)
            if _n == 0:
                raise IOError("no Bonn pair with an object: " + "; ".join(f"{k}: {v}" for k, v in list(_notes.items())[:4]))
            BONN_STAGE.commit([BONN_DIR])
        except Exception as e:
            BONN_STATUS["note"] = f"{type(e).__name__}: {str(e)[:300]}"
            shutil.rmtree(BONN_DIR, ignore_errors=True)
            _release_person_model()

if os.path.exists(os.path.join(BONN_DIR, "a3.json")):
    _brec = json.load(open(os.path.join(BONN_DIR, "a3.json")))
    BONN_A3 = _brec["a3"]
    BONN_STATUS.update(download=_brec.get("download", {}), person_model=_brec.get("person_model", ""), notes=_brec.get("notes", {}))
    for _p in sorted(glob.glob(os.path.join(BONN_GT_DIR, "*", "meta.json"))):
        _m = json.load(open(_p)); _m["dir"] = os.path.dirname(_p)
        BONN_PAIR_META.append(_m)
    BONN_PAIR_META.sort(key=lambda m_: m_["index"])
    BONN_STATUS.update(available=bool(BONN_PAIR_META), sequences=sorted({m_["seq"] for m_ in BONN_PAIR_META}))
BONN_QC = {}
if os.path.exists(os.path.join(BONN_DIR, "pose_refine.csv")):
    _pr = pd.read_csv(os.path.join(BONN_DIR, "pose_refine.csv"))
    _ok = _pr[_pr["refined"].astype(bool)]
    BONN_QC = dict(n_pairs_tried=len(_pr), n_refined=len(_ok), refined_share=float(len(_ok) / max(len(_pr), 1)),
                   median_inlier_frac=float(_ok["inlier_frac"].median()) if len(_ok) else np.nan,
                   median_err_px=float(_ok["med_err_px"].median()) if len(_ok) else np.nan,
                   median_mocap_err_px=float(_pr["mocap_med_err_px"].median()),
                   median_rot_change_deg=float(_ok["rot_change_deg"].median()) if len(_ok) else np.nan,
                   median_trans_change_m=float(_ok["trans_change_m"].median()) if len(_ok) else np.nan,
                   median_flag_frac=float(_ok["flag_frac"].median()) if len(_ok) and "flag_frac" in _ok else np.nan,
                   p90_flag_frac=float(_ok["flag_frac"].quantile(0.9)) if len(_ok) and "flag_frac" in _ok else np.nan,
                   floor_found_share=float(_ok["floor_found"].astype(bool).mean()) if len(_ok) and "floor_found" in _ok else np.nan,
                   n_rejected_large=int(_ok["n_large"].sum()) if len(_ok) and "n_large" in _ok else 0,
                   n_rejected_photo=int(_ok["n_photo"].sum()) if len(_ok) and "n_photo" in _ok else 0,
                   n_rejected_thin=int(_ok["n_thin"].sum()) if len(_ok) and "n_thin" in _ok else 0,
                   n_rejected_flat=int(_ok["n_flat"].sum()) if len(_ok) and "n_flat" in _ok else 0)
    # quality gate of the ground truth: if the median refined pair still flags much of its static scene as moving,
    # the objects are not trustworthy and Bonn is not interpreted (v6.6)
    BONN_STATUS["suspect"] = bool(np.isfinite(BONN_QC["median_flag_frac"])
                                  and BONN_QC["median_flag_frac"] > BONN_DESIGN["qc_max_object_frac"])
if BONN_PAIR_META:
    MOVERS_BONN, _bn_noise, BONN_TAU_PAR = bonn_regimes(bonn_object_table(BONN_PAIR_META), BONN_PAIR_META)
    BONN_STATUS["noise_floor_px"] = _bn_noise
    MOVERS_BONN.to_csv(os.path.join(MANIFEST_DIR, "bonn_objects.csv"), index=False)
    print(f"Bonn RGB-D Dynamic: {len(BONN_PAIR_META)} pairs with at least one object from {len(BONN_STATUS['sequences'])} sequences "
          f"({sum(v.get('MiB', 0) for v in BONN_STATUS['download'].values()):,.0f} MiB downloaded); people: {BONN_STATUS['person_model']}")
    print(pd.DataFrame([dict(sequence=k, frames=v["n_frames"], convention=v["convention"],
                             warp_err=f"{v['median_rel_depth_err']:.3f}", A3="PASS" if v["passed"] else "FAIL")
                        for k, v in BONN_A3.items()]).to_string(index=False))
    if BONN_QC:
        print(f"relative pose re-estimated from static correspondences for {BONN_QC['n_refined']} of {BONN_QC['n_pairs_tried']} "
              f"pairs (the others are left out): median reprojection error {BONN_QC['median_mocap_err_px']:.2f} px with the "
              f"motion-capture pose, {BONN_QC['median_err_px']:.2f} px after (inliers {BONN_QC['median_inlier_frac']:.2f}); "
              f"median change {BONN_QC['median_rot_change_deg']:.2f} deg, {100 * BONN_QC['median_trans_change_m']:.1f} cm")
        print(f"share of the image flagged as moving outside people: median {BONN_QC['median_flag_frac']:.3f} (90th percentile "
              f"{BONN_QC['p90_flag_frac']:.3f}); regions not taken as objects: {BONN_QC['n_rejected_large']} too large, "
              f"{BONN_QC['n_rejected_photo']} explained by the static prediction, {BONN_QC['n_rejected_thin']} too thin, "
              f"{BONN_QC['n_rejected_flat']} flat on the floor (shadows); floor "
              f"found in {100 * BONN_QC['floor_found_share']:.0f}% of the pairs"
              + ("  -> QUALITY GATE FAILED: Bonn is not interpreted" if BONN_STATUS.get("suspect") else ""))
    _sc = np.array([m_["scale_log"] for m_ in BONN_PAIR_META], float)
    print(f"static scale check: median |log(Z_tri / Z)| over static pixels per pair = {np.nanmedian(np.abs(_sc)):.3f} "
          f"(90th percentile {np.nanpercentile(np.abs(_sc), 90):.3f}); subtracted from every object's x_geo")
    print(f"flow-noise floor: 90th percentile of the median |eps| of static patches = {_bn_noise:.2f} px; "
          f"parallel regime on Bonn: |eps| < {BONN_TAU_PAR:.2f} px")
    print("objects by kind and regime:")
    print(MOVERS_BONN.groupby(["cls", "regime"]).size().unstack(fill_value=0).to_string())
    print("objects by kind and floor contact:")
    print(MOVERS_BONN.groupby(["cls", "floor"]).size().unstack(fill_value=0).to_string())
    if BONN_STATUS.get("notes"):
        print("sequences left out:", BONN_STATUS["notes"])
else:
    print("Bonn RGB-D Dynamic is not used in this run:", BONN_STATUS["note"] or "no data")
release_memory(); mem_report("Bonn RGB-D Dynamic")

In [ ]:
# Cell No: 039
# 4.1 Model code: repositories cloned at pinned commits and imported from source.
#     No requirements.txt is installed (see Section 0.3); minimal dependencies were
#     installed under the constraints file in Section 0.3.2.
REPOS = {
    "dust3r":       dict(url="https://github.com/naver/dust3r.git",
                         commit="4c24a6ebf04809f2cfe59915e51779c8984aaa40", submodules=True,
                         syspath=["", "croco"], position="front"),
    "mast3r":       dict(url="https://github.com/naver/mast3r.git",
                         commit="f5209afc300cec36239a7ac992263f36847bbba0", submodules=True,
                         syspath=[""], position="back"),      # after dust3r: shares naver/dust3r code
    "vggt":         dict(url="https://github.com/facebookresearch/vggt.git",
                         commit="a288dd0f14786c93483e45524328726ab7b1b4ce", submodules=False,
                         syspath=[""], position="front"),
    "pi3":          dict(url="https://github.com/yyfz/Pi3.git",
                         commit="9fa3ddb3f8d53041f8b2738df404f62223bbaa7b", submodules=False,
                         syspath=[""], position="front"),
    "map-anything": dict(url="https://github.com/facebookresearch/map-anything.git",
                         commit="3d10cf7a3016fc0f9bb13a071ee66c47b10be0d9", submodules=False,
                         syspath=[""], position="front"),
}

def _git(args, cwd=None, timeout=900):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-400:]

REPO_STATUS = {}
with stage("clone_repos"):
    for name, spec in REPOS.items():
        d = os.path.join(REPO_DIR, name)
        note = []
        if not os.path.isdir(os.path.join(d, ".git")):
            shutil.rmtree(d, ignore_errors=True)
            ok, msg = _git(["clone", "--filter=blob:none", spec["url"], d])
            if not ok:
                REPO_STATUS[name] = dict(ok=False, commit=None, note=f"clone failed: {msg}")
                continue
        ok, msg = _git(["checkout", "-q", spec["commit"]], cwd=d)
        note.append("pinned commit" if ok else f"pinned commit unavailable, using HEAD ({msg[:80]})")
        if spec["submodules"]:
            ok_s, msg_s = _git(["submodule", "update", "--init", "--recursive"], cwd=d)
            if not ok_s:
                note.append(f"submodules failed: {msg_s[:120]}")
        _, head = _git(["rev-parse", "HEAD"], cwd=d)
        for sub in spec["syspath"]:
            p = os.path.join(d, sub) if sub else d
            if p not in sys.path:
                if spec["position"] == "front":
                    sys.path.insert(0, p)
                else:
                    sys.path.append(p)
        REPO_STATUS[name] = dict(ok=True, commit=head.strip()[:12], note="; ".join(note))
importlib.invalidate_caches()
for k, v in REPO_STATUS.items():
    print(f"{k:<14}{'PASS' if v['ok'] else 'FAIL':<6}{str(v['commit']):<14}{v['note']}")

In [ ]:
# Cell No: 040
# 4.2 Common adapter interface. Every adapter takes native 4:3 uint8 RGB images (640x480, or 500x375
#     for Virtual KITTI 2) and returns depth (and confidence) maps resampled to the native pixel grid.
from PIL import Image as _PIL

def resize_rgb(img, size, method=_PIL.LANCZOS):
    return np.array(_PIL.fromarray(img).resize(size, method))            # writable copy (torch.from_numpy)

def to_native(arr, H, W):
    return cv2.resize(np.asarray(arr, np.float32), (W, H), interpolation=cv2.INTER_LINEAR)

class ReconModel:
    name, family, training, checkpoint = "", "", "", ""
    input_size, precision = (0, 0), "fp32"

    def __init__(self):
        self.model = None
        self.status, self.note, self.n_params, self.load_seconds = "NOT LOADED", "", None, None

    # -- lifecycle ------------------------------------------------------------------
    def import_code(self):
        raise NotImplementedError

    def build(self):
        raise NotImplementedError

    def load(self):
        t0 = time.time()
        self.import_code()
        m = self.build().eval()
        for p in m.parameters():
            p.requires_grad_(False)
        self.model = m
        self.n_params = sum(p.numel() for p in m.parameters())
        self.load_seconds = time.time() - t0
        return self

    def to_gpu(self):
        self.model.to(device)
        return self

    def unload(self):
        # drop every reference to the weights and hand the memory back (GPU and host); the model's
        # results are already archived when this is called
        self.model = None
        release_memory()

    def autocast(self):
        if self.precision == "fp32":
            return contextlib.nullcontext()
        return torch.autocast(device_type=device.type, dtype=AMP_DTYPE)

    # -- inference (implemented per family) --------------------------------------------
    def pair(self, im0, im1):
        # depth/conf of view 0 given views (0, 1)
        raise NotImplementedError

    def mono(self, im0):
        raise NotImplementedError

    def multi(self, ims):
        raise NotImplementedError

print("Adapter base ready.")

In [ ]:
# Cell No: 041
# 4.3 CroCo-family adapters (DUSt3R, MASt3R, MonST3R): the asymmetric two-view network of the shared
#     naver/dust3r code base. Depth of view 1 is the z-coordinate of its pointmap; the single-image
#     reference is the image paired with itself (the monocular protocol of the DUSt3R paper).
#     (The v6.0 Easi3R attention probe on 10-frame clips was withdrawn after the first full run, D33.)

class CroCoModel(ReconModel):
    family, input_size, precision = "croco", (512, 384), "fp32"
    model_class = "dust3r"

    def import_code(self):
        if self.model_class == "mast3r":
            from mast3r.model import AsymmetricMASt3R as cls
        else:
            from dust3r.model import AsymmetricCroCo3DStereo as cls
        self._cls = cls

    def build(self):
        return self._cls.from_pretrained(self.checkpoint)

    def _tensor(self, img):
        x = torch.from_numpy(resize_rgb(img, self.input_size)).float().permute(2, 0, 1) / 255.0
        return (x - 0.5) / 0.5

    @torch.no_grad()
    def _forward(self, tens, names, pairs):
        W, H = self.input_size
        B = len(pairs)
        v1 = dict(img=torch.stack([tens[a] for a, b in pairs]).to(device),
                  true_shape=torch.tensor([[H, W]] * B, dtype=torch.int32),
                  instance=[names[a] for a, b in pairs], idx=[a for a, b in pairs])
        v2 = dict(img=torch.stack([tens[b] for a, b in pairs]).to(device),
                  true_shape=torch.tensor([[H, W]] * B, dtype=torch.int32),
                  instance=[names[b] for a, b in pairs], idx=[b for a, b in pairs])
        pred1, pred2 = self.model(v1, v2)
        return pred1, pred2

    def _depth_conf(self, pred, k, Hn, Wn):
        z = pred["pts3d"][k][..., 2].float().cpu().numpy()
        c = pred["conf"][k].float().cpu().numpy()
        return to_native(z, Hn, Wn), to_native(c, Hn, Wn)

    def pair(self, im0, im1):
        Hn, Wn = im0.shape[:2]
        p1, _ = self._forward([self._tensor(im0), self._tensor(im1)], ["0", "1"], [(0, 1)])
        d, c = self._depth_conf(p1, 0, Hn, Wn)
        return dict(depth=d, conf=c)

    def mono(self, im0):
        Hn, Wn = im0.shape[:2]
        p1, _ = self._forward([self._tensor(im0)], ["0"], [(0, 0)])
        d, c = self._depth_conf(p1, 0, Hn, Wn)
        return dict(depth=d, conf=c)

    def pairs_from_frame0(self, im0, others):
        # depth of frame 0 paired (as reference view) with each image in `others`
        Hn, Wn = im0.shape[:2]
        tens = [self._tensor(im0)] + [self._tensor(o) for o in others]
        names = ["0"] + [f"x{k}" for k in range(len(others))]
        out = []
        for k in range(1, len(tens)):
            p1, _ = self._forward(tens, names, [(0, k)])
            d, c = self._depth_conf(p1, 0, Hn, Wn)
            out.append(dict(depth=d, conf=c))
        return out


class DUSt3RModel(CroCoModel):
    name, training = "DUSt3R", "static"
    checkpoint = "naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt"

class MonST3RModel(CroCoModel):
    name, training = "MonST3R", "dynamic (DUSt3R fine-tuned on video)"
    checkpoint = "Junyi42/MonST3R_PO-TA-S-W_ViTLarge_BaseDecoder_512_dpt"

class MASt3RModel(CroCoModel):
    name, training, model_class = "MASt3R", "static", "mast3r"
    checkpoint = "naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric"

print("CroCo-family adapters ready: DUSt3R, MASt3R, MonST3R.")

In [ ]:
# Cell No: 042
# 4.4 Multi-view adapters: VGGT-1B, pi3, MapAnything. Each takes N >= 1 frames jointly;
#     the depth of frame 0 is returned at native resolution.

class MultiViewModel(ReconModel):
    family = "multiview"

    def pair(self, im0, im1):
        return self.multi([im0, im1])

    def mono(self, im0):
        return self.multi([im0])

class VGGTModel(MultiViewModel):
    name, training, checkpoint = "VGGT-1B", "multi-dataset mixture", "facebook/VGGT-1B"
    input_size = (504, 378)

    def __init__(self):
        super().__init__()
        self.precision = {torch.float16: "fp16", torch.bfloat16: "bf16"}.get(AMP_DTYPE, "fp32")

    def import_code(self):
        from vggt.models.vggt import VGGT
        self._cls = VGGT

    def build(self):
        m = self._cls.from_pretrained(self.checkpoint)
        m.track_head = None                       # tracking head is not used
        return m

    @torch.no_grad()
    def multi(self, ims):
        Hn, Wn = ims[0].shape[:2]
        x = torch.stack([torch.from_numpy(resize_rgb(im, self.input_size, _PIL.BICUBIC)).float().permute(2, 0, 1) / 255.0
                         for im in ims])[None].to(device)
        with self.autocast():
            pred = self.model(x)
        d = pred["depth"][0, 0, :, :, 0].float().cpu().numpy()
        c = pred["depth_conf"][0, 0].float().cpu().numpy()
        return dict(depth=to_native(d, Hn, Wn), conf=to_native(c, Hn, Wn))

class Pi3Model(MultiViewModel):
    name, training, checkpoint = "pi3", "multi-dataset mixture", "yyfz233/Pi3"
    input_size = (560, 420)

    def __init__(self):
        super().__init__()
        self.precision = {torch.float16: "fp16", torch.bfloat16: "bf16"}.get(AMP_DTYPE, "fp32")

    def import_code(self):
        from pi3.models.pi3 import Pi3
        self._cls = Pi3

    def build(self):
        return self._cls.from_pretrained(self.checkpoint)

    @torch.no_grad()
    def multi(self, ims):
        Hn, Wn = ims[0].shape[:2]
        x = torch.stack([torch.from_numpy(resize_rgb(im, self.input_size)).float().permute(2, 0, 1) / 255.0
                         for im in ims])[None].to(device)
        with self.autocast():
            res = self.model(x)
        d = res["local_points"][0, 0, :, :, 2].float().cpu().numpy()
        c = torch.sigmoid(res["conf"][0, 0, :, :, 0].float()).cpu().numpy()
        return dict(depth=to_native(d, Hn, Wn), conf=to_native(c, Hn, Wn))

class MapAnythingModel(MultiViewModel):
    name, training, checkpoint = "MapAnything", "multi-dataset mixture", "facebook/map-anything"
    input_size, precision = (504, 378), "fp32"

    def import_code(self):
        from mapanything.models import MapAnything
        from uniception.models.encoders.image_normalizations import IMAGE_NORMALIZATION_DICT
        self._cls, self._norms = MapAnything, IMAGE_NORMALIZATION_DICT

    def build(self):
        return self._cls.from_pretrained(self.checkpoint)

    @torch.no_grad()
    def multi(self, ims):
        Hn, Wn = ims[0].shape[:2]
        W, H = self.input_size
        norm_type = self.model.encoder.data_norm_type
        nz = self._norms[norm_type]
        mean = torch.as_tensor(nz.mean).view(3, 1, 1); std = torch.as_tensor(nz.std).view(3, 1, 1)
        views = []
        for i, im in enumerate(ims):
            x = torch.from_numpy(resize_rgb(im, self.input_size)).float().permute(2, 0, 1) / 255.0
            views.append(dict(img=((x - mean) / std)[None], data_norm_type=[norm_type],
                              true_shape=np.int32([[H, W]]), idx=i, instance=str(i)))
        preds = self.model.infer(views, memory_efficient_inference=True, use_amp=False,
                                 apply_mask=False, mask_edges=False)
        d = preds[0]["depth_z"][0, :, :, 0].float().cpu().numpy()
        c = preds[0]["conf"][0].float().cpu().numpy()
        return dict(depth=to_native(d, Hn, Wn), conf=to_native(c, Hn, Wn))

print("Multi-view adapters ready: VGGT-1B, pi3, MapAnything.")

In [ ]:
# Cell No: 043
# 4.5 Model registry and evaluation stages.
#     Every model is evaluated in one checkpointed stage per testbed ("eval:<model>:<testbed>",
#     Section 0.5), so a later session restores finished (model, testbed) results exactly and
#     runs only what is missing. A model whose stages can all be restored is never downloaded.
#     The load check (finite, positive depth from two views, one view and, for multi-view models,
#     a 3-frame window) runs in the master loop on the same loaded copy that is then evaluated, so
#     each model is loaded once per session (Section 5).
MODEL_CLASSES = [DUSt3RModel, MASt3RModel, MonST3RModel, VGGTModel, Pi3Model, MapAnythingModel]
VK_TRAINING_EXPOSURE = {"MASt3R": True, "VGGT-1B": True}   # Virtual KITTI listed in the published training data
MEASURE_VERSION = "m6.2"                             # version of the measurement code in Sections 4.6 and 5

_EVAL_PREREG = {k: PREREG[k] for k in ("erosion_px", "min_object_pixels", "mask_fpr", "static_depth_range_synth",
                                        "static_depth_range_real", "static_depth_range_vk")}

def eval_testbeds(cls):
    tbs = ["MAIN", "FLAT", "ABL"]
    if META_REAL:
        tbs.append("REAL")
    if META_RABL:
        tbs.append("RABL")
    if cls.family == "multiview":
        tbs.append("E4")
    if VK_STATUS["available"] and VK_PAIR_META:
        tbs.append("VK")
    if KT_STATUS["available"] and KT_PAIR_META:
        tbs.append("KT")
    if AV2_STATUS["available"] and AV2_PAIR_META:
        tbs.append("AV2")
    if AV2X_STATUS["available"] and AV2X_PAIR_META:
        tbs.append("AV2X")                            # v6.4: the further Argoverse 2 logs (pooled with AV2 in every analysis)
    tbs += ["BSW", "WDR"]                             # v6.5: the designed experiments (D55)
    if BONN_STATUS["available"] and BONN_PAIR_META:
        tbs.append("BONN")                            # v6.5: Bonn RGB-D Dynamic (D56)
    return tbs

def _eval_upstream(tb):
    return {"MAIN": [RENDER_FP["MAIN"], FLOW_FP.get("MAIN", "")], "FLAT": [RENDER_FP["FLAT"]], "ABL": [RENDER_FP["ABL"]],
            "REAL": [TUM_STAGES["REAL"].fp, FLOW_FP.get("REAL", "")], "RABL": [RABL_STAGE.fp], "E4": [RENDER_FP["E4"]],
            "VK": [VK_GT_STAGE.fp], "KT": [KT_STAGE.fp], "AV2": [AV2_STAGE.fp], "AV2X": [AV2X_STAGE.fp],
            "BSW": [RENDER_FP["BSW"]], "WDR": [RENDER_FP["WDR"]], "BONN": [BONN_STAGE.fp]}[tb]

def eval_stage(cls, tb):
    m = cls()                                         # constructor only: no download, no weights
    cfg = dict(model=m.name, checkpoint=m.checkpoint, input=list(m.input_size), precision=m.precision, testbed=tb,
               measure=MEASURE_VERSION, prereg=_EVAL_PREREG, window_sizes=list(RUN_CONFIG["e4_window_sizes"]))
    st = Stage(f"eval:{m.name}:{tb}", "eval", cfg, upstream=_eval_upstream(tb))
    st.out_dir = os.path.join(CACHE_DIR, m.name, tb)
    return st

EVAL_STAGES = {cls.name: {tb: eval_stage(cls, tb) for tb in eval_testbeds(cls)} for cls in MODEL_CLASSES}
MODELS, MODEL_STATUS = {}, {}

def _check_depth(out, ref, tag):
    d = out["depth"]
    assert d.shape == ref.shape[:2], f"{tag}: depth shape {d.shape}"
    fin = np.isfinite(d)
    assert fin.mean() > 0.99, f"{tag}: only {fin.mean():.1%} finite"
    assert np.median(d[fin]) > 0, f"{tag}: non-positive median depth"

def load_check(m):
    # one forward pass of every kind the evaluation uses, on a CONVOY-3D pair and an E4 window
    ims = [load_rgb(META_MAIN[0]["dir"], k) for k in range(2)]
    _check_depth(m.pair(ims[0], ims[1]), ims[0], "two views")
    _check_depth(m.mono(ims[0]), ims[0], "single image")
    if m.family == "multiview":
        e4 = [load_rgb(META_E4[0]["dir"], k) for k in range(3)]
        _check_depth(m.multi(e4), e4[0], "3-frame window")

for cls in MODEL_CLASSES:
    m = cls()
    MODEL_STATUS[m.name] = dict(model=m.name, family=m.family, training=m.training, checkpoint=m.checkpoint,
                                input=f"{m.input_size[0]}x{m.input_size[1]}", precision=m.precision, status="PENDING",
                                params_M=None, load_s=None, peak_gpu_GiB=None, check_s=None, note="")
    MODELS[m.name] = cls
print("Models and their evaluation stages:")
for cls in MODEL_CLASSES:
    st = EVAL_STAGES[cls.name]
    n_av = sum(s_.available() is not None for s_ in st.values())
    print(f"  {cls.name:<12} {', '.join(st)}  ({n_av}/{len(st)} restorable from checkpoints)")

In [ ]:
# Cell No: 044
# 4.6 Measurement protocol shared by every model and baseline.
#   * static pixels: finite ground-truth depth inside the pre-registered range, not on a mover,
#     not within 9 px of a mover, not under a mover's moving shadow (Virtual KITTI 2: not on or
#     within 9 px of any vehicle, moving or parked);
#   * one scale factor per image and model: median(Z_true / Z_pred) over static pixels;
#   * validity-gate statistics per image: OLS slope of log Z_pred on log Z_true and AbsRel
#     after scale alignment, over static pixels;
#   * per mover: rho = median log(scale * Z_pred / Z_true) over its visible pixels eroded
#     by 2 px;
#   * dynamic-mask scores: threshold = 95th percentile over the image's static pixels
#     (5% false-positive rate); recall = share of mover pixels above it; AUROC of mover
#     pixels against a fixed random sample of static pixels.
from scipy.stats import rankdata

def static_mask(r, kind):
    d = r["depth"]
    mv = r["mover_id"] >= 0
    near = cv2.dilate(mv.astype(np.uint8), np.ones((19, 19), np.uint8)).astype(bool)
    lo, hi = PREREG["static_depth_range_real"] if kind == "real" else PREREG["static_depth_range_synth"]
    m = np.isfinite(d) & (d >= lo) & (d <= hi) & ~near
    if kind == "real":
        m &= r["sem"] == SEM["static_box"]
    else:
        m &= ~r["mover_shadow"]
    return m

def vk_static_mask(depth, inst):
    # Virtual KITTI 2: finite depth in range, not on any vehicle (moving or parked) or within 9 px of one
    veh = cv2.dilate((inst >= 0).astype(np.uint8), np.ones((19, 19), np.uint8)).astype(bool)
    lo, hi = PREREG["static_depth_range_vk"]
    return np.isfinite(depth) & (depth >= lo) & (depth <= hi) & ~veh

def mover_masks(scene, r0, r1, a=0, b=1):
    # eroded pixels of every mover in frame a that are visible in frame b (the measurement pixels)
    pB, zB, valid, vis = gt_correspondence(scene, r0, a, b, r1)
    out = []
    for j in range(len(scene["movers"])):
        out.append(erode_mask(r0["mover_id"] == j, PREREG["erosion_px"]) & vis)
    return out, pB, vis

def scale_and_gate(pred, gt, smask, max_px=40000, rng_seed=0):
    p, g = pred[smask].astype(np.float64), gt[smask].astype(np.float64)
    ok = np.isfinite(p) & (p > 0)
    p, g = p[ok], g[ok]
    if len(p) < 100:
        return dict(scale=np.nan, slope=np.nan, absrel=np.nan, n_static=int(len(p)))
    s = float(np.median(g / p))
    if len(p) > max_px:
        sel = np.random.default_rng(rng_seed).choice(len(p), max_px, replace=False)
        p, g = p[sel], g[sel]
    slope = float(np.polyfit(np.log(g), np.log(p), 1)[0])
    absrel = float(np.mean(np.abs(s * p - g) / g))
    return dict(scale=s, slope=slope, absrel=absrel, n_static=int(ok.sum()))

def mover_rho(pred, gt, mask, scale):
    p = pred[mask].astype(np.float64); g = gt[mask].astype(np.float64)
    ok = np.isfinite(p) & (p > 0) & np.isfinite(g)
    if ok.sum() < PREREG["min_object_pixels"] or not np.isfinite(scale):
        return np.nan
    return float(np.median(np.log(scale * p[ok] / g[ok])))

def _auroc(pos, neg):
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    r = rankdata(np.concatenate([pos, neg]))
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))

def mask_scores(score, smask, masks, seed=0, n_neg=20000):
    # recall at the image's 5%-FPR threshold and AUROC, for every mover mask
    s_static = score[smask]
    s_static = s_static[np.isfinite(s_static)]
    if len(s_static) < 100:
        return [dict(recall=np.nan, auroc=np.nan) for _ in masks]
    thr = np.quantile(s_static, 1 - PREREG["mask_fpr"])
    neg = s_static if len(s_static) <= n_neg else np.random.default_rng(seed).choice(s_static, n_neg, replace=False)
    out = []
    for m in masks:
        pos = score[m]; pos = pos[np.isfinite(pos)]
        if len(pos) < PREREG["min_object_pixels"]:
            out.append(dict(recall=np.nan, auroc=np.nan)); continue
        out.append(dict(recall=float(np.mean(pos > thr)), auroc=_auroc(pos, neg)))
    return out

M1_RESOLUTION_PX = 1e-6        # D27: exact offsets are compared at 1e-6 px (float64 noise is ~1e-11 px)

def epipolar_residual_map(scene, p_obs, a=0, b=1):
    # M1: |eps| of every pixel's correspondence under the true relative pose (frames a -> b). Rounded to
    # 1e-6 px: with exact correspondences, the static scene and a parallel mover both have eps = 0 up to
    # floating-point noise, and that noise must not decide which of the two a threshold flags.
    H, W = p_obs.shape[:2]
    vv, uu = np.mgrid[0:H, 0:W]
    cam0, R0, C0 = frame_camera(scene, a); cam1, R1, C1 = frame_camera(scene, b)
    R21, t21 = relative_pose(R0, C0, R1, C1)
    F = fundamental_matrix(cam0.K, cam1.K, R21, t21)
    e = np.abs(epipolar_offset(np.stack([uu, vv], -1).astype(np.float64), p_obs, F))
    return np.round(e / M1_RESOLUTION_PX) * M1_RESOLUTION_PX

def rigid_flow_residual_map(scene, depth_scaled, p_obs, a=0, b=1):
    # M2: |observed correspondence - correspondence implied by the depth and the true motion|
    cam0, R0, C0 = frame_camera(scene, a); cam1, R1, C1 = frame_camera(scene, b)
    H, W = depth_scaled.shape
    vv, uu = np.mgrid[0:H, 0:W]
    X0 = cam0.rays(uu.astype(np.float64), vv.astype(np.float64)) * depth_scaled[..., None].astype(np.float64)
    R21, t21 = relative_pose(R0, C0, R1, C1)
    X1 = X0 @ R21.T + t21
    with np.errstate(divide="ignore", invalid="ignore"):
        p_rig = cam1.project(X1)
    res = np.linalg.norm(p_obs - p_rig, axis=-1)
    res[X1[..., 2] <= 0.05] = np.nan
    return res

def classical_triangulation_depth(scene, p_obs, a=0, b=1):
    cam0, R0, C0 = frame_camera(scene, a); cam1, R1, C1 = frame_camera(scene, b)
    H, W = p_obs.shape[:2]
    vv, uu = np.mgrid[0:H, 0:W]
    R21, t21 = relative_pose(R0, C0, R1, C1)
    z, _ = triangulate_depth(np.stack([uu, vv], -1).astype(np.float64), p_obs, cam0.K, cam1.K, R21, t21)
    return z

def flow_to_pobs(flow):
    H, W = flow.shape[:2]
    vv, uu = np.mgrid[0:H, 0:W]
    return np.stack([uu, vv], -1).astype(np.float64) + flow

print("Measurement protocol ready.")

In [ ]:
# Cell No: 045
# 4.7 Model-independent baselines on every CONVOY-3D clip, CONVOY-REAL pair (and both conditions of
#     the real-background contact ablation) and Virtual KITTI 2 pair:
#   * classical two-view triangulation with the TRUE relative pose (positive control), from
#     exact correspondences and (CONVOY-3D, CONVOY-REAL) from estimated optical flow;
#   * M1 epipolar-residual mask from exact correspondences and (CONVOY-3D, CONVOY-REAL) from flow.
#   One checkpointed stage (Section 0.5).
BASELINE_ROWS = []
BASELINE_STAGE = Stage("baselines", "eval", dict(measure=MEASURE_VERSION, prereg=_EVAL_PREREG, m1_resolution=M1_RESOLUTION_PX,
                                                 sets=dict(real=len(META_REAL), rabl=len(META_RABL), vk=len(VK_PAIR_META))),
                       upstream=[RENDER_FP["MAIN"], TUM_STAGES["REAL"].fp, RABL_STAGE.fp, FLOW_FP.get("MAIN", ""),
                                 FLOW_FP.get("REAL", ""), VK_GT_STAGE.fp if VK_STATUS["available"] else ""])
BASELINE_PATH = os.path.join(CACHE_DIR, "baselines", "baselines_per_mover.csv")

def _baseline_rows(set_, idx, masks, gt, z_exact, m1_exact, sm_exact, seed, z_flow=None, m1_flow=None, sm_flow=None):
    sc_e = mask_scores(m1_exact, sm_exact, masks, seed=seed)
    sc_f = mask_scores(m1_flow, sm_flow, masks, seed=seed) if m1_flow is not None else [dict(recall=np.nan, auroc=np.nan)] * len(masks)
    rows = []
    for j, m in enumerate(masks):
        g = gt[m].astype(np.float64); ze = z_exact[m]
        okf = np.zeros(0, bool)
        if z_flow is not None:
            zf = z_flow[m]; okf = np.isfinite(zf) & (zf > 0)
        rows.append(dict(set=set_, scene=idx, mover=j, uid=f"{set_}-{idx:04d}-{j}",
                         rho_classical_flow=float(np.median(np.log(zf[okf] / g[okf]))) if okf.sum() >= PREREG["min_object_pixels"] else np.nan,
                         frac_behind_classical_flow=float(np.mean(zf <= 0)) if z_flow is not None and len(zf) else np.nan,
                         rho_classical_exact=float(np.median(np.log(ze[ze > 0] / g[ze > 0]))) if np.sum(ze > 0) >= PREREG["min_object_pixels"] else np.nan,
                         recall_M1_exact=sc_e[j]["recall"], auroc_M1_exact=sc_e[j]["auroc"],
                         recall_M1_flow=sc_f[j]["recall"], auroc_M1_flow=sc_f[j]["auroc"]))
    return rows

with stage("baselines"):
    if not BASELINE_STAGE.done():
        for meta in META_MAIN + META_REAL:
            kind = "real" if meta["set"] == "REAL" else "synth"
            sc = meta["scene"]
            r0, r1 = load_frame(meta["dir"], 0), load_frame(meta["dir"], 1)
            masks, pB, vis = mover_masks(sc, r0, r1)
            sm = static_mask(r0, kind)
            p_flow = flow_to_pobs(load_flow(meta))
            BASELINE_ROWS += _baseline_rows(meta["set"], meta["index"], masks, r0["depth"], classical_triangulation_depth(sc, pB),
                                            epipolar_residual_map(sc, pB), sm & vis, meta["index"],
                                            classical_triangulation_depth(sc, p_flow), epipolar_residual_map(sc, p_flow), sm)
        for meta in META_RABL:
            sc = meta["scene"]
            r0, r1 = load_frame(meta["dir"], 0), load_frame(meta["dir"], 1)
            masks, pB, vis = mover_masks(sc, r0, r1)
            sm = static_mask(r0, "real")
            rows = _baseline_rows("RABL", meta["index"], masks, r0["depth"], classical_triangulation_depth(sc, pB),
                                  epipolar_residual_map(sc, pB), sm & vis, meta["index"])
            BASELINE_ROWS += [dict(r, condition=meta["condition"]) for r in rows]
        for meta in VK_PAIR_META:
            vsc = VK_SCENE_OBJS[meta["seq"]]
            i, j = meta["frames"]
            g = vsc.pair_gt(i, j)
            masks = [erode_mask(g["mover_id"] == k, PREREG["erosion_px"]) & g["visible"] for k in range(len(g["movers"]))]
            sm = vk_static_mask(g["depth"], g["inst"])
            BASELINE_ROWS += _baseline_rows(meta["set"], meta["index"], masks, g["depth"], classical_triangulation_depth(g["scene"], g["pB"]),
                                            epipolar_residual_map(g["scene"], g["pB"]), sm & g["visible"], meta["index"])
        for _sc in VK_SCENE_OBJS.values():
            _sc._cache.clear()
        os.makedirs(os.path.dirname(BASELINE_PATH), exist_ok=True)
        pd.DataFrame(BASELINE_ROWS).to_csv(BASELINE_PATH, index=False)
        BASELINE_STAGE.commit([BASELINE_PATH])
BASELINES = pd.read_csv(BASELINE_PATH)
if "condition" not in BASELINES:
    BASELINES["condition"] = np.nan
release_memory()
print(f"baselines: {len(BASELINES)} movers")
_gt_b = pd.concat([MOVERS_MAIN, MOVERS_REAL, MOVERS_VK] if len(MOVERS_VK) else [MOVERS_MAIN, MOVERS_REAL], ignore_index=True)
_tmp = BASELINES[BASELINES["set"] != "RABL"].merge(_gt_b.reindex(columns=["uid", "regime", "motion_type", "x_par", "s", "gt_x_geo"]), on="uid")
print(_tmp.groupby(["set", "regime"])[["recall_M1_exact", "recall_M1_flow", "auroc_M1_exact"]].mean().round(3).to_string(na_rep="–"))
_p = _tmp[(_tmp["motion_type"] == "parallel") & (_tmp["s"] < 1)]
print(f"\nclassical triangulation, parallel movers: max |rho_exact - x_par| = "
      f"{np.nanmax(np.abs(_p['rho_classical_exact'] - _p['x_par'])):.2e} (exact correspondences); "
      f"median |rho_flow - x_par| = {np.nanmedian(np.abs(_p['rho_classical_flow'] - _p['x_par'])):.3f} ({FLOW_NAME} flow)")
_v = _tmp[_tmp["set"] == "VK"]
if len(_v):
    _vv = _v[np.isfinite(_v["gt_x_geo"])]
    print(f"Virtual KITTI 2: max |rho_exact - x_geo| = {np.nanmax(np.abs(_vv['rho_classical_exact'] - _vv['gt_x_geo'])):.2e} "
          f"over {len(_vv)} moving vehicles (exact correspondences; a check of the ground-truth pipeline)")

_ = mem_report("baselines")

In [ ]:
# Cell No: 046
# 5.1 Master evaluation loop: one checkpointed stage per (model, testbed). A model is loaded only
#     if at least one of its stages must be computed. Its load check runs on the loaded copy, each
#     finished (model, testbed) stage is archived at once (checkpoints/ and convoy_checkpoints.zip), and
#     the model is released, with GPU and host memory handed back, before the next model is loaded.
#     A model is loaded only if enough RAM is free (MEMORY_GUARD_GIB); otherwise it is deferred,
#     reported, and evaluated by the next session that attaches this run's archive.
EXEMPLAR_MAIN = set(range(min(10, len(META_MAIN))))
EXEMPLAR_REAL = set(range(min(3, len(META_REAL))))
EXEMPLAR_ABL = set(range(min(6, N_ABL)))
EXEMPLAR_RABL = set(range(min(6, len({m["index"] for m in META_RABL}))))
EXEMPLAR_VK = set(range(min(8, len(VK_PAIR_META))))
EXEMPLAR_KT = set(range(min(6, len(KT_PAIR_META))))
EXEMPLAR_AV2 = set(range(min(6, len(AV2_PAIR_META))))
EXEMPLAR_BSW = set(range(min(4, N_BSW)))
EXEMPLAR_WDR = set(range(min(4, N_WDR)))
EXEMPLAR_BONN = set(range(min(6, len(BONN_PAIR_META))))

def _mono_dup_rows(m, rows, img0, gt, sm, masks):
    # v6.2 single-image protocol check (multi-view models): the image given twice instead of once
    if m.family != "multiview":
        return None
    od = m.multi([img0, img0])
    gd = scale_and_gate(od["depth"], gt, sm)
    for r, mk in zip(rows, masks):
        r["rho_mono_dup"] = mover_rho(od["depth"], gt, mk, gd["scale"])
    return gd

def _down2(a):
    a = np.asarray(a, np.float32)
    return cv2.resize(np.nan_to_num(a, nan=0.0, posinf=0.0), (a.shape[1] // 2, a.shape[0] // 2),
                      interpolation=cv2.INTER_AREA).astype(np.float16)

def _score_block(o, om, sc, pB, vis, p_flow, sm, masks, scale, mono_scale, seed):
    # per-mover recall / AUROC of the model-based masks M2 (exact and flow), M4 and M5
    dsc = o["depth"].astype(np.float64) * scale
    m2e = rigid_flow_residual_map(sc, dsc, pB)
    # M5 (exploratory remedy): disagreement between the two-view and the single-image depth of
    # the same model, |log(two-view) - log(single image)| after each is scale-aligned
    with np.errstate(divide="ignore", invalid="ignore"):
        m5 = np.abs(np.log(np.clip(dsc, 1e-6, None)) - np.log(np.clip(om["depth"].astype(np.float64) * mono_scale, 1e-6, None)))
    res = {"M2_exact": mask_scores(m2e, sm & vis, masks, seed), "M4": mask_scores(-o["conf"].astype(np.float64), sm, masks, seed),
           "M5": mask_scores(m5, sm, masks, seed)}
    if p_flow is not None:
        res["M2_flow"] = mask_scores(rigid_flow_residual_map(sc, dsc, p_flow), sm, masks, seed)
    return res, m2e, m5

def _mover_rows(set_, idx, masks, o, om, gt, sm, g, gm, scores, extra=None, ko=None, gk=None):
    rows = []
    for j, mk in enumerate(masks):
        row = dict(set=set_, scene=idx, mover=j, uid=f"{set_}-{idx:04d}-{j}", n_px=int(mk.sum()),
                   rho=mover_rho(o["depth"], gt, mk, g["scale"]), rho_mono=mover_rho(om["depth"], gt, mk, gm["scale"]),
                   rho_ko=mover_rho(ko["depth"], gt, mk, gk["scale"]) if ko is not None else np.nan,
                   conf_mover=float(np.median(o["conf"][mk])) if mk.any() else np.nan,
                   conf_static=float(np.median(o["conf"][sm])) if sm.any() else np.nan, **(extra or {}))
        for meth, lst in scores.items():
            row[f"recall_{meth}"] = lst[j]["recall"]; row[f"auroc_{meth}"] = lst[j]["auroc"]
        rows.append(row)
    return rows

def _img_row(set_, idx, g, gm, **kw):
    return dict(set=set_, scene=idx, **g, mono_scale=gm["scale"], mono_slope=gm["slope"], mono_absrel=gm["absrel"], **kw)

def _flat_gate(o, anchors):
    zp, zt = [], []
    for a in anchors:
        u, v = int(round(a["u"])), int(round(a["v"]))
        zp.append(float(np.median(o["depth"][max(v - 2, 0):v + 3, max(u - 2, 0):u + 3])))
        zt.append(a["Z"])
    zp, zt = np.asarray(zp), np.asarray(zt)
    ok = np.isfinite(zp) & (zp > 0)
    if ok.sum() < 4:
        return dict(scale=np.nan, slope=np.nan, absrel=np.nan, n_static=int(ok.sum()))
    s = float(np.median(zt[ok] / zp[ok]))
    return dict(scale=s, slope=float(np.polyfit(np.log(zt[ok]), np.log(zp[ok]), 1)[0]),
                absrel=float(np.mean(np.abs(s * zp[ok] - zt[ok]) / zt[ok])), n_static=int(ok.sum()))

# ---------------------------------------------------------------- one function per testbed
def _eval_main(m, name):
    img, mov, ex = [], [], {}
    for meta in META_MAIN:
        sc, d, idx = meta["scene"], meta["dir"], meta["index"]
        ims = [load_rgb(d, k) for k in range(2)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, pB, vis = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "synth"); gt = r0["depth"]
        o, om = m.pair(ims[0], ims[1]), m.mono(ims[0])
        has_ko = os.path.exists(os.path.join(d, "frame1_ko.png"))
        ko = m.pair(ims[0], load_rgb(d, 1, "_ko")) if has_ko else None
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        gk = scale_and_gate(ko["depth"], gt, sm) if has_ko else None
        img.append(_img_row("MAIN", idx, g, gm))
        scores, m2e, m5 = _score_block(o, om, sc, pB, vis, flow_to_pobs(load_flow(meta)), sm, masks, g["scale"], gm["scale"],
                                       seed_for(name, "MAIN", idx))
        mov += _mover_rows("MAIN", idx, masks, o, om, gt, sm, g, gm, scores, ko=ko, gk=gk)
        if idx in EXEMPLAR_MAIN:
            ex[f"MAIN-{idx}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]),
                                     m2=_down2(m2e), m5=_down2(m5), conf=_down2(o["conf"]),
                                     **({"ko": _down2(ko["depth"] * gk["scale"])} if has_ko else {}))
    return dict(images=img, movers=mov), ex

def _eval_flat(m, name):
    img = []
    for meta in META_FLAT:
        o = m.pair(load_rgb(meta["dir"], 0), load_rgb(meta["dir"], 1))
        img.append(dict(set="FLAT", scene=meta["index"], **_flat_gate(o, meta["anchors"])))
    return dict(images=img), {}

def _eval_real(m, name):
    img, mov, ex = [], [], {}
    for meta in META_REAL:
        sc, d, idx = meta["scene"], meta["dir"], meta["index"]
        ims = [load_rgb(d, k) for k in range(2)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, pB, vis = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "real"); gt = r0["depth"]
        o, om = m.pair(ims[0], ims[1]), m.mono(ims[0])
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row("REAL", idx, g, gm, seq=meta["seq"]))
        scores, m2e, m5 = _score_block(o, om, sc, pB, vis, flow_to_pobs(load_flow(meta)), sm, masks, g["scale"], gm["scale"],
                                       seed_for(name, "REAL", idx))
        rows = _mover_rows("REAL", idx, masks, o, om, gt, sm, g, gm, scores)
        gd = _mono_dup_rows(m, rows, ims[0], gt, sm, masks)
        if gd is not None:
            img[-1].update(mono_dup_slope=gd["slope"], mono_dup_absrel=gd["absrel"])
        mov += rows
        if idx in EXEMPLAR_REAL:
            ex[f"REAL-{idx}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]),
                                     m2=_down2(m2e), m5=_down2(m5), conf=_down2(o["conf"]))
    return dict(images=img, movers=mov), ex

def _eval_abl(m, name):
    # E5: every condition of every base scene; rows keyed by (uid, condition)
    img, mov, ex = [], [], {}
    for meta in META_ABL:
        sc, d, idx, cond = meta["scene"], meta["dir"], meta["index"], meta["condition"]
        ims = [load_rgb(d, k) for k in range(2)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, pB, vis = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "synth"); gt = r0["depth"]
        o, om = m.pair(ims[0], ims[1]), m.mono(ims[0])
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row("ABL", idx, g, gm, condition=cond))
        scores, m2e, m5 = _score_block(o, om, sc, pB, vis, None, sm, masks, g["scale"], gm["scale"], seed_for(name, "ABL", idx, cond))
        mov += _mover_rows("ABL", idx, masks, o, om, gt, sm, g, gm, scores, extra=dict(condition=cond))
        if idx in EXEMPLAR_ABL:
            ex[f"ABL-{idx}-{cond}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]))
    return dict(images=img, movers=mov), ex

def _eval_e4(m, name):
    rows = []
    for meta in META_E4:
        sc, d, idx = meta["scene"], meta["dir"], meta["index"]
        K = len(sc["cams"])
        ims = [load_rgb(d, k) for k in range(K)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, _, _ = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "synth")
        for Kw in RUN_CONFIG["e4_window_sizes"]:
            o = m.multi(ims[:Kw]); g = scale_and_gate(o["depth"], r0["depth"], sm)
            for j, mk in enumerate(masks):
                rows.append(dict(path=sc["camera"]["path"], scene=idx, mover=j, uid=f"E4-{idx:04d}-{j}", K=Kw,
                                 rho=mover_rho(o["depth"], r0["depth"], mk, g["scale"]), slope=g["slope"], absrel=g["absrel"]))
    return dict(e4=rows), {}

def _eval_vk(m, name):
    img, mov, ex = [], [], {}
    for meta in VK_PAIR_META:
        vsc = VK_SCENE_OBJS[meta["seq"]]; i, j = meta["frames"]; idx = meta["index"]
        gtp = vsc.pair_gt(i, j)
        A, B = vsc.frame_gt(i), vsc.frame_gt(j)
        masks = [erode_mask(gtp["mover_id"] == k, PREREG["erosion_px"]) & gtp["visible"] for k in range(len(gtp["movers"]))]
        sm = vk_static_mask(A["depth"], A["inst"]); gt = A["depth"]
        o, om = m.pair(A["rgb"], B["rgb"]), m.mono(A["rgb"])
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row("VK", idx, g, gm, seq=meta["seq"]))
        scores, m2e, m5 = _score_block(o, om, gtp["scene"], gtp["pB"], gtp["visible"], None, sm, masks, g["scale"], gm["scale"],
                                       seed_for(name, "VK", idx))
        mov += _mover_rows("VK", idx, masks, o, om, gt, sm, g, gm, scores)
        if idx in EXEMPLAR_VK:
            ex[f"VK-{idx}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]),
                                   m2=_down2(m2e), m5=_down2(m5))
    return dict(images=img, movers=mov), ex

def _eval_lidar(m, set_, metas, exemplars):
    # real driving with LiDAR (KITTI tracking, Argoverse 2; exploratory): sparse ground truth from the
    # nearest LiDAR return per pixel; no dense masks, so no mask scores
    img, mov, ex = [], [], {}
    for meta in metas:
        d, idx = meta["dir"], meta["index"]
        A = np.asarray(Image.open(os.path.join(d, "rgb0.png")).convert("RGB"))
        B = np.asarray(Image.open(os.path.join(d, "rgb1.png")).convert("RGB"))
        z = np.load(os.path.join(d, "gt.npz")); gt, sm = z["depth"], z["static"]
        masks = []
        for v in (meta["objects"] if "objects" in meta else meta["vehicles"]):      # Bonn (v6.5) stores objects
            mk = np.zeros(gt.shape, bool)
            px = np.asarray(v["pixels"], int).reshape(-1, 2)
            if len(px):
                mk[px[:, 1], px[:, 0]] = True
            masks.append(mk & np.isfinite(gt))
        o, om = m.pair(A, B), m.mono(A)
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row(set_, idx, g, gm, seq=meta["seq"]))
        mov += _mover_rows(set_, idx, masks, o, om, gt, sm, g, gm, {})
        if idx in exemplars:
            ex[f"{set_}-{idx}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]))
    return dict(images=img, movers=mov), ex

def _eval_kitti(m, name):
    return _eval_lidar(m, "KT", KT_PAIR_META, EXEMPLAR_KT)

def _eval_av2(m, name):
    return _eval_lidar(m, "AV2", AV2_PAIR_META, EXEMPLAR_AV2)

def _eval_av2x(m, name):
    # v6.4: the further Argoverse 2 logs, under the same set label (their pair indices continue those of AV2)
    return _eval_lidar(m, "AV2", AV2X_PAIR_META, set())

def _eval_designed(m, name, set_, metas, exemplars, mono_shared):
    # v6.5 designed experiments (D55): every level of every base scene; rows keyed by (uid, condition). In the baseline
    # sweep the first image is the same at every level, so its single-image prediction is computed once per base scene.
    img, mov, ex, mono_cache = [], [], {}, {}
    for meta in metas:
        sc, d, idx, cond = meta["scene"], meta["dir"], meta["index"], meta["condition"]
        ims = [load_rgb(d, k) for k in range(2)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, pB, vis = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "synth"); gt = r0["depth"]
        o = m.pair(ims[0], ims[1])
        if mono_shared:
            if idx not in mono_cache:
                mono_cache.clear(); mono_cache[idx] = m.mono(ims[0])
            om = mono_cache[idx]
        else:
            om = m.mono(ims[0])
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row(set_, idx, g, gm, condition=cond))
        scores, m2e, m5 = _score_block(o, om, sc, pB, vis, None, sm, masks, g["scale"], gm["scale"], seed_for(name, set_, idx, cond))
        mov += _mover_rows(set_, idx, masks, o, om, gt, sm, g, gm, scores, extra=dict(condition=cond))
        if idx in exemplars:
            ex[f"{set_}-{idx}-{cond}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]),
                                              m2=_down2(np.clip(m2e, 0.0, 6.0e4)))        # within the float16 range
    return dict(images=img, movers=mov), ex

def _eval_bsw(m, name):
    return _eval_designed(m, name, "BSW", META_BSW, EXEMPLAR_BSW, mono_shared=True)

def _eval_wdr(m, name):
    return _eval_designed(m, name, "WDR", META_WDR, EXEMPLAR_WDR, mono_shared=False)

def _eval_bonn(m, name):
    # v6.5 Bonn RGB-D Dynamic (D56): measured depth, objects from moving regions (Section 3.12); no dense masks
    return _eval_lidar(m, "BONN", BONN_PAIR_META, EXEMPLAR_BONN)

def _eval_rabl(m, name):
    # E5-REAL: both conditions of every real-background base pair; rows keyed by (uid, condition)
    img, mov, ex = [], [], {}
    for meta in META_RABL:
        sc, d, idx, cond = meta["scene"], meta["dir"], meta["index"], meta["condition"]
        ims = [load_rgb(d, k) for k in range(2)]
        r0, r1 = load_frame(d, 0), load_frame(d, 1)
        masks, pB, vis = mover_masks(sc, r0, r1)
        sm = static_mask(r0, "real"); gt = r0["depth"]
        o, om = m.pair(ims[0], ims[1]), m.mono(ims[0])
        g, gm = scale_and_gate(o["depth"], gt, sm), scale_and_gate(om["depth"], gt, sm)
        img.append(_img_row("RABL", idx, g, gm, condition=cond, seq=meta["seq"]))
        scores, m2e, m5 = _score_block(o, om, sc, pB, vis, None, sm, masks, g["scale"], gm["scale"], seed_for(name, "RABL", idx, cond))
        rows = _mover_rows("RABL", idx, masks, o, om, gt, sm, g, gm, scores, extra=dict(condition=cond))
        gd = _mono_dup_rows(m, rows, ims[0], gt, sm, masks)
        if gd is not None:
            img[-1].update(mono_dup_slope=gd["slope"], mono_dup_absrel=gd["absrel"])
        mov += rows
        if idx in EXEMPLAR_RABL:
            ex[f"RABL-{idx}-{cond}"] = dict(depth=_down2(o["depth"] * g["scale"]), mono=_down2(om["depth"] * gm["scale"]))
    return dict(images=img, movers=mov), ex

EVAL_FUNCTIONS = {"MAIN": _eval_main, "FLAT": _eval_flat, "REAL": _eval_real, "RABL": _eval_rabl, "ABL": _eval_abl,
                  "E4": _eval_e4, "VK": _eval_vk, "KT": _eval_kitti, "AV2": _eval_av2, "AV2X": _eval_av2x,
                  "BSW": _eval_bsw, "WDR": _eval_wdr, "BONN": _eval_bonn}

def _write_stage(st, tables, exemplars, seconds, status_record):
    shutil.rmtree(st.out_dir, ignore_errors=True); os.makedirs(st.out_dir)
    for k, rows in tables.items():
        pd.DataFrame(rows).to_csv(os.path.join(st.out_dir, f"{k}.csv"), index=False)
    np.savez_compressed(os.path.join(st.out_dir, "exemplars.npz"),
                        **{f"{k}|{kk}": vv for k, v in exemplars.items() for kk, vv in v.items()})
    with open(os.path.join(st.out_dir, "done.json"), "w") as f:
        json.dump(dict(stage=st.name, seconds=round(seconds, 1),
                       finished=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())), f, indent=1)
    st.commit([st.out_dir], extra=dict(seconds=round(seconds, 1), status_record=status_record))

def evaluate_model(name, cls):
    stages, rec = EVAL_STAGES[name], MODEL_STATUS[name]
    todo = []
    for tb, st in stages.items():
        if not st.done():                    # prints RESTORED (inference skipped) or RUNNING for each testbed
            todo.append(tb)
    if not todo:
        st0 = stages["MAIN"]
        prev = (_read_json(st0.reg_path) or {}).get("extra", {}).get("status_record") or {}
        rec.update(status="PASS", params_M=prev.get("params_M"), load_s=prev.get("load_s"), peak_gpu_GiB=prev.get("peak_gpu_GiB"),
                   check_s=0.0, note=f"all {len(stages)} evaluation stages restored; not loaded")
        return
    release_memory()
    r = mem_report(f"before loading {name}")
    if r["available"] < MEMORY_GUARD_GIB:
        rec.update(status="DEFERRED", note=f"not loaded: {r['available']:.1f} GiB RAM free (< {MEMORY_GUARD_GIB} GiB); "
                                           f"{len(todo)} stage(s) left for the next session")
        print(f"[{name}] DEFERRED: {rec['note']}")
        return
    t0 = time.time()
    m = cls()
    try:
        m.load(); m.to_gpu()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        load_check(m)
        rec.update(status="PASS", params_M=round(m.n_params / 1e6, 1), load_s=round(m.load_seconds, 1),
                   check_s=round(time.time() - t0, 1),
                   note="loaded and ran" + (f"; {len(stages) - len(todo)} of {len(stages)} stages restored" if len(todo) < len(stages) else ""))
        print(f"[{name}] PASS: loaded ({rec['params_M']} M parameters, {rec['load_s']} s) and passed the load check")
        for tb in todo:
            t1 = time.time()
            tables, ex = EVAL_FUNCTIONS[tb](m, name)
            rec["peak_gpu_GiB"] = round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0
            _write_stage(stages[tb], tables, ex, time.time() - t1, dict(rec))
            del tables, ex
            if tb == "VK":
                for _sc in VK_SCENE_OBJS.values():
                    _sc._cache.clear()
            release_memory()
    except Exception as e:
        rec.update(status="FAIL", note=f"{type(e).__name__}: {str(e)[:200]}")
        print(f"[{name}] FAIL: {rec['note']}")
    finally:
        m.unload(); del m
        release_memory()
        mem_report(f"after releasing {name}")

def _free_checkpoint_if_disk_low(cls, min_free_gib=15.0):
    # once a model's results are checkpointed its weights are no longer needed; free them only when
    # the disk is getting full (a re-run restores the results, not the weights)
    free = shutil.disk_usage(os.path.expanduser("~")).free / 2**30
    if free >= min_free_gib:
        return
    hub = os.environ.get("HF_HUB_CACHE") or os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
    d = os.path.join(hub, "models--" + cls.checkpoint.replace("/", "--"))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f"[disk] freed the downloaded weights of {cls.checkpoint} ({free:.1f} GiB free before)")

with stage("master_evaluation"):
    for _name, _cls in MODELS.items():
        evaluate_model(_name, _cls)          # failures are reported and the model is excluded from every analysis;
        _free_checkpoint_if_disk_low(_cls, min_free_gib=1e9)   # weights are not needed once the results are archived
EVALUATED_MODELS = [n for n in MODELS if MODEL_STATUS[n]["status"] == "PASS"
                    and all(os.path.exists(os.path.join(st.out_dir, "done.json")) for st in EVAL_STAGES[n].values())]
MODEL_TABLE = pd.DataFrame(MODEL_STATUS.values())
print("\n" + MODEL_TABLE.to_string(index=False, na_rep="–"))
print("\n" + "=" * 88)
_n_restored = sum("not loaded" in r["note"] for r in MODEL_STATUS.values())
print(f"MODEL SUMMARY: {len(EVALUATED_MODELS)}/{len(MODEL_STATUS)} models evaluated: "
      f"{len(EVALUATED_MODELS) - _n_restored} ran on the GPU in this session, {_n_restored} restored from checkpoints")
print("=" * 88)
for k, r_ in MODEL_STATUS.items():
    print(f"  [{'PASS' if k in EVALUATED_MODELS else r_['status']}] {k:<12} testbeds: {', '.join(EVAL_STAGES[k])}"
          + ("" if k in EVALUATED_MODELS else f"  -> left out of every experiment: {r_['note']}"))
if any(r_["status"] == "DEFERRED" for r_ in MODEL_STATUS.values()):
    print("Deferred models: attach convoy_checkpoints.zip (or this notebook's output) to a new session and run all cells; "
          "every finished stage is restored and only the deferred models are evaluated.")
MODEL_TABLE.to_csv(os.path.join(LOG_DIR, "model_status.csv"), index=False)
print(f"\nEvaluated models ({len(EVALUATED_MODELS)}): {', '.join(EVALUATED_MODELS)}")
_et = stage_table("eval:")
if len(_et):
    print(_et.groupby("status").size().to_string())
    print(f"checkpoints/: {len(glob.glob(os.path.join(CKPT_DIR, '*.zip')))} stage archives, "
          f"{sum(os.path.getsize(p) for p in glob.glob(os.path.join(CKPT_DIR, '*.zip'))) / 2**20:.0f} MiB")
# every model result is in convoy_checkpoints.zip now: offer it (and copy it to the private dataset,
# if configured), so that a session lost during the analysis still leaves a complete resume point
print("Dataset copy:", backup_to_dataset([ZIP_PATH], "after the master evaluation"))
offer_downloads([ZIP_PATH])

In [ ]:
# Cell No: 047
# 5.2 Collect the per-stage results of every evaluated model into analysis tables. No model is in
#     memory from here on; memory is released and reported before the analysis starts.
release_memory(); _ = mem_report("analysis start")

def _read(path):
    return pd.read_csv(path) if os.path.exists(path) and os.path.getsize(path) > 5 else pd.DataFrame()

_imgs, _movs, _e4s = [], [], []
EXEMPLARS, RUN_TIMING = {}, {}
for _n in EVALUATED_MODELS:
    EXEMPLARS[_n], RUN_TIMING[_n] = {}, {}
    for _tb, _st in EVAL_STAGES[_n].items():
        for _lst, _f in ((_imgs, "images.csv"), (_movs, "movers.csv"), (_e4s, "e4.csv")):
            _df = _read(os.path.join(_st.out_dir, _f))
            if len(_df):
                _df["model"] = _n
                _lst.append(_df)
        _z = np.load(os.path.join(_st.out_dir, "exemplars.npz"))
        EXEMPLARS[_n].update({k: _z[k].astype(np.float32) for k in _z.files})
        with open(os.path.join(_st.out_dir, "done.json")) as _f:
            RUN_TIMING[_n][_tb] = json.load(_f)["seconds"]

def _cols(df, cols):
    return df.reindex(columns=cols)

IMG_RES = pd.concat(_imgs, ignore_index=True) if _imgs else pd.DataFrame()
_gtcols = ["uid", "cls", "style", "s", "x_par", "eps_target", "eps_center", "beyond", "motion_type", "Z_center",
           "gt_Z_true", "gt_eps_med", "gt_eps_max", "gt_x_geo", "gt_parallax_med", "gt_area_px", "gt_visible_frac",
           "gt_n_eroded", "regime", "measurable", "identifiable", "cam_kind", "focal", "baseline"]
_extra = ["testbed", "seq", "speed_ratio", "gt_occupancy", "gt_truncation", "gt_vis_in_b", "track", "label",
          "gt_contact", "gt_wheels_visible", "gt_wheels_in_view", "gt_n_lidar_px", "gt_occlusion", "state", "gt_dt_ms",
          "kind", "floor", "gt_height_above_floor", "gt_x_geo_raw"]
MOVERS_ALL = pd.concat([_cols(MOVERS_MAIN.assign(testbed="CONVOY-3D"), _gtcols + _extra),
                        _cols(MOVERS_REAL.assign(testbed="CONVOY-REAL"), _gtcols + _extra) if len(MOVERS_REAL) else None,
                        _cols(MOVERS_VK.assign(testbed="Virtual KITTI 2"), _gtcols + _extra) if len(MOVERS_VK) else None,
                        _cols(MOVERS_KT.assign(testbed="KITTI tracking"), _gtcols + _extra) if len(MOVERS_KT) else None,
                        _cols(MOVERS_AV2.assign(testbed="Argoverse 2"), _gtcols + _extra) if len(MOVERS_AV2) else None,
                        _cols(MOVERS_BONN.assign(testbed="Bonn RGB-D"), _gtcols + _extra) if len(MOVERS_BONN) else None],
                       ignore_index=True)
_mr = pd.concat(_movs, ignore_index=True) if _movs else pd.DataFrame(columns=["set", "uid", "condition"])
MOV_RES = (_mr[~_mr["set"].isin(["ABL", "RABL", "BSW", "WDR"])].drop(columns=["condition"], errors="ignore")
           .merge(MOVERS_ALL, on="uid", how="left"))
# E5 ablation: one row per (mover, condition, model), joined with the ground truth of that condition
_abl_cols = _gtcols + ["condition", "contact_px0", "contact_px1", "has_wall", "lift", "support"]
ABL_RES = _mr[_mr["set"] == "ABL"].merge(_cols(MOVERS_ABL, _abl_cols), on=["uid", "condition"], how="left")
# E5-REAL: one row per (mover, condition, model), joined with the ground truth of that condition
RABL_RES = (_mr[_mr["set"] == "RABL"].merge(_cols(MOVERS_RABL, _abl_cols + ["seq"]), on=["uid", "condition"], how="left")
            if len(MOVERS_RABL) else pd.DataFrame())
# v6.5 designed experiments: one row per (mover, level, model), joined with the ground truth of that level
_dsg_cols = _abl_cols + ["factor", "cover_m"]
BSW_RES = _mr[_mr["set"] == "BSW"].merge(_cols(MOVERS_BSW, _dsg_cols), on=["uid", "condition"], how="left")
WDR_RES = _mr[_mr["set"] == "WDR"].merge(_cols(MOVERS_WDR, _dsg_cols), on=["uid", "condition"], how="left")
E4_RES = (pd.concat(_e4s, ignore_index=True).merge(MOVERS_E4[["uid", "cls", "s", "x_par", "measurable", "identifiable",
                                                               "gt_parallax_med", "gtlast_eps_med", "omega_deg"]], on="uid", how="left")
          if _e4s else pd.DataFrame())
BASE_RES = BASELINES.merge(MOVERS_ALL, on="uid", how="left")
MODEL_TRAINING = {n: MODEL_STATUS[n]["training"] for n in EVALUATED_MODELS}
MODEL_FAMILY = {n: MODEL_STATUS[n]["family"] for n in EVALUATED_MODELS}
print(f"image-level rows: {len(IMG_RES)}  mover-level rows: {len(MOV_RES)} "
      f"({MOV_RES['set'].value_counts().to_dict() if len(MOV_RES) else {}})")
print(f"ablation rows: {len(ABL_RES)}  real-background ablation rows: {len(RABL_RES)}  E4 rows: {len(E4_RES)}  "
      f"baseline rows: {len(BASE_RES)}")
print(f"designed experiments (v6.5): baseline sweep rows {len(BSW_RES)}, dose-response rows {len(WDR_RES)}; "
      f"Bonn object rows {int((MOV_RES['set'] == 'BONN').sum()) if len(MOV_RES) else 0}")
print("inference time per model and testbed (min):")
print(pd.DataFrame(RUN_TIMING).T.div(60).round(1).to_string(na_rep="–"))

In [ ]:
# Cell No: 048
# 6.A Statistics: cluster bootstrap (vectorised for linear statistics), OLS slope with R^2,
#     Holm-Bonferroni, three-level verdicts.
from scipy.stats import spearmanr

class ClusterBootstrap:
    def __init__(self, cluster_ids, n_boot=None, seed=0):
        n_boot = n_boot or RUN_CONFIG["n_boot"]
        self.ids, inv = np.unique(np.asarray(cluster_ids), return_inverse=True)
        self.C, self.inv, self.B = len(self.ids), inv, n_boot
        rng = np.random.default_rng(seed)
        self.draws = rng.integers(0, self.C, size=(n_boot, self.C))
        self.counts = np.zeros((n_boot, self.C), np.float64)
        np.add.at(self.counts, (np.repeat(np.arange(n_boot), self.C), self.draws.ravel()), 1.0)
        self.members = [np.nonzero(inv == c)[0] for c in range(self.C)]

    def sums(self, cols):
        # per-cluster sums of each column -> (C, k)
        S = np.zeros((self.C, len(cols)))
        for k, c in enumerate(cols):
            np.add.at(S[:, k], self.inv, c)
        return S

    def linear(self, cols):
        # bootstrap distribution of the column totals -> (B, k), and the point totals (k,)
        S = self.sums(cols)
        return self.counts @ S, S.sum(0)

    def resample_indices(self, b):
        return np.concatenate([self.members[c] for c in self.draws[b]])

def ols_fit(x, y):
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    A = np.vstack([x, np.ones_like(x)]).T
    (slope, icpt), *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - (slope * x + icpt)
    r2 = 1 - resid.var() / y.var() if y.var() > 0 else np.nan
    return float(slope), float(icpt), float(r2)

def slope_with_ci(x, y, clusters, seed, h0=None):
    x, y, clusters = np.asarray(x, float), np.asarray(y, float), np.asarray(clusters)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, clusters = x[ok], y[ok], clusters[ok]
    out = dict(n=int(len(x)), n_clusters=int(len(np.unique(clusters))))
    if len(x) < 8 or len(np.unique(clusters)) < 5 or np.var(x) == 0:
        return {**out, "beta": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "r2": np.nan, "intercept": np.nan, "p_h0": np.nan}
    beta, icpt, r2 = ols_fit(x, y)
    bs = ClusterBootstrap(clusters, seed=seed)
    T, _ = bs.linear([np.ones_like(x), x, y, x * x, x * y])
    with np.errstate(invalid="ignore", divide="ignore"):
        bb = (T[:, 0] * T[:, 4] - T[:, 1] * T[:, 2]) / (T[:, 0] * T[:, 3] - T[:, 1] ** 2)
    bb = bb[np.isfinite(bb)]
    lo, hi = np.percentile(bb, [2.5, 97.5])
    p = float((1 + np.sum(bb <= h0)) / (len(bb) + 1)) if h0 is not None else np.nan
    return {**out, "beta": beta, "ci_lo": float(lo), "ci_hi": float(hi), "r2": r2, "intercept": icpt, "p_h0": p}

def boot_stat(df, cluster_col, fn, seed, n_boot=None, cols=None):
    # generic (non-linear) statistic: point estimate and percentile CI. fn receives a dict of
    # NumPy columns (fast); cols lists the columns it needs.
    cols = cols or [c for c in df.columns if c != cluster_col]
    arrs = {c: df[c].to_numpy() for c in cols}
    bs = ClusterBootstrap(df[cluster_col].values, n_boot=n_boot, seed=seed)
    point = fn(arrs)
    vals = []
    for b in range(bs.B):
        idx = bs.resample_indices(b)
        v = fn({c: a[idx] for c, a in arrs.items()})
        if np.isfinite(v):
            vals.append(v)
    if len(vals) < 50:
        return point, np.nan, np.nan, np.array(vals)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, float(lo), float(hi), np.asarray(vals)

def holm(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.minimum(1.0, np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]))
    tmp = np.empty(m); tmp[idx] = adj
    out[ok] = tmp
    return out

def verdict_p1(beta, lo, hi, r2):
    if not np.isfinite(beta):
        return "NOT TESTABLE"
    a, b = PREREG["beta_ci_band"]
    if hi < a or lo > b or r2 < PREREG["r2_min"]:
        return "FALSIFIED"
    if PREREG["beta_support_range"][0] <= beta <= PREREG["beta_support_range"][1] and lo >= a and hi <= b:
        return "SUPPORTED"
    return "PARTIAL"

def e1_mask(d):
    # E1 movers: purely parallel (eps = 0 at every pixel), s < 1, measurable, identifiable
    return ((d["motion_type"] == "parallel") & (d["s"] < 1) & d["measurable"].astype(bool)
            & d["identifiable"].astype(bool))

def h9_mask(d):
    # H9 movers (Virtual KITTI 2): parallel regime, measurable, identifiable, x_geo in the design range
    lo, hi = PREREG["h9_x_range"]
    x = pd.to_numeric(d["gt_x_geo"], errors="coerce")
    return ((d["regime"] == "parallel") & d["measurable"].astype(bool) & d["identifiable"].astype(bool)
            & np.isfinite(x) & (x >= lo) & (x <= hi))

def prior_weight(df):
    # Zhat = w Zhat_mono + (1 - w) Z_geo  =>  w = (e^rho - e^x_geo) / (e^rho_mono - e^x_geo),
    # defined only where the two hypotheses are separable (|x_geo - rho_mono| > 0.10)
    x, r, rm = df["gt_x_geo"].values, df["rho"].values, df["rho_mono"].values
    sep = np.abs(x - rm) > PREREG["w_separability_min"]
    with np.errstate(invalid="ignore", over="ignore", divide="ignore"):
        w = (np.exp(r) - np.exp(x)) / (np.exp(rm) - np.exp(x))
    return np.where(sep & np.isfinite(w), w, np.nan)

def slope_boot(x, y, clusters, seed):
    # OLS slope and its cluster-bootstrap draws
    x, y, clusters = np.asarray(x, float), np.asarray(y, float), np.asarray(clusters)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, clusters = x[ok], y[ok], clusters[ok]
    if len(x) < 8 or len(np.unique(clusters)) < 5:
        return np.nan, np.array([])
    beta = ols_fit(x, y)[0]
    T, _ = ClusterBootstrap(clusters, seed=seed).linear([np.ones_like(x), x, y, x * x, x * y])
    with np.errstate(invalid="ignore", divide="ignore"):
        bb = (T[:, 0] * T[:, 4] - T[:, 1] * T[:, 2]) / (T[:, 0] * T[:, 3] - T[:, 1] ** 2)
    return beta, bb

def multi_slopes(x, ys, clusters, seed):
    # OLS slopes of several responses on the SAME regressor (paired design) with their joint
    # cluster-bootstrap draws: returns (slopes (k,), draws (B, k), n, n_clusters)
    x = np.asarray(x, float); Y = np.column_stack([np.asarray(y, float) for y in ys])
    cl = np.asarray(clusters)
    ok = np.isfinite(x) & np.isfinite(Y).all(1)
    x, Y, cl = x[ok], Y[ok], cl[ok]
    k, n, nc = Y.shape[1], len(x), len(np.unique(cl))
    if n < 8 or nc < 5 or np.var(x) == 0:
        return np.full(k, np.nan), np.zeros((0, k)), n, nc
    cols = [np.ones_like(x), x, x * x] + [Y[:, i] for i in range(k)] + [x * Y[:, i] for i in range(k)]
    T, P = ClusterBootstrap(cl, seed=seed).linear(cols)
    def _sl(t):
        den = t[:, 0] * t[:, 2] - t[:, 1] ** 2
        with np.errstate(invalid="ignore", divide="ignore"):
            return (t[:, :1] * t[:, 3 + k:3 + 2 * k] - t[:, 1:2] * t[:, 3:3 + k]) / den[:, None]
    return _sl(P[None])[0], _sl(T), n, nc

def contrast_ci(point, draws, weights):
    # point estimate and 95% percentile CI of a linear contrast of jointly bootstrapped statistics
    w = np.asarray(weights, float)
    est = float(np.dot(point, w))
    if len(draws) < 50 or not np.isfinite(est):
        return est, np.nan, np.nan
    v = draws @ w; v = v[np.isfinite(v)]
    if len(v) < 50:
        return est, np.nan, np.nan
    lo, hi = np.percentile(v, [2.5, 97.5])
    return est, float(lo), float(hi)

def paired_slopes(x, y_a, y_b, clusters, seed):
    # slopes of y_a and y_b on the same x, and the bootstrap distribution of their difference
    b, draws, n, nc = multi_slopes(x, [y_a, y_b], clusters, seed)
    return b[0], b[1], (draws[:, 0] - draws[:, 1]) if len(draws) else np.array([])

def _ci(bb):
    bb = np.asarray(bb)[np.isfinite(bb)] if len(bb) else np.array([])
    return tuple(np.percentile(bb, [2.5, 97.5])) if len(bb) > 50 else (np.nan, np.nan)

def verdict_positive(lo, hi):
    # registered v6 rule for a contrast predicted to be positive
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "NOT TESTABLE"
    return "SUPPORTED" if lo > 0 else ("CONTRADICTED" if hi < 0 else "NOT SUPPORTED")

BOOT_N = RUN_CONFIG["n_boot"]
print(f"Statistics ready (cluster bootstrap, {BOOT_N} resamples).")

In [ ]:
# Cell No: 049
# 6.0 Static-scene validity gate (pre-registered). For every model and testbed: median over
#     images of the log-depth slope on static pixels and of AbsRel after scale alignment.
#     PASS: slope in [0.7, 1.3] and AbsRel <= 0.25. LEGACY-FLAT is the negative control.
TESTBED_NAMES = {"MAIN": "CONVOY-3D", "ABL": "CONVOY-3D ablation (E5)", "REAL": "CONVOY-REAL",
                 "RABL": "CONVOY-REAL contact ablation (E5-REAL)", "VK": "Virtual KITTI 2", "KT": "KITTI tracking",
                 "AV2": "Argoverse 2", "BSW": "CONVOY-3D baseline sweep (H12)", "WDR": "CONVOY-3D dose-response (H13)",
                 "BONN": "Bonn RGB-D Dynamic", "FLAT": "LEGACY-FLAT (control)"}
_rows = []
for (model, st), grp in IMG_RES.groupby(["model", "set"]):
    sl, ar = grp["slope"].median(), grp["absrel"].median()
    ok = (PREREG["gate_slope_range"][0] <= sl <= PREREG["gate_slope_range"][1]) and (ar <= PREREG["gate_absrel_max"])
    _rows.append(dict(model=model, testbed=TESTBED_NAMES.get(st, st), set=st, n_images=len(grp),
                      slope_median=sl, slope_q25=grp["slope"].quantile(0.25), slope_q75=grp["slope"].quantile(0.75),
                      absrel_median=ar, absrel_q25=grp["absrel"].quantile(0.25), absrel_q75=grp["absrel"].quantile(0.75),
                      mono_slope_median=grp["mono_slope"].median() if "mono_slope" in grp else np.nan,
                      gate="PASS" if ok else "FAIL"))
GATE = pd.DataFrame(_rows)
GATE_PASS = {(r.model, r.set): r.gate == "PASS" for r in GATE.itertuples()}
_order = ["MAIN", "ABL", "REAL", "RABL", "VK", "KT", "AV2", "BSW", "WDR", "BONN", "FLAT"]
GATE["set"] = pd.Categorical(GATE["set"], _order, ordered=True)
GATE = GATE.sort_values(["set", "model"]).reset_index(drop=True)
print(GATE[["testbed", "model", "n_images", "slope_median", "absrel_median", "mono_slope_median", "gate"]]
      .to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
_flat = GATE[GATE["set"] == "FLAT"]
_main = GATE[GATE["set"] == "MAIN"]
print(f"\nCONVOY-3D gate passed by {int((_main['gate'] == 'PASS').sum())}/{len(_main)} models; "
      f"LEGACY-FLAT control rejected for {int((_flat['gate'] == 'FAIL').sum())}/{len(_flat)} models "
      f"(expected: rejected for all).")
GATE.to_csv(os.path.join(RESULTS_DIR, "gate.csv"), index=False)
def interpretable(model, testbed):
    # a model x testbed combination is interpreted only if it passes the gate (Section 6.0)
    return bool(GATE_PASS.get((model, testbed), False))

# models whose results on each testbed are interpreted (and drawn in the figures)
MODELS_OK = {tb: [m for m in EVALUATED_MODELS if interpretable(m, tb)]
             for tb in ["MAIN", "ABL", "REAL", "RABL", "VK", "KT", "AV2", "BSW", "WDR", "BONN"]}
for _tb, _ok in MODELS_OK.items():
    _no = [m for m in EVALUATED_MODELS if m not in _ok and (m, _tb) in GATE_PASS]
    if _no:
        print(f"Not interpreted on {TESTBED_NAMES[_tb]} (gate FAIL): {', '.join(_no)}")

In [ ]:
# Cell No: 050
# 6.0b Estimator validation with oracle predictors of known prior weight.
#      Before reading any model result, the E1 and E3 estimators are run on synthetic
#      predictors whose answer is known: on every mover pixel the oracle predicts
#      Zhat = w * Z_true + (1 - w) * Z_geo (w fixed), with 3% log-normal noise and an arbitrary
#      global scale per image; its "single-image" prediction is Z_true with independent noise.
#      Same movers, same scale alignment, same regressions as for the real models.
#      Expected: beta_s = beta(w) from the slope-weight identity (Section 2.4), median w = w.
#      The same check is run for the Virtual KITTI 2 estimator of H9. One checkpointed stage.
ORACLE_WS = [0.0, 0.25, 0.5, 0.75, 1.0]
ORACLE_STAGE = Stage("oracle", "eval", dict(ws=ORACLE_WS, measure=MEASURE_VERSION, prereg=_EVAL_PREREG, vk=len(VK_PAIR_META)),
                     upstream=[RENDER_FP["MAIN"], VK_GT_STAGE.fp if VK_STATUS["available"] else ""])
ORACLE_DIR = os.path.join(CACHE_DIR, "oracle")
with stage("oracle_validation"):
    if not ORACLE_STAGE.done():
        _orng = np.random.default_rng(seed_for("oracle"))
        _om_rows, _ov_rows = [], []
        for meta in META_MAIN:
            sc = meta["scene"]
            r0, r1 = load_frame(meta["dir"], 0), load_frame(meta["dir"], 1)
            masks, pB, vis = mover_masks(sc, r0, r1)
            sm = static_mask(r0, "synth")
            H, W = r0["depth"].shape
            vv, uu = np.mgrid[0:H, 0:W]
            _, zgeo, _ = pixel_epipolar_quantities(sc, 0, 1, np.stack([uu, vv], -1).astype(np.float64), pB)
            zt = np.where(np.isfinite(r0["depth"]), r0["depth"], 1e3).astype(np.float64)
            noise_p = np.exp(_orng.normal(0, 0.03, zt.shape)); noise_m = np.exp(_orng.normal(0, 0.03, zt.shape))
            scale_p, scale_m = np.exp(_orng.normal(0, 1)), np.exp(_orng.normal(0, 1))
            mono = zt * noise_m * scale_m
            g_m = scale_and_gate(mono, r0["depth"], sm)
            ok = (r0["mover_id"] >= 0) & np.isfinite(zgeo) & (zgeo > 0)
            for w in ORACLE_WS:
                pred = zt.copy()
                pred[ok] = w * zt[ok] + (1 - w) * zgeo[ok]
                pred = pred * noise_p * scale_p
                g = scale_and_gate(pred, r0["depth"], sm)
                for j, mk in enumerate(masks):
                    _om_rows.append(dict(w=w, uid=f"MAIN-{meta['index']:04d}-{j}", scene=meta["index"],
                                         rho=mover_rho(pred, r0["depth"], mk, g["scale"]),
                                         rho_mono=mover_rho(mono, r0["depth"], mk, g_m["scale"])))
        for meta in VK_PAIR_META:
            vsc = VK_SCENE_OBJS[meta["seq"]]; i, j = meta["frames"]
            g = vsc.pair_gt(i, j)
            zt = np.where(np.isfinite(g["depth"]), g["depth"], 1e3).astype(np.float64)
            sm = vk_static_mask(g["depth"], g["inst"])
            masks = [erode_mask(g["mover_id"] == k, PREREG["erosion_px"]) & g["visible"] for k in range(len(g["movers"]))]
            noise_p = np.exp(_orng.normal(0, 0.03, zt.shape)); scale_p = np.exp(_orng.normal(0, 1))
            ok = (g["mover_id"] >= 0) & np.isfinite(g["zgeo"]) & (g["zgeo"] > 0)
            for w in ORACLE_WS:
                pred = zt.copy()
                pred[ok] = w * zt[ok] + (1 - w) * g["zgeo"][ok]
                pred = pred * noise_p * scale_p
                gg = scale_and_gate(pred, g["depth"], sm)
                for k, mk in enumerate(masks):
                    _ov_rows.append(dict(w=w, uid=f"VK-{meta['index']:04d}-{k}", scene=meta["index"],
                                         rho=mover_rho(pred, g["depth"], mk, gg["scale"])))
        os.makedirs(ORACLE_DIR, exist_ok=True)
        pd.DataFrame(_om_rows).to_csv(os.path.join(ORACLE_DIR, "oracle_main.csv"), index=False)
        pd.DataFrame(_ov_rows, columns=["w", "uid", "scene", "rho"]).to_csv(os.path.join(ORACLE_DIR, "oracle_vkitti.csv"), index=False)
        ORACLE_STAGE.commit([ORACLE_DIR])
_om = pd.read_csv(os.path.join(ORACLE_DIR, "oracle_main.csv"))
_ov = _read(os.path.join(ORACLE_DIR, "oracle_vkitti.csv"))

_mv = MOVERS_MAIN[["uid", "x_par", "gt_x_geo", "motion_type", "s", "measurable", "identifiable", "regime", "gt_eps_med"]]
ORACLE_ROWS = []
for w in ORACLE_WS:
    d = _om[np.isclose(_om["w"], w)].merge(_mv, on="uid")
    dp = d[e1_mask(d)]
    r = slope_with_ci(dp["x_par"], dp["rho"], dp["scene"], seed_for("oracle-e1", w))
    beta_theory = float(np.polyfit(dp["x_par"], np.log(w + (1 - w) * np.exp(dp["x_par"])), 1)[0]) if len(dp) > 2 else np.nan
    de = d[d["measurable"].astype(bool) & (d["s"] < 1) & np.isfinite(d["gt_x_geo"])].copy()
    de["w"] = prior_weight(de)
    ORACLE_ROWS.append(dict(w_true=w, n=r["n"], beta_s=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                            beta_theory_identity=beta_theory, verdict_P1=verdict_p1(r["beta"], r["ci_lo"], r["ci_hi"], r["r2"]),
                            median_w_recovered=float(np.nanmedian(de["w"])) if len(de) else np.nan))
ORACLE_TABLE = pd.DataFrame(ORACLE_ROWS)
print(ORACLE_TABLE.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
_dev = np.nanmax(np.abs(ORACLE_TABLE["beta_s"] - ORACLE_TABLE["beta_theory_identity"]))
_wdev = np.nanmax(np.abs(ORACLE_TABLE["median_w_recovered"] - ORACLE_TABLE["w_true"]))
_v0 = ORACLE_TABLE.loc[ORACLE_TABLE["w_true"] == 0.0, "verdict_P1"].iloc[0]
_v1 = ORACLE_TABLE.loc[ORACLE_TABLE["w_true"] == 1.0, "verdict_P1"].iloc[0]
print(f"\nmax |beta_s(recovered) - beta(w)(identity)| = {_dev:.3f};  max |median w(recovered) - w| = {_wdev:.3f}")
print(f"P1 verdict for the pure-geometry oracle (w = 0): {_v0};  for the pure-prior oracle (w = 1): {_v1}")
ORACLE_OK = bool(_dev < 0.1 and _wdev < 0.1 and _v0 == "SUPPORTED" and _v1 == "FALSIFIED")
print("Estimator validation:", "PASS" if ORACLE_OK else "CHECK -- interpret model-level E1/E3 results with caution")
ORACLE_TABLE.to_csv(os.path.join(RESULTS_DIR, "oracle_validation.csv"), index=False)

# Virtual KITTI 2 estimator of H9 (regression of rho on x_geo over the H9 movers; real-looking images,
# no exact parallel motion)
ORACLE_VK = pd.DataFrame()
if VK_STATUS["available"] and len(MOVERS_VK) and len(_ov):
    _rows = []
    for w in ORACLE_WS:
        d = _ov[np.isclose(_ov["w"], w)].merge(MOVERS_VK[["uid", "gt_x_geo", "regime", "measurable", "identifiable"]], on="uid")
        dp = d[h9_mask(d)]
        r = slope_with_ci(dp["gt_x_geo"], dp["rho"], dp["scene"], seed_for("oracle-vk", w))
        bt = float(np.polyfit(dp["gt_x_geo"], np.log(w + (1 - w) * np.exp(dp["gt_x_geo"])), 1)[0]) if len(dp) > 2 else np.nan
        _rows.append(dict(w_true=w, n=r["n"], beta_geo=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], beta_theory_identity=bt))
    ORACLE_VK = pd.DataFrame(_rows)
    if ORACLE_VK["beta_geo"].notna().any():
        print("\nVirtual KITTI 2 (H9 estimator), planted predictors:")
        print(ORACLE_VK.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
        print(f"max |beta_geo(recovered) - beta(w)(identity)| = "
              f"{np.nanmax(np.abs(ORACLE_VK['beta_geo'] - ORACLE_VK['beta_theory_identity'])):.3f}")
        ORACLE_VK.to_csv(os.path.join(RESULTS_DIR, "oracle_validation_vkitti.csv"), index=False)
    else:
        print("\nVirtual KITTI 2: too few H9 movers for the oracle check (not reported).")
        ORACLE_VK = pd.DataFrame()

In [ ]:
# Cell No: 051
# 6.1 E1 / Prediction 1: is the depth of a parallel mover the static-equivalent depth?
#     beta_s = OLS slope of rho = log(Zhat/Z) on x = -log(1-s), purely parallel movers
#     (eps = 0 at every pixel), measurable and identifiable, CONVOY-3D.
E1_ROWS, E1_DATA, E1_PARALLAX = [], {}, []
_base = BASE_RES[(BASE_RES["set"] == "MAIN") & e1_mask(BASE_RES)]
CONTROL_E1 = {
    "Classical triangulation, exact correspondences": slope_with_ci(_base["x_par"], _base["rho_classical_exact"], _base["scene"], seed_for("e1", "cx")),
    f"Classical triangulation, {FLOW_NAME} flow": slope_with_ci(_base["x_par"], _base["rho_classical_flow"], _base["scene"], seed_for("e1", "cf")),
}
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN")]
    dp = d[e1_mask(d)]
    r = slope_with_ci(dp["x_par"], dp["rho"], dp["scene"], seed_for("e1", model), h0=0.5)
    rm = slope_with_ci(dp["x_par"], dp["rho_mono"], dp["scene"], seed_for("e1m", model))
    dg = d[(d["regime"] == "parallel") & (d["s"] < 1) & d["measurable"].astype(bool) & d["identifiable"].astype(bool)]
    rg = slope_with_ci(dg["gt_x_geo"], dg["rho"], dg["scene"], seed_for("e1g", model))
    gate_ok = GATE_PASS.get((model, "MAIN"), False)
    E1_ROWS.append(dict(model=model, training=MODEL_TRAINING[model], n=r["n"], n_clips=r["n_clusters"],
                        beta_s=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], intercept=r["intercept"], r2=r["r2"],
                        p_beta_le_half=r["p_h0"], beta_mono=rm["beta"], beta_mono_lo=rm["ci_lo"], beta_mono_hi=rm["ci_hi"],
                        beta_geo=rg["beta"], beta_geo_lo=rg["ci_lo"], beta_geo_hi=rg["ci_hi"], n_geo=rg["n"],
                        w_eff=beta_to_effective_w(r["beta"]) if np.isfinite(r["beta"]) else np.nan,
                        gate=("PASS" if gate_ok else "FAIL")))
    E1_DATA[model] = dp[["uid", "scene", "x_par", "rho", "rho_mono", "rho_ko", "gt_parallax_med", "cls", "cam_kind",
                         "gt_Z_true", "s"]].copy()
    for lo_, hi_ in [(0, 2), (2, 5), (5, 10), (10, 20), (20, 40), (40, 1e9)]:
        db = d[(d["motion_type"] == "parallel") & (d["s"] < 1) & d["measurable"].astype(bool)
               & (d["gt_parallax_med"] >= lo_) & (d["gt_parallax_med"] < hi_)]
        rb = slope_with_ci(db["x_par"], db["rho"], db["scene"], seed_for("e1p", model, lo_))
        E1_PARALLAX.append(dict(model=model, par_lo=lo_, par_hi=hi_, n=rb["n"], beta=rb["beta"], ci_lo=rb["ci_lo"], ci_hi=rb["ci_hi"]))
E1_TABLE = pd.DataFrame(E1_ROWS)
E1_TABLE["p_holm"] = holm(E1_TABLE["p_beta_le_half"].values)
E1_TABLE["verdict"] = [verdict_p1(b, l, h, r2) if g == "PASS" else "NOT INTERPRETABLE (gate)"
                       for b, l, h, r2, g in zip(E1_TABLE["beta_s"], E1_TABLE["ci_lo"], E1_TABLE["ci_hi"], E1_TABLE["r2"], E1_TABLE["gate"])]
E1_PARALLAX = pd.DataFrame(E1_PARALLAX)

print("Positive controls (same movers):")
for k, v in CONTROL_E1.items():
    print(f"   {k:<52} beta = {v['beta']:.3f} [{v['ci_lo']:.3f}, {v['ci_hi']:.3f}], R^2 = {v['r2']:.3f}, n = {v['n']}")
print("\nE1 (Prediction 1), CONVOY-3D, purely parallel movers:")
_cols = ["model", "training", "n", "n_clips", "beta_s", "ci_lo", "ci_hi", "r2", "beta_mono", "beta_geo", "w_eff", "p_holm", "verdict"]
print(E1_TABLE[_cols].to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
E1_TABLE.to_csv(os.path.join(RESULTS_DIR, "E1_beta_s.csv"), index=False)
E1_PARALLAX.to_csv(os.path.join(RESULTS_DIR, "E1_beta_by_parallax.csv"), index=False)

In [ ]:
# Cell No: 052
# 6.2 E2 / Prediction 2: do dynamic masks miss parallel motion?
#     Recall at a 5% static false-positive rate, parallel (|eps| < 1 px) vs perpendicular
#     (eps >= 3 px), stratified by |s| tertile x mover-area tertile (matched |s| and size),
#     cluster bootstrap over clips. The attention-based mask of the locked list (Easi3R) is not
#     evaluated: its 3-frame version (v5 M3) and its 10-frame version (v6.0 E7) are withdrawn (D23, D33).
METHOD_LABELS = {"M1_exact": "M1 epipolar residual (exact corr.)", "M1_flow": "M1 epipolar residual (flow)",
                 "M2_exact": "M2 rigid-flow residual (exact corr.)", "M2_flow": "M2 rigid-flow residual (flow)",
                 "M4": "M4 model confidence", "M5": "M5 two-view vs single-image depth (exploratory remedy)"}
MODEL_METHODS = ["M2_exact", "M2_flow", "M4", "M5"]

def stratified_recall_difference(df, recall_col, seed):
    d = df[df["regime"].isin(["parallel", "perpendicular"]) & np.isfinite(df[recall_col])].copy()
    if d["regime"].nunique() < 2 or len(d) < 10:
        return dict(diff=np.nan, ci_lo=np.nan, ci_hi=np.nan, rec_par=np.nan, rec_perp=np.nan, n_par=0, n_perp=0)
    d["s_bin"] = pd.qcut(d["s"].abs().rank(method="first"), 3, labels=False)
    d["a_bin"] = pd.qcut(d["gt_area_px"].rank(method="first"), 3, labels=False)
    d["stratum"] = d["s_bin"] * 3 + d["a_bin"]
    cols = []
    for k in range(9):
        ink = (d["stratum"] == k).values
        par = ink & (d["regime"] == "parallel").values; perp = ink & (d["regime"] == "perpendicular").values
        rv = d[recall_col].values
        cols += [par.astype(float), rv * par, perp.astype(float), rv * perp]
    bs = ClusterBootstrap(d["scene"].values, seed=seed)
    T, P = bs.linear(cols)
    def _diff(t):
        t = t.reshape(-1, 9, 4)
        n_par, s_par, n_perp, s_perp = t[..., 0], t[..., 1], t[..., 2], t[..., 3]
        ok = (n_par > 0) & (n_perp > 0)
        w = np.where(ok, n_par + n_perp, 0.0)
        with np.errstate(invalid="ignore", divide="ignore"):
            dk = np.where(ok, s_perp / np.maximum(n_perp, 1) - s_par / np.maximum(n_par, 1), 0.0)
            return (w * dk).sum(-1) / w.sum(-1)
    point = float(_diff(P[None])[0]); boots = _diff(T); boots = boots[np.isfinite(boots)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return dict(diff=point, ci_lo=float(lo), ci_hi=float(hi),
                rec_par=float(d.loc[d["regime"] == "parallel", recall_col].mean()),
                rec_perp=float(d.loc[d["regime"] == "perpendicular", recall_col].mean()),
                n_par=int((d["regime"] == "parallel").sum()), n_perp=int((d["regime"] == "perpendicular").sum()))

def verdict_p2(r):
    if not np.isfinite(r["diff"]):
        return "NOT TESTABLE"
    if r["ci_lo"] <= 0 <= r["ci_hi"] or r["ci_hi"] < 0:
        return "FALSIFIED"
    return "SUPPORTED" if r["diff"] >= PREREG["recall_diff_min"] else "PARTIAL"

E2_ROWS = []
_bm = BASE_RES[(BASE_RES["set"] == "MAIN") & BASE_RES["measurable"].astype(bool)]
for meth in ["M1_exact", "M1_flow"]:
    r = stratified_recall_difference(_bm, f"recall_{meth}", seed_for("e2", meth))
    E2_ROWS.append(dict(model="(model-independent)", method=METHOD_LABELS[meth], method_key=meth, **r,
                        auroc_par=float(_bm.loc[_bm["regime"] == "parallel", f"auroc_{meth}"].mean()),
                        auroc_perp=float(_bm.loc[_bm["regime"] == "perpendicular", f"auroc_{meth}"].mean())))
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & MOV_RES["measurable"].astype(bool)]
    for meth in MODEL_METHODS:
        if f"recall_{meth}" not in d or d[f"recall_{meth}"].notna().sum() == 0:
            continue
        r = stratified_recall_difference(d, f"recall_{meth}", seed_for("e2", model, meth))
        E2_ROWS.append(dict(model=model, method=METHOD_LABELS[meth], method_key=meth, **r,
                            auroc_par=float(d.loc[d["regime"] == "parallel", f"auroc_{meth}"].mean()),
                            auroc_perp=float(d.loc[d["regime"] == "perpendicular", f"auroc_{meth}"].mean())))
E2_TABLE = pd.DataFrame(E2_ROWS)
E2_TABLE["verdict"] = [("NOT INTERPRETABLE (gate)" if r["model"] != "(model-independent)" and not interpretable(r["model"], "MAIN")
                        else "EXPLORATORY" if r["method_key"] == "M5" else verdict_p2(r)) for r in E2_ROWS]
print(E2_TABLE[["model", "method", "n_par", "n_perp", "rec_par", "rec_perp", "diff", "ci_lo", "ci_hi",
                "auroc_par", "auroc_perp", "verdict"]].to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
E2_TABLE.to_csv(os.path.join(RESULTS_DIR, "E2_masks.csv"), index=False)

# recall as a function of the exact epipolar offset (for Figure 4)
EPS_BINS = [0, 0.25, 1, 2, 3, 6, 10, 1e9]
_rows = []
for model in ["(model-independent)"] + EVALUATED_MODELS:
    if model == "(model-independent)":
        src, meths = _bm, ["M1_exact", "M1_flow"]
    else:
        src = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & MOV_RES["measurable"].astype(bool)]
        meths = [m for m in MODEL_METHODS if f"recall_{m}" in src and src[f"recall_{m}"].notna().any()]
    src = src.assign(eps_bin=pd.cut(src["gt_eps_med"], EPS_BINS, right=False, labels=False))
    for meth in meths:
        for b, g in src.groupby("eps_bin"):
            _rows.append(dict(model=model, method_key=meth, eps_bin=int(b), eps_lo=EPS_BINS[int(b)],
                              eps_hi=EPS_BINS[int(b) + 1], n=len(g), recall=g[f"recall_{meth}"].mean(),
                              auroc=g[f"auroc_{meth}"].mean()))
E2_BY_EPS = pd.DataFrame(_rows)
E2_BY_EPS.to_csv(os.path.join(RESULTS_DIR, "E2_recall_by_eps.csv"), index=False)

# M5 (exploratory): recall on PARALLEL movers as a function of |x| = |log(1/(1-s))|, the depth
# discrepancy that the two-view prediction inherits and the single-image prediction does not
_rows = []
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & MOV_RES["measurable"].astype(bool)
                & (MOV_RES["motion_type"] == "parallel") & (MOV_RES["s"] < 1)]
    if "recall_M5" not in d or not len(d):
        continue
    for lo_, hi_ in [(0, 0.1), (0.1, 0.3), (0.3, 0.7), (0.7, 3.0)]:
        g = d[(d["x_par"].abs() >= lo_) & (d["x_par"].abs() < hi_)]
        _rows.append(dict(model=model, abs_x_lo=lo_, abs_x_hi=hi_, n=len(g), recall_M5=g["recall_M5"].mean(),
                          recall_M2_exact=g["recall_M2_exact"].mean()))
E2_M5_BY_X = pd.DataFrame(_rows)
if len(E2_M5_BY_X):
    print("\nM5 (exploratory remedy) vs M2 on parallel movers, by |x| = |-log(1-s)|:")
    print(E2_M5_BY_X.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
    E2_M5_BY_X.to_csv(os.path.join(RESULTS_DIR, "E2_M5_by_x.csv"), index=False)

In [ ]:
# Cell No: 053
# 6.3 E3 / Prediction 3: prior takeover. Prior weight per mover,
#     Zhat = w * Zhat_mono + (1 - w) * Z_geo  =>  w = (e^rho - e^x_geo) / (e^rho_mono - e^x_geo),
#     computed where the hypotheses are separable (|x_geo - rho_mono| > 0.10).
#     P3a: Spearman(w, |eps|) > 0.5.  P3b: at |eps| < 1 px, w(car) > w(crate).
def _spearman(c):
    if len(c["w"]) < 8:
        return np.nan
    return float(spearmanr(c["gt_eps_med"], c["w"]).correlation)

def _class_diff(c):
    a = c["w"][c["cls"] == "car"]; b = c["w"][c["cls"] != "car"]
    return float(np.median(a) - np.median(b)) if len(a) >= 3 and len(b) >= 3 else np.nan

E3_ROWS, E3_W = [], {}
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & MOV_RES["measurable"].astype(bool)
                & (MOV_RES["s"] < 1) & np.isfinite(MOV_RES["gt_x_geo"])].copy()
    d["w"] = prior_weight(d)
    d = d[np.isfinite(d["w"])]
    E3_W[model] = d
    rs, rs_lo, rs_hi, _ = boot_stat(d, "scene", _spearman, seed_for("e3s", model), cols=["gt_eps_med", "w"])
    dp = d[d["regime"] == "parallel"]
    cd, cd_lo, cd_hi, _ = boot_stat(dp, "scene", _class_diff, seed_for("e3c", model), cols=["cls", "w"])
    med = {c: float(dp.loc[dp["cls"] == c, "w"].median()) if (dp["cls"] == c).sum() else np.nan
           for c in ["car", "crate", "crate_floating"]}
    v3a = ("SUPPORTED" if rs > PREREG["spearman_min"] and rs_lo > 0 else
           "FALSIFIED" if not (rs_lo > 0) else "PARTIAL") if np.isfinite(rs_lo) else "NOT TESTABLE"
    v3b = ("SUPPORTED" if cd_lo > 0 else "NOT SUPPORTED") if np.isfinite(cd_lo) else "NOT TESTABLE"
    if not interpretable(model, "MAIN"):
        v3a = v3b = "NOT INTERPRETABLE (gate)"
    dynamic_trained = MODEL_TRAINING[model].startswith("dynamic")
    lock_c = None
    if dynamic_trained and np.isfinite(cd_lo) and interpretable(model, "MAIN"):
        beats_all = all(np.isfinite(v) and v > 0.5 for v in med.values())
        lock_c = "FALSIFIED (c)" if beats_all and cd_lo <= 0 <= cd_hi else "not triggered"
    E3_ROWS.append(dict(model=model, training=MODEL_TRAINING[model], n_w=len(d), n_parallel=len(dp),
                        median_w_parallel=float(dp["w"].median()) if len(dp) else np.nan,
                        median_w_perp=float(d.loc[d["regime"] == "perpendicular", "w"].median()),
                        spearman_w_eps=rs, sp_lo=rs_lo, sp_hi=rs_hi, verdict_P3a=v3a,
                        w_car=med["car"], w_crate=med["crate"], w_crate_floating=med["crate_floating"],
                        class_diff=cd, cd_lo=cd_lo, cd_hi=cd_hi, verdict_P3b=v3b,
                        lock_condition_c=lock_c or ("n/a (not dynamic-trained)" if not dynamic_trained else
                                                    "n/a (gate)" if not interpretable(model, "MAIN") else "n/a (not testable)")))
E3_TABLE = pd.DataFrame(E3_ROWS)
print(E3_TABLE.drop(columns=["training"]).to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
E3_TABLE.to_csv(os.path.join(RESULTS_DIR, "E3_prior_weight.csv"), index=False)
_rows = []
for model, d in E3_W.items():
    d = d.assign(eps_bin=pd.cut(d["gt_eps_med"], EPS_BINS, right=False, labels=False))
    for b, g in d.groupby("eps_bin"):
        _rows.append(dict(model=model, eps_bin=int(b), eps_lo=EPS_BINS[int(b)], eps_hi=EPS_BINS[int(b) + 1], n=len(g),
                          w_median=g["w"].median(), w_q25=g["w"].quantile(0.25), w_q75=g["w"].quantile(0.75)))
E3_BY_EPS = pd.DataFrame(_rows)
E3_BY_EPS.to_csv(os.path.join(RESULTS_DIR, "E3_w_by_eps.csv"), index=False)

In [ ]:
# Cell No: 054
# 6.4 E4 / Prediction 4: does the degeneracy persist over 2-8 frames on a straight camera
#     path, and break when the translation path bends? Multi-view models only. Each curved
#     clip is the twin of a straight clip (same street, movers, frames 0-1 and rotations), so
#     the straight-minus-curved difference is a paired comparison (cluster = twin).
E4_ROWS, E4_SUMMARY = [], []
for model in [m for m in EVALUATED_MODELS if MODEL_FAMILY[m] == "multiview"]:
    d = E4_RES[(E4_RES["model"] == model) & E4_RES["measurable"].astype(bool) & E4_RES["identifiable"].astype(bool)
               & (E4_RES["s"] < 1)].copy()
    d["twin"] = d["scene"] % N_E4
    res, n_k = {}, {}
    for K in RUN_CONFIG["e4_window_sizes"]:
        st = d[(d["path"] == "straight") & (d["K"] == K)][["twin", "mover", "x_par", "rho"]]
        cv = d[(d["path"] == "curved") & (d["K"] == K)][["twin", "mover", "rho", "gtlast_eps_med"]]
        m_ = st.merge(cv, on=["twin", "mover"], suffixes=("_st", "_cv"))
        b_st, b_cv, diff_b = paired_slopes(m_["x_par"], m_["rho_st"], m_["rho_cv"], m_["twin"], seed_for("e4", model, K))
        lo_st, hi_st = _ci(slope_boot(m_["x_par"], m_["rho_st"], m_["twin"], seed_for("e4s", model, K))[1])
        lo_cv, hi_cv = _ci(slope_boot(m_["x_par"], m_["rho_cv"], m_["twin"], seed_for("e4c", model, K))[1])
        dlo, dhi = _ci(diff_b)
        n_k[K] = len(m_)
        res[K] = dict(b_st=b_st, lo_st=lo_st, hi_st=hi_st, b_cv=b_cv, lo_cv=lo_cv, hi_cv=hi_cv,
                      diff=b_st - b_cv, dlo=dlo, dhi=dhi)
        for path, b, lo, hi in (("straight", b_st, lo_st, hi_st), ("curved", b_cv, lo_cv, hi_cv)):
            E4_ROWS.append(dict(model=model, path=path, K=K, n=len(m_), n_twins=int(m_["twin"].nunique()),
                                beta_s=b, ci_lo=lo, ci_hi=hi,
                                eps_first_last_median=float(m_["gtlast_eps_med"].median()) if path == "curved" and len(m_) else 0.0))
    Kmax = max(RUN_CONFIG["e4_window_sizes"])
    straight_in = all(PREREG["beta_support_range"][0] <= res[K]["b_st"] <= PREREG["beta_support_range"][1]
                      for K in RUN_CONFIG["e4_window_sizes"])
    straight_fail = any(res[K]["hi_st"] < PREREG["beta_ci_band"][0] or res[K]["lo_st"] > PREREG["beta_ci_band"][1]
                        for K in RUN_CONFIG["e4_window_sizes"])
    r8 = res[Kmax]
    v = ("NOT INTERPRETABLE (gate)" if not interpretable(model, "MAIN") else "NOT TESTABLE" if not np.isfinite(r8["b_st"]) else
         "FALSIFIED" if straight_fail else
         "SUPPORTED" if straight_in and r8["diff"] >= PREREG["e4_drop_min"] and r8["dlo"] > 0 else "PARTIAL")
    E4_SUMMARY.append(dict(model=model, **{f"straight_beta_K{K}": res[K]["b_st"] for K in RUN_CONFIG["e4_window_sizes"]},
                           **{f"curved_beta_K{K}": res[K]["b_cv"] for K in RUN_CONFIG["e4_window_sizes"]},
                           drop_K8=r8["diff"], drop_lo=r8["dlo"], drop_hi=r8["dhi"], n_K8=n_k[Kmax], verdict=v))
E4_TABLE = pd.DataFrame(E4_ROWS); E4_SUMMARY = pd.DataFrame(E4_SUMMARY)
# built-in check: at K = 2 a straight clip and its curved twin are the same two images
_k2 = E4_RES[E4_RES["K"] == 2].copy()
if len(_k2):
    _k2["twin"] = _k2["scene"] % N_E4
    _pv = _k2.pivot_table(index=["model", "twin", "mover"], columns="path", values="rho")
    if {"straight", "curved"} <= set(_pv.columns):
        print(f"Twin check (K = 2, identical inputs): max |rho_straight - rho_curved| = "
              f"{np.nanmax(np.abs(_pv['straight'] - _pv['curved'])):.2e}")
print(E4_TABLE.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
print()
print(E4_SUMMARY.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
E4_TABLE.to_csv(os.path.join(RESULTS_DIR, "E4_multiview.csv"), index=False)
E4_SUMMARY.to_csv(os.path.join(RESULTS_DIR, "E4_summary.csv"), index=False)

In [ ]:
# Cell No: 055
# 6.5 A1 correspondence knockout (causal test of assumption A1). The second view is
#     re-rendered with every mover deleted, so the mover has no correspondence. If the
#     two-view depth of a parallel mover comes from cross-view correspondence, beta_s must
#     drop towards the single-image value; if it comes from monocular cues, nothing changes.
#     Pre-registered rule: A1 holds for a model if the paired-bootstrap CI of
#     beta_s(full) - beta_s(knockout) lies above 0.
A1_ROWS = []
for model in EVALUATED_MODELS:
    d = E1_DATA[model]
    d = d[np.isfinite(d["rho_ko"])]
    full = slope_boot(d["x_par"], d["rho"], d["scene"], seed_for("a1f", model))
    ko = slope_boot(d["x_par"], d["rho_ko"], d["scene"], seed_for("a1k", model))
    mono = slope_boot(d["x_par"], d["rho_mono"], d["scene"], seed_for("a1m", model))
    nb = min(len(full[1]), len(ko[1]))
    # paired bootstrap of the drop (same clips in both conditions): reuse the same seed
    T, _ = ClusterBootstrap(d["scene"].values, seed=seed_for("a1d", model)).linear(
        [np.ones(len(d)), d["x_par"].values, d["rho"].values, d["x_par"].values ** 2, (d["x_par"] * d["rho"]).values,
         d["rho_ko"].values, (d["x_par"] * d["rho_ko"]).values]) if len(d) >= 8 else (np.zeros((0, 7)), None)
    if len(T):
        with np.errstate(invalid="ignore", divide="ignore"):
            den = T[:, 0] * T[:, 3] - T[:, 1] ** 2
            bfull = (T[:, 0] * T[:, 4] - T[:, 1] * T[:, 2]) / den
            bko = (T[:, 0] * T[:, 6] - T[:, 1] * T[:, 5]) / den
        drop = bfull - bko
        dlo, dhi = np.nanpercentile(drop, [2.5, 97.5])
    else:
        dlo = dhi = np.nan
    ci = lambda bb: np.nanpercentile(bb, [2.5, 97.5]) if len(bb) > 50 else (np.nan, np.nan)
    flo, fhi = ci(full[1]); klo, khi = ci(ko[1]); mlo, mhi = ci(mono[1])
    A1_ROWS.append(dict(model=model, n=len(d), n_clips=int(d["scene"].nunique()),
                        beta_full=full[0], full_lo=flo, full_hi=fhi, beta_knockout=ko[0], ko_lo=klo, ko_hi=khi,
                        beta_mono=mono[0], mono_lo=mlo, mono_hi=mhi, drop=full[0] - ko[0], drop_lo=dlo, drop_hi=dhi,
                        median_abs_rho_change=float(np.median(np.abs(d["rho"] - d["rho_ko"]))) if len(d) else np.nan,
                        static_sensitivity=float(np.median(np.abs(d.loc[d["s"].abs() < 0.15, "rho"] - d.loc[d["s"].abs() < 0.15, "rho_ko"])))
                        if (d["s"].abs() < 0.15).sum() >= 3 else np.nan,
                        A1=("NOT INTERPRETABLE (gate)" if not interpretable(model, "MAIN")
                            else "HOLDS (deleting the correspondence lowers beta_s)" if np.isfinite(dlo) and dlo > 0
                            else "NOT DETECTED (depth does not depend on the correspondence)" if np.isfinite(dlo)
                            else "NOT TESTABLE")))
A1_TABLE = pd.DataFrame(A1_ROWS)
print(A1_TABLE.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
A1_TABLE.to_csv(os.path.join(RESULTS_DIR, "A1_knockout.csv"), index=False)

In [ ]:
# Cell No: 056
# 6.6 Replication on real images (CONVOY-REAL: TUM backgrounds, motion-capture poses,
#     inserted movers with exact motion). Same protocol as E1 and E2.
REAL_E1_ROWS, REAL_E2_ROWS = [], []
_rb = BASE_RES[(BASE_RES["set"] == "REAL")] if len(BASE_RES) else pd.DataFrame()
if len(_rb):
    _rbp = _rb[e1_mask(_rb)]
    for lab, col in [("Classical triangulation, exact correspondences", "rho_classical_exact"),
                     (f"Classical triangulation, {FLOW_NAME} flow", "rho_classical_flow")]:
        r = slope_with_ci(_rbp["x_par"], _rbp[col], _rbp["scene"], seed_for("real-c", col))
        REAL_E1_ROWS.append(dict(model=lab, gate="-", n=r["n"], n_pairs=r["n_clusters"], beta_s=r["beta"],
                                 ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], r2=r["r2"], beta_mono=np.nan, verdict="control"))
    for meth in ["M1_exact", "M1_flow"]:
        r = stratified_recall_difference(_rb[_rb["measurable"].astype(bool)], f"recall_{meth}", seed_for("real-e2", meth))
        REAL_E2_ROWS.append(dict(model="(model-independent)", method=METHOD_LABELS[meth], **r, verdict=verdict_p2(r)))
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "REAL")]
    if not len(d):
        continue
    dp = d[e1_mask(d)]
    r = slope_with_ci(dp["x_par"], dp["rho"], dp["scene"], seed_for("real-e1", model), h0=0.5)
    rm = slope_with_ci(dp["x_par"], dp["rho_mono"], dp["scene"], seed_for("real-e1m", model))
    gate_ok = GATE_PASS.get((model, "REAL"), False)
    REAL_E1_ROWS.append(dict(model=model, gate="PASS" if gate_ok else "FAIL", n=r["n"], n_pairs=r["n_clusters"],
                             beta_s=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], r2=r["r2"], beta_mono=rm["beta"],
                             verdict=verdict_p1(r["beta"], r["ci_lo"], r["ci_hi"], r["r2"]) if gate_ok else "NOT INTERPRETABLE (gate)"))
    dm = d[d["measurable"].astype(bool)]
    for meth in ["M2_exact", "M2_flow", "M4", "M5"]:
        if f"recall_{meth}" not in dm or dm[f"recall_{meth}"].notna().sum() == 0:
            continue
        r = stratified_recall_difference(dm, f"recall_{meth}", seed_for("real-e2", model, meth))
        REAL_E2_ROWS.append(dict(model=model, method=METHOD_LABELS[meth], **r,
                                 verdict=("NOT INTERPRETABLE (gate)" if not gate_ok else
                                          "EXPLORATORY" if meth == "M5" else verdict_p2(r))))
REAL_E1 = pd.DataFrame(REAL_E1_ROWS, columns=["model", "gate", "n", "n_pairs", "beta_s", "ci_lo", "ci_hi", "r2", "beta_mono", "verdict"])
REAL_E2 = pd.DataFrame(REAL_E2_ROWS, columns=["model", "method", "diff", "ci_lo", "ci_hi", "rec_par", "rec_perp", "n_par", "n_perp", "verdict"])
print("CONVOY-REAL, E1 replication (purely parallel movers):")
print(REAL_E1.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
print("\nCONVOY-REAL, E2 replication:")
if len(REAL_E2):
    print(REAL_E2[["model", "method", "n_par", "n_perp", "rec_par", "rec_perp", "diff", "ci_lo", "ci_hi", "verdict"]]
          .to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
REAL_E1.to_csv(os.path.join(RESULTS_DIR, "real_E1.csv"), index=False)
REAL_E2.to_csv(os.path.join(RESULTS_DIR, "real_E2.csv"), index=False)

In [ ]:
# Cell No: 057
# 6.7 E5 ground-contact ablation (H5-H8 registered; shadow and class contrasts exploratory).
E5_CONDS = ABLATION_DESIGN["conditions"]
_E5_VALS = ["rho", "rho_mono", "gt_n_eroded", "gt_parallax_med", "gt_visible_frac", "contact_px0", "contact_px1", "has_wall"]

def e5_wide(model):
    # one row per base mover, one column per (quantity, condition)
    d = ABL_RES[ABL_RES["model"] == model].copy()
    d["has_wall"] = d["has_wall"].astype(float)
    w = d.pivot_table(index=["uid", "scene"], columns="condition", values=_E5_VALS, aggfunc="first")
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    gt = MOVERS_ABL[MOVERS_ABL["condition"] == "G"][["uid", "cls", "s", "x_par"]]
    w = w.reset_index().merge(gt, on="uid", how="left")
    for c in E5_CONDS:
        w[f"ok_{c}"] = (w.get(f"gt_n_eroded_{c}", 0) >= PREREG["min_object_pixels"]) & \
                       (w.get(f"gt_parallax_med_{c}", 0) >= PREREG["parallax_min_px"]) & np.isfinite(w.get(f"rho_{c}", np.nan))
    w["vis_ok"] = np.all([w[f"gt_visible_frac_{c}"] >= 0.85 for c in ("G", "NS", "F")], axis=0)
    w["walled"] = ((w["has_wall_CH"] == 1) & (w["contact_px0_CH"] == 0) & (w["contact_px1_CH"] == 0)
                   & (w["contact_px0_G"] >= 5) & (w["contact_px1_G"] >= 5))
    return w

def _sample(w, conds, walled=False):
    m = w["vis_ok"] & np.all([w[f"ok_{c}"] for c in conds], axis=0)
    return w[m & w["walled"]] if walled else w[m]

E5_TESTS_SPEC = [  # (id, label, conditions, weights, needs wall, registered)
    ("H5", "beta(F) - beta(NS): lifting removes the contact", ["F", "NS"], [1, -1], False, True),
    ("H6", "beta(CH) - beta(G): hiding the contact", ["CH", "G"], [1, -1], True, True),
    ("H7", "(beta_CH - beta_G) - (beta_FH - beta_F): the wall acts through the contact", ["CH", "G", "FH", "F"], [1, -1, -1, 1], True, True),
    ("SH", "beta(NS) - beta(G): removing the contact shadow", ["NS", "G"], [1, -1], False, False),
]

def _group_slope_diff(x, y, is_a, clusters, seed):
    # slope in group a minus slope in group b (unpaired movers, clusters resampled jointly)
    x, y, a = np.asarray(x, float), np.asarray(y, float), np.asarray(is_a, bool)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, a, cl = x[ok], y[ok], a[ok], np.asarray(clusters)[ok]
    if a.sum() < 8 or (~a).sum() < 8 or len(np.unique(cl)) < 5:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    cols = []
    for g in (a, ~a):
        gf = g.astype(float)
        cols += [gf, gf * x, gf * x * x, gf * y, gf * x * y]
    T, P = ClusterBootstrap(cl, seed=seed).linear(cols)
    def sl(t, o):
        with np.errstate(invalid="ignore", divide="ignore"):
            return (t[:, o] * t[:, o + 4] - t[:, o + 1] * t[:, o + 3]) / (t[:, o] * t[:, o + 2] - t[:, o + 1] ** 2)
    pa, pb = sl(P[None], 0)[0], sl(P[None], 5)[0]
    dd = sl(T, 0) - sl(T, 5); dd = dd[np.isfinite(dd)]
    lo, hi = np.percentile(dd, [2.5, 97.5]) if len(dd) > 50 else (np.nan, np.nan)
    return float(pa), float(pb), float(pa - pb), float(lo), float(hi)

E5_BETA_ROWS, E5_TEST_ROWS, E5_WIDE = [], [], {}
for model in EVALUATED_MODELS:
    w = e5_wide(model)
    E5_WIDE[model] = w
    gate_ok = GATE_PASS.get((model, "ABL"), False)
    # slopes per condition on the common sample (valid in all five conditions, wall built and contact hidden)
    s5 = _sample(w, E5_CONDS, walled=True)
    b5, d5, n5, nc5 = multi_slopes(s5["x_par"], [s5[f"rho_{c}"] for c in E5_CONDS], s5["scene"], seed_for("e5-all", model))
    for k, c in enumerate(E5_CONDS):
        lo, hi = _ci(d5[:, k]) if len(d5) else (np.nan, np.nan)
        E5_BETA_ROWS.append(dict(model=model, condition=c, label=ABLATION_DESIGN["labels"][c], n=n5, n_scenes=nc5,
                                 beta=b5[k], ci_lo=lo, ci_hi=hi, median_rho_mono=float(s5[f"rho_mono_{c}"].median()) if len(s5) else np.nan))
    for tid, lab, conds, wts, walled, registered in E5_TESTS_SPEC:
        sm_ = _sample(w, conds, walled)
        b, dr, n, nc = multi_slopes(sm_["x_par"], [sm_[f"rho_{c}"] for c in conds], sm_["scene"], seed_for("e5", model, tid))
        est, lo, hi = contrast_ci(b, dr, wts)
        p = float((1 + np.sum(dr @ np.asarray(wts, float) <= 0)) / (len(dr) + 1)) if len(dr) else np.nan
        v = verdict_positive(lo, hi) if gate_ok else "NOT INTERPRETABLE (gate)"
        E5_TEST_ROWS.append(dict(model=model, test=tid, contrast=lab, registered=registered, n=n, n_scenes=nc,
                                 **{f"beta_{c}": b[i] for i, c in enumerate(conds)}, estimate=est, ci_lo=lo, ci_hi=hi,
                                 p_boot=p, verdict=v if registered else ("EXPLORATORY" if gate_ok else v)))
    # H8: single-image depth of the same mover, floating vs grounded without shadow
    s8 = w[w["vis_ok"] & (w["gt_n_eroded_F"] >= PREREG["min_object_pixels"]) & (w["gt_n_eroded_NS"] >= PREREG["min_object_pixels"])
           & np.isfinite(w["rho_mono_F"]) & np.isfinite(w["rho_mono_NS"])]
    if len(s8) >= 8 and s8["scene"].nunique() >= 5:
        est, lo, hi, dr = boot_stat(s8, "scene", lambda c: float(np.median(c["rho_mono_F"]) - np.median(c["rho_mono_NS"])),
                                    seed_for("e5-h8", model), cols=["rho_mono_F", "rho_mono_NS"])
        p = float((1 + np.sum(dr <= 0)) / (len(dr) + 1))
    else:
        est = lo = hi = p = np.nan
    E5_TEST_ROWS.append(dict(model=model, test="H8", contrast="median rho_mono(F) - median rho_mono(NS): single-image depth of a "
                             "lifted object", registered=True, n=len(s8), n_scenes=int(s8["scene"].nunique()), estimate=est,
                             ci_lo=lo, ci_hi=hi, p_boot=p, verdict=verdict_positive(lo, hi) if gate_ok else "NOT INTERPRETABLE (gate)"))
    # exploratory: floating car vs floating crate (different movers, same scenes)
    sf = _sample(w, ["F"])
    ba, bb_, est, lo, hi = _group_slope_diff(sf["x_par"], sf["rho_F"], sf["cls"] == "car", sf["scene"], seed_for("e5-cls", model))
    E5_TEST_ROWS.append(dict(model=model, test="CL", contrast="beta(F, car) - beta(F, crate): floating car vs floating crate",
                             registered=False, n=len(sf), n_scenes=int(sf["scene"].nunique()), beta_car=ba, beta_crate=bb_,
                             estimate=est, ci_lo=lo, ci_hi=hi, p_boot=np.nan, verdict="EXPLORATORY" if gate_ok else "NOT INTERPRETABLE (gate)"))
E5_BETA = pd.DataFrame(E5_BETA_ROWS)
E5_TESTS = pd.DataFrame(E5_TEST_ROWS)
for _t in ["H5", "H6", "H7", "H8"]:
    _m = E5_TESTS["test"] == _t
    E5_TESTS.loc[_m, "p_holm"] = holm(E5_TESTS.loc[_m, "p_boot"].values)
print("E5: beta_s per condition (common sample: valid in all five conditions, wall built, contact hidden in CH):")
print(E5_BETA.pivot_table(index="model", columns="condition", values="beta")[E5_CONDS]
      .to_string(float_format=lambda v: f"{v:.3f}"), f"\n(n = {E5_BETA.groupby('model')['n'].first().to_dict()})")
print("\nE5 contrasts:")
with pd.option_context("display.max_colwidth", 50, "display.width", 250):
    print(E5_TESTS[["model", "test", "n", "n_scenes", "estimate", "ci_lo", "ci_hi", "p_holm", "verdict"]]
          .to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
E5_BETA.to_csv(os.path.join(RESULTS_DIR, "E5_beta_by_condition.csv"), index=False)
E5_TESTS.to_csv(os.path.join(RESULTS_DIR, "E5_contrasts.csv"), index=False)

In [ ]:
# Cell No: 058
# 6.7c Exploratory (v6.2, AMENDMENT_V6_2): do the masks follow the support cue? Recall of the
#      rigid-flow residual M2 (exact correspondences, at the image's 5% static false-positive rate) on
#      the same parallel movers in every ablation condition. A model that uses more geometry for a
#      mover leaves a smaller residual on it, so M2 should catch fewer movers when the contact is
#      hidden (G -> CH) or the mover is lifted (NS -> F). Paired by mover; cluster = base scene.
def _dir_label(lo, hi):
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "NOT TESTABLE"
    return "lower" if hi < 0 else "higher" if lo > 0 else "no clear change"

E5M_ROWS, E5M_TEST_ROWS = [], []
for model in MODELS_OK["ABL"]:
    w = E5_WIDE.get(model)
    if w is None:
        continue
    s5 = _sample(w, E5_CONDS, walled=True)[["uid"]]
    d = ABL_RES[(ABL_RES["model"] == model) & ABL_RES["uid"].isin(s5["uid"])]
    r = d.pivot_table(index=["uid", "scene"], columns="condition", values="recall_M2_exact", aggfunc="first").reset_index()
    r = r.dropna(subset=[c for c in E5_CONDS if c in r.columns])
    if len(r) < 8 or r["scene"].nunique() < 5:
        continue
    for c in E5_CONDS:
        est, lo, hi, _ = boot_stat(r, "scene", lambda q, c=c: float(np.mean(q[c])), seed_for("e5m", model, c), cols=[c])
        E5M_ROWS.append(dict(model=model, condition=c, label=ABLATION_DESIGN["labels"][c], n=len(r), recall_M2=est, ci_lo=lo, ci_hi=hi))
    for a, b, what in (("CH", "G", "contact hidden - grounded"), ("F", "NS", "lifted - grounded without shadow")):
        est, lo, hi, _ = boot_stat(r, "scene", lambda q, a=a, b=b: float(np.mean(q[a] - q[b])), seed_for("e5m-c", model, a, b),
                                   cols=[a, b])
        E5M_TEST_ROWS.append(dict(model=model, contrast=f"{a} - {b}", meaning=what, n=len(r), n_scenes=int(r["scene"].nunique()),
                                  estimate=est, ci_lo=lo, ci_hi=hi, direction=_dir_label(lo, hi)))
E5_MASKS = pd.DataFrame(E5M_ROWS)
E5_MASK_TESTS = pd.DataFrame(E5M_TEST_ROWS)
if len(E5_MASKS):
    print("Exploratory: M2 recall (exact correspondences) on the same parallel movers in each ablation condition:")
    print(E5_MASKS.pivot_table(index="model", columns="condition", values="recall_M2")[E5_CONDS]
          .to_string(float_format=lambda v: f"{v:.3f}"))
    print("\nPaired changes (95% cluster-bootstrap intervals over base scenes):")
    print(E5_MASK_TESTS[["model", "contrast", "n", "estimate", "ci_lo", "ci_hi", "direction"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    E5_MASKS.to_csv(os.path.join(RESULTS_DIR, "E5_masks_by_condition.csv"), index=False)
    E5_MASK_TESTS.to_csv(os.path.join(RESULTS_DIR, "E5_masks_contrasts.csv"), index=False)
else:
    print("Masks by ablation condition: no interpretable model with enough movers in this run.")

In [ ]:
# Cell No: 059
# 6.7b E5-REAL: the ground-contact ablation on real backgrounds (H11a, H11b; registered in v6.1).
RABL_CONDS = RABL_DESIGN["conditions"]
_RABL_VALS = ["rho", "rho_mono", "gt_n_eroded", "gt_parallax_med", "gt_visible_frac", "contact_px0", "contact_px1"]

def rabl_wide(model):
    # one row per crate, one column per (quantity, condition)
    d = RABL_RES[RABL_RES["model"] == model]
    if not len(d):
        return pd.DataFrame()
    w = d.pivot_table(index=["uid", "scene"], columns="condition", values=_RABL_VALS, aggfunc="first")
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    gt = MOVERS_RABL[MOVERS_RABL["condition"] == "G"][["uid", "s", "x_par", "seq"]]
    w = w.reset_index().merge(gt, on="uid", how="left")
    for c in RABL_CONDS:
        w[f"ok_{c}"] = ((w[f"gt_n_eroded_{c}"] >= PREREG["min_object_pixels"]) & (w[f"gt_parallax_med_{c}"] >= PREREG["parallax_min_px"])
                        & (w[f"gt_visible_frac_{c}"] >= 0.85) & np.isfinite(w[f"rho_{c}"]))
    # the contact is visible in both views when grounded and absent when lifted (measured on the composites)
    w["contact_ok"] = ((w["contact_px0_G"] >= 5) & (w["contact_px1_G"] >= 5) & (w["contact_px0_F"] == 0) & (w["contact_px1_F"] == 0))
    w["sample"] = w["ok_G"] & w["ok_F"] & w["contact_ok"] & np.isfinite(w["x_par"])
    return w

RABL_BETA_ROWS, RABL_TEST_ROWS, RABL_WIDE = [], [], {}
# positive control on the same composites: classical triangulation with the true pose follows x exactly
_rb = BASELINES[BASELINES["set"] == "RABL"].merge(MOVERS_RABL[["uid", "condition", "x_par", "measurable", "identifiable"]],
                                                   on=["uid", "condition"], how="left") if len(MOVERS_RABL) else pd.DataFrame()
RABL_CONTROL = {}
for c in RABL_CONDS:
    if len(_rb):
        dc = _rb[(_rb["condition"] == c) & _rb["measurable"].astype(bool) & _rb["identifiable"].astype(bool)]
        RABL_CONTROL[c] = slope_with_ci(dc["x_par"], dc["rho_classical_exact"], dc["scene"], seed_for("rabl-control", c))
for model in EVALUATED_MODELS:
    if "RABL" not in EVAL_STAGES[model]:
        continue
    w = rabl_wide(model)
    if not len(w):
        continue
    RABL_WIDE[model] = w
    ok = interpretable(model, "RABL")
    s_ = w[w["sample"]]
    b, dr, n, nc = multi_slopes(s_["x_par"], [s_["rho_G"], s_["rho_F"]], s_["scene"], seed_for("rabl", model))
    for k, c in enumerate(RABL_CONDS):
        lo, hi = _ci(dr[:, k]) if len(dr) else (np.nan, np.nan)
        RABL_BETA_ROWS.append(dict(model=model, condition=c, label=RABL_DESIGN["labels"][c], n=n, n_pairs=nc, beta=b[k],
                                   ci_lo=lo, ci_hi=hi, median_rho_mono=float(s_[f"rho_mono_{c}"].median()) if len(s_) else np.nan))
    est, lo, hi = contrast_ci(b, dr, [-1, 1])
    p = float((1 + np.sum(dr @ np.array([-1.0, 1.0]) <= 0)) / (len(dr) + 1)) if len(dr) else np.nan
    if n < 8 or nc < 5:                                  # registered minimum sample (AMENDMENT_V6_1)
        lo = hi = p = np.nan
    RABL_TEST_ROWS.append(dict(model=model, test="H11a", contrast="beta(F) - beta(G): lifting the crate off the real support",
                               n=n, n_pairs=nc, beta_G=b[0], beta_F=b[1], estimate=est, ci_lo=lo, ci_hi=hi, p_boot=p,
                               verdict=verdict_positive(lo, hi) if ok else "NOT INTERPRETABLE (gate)"))
    s8 = w[w["ok_G"] & w["ok_F"] & w["contact_ok"] & np.isfinite(w["rho_mono_G"]) & np.isfinite(w["rho_mono_F"])]
    if len(s8) >= 8 and s8["scene"].nunique() >= 5:
        est, lo, hi, d8 = boot_stat(s8, "scene", lambda c_: float(np.median(c_["rho_mono_F"]) - np.median(c_["rho_mono_G"])),
                                    seed_for("rabl-h11b", model), cols=["rho_mono_F", "rho_mono_G"])
        p = float((1 + np.sum(d8 <= 0)) / (len(d8) + 1))
    else:
        est = lo = hi = p = np.nan
    RABL_TEST_ROWS.append(dict(model=model, test="H11b", contrast="median rho_mono(F) - median rho_mono(G): single-image depth "
                               "of the lifted crate", n=len(s8), n_pairs=int(s8["scene"].nunique()), estimate=est, ci_lo=lo,
                               ci_hi=hi, p_boot=p, verdict=verdict_positive(lo, hi) if ok else "NOT INTERPRETABLE (gate)"))
RABL_BETA = pd.DataFrame(RABL_BETA_ROWS)
RABL_TESTS = pd.DataFrame(RABL_TEST_ROWS)
for _t in ["H11a", "H11b"]:
    if len(RABL_TESTS):
        _m = RABL_TESTS["test"] == _t
        RABL_TESTS.loc[_m, "p_holm"] = holm(RABL_TESTS.loc[_m, "p_boot"].values)
if len(RABL_TESTS):
    print("Positive control (classical triangulation, true pose, same composites): "
          + "; ".join(f"{RABL_DESIGN['labels'][c]}: beta = {v['beta']:.3f} [{v['ci_lo']:.3f}, {v['ci_hi']:.3f}], n = {v['n']}"
                      for c, v in RABL_CONTROL.items()))
    print("\nE5-REAL: beta_s per condition (same crates in both conditions):")
    print(RABL_BETA.pivot_table(index="model", columns="condition", values="beta")[RABL_CONDS]
          .to_string(float_format=lambda v: f"{v:.3f}"), f"\n(n = {RABL_BETA.groupby('model')['n'].first().to_dict()})")
    print("\nE5-REAL contrasts:")
    with pd.option_context("display.max_colwidth", 50, "display.width", 250):
        print(RABL_TESTS[["model", "test", "n", "n_pairs", "estimate", "ci_lo", "ci_hi", "p_holm", "verdict"]]
              .to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
    RABL_BETA.to_csv(os.path.join(RESULTS_DIR, "E5real_beta_by_condition.csv"), index=False)
    RABL_TESTS.to_csv(os.path.join(RESULTS_DIR, "E5real_contrasts.csv"), index=False)
else:
    print("E5-REAL is not part of this run (no real-background composites).")

In [ ]:
# Cell No: 060
# 6.7d Single-image protocol check (v6.2). The single-image reference of a multi-view model is the
#      model given one image. pi3 contradicted H11b in v6.1 with a very wide interval, so before any
#      single-image claim is made the same frames are also given as the image twice (the protocol
#      used for the CroCo models). For every multi-view model: H11b under both protocols on the same
#      crates, their per-crate agreement, the static-scene quality of both single-image depths, and
#      the CONVOY-REAL single-image slope under both. A model whose single-image results depend on
#      the protocol is reported as such and its single-image claims are not made.
PROTO_ROWS = []
_mv_models = [m for m in EVALUATED_MODELS if MODEL_FAMILY.get(m) == "multiview"]
for model in _mv_models:
    rec = dict(model=model)
    w = RABL_WIDE.get(model)
    d = RABL_RES[RABL_RES["model"] == model] if len(RABL_RES) else pd.DataFrame()
    if w is not None and len(d) and "rho_mono_dup" in d:
        dup = d.pivot_table(index="uid", columns="condition", values="rho_mono_dup", aggfunc="first")
        dup.columns = [f"rho_mono_dup_{c}" for c in dup.columns]
        s8 = w[w["ok_G"] & w["ok_F"] & w["contact_ok"]].merge(dup.reset_index(), on="uid", how="left")
        s8 = s8[np.isfinite(s8["rho_mono_G"]) & np.isfinite(s8["rho_mono_F"]) & np.isfinite(s8["rho_mono_dup_G"])
                & np.isfinite(s8["rho_mono_dup_F"])]
        if len(s8) >= 8 and s8["scene"].nunique() >= 5:
            for tag, cg, cf in (("one image", "rho_mono_G", "rho_mono_F"), ("image twice", "rho_mono_dup_G", "rho_mono_dup_F")):
                est, lo, hi, _ = boot_stat(s8, "scene", lambda q, cg=cg, cf=cf: float(np.median(q[cf]) - np.median(q[cg])),
                                           seed_for("proto", model, tag), cols=[cg, cf])
                key = "one" if tag == "one image" else "twice"
                rec.update({f"H11b_{key}": est, f"H11b_{key}_lo": lo, f"H11b_{key}_hi": hi})
            both = np.r_[s8["rho_mono_G"] - s8["rho_mono_dup_G"], s8["rho_mono_F"] - s8["rho_mono_dup_F"]]
            rec.update(n_crates=len(s8), median_abs_diff=float(np.median(np.abs(both))),
                       spearman=float(pd.Series(np.r_[s8["rho_mono_G"], s8["rho_mono_F"]]).corr(
                           pd.Series(np.r_[s8["rho_mono_dup_G"], s8["rho_mono_dup_F"]]), method="spearman")))
        im = IMG_RES[(IMG_RES["model"] == model) & (IMG_RES["set"] == "RABL")]
        if "mono_dup_absrel" in im:
            rec.update(static_absrel_one=float(im["mono_absrel"].median()), static_absrel_twice=float(im["mono_dup_absrel"].median()))
    rr = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "REAL") & (MOV_RES["motion_type"] == "parallel")
                 & MOV_RES["measurable"].astype(bool) & MOV_RES["identifiable"].astype(bool)] if len(MOV_RES) else pd.DataFrame()
    if len(rr) and "rho_mono_dup" in rr:
        for key, col in (("one", "rho_mono"), ("twice", "rho_mono_dup")):
            sl = slope_with_ci(rr["x_par"], rr[col], rr["scene"], seed_for("proto-real", model, key))
            rec.update({f"real_mono_slope_{key}": sl["beta"], f"real_mono_r2_{key}": sl.get("r2", np.nan)})
    if "H11b_one" in rec:
        same_sign = np.sign(rec["H11b_one"]) == np.sign(rec["H11b_twice"])
        overlap = rec["H11b_one_lo"] <= rec["H11b_twice_hi"] and rec["H11b_twice_lo"] <= rec["H11b_one_hi"]
        rec["protocol"] = "consistent" if (same_sign and overlap) else "PROTOCOL-DEPENDENT"
    PROTO_ROWS.append(rec)
PROTOCOL_CHECK = pd.DataFrame(PROTO_ROWS)
PROTOCOL_DEPENDENT = set(PROTOCOL_CHECK.loc[PROTOCOL_CHECK.get("protocol", pd.Series(dtype=str)) == "PROTOCOL-DEPENDENT", "model"]) \
    if "protocol" in PROTOCOL_CHECK else set()
if len(PROTOCOL_CHECK):
    print("Single-image protocol check (multi-view models; E5-REAL crates and CONVOY-REAL movers):")
    with pd.option_context("display.width", 250):
        print(PROTOCOL_CHECK.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
    PROTOCOL_CHECK.to_csv(os.path.join(RESULTS_DIR, "single_image_protocol_check.csv"), index=False)
    if PROTOCOL_DEPENDENT:
        print("Single-image results depend on the input protocol for: " + ", ".join({"pi3": "π³", "VGGT-1B": "VGGT"}.get(m, m) for m in sorted(PROTOCOL_DEPENDENT))
              + "; their single-image claims (H8, H11b) are reported but not made in the paper.")
else:
    print("No multi-view model in this run.")

In [ ]:
# Cell No: 061
# 6.8 E6 driving scenes (Virtual KITTI 2): H9 and the prevalence of undetectable motion.
def verdict_h9(beta, lo, hi, n, n_pairs):
    if not np.isfinite(beta) or n < PREREG["h9_min_movers"] or n_pairs < PREREG["h9_min_pairs"]:
        return "NOT TESTABLE"
    if np.isfinite(lo) and lo > PREREG["h9_beta_max"]:
        return "FALSIFIED"
    if beta < PREREG["h9_beta_max"] and np.isfinite(hi) and hi < PREREG["h9_ci_hi_max"]:
        return "SUPPORTED"
    return "PARTIAL"

E6_ROWS, E6_DATA, E6_MASKS = [], {}, []
if VK_STATUS["available"] and len(MOVERS_VK):
    _bv = BASE_RES[(BASE_RES["set"] == "VK")]
    _bvp = _bv[h9_mask(_bv)]
    r = slope_with_ci(_bvp["gt_x_geo"], _bvp["rho_classical_exact"], _bvp["scene"], seed_for("e6", "classical"))
    E6_ROWS.append(dict(model="Classical triangulation, exact correspondences", trained_on_vkitti="-", gate="-", n=r["n"],
                        n_pairs=r["n_clusters"], beta_geo=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], r2=r["r2"],
                        beta_mono=np.nan, w_eff=np.nan, verdict="control"))
    for model in EVALUATED_MODELS:
        d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "VK")]
        if not len(d):
            continue
        dp = d[h9_mask(d)]
        r = slope_with_ci(dp["gt_x_geo"], dp["rho"], dp["scene"], seed_for("e6", model))
        rm = slope_with_ci(dp["gt_x_geo"], dp["rho_mono"], dp["scene"], seed_for("e6m", model))
        gate_ok = GATE_PASS.get((model, "VK"), False)
        v = verdict_h9(r["beta"], r["ci_lo"], r["ci_hi"], r["n"], r["n_clusters"]) if gate_ok else "NOT INTERPRETABLE (gate)"
        E6_ROWS.append(dict(model=model, trained_on_vkitti="yes" if VK_TRAINING_EXPOSURE.get(model) else "no",
                            gate="PASS" if gate_ok else "FAIL", n=r["n"], n_pairs=r["n_clusters"], beta_geo=r["beta"],
                            ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], r2=r["r2"], beta_mono=rm["beta"],
                            w_eff=beta_to_effective_w(r["beta"]) if np.isfinite(r["beta"]) else np.nan, verdict=v))
        E6_DATA[model] = dp[["uid", "scene", "seq", "gt_x_geo", "rho", "rho_mono", "s", "gt_Z_true", "gt_parallax_med"]].copy()
        # exploratory: model-based dynamic masks on moving vehicles, by regime. M1 is not listed: with exact
        # correspondences its static false-positive threshold is 0 px, so on these frames it flags any
        # offset above the precision of the released poses, which no real two-view test can do.
        dm = d[d["measurable"].astype(bool)]
        for reg, g in dm.groupby("regime"):
            E6_MASKS.append(dict(model=model, regime=reg, n=len(g), recall_M2_exact=g["recall_M2_exact"].mean(),
                                 recall_M5=g["recall_M5"].mean()))
E6_TABLE = pd.DataFrame(E6_ROWS, columns=["model", "trained_on_vkitti", "gate", "n", "n_pairs", "beta_geo", "ci_lo", "ci_hi",
                                          "r2", "beta_mono", "w_eff", "verdict"])
E6_MASKS = pd.DataFrame(E6_MASKS, columns=["model", "regime", "n", "recall_M2_exact", "recall_M5"])

# prevalence of motion that two views (and K views) cannot see
PREVALENCE_ALL = pd.concat([PREVALENCE.assign(dataset="TUM fr3 walking (people)") if len(PREVALENCE) else None,
                            PREVALENCE_VK.assign(dataset="Virtual KITTI 2 (vehicles)") if len(PREVALENCE_VK) else None],
                           ignore_index=True) if (len(PREVALENCE) or len(PREVALENCE_VK)) else pd.DataFrame()
if len(KVIEW_VK):
    KVIEW_SUMMARY = (KVIEW_VK.assign(static_fit=KVIEW_VK["residual_px"] < 1.0)
                     .groupby("K").agg(n_tracks=("static_fit", "size"), share_static_fit_below_1px=("static_fit", "mean"),
                                       median_residual_px=("residual_px", "median")).reset_index())
else:
    KVIEW_SUMMARY = pd.DataFrame(columns=["K", "n_tracks", "share_static_fit_below_1px", "median_residual_px"])

if len(E6_TABLE):
    print("E6 / H9 (exploratory in v6.1, D34), Virtual KITTI 2 moving vehicles (parallel regime, measurable, identifiable, "
          "x_geo in the design range):")
    print(E6_TABLE.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
    print("\nExploratory: mean recall at 5% static FPR on moving vehicles, by regime:")
    print(E6_MASKS.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
else:
    print("E6 is not run: Virtual KITTI 2 is not available in this run (" + VK_STATUS["note"] + ").")
if len(PREVALENCE_ALL):
    print("\nShare of moving pixels per motion regime (two views):")
    print(PREVALENCE_ALL.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
if len(KVIEW_SUMMARY):
    print("\nVirtual KITTI 2 moving vehicles whose K-frame track a static point fits within 1 px:")
    print(KVIEW_SUMMARY.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
E6_TABLE.to_csv(os.path.join(RESULTS_DIR, "E6_vkitti_h9.csv"), index=False)
E6_MASKS.to_csv(os.path.join(RESULTS_DIR, "E6_vkitti_masks.csv"), index=False)
PREVALENCE_ALL.to_csv(os.path.join(RESULTS_DIR, "prevalence_two_view.csv"), index=False)
KVIEW_SUMMARY.to_csv(os.path.join(RESULTS_DIR, "prevalence_kview_vkitti.csv"), index=False)

In [ ]:
# Cell No: 062
# 6.8b Exploratory (v6.2, extended in v6.3 and v6.4): ground contact and the convoy case in driving.
#      Virtual KITTI 2 (exact ground truth), Argoverse 2 (real, LiDAR; automatic download) and, when
#      attached, KITTI tracking (real, LiDAR). Per-object geometry share g (Section 6.8b text), medians
#      with cluster-bootstrap intervals. v6.3: the cluster is the vehicle track (a vehicle seen in several
#      pairs counts once per resample), and the hidden-contact contrast is repeated on vehicles matched
#      for distance, image size and geometric lever. v6.4: the same contrast on real Argoverse 2 vehicles
#      (contact labels of Section 3.11b, D50), on all 64 logs. No verdicts: directions only.
def geo_share_obj(rho, rho_mono, x):
    a, b, c = np.exp(np.asarray(rho, float)), np.exp(np.asarray(rho_mono, float)), np.exp(np.asarray(x, float))
    with np.errstate(divide="ignore", invalid="ignore"):
        return (a - b) / (c - b)

def _cboot(df, fn, seed, B=None, cluster="cl"):
    # cluster bootstrap of a statistic fn(df) -> float; percentile 95% CI (2,000 resamples: these
    # medians are exploratory and the resampling is done in pandas)
    B = B or min(BOOT_N, 2000)
    est = fn(df)
    groups = [g for _, g in df.groupby(cluster)]
    if len(groups) < 5 or not np.isfinite(est):
        return est, np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = np.empty(B)
    for b in range(B):
        draws[b] = fn(pd.concat([groups[k] for k in rng.integers(0, len(groups), len(groups))], ignore_index=True))
    draws = draws[np.isfinite(draws)]
    if len(draws) < 0.9 * B:
        return est, np.nan, np.nan
    return est, float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

def _dsample(set_, model):
    d = MOV_RES[(MOV_RES["set"] == set_) & (MOV_RES["model"] == model)].copy() if len(MOV_RES) else pd.DataFrame()
    if not len(d):
        return d
    if set_ == "VK":
        ok = (d["gt_n_eroded"] >= PREREG["x6_min_eroded_px"])
    else:
        ok = d["measurable"].astype(bool)
    ok &= (d["regime"] == "parallel") & np.isfinite(d["rho"]) & np.isfinite(d["rho_mono"])
    d = d[ok].copy()
    # identifiable (parallax >= 5 px) is required for the contact test and the slope, not for the convoy
    # case: a vehicle that keeps pace with the camera has almost no parallax by construction, and its
    # geometric answer then lies far away, which is exactly what the test needs
    d["ident"] = d["gt_parallax_med"] >= PREREG["parallax_min_px"]
    d["g"] = geo_share_obj(d["rho"], d["rho_mono"], d["gt_x_geo"])
    d["lever"] = np.abs(d["gt_x_geo"] - d["rho_mono"])
    d["cl"] = d["seq"].astype(str) + "/" + d["track"].astype(str)        # vehicle track (v6.3 cluster)
    return d

def _pooled_sd(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.sqrt((np.nanvar(a, ddof=1) + np.nanvar(b, ddof=1)) / 2)) if len(a) > 1 and len(b) > 1 else np.nan

def _smd(a, b, sd):
    # standardized mean difference; before and after matching both use the pooled SD of the unmatched groups
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float((np.nanmean(a) - np.nanmean(b)) / sd) if len(a) and len(b) and np.isfinite(sd) and sd > 0 else np.nan

_MATCH_COV = ("logZ", "logA", "gt_x_geo")

def match_hidden_visible(dl, caliper):
    # every vehicle with hidden contact is paired with the nearest vehicle with visible contact (with
    # replacement) on log Z, log image area and x_geo in pooled-SD units, within the caliper on each
    dl = dl.assign(logZ=np.log(dl["gt_Z_true"].astype(float)), logA=np.log(dl["gt_area_px"].astype(float)))
    hid, vis = dl[dl["gt_contact"] == "hidden"], dl[dl["gt_contact"] == "visible"]
    sd = dl[list(_MATCH_COV)].std(ddof=1).replace(0, np.nan).to_numpy(float)
    zv = vis[list(_MATCH_COV)].to_numpy(float) / sd
    rows = []
    for (_, h), zh in zip(hid.iterrows(), hid[list(_MATCH_COV)].to_numpy(float) / sd):
        dist = np.abs(zv - zh)
        ok = np.all(dist <= caliper, axis=1) & np.all(np.isfinite(dist), axis=1)
        if not ok.any():
            continue
        k = int(np.argmin(np.where(ok, np.sqrt((dist ** 2).sum(1)), np.inf)))
        v = vis.iloc[k]
        rows.append(dict(cl=h["cl"], g_h=h["g"], g_v=v["g"], **{f"h_{c}": h[c] for c in _MATCH_COV}, **{f"v_{c}": v[c] for c in _MATCH_COV}))
    return pd.DataFrame(rows), hid, vis

_med_g = lambda q: float(np.median(q["g"])) if len(q) else np.nan
X6_CONTACT_ROWS, X6_CONTACT_TESTS, X6_MATCH_ROWS, X6_PACE_ROWS, REAL_BETA_ROWS = [], [], [], [], []
REAL_DRIVE = [(k, n, df) for k, n, df in (("AV2", "Argoverse 2", MOVERS_AV2), ("KT", "KITTI tracking", MOVERS_KT)) if len(df)]
_drive = [("VK", "Virtual KITTI 2", MODELS_OK["VK"])] + [(k, n, MODELS_OK[k]) for k, n, _ in REAL_DRIVE]
for set_, dname, models in _drive:
    for model in models:
        d = _dsample(set_, model)
        if not len(d):
            continue
        # (1) ground contact: Virtual KITTI 2 (wheels tested with instance masks, D37) and, v6.4, Argoverse 2 (wheels
        #     tested with geometry and LiDAR, D50); KITTI tracking has no contact labels
        if set_ in ("VK", "AV2") and "gt_contact" in d and d["gt_contact"].isin(["visible", "hidden"]).any():
            _sk = () if set_ == "VK" else (set_,)          # Virtual KITTI 2 keeps its v6.3 resampling seeds
            dl = d[d["ident"] & np.isfinite(d["g"]) & (d["lever"] >= PREREG["x6_min_lever"]) & d["gt_contact"].isin(["visible", "hidden"])]
            for grp in ("visible", "hidden"):
                q = dl[dl["gt_contact"] == grp]
                est, lo, hi = _cboot(q, _med_g, seed_for("x6c", *_sk, model, grp)) if len(q) >= 8 else (_med_g(q), np.nan, np.nan)
                X6_CONTACT_ROWS.append(dict(dataset=dname, model=model, contact=grp, n=len(q), n_tracks=int(q["cl"].nunique()),
                                            median_g=est, ci_lo=lo, ci_hi=hi))
            nh, nv = int((dl["gt_contact"] == "hidden").sum()), int((dl["gt_contact"] == "visible").sum())
            if nh >= 8 and nv >= 8:
                fn = lambda q: (float(np.median(q.loc[q["gt_contact"] == "hidden", "g"])) - float(np.median(q.loc[q["gt_contact"] == "visible", "g"]))
                                if (q["gt_contact"] == "hidden").any() and (q["gt_contact"] == "visible").any() else np.nan)
                est, lo, hi = _cboot(dl, fn, seed_for("x6c-diff", *_sk, model))
            else:
                est = lo = hi = np.nan
            X6_CONTACT_TESTS.append(dict(dataset=dname, model=model, n_hidden=nh, n_visible=nv,
                                         n_hidden_tracks=int(dl.loc[dl["gt_contact"] == "hidden", "cl"].nunique()),
                                         diff_hidden_minus_visible=est, ci_lo=lo, ci_hi=hi,
                                         direction=_dir_label(lo, hi) if np.isfinite(est) else "NOT TESTABLE"))
            # (1b, v6.3) the same contrast on matched vehicles: distance, image size and lever held fixed
            mt, hid, vis = match_hidden_visible(dl, PREREG["x6_match_caliper_sd"])
            rec = dict(dataset=dname, model=model, n_hidden=nh, n_matched=len(mt), n_matched_tracks=int(mt["cl"].nunique()) if len(mt) else 0)
            if len(mt) >= 8:
                fnm = lambda q: float(np.median(q["g_h"] - q["g_v"])) if len(q) else np.nan
                est, lo, hi = _cboot(mt, fnm, seed_for("x6c-match", *_sk, model))
            else:
                est, lo, hi = (float(np.median(mt["g_h"] - mt["g_v"])) if len(mt) else np.nan), np.nan, np.nan
            rec.update(median_paired_diff=est, ci_lo=lo, ci_hi=hi,
                       direction=_dir_label(lo, hi) if np.isfinite(lo) else "NOT TESTABLE")
            for c in _MATCH_COV:                               # covariate balance (standardized mean difference)
                _sd = _pooled_sd(hid[c], vis[c])
                rec[f"smd_before_{c}"] = _smd(hid[c], vis[c], _sd)
                rec[f"smd_after_{c}"] = _smd(mt[f"h_{c}"], mt[f"v_{c}"], _sd) if len(mt) else np.nan
            X6_MATCH_ROWS.append(rec)
        # (2) the convoy case: vehicles keeping pace with the camera
        lo_s, hi_s = PREREG["x6_pace_s"]
        p1 = d[(d["s"] >= lo_s) & (d["s"] < 1.0) & np.isfinite(d["g"]) & (d["lever"] >= PREREG["x6_min_lever"])]
        p2 = d[(d["s"] >= 1.0) & (d["s"] < hi_s)]
        est, lo, hi = _cboot(p1, _med_g, seed_for("x6p", set_, model)) if len(p1) >= 8 else (_med_g(p1), np.nan, np.nan)
        closer = float(np.mean(np.abs(p1["rho"] - p1["rho_mono"]) < np.abs(p1["rho"] - p1["gt_x_geo"]))) if len(p1) else np.nan
        X6_PACE_ROWS.append(dict(dataset=dname, set=set_, model=model, n_s_below_1=len(p1), n_tracks=int(p1["cl"].nunique()) if len(p1) else 0,
                                 median_g=est, ci_lo=lo, ci_hi=hi, share_closer_to_single_image=closer,
                                 median_x_geo=float(p1["gt_x_geo"].median()) if len(p1) else np.nan, n_s_at_least_1=len(p2),
                                 share_placed_farther_than_single_image=float(np.mean(p2["rho"] > p2["rho_mono"])) if len(p2) else np.nan))
        # (3) real driving: the slope beta_geo on parallel vehicles (as H9, exploratory)
        if set_ in ("KT", "AV2"):
            lo_x, hi_x = PREREG["h9_x_range"]
            q = d[d["ident"] & d["gt_x_geo"].between(lo_x, hi_x)]
            if len(q) >= 20 and q["cl"].nunique() >= 5:
                sl = slope_with_ci(q["gt_x_geo"], q["rho"], q["cl"], seed_for("real-beta", set_, model))
                REAL_BETA_ROWS.append(dict(dataset=dname, set=set_, model=model, n=len(q), n_tracks=int(q["cl"].nunique()),
                                           beta_geo=sl["beta"], ci_lo=sl["ci_lo"], ci_hi=sl["ci_hi"], r2=sl["r2"]))
            else:
                REAL_BETA_ROWS.append(dict(dataset=dname, set=set_, model=model, n=len(q), n_tracks=int(q["cl"].nunique()) if len(q) else 0,
                                           beta_geo=np.nan, ci_lo=np.nan, ci_hi=np.nan, r2=np.nan))
X6_CONTACT = pd.DataFrame(X6_CONTACT_ROWS)
X6_CONTACT_TEST = pd.DataFrame(X6_CONTACT_TESTS)
X6_MATCHED = pd.DataFrame(X6_MATCH_ROWS)
X6_PACE = pd.DataFrame(X6_PACE_ROWS)
REAL_BETA = pd.DataFrame(REAL_BETA_ROWS)
_prev = []
for set_, dname, df in REAL_DRIVE:
    _mk = df[df["measurable"].astype(bool)]
    if not len(_mk):
        continue
    _tau = {"AV2": AV2_TAU_PAR, "KT": KT_TAU_PAR}[set_]
    _metas = AV2_PAIRS_ALL if set_ == "AV2" else KT_PAIR_META
    _dts = [(m_.get("baseline"), m_.get("dt_s") or (0.1 * (m_["frames"][1] - m_["frames"][0]) if set_ == "KT" else None))
            for m_ in _metas]                                   # KITTI tracking: 10 Hz frames
    _spd = np.array([b_ / t_ for b_, t_ in _dts if b_ and t_], float)
    _prev.append(dict(dataset=f"{dname} (moving vehicles)", n_logs=int(_mk["seq"].nunique()), n_vehicles=len(_mk),
                      n_pairs=int(_mk["scene"].nunique()),
                      frac_parallel=float((_mk["regime"] == "parallel").mean()),
                      frac_intermediate=float((_mk["regime"] == "intermediate").mean()),
                      frac_perpendicular=float((_mk["regime"] == "perpendicular").mean()),
                      frac_parallel_lidar_px=float(_mk.loc[_mk["regime"] == "parallel", "gt_n_lidar_px"].sum()
                                                   / max(_mk["gt_n_lidar_px"].sum(), 1)),
                      frac_keeping_pace=float(_mk["s"].between(*PREREG["x6_pace_s"]).mean()),
                      parallel_threshold_px=_tau,
                      ego_speed_median_mps=float(np.median(_spd)) if len(_spd) else np.nan,
                      ego_speed_p90_mps=float(np.quantile(_spd, 0.9)) if len(_spd) else np.nan))
REAL_PREVALENCE = pd.DataFrame(_prev)
with pd.option_context("display.width", 250):
    for title, df in (("Ground contact (Virtual KITTI 2; Argoverse 2 from v6.4), median per-object geometry share g:", X6_CONTACT),
                      ("Hidden minus visible contact (clusters: vehicle tracks):", X6_CONTACT_TEST),
                      ("Hidden minus visible contact on matched vehicles (log Z, log area, x_geo within 0.5 SD):", X6_MATCHED),
                      ("The convoy case (vehicles keeping pace with the camera):", X6_PACE),
                      ("Real driving: share of moving vehicles per regime:", REAL_PREVALENCE),
                      ("Real driving: beta_geo on parallel vehicles (exploratory):", REAL_BETA)):
        if len(df):
            print("\n" + title)
            print(df.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
for _n, _df in (("X6_contact", X6_CONTACT), ("X6_contact_test", X6_CONTACT_TEST), ("X6_contact_matched", X6_MATCHED),
                ("X6_pace", X6_PACE), ("real_driving_prevalence", REAL_PREVALENCE), ("real_driving_beta", REAL_BETA)):
    if len(_df):
        _df.to_csv(os.path.join(RESULTS_DIR, f"{_n}.csv"), index=False)
for _k, _n, _st in (("AV2", "Argoverse 2", AV2_STATUS), ("KT", "KITTI tracking", KT_STATUS)):
    if not _st["available"]:
        print(f"\n{_n} analyses are not part of this run: {_st['note']}")

In [ ]:
# Cell No: 063
# 6.9 Synthesis (descriptive, v6.4, D51): the geometry share by ground support in every testbed, each testbed's
#     contrast between unsupported or hidden-contact objects and grounded ones, and the depth overestimate a slope
#     implies for an object keeping pace with the camera (s = 0.85). Only interpretable model x testbed results enter.
X_IMPLIED = float(-np.log(1.0 - PREREG["implied_factor_s"]))
_PAPER_NAME = {"pi3": "π³", "VGGT-1B": "VGGT"}                 # model names as printed in the paper (also Section 10)
SUPPORT_LEVEL_ROWS, SUPPORT_CONTRAST_ROWS = [], []

def _level(model, testbed, condition, support, stat, est, lo, hi, n, n_clusters, source):
    r = dict(model=model, testbed=testbed, condition=condition, support=support, statistic=stat, estimate=est, ci_lo=lo,
             ci_hi=hi, n=n, n_clusters=n_clusters, source=source, implied_factor=np.nan, implied_lo=np.nan, implied_hi=np.nan)
    if stat == "beta_s":                    # exp(beta x) is increasing in beta, so the interval maps through it
        r.update(implied_factor=float(np.exp(est * X_IMPLIED)) if np.isfinite(est) else np.nan,
                 implied_lo=float(np.exp(lo * X_IMPLIED)) if np.isfinite(lo) else np.nan,
                 implied_hi=float(np.exp(hi * X_IMPLIED)) if np.isfinite(hi) else np.nan)
    SUPPORT_LEVEL_ROWS.append(r)

def _contrast(model, testbed, contrast, stat, est, lo, hi, n, source):
    SUPPORT_CONTRAST_ROWS.append(dict(model=model, testbed=testbed, contrast=contrast, statistic=stat, estimate=est, ci_lo=lo,
                                      ci_hi=hi, n=n, direction=_dir_label(lo, hi) if np.isfinite(est) else "NOT TESTABLE",
                                      source=source))

# CONVOY-3D: E1 movers by support (cars and crates stand on the street, floating crates do not)
for model in MODELS_OK["MAIN"]:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN")]
    dp = d[e1_mask(d)]
    for lab, sel, sup in (("cars and crates", dp["cls"].isin(["car", "crate"]), "grounded"),
                          ("floating crates", dp["cls"] == "crate_floating", "unsupported")):
        r = slope_with_ci(dp.loc[sel, "x_par"], dp.loc[sel, "rho"], dp.loc[sel, "scene"], seed_for("synth-c3d", model, lab))
        _level(model, "CONVOY-3D", lab, sup, "beta_s", r["beta"], r["ci_lo"], r["ci_hi"], r["n"], r["n_clusters"], "E1 movers")
    _, _, est, lo, hi = _group_slope_diff(dp["x_par"], dp["rho"], dp["cls"] == "crate_floating", dp["scene"],
                                          seed_for("synth-c3d-diff", model))
    _contrast(model, "CONVOY-3D", "floating crates - cars and crates", "beta_s", est, lo, hi, len(dp), "new in v6.4 (unpaired)")
# CONVOY-REAL: every inserted mover is a floating crate (D52)
for r in REAL_E1.itertuples():
    if r.verdict != "control" and r.model in MODELS_OK["REAL"]:
        _level(r.model, "CONVOY-REAL", "floating crates (every mover)", "unsupported", "beta_s", r.beta_s, r.ci_lo, r.ci_hi,
               r.n, r.n_pairs, "T8")
# E5: the same movers in five conditions
_E5_SUPPORT = {"G": "grounded", "NS": "grounded", "CH": "contact hidden", "F": "unsupported", "FH": "unsupported"}
for r in (E5_BETA[E5_BETA["model"].isin(MODELS_OK["ABL"])].itertuples() if len(E5_BETA) else []):
    _level(r.model, "E5 ablation", ABLATION_DESIGN["labels"][r.condition], _E5_SUPPORT[r.condition], "beta_s", r.beta,
           r.ci_lo, r.ci_hi, r.n, r.n_scenes, "T9a")
for r in (E5_TESTS[E5_TESTS["test"].isin(["H5", "H6"]) & E5_TESTS["model"].isin(MODELS_OK["ABL"])].itertuples()
          if len(E5_TESTS) else []):
    _contrast(r.model, "E5 ablation", {"H5": "floating - no shadow (H5)", "H6": "contact hidden - grounded (H6)"}[r.test],
              "beta_s", r.estimate, r.ci_lo, r.ci_hi, r.n, "T9b")
# E5-REAL: the same crates on real images
for r in (RABL_BETA[RABL_BETA["model"].isin(MODELS_OK["RABL"])].itertuples() if len(RABL_BETA) else []):
    _level(r.model, "E5-REAL", RABL_DESIGN["labels"][r.condition], {"G": "grounded", "F": "unsupported"}[r.condition], "beta_s",
           r.beta, r.ci_lo, r.ci_hi, r.n, r.n_pairs, "T11a")
for r in (RABL_TESTS[(RABL_TESTS["test"] == "H11a") & RABL_TESTS["model"].isin(MODELS_OK["RABL"])].itertuples()
          if len(RABL_TESTS) else []):
    _contrast(r.model, "E5-REAL", "lifted - grounded (H11a)", "beta_s", r.estimate, r.ci_lo, r.ci_hi, r.n, "T11b")
# driving: per-object share g of parallel vehicles by wheel contact
_DRIVE_TB = {"Virtual KITTI 2": "VK", "Argoverse 2": "AV2"}
_ok_drive = lambda ds, m: m in MODELS_OK.get(_DRIVE_TB.get(ds, ""), [])
for r in (X6_CONTACT.itertuples() if len(X6_CONTACT) else []):
    if _ok_drive(r.dataset, r.model):
        _level(r.model, r.dataset, f"contact {r.contact}", "grounded" if r.contact == "visible" else "contact hidden", "median_g",
               r.median_g, r.ci_lo, r.ci_hi, r.n, r.n_tracks, "T15a")
for r in (X6_CONTACT_TEST.itertuples() if len(X6_CONTACT_TEST) else []):
    if _ok_drive(r.dataset, r.model):
        _contrast(r.model, r.dataset, "contact hidden - visible", "median_g", r.diff_hidden_minus_visible, r.ci_lo, r.ci_hi,
                  r.n_hidden + r.n_visible, "T15a")
for r in (X6_MATCHED.itertuples() if len(X6_MATCHED) else []):
    if _ok_drive(r.dataset, r.model):
        _contrast(r.model, r.dataset, "contact hidden - visible, matched", "median_g", r.median_paired_diff, r.ci_lo, r.ci_hi,
                  r.n_matched, "T15a")

_lcols = ["model", "testbed", "condition", "support", "statistic", "estimate", "ci_lo", "ci_hi", "n", "n_clusters", "source",
          "implied_factor", "implied_lo", "implied_hi"]
SUPPORT_LEVELS = pd.DataFrame(SUPPORT_LEVEL_ROWS, columns=_lcols)
SUPPORT_CONTRASTS = pd.DataFrame(SUPPORT_CONTRAST_ROWS, columns=["model", "testbed", "contrast", "statistic", "estimate", "ci_lo",
                                                                 "ci_hi", "n", "direction", "source"])
_ci3 = lambda e, l, h: "–" if not np.isfinite(e) else (f"{e:.2f}" if not (np.isfinite(l) and np.isfinite(h)) else f"{e:.2f} [{l:.2f}, {h:.2f}]")
if len(SUPPORT_LEVELS):
    _sl = SUPPORT_LEVELS.assign(row=SUPPORT_LEVELS["testbed"] + ": " + SUPPORT_LEVELS["condition"] + " (" + SUPPORT_LEVELS["support"] + ")",
                                value=[_ci3(*v) for v in SUPPORT_LEVELS[["estimate", "ci_lo", "ci_hi"]].to_numpy(float)])
    _order = list(dict.fromkeys(_sl["row"]))
    with pd.option_context("display.width", 250, "display.max_colwidth", 60):
        print("Geometry share by ground support (beta_s on CONVOY-3D, CONVOY-REAL, E5 and E5-REAL; median per-object g in driving):")
        print(_sl.pivot_table(index="row", columns="model", values="value", aggfunc="first").reindex(_order)
              .rename(columns=lambda c: _PAPER_NAME.get(c, c)).to_string(na_rep="–"))
        _im = _sl[_sl["statistic"] == "beta_s"].assign(
            factor=[_ci3(*v) for v in _sl.loc[_sl["statistic"] == "beta_s", ["implied_factor", "implied_lo", "implied_hi"]].to_numpy(float)])
        print(f"\nDepth overestimate implied by each slope for an object keeping pace with the camera (s = {PREREG['implied_factor_s']}, "
              f"exp(beta x) with x = {X_IMPLIED:.2f}; 1 = correct depth, {np.exp(X_IMPLIED):.1f} = the geometric answer):")
        print(_im.pivot_table(index="row", columns="model", values="factor", aggfunc="first").reindex([o for o in _order if o in set(_im["row"])])
              .rename(columns=lambda c: _PAPER_NAME.get(c, c)).to_string(na_rep="–"))
if len(SUPPORT_CONTRASTS):
    _sc = SUPPORT_CONTRASTS.assign(row=SUPPORT_CONTRASTS["testbed"] + ": " + SUPPORT_CONTRASTS["contrast"],
                                   value=[f"{_ci3(e, l, h)} {d_}" if np.isfinite(e) else "–" for e, l, h, d_ in
                                          SUPPORT_CONTRASTS[["estimate", "ci_lo", "ci_hi", "direction"]].itertuples(index=False)])
    with pd.option_context("display.width", 250, "display.max_colwidth", 60):
        print("\nUnsupported or hidden-contact minus grounded, within each testbed (95% cluster-bootstrap intervals; direction from "
              "the interval):")
        print(_sc.pivot_table(index="row", columns="model", values="value", aggfunc="first").reindex(list(dict.fromkeys(_sc["row"])))
              .rename(columns=lambda c: _PAPER_NAME.get(c, c)).to_string(na_rep="–"))
    _t = SUPPORT_CONTRASTS[SUPPORT_CONTRASTS["direction"] != "NOT TESTABLE"]
    SUPPORT_COUNT = _t.groupby(["testbed", "contrast"], sort=False)["direction"].agg(
        higher=lambda v: int((v == "higher").sum()), no_clear_change=lambda v: int((v == "no clear change").sum()),
        lower=lambda v: int((v == "lower").sum())).reset_index()
    print("\nModels per direction, for each contrast:")
    print(SUPPORT_COUNT.to_string(index=False))
else:
    SUPPORT_COUNT = pd.DataFrame()
SUPPORT_LEVELS.to_csv(os.path.join(RESULTS_DIR, "support_levels.csv"), index=False)
SUPPORT_CONTRASTS.to_csv(os.path.join(RESULTS_DIR, "support_contrasts.csv"), index=False)

In [ ]:
# Cell No: 064
# 6.10 Designed experiments (v6.5, D55): H12 baseline sweep and H13 contact dose-response (registered), with the shape of
#      each curve, the sweep by support and M2 recall by level (exploratory). Paired by mover; cluster = base scene.
_DSG_VALS = ["rho", "rho_mono", "gt_n_eroded", "gt_parallax_med", "gt_visible_frac", "measurable", "contact_px0", "contact_px1",
             "has_wall", "recall_M2_exact"]

def designed_wide(res, movers, model):
    # one row per base mover, one column per (quantity, level)
    d = res[res["model"] == model].copy()
    if not len(d):
        return pd.DataFrame()
    for c in ("has_wall", "measurable"):
        d[c] = d[c].astype(float)
    w = d.pivot_table(index=["uid", "scene"], columns="condition", values=[c for c in _DSG_VALS if c in d], aggfunc="first")
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    gt = movers.drop_duplicates("uid")[["uid", "cls", "s", "x_par"]]
    return w.reset_index().merge(gt, on="uid", how="left")

def group_multi_slopes(x, ys, group, clusters, seed):
    """Slopes of several paired responses on the same x in two groups of movers that share clusters (joint cluster
    bootstrap). Returns slopes (2, k) [group True, group False], draws (B, 2, k) and the group sizes."""
    x = np.asarray(x, float); Y = np.column_stack([np.asarray(y, float) for y in ys]); gp = np.asarray(group, bool)
    cl = np.asarray(clusters)
    ok = np.isfinite(x) & np.isfinite(Y).all(1)
    x, Y, gp, cl = x[ok], Y[ok], gp[ok], cl[ok]
    k = Y.shape[1]
    if min(gp.sum(), (~gp).sum()) < 8 or len(np.unique(cl)) < 5:
        return np.full((2, k), np.nan), np.zeros((0, 2, k)), (int(gp.sum()), int((~gp).sum()))
    cols = []
    for g in (gp, ~gp):
        gf = g.astype(float)
        cols += [gf, gf * x, gf * x * x] + [gf * Y[:, i] for i in range(k)] + [gf * x * Y[:, i] for i in range(k)]
    T, P = ClusterBootstrap(cl, seed=seed).linear(cols)
    L = 3 + 2 * k
    def _sl(t, o):
        den = t[:, o] * t[:, o + 2] - t[:, o + 1] ** 2
        with np.errstate(invalid="ignore", divide="ignore"):
            return (t[:, o:o + 1] * t[:, o + 3 + k:o + 3 + 2 * k] - t[:, o + 1:o + 2] * t[:, o + 3:o + 3 + k]) / den[:, None]
    return (np.stack([_sl(P[None], 0)[0], _sl(P[None], L)[0]]), np.stack([_sl(T, 0), _sl(T, L)], axis=1),
            (int(gp.sum()), int((~gp).sum())))

def _mean_ci(df, cols, seed):
    # mean of each column with a cluster-bootstrap interval (scenes), and the draws (linear statistic)
    bs = ClusterBootstrap(df["scene"].values, seed=seed)
    T, P = bs.linear([np.ones(len(df))] + [df[c].to_numpy(float) for c in cols])
    with np.errstate(invalid="ignore", divide="ignore"):
        dr = T[:, 1:] / T[:, :1]
    return P[1:] / P[0], dr

def _p_one_sided(draws):
    draws = np.asarray(draws)[np.isfinite(draws)]
    return float((1 + np.sum(draws <= 0)) / (len(draws) + 1)) if len(draws) else np.nan

_ok_dose = lambda n, nc: n >= PREREG["dose_min_movers"] and nc >= PREREG["dose_min_scenes"]
BSW_BETA_ROWS, BSW_TEST_ROWS, BSW_RECALL_ROWS, WDR_BETA_ROWS, WDR_TEST_ROWS, WDR_RECALL_ROWS = [], [], [], [], [], []
BSW_WIDE, WDR_WIDE = {}, {}
_hi12, _lo12 = PREREG["h12_contrast"]
_hi13, _lo13 = PREREG["h13_contrast"]
for model in EVALUATED_MODELS:
    # ---------------------------------------------------------------- baseline sweep (H12)
    w = designed_wide(BSW_RES, MOVERS_BSW, model)
    if len(w):
        BSW_WIDE[model] = w
        gate_ok = GATE_PASS.get((model, "BSW"), False)
        meas = np.all([w.get(f"measurable_{lv}", 0) == 1 for lv in BSW_LEVELS], axis=0)
        fin = np.all([np.isfinite(w.get(f"rho_{lv}", np.nan)) for lv in BSW_LEVELS], axis=0)
        w["h12_sample"] = meas & fin & (w.get("gt_parallax_med_B1", 0) >= PREREG["parallax_min_px"])
        s_ = w[w["h12_sample"]]
        b, dr, n, nc = multi_slopes(s_["x_par"], [s_[f"rho_{lv}"] for lv in BSW_LEVELS], s_["scene"], seed_for("bsw", model))
        for k, lv in enumerate(BSW_LEVELS):
            lo, hi = _ci(dr[:, k]) if len(dr) else (np.nan, np.nan)
            BSW_BETA_ROWS.append(dict(model=model, support="all", level=lv, factor=SWEEP_DESIGN["levels"][lv], n=n, n_scenes=nc,
                                      beta=b[k], ci_lo=lo, ci_hi=hi,
                                      median_parallax=float(s_[f"gt_parallax_med_{lv}"].median()) if len(s_) else np.nan,
                                      median_rho_mono=float(s_[f"rho_mono_{lv}"].median()) if len(s_) else np.nan))
        wts = [1.0 if lv == _hi12 else -1.0 if lv == _lo12 else 0.0 for lv in BSW_LEVELS]
        est, lo, hi = contrast_ci(b, dr, wts)
        p = _p_one_sided(dr @ np.asarray(wts)) if len(dr) else np.nan
        registered = model in PREREG["h12_models"]
        testable = _ok_dose(n, nc)
        v = ("NOT INTERPRETABLE (gate)" if not gate_ok else "NOT TESTABLE" if not testable
             else verdict_positive(lo, hi) if registered else "NO PREDICTION")
        BSW_TEST_ROWS.append(dict(model=model, test="H12", contrast=f"beta({SWEEP_DESIGN['labels'][_hi12]}) - beta({SWEEP_DESIGN['labels'][_lo12]})",
                                  registered=registered, n=n, n_scenes=nc, estimate=est, ci_lo=lo, ci_hi=hi, p_boot=p, verdict=v))
        # exploratory: the same rise for grounded (cars, crates) and floating movers, and its difference
        pt, drg, (n_f, n_g) = group_multi_slopes(s_["x_par"], [s_[f"rho_{lv}"] for lv in BSW_LEVELS], s_["cls"] == "crate_floating",
                                                 s_["scene"], seed_for("bsw-support", model))
        for gi, sup in enumerate(("floating", "grounded")):
            for k, lv in enumerate(BSW_LEVELS):
                lo_, hi_ = _ci(drg[:, gi, k]) if len(drg) else (np.nan, np.nan)
                BSW_BETA_ROWS.append(dict(model=model, support=sup, level=lv, factor=SWEEP_DESIGN["levels"][lv],
                                          n=(n_f, n_g)[gi], n_scenes=nc, beta=pt[gi, k], ci_lo=lo_, ci_hi=hi_))
        rise = pt @ np.asarray(wts)
        rdr = drg @ np.asarray(wts) if len(drg) else np.zeros((0, 2))
        for gi, sup in enumerate(("floating", "grounded")):
            lo_, hi_ = _ci(rdr[:, gi]) if len(rdr) else (np.nan, np.nan)
            BSW_TEST_ROWS.append(dict(model=model, test=f"H12-{sup}", contrast=f"rise x0.5 -> x3, {sup} movers", registered=False,
                                      n=(n_f, n_g)[gi], n_scenes=nc, estimate=rise[gi], ci_lo=lo_, ci_hi=hi_, p_boot=np.nan,
                                      verdict="EXPLORATORY" if gate_ok else "NOT INTERPRETABLE (gate)"))
        dd = rdr[:, 0] - rdr[:, 1] if len(rdr) else np.zeros(0)
        lo_, hi_ = _ci(dd) if len(dd) else (np.nan, np.nan)
        BSW_TEST_ROWS.append(dict(model=model, test="H12-interaction", contrast="rise of floating minus rise of grounded movers",
                                  registered=False, n=n_f + n_g, n_scenes=nc, estimate=float(rise[0] - rise[1]), ci_lo=lo_, ci_hi=hi_,
                                  p_boot=np.nan, verdict="EXPLORATORY" if gate_ok else "NOT INTERPRETABLE (gate)"))
        # exploratory: M2 recall at every baseline on the same movers
        rc = s_.dropna(subset=[f"recall_M2_exact_{lv}" for lv in BSW_LEVELS])
        if _ok_dose(len(rc), rc["scene"].nunique()):
            mu, mdr = _mean_ci(rc, [f"recall_M2_exact_{lv}" for lv in BSW_LEVELS], seed_for("bsw-m2", model))
            for k, lv in enumerate(BSW_LEVELS):
                lo_, hi_ = _ci(mdr[:, k])
                BSW_RECALL_ROWS.append(dict(model=model, level=lv, factor=SWEEP_DESIGN["levels"][lv], n=len(rc), recall_M2=mu[k],
                                            ci_lo=lo_, ci_hi=hi_))
    # ---------------------------------------------------------------- contact dose-response (H13)
    w = designed_wide(WDR_RES, MOVERS_WDR, model)
    if len(w):
        WDR_WIDE[model] = w
        gate_ok = GATE_PASS.get((model, "WDR"), False)
        for lv in WDR_LEVELS:
            w[f"ok_{lv}"] = ((w.get(f"gt_n_eroded_{lv}", 0) >= PREREG["min_object_pixels"])
                             & (w.get(f"gt_parallax_med_{lv}", 0) >= PREREG["parallax_min_px"]) & np.isfinite(w.get(f"rho_{lv}", np.nan)))
        cmin = PREREG["h13_contact_px_min"]
        w["contact_G"] = (w.get("contact_px0_G", 0) >= cmin) & (w.get("contact_px1_G", 0) >= cmin) & (w.get("gt_visible_frac_G", 0) >= 0.85)
        for lv in WDR_LEVELS[1:]:
            w[f"hidden_{lv}"] = ((w.get(f"has_wall_{lv}", 0) == 1) & (w.get(f"contact_px0_{lv}", 1) == 0)
                                 & (w.get(f"contact_px1_{lv}", 1) == 0))
        s13 = w[w["contact_G"] & w[f"ok_{_lo13}"] & w[f"ok_{_hi13}"] & w[f"hidden_{_hi13}"]]
        b, dr, n, nc = multi_slopes(s13["x_par"], [s13[f"rho_{_hi13}"], s13[f"rho_{_lo13}"]], s13["scene"], seed_for("wdr-h13", model))
        est, lo, hi = contrast_ci(b, dr, [1, -1])
        p = _p_one_sided(dr @ np.array([1.0, -1.0])) if len(dr) else np.nan
        v = ("NOT INTERPRETABLE (gate)" if not gate_ok else "NOT TESTABLE" if not _ok_dose(n, nc) else verdict_positive(lo, hi))
        WDR_TEST_ROWS.append(dict(model=model, test="H13", contrast=f"beta({WALL_DOSE_DESIGN['labels'][_hi13]} wall) - beta(no wall)",
                                  registered=True, n=n, n_scenes=nc, estimate=est, ci_lo=lo, ci_hi=hi, p_boot=p, verdict=v,
                                  beta_hi=b[0], beta_lo=b[1]))
        # exploratory: the whole curve on movers valid at every height whose contact every wall hides
        sa = w[w["contact_G"] & np.all([w[f"ok_{lv}"] for lv in WDR_LEVELS], axis=0)
               & np.all([w[f"hidden_{lv}"] for lv in WDR_LEVELS[1:]], axis=0)]
        b, dr, n, nc = multi_slopes(sa["x_par"], [sa[f"rho_{lv}"] for lv in WDR_LEVELS], sa["scene"], seed_for("wdr-all", model))
        for k, lv in enumerate(WDR_LEVELS):
            lo_, hi_ = _ci(dr[:, k]) if len(dr) else (np.nan, np.nan)
            WDR_BETA_ROWS.append(dict(model=model, level=lv, cover_m=WALL_DOSE_DESIGN["levels"][lv], n=n, n_scenes=nc, beta=b[k],
                                      ci_lo=lo_, ci_hi=hi_, median_rho_mono=float(sa[f"rho_mono_{lv}"].median()) if len(sa) else np.nan,
                                      median_visible_px=float(sa[f"gt_n_eroded_{lv}"].median()) if len(sa) else np.nan,
                                      median_hidden_share=float((1.0 - sa[f"gt_visible_frac_{lv}"]).median()) if len(sa) else np.nan))
        i15, i70, i0 = WDR_LEVELS.index("W15"), WDR_LEVELS.index("W70"), WDR_LEVELS.index("G")
        est, lo_, hi_ = contrast_ci(b, dr, [1.0 if k == i70 else -1.0 if k == i15 else 0.0 for k in range(len(WDR_LEVELS))])
        WDR_TEST_ROWS.append(dict(model=model, test="dose 70-15", contrast="beta(70 cm) - beta(15 cm)", registered=False, n=n, n_scenes=nc,
                                  estimate=est, ci_lo=lo_, ci_hi=hi_, p_boot=np.nan,
                                  verdict="EXPLORATORY" if gate_ok else "NOT INTERPRETABLE (gate)"))
        e70, lo70, hi70 = contrast_ci(b, dr, [1.0 if k == i70 else -1.0 if k == i0 else 0.0 for k in range(len(WDR_LEVELS))])
        WDR_TEST_ROWS.append(dict(model=model, test="dose 70-0", contrast="beta(70 cm) - beta(no wall)", registered=False, n=n,
                                  n_scenes=nc, estimate=e70, ci_lo=lo70, ci_hi=hi70, p_boot=np.nan,
                                  verdict="EXPLORATORY" if gate_ok else "NOT INTERPRETABLE (gate)"))
        # the share of the 70 cm effect reached at 15 cm is a ratio: it is reported only where the 70 cm effect itself is
        # clearly positive (its interval excludes 0); otherwise the denominator can be ~0 and the ratio means nothing
        share, lo_, hi_ = np.nan, np.nan, np.nan
        if np.isfinite(lo70) and lo70 > 0:
            with np.errstate(invalid="ignore", divide="ignore"):
                share = float((b[i15] - b[i0]) / (b[i70] - b[i0]))
                sdr = (dr[:, i15] - dr[:, i0]) / (dr[:, i70] - dr[:, i0]) if len(dr) else np.zeros(0)
            lo_, hi_ = _ci(sdr[np.isfinite(sdr)]) if np.isfinite(sdr).sum() else (np.nan, np.nan)
        WDR_TEST_ROWS.append(dict(model=model, test="dose share", contrast="(beta 15 cm - beta no wall) / (beta 70 cm - beta no wall)",
                                  registered=False, n=n, n_scenes=nc, estimate=share, ci_lo=lo_, ci_hi=hi_, p_boot=np.nan,
                                  verdict=("NOT INTERPRETABLE (gate)" if not gate_ok else "EXPLORATORY" if np.isfinite(share)
                                           else "NOT ESTIMABLE (no clear 70 cm effect)")))
        rc = sa.dropna(subset=[f"recall_M2_exact_{lv}" for lv in WDR_LEVELS])
        if _ok_dose(len(rc), rc["scene"].nunique()):
            mu, mdr = _mean_ci(rc, [f"recall_M2_exact_{lv}" for lv in WDR_LEVELS], seed_for("wdr-m2", model))
            for k, lv in enumerate(WDR_LEVELS):
                lo_, hi_ = _ci(mdr[:, k])
                WDR_RECALL_ROWS.append(dict(model=model, level=lv, cover_m=WALL_DOSE_DESIGN["levels"][lv], n=len(rc), recall_M2=mu[k],
                                            ci_lo=lo_, ci_hi=hi_))
BSW_BETA, BSW_TESTS, BSW_RECALL = pd.DataFrame(BSW_BETA_ROWS), pd.DataFrame(BSW_TEST_ROWS), pd.DataFrame(BSW_RECALL_ROWS)
WDR_BETA, WDR_TESTS, WDR_RECALL = pd.DataFrame(WDR_BETA_ROWS), pd.DataFrame(WDR_TEST_ROWS), pd.DataFrame(WDR_RECALL_ROWS)
for _df, _t, _models in ((BSW_TESTS, "H12", PREREG["h12_models"]), (WDR_TESTS, "H13", None)):
    if len(_df):
        _m = (_df["test"] == _t) & (_df["model"].isin(_models) if _models else True) & _df["verdict"].isin(
            ["SUPPORTED", "NOT SUPPORTED", "CONTRADICTED"])
        _df["p_holm"] = np.nan
        _df.loc[_m, "p_holm"] = holm(_df.loc[_m, "p_boot"].values)

_f3 = lambda v: f"{v:.3f}"
with pd.option_context("display.width", 250, "display.max_colwidth", 60):
    if len(BSW_BETA):
        print("Baseline sweep: beta_s at each baseline (common sample: measurable at every baseline, identifiable at x1):")
        print(BSW_BETA[BSW_BETA["support"] == "all"].pivot_table(index="model", columns="level", values="beta")[BSW_LEVELS]
              .rename(columns=SWEEP_DESIGN["labels"]).to_string(float_format=_f3))
        _pp = BSW_BETA[BSW_BETA["support"] == "all"].groupby("level")["median_parallax"].median().reindex(BSW_LEVELS)
        print("median parallax of the sample (px): " + ", ".join(f"{SWEEP_DESIGN['labels'][k]} {v:.1f}" for k, v in _pp.items()))
        print("\nH12 (registered for DUSt3R and MASt3R) and the exploratory split by support:")
        print(BSW_TESTS[["model", "test", "n", "n_scenes", "estimate", "ci_lo", "ci_hi", "p_holm", "verdict"]]
              .to_string(index=False, na_rep="–", float_format=_f3))
    if len(BSW_RECALL):
        print("\nM2 recall (exact correspondences) at each baseline, same movers:")
        print(BSW_RECALL.pivot_table(index="model", columns="level", values="recall_M2")[BSW_LEVELS]
              .rename(columns=SWEEP_DESIGN["labels"]).to_string(float_format=_f3))
    if len(WDR_BETA):
        print("\nContact dose-response: beta_s at each wall height (common sample of all four heights):")
        print(WDR_BETA.pivot_table(index="model", columns="level", values="beta")[WDR_LEVELS]
              .rename(columns=WALL_DOSE_DESIGN["labels"]).to_string(float_format=_f3))
        _hs = WDR_BETA.groupby("level")["median_hidden_share"].median().reindex(WDR_LEVELS)
        print("median share of the mover hidden in view 1 (same sample): "
              + ", ".join(f"{WALL_DOSE_DESIGN['labels'][k]} {v:.2f}" for k, v in _hs.items()))
    if len(WDR_TESTS):
        print("\nH13 (registered, every model) and the shape of the curve (exploratory):")
        print(WDR_TESTS[["model", "test", "n", "n_scenes", "estimate", "ci_lo", "ci_hi", "p_holm", "verdict"]]
              .to_string(index=False, na_rep="–", float_format=_f3))
    if len(WDR_RECALL):
        print("\nM2 recall at each wall height, same movers:")
        print(WDR_RECALL.pivot_table(index="model", columns="level", values="recall_M2")[WDR_LEVELS]
              .rename(columns=WALL_DOSE_DESIGN["labels"]).to_string(float_format=_f3))
for _n, _df in (("BSW_beta", BSW_BETA), ("BSW_tests", BSW_TESTS), ("BSW_recall", BSW_RECALL), ("WDR_beta", WDR_BETA),
                ("WDR_tests", WDR_TESTS), ("WDR_recall", WDR_RECALL)):
    _df.to_csv(os.path.join(RESULTS_DIR, f"{_n}.csv"), index=False)

In [ ]:
# Cell No: 065
# 6.11 Two channels: the cue model (v6.5, exploratory, D57 and D59). Per model: fit on CONVOY-3D and E5, cross-validation
#      against the constant-weight and support-free models, out-of-sample slope predictions for every other testbed, the
#      hump, MonST3R against DUSt3R, the mask trade-off, and CONVOY-REAL by parallax.
_SUPPORTS = ("contact in view", "contact hidden", "unsupported")
_E5_SUP = {"G": "contact in view", "NS": "contact in view", "CH": "contact hidden", "F": "unsupported", "FH": "unsupported"}


def _phi(d, kappa):
    d = np.asarray(d, float)
    return np.ones_like(d) if kappa <= 0 else d * d / (d * d + kappa * kappa)


def _frame(x, rho, rho_mono, d, support, cl, **extra):
    df = pd.DataFrame(dict(x=np.asarray(x, float), rho=np.asarray(rho, float), rho_mono=np.asarray(rho_mono, float),
                           d=np.asarray(d, float), support=np.asarray(support, object), cl=np.asarray(cl, object), **extra))
    df = df[np.isfinite(df[["x", "rho", "rho_mono", "d"]]).all(1)].copy()
    df["y"], df["z"] = df["rho"] - df["rho_mono"], df["x"] - df["rho_mono"]
    return df


def cue_fit(df, per_support=True, kappas=None):
    # least squares of y = c_support * phi(d; kappa) * z; kappa on the grid, c in closed form for each kappa
    kappas = CUE_DESIGN["kappa_grid_px"] if kappas is None else kappas
    y, z, d = df["y"].to_numpy(float), df["z"].to_numpy(float), df["d"].to_numpy(float)
    groups = df["support"].to_numpy(object) if per_support else np.full(len(df), "all", object)
    best = None
    for k in kappas:
        a = _phi(d, k) * z
        c, sse = {}, 0.0
        for g in np.unique(groups):
            m = groups == g
            den = float(np.sum(a[m] ** 2))
            c[g] = float(np.sum(a[m] * y[m]) / den) if den > 0 else 0.0
            sse += float(np.sum((y[m] - c[g] * a[m]) ** 2))
        if best is None or sse < best["sse"]:
            best = dict(kappa=float(k), c=c, sse=sse, n=len(df), per_support=per_support)
    return best


def cue_predict(fit, df):
    c = df["support"].map(fit["c"]).to_numpy(float) if fit["per_support"] else np.full(len(df), fit["c"]["all"])
    return df["rho_mono"].to_numpy(float) + c * _phi(df["d"], fit["kappa"]) * df["z"].to_numpy(float)


def cue_training(model):
    # CONVOY-3D parallel movers (measurable; parallax not required) and the five E5 conditions (the E5 sample rules)
    parts = []
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN")]
    d = d[(d["motion_type"] == "parallel") & (d["s"] < 1) & d["measurable"].astype(bool)]
    parts.append(_frame(d["x_par"], d["rho"], d["rho_mono"], d["gt_parallax_med"],
                        np.where(d["cls"] == "crate_floating", "unsupported", "contact in view"), "MAIN-" + d["scene"].astype(str),
                        source="CONVOY-3D", uid=d["uid"].to_numpy()))
    w = E5_WIDE.get(model)
    if w is not None and len(w):
        for c in E5_CONDS:
            m = w[f"ok_{c}"] & (w["vis_ok"] if c in ("G", "NS", "F") else w["walled"])
            q = w[m]
            parts.append(_frame(q["x_par"], q[f"rho_{c}"], q[f"rho_mono_{c}"], q[f"gt_parallax_med_{c}"], [_E5_SUP[c]] * len(q),
                                "ABL-" + q["scene"].astype(str), source=f"E5 {c}", uid=q["uid"].to_numpy()))
    return pd.concat(parts, ignore_index=True)


def cue_tests(model):
    # samples on which the fitted model predicts the slope (each with its own testbed rules); x is x = -log(1-s) where s
    # was designed and x_geo in driving
    out = []
    if model in MODELS_OK["BSW"] and model in BSW_WIDE:
        w = BSW_WIDE[model]; q = w[w["h12_sample"]]
        sup = np.where(q["cls"] == "crate_floating", "unsupported", "contact in view")
        for lv in BSW_LEVELS:
            for lab, m in (("all", np.ones(len(q), bool)), ("grounded", sup == "contact in view"), ("floating", sup == "unsupported")):
                out.append((f"baseline sweep {SWEEP_DESIGN['labels'][lv]}", lab, "BSW", lv,
                            _frame(q["x_par"][m], q[f"rho_{lv}"][m], q[f"rho_mono_{lv}"][m], q[f"gt_parallax_med_{lv}"][m], sup[m],
                                   q["scene"][m].astype(str))))
    if model in MODELS_OK["WDR"] and model in WDR_WIDE:
        w = WDR_WIDE[model]
        q = w[w["contact_G"] & np.all([w[f"ok_{lv}"] for lv in WDR_LEVELS], axis=0)
              & np.all([w[f"hidden_{lv}"] for lv in WDR_LEVELS[1:]], axis=0)]
        for lv in WDR_LEVELS:
            out.append((f"dose-response {WALL_DOSE_DESIGN['labels'][lv]}", "all", "WDR", lv,
                        _frame(q["x_par"], q[f"rho_{lv}"], q[f"rho_mono_{lv}"], q[f"gt_parallax_med_{lv}"],
                               ["contact in view" if lv == "G" else "contact hidden"] * len(q), q["scene"].astype(str))))
    if model in MODELS_OK["REAL"]:
        d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "REAL")]
        d = d[e1_mask(d)]
        out.append(("CONVOY-REAL", "all", "REAL", "", _frame(d["x_par"], d["rho"], d["rho_mono"], d["gt_parallax_med"],
                                                             ["unsupported"] * len(d), d["scene"].astype(str))))
    if model in MODELS_OK["RABL"] and model in RABL_WIDE:
        w = RABL_WIDE[model]; q = w[w["sample"]]
        for c, sup in (("G", "contact in view"), ("F", "unsupported")):
            out.append((f"E5-REAL {RABL_DESIGN['labels'][c]}", "all", "RABL", c,
                        _frame(q["x_par"], q[f"rho_{c}"], q[f"rho_mono_{c}"], q[f"gt_parallax_med_{c}"], [sup] * len(q),
                               q["scene"].astype(str))))
    for set_, name in (("VK", "Virtual KITTI 2"), ("AV2", "Argoverse 2")):
        if model not in MODELS_OK.get(set_, []):
            continue
        d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == set_)]
        if not len(d) or "gt_contact" not in d:
            continue
        big = (d["gt_n_eroded"] >= PREREG["x6_min_eroded_px"]) if set_ == "VK" else d["measurable"].astype(bool)
        lo_x, hi_x = PREREG["h9_x_range"]
        d = d[big & (d["regime"] == "parallel") & (d["gt_parallax_med"] >= PREREG["parallax_min_px"])
              & d["gt_contact"].isin(["visible", "hidden"]) & d["gt_x_geo"].between(lo_x, hi_x)]
        for cont, sup in (("visible", "contact in view"), ("hidden", "contact hidden")):
            q = d[d["gt_contact"] == cont]
            out.append((f"{name}, contact {cont}", "all", set_, cont,
                        _frame(q["gt_x_geo"], q["rho"], q["rho_mono"], q["gt_parallax_med"], [sup] * len(q),
                               q["seq"].astype(str) + "/" + q["track"].astype(str))))
    return out


def _fold_ids(cl, n_folds, seed):
    u = np.unique(cl)
    perm = np.random.default_rng(seed).permutation(len(u))
    return dict(zip(u[perm], np.arange(len(u)) % n_folds))


CUE_FIT_ROWS, CUE_CV_ROWS, CUE_PRED_ROWS, CUE_HUMP_ROWS, CUE_FITS, CUE_TRAIN = [], [], [], [], {}, {}
for model in EVALUATED_MODELS:
    if model not in MODELS_OK["MAIN"] or model not in MODELS_OK["ABL"]:
        continue
    tr = cue_training(model)
    if len(tr) < 40 or tr["cl"].nunique() < 10:
        continue
    CUE_TRAIN[model] = tr
    fits = {"two channels": cue_fit(tr), "contact only (kappa = 0)": cue_fit(tr, kappas=[0.0]),
            "parallax only (one c)": cue_fit(tr, per_support=False)}
    CUE_FITS[model] = fits["two channels"]
    for name, f_ in fits.items():
        CUE_FIT_ROWS.append(dict(model=model, fit=name, kappa_px=f_["kappa"], n=f_["n"], n_clusters=int(tr["cl"].nunique()),
                                 rmse=float(np.sqrt(f_["sse"] / max(f_["n"], 1))),
                                 **{f"c_{s.replace(' ', '_')}": f_["c"].get(s, f_["c"].get("all", np.nan)) for s in _SUPPORTS}))
    # 10-fold cross-validation over scenes: squared error of every held-out object under each model
    fold = tr["cl"].map(_fold_ids(tr["cl"].to_numpy(), CUE_DESIGN["n_folds"], seed_for("cue-folds", model))).to_numpy()
    se = {k: np.full(len(tr), np.nan) for k in fits}
    for f_id in range(CUE_DESIGN["n_folds"]):
        trn, tst = tr[fold != f_id], tr[fold == f_id]
        if not len(tst) or len(trn) < 20:
            continue
        for name, per_sup, ks in (("two channels", True, None), ("contact only (kappa = 0)", True, [0.0]),
                                  ("parallax only (one c)", False, None)):
            fit_k = cue_fit(trn, per_support=per_sup, kappas=ks)
            se[name][fold == f_id] = (tst["rho"].to_numpy(float) - cue_predict(fit_k, tst)) ** 2
    okcv = np.all([np.isfinite(v) for v in se.values()], axis=0)
    bs = ClusterBootstrap(tr["cl"].to_numpy()[okcv], seed=seed_for("cue-cv", model))
    for other in ("contact only (kappa = 0)", "parallax only (one c)"):
        diff = se[other][okcv] - se["two channels"][okcv]
        T, P = bs.linear([np.ones(int(okcv.sum())), diff])
        dr = T[:, 1] / T[:, 0]
        lo, hi = _ci(dr)
        CUE_CV_ROWS.append(dict(model=model, comparison=f"{other} minus two channels", n=int(okcv.sum()),
                                mse_two_channels=float(np.mean(se["two channels"][okcv])), mse_other=float(np.mean(se[other][okcv])),
                                delta_mse=float(P[1] / P[0]), ci_lo=lo, ci_hi=hi,
                                direction="two channels predict better" if lo > 0 else "worse" if hi < 0 else "no clear difference"))
    # in-sample slopes by support and condition (reference), then out-of-sample predictions
    fit = fits["two channels"]
    tr = tr.assign(rho_hat=cue_predict(fit, tr))
    for src, g in tr.groupby("source", sort=False):
        for sup, q in g.groupby("support"):
            r = slope_with_ci(q["x"], q["rho"], q["cl"], seed_for("cue-in", model, src, sup))
            CUE_PRED_ROWS.append(dict(model=model, testbed=src, group=sup, set="fit", level="", role="fitted", n=r["n"],
                                      n_clusters=r["n_clusters"], beta_observed=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                                      beta_predicted=ols_fit(q["x"], q["rho_hat"])[0] if len(q) >= 8 else np.nan,
                                      median_parallax=float(q["d"].median())))
    for name, grp, set_, lv, q in cue_tests(model):
        if len(q) < 8 or q["cl"].nunique() < 5:
            continue
        r = slope_with_ci(q["x"], q["rho"], q["cl"], seed_for("cue-out", model, name, grp))
        CUE_PRED_ROWS.append(dict(model=model, testbed=name, group=grp, set=set_, level=lv, role="predicted", n=r["n"],
                                  n_clusters=r["n_clusters"], beta_observed=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                                  beta_predicted=ols_fit(q["x"], cue_predict(fit, q))[0], median_parallax=float(q["d"].median())))
    # the hump: CONVOY-3D movers binned by x; observed and fitted medians (two channels and contact only)
    main = tr[tr["source"] == "CONVOY-3D"].assign(rho_c0=cue_predict(fits["contact only (kappa = 0)"], tr[tr["source"] == "CONVOY-3D"]))
    xb = CUE_DESIGN["x_bins"]
    for k in range(len(xb) - 1):
        q = main[(main["x"] >= xb[k]) & (main["x"] < xb[k + 1])]
        if len(q) >= 5:
            CUE_HUMP_ROWS.append(dict(model=model, x_lo=xb[k], x_hi=xb[k + 1], n=len(q), x_median=float(q["x"].median()),
                                      rho_observed=float(q["rho"].median()), rho_two_channels=float(q["rho_hat"].median()),
                                      rho_contact_only=float(q["rho_c0"].median()), parallax_median=float(q["d"].median())))
CUE_FIT_TABLE, CUE_CV, CUE_PRED, CUE_HUMP = (pd.DataFrame(CUE_FIT_ROWS), pd.DataFrame(CUE_CV_ROWS), pd.DataFrame(CUE_PRED_ROWS),
                                              pd.DataFrame(CUE_HUMP_ROWS))
# how much of the hump each fit reproduces (v6.6): the fall of the median rho from the bin where the observed median peaks
# to the bin of largest x, observed and as each fit has it
_fall = []
for _m, _g in (CUE_HUMP.groupby("model") if len(CUE_HUMP) else []):
    _g = _g.sort_values("x_median").reset_index(drop=True)
    if len(_g) >= 3:
        _k = int(_g["rho_observed"].idxmax())
        _fall.append(dict(model=_m, x_peak=float(_g["x_median"].iloc[_k]), x_last=float(_g["x_median"].iloc[-1]),
                          fall_observed=float(_g["rho_observed"].iloc[_k] - _g["rho_observed"].iloc[-1]),
                          fall_two_channels=float(_g["rho_two_channels"].iloc[_k] - _g["rho_two_channels"].iloc[-1]),
                          fall_contact_only=float(_g["rho_contact_only"].iloc[_k] - _g["rho_contact_only"].iloc[-1])))
CUE_HUMP_FALL = pd.DataFrame(_fall)
if len(CUE_PRED):
    _p = CUE_PRED[CUE_PRED["role"] == "predicted"]
    CUE_PRED["inside_ci"] = (CUE_PRED["beta_predicted"] >= CUE_PRED["ci_lo"]) & (CUE_PRED["beta_predicted"] <= CUE_PRED["ci_hi"])
    _pp = CUE_PRED[(CUE_PRED["role"] == "predicted") & CUE_PRED[["beta_observed", "beta_predicted"]].notna().all(1)]
    CUE_CALIBRATION = dict(n=len(_pp), pearson_r=float(np.corrcoef(_pp["beta_observed"], _pp["beta_predicted"])[0, 1]) if len(_pp) >= 3 else np.nan,
                           mean_abs_error=float(np.mean(np.abs(_pp["beta_observed"] - _pp["beta_predicted"]))) if len(_pp) else np.nan,
                           share_inside_ci=float(_pp["inside_ci"].mean()) if len(_pp) else np.nan)
else:
    CUE_CALIBRATION = dict(n=0, pearson_r=np.nan, mean_abs_error=np.nan, share_inside_ci=np.nan)

# ---- MonST3R against DUSt3R on the same movers --------------------------------------------------------------------
MONDUS_ROWS = []
if all(m in EVALUATED_MODELS for m in ("DUSt3R", "MonST3R")):
    _a = MOV_RES[(MOV_RES["set"] == "MAIN") & MOV_RES["model"].isin(["DUSt3R", "MonST3R"])]
    _a = _a[e1_mask(_a)].pivot_table(index=["uid", "scene"], columns="model", values=["rho"], aggfunc="first")
    _a.columns = [f"{a}_{b}" for a, b in _a.columns]
    _a = _a.reset_index().merge(MOVERS_MAIN[["uid", "x_par", "gt_parallax_med"]], on="uid", how="left")
    if all(m in MODELS_OK["MAIN"] for m in ("DUSt3R", "MonST3R")):
        for lab, sel in (("parallax > 40 px", _a["gt_parallax_med"] > 40), ("parallax 5-40 px", _a["gt_parallax_med"] <= 40)):
            q = _a[sel]
            b, dr, n, nc = multi_slopes(q["x_par"], [q["rho_MonST3R"], q["rho_DUSt3R"]], q["scene"], seed_for("mondus", lab))
            est, lo, hi = contrast_ci(b, dr, [1, -1])
            MONDUS_ROWS.append(dict(contrast=f"beta_s, CONVOY-3D movers with {lab}", n=n, n_scenes=nc, MonST3R=b[0], DUSt3R=b[1],
                                    difference=est, ci_lo=lo, ci_hi=hi, direction=_dir_label(lo, hi)))
    if all(m in MODELS_OK["ABL"] and m in E5_WIDE for m in ("DUSt3R", "MonST3R")):
        _wm, _wd = E5_WIDE["MonST3R"], E5_WIDE["DUSt3R"]
        _j = _wm.merge(_wd, on=["uid", "scene"], suffixes=("_m", "_d"))
        q = _j[_j["walled_m"] & _j["ok_G_m"] & _j["ok_CH_m"] & _j["ok_G_d"] & _j["ok_CH_d"] & _j["vis_ok_m"]]
        b, dr, n, nc = multi_slopes(q["x_par_m"], [q["rho_CH_m"], q["rho_G_m"], q["rho_CH_d"], q["rho_G_d"]], q["scene"],
                                    seed_for("mondus-h6"))
        est, lo, hi = contrast_ci(b, dr, [1, -1, -1, 1])
        MONDUS_ROWS.append(dict(contrast="H6 contrast beta(CH) - beta(G), E5 movers", n=n, n_scenes=nc, MonST3R=b[0] - b[1],
                                DUSt3R=b[2] - b[3], difference=est, ci_lo=lo, ci_hi=hi, direction=_dir_label(lo, hi)))
    if all(m in MODELS_OK["BSW"] and m in BSW_WIDE for m in ("DUSt3R", "MonST3R")):
        _j = BSW_WIDE["MonST3R"].merge(BSW_WIDE["DUSt3R"], on=["uid", "scene"], suffixes=("_m", "_d"))
        q = _j[_j["h12_sample_m"] & _j["h12_sample_d"]]
        b, dr, n, nc = multi_slopes(q["x_par_m"], [q[f"rho_{_hi12}_m"], q[f"rho_{_lo12}_m"], q[f"rho_{_hi12}_d"], q[f"rho_{_lo12}_d"]],
                                    q["scene"], seed_for("mondus-bsw"))
        est, lo, hi = contrast_ci(b, dr, [1, -1, -1, 1])
        MONDUS_ROWS.append(dict(contrast="rise of beta_s from x0.5 to x3, baseline sweep", n=n, n_scenes=nc, MonST3R=b[0] - b[1],
                                DUSt3R=b[2] - b[3], difference=est, ci_lo=lo, ci_hi=hi, direction=_dir_label(lo, hi)))
MONDUS = pd.DataFrame(MONDUS_ROWS)

# ---- masks that defeat themselves: change of M2 recall against change of beta_s ---------------------------------------
TRADEOFF_ROWS = []
for _src, _beta, _rec, _key, _ref, _lab in (
        ("E5", E5_BETA, E5_MASKS, "condition", "G", ABLATION_DESIGN["labels"]),
        ("baseline sweep", BSW_BETA[BSW_BETA["support"] == "all"] if len(BSW_BETA) else BSW_BETA, BSW_RECALL, "level", _lo12, SWEEP_DESIGN["labels"]),
        ("dose-response", WDR_BETA, WDR_RECALL, "level", "G", WALL_DOSE_DESIGN["labels"])):
    if not len(_beta) or not len(_rec):
        continue
    _rc = _rec.rename(columns={"recall_M2": "recall"})
    for model in _rec["model"].unique():
        b_ = _beta[_beta["model"] == model].set_index(_key)["beta"]
        r_ = _rc[_rc["model"] == model].set_index(_key)["recall"]
        if _ref not in b_ or _ref not in r_:
            continue
        for lv in r_.index:
            if lv == _ref or lv not in b_:
                continue
            TRADEOFF_ROWS.append(dict(source=_src, model=model, level=lv, label=_lab.get(lv, lv),
                                      delta_beta=float(b_[lv] - b_[_ref]), delta_recall=float(r_[lv] - r_[_ref])))
TRADEOFF = pd.DataFrame(TRADEOFF_ROWS)
if len(TRADEOFF) >= 5:
    from scipy.stats import spearmanr as _spr
    TRADEOFF_SUMMARY = dict(n=len(TRADEOFF), spearman=float(_spr(TRADEOFF["delta_beta"], TRADEOFF["delta_recall"])[0]),
                            share_opposite_sign=float(np.mean(np.sign(TRADEOFF["delta_beta"]) != np.sign(TRADEOFF["delta_recall"]))))
else:
    TRADEOFF_SUMMARY = dict(n=len(TRADEOFF), spearman=np.nan, share_opposite_sign=np.nan)

# ---- CONVOY-REAL against CONVOY-3D floating crates, within parallax bins (D57) ----------------------------------------
REALPAR_ROWS = []
_pb = [5, 20, 40, 80, 1e9]
for model in EVALUATED_MODELS:
    if model not in MODELS_OK["MAIN"] or model not in MODELS_OK["REAL"]:
        continue
    for tb, d in (("CONVOY-3D floating crates", MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN")]),
                  ("CONVOY-REAL", MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "REAL")])):
        d = d[e1_mask(d)]
        if tb.startswith("CONVOY-3D"):
            d = d[d["cls"] == "crate_floating"]
        REALPAR_ROWS.append(dict(model=model, testbed=tb, bin="all", n=len(d), median_parallax=float(d["gt_parallax_med"].median()) if len(d) else np.nan,
                                 **{k: v for k, v in slope_with_ci(d["x_par"], d["rho"], d["scene"], seed_for("realpar", model, tb)).items()
                                    if k in ("beta", "ci_lo", "ci_hi")}))
        for k in range(len(_pb) - 1):
            q = d[(d["gt_parallax_med"] >= _pb[k]) & (d["gt_parallax_med"] < _pb[k + 1])]
            r = slope_with_ci(q["x_par"], q["rho"], q["scene"], seed_for("realpar", model, tb, k)) if len(q) >= 12 else dict(beta=np.nan, ci_lo=np.nan, ci_hi=np.nan)
            REALPAR_ROWS.append(dict(model=model, testbed=tb, bin=f"{_pb[k]:g}-{_pb[k + 1]:g} px" if _pb[k + 1] < 1e8 else f">{_pb[k]:g} px",
                                     n=len(q), median_parallax=float(q["gt_parallax_med"].median()) if len(q) else np.nan,
                                     beta=r["beta"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"]))
REAL_PARALLAX = pd.DataFrame(REALPAR_ROWS)

_f3 = lambda v: f"{v:.3f}"
with pd.option_context("display.width", 250, "display.max_colwidth", 70):
    if len(CUE_FIT_TABLE):
        print("Cue model fitted on CONVOY-3D and E5 (rho - rho_mono = w (x - rho_mono), w = c d^2 / (d^2 + kappa^2)):")
        print(CUE_FIT_TABLE.to_string(index=False, na_rep="–", float_format=_f3))
        print("\n10-fold cross-validation over scenes (difference of the mean squared error of held-out rho; > 0: the two-channel "
              "model predicts better):")
        print(CUE_CV.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.4f}"))
    if len(CUE_PRED):
        print("\nObserved and predicted slopes (fitted rows: CONVOY-3D and E5; predicted rows: no refitting):")
        print(CUE_PRED.drop(columns=["set", "level"]).to_string(index=False, na_rep="–", float_format=_f3))
        print(f"\npredicted vs observed over {CUE_CALIBRATION['n']} out-of-sample slopes: Pearson r = {CUE_CALIBRATION['pearson_r']:.2f}, "
              f"mean |error| = {CUE_CALIBRATION['mean_abs_error']:.3f}, prediction inside the observed 95% interval: "
              f"{100 * CUE_CALIBRATION['share_inside_ci']:.0f}%")
    if len(CUE_HUMP):
        print("\nThe hump: median rho of CONVOY-3D parallel movers by x, observed and fitted:")
        print(CUE_HUMP.to_string(index=False, float_format=_f3))
        if len(CUE_HUMP_FALL):
            print("fall of the median rho from the observed peak to the largest x (observed, two channels, contact only):")
            print(CUE_HUMP_FALL.to_string(index=False, float_format=_f3))
    if len(MONDUS):
        print("\nMonST3R minus DUSt3R on the same movers (95% cluster-bootstrap intervals):")
        print(MONDUS.to_string(index=False, na_rep="–", float_format=_f3))
    if len(TRADEOFF):
        print(f"\nMask trade-off: change of M2 recall against change of beta_s ({TRADEOFF_SUMMARY['n']} model x condition pairs): "
              f"Spearman {TRADEOFF_SUMMARY['spearman']:.2f}, opposite signs in {100 * TRADEOFF_SUMMARY['share_opposite_sign']:.0f}%")
    if len(REAL_PARALLAX):
        print("\nCONVOY-REAL and CONVOY-3D floating crates by parallax (beta_s, 95% CI; bins with >= 12 movers):")
        print(REAL_PARALLAX.to_string(index=False, na_rep="–", float_format=_f3))
for _n, _df in (("cue_fits", CUE_FIT_TABLE), ("cue_cv", CUE_CV), ("cue_predictions", CUE_PRED), ("cue_hump", CUE_HUMP),
                ("cue_hump_fall", CUE_HUMP_FALL),
                ("monst3r_vs_dust3r", MONDUS), ("mask_tradeoff", TRADEOFF), ("convoy_real_by_parallax", REAL_PARALLAX)):
    _df.to_csv(os.path.join(RESULTS_DIR, f"{_n}.csv"), index=False)

In [ ]:
# Cell No: 066
# 6.12 Bonn RGB-D Dynamic (v6.5, ground truth corrected in v6.6; exploratory, D56, D62): prevalence of the motion regimes
#      of real objects, and the per-object geometry share g of parallel objects by kind and by floor contact. Cluster =
#      image pair. No verdicts; nothing is reported if the ground truth fails its quality gate (Section 3.12).
BONN_PREV_ROWS, BONN_G_ROWS, BONN_TEST_ROWS = [], [], []
if len(MOVERS_BONN):
    _mk = MOVERS_BONN[MOVERS_BONN["measurable"].astype(bool)]
    for kind, g in [("all objects", _mk)] + list(_mk.groupby("cls")):
        BONN_PREV_ROWS.append(dict(kind=kind, n_objects=len(g), n_pairs=int(g["scene"].nunique()),
                                   frac_parallel=float((g["regime"] == "parallel").mean()),
                                   frac_intermediate=float((g["regime"] == "intermediate").mean()),
                                   frac_perpendicular=float((g["regime"] == "perpendicular").mean()),
                                   frac_no_static_solution=float(g["gt_no_static_solution"].astype(bool).mean()),
                                   frac_on_floor=float((g["floor"] == "on the floor").mean()),
                                   frac_off_floor=float((g["floor"] == "off the floor").mean()),
                                   median_parallax_px=float(g["gt_parallax_med"].median()), median_depth_m=float(g["gt_Z_true"].median())))
    for model in ([] if BONN_STATUS.get("suspect") else MODELS_OK.get("BONN", [])):    # v6.6: quality gate of the objects
        d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "BONN")].copy()
        if not len(d):
            continue
        d = d[d["measurable"].astype(bool) & (d["regime"] == "parallel") & (d["gt_parallax_med"] >= PREREG["parallax_min_px"])
              & np.isfinite(d["rho"]) & np.isfinite(d["rho_mono"]) & np.isfinite(d["gt_x_geo"])].copy()
        d["g"] = geo_share_obj(d["rho"], d["rho_mono"], d["gt_x_geo"])
        d = d[np.isfinite(d["g"]) & (np.abs(d["gt_x_geo"] - d["rho_mono"]) >= PREREG["x6_min_lever"])]
        d["cl"] = d["scene"]
        groups = [("kind", k, q) for k, q in d.groupby("cls")] + [("floor contact", f_, q) for f_, q in d.groupby("floor")
                                                                  if f_ != "undetermined"]
        for gtype, gname, q in groups:
            est, lo, hi = _cboot(q, _med_g, seed_for("bonn-g", model, gname)) if len(q) >= 8 else (_med_g(q), np.nan, np.nan)
            BONN_G_ROWS.append(dict(model=model, grouping=gtype, group=gname, n=len(q), n_pairs=int(q["cl"].nunique()),
                                    median_g=est, ci_lo=lo, ci_hi=hi, median_x_geo=float(q["gt_x_geo"].median()) if len(q) else np.nan))
        on, off = d[d["floor"] == "on the floor"], d[d["floor"] == "off the floor"]
        if len(on) >= 8 and len(off) >= 8:
            fn = lambda q: (float(np.median(q.loc[q["floor"] == "off the floor", "g"]) - np.median(q.loc[q["floor"] == "on the floor", "g"]))
                            if (q["floor"] == "off the floor").any() and (q["floor"] == "on the floor").any() else np.nan)
            est, lo, hi = _cboot(d[d["floor"].isin(["on the floor", "off the floor"])], fn, seed_for("bonn-diff", model))
        else:
            est = lo = hi = np.nan
        BONN_TEST_ROWS.append(dict(model=model, contrast="off the floor minus on the floor (median g)", n_off=len(off), n_on=len(on),
                                   estimate=est, ci_lo=lo, ci_hi=hi, direction=_dir_label(lo, hi) if np.isfinite(est) else "NOT TESTABLE"))
BONN_PREV = pd.DataFrame(BONN_PREV_ROWS)
BONN_G = pd.DataFrame(BONN_G_ROWS)
BONN_TESTS = pd.DataFrame(BONN_TEST_ROWS)
if len(BONN_PREV):
    with pd.option_context("display.width", 250):
        print("Bonn RGB-D Dynamic: motion regimes of real objects (parallel: |eps| below the flow-noise floor "
              f"{BONN_TAU_PAR:.2f} px):")
        print(BONN_PREV.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
        if len(BONN_G):
            print("\nMedian per-object geometry share g of parallel, identifiable objects (|x - rho_mono| >= 0.3), by kind and floor contact:")
            print(BONN_G.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
            print("\nObjects off the floor minus objects on it:")
            print(BONN_TESTS.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
        elif BONN_STATUS.get("suspect"):
            print("\nThe Bonn objects fail the quality gate (much of the static scene is flagged as moving), so no geometry "
                  "share is reported.")
        else:
            print("\nNo model has enough parallel Bonn objects for the geometry share in this run.")
else:
    print("Bonn RGB-D Dynamic is not part of this run:", BONN_STATUS["note"] or "no data")
for _n, _df in (("bonn_prevalence", BONN_PREV), ("bonn_geometry_share", BONN_G), ("bonn_floor_contrast", BONN_TESTS)):
    _df.to_csv(os.path.join(RESULTS_DIR, f"{_n}.csv"), index=False)

In [ ]:
# Cell No: 067
# 6.13 The static twin test (v6.5, exploratory, D58): Argoverse 2 vehicles of all logs; twin height H (1 - e^x_geo) and
#      epipolar offset, both thresholded at 5% false positives on parked vehicles; recall on moving vehicles by regime and
#      by s; the detection curve implied by the parked distribution for exactly parallel motion. Cluster = vehicle track.
#      v6.6 (D63): the same tests with optical flow instead of labelled correspondences and a LiDAR road plane instead of
#      the box bottom, and the registered H14 (twin minus epipolar in the parallel regime, paired over the same vehicles).
TWIN_ROWS, TWIN_BY_S, TWIN_CURVE = [], [], []
TWIN_VEH, TWIN_THRESH = pd.DataFrame(), {}


def _rec_ci(q, cols, seed):
    # share of vehicles flagged by each test, with cluster-bootstrap intervals over vehicle tracks
    if len(q) < 5 or q["cl"].nunique() < 5:
        return {c: (float(q[c].mean()) if len(q) else np.nan, np.nan, np.nan) for c in cols}
    T, P = ClusterBootstrap(q["cl"].to_numpy(), seed=seed).linear([np.ones(len(q))] + [q[c].to_numpy(float) for c in cols])
    out = {}
    for k, c in enumerate(cols):
        lo, hi = _ci(T[:, k + 1] / T[:, 0])
        out[c] = (float(P[k + 1] / P[0]), lo, hi)
    return out


if len(AV2_TWIN) and len(MOVERS_AV2_ALL):
    _t = MOVERS_AV2_ALL.merge(AV2_TWIN.drop(columns=["track"]), on=["scene", "mover"], how="left")
    _t = _t[_t["measurable"].astype(bool) & np.isfinite(_t["H_cam"]) & np.isfinite(_t["gt_eps_med"])].copy()
    _t["h_twin"] = _t["H_cam"] * (1.0 - np.exp(_t["gt_x_geo"]))
    _t["no_static_solution"] = ~np.isfinite(_t["gt_x_geo"])
    _t["cl"] = _t["seq"].astype(str) + "/" + _t["track"].astype(str)
    _pk = _t[(_t["state"] == "parked") & np.isfinite(_t["h_twin"])]
    q_ = 1.0 - PREREG["twin_fpr"]
    if len(_pk) >= 20:
        TWIN_THRESH = dict(n_parked=len(_pk), eps_px=float(np.quantile(_pk["gt_eps_med"], q_)),
                           height_m=float(np.quantile(np.abs(_pk["h_twin"]), q_)), H_median_m=float(_t["H_cam"].median()))
        _mv = _t[_t["state"] == "moving"].copy()
        _mv["det_epipolar"] = _mv["gt_eps_med"] > TWIN_THRESH["eps_px"]
        _mv["det_twin"] = np.where(_mv["no_static_solution"], False, np.abs(_mv["h_twin"]) > TWIN_THRESH["height_m"])
        _mv["det_either"] = _mv["det_epipolar"] | _mv["det_twin"]
        _mv["twin_below_road"] = _mv["det_twin"] & (_mv["h_twin"] < 0)
        TWIN_VEH = _mv


        _cols = ["det_epipolar", "det_twin", "det_either", "twin_below_road"]
        _valid = _mv[~_mv["no_static_solution"]]
        for reg in ("parallel", "intermediate", "perpendicular", "all"):
            q = _valid if reg == "all" else _valid[_valid["regime"] == reg]
            r = _rec_ci(q, _cols, seed_for("twin", reg))
            # what the twin test adds: its share among the vehicles the epipolar test misses (same exact correspondences)
            qm = q[~q["det_epipolar"]]
            rm = _rec_ci(qm, ["det_twin"], seed_for("twin-miss", reg))["det_twin"]
            TWIN_ROWS.append(dict(regime=reg, n=len(q), n_tracks=int(q["cl"].nunique()),
                                  **{f"{c}": r[c][0] for c in _cols}, **{f"{c}_lo": r[c][1] for c in _cols},
                                  **{f"{c}_hi": r[c][2] for c in _cols},
                                  n_epi_miss=len(qm), twin_given_epi_miss=rm[0], twin_given_epi_miss_lo=rm[1],
                                  twin_given_epi_miss_hi=rm[2]))
        _sb = TWIN_DESIGN["s_bins"]
        for k in range(len(_sb) - 1):
            q = _valid[(_valid["s"] >= _sb[k]) & (_valid["s"] < _sb[k + 1])]
            if not len(q):
                continue
            r = _rec_ci(q, _cols[:3], seed_for("twin-s", k))
            TWIN_BY_S.append(dict(s_lo=_sb[k], s_hi=_sb[k + 1], s_median=float(q["s"].median()), n=len(q),
                                  frac_parallel=float((q["regime"] == "parallel").mean()),
                                  **{c: r[c][0] for c in _cols[:3]}, **{f"{c}_lo": r[c][1] for c in _cols[:3]},
                                  **{f"{c}_hi": r[c][2] for c in _cols[:3]}))
        # detection curve for exactly parallel motion at the median camera height: the twin height -H s / (1 - s) plus
        # the noise of parked vehicles (their twin heights), against the same threshold
        _noise = _pk["h_twin"].to_numpy(float)
        for s_ in np.linspace(*TWIN_DESIGN["theory_s"], 75):
            h0 = -TWIN_THRESH["H_median_m"] * s_ / (1.0 - s_)
            TWIN_CURVE.append(dict(s=float(s_), twin_height_m=float(h0),
                                   p_detect_twin=float(np.mean(np.abs(h0 + _noise) > TWIN_THRESH["height_m"])),
                                   p_detect_epipolar=PREREG["twin_fpr"]))
        TWIN_THRESH.update(n_moving=len(_mv), n_no_static_solution=int(_mv["no_static_solution"].sum()),
                           parked_fpr_epipolar=float(np.mean(_pk["gt_eps_med"] > TWIN_THRESH["eps_px"])),
                           parked_fpr_twin=float(np.mean(np.abs(_pk["h_twin"]) > TWIN_THRESH["height_m"])))
TWIN_RECALL, TWIN_S, TWIN_THEORY = pd.DataFrame(TWIN_ROWS), pd.DataFrame(TWIN_BY_S), pd.DataFrame(TWIN_CURVE)
if len(TWIN_RECALL):
    print(f"Twin test on Argoverse 2: thresholds at {100 * PREREG['twin_fpr']:.0f}% false positives on {TWIN_THRESH['n_parked']} parked "
          f"vehicles: |eps| > {TWIN_THRESH['eps_px']:.2f} px (epipolar), |twin height| > {TWIN_THRESH['height_m']:.2f} m (twin); "
          f"median camera height above the contact plane {TWIN_THRESH['H_median_m']:.2f} m")
    print(f"moving vehicles: {TWIN_THRESH['n_moving']}, of which {TWIN_THRESH['n_no_static_solution']} have no static solution "
          f"(s >= 1: the twin lies behind the camera, which any cheirality check rejects)")
    with pd.option_context("display.width", 250):
        print("\nShare of moving vehicles flagged, by regime (95% intervals over tracks):")
        print(TWIN_RECALL[["regime", "n", "n_tracks", "det_epipolar", "det_twin", "det_either", "twin_below_road", "n_epi_miss",
                           "twin_given_epi_miss"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
        print("\nBy parallel fraction s:")
        print(TWIN_S[["s_lo", "s_hi", "n", "frac_parallel", "det_epipolar", "det_twin", "det_either"]]
              .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    _par = TWIN_RECALL[TWIN_RECALL["regime"] == "parallel"]
    if len(_par) and _par["n"].iloc[0]:
        _r = _par.iloc[0]
        if _r["n_epi_miss"]:
            _ci_t = (f" [{_r['twin_given_epi_miss_lo']:.2f}, {_r['twin_given_epi_miss_hi']:.2f}]"
                     if np.isfinite(_r["twin_given_epi_miss_lo"]) else " (too few tracks for an interval)")
            print(f"\nParallel regime: the epipolar test misses {int(_r['n_epi_miss'])} of {int(_r['n'])} moving vehicles even with "
                  f"exact correspondences; the twin test flags {_r['twin_given_epi_miss']:.2f}{_ci_t} of those.")
        else:
            print("\nParallel regime: with exact correspondences the epipolar test misses none of the moving vehicles in this run.")
else:
    print("The twin test is not part of this run:", AV2T_STATUS["note"] or "no Argoverse 2 camera heights")
for _n, _df in (("twin_recall_by_regime", TWIN_RECALL), ("twin_recall_by_s", TWIN_S), ("twin_detection_curve", TWIN_THEORY)):
    _df.to_csv(os.path.join(RESULTS_DIR, f"{_n}.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "twin_thresholds.json"), "w") as f:
    json.dump(TWIN_THRESH, f, indent=1)

# ---- the twin test with optical flow (v6.6, D63; H14 registered in AMENDMENT_V6_6) -----------------------------------
# Same vehicles, but every quantity from the flow at their LiDAR pixels: the epipolar offset, and the 10th percentile of
# the heights of the static reconstruction above the road plane fitted to the LiDAR (no box). Thresholds at 5% false
# positives on parked vehicles: eps above its 95th percentile; the low height outside its 2.5-97.5% range.
FTWIN_ROWS, FTWIN_THRESH, H14_RESULT, FTWIN_VEH = [], {}, {}, pd.DataFrame()
if len(AV2_FLOWTWIN) and len(MOVERS_AV2_ALL):
    _f = MOVERS_AV2_ALL.merge(AV2_FLOWTWIN, on=["scene", "mover"], how="inner")
    _f = _f[_f["measurable"].astype(bool) & (_f["n_flow_px"] >= FLOWTWIN_DESIGN["min_px"]) & np.isfinite(_f["eps_flow"])].copy()
    _f["cl"] = _f["seq"].astype(str) + "/" + _f["track"].astype(str)
    _f["static_solution"] = (_f["frac_behind"] <= FLOWTWIN_DESIGN["behind_frac"]) & np.isfinite(_f["h_low_flow"])
    _fpk = _f[(_f["state"] == "parked") & _f["static_solution"]]
    if len(_fpk) >= 20:
        _a = PREREG["twin_fpr"]
        FTWIN_THRESH = dict(n_parked=len(_fpk), eps_px=float(np.quantile(_fpk["eps_flow"], 1 - _a)),
                            h_low_lo_m=float(np.quantile(_fpk["h_low_flow"], _a / 2)),
                            h_low_hi_m=float(np.quantile(_fpk["h_low_flow"], 1 - _a / 2)),
                            parked_h_low_true_median_m=float(_fpk["h_low_true"].median()),
                            parked_x_flow_abs_median=float(np.median(np.abs(_fpk["x_flow"]))), flow=AV2F_STATUS.get("flow", ""))
        _fm = _f[_f["state"] == "moving"].copy()
        _fm["det_epipolar"] = _fm["eps_flow"] > FTWIN_THRESH["eps_px"]
        _fm["det_twin"] = _fm["static_solution"] & ((_fm["h_low_flow"] < FTWIN_THRESH["h_low_lo_m"])
                                                    | (_fm["h_low_flow"] > FTWIN_THRESH["h_low_hi_m"]))
        _fm["det_either"] = _fm["det_epipolar"] | _fm["det_twin"]
        _fm["twin_below_road"] = _fm["det_twin"] & (_fm["h_low_flow"] < FTWIN_THRESH["h_low_lo_m"])
        FTWIN_VEH = _fm
        _fv = _fm[_fm["static_solution"]]
        _cols = ["det_epipolar", "det_twin", "det_either", "twin_below_road"]
        for reg in ("parallel", "intermediate", "perpendicular", "all"):
            q = _fv if reg == "all" else _fv[_fv["regime"] == reg]
            r = _rec_ci(q, _cols, seed_for("ftwin", reg))
            qm = q[~q["det_epipolar"]]
            rm = _rec_ci(qm, ["det_twin"], seed_for("ftwin-miss", reg))["det_twin"]
            FTWIN_ROWS.append(dict(regime=reg, n=len(q), n_tracks=int(q["cl"].nunique()), **{c: r[c][0] for c in _cols},
                                   **{f"{c}_lo": r[c][1] for c in _cols}, **{f"{c}_hi": r[c][2] for c in _cols},
                                   n_epi_miss=len(qm), twin_given_epi_miss=rm[0], twin_given_epi_miss_lo=rm[1],
                                   twin_given_epi_miss_hi=rm[2]))
        FTWIN_THRESH.update(n_moving=len(_fm), n_no_static_solution=int((~_fm["static_solution"]).sum()),
                            parked_fpr_epipolar=float(np.mean(_fpk["eps_flow"] > FTWIN_THRESH["eps_px"])),
                            parked_fpr_twin=float(np.mean((_fpk["h_low_flow"] < FTWIN_THRESH["h_low_lo_m"])
                                                          | (_fpk["h_low_flow"] > FTWIN_THRESH["h_low_hi_m"]))))
        # H14 (registered): parallel-regime moving vehicles with a static solution, det_twin - det_epipolar (paired)
        q = _fv[_fv["regime"] == "parallel"]
        H14_RESULT = dict(n=len(q), n_tracks=int(q["cl"].nunique()), det_twin=float(q["det_twin"].mean()) if len(q) else np.nan,
                          det_epipolar=float(q["det_epipolar"].mean()) if len(q) else np.nan, estimate=np.nan, ci_lo=np.nan,
                          ci_hi=np.nan, verdict="NOT TESTABLE")
        if len(q) >= PREREG["h14_min_vehicles"] and q["cl"].nunique() >= PREREG["h14_min_tracks"]:
            _T, _P = ClusterBootstrap(q["cl"].to_numpy(), seed=seed_for("h14")).linear(
                [np.ones(len(q)), q["det_twin"].to_numpy(float) - q["det_epipolar"].to_numpy(float)])
            H14_RESULT["estimate"] = float(_P[1] / _P[0])
            H14_RESULT["ci_lo"], H14_RESULT["ci_hi"] = _ci(_T[:, 1] / _T[:, 0])
            H14_RESULT["verdict"] = verdict_positive(H14_RESULT["ci_lo"], H14_RESULT["ci_hi"])
FTWIN_RECALL = pd.DataFrame(FTWIN_ROWS)
if len(FTWIN_RECALL):
    print(f"\nTwin test with optical flow ({FTWIN_THRESH['flow']}): thresholds at {100 * PREREG['twin_fpr']:.0f}% false positives on "
          f"{FTWIN_THRESH['n_parked']} parked vehicles: |eps| > {FTWIN_THRESH['eps_px']:.2f} px; lowest static-reconstruction height "
          f"outside [{FTWIN_THRESH['h_low_lo_m']:.2f}, {FTWIN_THRESH['h_low_hi_m']:.2f}] m (parked: lowest LiDAR returns "
          f"{FTWIN_THRESH['parked_h_low_true_median_m']:.2f} m above the road, median |x_flow| {FTWIN_THRESH['parked_x_flow_abs_median']:.3f})")
    print(f"moving vehicles with flow: {FTWIN_THRESH['n_moving']}, of which {FTWIN_THRESH['n_no_static_solution']} have no static "
          f"solution (most pixels triangulate behind the camera)")
    with pd.option_context("display.width", 250):
        print(FTWIN_RECALL[["regime", "n", "n_tracks", "det_epipolar", "det_twin", "det_either", "twin_below_road", "n_epi_miss",
                            "twin_given_epi_miss"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    _h = H14_RESULT
    print(f"H14 (registered): parallel regime, twin minus epipolar with the same flow = "
          + (f"{_h['estimate']:+.3f} [{_h['ci_lo']:+.3f}, {_h['ci_hi']:+.3f}]" if np.isfinite(_h["estimate"]) else "–")
          + f" over {_h['n']} vehicles / {_h['n_tracks']} tracks (twin {_h['det_twin']:.3f}, epipolar {_h['det_epipolar']:.3f}): "
          f"{_h['verdict']}")
else:
    print("\nThe twin test with optical flow is not part of this run:", AV2F_STATUS["note"] or "fewer than 20 parked vehicles with flow")
FTWIN_RECALL.to_csv(os.path.join(RESULTS_DIR, "twin_flow_recall_by_regime.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "twin_flow_thresholds.json"), "w") as f:
    json.dump(dict(thresholds=FTWIN_THRESH, H14=H14_RESULT), f, indent=1, default=float)

In [ ]:
# Cell No: 068
# 7. Statistical summary: every registered test (locked P1-P4 and A1, v6 H5-H8, v6.1 H11a-b, v6.5 H12-H13, v6.6 H14), its
#    estimate, interval and verdict; H9 is listed as exploratory (its sample rule changed, D34). For H12 only
#    DUSt3R and MASt3R carry a prediction; the other models are listed with "NO PREDICTION".
SUMMARY_ROWS = []
def _add(prediction, model, statistic, est, lo, hi, n, verdict, p_holm=np.nan, family="locked (v5)"):
    SUMMARY_ROWS.append(dict(family=family, prediction=prediction, model=model, statistic=statistic, estimate=est,
                             ci_lo=lo, ci_hi=hi, p_holm=p_holm, n=n, verdict=verdict))

for r in E1_TABLE.itertuples():
    _add("P1 degeneracy (CONVOY-3D)", r.model, "beta_s", r.beta_s, r.ci_lo, r.ci_hi, r.n, r.verdict, r.p_holm)
for r in REAL_E1.itertuples():
    if r.verdict != "control":
        _add("P1 degeneracy (CONVOY-REAL)", r.model, "beta_s", r.beta_s, r.ci_lo, r.ci_hi, r.n, r.verdict)
for r in E2_TABLE[E2_TABLE["method_key"] != "M5"].itertuples():
    _add("P2 masks (CONVOY-3D)", f"{r.model} / {r.method}", "recall(perp) - recall(par)", r.diff, r.ci_lo, r.ci_hi,
         r.n_par + r.n_perp, r.verdict)
for r in E3_TABLE.itertuples():
    _add("P3a prior weight vs eps", r.model, "Spearman(w, |eps|)", r.spearman_w_eps, r.sp_lo, r.sp_hi, r.n_w, r.verdict_P3a)
    _add("P3b class dependence at eps~0", r.model, "w(car) - w(crate)", r.class_diff, r.cd_lo, r.cd_hi, r.n_parallel, r.verdict_P3b)
for r in E4_SUMMARY.itertuples():
    _add("P4 convoy persistence", r.model, "beta_s(straight,K=8) - beta_s(curved,K=8)", r.drop_K8, r.drop_lo, r.drop_hi,
         r.n_K8, r.verdict)
for r in A1_TABLE.itertuples():
    _add("A1 correspondence (knockout)", r.model, "beta_s(full) - beta_s(knockout)", r.drop, r.drop_lo, r.drop_hi, r.n, r.A1)
_e5_names = {"H5": "H5 lifting raises the geometry share", "H6": "H6 hiding the contact raises it",
             "H7": "H7 the wall acts through the contact", "H8": "H8 lifted objects look farther (single image)"}
for r in E5_TESTS[E5_TESTS["registered"].astype(bool)].itertuples():
    _add(_e5_names[r.test], r.model, r.contrast.split(":")[0], r.estimate, r.ci_lo, r.ci_hi, r.n, r.verdict,
         getattr(r, "p_holm", np.nan), family="registered (v6)")
_h11_names = {"H11a": "H11a real images: lifting raises the geometry share", "H11b": "H11b real images: lifted crates look farther (single image)"}
for r in RABL_TESTS.itertuples() if len(RABL_TESTS) else []:
    _add(_h11_names[r.test], r.model, r.contrast.split(":")[0], r.estimate, r.ci_lo, r.ci_hi, r.n, r.verdict,
         getattr(r, "p_holm", np.nan), family="registered (v6.1)")
_h1213_names = {"H12": "H12 a longer baseline raises the geometry share (baseline sweep)",
                "H13": "H13 hiding the contact with the lowest wall raises it (dose-response)"}
for _df in (BSW_TESTS, WDR_TESTS):
    for r in (_df[_df["test"].isin(["H12", "H13"])].itertuples() if len(_df) else []):
        _add(_h1213_names[r.test], r.model, r.contrast, r.estimate, r.ci_lo, r.ci_hi, r.n, r.verdict,
             getattr(r, "p_holm", np.nan), family="registered (v6.5)")
if H14_RESULT:
    _add("H14 twin test with flow flags more parallel movers than the epipolar test (Argoverse 2)",
         f"optical flow ({FTWIN_THRESH.get('flow', '').split(' (')[0] or 'flow'})", "det_twin - det_epipolar (parallel regime)",
         H14_RESULT["estimate"], H14_RESULT["ci_lo"], H14_RESULT["ci_hi"], H14_RESULT["n"], H14_RESULT["verdict"],
         family="registered (v6.6)")
for r in E6_TABLE[E6_TABLE["verdict"] != "control"].itertuples():
    _add("H9 driving: beta_geo < 0.3 (Virtual KITTI 2)", r.model + (" (trained on Virtual KITTI)" if r.trained_on_vkitti == "yes" else ""),
         "beta_geo", r.beta_geo, r.ci_lo, r.ci_hi, r.n, r.verdict, family="exploratory (v6.1 sample rule)")
SUMMARY = pd.DataFrame(SUMMARY_ROWS)
with pd.option_context("display.max_colwidth", 60, "display.width", 250):
    print(SUMMARY.drop(columns=["family"]).to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
SUMMARY.to_csv(os.path.join(RESULTS_DIR, "prediction_summary.csv"), index=False)
print("\nVerdict counts:")
print(SUMMARY.groupby(["prediction", "verdict"]).size().unstack(fill_value=0).to_string())

In [ ]:
# Cell No: 069
# 8. Robustness of beta_s (E1 movers): camera heading, mover class, depth,
#    focal length and translational parallax; and the exploratory regime s > 1, where no
#    static point explains the two views (Z* < 0).
ROBUST_ROWS = []
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN")]
    dp = d[e1_mask(d)].copy()
    dp["depth_tercile"] = pd.qcut(dp["gt_Z_true"].rank(method="first"), 3, labels=["near", "mid", "far"])
    dp["focal_tercile"] = pd.qcut(dp["focal"].rank(method="first"), 3, labels=["short f", "mid f", "long f"])
    for factor in ["cam_kind", "cls", "depth_tercile", "focal_tercile"]:
        for level, g in dp.groupby(factor, observed=True):
            r = slope_with_ci(g["x_par"], g["rho"], g["scene"], seed_for("rob", model, factor, str(level)))
            ROBUST_ROWS.append(dict(model=model, factor=factor, level=str(level), n=r["n"], beta_s=r["beta"],
                                    ci_lo=r["ci_lo"], ci_hi=r["ci_hi"]))
ROBUST = pd.DataFrame(ROBUST_ROWS)
_piv = ROBUST.pivot_table(index=["factor", "level"], columns="model", values="beta_s")
print("beta_s by condition (rows) and model (columns):")
print(_piv.to_string(na_rep="–", float_format=lambda v: f"{v:.2f}"))
ROBUST.to_csv(os.path.join(RESULTS_DIR, "robustness.csv"), index=False)

# exploratory: s > 1 (the static explanation lies behind the camera)
BEYOND_ROWS = []
for model in EVALUATED_MODELS:
    d = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & (MOV_RES["s"] > 1)
                & (MOV_RES["regime"] == "parallel") & MOV_RES["measurable"].astype(bool)]
    if len(d) < 5:
        continue
    diff, lo, hi, _ = boot_stat(d, "scene", lambda c: float(np.median(c["rho"] - c["rho_mono"])), seed_for("beyond", model),
                                cols=["rho", "rho_mono"])
    BEYOND_ROWS.append(dict(model=model, n=len(d), median_rho=float(d["rho"].median()),
                            median_rho_mono=float(d["rho_mono"].median()), median_rho_minus_mono=diff, ci_lo=lo, ci_hi=hi,
                            frac_placed_farther_than_mono=float(np.mean(d["rho"] > d["rho_mono"]))))
BEYOND = pd.DataFrame(BEYOND_ROWS)
print("\nExploratory: parallel movers with s > 1 (no static explanation in front of the camera):")
print(BEYOND.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}") if len(BEYOND) else "  (too few movers)")
BEYOND.to_csv(os.path.join(RESULTS_DIR, "s_beyond_one.csv"), index=False)

In [ ]:
# Cell No: 070
# 9. Failure analysis: where do models depart from the static-equivalent depth?
#    Residual of each E1 mover from its model's fitted line, related (Spearman) to ground-truth
#    covariates; the largest residuals are listed for inspection.
FAIL_ROWS, WORST = [], []
for model in EVALUATED_MODELS:
    d = E1_DATA[model].copy()
    d = d[np.isfinite(d["rho"])]
    if len(d) < 10:
        continue
    b, a, _ = ols_fit(d["x_par"], d["rho"])
    d["abs_resid"] = np.abs(d["rho"] - (b * d["x_par"] + a))
    d["dev_from_geometry"] = np.abs(d["rho"] - d["x_par"])
    for cov in ["gt_parallax_med", "gt_Z_true", "s"]:
        rs = spearmanr(d[cov], d["dev_from_geometry"]).correlation
        FAIL_ROWS.append(dict(model=model, covariate=cov, spearman_with_deviation=float(rs)))
    for cls_, g in d.groupby("cls"):
        FAIL_ROWS.append(dict(model=model, covariate=f"class={cls_}", spearman_with_deviation=np.nan,
                              median_deviation=float(g["dev_from_geometry"].median()), n=len(g)))
    w = d.sort_values("dev_from_geometry", ascending=False).head(3)
    for r in w.itertuples():
        WORST.append(dict(model=model, uid=r.uid, cls=r.cls, cam_kind=r.cam_kind, s=r.s, parallax_px=r.gt_parallax_med,
                          Z=r.gt_Z_true, rho=r.rho, x_par=r.x_par, rho_mono=r.rho_mono))
FAILURE = pd.DataFrame(FAIL_ROWS, columns=["model", "covariate", "spearman_with_deviation", "median_deviation", "n"])
WORST = pd.DataFrame(WORST)
print("Spearman correlation of |rho - x| (deviation from the static-equivalent depth) with covariates:")
print(FAILURE[FAILURE["covariate"].isin(["gt_parallax_med", "gt_Z_true", "s"])]
      .pivot(index="covariate", columns="model", values="spearman_with_deviation").to_string(na_rep="–", float_format=lambda v: f"{v:.2f}"))
print("\nMedian deviation by class:")
print(FAILURE[FAILURE["covariate"].str.startswith("class=")]
      .pivot(index="covariate", columns="model", values="median_deviation").to_string(na_rep="–", float_format=lambda v: f"{v:.3f}"))
print("\nLargest deviations (3 per model):")
print(WORST.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.3f}"))
FAILURE.to_csv(os.path.join(RESULTS_DIR, "failure_covariates.csv"), index=False)
WORST.to_csv(os.path.join(RESULTS_DIR, "largest_deviations.csv"), index=False)

In [ ]:
# Cell No: 071
# 10.0 Figure style. Exact CVPR sizes: double-column figures 5.5 x 1.8 in, single-column
#      2.8 x 2.0 in, saved WITHOUT a tight bounding box so the files have exactly these sizes.
#      Text is set for print at 100% scale (5-7 pt), in regular weight only: no bold anywhere,
#      panel labels included (mathtext bold is mapped to the regular face as well). Fonts are
#      embedded as TrueType (Type 42). Every figure passes an automatic layout check before it is
#      saved: no text outside the canvas, no overlapping text, no legend covering plotted data.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as _fm
from matplotlib.lines import Line2D
from matplotlib.text import Text
from IPython.display import display, Image as _IPyImage

FIG_DOUBLE, FIG_SINGLE = (5.5, 1.8), (2.8, 2.0)
_installed = {f.name for f in _fm.fontManager.ttflist}
FIG_FONT = next((f for f in ["Arial", "Helvetica", "Liberation Sans", "TeX Gyre Heros", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
                 if f in _installed), "DejaVu Sans")
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [FIG_FONT, "DejaVu Sans"], "font.weight": "normal",
    "font.size": 6.5, "axes.titlesize": 6.5, "axes.labelsize": 6.5, "axes.titleweight": "normal",
    "axes.labelweight": "normal", "figure.titleweight": "normal",
    "xtick.labelsize": 5.8, "ytick.labelsize": 5.8, "legend.fontsize": 5.6, "legend.frameon": True,
    "legend.framealpha": 0.92, "legend.facecolor": "white", "legend.edgecolor": "none", "legend.fancybox": False,
    "legend.borderpad": 0.25,
    "legend.title_fontsize": 5.6, "legend.handlelength": 1.4, "legend.handletextpad": 0.4,
    "legend.borderaxespad": 0.2, "legend.columnspacing": 0.8, "legend.labelspacing": 0.25,
    "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4, "xtick.major.size": 2.0, "ytick.major.size": 2.0, "xtick.minor.size": 1.2,
    "ytick.minor.size": 1.2, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5, "axes.labelpad": 1.5,
    "axes.titlepad": 2.5, "lines.linewidth": 0.9, "lines.markersize": 3.0, "lines.markeredgewidth": 0.5,
    "errorbar.capsize": 1.2, "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 200, "savefig.dpi": 600, "pdf.fonttype": 42, "ps.fonttype": 42, "axes.grid": False,
    "mathtext.fontset": "custom", "mathtext.rm": FIG_FONT, "mathtext.it": f"{FIG_FONT}:italic",
    "mathtext.bf": FIG_FONT, "mathtext.sf": FIG_FONT, "mathtext.cal": FIG_FONT, "mathtext.default": "regular",
    "image.interpolation": "nearest", "savefig.facecolor": "white", "figure.facecolor": "white",
})
MODEL_COLORS = {"DUSt3R": "#0072B2", "MASt3R": "#56B4E9", "MonST3R": "#D55E00",
                "VGGT-1B": "#009E73", "pi3": "#CC79A7", "MapAnything": "#E69F00"}
MODEL_MARKERS = {"DUSt3R": "o", "MASt3R": "o", "MonST3R": "o", "VGGT-1B": "s", "pi3": "s", "MapAnything": "s"}
MODEL_LABELS = {"pi3": "π³", "VGGT-1B": "VGGT"}
CONTROL_COLOR, MONO_COLOR, GRID_COLOR = "#000000", "#8C8C8C", "#D9D9D9"
SUPPORT_SHADE = "#E8F2E8"
PAR_SHADE, PERP_SHADE = "#E6EEF7", "#FBEDE6"
REGIME_COLORS = {"parallel": "#0072B2", "intermediate": "#999999", "perpendicular": "#D55E00"}
_extra_palette = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442"]
for _i, _m in enumerate([m for m in EVALUATED_MODELS if m not in MODEL_COLORS]):
    MODEL_COLORS[_m] = _extra_palette[_i % len(_extra_palette)]
    MODEL_MARKERS[_m] = "o" if MODEL_FAMILY.get(_m) == "croco" else "s"
mcol = lambda m: MODEL_COLORS.get(m, "#666666")
mlab = lambda m: MODEL_LABELS.get(m, m)

def gate_caption_note(*testbeds):
    # caption sentence naming the models left out of a figure because they fail the static-scene gate
    parts = []
    for tb in testbeds:
        no = [mlab(m) for m in EVALUATED_MODELS if m not in MODELS_OK[tb] and (m, tb) in GATE_PASS]
        if no:
            parts.append(f"{', '.join(no)} ({TESTBED_NAMES[tb]})")
    return (" Not shown because they fail the static-scene gate: " + "; ".join(parts) + ".") if parts else ""

def panel_label(ax, s, dx=-0.02, dy=1.02, ha="right", fig=None):
    # "(a)", "(b)", ... in regular weight, slightly larger than the body text
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=7, fontweight="normal", va="bottom", ha=ha)

def fig_panel_label(fig, x, y, s):
    fig.text(x, y, s, fontsize=7, fontweight="normal", va="top", ha="left")

# v6.5 (D60): every panel carries a short title "(a) Title" in regular weight, matching the bold lead of its caption
PANEL_TITLE_FS = 6.2

def panel_title(ax, label, title, x=0.0, pad=2.5, loc="left"):
    ax.set_title(f"({label}) {title}" if label else title, fontsize=PANEL_TITLE_FS, loc=loc, pad=pad, x=x)

def fig_panel_title(fig, x, y, label, title, ha="left"):
    fig.text(x, y, f"({label}) {title}" if label else title, fontsize=PANEL_TITLE_FS, va="top", ha=ha)

def title_axes(fig, rect, label, title, x=0.0):
    # an invisible axes that only carries a panel title: an image or schematic panel then shares the title line of the
    # plot panels whose axes have the same top
    a = fig.add_axes(rect)
    a.set_xticks([]); a.set_yticks([])
    a.set_axis_off()
    panel_title(a, label, title, x=x)
    return a

def empty_panel(ax, msg):
    # a panel without data in this run says so instead of showing empty axes
    ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha="center", va="center", fontsize=5.2, color="#7F7F7F")
    ax.set_yticks([])

def _text_polygon(t, rend):
    # exact outline of a text in display units: the axis-aligned box for 0/90 degree text, and
    # the rotated rectangle (same centre, unrotated width/height) for slanted tick labels
    bb = t.get_window_extent(rend)
    box = np.array([[bb.x0, bb.y0], [bb.x1, bb.y0], [bb.x1, bb.y1], [bb.x0, bb.y1]])
    ang = np.deg2rad(t.get_rotation())
    s = t.get_text()
    if abs(np.sin(2 * ang)) < 1e-6 or "\n" in s or "$" in s:
        return box
    w, h, _ = rend.get_text_width_height_descent(s, t.get_fontproperties(), ismath=False)
    c = np.array([(bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2])
    u, v = np.array([np.cos(ang), np.sin(ang)]), np.array([-np.sin(ang), np.cos(ang)])
    return np.array([c + sx * w / 2 * u + sy * h / 2 * v for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))])

def _polygons_overlap(a, b, tol=1.0):
    # separating-axis test for two convex quadrilaterals; overlap must exceed tol pixels on every axis
    for poly in (a, b):
        for i in range(4):
            e = poly[(i + 1) % 4] - poly[i]
            n = np.array([-e[1], e[0]]); nn = np.linalg.norm(n)
            if nn == 0:
                continue
            n = n / nn
            pa, pb = a @ n, b @ n
            if min(pa.max(), pb.max()) - max(pa.min(), pb.min()) <= tol:
                return False
    return True

def _data_points(ax):
    # display coordinates of everything plotted in an axes (line vertices densified along segments,
    # markers, scatter offsets); reference lines are included, images and shaded spans are not
    pts = []
    for ln in ax.get_lines():
        if not ln.get_visible():
            continue
        if not all(ln.get_transform().contains_branch_seperately(ax.transData)):
            continue                                       # axhline / axvline reference lines span the axes
        xy = np.asarray(ln.get_xydata(), float)
        xy = xy[np.isfinite(xy).all(1)] if xy.ndim == 2 else np.zeros((0, 2))
        if not len(xy):
            continue
        P = ln.get_transform().transform(xy)
        if len(P) > 1 and ln.get_linestyle() not in ("None", "", " ", "none"):
            t = np.linspace(0, 1, 24)[:, None, None]
            P = np.concatenate([P, (P[:-1][None] * (1 - t) + P[1:][None] * t).reshape(-1, 2)])
        pts.append(P)
    for coll in ax.collections:
        if not coll.get_visible():
            continue
        try:
            offs = np.asarray(coll.get_offsets(), float)
            if len(offs) and np.isfinite(offs).all():
                pts.append(coll.get_offset_transform().transform(offs))
            for path in coll.get_paths():                 # errorbar whiskers (LineCollection)
                v = path.vertices
                if len(v) == 2 and np.isfinite(v).all():
                    P = coll.get_transform().transform(v)
                    t = np.linspace(0, 1, 12)[:, None]
                    pts.append(P[0] * (1 - t) + P[1] * t)
        except Exception:
            pass
    return np.concatenate(pts) if pts else np.zeros((0, 2))

def _legend_hits(leg, axes, rend):
    bb = leg.get_window_extent(rend).expanded(1.0, 1.0)
    n = 0
    for ax in axes:
        P = _data_points(ax)
        if len(P):
            n += int(((P[:, 0] > bb.x0 + 1) & (P[:, 0] < bb.x1 - 1) & (P[:, 1] > bb.y0 + 1) & (P[:, 1] < bb.y1 - 1)).sum())
    return n

_LEG_LOCS = ["upper left", "upper right", "lower left", "lower right", "center left", "center right", "upper center",
             "lower center", "center"]

def _relocate_legends(fig):
    # an axes legend that covers plotted data is moved to the first standard position inside the axes where it
    # covers none (it stays where it was if every position covers some)
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or not leg.get_visible() or _legend_hits(leg, [ax], rend) == 0:
            continue
        best = (None, _legend_hits(leg, [ax], rend))
        for loc in _LEG_LOCS:
            leg.set_bbox_to_anchor(ax.bbox)
            if hasattr(leg, "set_loc"):
                leg.set_loc(loc)
            else:
                leg._loc = leg.codes[loc]
            fig.canvas.draw()
            h = _legend_hits(leg, [ax], rend)
            if h < best[1]:
                best = (loc, h)
            if h == 0:
                break
        if best[0] is not None and _legend_hits(leg, [ax], rend) != 0:
            if hasattr(leg, "set_loc"):
                leg.set_loc(best[0])
            else:
                leg._loc = leg.codes[best[0]]

def _legend_overlaps(fig, rend):
    out = []
    legends = [(ax.get_legend(), [ax]) for ax in fig.axes if ax.get_legend() is not None]
    legends += [(lg, fig.axes) for lg in fig.legends]
    for leg, axes in legends:
        if not leg.get_visible():
            continue
        bb = leg.get_window_extent(rend).expanded(1.0, 1.0)
        for ax in axes:
            P = _data_points(ax)
            if not len(P):
                continue
            inside = (P[:, 0] > bb.x0 + 1) & (P[:, 0] < bb.x1 - 1) & (P[:, 1] > bb.y0 + 1) & (P[:, 1] < bb.y1 - 1)
            if inside.any():
                out.append((leg.get_texts()[0].get_text()[:16] if leg.get_texts() else "legend", int(inside.sum())))
    return out

def _layout_check(fig):
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    W, H = fig.bbox.width, fig.bbox.height
    hidden = set()
    for ax in fig.axes:                      # tick labels outside the view limits are not drawn
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            lo, hi = min(lim), max(lim)
            for tk in axis.get_major_ticks() + axis.get_minor_ticks():
                loc = tk.get_loc()
                if loc is None or not (lo - 1e-9 * abs(hi - lo) <= loc <= hi + 1e-9 * abs(hi - lo)):
                    hidden.update([id(tk.label1), id(tk.label2)])
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip() and id(t) not in hidden
             and t.get_window_extent(rend).width > 0]
    bold = [t.get_text()[:20] for t in texts if t.get_fontweight() not in ("normal", 400, "regular", "book")]
    boxes = [t.get_window_extent(rend) for t in texts]
    outside = [t.get_text() for t, b in zip(texts, boxes) if b.x0 < -1 or b.y0 < -1 or b.x1 > W + 1 or b.y1 > H + 1]
    polys = [_text_polygon(t, rend) for t in texts]
    overlaps = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            a, b = boxes[i], boxes[j]
            ix = min(a.x1, b.x1) - max(a.x0, b.x0); iy = min(a.y1, b.y1) - max(a.y0, b.y0)
            if ix > 1.0 and iy > 1.0 and _polygons_overlap(polys[i], polys[j]):
                overlaps.append((texts[i].get_text()[:20], texts[j].get_text()[:20]))
    return outside, overlaps, _legend_overlaps(fig, rend), bold

FIGURE_MANIFEST = []

# ---- captions (v6.3): every panel of a caption opens with a bold lead phrase ----------------------
# Captions are written in light Markdown: "(a) **Lead.** text". The notebook shows them rendered;
# figures.json keeps the Markdown, a plain-text and a LaTeX version, and captions.tex (Section 13)
# holds ready-to-paste \caption{...} lines with \textbf lead-ins. The figure graphics themselves stay in
# regular weight (D32).
def cap(label, lead, text=""):
    return f"({label}) **{lead}.** {text}".rstrip()

_TEX_TOKENS = [(r"\brho - rho_mono = w \(x - rho_mono\)", r"$\\rho-\\rho_{\\mathrm{mono}}=w\\,(x-\\rho_{\\mathrm{mono}})$"),
               (r"\bw = c_support d\^2/\(d\^2 \+ kappa\^2\)", r"$w=c_{\\mathrm{support}}\\,d^2/(d^2+\\kappa^2)$"),
               (r"\|x - rho_mono\|", r"$|x-\\rho_{\\mathrm{mono}}|$"), (r"\|eps\|", r"$|\\varepsilon|$"),
               (r"(?<![\w$])-s/\(1-s\)", r"$-s/(1-s)$"), (r"\bH s/\(1-s\)", r"$H s/(1-s)$"),
               (r"0\.85 (<=|≤) s < 1\.15", r"$0.85 \\leq s < 1.15$"), (r"0\.85 (<=|≤) s < 1\b", r"$0.85 \\leq s < 1$"),
               (r"\bbeta_s\b", r"$\\beta_s$"), (r"\bbeta_geo\b", r"$\\beta_{\\mathrm{geo}}$"), (r"\bbeta\(w\)", r"$\\beta(w)$"),
               (r"\bx_geo\b", r"$x_{\\mathrm{geo}}$"), (r"\bZ_geo\b", r"$Z_{\\mathrm{geo}}$"), (r"\brho_mono\b", r"$\\rho_{\\mathrm{mono}}$"),
               (r"\bx = -log\(1-s\)", r"$x=-\\log(1-s)$"), (r"\bX/\(1-s\)", r"$X/(1-s)$"), (r"\bZ/\(1-s\)", r"$Z/(1-s)$"),
               (r"\bs b\b", r"$s\\,\\mathbf{b}$"), (r"(?<![\w$])s >= 1\b", r"$s \\geq 1$"), (r"(?<![\w$])s < 1\b", r"$s < 1$"),
               (r" <= ", r" $\\leq$ "), (r" >= ", r" $\\geq$ "), (r"\bkappa\b", r"$\\kappa$"), (r"\beps\b", r"$\\varepsilon$"),
               (r"\brho\b", r"$\\rho$")]
_TEX_CHARS = [("π³", r"$\pi^3$"), ("β_s", r"$\beta_s$"), ("β", r"$\beta$"), ("ε", r"$\varepsilon$"), ("ρ", r"$\rho$"), ("≥", r"$\geq$"), ("≤", r"$\leq$"),
              ("−", "$-$"), ("–", "--"), ("×", r"$\times$"), ("°", r"$^\circ$"), ("δ", r"$\delta$"), ("≈", r"$\approx$"),
              ("·", r"$\cdot$"), ("σ", r"$\sigma$"), ("ω", r"$\omega$"), ("ψ", r"$\psi$"), ("³", r"$^3$"), ("²", r"$^2$")]

def caption_plain(s):
    return re.sub(r"\*\*(.+?)\*\*", r"\1", s)

def caption_tex(s):
    out = re.sub(r"\*\*(.+?)\*\*", lambda m_: "\x01" + m_.group(1) + "\x02", s)
    for a, b in (("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("#", r"\#")):
        out = out.replace(a, b)
    def _outside_math(fn, text):
        parts = re.split(r"(\$[^$]*\$)", text)              # math segments stay as they are
        return "".join(fn(p_) if k % 2 == 0 else p_ for k, p_ in enumerate(parts))
    for pat, rep in _TEX_TOKENS:
        out = _outside_math(lambda t, pat=pat, rep=rep: re.sub(pat, rep, t), out)
    for a, b in _TEX_CHARS:
        out = _outside_math(lambda t, a=a, b=b: t.replace(a, b), out)
    parts = re.split(r"(\$[^$]*\$)", out)
    for k in range(0, len(parts), 2):
        parts[k] = parts[k].replace("_", r"\_").replace("<", "$<$").replace(">", "$>$").replace("^", r"\^{}")
        parts[k] = re.sub(r"(?<=[0-9)])-(?=[0-9(])", "--", parts[k])          # number and panel ranges: en dash
    return "".join(parts).replace("\x01", r"\textbf{").replace("\x02", "}")

def _show_caption(title, caption):
    try:
        from IPython.display import Markdown as _Md
        display(_Md(f"**{title}.** {caption}"))
    except ImportError:
        print(f"{title}. {caption_plain(caption)}")

def save_figure(fig, fig_id, title, question, source, placement, data, caption=""):
    _relocate_legends(fig)
    outside, overlaps, legend_hits, bold = _layout_check(fig)
    w, h = fig.get_size_inches()
    pdf, png = os.path.join(FIGURE_DIR, f"{fig_id}.pdf"), os.path.join(FIGURE_DIR, f"{fig_id}.png")
    fig.savefig(pdf); fig.savefig(png, dpi=300)
    with open(os.path.join(FIGURE_DIR, f"{fig_id}_data.json"), "w") as f:
        json.dump(data, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    FIGURE_MANIFEST.append(dict(figure=fig_id, title=title, question=question, source=source, placement=placement,
                                caption=caption, caption_plain=caption_plain(caption),
                                caption_tex=f"\\textbf{{{caption_tex(title)}.}} {caption_tex(caption)}",
                                size_in=f"{w:.2f} x {h:.2f}", text_outside=len(outside),
                                text_overlaps=len(overlaps), legend_over_data=len(legend_hits), bold_text=len(bold),
                                pdf=pdf, png=png))
    plt.close(fig)                       # the figure is shown from its saved PNG, not drawn a second time (D32)
    del fig
    display(_IPyImage(filename=png, width=int(round(w * 150))))
    release_memory()
    ok = not outside and not overlaps and not legend_hits and not bold
    qa = "PASS" if ok else (f"CHECK (outside: {outside[:3]}, overlaps: {overlaps[:3]}, legend over data: {legend_hits[:3]}, "
                            f"bold: {bold[:3]})")
    print(f"{fig_id}: {w:.2f} x {h:.2f} in, font {FIG_FONT}, layout QA {qa}")
    if caption:
        _show_caption(title, caption)

def clean_image_axes(a, frame=True, lw=0.4):
    a.set_xticks([]); a.set_yticks([])
    for sp in a.spines.values():
        sp.set_visible(frame); sp.set_linewidth(lw)

def tag(a, s, loc="lower left", color="white", fs=5.0):
    x, ha = (0.03, "left") if "left" in loc else (0.97, "right")
    y, va = (0.04, "bottom") if "lower" in loc else (0.96, "top")
    a.text(x, y, s, transform=a.transAxes, fontsize=fs, color=color, va=va, ha=ha,
           bbox=dict(boxstyle="square,pad=0.14", fc="black", ec="none", alpha=0.55))

def crop_around(img, bbox, out_hw, pad=0.35):
    # crop an image around a bounding box (u0, v0, u1, v1) with relative padding, fixed aspect
    H, W = img.shape[:2]
    u0, v0, u1, v1 = bbox
    cu, cv = (u0 + u1) / 2, (v0 + v1) / 2
    h = (v1 - v0) * (1 + 2 * pad); w = (u1 - u0) * (1 + 2 * pad)
    ar = out_hw[1] / out_hw[0]
    if w / max(h, 1) > ar:
        h = w / ar
    else:
        w = h * ar
    h, w = min(h, H), min(w, W)
    a0 = int(np.clip(cu - w / 2, 0, W - w)); b0 = int(np.clip(cv - h / 2, 0, H - h))
    return img[b0:b0 + int(h), a0:a0 + int(w)]

def mask_bbox(mask, margin=0):
    vv, uu = np.nonzero(mask)
    return (uu.min() - margin, vv.min() - margin, uu.max() + margin, vv.max() + margin) if len(vv) else None

def depth_to_rgb(d, vmin=None, vmax=None, cmap="turbo"):
    # colour encodes log depth between the limits vmin, vmax (log metres): near = warm, far = cool;
    # missing or infinite depth (sky) is white
    ld = np.log(np.clip(np.nan_to_num(d, nan=np.nan, posinf=np.nan), 1e-3, None))
    fin = np.isfinite(ld)
    vmin = np.nanpercentile(ld[fin], 2) if vmin is None else vmin
    vmax = np.nanpercentile(ld[fin], 98) if vmax is None else vmax
    x = 1.0 - np.clip((ld - vmin) / max(vmax - vmin, 1e-6), 0, 1)
    rgb = plt.get_cmap(cmap)(np.clip(np.nan_to_num(x, nan=0.0), 0.04, 0.96))[..., :3]
    rgb[~fin] = 1.0
    return rgb

def ci_err(est, lo, hi):
    # asymmetric error-bar lengths, or None when the interval is missing
    return [[est - lo], [hi - est]] if np.isfinite(lo) and np.isfinite(hi) and np.isfinite(est) else None

print(f"Figure style ready: double column {FIG_DOUBLE} in, single column {FIG_SINGLE} in, font {FIG_FONT}, regular weight only.")

In [ ]:
# Cell No: 072
# Figure 1 (teaser, double column). (a) Parallel motion as an exact counterfactual: a point moving
# by s*b and a static point at X/(1-s) give the same pixels in both views. (b) A parallel mover in
# CONVOY-3D: both frames, true depth and a model's depth on the same colour scale. (c) The probe:
# log depth error rho against the geometric answer x = -log(1-s) (binned medians): geometry alone
# gives the diagonal (beta = 1), the single-image prior gives zero (beta = 0).
def _pick_showcase(model):
    base = MOV_RES[(MOV_RES["model"] == model) & (MOV_RES["set"] == "MAIN") & MOV_RES["scene"].isin(EXEMPLAR_MAIN)]
    for cond in [(base["motion_type"] == "parallel") & base["measurable"].astype(bool) & base["identifiable"].astype(bool)
                 & (base["s"] > 0.25) & (base["s"] < 0.9),
                 base["motion_type"] == "parallel", base["s"].notna()]:
        cand = base[cond]
        if len(cand):
            return cand.sort_values("gt_area_px", ascending=False).iloc[0]

_M1 = MODELS_OK["MAIN"] or EVALUATED_MODELS
_showcase_model = next((m for m in ["MASt3R", "DUSt3R", "VGGT-1B", "pi3", "MonST3R"] if m in _M1), _M1[0])
_sc_row = _pick_showcase(_showcase_model)
_meta = META_MAIN[int(_sc_row["scene"])]
_mv = _meta["scene"]["movers"][int(_sc_row["mover"])]
_r0 = load_frame(_meta["dir"], 0); _rgb1 = load_rgb(_meta["dir"], 1)
_pred = cv2.resize(EXEMPLARS[_showcase_model][f"MAIN-{int(_sc_row['scene'])}|depth"], (_r0["depth"].shape[1], _r0["depth"].shape[0]),
                   interpolation=cv2.INTER_LINEAR)
_bb = mask_bbox(_r0["mover_id"] == int(_sc_row["mover"])) or _mv["bbox0"]
_fin = np.isfinite(_r0["depth"])
_vmin = np.log(np.nanpercentile(_r0["depth"][_fin], 2)); _vmax = np.log(np.nanpercentile(_r0["depth"][_fin], 97))

fig = plt.figure(figsize=FIG_DOUBLE)
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.5, 1.3], left=0.012, right=0.985, bottom=0.2, top=0.9, wspace=0.28)
# (a) schematic, top view: forward (camera translation) to the right, lateral upwards
ax = fig.add_subplot(gs[0]); ax.set_anchor("W")
fig_panel_title(fig, 0.004, 0.985, "a", "The counterfactual")
s_, b_ = 0.5, 2.0
X = np.array([4.0, 2.2]); Xm = X + np.array([s_ * b_, 0.0]); Xs = X / (1 - s_)
C1, C2 = np.array([0.0, 0.0]), np.array([b_, 0.0])
for C in (C1, C2):
    ax.plot([C[0], Xs[0]], [C[1], Xs[1]], color="#7F7F7F", lw=0.55, ls=(0, (3, 2)))
ax.annotate("", xy=C2, xytext=C1, arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#0072B2", mutation_scale=6,
                                                  shrinkA=0, shrinkB=2))
ax.annotate("", xy=Xm, xytext=X, arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#D55E00", mutation_scale=6,
                                                 shrinkA=2, shrinkB=2))
for P in (C1, C2):
    ax.add_patch(mpatches.Polygon([[P[0] - 0.32, P[1] - 0.36], [P[0] - 0.32, P[1] + 0.36], [P[0] + 0.3, P[1]]],
                                  closed=True, fc="#0072B2", ec="none"))
ax.plot(*X, "o", color="#D55E00", ms=3.2); ax.plot(*Xm, "o", mfc="white", mec="#D55E00", ms=3.2, mew=0.7)
ax.plot(*Xs, "*", color="black", ms=5.5)
ax.text(C1[0], C1[1] - 0.95, "$C_1$", fontsize=6, ha="center")
ax.text(C2[0], C2[1] - 0.95, "$C_2$", fontsize=6, ha="center")
ax.text(1.0, -0.62, "$b$", fontsize=6, color="#0072B2", ha="center")
ax.text(X[0] - 0.35, X[1] + 0.5, "$X$", fontsize=6, color="#D55E00", ha="center")
ax.text(Xm[0] + 1.25, Xm[1] - 0.55, "$X + sb$", fontsize=6, color="#D55E00", ha="center")
ax.text(Xs[0] - 0.3, Xs[1] + 0.55, "static $X/(1-s)$", fontsize=6, ha="center")
ax.set_xlim(-0.8, 9.8); ax.set_ylim(-1.35, 5.6); ax.set_aspect("equal")
clean_image_axes(ax, frame=False)
ax.text(0.5, -0.03, "moving and static point:\nthe same pixels in both views", transform=ax.transAxes, fontsize=5.6,
        ha="center", va="top")
# (b) mover crops: frame 1, frame 2, true depth, model depth
sub = gs[1].subgridspec(2, 2, wspace=0.04, hspace=0.05)
tiles = [(crop_around(_r0["rgb"], _bb, (3, 4)), "frame 1"), (crop_around(_rgb1, _bb, (3, 4)), "frame 2"),
         (crop_around(depth_to_rgb(_r0["depth"], _vmin, _vmax), _bb, (3, 4)), "true depth"),
         (crop_around(depth_to_rgb(_pred, _vmin, _vmax), _bb, (3, 4)), f"{mlab(_showcase_model)}, two views")]
for k, (img, lab) in enumerate(tiles):
    a2 = fig.add_subplot(sub[k // 2, k % 2])
    a2.imshow(img); clean_image_axes(a2); tag(a2, lab)
_ax_b = fig.axes[1]
fig_panel_title(fig, _ax_b.get_position().x0 - 0.012, 0.985, "b", "A parallel mover")
fig.text((fig.axes[1].get_position().x0 + fig.axes[2].get_position().x1) / 2, 0.075,
         f"parallel mover, s = {_mv['s']:.2f}: true Z = {_sc_row['gt_Z_true']:.1f} m, geometric answer Z/(1 − s) = "
         f"{_sc_row['gt_Z_true'] / (1 - _mv['s']):.1f} m", fontsize=5.4, ha="center")
# (c) the probe
ax = fig.add_subplot(gs[2])
fig_panel_title(fig, ax.get_position().x0 - 0.045, 0.985, "c", "Prior or geometry")
xs = np.linspace(-0.55, 1.75, 10)
ax.plot(xs, xs, color=CONTROL_COLOR, lw=0.7, ls=(0, (4, 2)))
ax.axhline(0, color=MONO_COLOR, lw=0.7, ls=":")
ax.text(0.62, 0.72, "geometry only, β = 1", fontsize=5.2, ha="center", va="bottom", color=CONTROL_COLOR, rotation=45,
        rotation_mode="anchor", transform_rotates_text=True)
ax.text(1.72, -0.06, "prior only, β = 0", fontsize=5.2, ha="right", va="top", color="#666666")
fig1_data = {}
for model in MODELS_OK["MAIN"]:
    d = E1_DATA[model]
    if len(d) < 6:
        continue
    q = pd.qcut(d["x_par"].rank(method="first"), min(6, len(d) // 3), labels=False)
    g = d.groupby(q).agg(x=("x_par", "median"), y=("rho", "median"))
    ax.plot(g["x"], g["y"], marker=MODEL_MARKERS[model], color=mcol(model), lw=0.8, ms=2.6, label=mlab(model))
    fig1_data[model] = g.to_dict("list")
ax.set_xlabel("geometric answer x = −log(1 − s)"); ax.set_ylabel("log depth error ρ")
ax.set_xlim(-0.55, 1.75); ax.set_ylim(-0.7, 1.75)
ax.legend(loc="upper left", ncol=2, fontsize=5.0, handlelength=1.2, columnspacing=0.7)
save_figure(fig, "fig1_teaser", "Parallel motion as an exact counterfactual",
            "How much of the cross-view geometry do feed-forward models use for a moving object?",
            "E1, CONVOY-3D", "double column", dict(binned=fig1_data, showcase=dict(uid=_sc_row["uid"], model=_showcase_model)),
            caption=cap("a", "The counterfactual", "A point moving parallel to the camera baseline by s b and a static point at "
                        "X/(1-s) project to the same pixels in both views.")
                    + " " + cap("b", "A parallel mover", "A mover in CONVOY-3D and the depth a model predicts for it from both views.")
                    + " " + cap("c", "Prior or geometry", "Log depth error against the geometric answer x = -log(1-s), binned medians "
                                "over purely parallel movers: a model that used only geometry would follow the diagonal, a model that "
                                "uses only its single-image prior stays at zero.") + gate_caption_note("MAIN"))

In [ ]:
# Cell No: 073
# Figure 2 (double column): the testbeds and the two checks that make the probe interpretable.
# (a-c) Three testbeds (CONVOY-3D, then CONVOY-REAL and Virtual KITTI 2 when they are part of the run,
# filled up with LEGACY-FLAT and a second CONVOY-3D camera): image with the movers outlined by regime,
# and exact depth. Then the static-scene validity gate and the oracle check (beta_s recovered from
# planted predictors against the slope-weight identity). Panel letters follow the panels drawn.
def _exemplar_scene(movers, metas, prefer_kind=None):
    best, best_n = metas[0], -1
    for meta in metas[: min(len(metas), 30)]:
        mv = movers[movers["scene"] == meta["index"]]
        n = mv["regime"].nunique() + 0.1 * len(mv) + (0.5 if prefer_kind and meta["scene"]["camera"]["kind"] == prefer_kind else 0)
        if n > best_n:
            best, best_n = meta, n
    return best

def _outline(ax, mask, color, lw=0.7):
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=lw)

def _main_panel(meta, name):
    fr = load_frame(meta["dir"], 0)
    rg = {int(r.idx): r.regime for r in MOVERS_MAIN[MOVERS_MAIN["scene"] == meta["index"]].itertuples()}
    return (name, fr["rgb"], depth_to_rgb(fr["depth"]), [(fr["mover_id"] == j, REGIME_COLORS[r]) for j, r in rg.items()])

_tb_panels = [_main_panel(_exemplar_scene(MOVERS_MAIN, META_MAIN, prefer_kind="forward"), "CONVOY-3D")]
if META_REAL:
    _mr = _exemplar_scene(MOVERS_REAL, META_REAL)
    _fr = load_frame(_mr["dir"], 0)
    _rgr = {int(r.idx): r.regime for r in MOVERS_REAL[MOVERS_REAL["scene"] == _mr["index"]].itertuples()}
    _tb_panels.append(("CONVOY-REAL", _fr["rgb"], depth_to_rgb(_fr["depth"]),
                       [(_fr["mover_id"] == j, REGIME_COLORS[r]) for j, r in _rgr.items()]))
if VK_STATUS["available"] and len(MOVERS_VK):
    _vk_best = MOVERS_VK.groupby("scene").agg(n=("regime", "size"), k=("regime", "nunique")).sort_values(["k", "n"]).index[-1]
    _vm = VK_PAIR_META[int(_vk_best)]
    _vs = VK_SCENE_OBJS[_vm["seq"]]; _vg = _vs.pair_gt(*_vm["frames"])
    _rgv = MOVERS_VK[MOVERS_VK["scene"] == _vk_best].set_index("mover")["regime"].to_dict()
    _tb_panels.append(("Virtual KITTI 2", _vs.rgb(_vm["frames"][0]), depth_to_rgb(np.where(np.isfinite(_vg["depth"]), _vg["depth"], 655.0)),
                       [(_vg["mover_id"] == k, REGIME_COLORS[r]) for k, r in _rgv.items()]))
if len(_tb_panels) < 3:
    _fl = META_FLAT[0]
    _tb_panels.append(("LEGACY-FLAT", load_rgb(_fl["dir"], 0), load_rgb(_fl["dir"], 1) / 255.0, []))
if len(_tb_panels) < 3:
    _ms = [mm for mm in META_MAIN if mm["scene"]["camera"]["kind"] == "side"]
    if _ms:
        _tb_panels.append(_main_panel(_exemplar_scene(MOVERS_MAIN, _ms), "CONVOY-3D, side"))
_L2 = "abcdefg"
_n_tb = len(_tb_panels)

fig = plt.figure(figsize=FIG_DOUBLE)
gs = fig.add_gridspec(2, 3, left=0.008, right=0.505, bottom=0.17, top=0.885, wspace=0.035, hspace=0.04)
for c, (name, top, bottom, outlines) in enumerate(_tb_panels):
    a_top = fig.add_subplot(gs[0, c]); a_bot = fig.add_subplot(gs[1, c])
    a_top.imshow(top); a_bot.imshow(bottom)
    for m, col in outlines:
        _outline(a_top, m, col)
    clean_image_axes(a_top); clean_image_axes(a_bot)
    panel_title(a_top, _L2[c], name)
    tag(a_bot, {"LEGACY-FLAT": "second view", "CONVOY-REAL": "depth"}.get(name, "exact depth"), fs=4.8)
fig.legend(handles=[Line2D([], [], color=cc, lw=1.0, label=rg) for rg, cc in REGIME_COLORS.items()],
           loc="lower center", bbox_to_anchor=(0.257, 0.0), ncol=3, fontsize=5.3, handlelength=1.2, columnspacing=1.2)
# (d) gate
ax = fig.add_axes([0.585, 0.24, 0.17, 0.645])
panel_title(ax, _L2[_n_tb], "Validity gate", x=-0.3)
ax.add_patch(mpatches.Rectangle((0, PREREG["gate_slope_range"][0]), PREREG["gate_absrel_max"],
                                PREREG["gate_slope_range"][1] - PREREG["gate_slope_range"][0], fc=SUPPORT_SHADE, ec="none", zorder=0))
ax.text(0.012, PREREG["gate_slope_range"][1] + 0.04, "pass", fontsize=5.2, color="#2E7D32", va="bottom")
TB_MARK = {"MAIN": "o", "ABL": "o", "REAL": "^", "VK": "D", "KT": "v", "AV2": "P", "BONN": "h", "FLAT": "X"}
for r in GATE.itertuples():
    if str(r.set) in ("ABL", "RABL", "BSW", "WDR"):   # same backgrounds as CONVOY-3D and CONVOY-REAL; listed in T2
        continue
    ax.plot(min(r.absrel_median, 0.95), r.slope_median, TB_MARK[str(r.set)], color=mcol(r.model), ms=3.0,
            mec="black" if str(r.set) == "FLAT" else mcol(r.model), mew=0.3, alpha=0.9)
ax.set_xlabel("AbsRel, static pixels"); ax.set_ylabel("log-depth slope, static pixels")
ax.set_xlim(0, 1.0); ax.set_ylim(-0.3, 1.45); ax.set_xticks([0, 0.5, 1.0])
_present = [k for k in ["MAIN", "REAL", "VK", "AV2", "KT", "BONN", "FLAT"] if k in set(GATE["set"].astype(str))]
_tbl = {"MAIN": "CONVOY-3D", "REAL": "CONVOY-REAL", "VK": "VKITTI 2", "AV2": "Argoverse 2", "KT": "KITTI", "BONN": "Bonn",
        "FLAT": "LEGACY-FLAT"}
ax.legend(handles=[Line2D([], [], ls="", marker=TB_MARK[k], color="#555555", mec="black" if k == "FLAT" else "#555555",
                          mew=0.3, ms=3, label=_tbl[k]) for k in _present], loc="upper right", fontsize=4.9, handletextpad=0.2)
# (e) oracle
ax = fig.add_axes([0.835, 0.24, 0.145, 0.645])
panel_title(ax, _L2[_n_tb + 1], "Estimator check", x=-0.3)
ax.plot(SLOPE_WEIGHT_TABLE["w"], SLOPE_WEIGHT_TABLE["beta_ols_over_design"], color=CONTROL_COLOR, lw=0.8, label="identity β(w)")
ax.errorbar(ORACLE_TABLE["w_true"], ORACLE_TABLE["beta_s"],
            yerr=[ORACLE_TABLE["beta_s"] - ORACLE_TABLE["ci_lo"], ORACLE_TABLE["ci_hi"] - ORACLE_TABLE["beta_s"]],
            fmt="o", color="#0072B2", ms=2.8, capsize=1.0, lw=0.7, label="CONVOY-3D")
if len(ORACLE_VK) and ORACLE_VK["beta_geo"].notna().any():
    ax.plot(ORACLE_VK["w_true"], ORACLE_VK["beta_geo"], "D", mfc="white", mec="#D55E00", ms=2.6, mew=0.7, label="VKITTI 2")
ax.set_xlabel("planted weight w"); ax.set_ylabel("recovered β$_s$")
ax.set_xlim(-0.08, 1.08); ax.set_ylim(-0.12, 1.12); ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
ax.legend(loc="lower left", fontsize=4.9, handlelength=1.0)
fig.legend(handles=[Line2D([], [], ls="", marker="o", color=mcol(m), ms=2.8, label=mlab(m)) for m in EVALUATED_MODELS],
           loc="lower center", bbox_to_anchor=(0.66, 0.0), ncol=len(EVALUATED_MODELS), fontsize=5.0, handletextpad=0.1,
           columnspacing=0.7)
_tb_desc = {"CONVOY-3D": "CONVOY-3D (ray-cast streets)",
            "CONVOY-REAL": "CONVOY-REAL (real TUM images and their measured depth, with inserted movers of exact depth)",
            "Virtual KITTI 2": "Virtual KITTI 2 (photo-realistic driving)",
            "LEGACY-FLAT": "LEGACY-FLAT, the 2-D testbed of the previous version (bottom: its second view)",
            "CONVOY-3D, side": "CONVOY-3D with a side-looking camera"}
_tb_list = [_tb_desc[p[0]] for p in _tb_panels]
_cap2 = (f"({_L2[0]})-({_L2[_n_tb - 1]}) **Testbeds.** " + (", ".join(_tb_list[:-1]) + " and " + _tb_list[-1] if _n_tb > 1 else _tb_list[0])
         + "; top: first view with the movers outlined by their exact motion regime, bottom: ground-truth depth. "
         + cap(_L2[_n_tb], "Validity gate", "Medians over images of AbsRel and of the log-depth slope on static pixels; "
               "the shaded box is the pre-registered pass region"))
_gf = GATE[GATE["set"] == "FLAT"]
if len(_gf):
    _nf = int((_gf["gate"] == "FAIL").sum())
    _cap2 += (", and LEGACY-FLAT falls outside it for every model" if _nf == len(_gf)
              else f", and LEGACY-FLAT falls outside it for {_nf} of {len(_gf)} models")
_odev = float(np.nanmax(np.abs(ORACLE_TABLE["beta_s"] - ORACLE_TABLE["beta_theory_identity"])))
_cap2 += (". " + cap(_L2[_n_tb + 1], "Estimator check", "Predictors with a planted prior weight w") +
          (f" recover the slope-weight identity (largest deviation {_odev:.3f}), so beta_s reads as a geometry share" if ORACLE_OK else
           f": recovered beta_s against the slope-weight identity (largest deviation {_odev:.3f}; the estimator check did not pass)")
          + ("; open diamonds: the Virtual KITTI 2 estimator of H9." if len(ORACLE_VK) and ORACLE_VK["beta_geo"].notna().any() else "."))
save_figure(fig, "fig2_testbeds_checks", "Testbeds and the checks that make the probe interpretable",
            "Do the models reconstruct the static scene of each testbed, and does the estimator recover a known answer?",
            "Sections 3, 6.0 and 6.0b", "double column",
            dict(gate=GATE.astype({"set": str}).to_dict("records"), oracle=ORACLE_TABLE.to_dict("records"),
                 oracle_vk=ORACLE_VK.to_dict("records") if len(ORACLE_VK) else []),
            caption=_cap2)

In [ ]:
# Cell No: 074
# Figure 3 (double column): how much geometry the models use. (a) beta_s with 95% CI per model,
# the single-image control and classical triangulation; top axis: the equivalent prior weight w
# (slope-weight identity). (b) A1 knockout: beta_s with the mover deleted from view 2 (rows aligned
# with (a)). (c) beta_s by translational parallax: geometry can only be used where it determines depth.
# A panel whose estimates are not available in this run is left out.
_SW_W = SLOPE_WEIGHT_TABLE["w"].to_numpy(float); _SW_B = SLOPE_WEIGHT_TABLE["beta_ols_over_design"].to_numpy(float)

def _interp_lin(x, xp, fp):
    # np.interp with linear extrapolation beyond both ends (keeps the axis mapping strictly monotone)
    x = np.asarray(x, float); y = np.interp(x, xp, fp)
    lo, hi = x < xp[0], x > xp[-1]
    y = np.where(lo, fp[0] + (x - xp[0]) * (fp[1] - fp[0]) / (xp[1] - xp[0]), y)
    return np.where(hi, fp[-1] + (x - xp[-1]) * (fp[-1] - fp[-2]) / (xp[-1] - xp[-2]), y)

_w_of_beta = lambda b: _interp_lin(b, _SW_B[::-1], _SW_W[::-1])
_beta_of_w = lambda w: _interp_lin(w, _SW_W, _SW_B)
_A1_OK = A1_TABLE[A1_TABLE["model"].isin(MODELS_OK["MAIN"])] if len(A1_TABLE) else A1_TABLE
_PAR_OK = E1_PARALLAX[E1_PARALLAX["model"].isin(MODELS_OK["MAIN"])] if len(E1_PARALLAX) else E1_PARALLAX
_has_a1 = len(_A1_OK) > 0 and np.isfinite(pd.to_numeric(_A1_OK["beta_knockout"], errors="coerce")).any()
_has_par = len(_PAR_OK) > 0 and bool(((_PAR_OK["n"] >= 8) & np.isfinite(_PAR_OK["beta"])).any())
_rows3 = [(r.model, mlab(r.model), r.beta_s, r.ci_lo, r.ci_hi, r.beta_mono, mcol(r.model)) for r in E1_TABLE.itertuples()
          if r.model in MODELS_OK["MAIN"]]
_rows3 += [(None, "triang., exact" if "exact" in k else "triang., flow", v["beta"], v["ci_lo"], v["ci_hi"], np.nan,
            CONTROL_COLOR) for k, v in CONTROL_E1.items()]
_ypos3 = {r[1]: len(_rows3) - 1 - i for i, r in enumerate(_rows3)}

def _forest_frame(a):
    a.axvspan(*PREREG["beta_support_range"], color=SUPPORT_SHADE, zorder=0, lw=0)
    a.axvline(0, color=GRID_COLOR, lw=0.5, zorder=0); a.axvline(1, color=GRID_COLOR, lw=0.5, zorder=0)
    a.set_xlim(-0.22, 1.3); a.set_ylim(-0.7, len(_rows3) - 0.3)

def _fig3_forest(ax):
    _forest_frame(ax)
    for m, lab, b, lo, hi, bm, c in _rows3:
        y = _ypos3[lab]
        if np.isfinite(b):
            ax.errorbar(b, y, xerr=ci_err(b, lo, hi), fmt="o", color=c, ms=3, capsize=1.2, lw=0.8)
        if np.isfinite(bm):
            ax.plot(bm, y, "o", mfc="white", mec=MONO_COLOR, ms=3, mew=0.6)
    ax.set_yticks([_ypos3[r[1]] for r in _rows3]); ax.set_yticklabels([r[1] for r in _rows3])
    ax.set_xlabel("β$_s$ (95% CI)")
    top = ax.secondary_xaxis("top", functions=(_w_of_beta, _beta_of_w))
    top.set_xticks([1.0, 0.8, 0.6, 0.4, 0.0]); top.tick_params(labelsize=5.4, pad=1)
    top.set_xlabel("equivalent prior weight w", fontsize=5.8, labelpad=2)
    ax.legend(handles=[Line2D([], [], ls="", marker="o", color="#444444", ms=3, label="two views"),
                       Line2D([], [], ls="", marker="o", mfc="white", mec=MONO_COLOR, ms=3, label="one view")],
              loc="lower left", fontsize=5.0)

def _fig3_knockout(ax):
    _forest_frame(ax)
    for r in _A1_OK.itertuples():
        y = _ypos3[mlab(r.model)]; c = mcol(r.model)
        if np.isfinite(r.beta_full):
            ax.plot(r.beta_full, y, "o", color=c, ms=2.2, alpha=0.5)
        if np.isfinite(r.beta_knockout):
            ax.errorbar(r.beta_knockout, y, xerr=ci_err(r.beta_knockout, r.ko_lo, r.ko_hi), fmt="s", mfc="white", mec=c, color=c,
                        ms=3, capsize=1.2, lw=0.8, mew=0.7)
    plt.setp(ax.get_yticklabels(), visible=False); ax.tick_params(axis="y", length=0)
    ax.set_xlabel("β$_s$, mover deleted from view 2")
    ax.legend(handles=[Line2D([], [], ls="", marker="s", mfc="white", mec="#444444", ms=3, mew=0.7, label="mover deleted"),
                       Line2D([], [], ls="", marker="o", color="#444444", alpha=0.5, ms=2.2, label="full")],
              loc="lower left", fontsize=5.0)

def _fig3_parallax(ax):
    bins = _PAR_OK[["par_lo", "par_hi"]].drop_duplicates().sort_values("par_lo").values.tolist()
    labs = [(f"<{int(b)}" if a == 0 else f"{int(a)}–{int(b)}") if b < 1e8 else f">{int(a)}" for a, b in bins]
    ax.axhspan(*PREREG["beta_support_range"], color=SUPPORT_SHADE, zorder=0, lw=0)
    n_low = sum(1 for _, b in bins if b <= PREREG["parallax_min_px"])
    if n_low:
        ax.axvspan(-0.5, n_low - 0.5, color="#F2F2F2", zorder=0, lw=0)
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    for model in MODELS_OK["MAIN"]:
        g = _PAR_OK[_PAR_OK["model"] == model].sort_values("par_lo")
        ok = (g["n"] >= 8).values & np.isfinite(g["beta"].values)
        if ok.any():
            ax.plot(np.arange(len(g))[ok], g["beta"].values[ok], marker=MODEL_MARKERS[model], color=mcol(model), ms=2.6, lw=0.8,
                    label=mlab(model))
    ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, fontsize=5.4)
    ax.set_xlabel("translational parallax (px)"); ax.set_ylabel("β$_s$", labelpad=1.0)
    ax.set_ylim(-0.25, 1.3); ax.set_xlim(-0.5, len(labs) - 0.5)
    if n_low:
        ax.text((n_low - 1) / 2, 1.22, "not identifiable", fontsize=5.0, color="#666666", ha="center", va="top")
    ax.yaxis.tick_right(); ax.yaxis.set_label_position("right")
    ax.spines["right"].set_visible(True); ax.spines["left"].set_visible(False)
    # no legend: the curves carry the model colours of (a) (a six-entry legend did not fit beside six curves)

_wr = [1.3] + ([0.78] if _has_a1 else []) + ([1.15] if _has_par else [])
fig = plt.figure(figsize=FIG_DOUBLE)
gs = fig.add_gridspec(1, len(_wr), width_ratios=_wr, left=0.112, right=0.93 if _has_par else 0.97, bottom=0.215, top=0.8,
                      wspace=0.12)
_ax3 = [fig.add_subplot(gs[0])]
_fig3_forest(_ax3[0])
if _has_a1:
    _ax3.append(fig.add_subplot(gs[1], sharey=_ax3[0]))
    _fig3_knockout(_ax3[-1])
if _has_par:
    _ax3.append(fig.add_subplot(gs[len(_ax3)]))
    _fig3_parallax(_ax3[-1])
_t3 = ["Geometry share"] + (["Knockout (A1)"] if _has_a1 else []) + (["Parallax"] if _has_par else [])
for i, a in enumerate(_ax3):
    fig_panel_title(fig, 0.004 if i == 0 else a.get_position().x0 - 0.005, 0.985, "abc"[i], _t3[i])
_cap3 = cap("a", "Geometry share", "beta_s of every model for purely parallel movers in CONVOY-3D (95% cluster-bootstrap intervals), "
            "the same model's single-image prediction (open circles) and classical triangulation with the true pose from exact "
            f"correspondences and from {FLOW_NAME} optical flow (triang., flow); the top axis "
            "gives the equivalent prior weight, the shaded band is the range the locked Prediction 1 expected.")
_k3 = iter("bc")
if _has_a1:
    _cap3 += " " + cap(next(_k3), "Knockout (A1)", "With the mover deleted from the second view, the remaining slope is "
                       "the part not carried by the mover's own correspondence.")
if _has_par:
    _cap3 += " " + cap(next(_k3), "Parallax", f"beta_s by translational parallax (model colours as in (a)); movers below "
                       f"{PREREG['parallax_min_px']:g} px (shaded) are not identifiable from two views and are excluded from (a).")
_cap3 += gate_caption_note("MAIN")
save_figure(fig, "fig3_geometry_share", "How much geometry the models use",
            "What is beta_s, does it come from the correspondence, and does it need parallax?", "Sections 6.1 and 6.5",
            "double column", dict(e1=E1_TABLE[E1_TABLE["model"].isin(MODELS_OK["MAIN"])].to_dict("records"),
                                  parallax=_PAR_OK.to_dict("records"), a1=_A1_OK.to_dict("records"), controls=CONTROL_E1),
            caption=_cap3)

In [ ]:
# Cell No: 075
# Figure 4 (double column): E5, what controls the geometry share. (a) One mover in four of the five
# conditions (same s, same camera, same scene). (b) beta_s per condition (common sample; 95% CI).
# (c) The registered contrasts H5-H7. (d) H8: the single-image depth of a lifted object.
_cond_lab = {"G": "grounded", "NS": "no shadow", "CH": "contact hidden", "F": "floating", "FH": "floating + wall"}
_M4 = MODELS_OK["ABL"]
_E5B = E5_BETA[E5_BETA["model"].isin(_M4)] if len(E5_BETA) else E5_BETA
_has_e5 = len(_E5B) > 0 and np.isfinite(_E5B["beta"]).any()
if _has_e5:
    _w0 = E5_WIDE[_M4[0]]
    _cands = _sample(_w0, E5_CONDS, walled=True)
    _cands = _cands[_cands["scene"].isin(sorted({m["index"] for m in META_ABL}))]
    _ex = _cands.sort_values("gt_n_eroded_G", ascending=False).iloc[0] if len(_cands) else _w0.iloc[0]
    _ex_scene, _ex_mover = int(_ex["scene"]), int(str(_ex["uid"]).rsplit("-", 1)[1])
    _ex_meta = {m["condition"]: m for m in META_ABL if m["index"] == _ex_scene}
    _frames = {c: load_frame(_ex_meta[c]["dir"], 0) for c in ["G", "CH", "F", "FH"]}
    _union = np.zeros_like(_frames["G"]["mover_id"], bool)
    for c, fr in _frames.items():
        _union |= fr["mover_id"] == _ex_mover
    _bbx = mask_bbox(_union)

    fig = plt.figure(figsize=FIG_DOUBLE)
    _tw, _gap = 0.615, 0.03                                   # tile width and gap (inches); tiles are 4:3
    _th = 0.75 * _tw
    _y0 = (0.26 * 1.8)
    for k, c in enumerate(["G", "CH", "F", "FH"]):
        r_, c_ = k // 2, k % 2
        a = fig.add_axes([(0.03 + c_ * (_tw + _gap)) / 5.5, (_y0 + (1 - r_) * (_th + _gap)) / 1.8, _tw / 5.5, _th / 1.8])
        a.imshow(crop_around(_frames[c]["rgb"], _bbx, (3, 4), pad=0.45), aspect="auto"); clean_image_axes(a)
        tag(a, _cond_lab[c], fs=4.8)
    fig_panel_title(fig, 0.004, 0.985, "a", "One mover")
    fig.text((0.03 + _tw + _gap / 2) / 5.5, 0.2, f"one mover, s = {_ex['s']:.2f}, ε = 0", fontsize=5.0, ha="center", va="top")
    # (b) beta by condition
    ax = fig.add_axes([0.305, 0.25, 0.29, 0.635])
    fig_panel_title(fig, 0.258, 0.985, "b", "Geometry share per condition")
    _nm = len(_M4)
    for i, model in enumerate(_M4):
        g = _E5B[_E5B["model"] == model].set_index("condition").reindex(E5_CONDS)
        xo = np.arange(len(E5_CONDS)) + (i - (_nm - 1) / 2) * 0.1
        ax.plot(xo, g["beta"], color=mcol(model), lw=0.6, alpha=0.55)
        ax.errorbar(xo, g["beta"], yerr=[(g["beta"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["beta"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.4, capsize=0.8, lw=0.6, label=mlab(model))
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    ax.axvspan(2.5, 4.5, color="#F2F2F2", zorder=0, lw=0)
    ax.set_xticks(range(len(E5_CONDS))); ax.set_xticklabels(["grounded", "no\nshadow", "contact\nhidden", "floating", "floating\n+ wall"],
                                                            fontsize=5.2)
    ax.set_xlim(-0.5, len(E5_CONDS) - 0.5); ax.set_ylabel("β$_s$ (95% CI)")
    _ylo = np.nanmin(_E5B["ci_lo"]) if _E5B["ci_lo"].notna().any() else 0.0
    _yhi = np.nanmax(_E5B["ci_hi"]) if _E5B["ci_hi"].notna().any() else 1.0
    ax.set_ylim(min(-0.1, _ylo - 0.05), max(1.0, _yhi + 0.05))
    _n5 = int(_E5B["n"].max()) if len(_E5B) else 0
    ax.text(0.02, 0.97, f"same {_n5} movers in every condition", transform=ax.transAxes, fontsize=5.0, va="top", color="#555555")
    # (c) contrasts H5-H7
    ax = fig.add_axes([0.665, 0.25, 0.2, 0.635])
    fig_panel_title(fig, 0.618, 0.985, "c", "Registered contrasts")
    _tests = ["H5", "H6", "H7"]
    for i, model in enumerate(_M4):
        g = E5_TESTS[(E5_TESTS["model"] == model)].set_index("test").reindex(_tests)
        xo = np.arange(len(_tests)) + (i - (_nm - 1) / 2) * 0.11
        ax.errorbar(xo, g["estimate"], yerr=[(g["estimate"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["estimate"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.4, capsize=0.8, lw=0.6)
    ax.axhline(0, color="#7F7F7F", lw=0.6, zorder=0)
    ax.set_xticks(range(3)); ax.set_xticklabels(["F − NS", "CH − G", "interaction"], fontsize=5.2)
    ax.set_xlim(-0.5, 2.5); ax.set_ylabel("difference in β$_s$")
    for k, t in enumerate(_tests):
        ax.text(k, 1.0, t, transform=ax.get_xaxis_transform(), fontsize=5.0, ha="center", va="bottom", color="#555555")
    # (d) H8
    ax = fig.add_axes([0.925, 0.25, 0.07, 0.635])
    fig_panel_title(fig, 0.868, 0.985, "d", "One view")
    for i, model in enumerate(_M4):
        g = E5_TESTS[(E5_TESTS["model"] == model) & (E5_TESTS["test"] == "H8")]
        if len(g):
            r = g.iloc[0]
            ax.errorbar((i - (_nm - 1) / 2) * 0.12, r["estimate"], yerr=ci_err(r["estimate"], r["ci_lo"], r["ci_hi"]),
                        fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.4, capsize=0.8, lw=0.6)
    ax.axhline(0, color="#7F7F7F", lw=0.6, zorder=0)
    ax.set_xlim(-0.5, 0.5); ax.set_xticks([0]); ax.set_xticklabels(["H8"], fontsize=5.2)
    ax.set_ylabel("Δ single-image ρ", labelpad=1)
    fig.legend(handles=[Line2D([], [], ls="-", lw=0.6, marker=MODEL_MARKERS[m], color=mcol(m), ms=2.4, label=mlab(m))
                        for m in _M4], loc="lower center", bbox_to_anchor=(0.64, 0.0), ncol=len(_M4),
               fontsize=5.0, handlelength=1.4, columnspacing=0.9)
    save_figure(fig, "fig4_ablation", "What controls the geometry share: ground contact",
                "Does removing or hiding an object's ground contact make the models use more geometry?", "Section 6.7",
                "double column", dict(beta=_E5B.to_dict("records"), tests=E5_TESTS[E5_TESTS["model"].isin(_M4)].to_dict("records"),
                                      exemplar=dict(scene=_ex_scene, mover=_ex_mover)),
                caption=cap("a", "One mover", "The same parallel mover rendered grounded, behind a low wall "
                            "that hides its ground contact in both views, lifted off the ground, and lifted behind the wall; "
                            "everything else is identical.")
                        + " " + cap("b", "Geometry share per condition", "beta_s on the same movers; shaded: the two lifted conditions.")
                        + " " + cap("c", "Registered contrasts", "Lifting the object (H5), hiding its contact (H6) and their "
                                    "interaction (H7).")
                        + " " + cap("d", "One view", "Shift of the single-image depth of the same object when it is "
                                    "lifted (H8).")
                        + " Intervals are 95% cluster-bootstrap intervals over base scenes." + gate_caption_note("ABL"))

else:
    print("Figure 4 is not drawn: the ablation produced no estimable geometry share in this run.")

In [ ]:
# Cell No: 076
# Figure 5 (double column): E5-REAL, the support cue on real images. (a) Two crates, each inserted
# into a real TUM frame grounded with a contact shadow and lifted without one (same pair, crate, s
# and texture). (b) beta_s of the same crates in both conditions, per model, with classical
# triangulation on the same composites. (c) H11a, beta(lifted) - beta(grounded). (d) H11b, the shift
# of the single-image depth when the crate is lifted.
_MR = [m for m in MODELS_OK["RABL"] if m in RABL_WIDE]
_RB = RABL_BETA[RABL_BETA["model"].isin(_MR)] if len(RABL_BETA) else RABL_BETA
_has_rabl = len(_RB) > 0 and bool(np.isfinite(_RB["beta"]).any())
if _has_rabl:
    _wr = RABL_WIDE[_MR[0]]
    _cand = _wr[_wr["sample"]].sort_values("gt_n_eroded_G", ascending=False)
    _picks = []
    for _seq in _cand["seq"].dropna().unique():                 # one crate per sequence first, then the largest others
        _picks.append(_cand[_cand["seq"] == _seq].iloc[0])
    for _, r in _cand.iterrows():
        if len(_picks) >= 2:
            break
        if all(r["uid"] != p["uid"] for p in _picks):
            _picks.append(r)
    _picks = _picks[:2]
    _rmeta = {(m["index"], m["condition"]): m for m in META_RABL}
    _tiles = []
    for r in _picks:
        _sc, _mv = int(r["scene"]), int(str(r["uid"]).rsplit("-", 1)[1])
        fr = {c: load_frame(_rmeta[(_sc, c)]["dir"], 0) for c in RABL_CONDS}
        _u = np.zeros_like(fr["G"]["mover_id"], bool)
        for c in RABL_CONDS:
            _u |= fr[c]["mover_id"] == _mv
        _bb = mask_bbox(_u)
        _tiles.append({c: crop_around(fr[c]["rgb"], _bb, (3, 4), pad=0.65) for c in RABL_CONDS} | dict(s=float(r["s"]), seq=r["seq"]))
        del fr
    _cl = {"G": "grounded", "F": "lifted"}

    fig = plt.figure(figsize=FIG_DOUBLE)
    _tw, _gap = 0.615, 0.03
    _th = 0.75 * _tw
    _y0 = 0.26 * 1.8
    for r_, t in enumerate(_tiles):
        for c_, c in enumerate(RABL_CONDS):
            a = fig.add_axes([(0.03 + c_ * (_tw + _gap)) / 5.5, (_y0 + (1 - r_) * (_th + _gap)) / 1.8, _tw / 5.5, _th / 1.8])
            a.imshow(t[c], aspect="auto"); clean_image_axes(a)
            tag(a, _cl[c], fs=4.8)
    fig_panel_title(fig, 0.004, 0.985, "a", "Real support")
    fig.text((0.03 + _tw + _gap / 2) / 5.5, 0.2, "real TUM frames, same crate, ε = 0", fontsize=5.0, ha="center", va="top")
    # (b) beta_s grounded vs lifted, paired by model, with the classical control
    ax = fig.add_axes([0.335, 0.25, 0.22, 0.635])
    fig_panel_title(fig, 0.272, 0.985, "b", "Geometry share")
    _nm = len(_MR)
    for i, model in enumerate(_MR):
        g = _RB[_RB["model"] == model].set_index("condition").reindex(RABL_CONDS)
        xo = np.arange(len(RABL_CONDS)) + (i - (_nm - 1) / 2) * 0.09
        ax.plot(xo, g["beta"], color=mcol(model), lw=0.6, alpha=0.55)
        ax.errorbar(xo, g["beta"], yerr=[(g["beta"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["beta"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.4, capsize=0.8, lw=0.6)
    _ctl = [RABL_CONTROL.get(c, {}).get("beta", np.nan) for c in RABL_CONDS]
    ax.plot(np.arange(len(RABL_CONDS)) + 0.5 * 0.09 * (_nm + 1), _ctl, ls="none", marker="D", ms=2.2, color=CONTROL_COLOR)
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    ax.set_xticks(range(len(RABL_CONDS))); ax.set_xticklabels(["grounded,\ncontact shadow", "lifted,\nno shadow"], fontsize=5.2)
    ax.set_xlim(-0.55, len(RABL_CONDS) - 0.45); ax.set_ylabel("β$_s$ (95% CI)")
    _lo = np.nanmin(_RB["ci_lo"]) if _RB["ci_lo"].notna().any() else 0.0
    _hi = np.nanmax(_RB["ci_hi"]) if _RB["ci_hi"].notna().any() else 1.0
    ax.set_ylim(min(-0.1, _lo - 0.05), max(1.32, _hi + 0.05))        # room above the triangulation control (v6.5)
    _nr = int(_RB["n"].max())
    ax.text(0.03, 0.97, f"same {_nr} crates, {int(_RB['n_pairs'].max())} pairs", transform=ax.transAxes, fontsize=5.0, va="top",
            color="#555555")
    # (c) H11a and (d) H11b
    for k_, (t_, lab_, x0_, lx_, tt_) in enumerate([("H11a", "difference in β$_s$", 0.645, 0.6, "Lifting (H11a)"),
                                                    ("H11b", "Δ single-image ρ", 0.865, 0.82, "One view (H11b)")]):
        ax = fig.add_axes([x0_, 0.25, 0.1, 0.635])
        fig_panel_title(fig, lx_, 0.985, "cd"[k_], tt_)
        for i, model in enumerate(_MR):
            g = RABL_TESTS[(RABL_TESTS["model"] == model) & (RABL_TESTS["test"] == t_)]
            if len(g) and np.isfinite(g.iloc[0]["estimate"]):
                r = g.iloc[0]
                ax.errorbar((i - (_nm - 1) / 2) * 0.12, r["estimate"], yerr=ci_err(r["estimate"], r["ci_lo"], r["ci_hi"]),
                            fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.4, capsize=0.8, lw=0.6)
        ax.axhline(0, color="#7F7F7F", lw=0.6, zorder=0)
        ax.set_xlim(-0.5, 0.5); ax.set_xticks([0]); ax.set_xticklabels([t_ + "\nlifted − grounded"], fontsize=5.2)
        ax.set_ylabel(lab_, labelpad=1)
    _h = [Line2D([], [], ls="-", lw=0.6, marker=MODEL_MARKERS[m], color=mcol(m), ms=2.4, label=mlab(m)) for m in _MR]
    _h += [Line2D([], [], ls="none", marker="D", ms=2.2, color=CONTROL_COLOR, label="triangulation, true pose")]
    fig.legend(handles=_h, loc="lower center", bbox_to_anchor=(0.665, 0.0), ncol=len(_h), fontsize=5.0, handlelength=1.4,
               columnspacing=0.8)
    save_figure(fig, "fig5_real_contact", "The support cue on real images",
                "Does lifting an object off its real support, with everything else fixed, make the models use more geometry?",
                "Section 6.7b", "double column",
                dict(beta=_RB.to_dict("records"), tests=RABL_TESTS[RABL_TESTS["model"].isin(_MR)].to_dict("records"),
                     control={c: {k: v for k, v in RABL_CONTROL[c].items() if np.isscalar(v)} for c in RABL_CONTROL},
                     exemplars=[dict(uid=str(p["uid"]), seq=str(p["seq"])) for p in _picks]),
                caption=cap("a", "Real support", "The same textured crate inserted into real TUM RGB-D frames, "
                            "resting on the real support plane with a soft contact shadow (grounded) and lifted 0.25-0.50 m "
                            "without a shadow (lifted); the frame pair, the crate, its texture and its parallel motion (ε = 0) "
                            "are identical.")
                        + " " + cap("b", "Geometry share", "beta_s of the same crates in both conditions; diamonds: classical "
                                    "triangulation with the true pose on the same composites.")
                        + " " + cap("c", "Lifting (H11a)", "The change of beta_s when the crate is lifted.")
                        + " " + cap("d", "One view (H11b)", "The shift of the single-image depth of the same crate "
                                    "when it is lifted.")
                        + " Intervals are 95% cluster-bootstrap intervals over image pairs." + gate_caption_note("RABL"))
    del _tiles
    release_memory()
else:
    print("Figure 5 is not drawn: the real-image contact ablation produced no estimable geometry share in this run.")

In [ ]:
# Cell No: 077
# Figure 6 (double column): what the geometry share implies for dynamic masks. Recall at the image's
# 5% static false-positive rate: (a) against the exact epipolar offset, M1 epipolar residual (no
# model) and M2 rigid-flow residual with each model's depth (CONVOY-3D); (b) exploratory, M2 recall
# on the same parallel movers in each ablation condition (Section 6.7c). The M5 remedy is in Fig. S3.
EPS_TICKLABELS = ["<0.25", "0.25–1", "1–2", "2–3", "3–6", "6–10", ">10"]

def _eps_axes(ax, ylabel=True):
    ax.axvspan(-0.5, 1.5, color=PAR_SHADE, zorder=0, lw=0)
    ax.axvspan(3.5, 6.5, color=PERP_SHADE, zorder=0, lw=0)
    ax.set_xticks(range(7)); ax.set_xticklabels(EPS_TICKLABELS, rotation=40, ha="right", rotation_mode="anchor", fontsize=5.2)
    ax.set_xlim(-0.5, 6.5); ax.set_ylim(-0.03, 1.06)
    ax.set_xlabel("epipolar offset ε (px)", labelpad=0.5)
    if ylabel:
        ax.set_ylabel(f"recall at {_fpr_pct} static FPR")

_E5M = E5_MASKS[E5_MASKS["model"].isin(MODELS_OK["ABL"])] if len(E5_MASKS) else E5_MASKS
_has_e5m = len(_E5M) > 0
_panels = ["m12"] + (["abl"] if _has_e5m else [])
_fpr_pct = f"{100 * PREREG['mask_fpr']:g}%"
_titles = {"m12": "Recall by epipolar offset", "abl": "Recall by support"}
_lt5 = {k: "ab"[i] for i, k in enumerate(_panels)}
fig = plt.figure(figsize=FIG_DOUBLE)
_x0, _gap = 0.07, 0.08
_w = (0.79 - _x0 - _gap * (len(_panels) - 1)) / len(_panels)
_axes = {k: fig.add_axes([_x0 + i * (_w + _gap), 0.29, _w, 0.6]) for i, k in enumerate(_panels)}
for i, k in enumerate(_panels):
    panel_title(_axes[k], "ab"[i], _titles[k], x=-0.08 if i == 0 else -0.06)
# (a) M1 and M2
ax = _axes["m12"]; _eps_axes(ax)
g = E2_BY_EPS[(E2_BY_EPS["model"] == "(model-independent)") & (E2_BY_EPS["method_key"] == "M1_exact") & (E2_BY_EPS["n"] >= 3)]
ax.plot(g.sort_values("eps_bin")["eps_bin"], g.sort_values("eps_bin")["recall"], color=CONTROL_COLOR, marker="o", ms=2.0, lw=0.9, zorder=5)
for model in MODELS_OK["MAIN"]:
    gg = E2_BY_EPS[(E2_BY_EPS["model"] == model) & (E2_BY_EPS["method_key"] == "M2_exact") & (E2_BY_EPS["n"] >= 3)].sort_values("eps_bin")
    if len(gg):
        ax.plot(gg["eps_bin"], gg["recall"], marker=MODEL_MARKERS[model], color=mcol(model), ms=2.0, lw=0.7)
# (b) M2 recall by ablation condition (same parallel movers, exact correspondences)
if _has_e5m:
    ax = _axes["abl"]
    _nm = len(MODELS_OK["ABL"])
    for i, model in enumerate(MODELS_OK["ABL"]):
        gg = _E5M[_E5M["model"] == model].set_index("condition").reindex(E5_CONDS)
        xo = np.arange(len(E5_CONDS)) + (i - (_nm - 1) / 2) * 0.09
        ax.plot(xo, gg["recall_M2"], color=mcol(model), lw=0.6, alpha=0.55)
        ax.errorbar(xo, gg["recall_M2"], yerr=[(gg["recall_M2"] - gg["ci_lo"]).clip(lower=0), (gg["ci_hi"] - gg["recall_M2"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[model], color=mcol(model), ms=2.2, capsize=0.8, lw=0.6)
    ax.axvspan(2.5, 4.5, color="#F2F2F2", zorder=0, lw=0)
    ax.set_xticks(range(len(E5_CONDS)))
    ax.set_xticklabels(["grounded", "no\nshadow", "contact\nhidden", "floating", "floating\n+ wall"], fontsize=5.2)
    ax.set_xlim(-0.5, len(E5_CONDS) - 0.5); ax.set_ylim(-0.03, 1.06)
    ax.text(0.02, 0.97, f"same {int(_E5M['n'].max())} parallel movers (ε = 0)", transform=ax.transAxes, fontsize=5.0, va="top",
            color="#555555")
_leg5 = [m for m in EVALUATED_MODELS if m in MODELS_OK["MAIN"] or m in MODELS_OK["ABL"]]
handles = [Line2D([], [], color=mcol(m), marker=MODEL_MARKERS[m], ms=2.4, lw=0.8, label=mlab(m)) for m in _leg5]
handles += [Line2D([], [], color=CONTROL_COLOR, marker="o", ms=2.0, lw=0.9, label="M1 (no model)")]
handles += [mpatches.Patch(color=PAR_SHADE, label=f"|ε| < {PREREG['eps_parallel_max']:g} px"),
            mpatches.Patch(color=PERP_SHADE, label=f"ε ≥ {PREREG['eps_perp_min']:g} px")]
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.805, 0.56), fontsize=5.2, handlelength=1.6, labelspacing=0.32)
_cap5 = (f"Recall at the image's {_fpr_pct} static false-positive rate. "
         + cap(_lt5["m12"], "Recall by epipolar offset", "The epipolar residual M1 (black) needs no model and is the most any "
               "two-view geometric test can see; the rigid-flow residual M2 (colours) uses each model's depth; shading marks the "
               "parallel and perpendicular regimes."))
if _has_e5m:
    _cap5 += " " + cap(_lt5["abl"], "Recall by support", "Exploratory: M2 recall on the same purely parallel movers in "
                       "each ground-contact condition of E5 (95% cluster-bootstrap intervals over base scenes); shaded: the lifted "
                       "conditions. The exploratory remedy M5 is shown in Fig. S3.")
_cap5 += gate_caption_note("MAIN", "ABL")
save_figure(fig, "fig6_masks", "Dynamic masks miss what geometry cannot see",
            "How does the recall of geometric masks depend on the epipolar offset and on the support cue?",
            "Sections 6.2 and 6.7c", "double column",
            dict(by_eps=E2_BY_EPS[E2_BY_EPS["model"].isin(["(model-independent)"] + MODELS_OK["MAIN"])].to_dict("records"),
                 by_condition=_E5M.to_dict("records") if _has_e5m else []),
            caption=_cap5)

In [ ]:
# Cell No: 078
# Figure 7 (double column): driving. (a) A Virtual KITTI 2 frame with its moving vehicles outlined by
# exact regime. (b) Motion that geometry cannot see: share of moving-vehicle tracks that a static
# point explains within 1 px over K = 2, 4, 8 frames (one metric for all bars). (c) Exploratory:
# g of vehicles keeping pace with the camera (0.85 <= s < 1): Virtual KITTI 2 and the real-driving data
# with LiDAR (Argoverse 2; KITTI tracking when attached). The H9 slope is reported in the tables only;
# the ground-contact contrast in traffic moved to Figure 10b in v6.4, next to the other testbeds.
_X6P = X6_PACE.copy() if len(X6_PACE) else X6_PACE
_has_p = len(_X6P) > 0 and np.isfinite(_X6P["median_g"]).any()
_P6 = {"img": [0.006, 0.2, 0.288, 0.66], "prev": [0.47, 0.24, 0.16, 0.62]}
if _has_p:
    _P6["pace"] = [0.775, 0.24, 0.215, 0.62]
_L6 = {k: "(" + "abc"[i] + ")" for i, k in enumerate(_P6)}

def _fig6_image(fig):
    cnt = MOVERS_VK[MOVERS_VK["measurable"].astype(bool)].groupby("scene").agg(n=("regime", "size"), k=("regime", "nunique"))
    best = int(cnt.sort_values(["k", "n"]).index[-1]) if len(cnt) else int(MOVERS_VK["scene"].iloc[0])
    vm = VK_PAIR_META[best]; vs = VK_SCENE_OBJS[vm["seq"]]; vg = vs.pair_gt(*vm["frames"])
    a = fig.add_axes(_P6["img"])
    a.imshow(vs.rgb(vm["frames"][0]), aspect="auto"); clean_image_axes(a)
    for k in range(len(vg["movers"])):
        reg = MOVERS_VK[(MOVERS_VK["scene"] == best) & (MOVERS_VK["mover"] == k)]["regime"]
        if len(reg) and (vg["mover_id"] == k).any():
            a.contour((vg["mover_id"] == k).astype(float), levels=[0.5], colors=[REGIME_COLORS[reg.iloc[0]]], linewidths=0.8)
    tag(a, f"{vm['seq']}, frames {vm['frames'][0]}–{vm['frames'][1]}", fs=4.8)
    panel_title(a, _L6["img"][1], "Exact regimes")
    return vm

def _fig7_prev(fig):
    ax = fig.add_axes(_P6["prev"])
    panel_title(ax, _L6["prev"][1], "Invisible motion", x=-0.9)
    k = KVIEW_SUMMARY.sort_values("K")
    yb = np.arange(len(k))[::-1]
    ax.barh(yb, k["share_static_fit_below_1px"], color="#0072B2", height=0.62)
    for y_, v in zip(yb, k["share_static_fit_below_1px"]):
        inside = v > 0.72
        ax.text(v - 0.03 if inside else v + 0.03, y_, f"{100 * v:.0f}%", va="center", ha="right" if inside else "left",
                fontsize=5.0, color="white" if inside else "black")
    ax.set_yticks(yb); ax.set_yticklabels([f"{int(K)} frames" for K in k["K"]], fontsize=5.2)
    ax.set_xlim(0, 1.0); ax.set_xticks([0, 0.5, 1.0]); ax.set_xticklabels(["0", "0.5", "1"])
    ax.set_ylim(-0.6, len(k) - 0.4); ax.tick_params(axis="y", length=0)
    ax.set_xlabel("vehicle tracks fit\nby a static point", labelpad=1, fontsize=5.6)

def _g_axis(ax):
    ax.axhline(0, color=MONO_COLOR, lw=0.6, ls=":")
    ax.axhline(1, color=CONTROL_COLOR, lw=0.6, ls=(0, (4, 2)))
    ax.set_ylim(-0.35, 1.15); ax.set_ylabel("geometry share g", labelpad=1)
    ax.text(1.0, 1.0, "geometry", transform=ax.get_yaxis_transform(), fontsize=4.8, ha="right", va="bottom", color="#555555")
    # at the left end of the line, clear of the markers, which sit near 0 when a model follows its prior
    ax.text(0.015, 0.0, "prior", transform=ax.get_yaxis_transform(), fontsize=4.8, ha="left", va="bottom", color="#555555")

def _fig7_pace(fig):
    ax = fig.add_axes(_P6["pace"])
    panel_title(ax, _L6["pace"][1], "The convoy case", x=-0.3)
    sets = [d for d in ("Virtual KITTI 2", "Argoverse 2", "KITTI tracking") if (_X6P["dataset"] == d).any()]
    ms = [m for m in EVALUATED_MODELS if (_X6P["model"] == m).any()]
    for k_, dset in enumerate(sets):
        for i, m in enumerate(ms):
            r = _X6P[(_X6P["dataset"] == dset) & (_X6P["model"] == m)]
            if not len(r) or not np.isfinite(r["median_g"].iloc[0]):
                continue
            r = r.iloc[0]
            ax.errorbar(k_ + (i - (len(ms) - 1) / 2) * 0.09, r["median_g"], yerr=ci_err(r["median_g"], r["ci_lo"], r["ci_hi"]),
                        fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6,
                        mfc="white" if dset == "Virtual KITTI 2" else mcol(m), mew=0.6)
    _g_axis(ax)
    _xl = {"Virtual KITTI 2": "VKITTI 2", "Argoverse 2": "Argoverse 2\n(real)", "KITTI tracking": "KITTI\n(real)"}
    ax.set_xticks(range(len(sets))); ax.set_xticklabels([_xl[d] for d in sets], fontsize=5.2)
    ax.set_xlim(-0.5, len(sets) - 0.5)

_K6 = sorted(int(k) for k in KVIEW_SUMMARY["K"]) if len(KVIEW_SUMMARY) else []
if VK_STATUS["available"] and len(MOVERS_VK):
    fig = plt.figure(figsize=FIG_DOUBLE)
    _vm = _fig6_image(fig)
    _fig7_prev(fig)
    if _has_p:
        _fig7_pace(fig)
    _h = [Line2D([], [], color=cc, lw=1.0, label=rg) for rg, cc in REGIME_COLORS.items()]
    _mm = [m for m in EVALUATED_MODELS if len(_X6P) and (_X6P["model"] == m).any()]
    _h += [Line2D([], [], ls="", marker=MODEL_MARKERS[m], color=mcol(m), ms=2.6, label=mlab(m)) for m in _mm]
    fig.legend(handles=_h, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(_h), fontsize=4.9, handletextpad=0.25,
               columnspacing=0.75, handlelength=1.1)
    _pv = (np.average(PREVALENCE_VK["frac_parallel"], weights=PREVALENCE_VK["n_moving_pixels"]) if len(PREVALENCE_VK) else np.nan)
    _cap = (cap(_L6["img"][1], "Exact regimes", "A Virtual KITTI 2 frame with its moving vehicles outlined by exact motion regime.")
            + " " + cap(_L6["prev"][1], "Invisible motion", "Share of moving-vehicle tracks whose image track over "
                        + ", ".join(str(k) for k in _K6) + " frames a static point explains within 1 px"
                        + (f" (between two views, {100 * _pv:.0f}% of moving-vehicle pixels lie within 1 px of their epipolar line)."
                           if np.isfinite(_pv) else ".")))
    if _has_p:
        _real = [d for d in ("Argoverse 2", "KITTI tracking") if (_X6P["dataset"] == d).any()]
        _cap += " " + cap(_L6["pace"][1], "The convoy case (exploratory)", "Median per-object geometry share g (0: the model's own "
                          "single-image depth, 1: the geometric answer) of vehicles keeping pace with the camera (0.85 ≤ s < 1, "
                          "whose geometric answer lies at least 6.7 times their distance); open markers: Virtual KITTI 2"
                          + (", filled: real images with LiDAR depth (" + " and ".join(_real) + ")" if _real else "") + ".")
    _cap += (" Intervals are 95% cluster-bootstrap intervals over vehicle tracks."
             + gate_caption_note("VK", *[k for k, _, _ in REAL_DRIVE if _has_p and (_X6P["set"] == k).any()]))
    save_figure(fig, "fig7_driving", "Driving: invisible motion and the convoy case",
                "How common is motion that geometry cannot see, and where do the models place vehicles that keep pace with the camera?",
                "Sections 6.8 and 6.8b", "double column",
                dict(kview=KVIEW_SUMMARY.to_dict("records"), pace=_X6P.to_dict("records") if _has_p else [],
                     prevalence=PREVALENCE_ALL.to_dict("records"), exemplar=_vm),
                caption=_cap)
else:
    print("Figure 7 is not drawn: Virtual KITTI 2 is not part of this run (" + VK_STATUS["note"] + ").")

In [ ]:
# Cell No: 079
# Figure 8 (double column): the prior weight and the number of views. (a) Prior weight w against the
# exact epipolar offset (median per bin). (b) w at |eps| < 1 px by mover class. (c) E4: beta_s against
# the number of frames given jointly to the multi-view models, straight camera path (filled, solid)
# and its curved twin (open, dashed).
_M7 = MODELS_OK["MAIN"]
_E4T = E4_TABLE[E4_TABLE["model"].isin(_M7)] if len(E4_TABLE) else E4_TABLE
_has_e4 = len(_E4T) > 0 and _E4T["beta_s"].notna().any()
fig = plt.figure(figsize=FIG_DOUBLE)
_ax_specs = ([[0.065, 0.29, 0.25, 0.6], [0.385, 0.29, 0.19, 0.6], [0.655, 0.29, 0.165, 0.6]] if _has_e4 else
             [[0.07, 0.29, 0.38, 0.6], [0.55, 0.29, 0.28, 0.6]])
_axs = [fig.add_axes(sp) for sp in _ax_specs]
for i, (a, t) in enumerate(zip(_axs, ["Prior weight by offset", "Prior weight by class", "More views"])):
    panel_title(a, "abc"[i], t, x=-0.14 if i == 0 else -0.25)
ax = _axs[0]
ax.axvspan(-0.5, 1.5, color=PAR_SHADE, zorder=0, lw=0); ax.axvspan(3.5, 6.5, color=PERP_SHADE, zorder=0, lw=0)
for model in _M7:
    g = E3_BY_EPS[(E3_BY_EPS["model"] == model) & (E3_BY_EPS["n"] >= 3)].sort_values("eps_bin")
    ax.plot(g["eps_bin"], g["w_median"], marker=MODEL_MARKERS[model], color=mcol(model), ms=2.4, lw=0.8)
ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0); ax.axhline(1, color=GRID_COLOR, lw=0.5, zorder=0)
ax.set_xticks(range(7)); ax.set_xticklabels(["<0.25", "0.25–1", "1–2", "2–3", "3–6", "6–10", ">10"], rotation=40, ha="right",
                                            rotation_mode="anchor", fontsize=5.2)
ax.set_xlim(-0.5, 6.5); ax.set_ylim(-0.3, 1.32)
ax.set_xlabel("epipolar offset ε (px)", labelpad=0.5); ax.set_ylabel("prior weight w (median)")
ax.text(6.4, 1.24, "w = 1: single-image depth", fontsize=5.0, ha="right", va="top", color="#555555")
ax.text(6.4, -0.22, "w = 0: geometric depth", fontsize=5.0, ha="right", va="bottom", color="#555555")
# (b) class at eps ~ 0
ax = _axs[1]
classes = [("car", "w_car", "car"), ("crate", "w_crate", "crate"), ("crate_floating", "w_crate_floating", "floating\ncrate")]
_nm = len(_M7)
for i, model in enumerate(_M7):
    r = E3_TABLE[E3_TABLE["model"] == model].iloc[0]
    xs_ = [k + (i - (_nm - 1) / 2) * 0.11 for k in range(3)]
    ax.plot(xs_, [r[c[1]] for c in classes], MODEL_MARKERS[model], color=mcol(model), ms=2.6)
ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0); ax.axhline(1, color=GRID_COLOR, lw=0.5, zorder=0)
ax.set_xticks(range(3)); ax.set_xticklabels([c[2] for c in classes], fontsize=5.4)
ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.3, 1.32); ax.set_ylabel("w at |ε| < 1 px (median)")
if _has_e4:
    ax = _axs[2]
    ax.axhspan(*PREREG["beta_support_range"], color=SUPPORT_SHADE, zorder=0, lw=0)
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    Ks = list(RUN_CONFIG["e4_window_sizes"])
    for model in sorted(set(_E4T["model"]), key=EVALUATED_MODELS.index):
        for path, ls, off in (("straight", "-", -0.07), ("curved", (0, (3, 1.5)), 0.07)):
            g = _E4T[(_E4T["model"] == model) & (_E4T["path"] == path)].sort_values("K")
            ax.errorbar(np.arange(len(g)) + off, g["beta_s"],
                        yerr=[(g["beta_s"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["beta_s"]).clip(lower=0)],
                        ls=ls, marker=MODEL_MARKERS[model], color=mcol(model), ms=2.6, lw=0.8, capsize=0.9,
                        mfc=mcol(model) if path == "straight" else "white", mew=0.7)
    ax.set_xticks(range(len(Ks))); ax.set_xticklabels([str(k) for k in Ks])
    ax.set_xlim(-0.4, len(Ks) - 0.6); ax.set_ylim(-0.25, 1.32)
    ax.set_xlabel("frames given jointly (K)"); ax.set_ylabel("β$_s$")
    ax.legend(handles=[Line2D([], [], color="#444444", ls="-", lw=0.8, marker="s", ms=2.6, label="straight path"),
                       Line2D([], [], color="#444444", ls=(0, (3, 1.5)), lw=0.8, marker="s", mfc="white", ms=2.6, label="curved twin")],
              loc="upper right", fontsize=4.9, handlelength=1.8)
fig.legend(handles=[Line2D([], [], color=mcol(m), marker=MODEL_MARKERS[m], ms=2.4, lw=0.8, label=mlab(m)) for m in _M7],
           loc="center left", bbox_to_anchor=(0.835 if _has_e4 else 0.86, 0.58), fontsize=5.1, handlelength=1.3, labelspacing=0.35)
save_figure(fig, "fig8_prior_multiview", "Prior weight, object class and number of views",
            "Does the prior weight grow with the epipolar offset and depend on the class, and do more views change beta_s?",
            "Sections 6.3 and 6.4", "double column",
            dict(by_eps=E3_BY_EPS[E3_BY_EPS["model"].isin(_M7)].to_dict("records"),
                 e3=E3_TABLE[E3_TABLE["model"].isin(_M7)].to_dict("records"), e4=_E4T.to_dict("records") if _has_e4 else []),
            caption=cap("a", "Prior weight by offset", "Prior weight w (0: the geometric static-hypothesis depth, 1: the "
                        "model's single-image depth) against the exact epipolar offset, median per bin, where the two hypotheses "
                        "are separable.")
                    + " " + cap("b", "Prior weight by class", "w for movers in the parallel regime by object class.")
                    + (" " + cap("c", "More views", "beta_s of the multi-view models given K frames jointly, on a straight camera "
                                 "path and on its curved twin (identical frames 1-2).") if _has_e4 else "") + gate_caption_note("MAIN"))

In [ ]:
# Cell No: 080
# Figure 9 (double column): qualitative. A parallel and a perpendicular CONVOY-3D mover and, when
# available, a moving Virtual KITTI 2 vehicle: image, exact depth, the model's two-view and single-image
# depth (same colour scale per row), and the dynamic scores M2 (rigid-flow residual) and M5 (two-view vs
# single-image depth), each divided by its 5%-FPR threshold on static pixels (1 = detection threshold).
_M8 = MODELS_OK["MAIN"] or EVALUATED_MODELS
_qmodel = next((m for m in ["DUSt3R", "MASt3R", "MonST3R"] if m in _M8), _M8[0])
_cand = MOV_RES[(MOV_RES["model"] == _qmodel) & (MOV_RES["set"] == "MAIN") & MOV_RES["scene"].isin(EXEMPLAR_MAIN)
                & MOV_RES["measurable"].astype(bool)]
_rows = []
_p = _cand[_cand["motion_type"] == "parallel"].sort_values("gt_area_px", ascending=False)
if len(_p):
    _rows.append(("parallel", "MAIN", _p.iloc[0]))
_q = _cand[_cand["regime"] == "perpendicular"].sort_values("gt_area_px", ascending=False)
if len(_q):
    _rows.append(("perpendicular", "MAIN", _q.iloc[0]))
if VK_STATUS["available"]:
    # driving pixel rule (>= 40 eroded visible px, as in Section 6.8b): almost no Virtual KITTI 2 vehicle meets the
    # CONVOY-3D measurability rule (D34), which left this row out in v6.3 (D54); a parallel vehicle is preferred
    _v = MOV_RES[(MOV_RES["model"] == _qmodel) & (MOV_RES["set"] == "VK") & MOV_RES["scene"].isin(EXEMPLAR_VK)
                 & (MOV_RES["gt_n_eroded"] >= PREREG["x6_min_eroded_px"])].copy()
    if len(_v):
        _v["_par"] = (_v["regime"] == "parallel").astype(int)
        _rows.append(("vehicle", "VK", _v.sort_values(["_par", "gt_area_px"], ascending=False).iloc[0]))

def _ratio_map(score, static_mask_half):
    s = np.asarray(score, np.float32)
    thr = np.nanpercentile(s[static_mask_half], 95) if static_mask_half.any() else np.nanmax(s)
    return np.clip(s / max(thr, 1e-6), 0, 2.5)

def _half(a, shape):
    return cv2.resize(np.asarray(a, np.float32), (shape[1], shape[0]), interpolation=cv2.INTER_AREA)

titles = ["image", "exact depth", "depth, two views", "depth, one view", "M2 / threshold", "M5 / threshold"]
_row_title = {"parallel": "Parallel", "perpendicular": "Perpendicular", "vehicle": "Driving"}
_row_cap = {"parallel": ("Parallel mover", "A CONVOY-3D mover with ε = 0: it leaves no geometric residual, so M2 cannot flag it."),
            "perpendicular": ("Perpendicular mover", "A mover off its epipolar line: M2 flags it whatever depth the model predicts."),
            "vehicle": ("Driving", "A moving Virtual KITTI 2 vehicle (exact depth and poses).")}
_nr = max(len(_rows), 1)
_th = 0.49; _tw = _th / 0.75; _gp = 0.035
_H9 = min(FIG_DOUBLE[1], 0.19 + _nr * _th + (_nr - 1) * 0.04 + 0.06)       # the canvas height follows the rows (D54)
fig = plt.figure(figsize=(FIG_DOUBLE[0], _H9))
_x0 = 0.74                                   # room for the row titles (v6.5)
_ytop = _H9 - 0.19
im_last = None
for i, (name, set_, row) in enumerate(_rows):
    ex = EXEMPLARS[_qmodel]
    if set_ == "MAIN":
        meta = META_MAIN[int(row["scene"])]
        r0 = load_frame(meta["dir"], 0)
        rgb, depth, mvmask = r0["rgb"], r0["depth"], r0["mover_id"] == int(row["mover"])
        sm = static_mask(r0, "synth"); key = f"MAIN-{int(row['scene'])}"
    else:
        meta = VK_PAIR_META[int(row["scene"])]
        vsc = VK_SCENE_OBJS[meta["seq"]]; g = vsc.pair_gt(*meta["frames"])
        rgb, depth, mvmask = vsc.rgb(meta["frames"][0]), g["depth"], g["mover_id"] == int(row["mover"])
        sm = vk_static_mask(g["depth"], g["inst"]); key = f"VK-{int(row['scene'])}"
    hs = ex[f"{key}|depth"].shape
    st_half = cv2.resize(sm.astype(np.uint8), (hs[1], hs[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    bb = [v * hs[0] / depth.shape[0] for v in mask_bbox(mvmask)]
    dfin = np.where(np.isfinite(depth), depth, np.nan)
    vmin = np.log(np.nanpercentile(dfin, 2)); vmax = np.log(np.nanpercentile(dfin, 97))
    m5 = np.abs(np.log(np.maximum(ex[f"{key}|depth"], 1e-6)) - np.log(np.maximum(ex[f"{key}|mono"], 1e-6)))
    tiles = [_half(rgb, hs) / 255.0, depth_to_rgb(_half(np.where(np.isfinite(depth), depth, 1e3), hs), vmin, vmax),
             depth_to_rgb(ex[f"{key}|depth"], vmin, vmax), depth_to_rgb(ex[f"{key}|mono"], vmin, vmax),
             _ratio_map(ex[f"{key}|m2"], st_half), _ratio_map(m5, st_half)]
    for j, t in enumerate(tiles):
        a = fig.add_axes([(_x0 + j * (_tw + _gp)) / 5.5, (_ytop - (i + 1) * _th - i * 0.04) / _H9, _tw / 5.5, _th / _H9])
        crop = crop_around(t, bb, (3, 4), pad=0.55)
        if j >= 4:
            im_last = a.imshow(crop, cmap="magma", vmin=0, vmax=2.5, aspect="auto")
        else:
            a.imshow(np.clip(crop, 0, 1), aspect="auto")
        clean_image_axes(a)
        if i == 0:
            a.set_title(titles[j], fontsize=5.6, pad=2)
        if j == 0:                                      # row titles (D60): "(a) Parallel", "(b) Perpendicular", "(c) Driving"
            lab = f"({'abc'[i]}) {_row_title[name]}\nε = {row['gt_eps_med']:.1f} px"
            a.text(-0.06, 0.5, lab, transform=a.transAxes, fontsize=5.3, ha="right", va="center")
if im_last is not None:
    _xr = (_x0 + 6 * _tw + 5 * _gp + 0.08) / 5.5
    cax = fig.add_axes([_xr, (_ytop - _nr * _th - (_nr - 1) * 0.04) / _H9, 0.012, (_nr * _th + (_nr - 1) * 0.04) / _H9])
    cb = fig.colorbar(im_last, cax=cax, ticks=[0, 1, 2])
    cb.ax.tick_params(labelsize=5.2, width=0.4, length=1.5); cb.outline.set_linewidth(0.4)
    cb.set_label("score / threshold", fontsize=5.4, labelpad=1)
save_figure(fig, "fig9_qualitative", "Qualitative: what geometry and the prior see",
            "Why do geometric masks miss parallel movers, and what does the single-image prediction add?",
            "Sections 6.1, 6.2 and 6.8", "double column", dict(model=_qmodel, movers=[r[2]["uid"] for r in _rows]),
            caption=" ".join(cap("abc"[i], *_row_cap[name]) for i, (name, _, _) in enumerate(_rows)) + " "
                    f"**Layout.** Crops around a mover ({mlab(_qmodel)}); depth panels share one colour scale per row. "
                    "**Reading the masks.** M2 and M5 are divided by their 5% false-positive threshold on the static pixels of "
                    "the same image, so values above 1 are detections. **What it shows.** A parallel mover leaves no geometric "
                    "residual; the disagreement between the two-view and the single-image depth (M5) can still mark it when the "
                    "model follows geometry.")

In [ ]:
# Cell No: 081
# Figure 10 (double column, v6.4, descriptive): one cue across the testbeds. (a) The slope beta_s of grounded objects
# and of unsupported or hidden-contact objects in every testbed with designed motion (CONVOY-3D by class, CONVOY-REAL,
# whose movers are all floating crates, E5, E5-REAL). (b) The same split in traffic: the median per-object share g of
# parallel vehicles whose wheel contact is visible or hidden (Virtual KITTI 2; real Argoverse 2 vehicles). (c) What
# the slope costs: the depth overestimate the E5 slopes imply for an object keeping pace with the camera (s = 0.85).
_SL = SUPPORT_LEVELS.copy() if len(SUPPORT_LEVELS) else pd.DataFrame(columns=["testbed", "condition", "model", "estimate"])

def _has_cat(tb, cond):
    return bool(((_SL["testbed"] == tb) & (_SL["condition"] == cond) & np.isfinite(pd.to_numeric(_SL["estimate"]))).any())

# tick codes, as in Figure 4: G = ground contact in view (shaded), F = floating or lifted, CH = contact hidden
_A_GROUPS = [("CONVOY-3D", [("cars and crates", "G"), ("floating crates", "F")]),
             ("CONVOY-REAL", [("floating crates (every mover)", "F")]),
             ("E5", [(ABLATION_DESIGN["labels"][c], c) for c in ("G", "CH")]),
             ("E5-REAL", [(RABL_DESIGN["labels"][c], c) for c in ("G", "F")])]
_TB_KEY = {"E5": "E5 ablation"}
_A_GROUPS = [(tb, [c for c in cats if _has_cat(_TB_KEY.get(tb, tb), c[0])]) for tb, cats in _A_GROUPS]
_A_GROUPS = [(tb, cats) for tb, cats in _A_GROUPS if cats]
_B_GROUPS = [(ds, [(f"contact {c}", code) for c, code in (("visible", "G"), ("hidden", "CH")) if _has_cat(ds, f"contact {c}")])
             for ds in ("Virtual KITTI 2", "Argoverse 2")]
_B_GROUPS = [(ds, cats) for ds, cats in _B_GROUPS if cats]
_C_CATS = [(ABLATION_DESIGN["labels"][c], c) for c in ("G", "CH", "F") if _has_cat("E5 ablation", ABLATION_DESIGN["labels"][c])]
_M10 = [m for m in EVALUATED_MODELS if (_SL["model"] == m).any()]

def _grouped_positions(groups, gap=0.5):
    pos, x = [], 0.0
    for tb, cats in groups:
        xs = [x + k for k in range(len(cats))]
        pos.append((tb, cats, xs))
        x = xs[-1] + 1.0 + gap
    return pos

def _support_of(tb, cond):
    s_ = _SL[(_SL["testbed"] == tb) & (_SL["condition"] == cond)]["support"]
    return s_.iloc[0] if len(s_) else ""

def _dots(ax, tb, cond, x, col_est="estimate", col_lo="ci_lo", col_hi="ci_hi"):
    for i, m in enumerate(_M10):
        r = _SL[(_SL["testbed"] == tb) & (_SL["condition"] == cond) & (_SL["model"] == m)]
        if not len(r) or not np.isfinite(r[col_est].iloc[0]):
            continue
        r = r.iloc[0]
        ax.errorbar(x + (i - (len(_M10) - 1) / 2) * 0.12, r[col_est], yerr=ci_err(r[col_est], r[col_lo], r[col_hi]),
                    fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6)

def _group_axis(ax, pos, tb_key=lambda tb: tb, names=None):
    ticks, labels = [], []
    for tb, cats, xs in pos:
        for (cond, lab), x in zip(cats, xs):
            if _support_of(tb_key(tb), cond) == "grounded":
                ax.axvspan(x - 0.5, x + 0.5, color=SUPPORT_SHADE, lw=0, zorder=0)
            ticks.append(x); labels.append(lab)
        name = (names or {}).get(tb, tb)
        if len(cats) == 1 and len(name) > 8 and "-" in name:      # a one-column group gets its long name on two lines
            name = name.replace("-", "-\n", 1)
        ax.text((xs[0] + xs[-1]) / 2, 1.0, name, transform=ax.get_xaxis_transform(), fontsize=5.0,
                ha="center", va="bottom", color="#555555", linespacing=1.0)
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=5.4)
    ax.set_xlim(pos[0][2][0] - 0.55, pos[-1][2][-1] + 0.55)
    ax.tick_params(axis="x", length=0)

if _A_GROUPS and _M10:
    fig = plt.figure(figsize=FIG_DOUBLE)
    _panels = ["a"] + (["b"] if _B_GROUPS else []) + (["c"] if _C_CATS else [])
    _lab = {k: "abc"[i] for i, k in enumerate(_panels)}
    _wA = 0.405 if len(_panels) == 3 else 0.55 if len(_panels) == 2 else 0.86
    # (a) slopes by support in the testbeds with designed motion
    ax = fig.add_axes([0.072, 0.2, _wA, 0.645])
    fig_panel_title(fig, 0.004, 0.985, _lab["a"], "Support decides the slope")
    _pa = _grouped_positions(_A_GROUPS)
    for tb, cats, xs in _pa:
        for (cond, _), x in zip(cats, xs):
            _dots(ax, _TB_KEY.get(tb, tb), cond, x)
    _group_axis(ax, _pa, tb_key=lambda tb: _TB_KEY.get(tb, tb))
    ax.axhline(0, color=MONO_COLOR, lw=0.6, ls=":", zorder=1)
    _sa = _SL[_SL["statistic"] == "beta_s"]
    ax.set_ylim(min(-0.2, float(np.nanmin(_sa["ci_lo"])) - 0.05 if _sa["ci_lo"].notna().any() else -0.2),
                max(1.0, float(np.nanmax(_sa["ci_hi"])) + 0.05 if _sa["ci_hi"].notna().any() else 1.0))
    ax.set_ylabel("β$_s$ (95% CI)", labelpad=1)
    _x_right = 0.072 + _wA
    # (b) driving: per-object share by wheel contact
    if _B_GROUPS:
        _xb = _x_right + 0.085
        ax = fig.add_axes([_xb, 0.2, 0.17, 0.645])
        fig_panel_title(fig, _xb - 0.07, 0.985, _lab["b"], "Contact in traffic")
        _pb = _grouped_positions(_B_GROUPS, gap=0.4)
        for ds, cats, xs in _pb:
            for (cond, _), x in zip(cats, xs):
                _dots(ax, ds, cond, x)
        _group_axis(ax, _pb, names={"Virtual KITTI 2": "VKITTI 2", "Argoverse 2": "AV2 (real)"})
        ax.axhline(0, color=MONO_COLOR, lw=0.6, ls=":", zorder=1)
        ax.set_ylim(-0.35, 1.0); ax.set_ylabel("geometry share g", labelpad=1)
        _x_right = _xb + 0.17
    # (c) the depth overestimate implied by the E5 slopes at s = 0.85
    if _C_CATS:
        _xc = _x_right + 0.09
        ax = fig.add_axes([_xc, 0.2, 0.99 - _xc, 0.645])
        fig_panel_title(fig, _xc - 0.075, 0.985, _lab["c"], "What the slope costs")
        for k, (cond, lab) in enumerate(_C_CATS):
            if _support_of("E5 ablation", cond) == "grounded":
                ax.axvspan(k - 0.5, k + 0.5, color=SUPPORT_SHADE, lw=0, zorder=0)
            _dots(ax, "E5 ablation", cond, k, "implied_factor", "implied_lo", "implied_hi")
        _geo = 1.0 / (1.0 - PREREG["implied_factor_s"])
        ax.set_yscale("log"); ax.set_ylim(0.8, _geo * 1.35)
        ax.axhline(1.0, color=MONO_COLOR, lw=0.6, ls=":", zorder=1)
        ax.axhline(_geo, color=CONTROL_COLOR, lw=0.6, ls=(0, (4, 2)), zorder=1)
        ax.set_yticks([1, 2, 4, _geo]); ax.set_yticklabels(["1", "2", "4", f"{_geo:.1f}"]); ax.minorticks_off()
        ax.text(1.0, _geo, "geometry", transform=ax.get_yaxis_transform(), fontsize=4.8, ha="right", va="bottom", color="#555555")
        ax.set_xticks(range(len(_C_CATS))); ax.set_xticklabels([lab for _, lab in _C_CATS], fontsize=5.4)
        ax.set_xlim(-0.6, len(_C_CATS) - 0.4); ax.tick_params(axis="x", length=0)
        ax.set_ylabel("depth overestimate at s = 0.85", labelpad=1, fontsize=5.6)
        ax.text(0.5, 1.0, "E5", transform=ax.transAxes, fontsize=5.0, ha="center", va="bottom", color="#555555")
    fig.legend(handles=[Line2D([], [], ls="", marker=MODEL_MARKERS[m], color=mcol(m), ms=2.6, label=mlab(m)) for m in _M10]
                       + [mpatches.Patch(color=SUPPORT_SHADE, label="contact in view")],
               loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(_M10) + 1, fontsize=5.0, handletextpad=0.3,
               columnspacing=0.9, handlelength=1.2)
    _n_real = int(len(SUPPORT_LEVELS[SUPPORT_LEVELS["testbed"] == "CONVOY-REAL"]))
    _cap10 = cap(_lab["a"], "Support decides the slope", "Geometry share β_s of objects whose ground contact is in view (G, "
                 "shaded), that float (F) or whose contact is hidden by a wall (CH), in every testbed with designed motion: "
                 "CONVOY-3D cars and crates against floating crates"
                 + (", CONVOY-REAL (every inserted mover is a floating crate)" if _n_real else "")
                 + ", the same E5 movers grounded and behind the wall, and the same E5-REAL crates on and above a real support.")
    if _B_GROUPS:
        _cap10 += " " + cap(_lab["b"], "Contact in traffic (exploratory)", "Median per-object geometry share g of parallel "
                            "vehicles whose wheel contact is visible (G, shaded) or hidden (CH), in " + " and ".join(
                                {"Virtual KITTI 2": "Virtual KITTI 2", "Argoverse 2": "real Argoverse 2 images with LiDAR depth"}[ds]
                                for ds, _ in _B_GROUPS) + "; 0 is the model's own single-image depth, 1 the geometric answer.")
    if _C_CATS:
        _cap10 += " " + cap(_lab["c"], "What the slope costs", f"Depth overestimate exp(β_s x) that the E5 slopes (G, CH, F) imply for an "
                            f"object keeping pace with the camera (s = {PREREG['implied_factor_s']}, whose geometric answer lies "
                            f"{1 / (1 - PREREG['implied_factor_s']):.1f} times too far); computed from the slope, not measured separately.")
    _cap10 += (" Intervals are 95% cluster-bootstrap intervals (scenes, image pairs or vehicle tracks). Descriptive: the grouping "
               "by support was chosen after the v6.3 results were read."
               + gate_caption_note("MAIN", "REAL", "ABL", "RABL", *[{"Virtual KITTI 2": "VK", "Argoverse 2": "AV2"}[ds] for ds, _ in _B_GROUPS]))
    save_figure(fig, "fig10_support", "One cue across the testbeds: ground support",
                "Does the same support cue explain the geometry share in every testbed, and what does it cost?",
                "Section 6.9", "double column",
                dict(levels=SUPPORT_LEVELS.to_dict("records"), contrasts=SUPPORT_CONTRASTS.to_dict("records")), caption=_cap10)
else:
    print("Figure 10 is not drawn: no support levels could be estimated in this run.")

In [ ]:
# Cell No: 082
# Figure 11 (double column, v6.5): the static twin. (a) Proposition 4 in a side view: a grounded object moving with the
# camera (s = 0.5) and its static twin, twice as large, sunk below the road. (b) Twin height over camera height against s:
# the law -s/(1-s) and the Argoverse 2 vehicles in the parallel regime. (c) Recall of the epipolar test, the twin test and
# either, by s, with the detection curve implied by parked vehicles. (d) v6.6: the parallel regime with labelled
# correspondences and with optical flow and the LiDAR road plane (H14); by motion regime if the flow test is not in the run.
_has_tw = len(TWIN_RECALL) > 0 and bool(TWIN_THRESH)
fig = plt.figure(figsize=FIG_DOUBLE)
# (a) schematic, side view: depth to the right, height up; camera height H = 1.5, s = 0.5. The drawing is anchored to the
# top of the plot panels and shares their title line.
# without Argoverse 2 camera heights only (a) and (b) are drawn, centred and wider
_xa, _wa = (0.005, 0.27) if _has_tw else (0.1, 0.36)
title_axes(fig, [_xa, 0.25, _wa, 0.62], "a", "The static twin")
ax = fig.add_axes([_xa, 0.14, _wa, 0.71])
ax.set_anchor("N")
_H, _s = 1.5, 0.5
_C = np.array([0.0, _H])
_obj = np.array([[2.2, 0.0], [3.4, 0.0], [3.4, 1.2], [2.2, 1.2]])
_twin = _C + (_obj - _C) / (1 - _s)
ax.add_patch(mpatches.Rectangle((-0.75, -2.15), 8.1, 2.15, fc="#EDEDED", ec="none", zorder=0))
ax.plot([-0.75, 7.35], [0, 0], color="#555555", lw=0.8, zorder=1)
ax.text(0.75, -0.12, "road", fontsize=5.2, color="#555555", ha="left", va="top")
for p_ in (_twin[0], _twin[1]):
    ax.plot([_C[0], p_[0]], [_C[1], p_[1]], color="#7F7F7F", lw=0.5, ls=(0, (3, 2)), zorder=2)
ax.add_patch(mpatches.Polygon(_obj, closed=True, fc="#D55E00", ec="none", alpha=0.85, zorder=3))
ax.add_patch(mpatches.Polygon(_twin, closed=True, fc="none", ec="black", lw=0.7, ls=(0, (2.5, 1.5)), zorder=3))
ax.add_patch(mpatches.Polygon([[-0.3, _H - 0.28], [-0.3, _H + 0.28], [0.25, _H]], closed=True, fc="#0072B2", ec="none", zorder=4))
ax.annotate("", xy=(1.25, _H), xytext=(0.3, _H), arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#0072B2", mutation_scale=5))
ax.annotate("", xy=(3.95, 0.6), xytext=(3.45, 0.6), arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#D55E00", mutation_scale=5))
ax.plot([-0.5, -0.5], [0, _H], color="#0072B2", lw=0.6)
ax.text(-0.43, _H / 2, "H", fontsize=5.6, color="#0072B2", ha="left", va="center")
ax.text(2.8, 1.3, "object", fontsize=5.2, color="#D55E00", ha="center", va="bottom")
ax.text(5.6, _twin[2, 1] + 0.1, "static twin", fontsize=5.2, ha="center", va="bottom")
ax.annotate("", xy=(4.05, _twin[0, 1]), xytext=(4.05, 0.0), arrowprops=dict(arrowstyle="<->", lw=0.6, color="black", mutation_scale=4))
ax.text(4.05, _twin[0, 1] - 0.1, "H s/(1−s)", fontsize=5.0, ha="center", va="top")
ax.set_xlim(-0.75, 7.35); ax.set_ylim(-2.15, 1.95); ax.set_aspect("equal")
clean_image_axes(ax, frame=False)
ax.text(0.5, -0.03, "s = 0.5: twice as large, sunk below the road", transform=ax.transAxes, fontsize=5.0, ha="center", va="top")
# (b) twin height against s
ax = fig.add_axes([0.355, 0.25, 0.18, 0.62] if _has_tw else [0.6, 0.25, 0.3, 0.62])
panel_title(ax, "b", "Twin height", x=-0.34 if _has_tw else -0.2)
_sg = np.linspace(-0.9, 0.83, 200)
ax.plot(_sg, -_sg / (1 - _sg), color=CONTROL_COLOR, lw=0.9, zorder=3, label="−s/(1−s)")
ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
if _has_tw and len(TWIN_VEH):
    _pv = TWIN_VEH[(TWIN_VEH["regime"] == "parallel") & np.isfinite(TWIN_VEH["h_twin"])]
    ax.scatter(_pv["s"], _pv["h_twin"] / _pv["H_cam"], s=2.5, color="#0072B2", alpha=0.6, lw=0, zorder=2, label="Argoverse 2")
    _bh = TWIN_THRESH["height_m"] / TWIN_THRESH["H_median_m"]
    ax.axhspan(-_bh, _bh, color="#F2F2F2", zorder=0, lw=0)
ax.set_xlim(-0.9, 0.85); ax.set_ylim(-4.2, 1.0); ax.set_yticks([-4, -3, -2, -1, 0])
ax.set_xlabel("parallel fraction s"); ax.set_ylabel("twin contact height / H", labelpad=1)
ax.legend(loc="lower left", fontsize=4.9, handlelength=1.2, markerscale=2)
if _has_tw:
    # (c) recall by s
    ax = fig.add_axes([0.6, 0.25, 0.2, 0.62])
    panel_title(ax, "c", "Twin test by s", x=-0.28)
    _ts = TWIN_S[TWIN_S["n"] >= 5]
    for col, colr, lab, ls in (("det_epipolar", MONO_COLOR, "epipolar", "-"), ("det_twin", "#0072B2", "twin", "-"),
                               ("det_either", CONTROL_COLOR, "either", (0, (3, 1.5)))):
        ax.errorbar(_ts["s_median"], _ts[col], yerr=[(_ts[col] - _ts[col + "_lo"]).clip(lower=0), (_ts[col + "_hi"] - _ts[col]).clip(lower=0)],
                    color=colr, marker="o", ms=2.0, lw=0.8, ls=ls, capsize=0.8, label=lab)
    ax.plot(TWIN_THEORY["s"], TWIN_THEORY["p_detect_twin"], color="#0072B2", lw=0.6, alpha=0.6, ls=(0, (1, 1)))
    ax.set_xlim(-0.9, 0.95); ax.set_ylim(-0.03, 1.05)
    ax.set_xlabel("parallel fraction s"); ax.set_ylabel("share flagged", labelpad=1)
    # (d) parallel regime: labelled correspondences against optical flow (v6.6), else by regime
    _ftp11 = FTWIN_RECALL[(FTWIN_RECALL["regime"] == "parallel") & (FTWIN_RECALL["n"] > 0)] if len(FTWIN_RECALL) else FTWIN_RECALL
    _flow11 = len(_ftp11) > 0
    ax = fig.add_axes([0.86, 0.25, 0.13, 0.62])
    _tests11 = (("det_epipolar", MONO_COLOR), ("det_twin", "#0072B2"), ("det_either", CONTROL_COLOR))
    if _flow11:
        panel_title(ax, "d", "Labels vs flow", x=-0.45)
        _grp = [TWIN_RECALL[TWIN_RECALL["regime"] == "parallel"].iloc[0], _ftp11.iloc[0]]
    else:
        panel_title(ax, "d", "By regime", x=-0.45)
        _grp = [r for _, r in TWIN_RECALL[TWIN_RECALL["regime"] != "all"].set_index("regime")
                .reindex(["parallel", "intermediate", "perpendicular"]).reset_index().iterrows()]
    for k, (col, colr) in enumerate(_tests11):
        _v = np.array([float(g[col]) if np.isfinite(g[col]) else 0.0 for g in _grp])
        _lo = np.array([float(g[col] - g[col + "_lo"]) if np.isfinite(g[col + "_lo"]) else 0.0 for g in _grp]).clip(0)
        _hi = np.array([float(g[col + "_hi"] - g[col]) if np.isfinite(g[col + "_hi"]) else 0.0 for g in _grp]).clip(0)
        ax.bar(np.arange(len(_grp)) + (k - 1) * 0.26, _v, width=0.24, color=colr, lw=0, yerr=[_lo, _hi],
               error_kw=dict(lw=0.5, capsize=0.8, ecolor="#333333"))
    ax.set_xticks(range(len(_grp)))
    ax.set_xticklabels(["labels", "flow"] if _flow11 else ["par.", "int.", "perp."], fontsize=5.2)
    ax.set_ylim(0, 1.12); ax.set_ylabel("share flagged", labelpad=1)
    if _flow11 and np.isfinite(H14_RESULT.get("estimate", np.nan)):
        ax.text(0.5, 0.985, f"H14 {H14_RESULT['estimate']:+.2f}", transform=ax.transAxes, fontsize=4.8, ha="center", va="top",
                color="#555555")
    # one key for (c) and (d)
    _hl11 = [Line2D([], [], color=MONO_COLOR, marker="o", ms=2.0, lw=0.8, label="epipolar"),
             Line2D([], [], color="#0072B2", marker="o", ms=2.0, lw=0.8, label="twin"),
             Line2D([], [], color=CONTROL_COLOR, marker="o", ms=2.0, lw=0.8, ls=(0, (3, 1.5)), label="either"),
             Line2D([], [], color="#0072B2", lw=0.6, alpha=0.6, ls=(0, (1, 1)), label="twin, parked-noise model")]
    fig.legend(handles=_hl11, loc="lower center", bbox_to_anchor=(0.78, 0.0), ncol=4, fontsize=4.8, handletextpad=0.3,
               columnspacing=0.9, handlelength=1.6)
_cap11 = (cap("a", "The static twin", "Side view of Proposition 4: a grounded object moving with the camera (s = 0.5) is "
              "explained by a static twin that is twice as large and whose contact lies H s/(1-s) below the road (H: camera height "
              "above the object's contact plane); the road would hide it. An oncoming object (s < 0) has a twin floating above the road.")
          + " " + cap("b", "Twin height", "Height of the twin's contact over H, against s: the law -s/(1-s)"
                      + (" and Argoverse 2 vehicles in the parallel regime (camera height from each vehicle's labelled box, x_geo from "
                         "its LiDAR pixels and labelled motion); shaded: the 95% range of parked vehicles." if _has_tw else ".")))
if _has_tw:
    _trp = TWIN_RECALL[(TWIN_RECALL["regime"] == "parallel") & (TWIN_RECALL["n"] > 0)]
    _trp = _trp.iloc[0] if len(_trp) else None
    _cap11 += (" " + cap("c", "Twin test by s", f"Share of moving Argoverse 2 vehicles flagged by the epipolar test (|eps| above its "
                         f"95th percentile over parked vehicles, {TWIN_THRESH['eps_px']:.2f} px), by the twin test (|twin height| above "
                         f"its 95th percentile, {TWIN_THRESH['height_m']:.2f} m) and by either, binned by s; dotted: the twin test's "
                         "detection curve for exactly parallel motion implied by the parked vehicles.")
               + " " + (cap("d", "Labels vs flow", "Parallel-regime vehicles flagged by each test (bars: epipolar, twin, either; "
                                "95% intervals over vehicle tracks) with the exact correspondences of panels (b-c) and with optical flow "
                                f"({FTWIN_THRESH.get('flow', '').split(' (')[0]}, forward-backward checked) and the road plane fitted to "
                                "the LiDAR returns instead of the labelled box; both at 5% false positives on parked vehicles. H14: "
                                + (f"twin minus epipolar with flow {H14_RESULT['estimate']:+.2f} [{H14_RESULT['ci_lo']:+.2f}, "
                                   f"{H14_RESULT['ci_hi']:+.2f}] over {H14_RESULT['n']} vehicles, {H14_RESULT['verdict']}."
                                   if np.isfinite(H14_RESULT.get("estimate", np.nan))
                                   else f"{H14_RESULT.get('verdict', 'NOT TESTABLE')} ({H14_RESULT.get('n', 0)} vehicles)."))
                        if _flow11 else
                        cap("d", "By regime", "The same shares by motion regime (bars: epipolar, twin, either)"
                            + (f"; of the {int(_trp['n_epi_miss'])} parallel-regime vehicles the epipolar test misses, the twin test "
                               f"flags {100 * _trp['twin_given_epi_miss']:.0f}%" if _trp is not None and _trp["n_epi_miss"] else "")
                            + ". Both tests use the same exact correspondences from labelled boxes, so the panels compare the tests, "
                              "not a deployed detector; 95% intervals over vehicle tracks.")))
save_figure(fig, "fig11_static_twin", "The static twin and the ground plane",
            "Where is the static twin of a grounded mover, and how much invisible motion does it expose?", "Sections 1 and 6.13",
            "double column", dict(thresholds=TWIN_THRESH, by_regime=TWIN_RECALL.to_dict("records"), by_s=TWIN_S.to_dict("records"),
                                  curve=TWIN_THEORY.to_dict("records"), flow_thresholds=FTWIN_THRESH,
                                  flow_by_regime=FTWIN_RECALL.to_dict("records"), H14=H14_RESULT), caption=_cap11)

In [ ]:
# Cell No: 083
# Figure 12 (double column, v6.5, exploratory): two channels. (a) The cue model fitted on CONVOY-3D and E5 predicts the slope
# of every other testbed without refitting: observed against predicted. (b) The hump: binned rho against x on CONVOY-3D,
# observed and as the two-channel and contact-only models fit it. (c) The fitted channels per model: the share of geometry at
# unlimited parallax for each support (contact) and the half-saturation parallax kappa (parallax). (d) Masks that defeat
# themselves: the change of M2 recall against the change of beta_s between conditions.
_TB_MARK12 = {"BSW": "o", "WDR": "s", "REAL": "^", "RABL": "v", "VK": "D", "AV2": "P"}
_TB_NAME12 = {"BSW": "baseline sweep", "WDR": "dose-response", "REAL": "CONVOY-REAL", "RABL": "E5-REAL", "VK": "VKITTI 2",
              "AV2": "Argoverse 2"}
_pred12 = CUE_PRED[(CUE_PRED["role"] == "predicted") & (CUE_PRED["group"] == "all")] if len(CUE_PRED) else CUE_PRED
_has12 = len(_pred12) > 0 and len(CUE_FIT_TABLE) > 0
if _has12:
    fig = plt.figure(figsize=FIG_DOUBLE)
    # (a) observed against predicted
    ax = fig.add_axes([0.08, 0.25, 0.18, 0.62])
    panel_title(ax, "a", "Out of sample", x=-0.33)
    _lo12 = float(np.nanmin(np.r_[_pred12["beta_predicted"], _pred12["ci_lo"], -0.1]))
    _hi12 = float(np.nanmax(np.r_[_pred12["beta_predicted"], _pred12["ci_hi"], 0.8]))
    _lim = [_lo12 - 0.05, _hi12 + 0.05]
    ax.plot(_lim, _lim, color=CONTROL_COLOR, lw=0.6, ls=(0, (4, 2)), zorder=1)
    for r in _pred12.itertuples():
        ax.errorbar(r.beta_predicted, r.beta_observed, yerr=ci_err(r.beta_observed, r.ci_lo, r.ci_hi), fmt=_TB_MARK12.get(r.set, "o"),
                    color=mcol(r.model), ms=2.2, capsize=0.6, lw=0.5, mfc="white" if r.set in ("BSW", "WDR") else mcol(r.model), mew=0.5)
    ax.set_xlim(_lim); ax.set_ylim(_lim)
    ax.set_xlabel("predicted β (cue model)"); ax.set_ylabel("observed β (95% CI)", labelpad=1)
    ax.text(0.04, 0.96, f"r = {CUE_CALIBRATION['pearson_r']:.2f}", transform=ax.transAxes, fontsize=5.0, va="top", color="#555555")
    # (b) the hump
    ax = fig.add_axes([0.335, 0.25, 0.18, 0.62])
    panel_title(ax, "b", "The hump", x=-0.25)
    _hm = [m for m in ("DUSt3R", "MASt3R") if len(CUE_HUMP) and (CUE_HUMP["model"] == m).any()] or \
          (list(CUE_HUMP["model"].unique()[:2]) if len(CUE_HUMP) else [])
    for m in _hm:
        g = CUE_HUMP[CUE_HUMP["model"] == m]
        ax.plot(g["x_median"], g["rho_observed"], MODEL_MARKERS[m], color=mcol(m), ms=2.6, ls="none", zorder=3)
        ax.plot(g["x_median"], g["rho_two_channels"], color=mcol(m), lw=0.9, zorder=2)
        ax.plot(g["x_median"], g["rho_contact_only"], color=mcol(m), lw=0.7, ls=(0, (2, 1.5)), alpha=0.8, zorder=2)
    ax.set_xlabel("geometric answer x"); ax.set_ylabel("median ρ", labelpad=1)
    if _hm:
        ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
        ax.margins(y=0.1)
        ax.legend(handles=[Line2D([], [], ls="none", marker="o", color="#555555", ms=2.6, label="observed"),
                           Line2D([], [], color="#555555", lw=0.9, label="two channels"),
                           Line2D([], [], color="#555555", lw=0.7, ls=(0, (2, 1.5)), label="contact only")],
                  loc="best", fontsize=4.8, handlelength=1.4)
    else:
        ax.set_xticks([]); empty_panel(ax, "fewer than 5 movers\nper bin of x\nin this run")
    # (c) the fitted channels
    ax = fig.add_axes([0.61, 0.25, 0.17, 0.62])
    panel_title(ax, "c", "Two channels", x=-0.42)
    _ft = CUE_FIT_TABLE[CUE_FIT_TABLE["fit"] == "two channels"].reset_index(drop=True)
    _sup_mark = {"c_contact_in_view": ("o", "in view"), "c_contact_hidden": ("^", "hidden"), "c_unsupported": ("s", "floating")}
    for i, r in _ft.iterrows():
        y = len(_ft) - 1 - i
        for col, (mk, _) in _sup_mark.items():
            ax.plot(r[col], y, mk, color=mcol(r["model"]), ms=2.8, mfc="white" if col == "c_contact_in_view" else mcol(r["model"]), mew=0.6)
        ax.text(1.02, y, f"κ {r['kappa_px']:.0f}", transform=ax.get_yaxis_transform(), fontsize=4.8, va="center", ha="left")
    ax.set_yticks(range(len(_ft))); ax.set_yticklabels([mlab(m) for m in _ft["model"][::-1]], fontsize=5.2)
    ax.axvline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    # the bottom band of the panel is kept free of data for the key of the three support classes
    ax.set_ylim(-1.45, len(_ft) - 0.4); ax.set_xlabel("c (geometry share at large parallax)", fontsize=5.6)
    ax.legend(handles=[Line2D([], [], ls="none", marker=mk, color="#555555", mfc="white" if k == "c_contact_in_view" else "#555555",
                              ms=2.8, label=lab) for k, (mk, lab) in _sup_mark.items()],
              loc="lower center", ncol=3, fontsize=4.6, handletextpad=0.3, columnspacing=0.8, handlelength=0.8, borderaxespad=0.1)
    # (d) masks that defeat themselves
    ax = fig.add_axes([0.875, 0.25, 0.115, 0.62])
    panel_title(ax, "d", "Masks", x=-0.5)
    _src_mark = {"E5": "o", "baseline sweep": "s", "dose-response": "^"}
    if len(TRADEOFF):
        for r in TRADEOFF.itertuples():
            ax.plot(r.delta_beta, r.delta_recall, _src_mark.get(r.source, "o"), color=mcol(r.model), ms=2.2, alpha=0.85)
        ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0); ax.axvline(0, color=GRID_COLOR, lw=0.5, zorder=0)
        ax.text(0.95, 0.95, f"ρ$_S$ = {TRADEOFF_SUMMARY['spearman']:.2f}", transform=ax.transAxes, fontsize=4.8, ha="right", va="top",
                color="#555555")
    ax.set_xlabel("Δβ$_s$", labelpad=0.5); ax.set_ylabel("Δ M2 recall", labelpad=1)
    _ms12 = [m for m in EVALUATED_MODELS if (_ft["model"] == m).any()]
    _hl = [Line2D([], [], ls="", marker="o", color=mcol(m), ms=2.6, label=mlab(m)) for m in _ms12]
    _hl += [Line2D([], [], ls="", marker=_TB_MARK12[k], color="#555555", mfc="white" if k in ("BSW", "WDR") else "#555555", ms=2.6,
                   label=_TB_NAME12[k]) for k in _TB_MARK12 if (_pred12["set"] == k).any()]
    fig.legend(handles=_hl, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(_hl), fontsize=4.7, handletextpad=0.15,
               columnspacing=0.6, handlelength=0.9)
    _cap12 = (cap("a", "Out of sample", "The cue model rho - rho_mono = w (x - rho_mono), w = c_support d^2/(d^2 + kappa^2), fitted on "
                  "CONVOY-3D and the E5 conditions, predicts the slope of every other testbed from that testbed's own objects "
                  "(geometric answer, single-image error, parallax, support) without refitting; observed slope (95% interval) "
                  f"against predicted slope (Pearson r = {CUE_CALIBRATION['pearson_r']:.2f}, mean absolute error "
                  f"{CUE_CALIBRATION['mean_abs_error']:.2f}; open markers: the new designed sets).")
              + " " + cap("b", "The hump", "Median rho of CONVOY-3D parallel movers by x (markers) and the fits of the two-channel model "
                          "(solid) and of the contact-only model with constant weight (dashed)" + (f", for {' and '.join(mlab(m) for m in _hm)}" if _hm else "")
                          + "; in the two-channel model, any fall at large x comes from the parallax, which vanishes as s approaches 1"
                          + ("; fall from the observed peak to the largest x: " + ", ".join(
                              f"{mlab(r.model)} {r.fall_observed:.2f} observed, {r.fall_two_channels:.2f} two channels, "
                              f"{r.fall_contact_only:.2f} contact only" for r in CUE_HUMP_FALL[CUE_HUMP_FALL["model"].isin(_hm)].itertuples())
                             if len(CUE_HUMP_FALL) and CUE_HUMP_FALL["model"].isin(_hm).any() else "") + ".")
              + " " + cap("c", "Two channels", "Fitted share of geometry at unlimited parallax for objects with the contact in view "
                          "(open circles), with the contact hidden (triangles) and floating, i.e. without support (squares), and the "
                          "half-saturation parallax kappa in px (right).")
              + " " + cap("d", "Masks", "Change of M2 recall against the change of beta_s between conditions of the same movers "
                          "(E5 conditions against grounded, baseline levels against x0.5, wall heights against no wall); "
                          f"Spearman {TRADEOFF_SUMMARY['spearman']:.2f}." if len(TRADEOFF) else "No paired recall in this run.")
              + " Exploratory (AMENDMENT_V6_5).")
    save_figure(fig, "fig12_two_channels", "Two channels: contact and parallax",
                "Does one cue-combination model with a contact channel and a parallax channel explain every testbed?",
                "Section 6.11", "double column", dict(fits=CUE_FIT_TABLE.to_dict("records"), predictions=CUE_PRED.to_dict("records"),
                                                      hump=CUE_HUMP.to_dict("records"), tradeoff=TRADEOFF.to_dict("records")),
                caption=_cap12)
else:
    print("Figure 12 is not drawn: the cue model could not be fitted in this run.")

In [ ]:
# Cell No: 084
# Figure 13 (double column, v6.5): the designed experiments. (a) One scene of each set: the second view at x0.5 and x3 the
# baseline, and one mover behind the 15 cm and the 70 cm wall. (b) H12: beta_s against the baseline factor. (c) The rise from
# x0.5 to x3 for grounded and floating movers. (d) H13: beta_s against the height of the wall.
_M13b = [m for m in MODELS_OK["BSW"] if len(BSW_BETA) and (BSW_BETA["model"] == m).any()]
_M13w = [m for m in MODELS_OK["WDR"] if len(WDR_BETA) and (WDR_BETA["model"] == m).any()]

def _crop_mover(meta, k, j, pad=0.6):
    fr = load_frame(meta["dir"], k)
    bb = mask_bbox(fr["mover_id"] == j)
    return crop_around(fr["rgb"], bb, (3, 4), pad=pad) if bb is not None else fr["rgb"]

def _bsw_example():
    w = BSW_WIDE.get(_M13b[0]) if _M13b else None
    if w is None or not len(w):
        return None
    q = w[w["h12_sample"]].sort_values(f"gt_n_eroded_{BSW_LEVELS[0]}", ascending=False)
    if not len(q):
        return None
    r = q.iloc[0]
    j = int(str(r["uid"]).rsplit("-", 1)[1])
    metas = {m["condition"]: m for m in META_BSW if m["index"] == int(r["scene"])}
    lo, hi = BSW_LEVELS[0], BSW_LEVELS[-1]
    fr_lo, fr_hi = load_frame(metas[lo]["dir"], 1), load_frame(metas[hi]["dir"], 1)
    union = (fr_lo["mover_id"] == j) | (fr_hi["mover_id"] == j)
    bb = mask_bbox(union)
    return [(crop_around(fr_lo["rgb"], bb, (3, 4), pad=0.5), f"view 2, {SWEEP_DESIGN['labels'][lo]}"),
            (crop_around(fr_hi["rgb"], bb, (3, 4), pad=0.5), f"view 2, {SWEEP_DESIGN['labels'][hi]}")], r

def _wdr_example():
    w = WDR_WIDE.get(_M13w[0]) if _M13w else None
    if w is None or not len(w):
        return None
    q = w[w["contact_G"] & w["hidden_W15"] & w["hidden_W70"] & w["ok_W70"]].sort_values("gt_n_eroded_G", ascending=False)
    if not len(q):
        return None
    r = q.iloc[0]
    j = int(str(r["uid"]).rsplit("-", 1)[1])
    metas = {m["condition"]: m for m in META_WDR if m["index"] == int(r["scene"])}
    frs = {lv: load_frame(metas[lv]["dir"], 0) for lv in ("G", "W15", "W70")}
    bb = mask_bbox(frs["G"]["mover_id"] == j)
    return [(crop_around(frs[lv]["rgb"], bb, (3, 4), pad=0.55), f"{WALL_DOSE_DESIGN['labels'][lv]} wall") for lv in ("W15", "W70")], r

_ex_b, _ex_w = _bsw_example(), _wdr_example()
if _M13b or _M13w:
    fig = plt.figure(figsize=FIG_DOUBLE)
    # (a) a 2 x 2 block of 4:3 tiles, centred on the plot panels and sharing their title line
    _th, _gap = 0.52, 0.03
    _tw = _th * 4 / 3
    _y0 = 0.25 * 1.8 + (0.62 * 1.8 - (2 * _th + _gap)) / 2
    title_axes(fig, [0.03 / 5.5, 0.25, (2 * _tw + _gap) / 5.5, 0.62], "a", "One factor at a time")
    for row, ex in enumerate((_ex_b, _ex_w)):
        if ex is None:
            continue
        for c_, (img, lab) in enumerate(ex[0]):
            a = fig.add_axes([(0.03 + c_ * (_tw + _gap)) / 5.5, (_y0 + (1 - row) * (_th + _gap)) / 1.8, _tw / 5.5, _th / 1.8])
            a.imshow(img, aspect="auto"); clean_image_axes(a); tag(a, lab, fs=4.6)
    fig.text((0.03 + _tw + _gap / 2) / 5.5, (_y0 - 0.04) / 1.8, "top: baseline sweep; bottom: walls", fontsize=5.0,
             ha="center", va="top")
    # (b) beta against the baseline factor
    ax = fig.add_axes([0.35, 0.25, 0.18, 0.62])
    panel_title(ax, "b", "Parallax (H12)", x=-0.25)
    _fx = [SWEEP_DESIGN["levels"][lv] for lv in BSW_LEVELS]
    for i, m in enumerate(_M13b):
        g = BSW_BETA[(BSW_BETA["model"] == m) & (BSW_BETA["support"] == "all")].set_index("level").reindex(BSW_LEVELS)
        xo = np.array(_fx) * (1 + (i - (len(_M13b) - 1) / 2) * 0.025)
        ax.errorbar(xo, g["beta"], yerr=[(g["beta"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["beta"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6, ls="-")
    ax.set_xscale("log"); ax.set_xticks(_fx); ax.set_xticklabels([SWEEP_DESIGN["labels"][lv] for lv in BSW_LEVELS]); ax.minorticks_off()
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    ax.set_xlabel("camera baseline"); ax.set_ylabel("β$_s$ (95% CI)", labelpad=1)
    # (c) rise by support
    ax = fig.add_axes([0.615, 0.25, 0.11, 0.62])
    panel_title(ax, "c", "By support", x=-0.55)
    _n13c = 0
    for i, m in enumerate(_M13b):
        for k, sup in enumerate(("grounded", "floating")):
            r = BSW_TESTS[(BSW_TESTS["model"] == m) & (BSW_TESTS["test"] == f"H12-{sup}")]
            if len(r) and np.isfinite(r["estimate"].iloc[0]):
                r = r.iloc[0]
                ax.errorbar(k + (i - (len(_M13b) - 1) / 2) * 0.1, r["estimate"], yerr=ci_err(r["estimate"], r["ci_lo"], r["ci_hi"]),
                            fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6)
                _n13c += 1
    if _n13c:
        ax.axhline(0, color="#7F7F7F", lw=0.6, zorder=0)
        ax.margins(y=0.12)
    else:
        empty_panel(ax, "too few movers\nper group\nin this run")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["grounded", "floating"], fontsize=5.2); ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel("rise of β$_s$, x0.5 → x3", labelpad=1, fontsize=5.6)
    # (d) beta against the wall height
    ax = fig.add_axes([0.81, 0.25, 0.18, 0.62])
    panel_title(ax, "d", "Contact (H13)", x=-0.28)
    _cv = [100 * WALL_DOSE_DESIGN["levels"][lv] for lv in WDR_LEVELS]
    for i, m in enumerate(_M13w):
        g = WDR_BETA[WDR_BETA["model"] == m].set_index("level").reindex(WDR_LEVELS)
        xo = np.array(_cv) + (i - (len(_M13w) - 1) / 2) * 1.6
        ax.errorbar(xo, g["beta"], yerr=[(g["beta"] - g["ci_lo"]).clip(lower=0), (g["ci_hi"] - g["beta"]).clip(lower=0)],
                    fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6, ls="-")
    ax.axvspan(-6, 6, color=SUPPORT_SHADE, lw=0, zorder=0)
    ax.axhline(0, color=GRID_COLOR, lw=0.5, zorder=0)
    ax.set_xticks(_cv); ax.set_xticklabels(["none", "15", "35", "70"]); ax.set_xlim(-8, 78)
    ax.margins(y=0.1)
    ax.set_xlabel("hidden band (cm)"); ax.set_ylabel("β$_s$ (95% CI)", labelpad=1)
    _mm = [m for m in EVALUATED_MODELS if m in set(_M13b) | set(_M13w)]
    fig.legend(handles=[Line2D([], [], ls="-", lw=0.6, marker=MODEL_MARKERS[m], color=mcol(m), ms=2.4, label=mlab(m)) for m in _mm],
               loc="lower center", bbox_to_anchor=(0.66, 0.0), ncol=len(_mm), fontsize=5.0, handlelength=1.4, columnspacing=0.9)
    _h12 = BSW_TESTS[(BSW_TESTS["test"] == "H12") & BSW_TESTS["model"].isin(PREREG["h12_models"])] if len(BSW_TESTS) else BSW_TESTS
    _h13 = WDR_TESTS[WDR_TESTS["test"] == "H13"] if len(WDR_TESTS) else WDR_TESTS
    _h13i = _h13[~_h13["verdict"].str.startswith("NOT INTERPRETABLE")] if len(_h13) else _h13
    _hs13 = WDR_BETA.groupby("level")["median_hidden_share"].median() if len(WDR_BETA) else pd.Series(dtype=float)
    _cap13 = (cap("a", "One factor at a time", "Top: the second view of one baseline-sweep scene at 0.5 and 3 times the baseline "
                  "(first view, s, eps = 0 and the geometric answer are identical); bottom: one mover behind the 15 cm and the 70 cm wall.")
              + " " + cap("b", "Parallax (H12)", "beta_s of the same movers at each baseline (measurable at every baseline, identifiable "
                          "at x1); H12 predicts a rise for DUSt3R and MASt3R"
                          + (" (" + "; ".join(f"{mlab(r.model)}: {r.estimate:.2f} [{r.ci_lo:.2f}, {r.ci_hi:.2f}], {r.verdict}"
                                               for r in _h12.itertuples()) + ")" if len(_h12) else "") + ".")
              + " " + cap("c", "By support", "The rise from x0.5 to x3 for grounded movers (cars, crates) and floating crates, "
                          "exploratory.")
              + " " + cap("d", "Contact (H13)", "beta_s of the same grounded movers with no wall (shaded) and behind walls built "
                          "to hide at least their lowest 15, 35 and 70 cm in both views"
                          + (" (median share of the mover hidden: " + ", ".join(f"{_hs13.get(lv, np.nan):.2f}" for lv in WDR_LEVELS)
                             + ")" if len(_hs13) else "")
                          + "; H13 predicts a rise at 15 cm for every model"
                          + (f" (SUPPORTED for {int((_h13i['verdict'] == 'SUPPORTED').sum())} of the {len(_h13i)} models that pass the "
                             "static-scene gate)" if len(_h13i) else "") + ".")
              + " Intervals are 95% cluster-bootstrap intervals over base scenes." + gate_caption_note("BSW", "WDR"))
    save_figure(fig, "fig13_designed", "Designed experiments: parallax and contact, one at a time",
                "Does a longer baseline alone raise the geometry share, and does the lowest wall that hides the contact suffice?",
                "Section 6.10", "double column", dict(sweep=BSW_BETA.to_dict("records"), sweep_tests=BSW_TESTS.to_dict("records"),
                                                      dose=WDR_BETA.to_dict("records"), dose_tests=WDR_TESTS.to_dict("records")),
                caption=_cap13)
else:
    print("Figure 13 is not drawn: no interpretable model on the designed sets in this run.")

In [ ]:
# Cell No: 085
# Figure 14 (double column, v6.5, exploratory): real objects that float (Bonn RGB-D Dynamic). (a) Four pairs, two with a
# balloon and two with a box, the moving object outlined (people in white). (b) Motion regimes of real objects by kind.
# (c) Median per-object geometry share g of parallel objects by kind and by floor contact.
_OBJ_COL, _PERSON_COL = "#F0E442", "#FFFFFF"
_bg = BONN_G[BONN_G["model"].isin(MODELS_OK.get("BONN", []))] if len(BONN_G) else BONN_G


def _bonn_examples(kind, k):
    # the k pairs with the largest object of this kind, from different sequences where possible
    cands = sorted(((max(o["n_px"] for o in m_["objects"] if o["kind"] == kind), m_["index"], m_) for m_ in BONN_PAIR_META
                    if any(o["kind"] == kind for o in m_["objects"])), key=lambda t: (-t[0], t[1]))
    out, seqs = [], set()
    for _, _, m_ in cands:
        if m_["seq"] not in seqs and len(out) < k:
            out.append(m_); seqs.add(m_["seq"])
    for _, _, m_ in cands:
        if len(out) < k and all(m_["index"] != o_["index"] for o_ in out):
            out.append(m_)
    return out


if len(BONN_PREV) and BONN_PAIR_META and not BONN_STATUS.get("suspect"):
    _ex14 = _bonn_examples("balloon", 2) + _bonn_examples("box", 2)
    fig = plt.figure(figsize=FIG_DOUBLE)
    # (a) a 2 x 2 block of 4:3 tiles, centred on the plot panels and sharing their title line
    _th, _gap = 0.52, 0.03
    _tw = _th * 4 / 3
    _y0 = 0.25 * 1.8 + (0.62 * 1.8 - (2 * _th + _gap)) / 2
    title_axes(fig, [0.03 / 5.5, 0.25, (2 * _tw + _gap) / 5.5, 0.62], "a", "Real objects")
    for k, m_ in enumerate(_ex14[:4]):
        img = np.asarray(Image.open(os.path.join(m_["dir"], "rgb0.png")).convert("RGB"))
        a = fig.add_axes([(0.03 + (k % 2) * (_tw + _gap)) / 5.5, (_y0 + (1 - k // 2) * (_th + _gap)) / 1.8, _tw / 5.5, _th / 1.8])
        a.imshow(img, aspect="auto"); clean_image_axes(a)
        for o in m_["objects"]:
            mk = np.zeros(img.shape[:2], bool)
            px = np.asarray(o["pixels"], int).reshape(-1, 2)
            mk[px[:, 1], px[:, 0]] = True
            a.contour(mk.astype(float), levels=[0.5], colors=[_PERSON_COL if o["kind"] == "person" else _OBJ_COL], linewidths=0.7)
        tag(a, m_["seq"].replace("moving_nonobstructing_", "").replace("_", " "), fs=4.6)
    # (b) regimes by kind
    ax = fig.add_axes([0.43, 0.25, 0.15, 0.62])
    panel_title(ax, "b", "Regimes", x=-0.6)
    _bp = BONN_PREV[BONN_PREV["kind"] != "all objects"].reset_index(drop=True)
    left = np.zeros(len(_bp))
    for col, reg in (("frac_parallel", "parallel"), ("frac_intermediate", "intermediate"), ("frac_perpendicular", "perpendicular")):
        ax.barh(np.arange(len(_bp))[::-1], _bp[col], left=left, color=REGIME_COLORS[reg], height=0.6, lw=0)
        left = left + _bp[col].to_numpy()
    ax.set_yticks(np.arange(len(_bp))[::-1]); ax.set_yticklabels([f"{k} ({n})" for k, n in zip(_bp["kind"], _bp["n_objects"])],
                                                                    fontsize=5.2)
    ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); ax.set_xticklabels(["0", "0.5", "1"]); ax.tick_params(axis="y", length=0)
    ax.set_xlabel("share of objects", labelpad=1)
    # (c) geometry share by kind and floor contact
    ax = fig.add_axes([0.685, 0.25, 0.305, 0.62])
    panel_title(ax, "c", "Geometry share", x=-0.17)
    _groups = [("kind", k) for k in ("balloon", "box", "person")] + [("floor contact", f_) for f_ in ("off the floor", "on the floor")]
    _groups = [g for g in _groups if len(_bg) and ((_bg["grouping"] == g[0]) & (_bg["group"] == g[1])).any()]
    _m14 = [m for m in EVALUATED_MODELS if len(_bg) and (_bg["model"] == m).any()]
    for k, (gt_, gn) in enumerate(_groups):
        for i, m in enumerate(_m14):
            r = _bg[(_bg["grouping"] == gt_) & (_bg["group"] == gn) & (_bg["model"] == m)]
            if len(r) and np.isfinite(r["median_g"].iloc[0]):
                r = r.iloc[0]
                _small = r["n"] < 8                              # no interval below 8 objects: open marker
                ax.errorbar(k + (i - (len(_m14) - 1) / 2) * 0.1, r["median_g"], yerr=ci_err(r["median_g"], r["ci_lo"], r["ci_hi"]),
                            fmt=MODEL_MARKERS[m], color=mcol(m), ms=2.2, capsize=0.8, lw=0.6, mfc="white" if _small else mcol(m),
                            mew=0.6)
    if _groups:
        ax.axvspan(len([g for g in _groups if g[0] == "kind"]) - 0.5, len(_groups) - 0.5, color="#F7F7F7", lw=0, zorder=0)
        ax.axhline(0, color=MONO_COLOR, lw=0.6, ls=":"); ax.axhline(1, color=CONTROL_COLOR, lw=0.6, ls=(0, (4, 2)))
        ax.set_xticks(range(len(_groups))); ax.set_xticklabels([gn.replace(" the floor", "\nthe floor") for _, gn in _groups], fontsize=5.0)
        ax.set_xlim(-0.5, len(_groups) - 0.5); ax.set_ylim(-0.4, 1.15)
    else:
        ax.set_xticks([]); empty_panel(ax, "no model has enough\nparallel objects\nin this run")
    ax.set_ylabel("geometry share g", labelpad=1)
    _h14 = [mpatches.Patch(color=REGIME_COLORS[r_], lw=0, label=r_) for r_ in ("parallel", "intermediate", "perpendicular")]
    _h14 += [Line2D([], [], ls="", marker=MODEL_MARKERS[m], color=mcol(m), ms=2.4, label=mlab(m)) for m in _m14]
    fig.legend(handles=_h14, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(_h14), fontsize=4.8, handletextpad=0.3,
               columnspacing=0.8, handlelength=1.0)
    _cap14 = (cap("a", "Real objects", "Bonn RGB-D Dynamic frames with a balloon (top) and a box (bottom); the moving object is "
                  "outlined in yellow and people in white. People are found by an instance segmenter; the rest of a moving region "
                  "is the object.")
              + " " + cap("b", "Regimes", f"Share of measurable moving objects in each motion regime, by kind (number of objects); "
                          f"parallel: median |eps| below the flow-noise floor ({BONN_TAU_PAR:.2f} px).")
              + " " + cap("c", "Geometry share", "Median per-object geometry share g (0, dotted: the model's own single-image depth; "
                          "1, dashed: the geometric answer) of parallel, identifiable objects with |x - rho_mono| >= 0.3, by kind and by "
                          "measured floor contact (shaded); 95% intervals over image pairs; open markers: fewer than 8 objects, "
                          "no interval." if _groups else
                          "No model has enough parallel objects in this run.")
              + " Exploratory: x_geo comes from optical flow and sensor depth, so no slope is fitted (D24)." + gate_caption_note("BONN"))
    save_figure(fig, "fig14_bonn", "Real objects that float", "How often is real object motion invisible, and how much geometry do "
                "the models use for real floating objects?", "Sections 3.12 and 6.12", "double column",
                dict(prevalence=BONN_PREV.to_dict("records"), geometry_share=BONN_G.to_dict("records"),
                     examples=[m_["dir"].rsplit(os.sep, 1)[-1] for m_ in _ex14]), caption=_cap14)
else:
    print("Figure 14 is not drawn: " + ("the Bonn objects fail the quality gate of Section 3.12" if BONN_STATUS.get("suspect") else
                                        "Bonn RGB-D Dynamic is not part of this run (" + (BONN_STATUS["note"] or "no data") + ")") + ".")

In [ ]:
# Cell No: 086
# Figure S1 (single column): assumption A2. The locked first-order epipolar offset f||delta_perp||/Z
#     and the corrected first-order offset (out-of-plane component only) against the exact offset.
fig = plt.figure(figsize=FIG_SINGLE)
ax = fig.add_axes([0.17, 0.19, 0.79, 0.74])
sub = A2_DIAG.sample(min(2500, len(A2_DIAG)), random_state=0)
ax.scatter(sub["eps_exact"], sub["eps_locked_first_order"], s=1.2, color="#D55E00", alpha=0.45, lw=0,
           label="locked: f‖δ$_⊥$‖/Z", rasterized=True)
ax.scatter(sub["eps_exact"], sub["eps_corrected_first_order"], s=1.2, color="#0072B2", alpha=0.45, lw=0,
           label="corrected: out-of-plane part", rasterized=True)
lim = [0.05, 400]
ax.plot(lim, lim, color="black", lw=0.6, ls=(0, (4, 2)))
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("exact epipolar offset ε (px)"); ax.set_ylabel("first-order prediction (px)")
ax.legend(loc="lower right", fontsize=5.0, markerscale=3)
panel_title(ax, None, "First-order versus exact offset")
save_figure(fig, "figS1_a2_first_order", "Assumption A2: first-order epipolar offset",
            "Is eps ~ f||delta_perp||/Z adequate?", "Section 2.3", "single column", dict(summary=A2_SUMMARY),
            caption="**First-order offset.** The locked first-order prediction of the epipolar offset (orange) and the corrected "
                    "prediction that keeps only the motion component leaving the epipolar plane (blue), against the exact "
                    "offset, over street-scene configurations. **Consequence.** Every regime in the paper is assigned from the "
                    "exact offset.")

# Figure S2 (double column): gallery of the testbeds (CONVOY-3D camera kinds, an ablation render,
#     CONVOY-REAL, a real-image contact-ablation composite, Virtual KITTI 2, TUM walking).
gal = []                                       # (image, tag on the tile, description in the caption)
for kind in ["forward", "oblique", "side"]:
    m = next((mm for mm in META_MAIN if mm["scene"]["camera"]["kind"] == kind), None)
    if m is not None:
        gal.append((load_rgb(m["dir"], 0), f"CONVOY-3D, {kind}",
                     f"CONVOY-3D with a {kind}{'' if kind == 'oblique' else '-looking'} camera"))
_abl_fh = next((mm for mm in META_ABL if mm["condition"] == "FH" and mm["index"] in EXEMPLAR_ABL), None)
if _abl_fh is not None:
    gal.append((load_rgb(_abl_fh["dir"], 0), "E5, floating behind wall",
                "the ground-contact ablation with the mover lifted and behind the wall"))
_short = {"fr3_structure_texture_far": "structure-texture", "fr3_long_office_household": "long office"}
for seq_name in list(dict.fromkeys(mm["seq"] for mm in META_REAL))[:1]:
    mm = next(mm for mm in META_REAL if mm["seq"] == seq_name)
    _sn = _short.get(seq_name, seq_name.replace('fr3_', ''))
    gal.append((load_rgb(mm["dir"], 0), f"CONVOY-REAL, {_sn}", f"a CONVOY-REAL composite (TUM {_sn})"))
_rg = next((mm for mm in META_RABL if mm["condition"] == "G" and mm["seq"] not in {m["seq"] for m in META_REAL[:1]}), None) \
    or next((mm for mm in META_RABL if mm["condition"] == "G"), None)
if _rg is not None:
    _sn = _short.get(_rg["seq"], _rg["seq"].replace('fr3_', ''))
    gal.append((load_rgb(_rg["dir"], 0), f"E5-REAL, grounded, {_sn}",
                f"a real-image contact-ablation composite with grounded crates (TUM {_sn})"))
if VK_STATUS["available"]:
    for name in list(VK_SCENE_OBJS)[:1]:
        f0 = VK_SELECTION[name]["pairs"][0][0] if VK_SELECTION[name]["pairs"] else None
        if f0 is not None:
            gal.append((VK_SCENE_OBJS[name].rgb(int(f0)), f"Virtual KITTI 2, {name}", f"a Virtual KITTI 2 frame ({name})"))
if AV2_STATUS["available"] and AV2_PAIR_META:
    _am = AV2_PAIR_META[0]
    gal.append((np.asarray(Image.open(os.path.join(_am["dir"], "rgb0.png")).convert("RGB")), f"Argoverse 2, {_am['seq']}",
                f"an Argoverse 2 front-camera frame (log {_am['seq']}, 4:3 window)"))
for name, p in list(TUM_THUMBS.items())[:1]:
    _sn = name.replace('fr3_', '')
    gal.append((np.asarray(Image.open(p).convert("RGB")), f"TUM {_sn}", f"a TUM frame with walking people ({_sn})"))
if len(gal) > 8:                               # eight tiles: the oblique CONVOY-3D camera gives way to the real-driving frame
    gal = [g for g in gal if g[1] != "CONVOY-3D, oblique"][:8]
fig = plt.figure(figsize=FIG_DOUBLE)
gs = fig.add_gridspec(2, 4, left=0.071, right=0.929, bottom=0.01, top=0.99, wspace=0.03, hspace=0.04)
for k, (img, lab, _) in enumerate(gal):
    a = fig.add_subplot(gs[k // 4, k % 4])
    a.imshow(img); clean_image_axes(a, frame=False)
    tag(a, f"({'abcdefgh'[k]}) {lab}", fs=4.8)            # tile titles (D60)
save_figure(fig, "figS2_gallery", "Testbed gallery", "What do the inputs look like?", "Section 3", "double column",
            dict(labels=[g[1] for g in gal]),
            caption=" ".join(cap("abcdefgh"[k], g[1], g[2][0].upper() + g[2][1:] + ".") for k, g in enumerate(gal)))

# Figure S3 (single column): the exploratory remedy M5 (disagreement between a model's two-view and
#     single-image depth) on parallel CONVOY-3D movers, by the depth discrepancy |x| that geometry
#     imposes, against M2 averaged over models (moved here from Fig. 6 in v6.2).
_M5X = E2_M5_BY_X[E2_M5_BY_X["model"].isin(MODELS_OK["MAIN"])] if len(E2_M5_BY_X) else E2_M5_BY_X
_m5 = _M5X[_M5X["n"] >= 3] if len(_M5X) else _M5X
if len(_m5):
    fig = plt.figure(figsize=FIG_SINGLE)
    ax = fig.add_axes([0.15, 0.25, 0.5, 0.68])
    _xv = [0, 0.1, 0.3, 0.7]
    for model in MODELS_OK["MAIN"]:
        gg = _m5[_m5["model"] == model]
        if len(gg):
            ax.plot([_xv.index(v) for v in gg["abs_x_lo"]], gg["recall_M5"], marker=MODEL_MARKERS[model], color=mcol(model), ms=2.0, lw=0.8)
    _m2m = _m5.groupby("abs_x_lo")["recall_M2_exact"].mean()
    ax.plot([_xv.index(v) for v in _m2m.index], _m2m.values, color=MONO_COLOR, ls=(0, (3, 1.5)), lw=0.9)
    ax.set_xticks(range(4)); ax.set_xticklabels(["<0.1", "0.1–0.3", "0.3–0.7", ">0.7"], rotation=40, ha="right",
                                                rotation_mode="anchor", fontsize=5.2)
    ax.set_xlim(-0.4, 3.4); ax.set_ylim(-0.03, 1.06)
    ax.set_xlabel("|x| = |log(1 − s)|", labelpad=0.5); ax.set_ylabel(f"recall at {100 * PREREG['mask_fpr']:g}% static FPR")
    _h = [Line2D([], [], color=mcol(m), marker=MODEL_MARKERS[m], ms=2.2, lw=0.8, label=mlab(m)) for m in MODELS_OK["MAIN"]
          if (_m5["model"] == m).any()]
    _h += [Line2D([], [], color=MONO_COLOR, ls=(0, (3, 1.5)), lw=0.9, label="M2, mean")]
    fig.legend(handles=_h, loc="center left", bbox_to_anchor=(0.67, 0.58), fontsize=5.0, handlelength=1.5, labelspacing=0.3)
    panel_title(ax, None, "Remedy M5 by depth discrepancy")
    save_figure(fig, "figS3_m5_remedy", "Exploratory remedy M5", "Does a model's own single-image depth expose parallel movers?",
                "Section 6.2", "single column", dict(m5_by_x=_m5.to_dict("records")),
                caption="**Remedy M5 (exploratory).** The disagreement between a model's two-view and single-image depth, on "
                        "purely parallel CONVOY-3D movers, by the depth discrepancy |x| that geometry imposes. **Reference.** "
                        "Dashed: the rigid-flow residual M2 averaged over models, on the same movers. Recall at the image's 5% "
                        "static false-positive rate." + gate_caption_note("MAIN"))
else:
    print("Figure S3 is not drawn: no M5 bins with at least 3 movers in this run.")
release_memory()

In [ ]:
# Cell No: 087
# Figure S4 (double column, v6.5): real vehicles, their wheel contact points and their static twin (Argoverse 2). Each tile:
#     one moving vehicle, its wheel contact points coloured by the visibility rule of Section 3.11b, with its parallel
#     fraction s and the height of its static twin above the road.
# Figure S5 (double column, v6.5): the designed sets. Top: the second view of one baseline-sweep scene at the four baselines.
#     Bottom: one dose-response scene with no wall and the three walls.
_PT_STYLE = {"visible": ("o", "#00C853"), "body": ("x", "#BDBDBD"), "object": ("x", "#FF9800"), "static": ("x", "#E53935")}


def _crop_offset(img, bbox, out_hw, pad):
    # the crop of crop_around and the image coordinates of its top-left corner
    H_, W_ = img.shape[:2]
    u0, v0, u1, v1 = bbox
    cu, cv = (u0 + u1) / 2, (v0 + v1) / 2
    h, w = (v1 - v0) * (1 + 2 * pad), (u1 - u0) * (1 + 2 * pad)
    ar = out_hw[1] / out_hw[0]
    if w / max(h, 1) > ar:
        h = w / ar
    else:
        w = h * ar
    h, w = min(h, H_), min(w, W_)
    a0 = int(np.clip(cu - w / 2, 0, W_ - w)); b0 = int(np.clip(cv - h / 2, 0, H_ - h))
    return img[b0:b0 + int(h), a0:a0 + int(w)], a0, b0


if len(AV2_TWIN) and len(MOVERS_AV2_ALL):
    _g = MOVERS_AV2_ALL.merge(AV2_TWIN.drop(columns=["track"]), on=["scene", "mover"], how="left")
    _g = _g[(_g["state"] == "moving") & _g["measurable"].astype(bool) & _g["gt_contact"].isin(["visible", "hidden"])
            & np.isfinite(_g["H_cam"])].copy()
    _g["h_twin"] = _g["H_cam"] * (1.0 - np.exp(_g["gt_x_geo"]))
    _g = _g[np.isfinite(_g["h_twin"])]                     # a vehicle without a static solution has no twin to show
    _g["_par"] = (_g["regime"] == "parallel").astype(int)
    _rows4 = [(c, _g[_g["gt_contact"] == c].sort_values(["_par", "gt_n_lidar_px"], ascending=False).drop_duplicates("track").head(4))
              for c in ("visible", "hidden")]
    _rows4 = [(c, q) for c, q in _rows4 if len(q)]
    _metas = {m_["index"]: m_ for m_ in AV2_PAIRS_ALL}
    if _rows4:
        fig = plt.figure(figsize=FIG_DOUBLE)
        # two rows (contact visible, contact hidden) of up to four 4:3 tiles, centred, with the row name on the left
        _th, _gap = 0.76, 0.04
        _tw = _th * 4 / 3
        _x0 = (5.5 - (4 * _tw + 3 * _gap + 0.14)) / 2 + 0.14
        _tags, _caps, _recs = "abcdefgh", [], []
        k = 0
        for row, (c, q) in enumerate(_rows4):
            y0 = 0.2 + (len(_rows4) - 1 - row) * (_th + _gap)
            fig.text((_x0 - 0.07) / 5.5, (y0 + _th / 2) / 1.8, f"contact {c}", rotation=90, ha="center", va="center", fontsize=5.2)
            for col, r in enumerate(q.itertuples()):
                m_ = _metas[int(r.scene)]
                img = np.asarray(Image.open(os.path.join(m_["dir"], "rgb0.png")).convert("RGB"))
                px = np.asarray(m_["vehicles"][int(r.mover)]["pixels"], int).reshape(-1, 2)
                pts = json.loads(r.wheel_points) if isinstance(r.wheel_points, str) else []
                allp = np.vstack([px] + ([np.array([[p_[0], p_[1]] for p_ in pts])] if pts else []))
                bb = (allp[:, 0].min(), allp[:, 1].min(), allp[:, 0].max(), allp[:, 1].max())
                crop, a0, b0 = _crop_offset(img, bb, (3, 4), 0.35)
                a = fig.add_axes([(_x0 + col * (_tw + _gap)) / 5.5, y0 / 1.8, _tw / 5.5, _th / 1.8])
                a.imshow(crop, aspect="auto"); clean_image_axes(a)
                for (u, v, st) in pts:
                    mk, colr = _PT_STYLE.get(st, ("o", "white"))
                    a.plot(u - a0, v - b0, mk, color=colr, ms=3.2, mew=0.9, mfc="none" if mk == "o" else colr)
                a.set_xlim(-0.5, crop.shape[1] - 0.5); a.set_ylim(crop.shape[0] - 0.5, -0.5)
                tag(a, f"({_tags[k]}) s = {r.s:.2f}, twin {r.h_twin:+.1f} m".replace("-", "−"), fs=4.5)
                _caps.append(cap(_tags[k], f"Contact {c}", f"A moving Argoverse 2 vehicle ({r.regime} regime, s = {r.s:.2f}); its "
                                 f"static twin's contact would lie {abs(r.h_twin):.1f} m {'below' if r.h_twin < 0 else 'above'} its "
                                 "contact plane."))
                _recs.append(dict(uid=r.uid, gt_contact=c, s=float(r.s), regime=r.regime, h_twin=float(r.h_twin), H_cam=float(r.H_cam)))
                k += 1
        fig.legend(handles=[Line2D([], [], ls="", marker=mk, color=colr, mfc="none" if mk == "o" else colr, ms=3.2, mew=0.9,
                                   label={"visible": "visible", "body": "hidden by its body", "object": "hidden by an object",
                                          "static": "hidden by static structure"}[st]) for st, (mk, colr) in _PT_STYLE.items()],
                   loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=4, fontsize=5.0, handletextpad=0.2, columnspacing=1.0)
        save_figure(fig, "figS4_av2_contact_twin", "Real vehicles: contact points and static twins",
                    "What does the contact rule see on real vehicles, and where are their static twins?", "Sections 3.11b, 3.11c and 6.13",
                    "double column", dict(vehicles=_recs),
                    caption=" ".join(_caps) + " **Markers.** Wheel contact points by the rule of Section 3.11b: visible (circles), "
                                              "hidden by the vehicle's own body, by another labelled object or by static structure "
                                              "(crosses).")
else:
    print("Figure S4 is not drawn: no Argoverse 2 camera heights in this run.")

# ---- Figure S5: the designed sets ------------------------------------------------------------------------------------
_s5b = sorted({m["index"] for m in META_BSW})
_s5w = sorted({m["index"] for m in META_WDR})
if _s5b and _s5w:
    def _richest(metas, idx_list, cond):
        best = None
        for i in idx_list[: min(12, len(idx_list))]:
            m_ = next((m for m in metas if m["index"] == i and m["condition"] == cond), None)
            if m_ is None:
                continue
            n = sum(1 for g_ in m_["gt01"] if g_.get("n_eroded", 0) >= PREREG["min_object_pixels"])
            if best is None or n > best[0]:
                best = (n, i)
        return best[1] if best else idx_list[0]
    _ib = _richest(META_BSW, _s5b, BSW_LEVELS[0])
    _iw = _richest(META_WDR, _s5w, "W70")
    fig = plt.figure(figsize=FIG_DOUBLE)
    gs = fig.add_gridspec(2, 4, left=0.071, right=0.929, bottom=0.01, top=0.99, wspace=0.03, hspace=0.04)
    _tiles = [(next(m for m in META_BSW if m["index"] == _ib and m["condition"] == lv), 1, f"baseline {SWEEP_DESIGN['labels'][lv]}, view 2")
              for lv in BSW_LEVELS]
    _tiles += [(next(m for m in META_WDR if m["index"] == _iw and m["condition"] == lv), 0, f"{WALL_DOSE_DESIGN['labels'][lv]} wall"
                if lv != "G" else "no wall") for lv in WDR_LEVELS]
    for k, (m_, fk, lab) in enumerate(_tiles):
        a = fig.add_subplot(gs[k // 4, k % 4])
        a.imshow(load_rgb(m_["dir"], fk)); clean_image_axes(a, frame=False)
        tag(a, f"({'abcdefgh'[k]}) {lab}", fs=4.8)
    save_figure(fig, "figS5_designed_gallery", "The designed sets", "What do the baseline sweep and the dose-response look like?",
                "Section 3.2b", "double column", dict(sweep_scene=int(_ib), dose_scene=int(_iw)),
                caption=" ".join(cap("abcd"[k], f"Baseline {SWEEP_DESIGN['labels'][lv]}", "The second view of the same scene; the first "
                                     "view, s and eps = 0 are identical at every baseline.") for k, lv in enumerate(BSW_LEVELS))
                        + " " + " ".join(cap("efgh"[k], "No wall" if lv == "G" else WALL_DOSE_DESIGN["labels"][lv] + " wall",
                                             "The first view of the same dose-response scene." if lv == "G" else
                                             f"Static walls built to hide at least the lowest {WALL_DOSE_DESIGN['labels'][lv]} of every "
                                             "visible point of each mover in both views.")
                                         for k, lv in enumerate(WDR_LEVELS)))
release_memory()

In [ ]:
# Cell No: 088
# 11. Tables: every table is written as CSV, JSON and a booktabs LaTeX fragment
#     (outputs/tables/T*.{csv,json,tex}) and printed here. Only what this run evaluated appears.
TABLE_MANIFEST = []

def _tex_escape(s):
    s = str(s)
    for a, b in (("\\", "\\textbackslash{}"), ("&", "\\&"), ("%", "\\%"), ("_", "\\_"), ("#", "\\#"), ("^", "\\^{}"), ("<", "$<$"), (">", "$>$"),
                 ("π³", "$\\pi^3$"), ("β", "$\\beta$"), ("ε", "$\\varepsilon$"), ("ρ", "$\\rho$"), ("≥", "$\\geq$"),
                 ("≤", "$\\leq$"), ("−", "-")):
        s = s.replace(a, b)
    return s

def _cell(v, nd):
    if v is None:
        return "--"
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (float, np.floating)):
        return "--" if not np.isfinite(v) else f"{v:.{nd}f}"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    return str(v)

def ci_str(est, lo, hi, nd=2):
    if est is None or not np.isfinite(est):
        return "--"
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return f"{est:.{nd}f}"
    return f"{est:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

def _cis(df, est, lo, hi, nd=2):
    return [ci_str(a, b, c, nd) for a, b, c in zip(df[est], df[lo], df[hi])]

def _finite(df, col):
    # paper tables list only estimates this run could compute (the result CSVs keep every row)
    return df[np.isfinite(pd.to_numeric(df[col], errors="coerce"))] if len(df) else df

_NI = "NOT INTERPRETABLE (gate)"

def _interp(df, col="verdict"):
    # rows of a model that fails the static-scene gate on the testbed are left out of the paper tables
    return df[df[col].astype(str) != _NI] if len(df) and col in df.columns else df

def _gate_note(tb):
    no = [MODEL_LABELS.get(m, m) for m in EVALUATED_MODELS if m not in MODELS_OK[tb] and (m, tb) in GATE_PASS]
    return f" Not interpreted because they fail the static-scene gate on {TESTBED_NAMES[tb]}: {', '.join(no)}." if no else ""

def _display_models(col):
    # paper names of the models (pi3 -> π³, VGGT-1B -> VGGT), also inside composite labels
    out = col.astype(str)
    for k, v in MODEL_LABELS.items():
        out = out.str.replace(rf"(?<![A-Za-z0-9]){re.escape(k)}(?![A-Za-z0-9])", v, regex=True)
    return out

def write_table(df, tid, caption, nd=2):
    df = df.reset_index(drop=True)
    if "model" in df.columns and len(df):
        df = df.assign(model=_display_models(df["model"]))
    if not len(df):
        print(f"\n{tid}: not written (no estimable rows in this run)")
        return
    base = os.path.join(TABLE_DIR, tid)
    df.to_csv(base + ".csv", index=False, na_rep="–")          # "–": not applicable
    df.to_json(base + ".json", orient="records", indent=1, force_ascii=False)
    cols = list(df.columns)
    lines = ["\\begin{table}[t]", "\\centering", "\\scriptsize", f"\\caption{{{_tex_escape(caption)}}}",
             f"\\label{{tab:{tid}}}", "\\begin{tabular}{" + "l" * len(cols) + "}", "\\toprule",
             " & ".join(_tex_escape(c) for c in cols) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(_tex_escape(_cell(r[c], nd)) for c in cols) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    with open(base + ".tex", "w") as f:
        f.write("\n".join(lines) + "\n")
    TABLE_MANIFEST.append(dict(table=tid, caption=caption, rows=len(df), csv=base + ".csv", tex=base + ".tex"))
    print(f"\n{tid}: {caption}")
    with pd.option_context("display.max_colwidth", 70, "display.width", 250):
        print(df.to_string(index=False, na_rep="–", float_format=lambda v: f"{v:.{nd}f}"))

# T1 models and data
t1m = MODEL_TABLE[MODEL_TABLE["model"].isin(EVALUATED_MODELS)][["model", "family", "training", "checkpoint", "input",
                                                                 "precision", "params_M", "peak_gpu_GiB"]].copy()
if VK_STATUS["available"]:
    t1m["virtual_kitti_in_training"] = ["yes" if VK_TRAINING_EXPOSURE.get(m) else "no" for m in t1m["model"]]
write_table(t1m, "T1a_models", "Models evaluated: family, training data, checkpoint, input size, precision, size, peak GPU memory"
            + (", and whether Virtual KITTI is among the published training data." if VK_STATUS["available"] else "."))
_t1d = [dict(dataset="CONVOY-3D main (+ knockout)", role="synthetic, exact", items=len(META_MAIN), movers=len(MOVERS_MAIN),
             check="exact", used_in="E1-E3, A1"),
        dict(dataset="CONVOY-3D E4 twins", role="synthetic, exact", items=len(META_E4), movers=len(MOVERS_E4), check="exact", used_in="E4"),
        dict(dataset="CONVOY-3D ablation (5 conditions)", role="synthetic, exact", items=len(META_ABL),
             movers=int((MOVERS_ABL["condition"] == "G").sum()), check="exact", used_in="E5"),
        dict(dataset="CONVOY-3D baseline sweep (4 baselines)", role="synthetic, exact (v6.5)", items=len(META_BSW),
             movers=int(MOVERS_BSW["uid"].nunique()), check="exact", used_in="H12"),
        dict(dataset="CONVOY-3D dose-response (4 wall heights)", role="synthetic, exact (v6.5)", items=len(META_WDR),
             movers=int(MOVERS_WDR["uid"].nunique()), check="exact", used_in="H13"),
        dict(dataset="LEGACY-FLAT", role="negative control", items=len(META_FLAT), movers=0, check="not applicable", used_in="gate")]
_a3_excluded = []
for k, v in A3_VALIDATION.items():
    if not v["passed"]:
        _a3_excluded.append(f"TUM {k}"); continue
    _t1d.append(dict(dataset=f"TUM {k}", role=TUM_SEQUENCES[k]["role"], items=v["n_frames"],
                     movers=len(MOVERS_REAL[MOVERS_REAL["seq"] == k]) if len(MOVERS_REAL) and "seq" in MOVERS_REAL else 0,
                     check=f"A3 {'PASS' if v['passed'] else 'FAIL'} ({v['median_rel_depth_err']:.3f})",
                     used_in=("CONVOY-REAL" + (", E5-REAL" if len(MOVERS_RABL) and (MOVERS_RABL["seq"] == k).any() else ""))
                     if TUM_SEQUENCES[k]["role"] == "static" else "prevalence"))
if len(MOVERS_RABL):
    _t1d.append(dict(dataset="E5-REAL composites (2 conditions)", role="real background, inserted crate",
                     items=int(MOVERS_RABL["scene"].nunique()), movers=int((MOVERS_RABL["condition"] == "G").sum()),
                     check="A3 of the source sequence", used_in="E5-REAL"))
for k, v in VK_A3.items():
    if not v["passed"]:
        _a3_excluded.append(f"Virtual KITTI 2 {k}"); continue
    _t1d.append(dict(dataset=f"Virtual KITTI 2 {k}", role="photo-realistic, exact", items=v["n_frames"],
                     movers=int((MOVERS_VK["seq"] == k).sum()) if len(MOVERS_VK) else 0,
                     check=f"A3 {'PASS' if v['passed'] else 'FAIL'} ({v['static_warp_err']:.3f} / {v['vehicle_warp_err']:.3f})",
                     used_in="E6"))
for k, v in KT_A3.items():
    if not v["passed"]:
        _a3_excluded.append(f"KITTI tracking {k}"); continue
    _t1d.append(dict(dataset=f"KITTI tracking {k}", role="real driving, LiDAR (exploratory)", items=v["n_frames"],
                     movers=int((MOVERS_KT["seq"] == k).sum()) if len(MOVERS_KT) else 0,
                     check=f"A3 PASS ({v['warp_err_' + v['convention']]:.3f})", used_in="E6x"))
# Argoverse 2: one row for all logs here; the per-log record is T1c (v6.4: 64 logs would not fit a paper table)
_av2_a3_all = {**AV2_A3, **AV2X_A3}
_av2_used = {k: v for k, v in _av2_a3_all.items() if v["passed"] and v.get("n_pairs_used", 0)}
_n_av2_fail = sum(1 for v in _av2_a3_all.values() if not v["passed"])
if _av2_used:
    _sw = np.array([v["static_warp_err"] for v in _av2_used.values()], float)
    _t1d.append(dict(dataset=f"Argoverse 2 ({len(_av2_used)} validation logs)", role="real driving, LiDAR (exploratory)",
                     items=int(sum(v["n_pairs_used"] for v in _av2_used.values())), movers=len(MOVERS_AV2),
                     check=f"A3 PASS (median {np.median(_sw):.3f}, max {np.max(_sw):.3f})", used_in="E6x, 6.9"))
if _n_av2_fail:
    _a3_excluded.append(f"{_n_av2_fail} Argoverse 2 log(s)")
for k, v in BONN_A3.items():                               # v6.5: Bonn RGB-D Dynamic
    if not v["passed"]:
        _a3_excluded.append(f"Bonn {k}"); continue
    _t1d.append(dict(dataset=f"Bonn RGB-D Dynamic {k}", role="real objects, measured depth (exploratory)", items=v["n_frames"],
                     movers=int((MOVERS_BONN["seq"] == k).sum()) if len(MOVERS_BONN) else 0,
                     check=f"A3 PASS ({v['median_rel_depth_err']:.3f}, {v['convention']})", used_in="6.12"))
write_table(pd.DataFrame(_t1d), "T1b_data", "Data: testbeds, sizes, and the ground-truth self-consistency test (A3: median relative "
            "depth disagreement after warping" + ("; static / vehicles for Virtual KITTI 2)." if VK_SCENE_OBJS else ").")
            + (" Excluded because their ground truth failed A3: " + ", ".join(_a3_excluded) + "." if _a3_excluded else ""), nd=3)
if _av2_a3_all:
    _t1c = pd.DataFrame([dict(log=k[:8], stage="v6.3" if k in AV2_A3 else "v6.4", pairs_used=v.get("n_pairs_used", 0),
                              moving_vehicles=int((MOVERS_AV2["seq"] == k[:8]).sum()) if len(MOVERS_AV2) else 0,
                              static_warp=v["static_warp_err"], grey_static_warp=v["photo_static_warp"],
                              grey_static_nowarp=v["photo_static_nowarp"], A3="PASS" if v["passed"] else "FAIL")
                         for k, v in _av2_a3_all.items()])
    _vcs = "; ".join(f"{lab}: {vc.get('note', '–')}" for lab, vc in (("v6.3 logs", AV2_VEHICLE_CHECK), ("v6.4 logs", AV2X_VEHICLE_CHECK)) if vc)
    write_table(_t1c, "T1c_argoverse2_logs", "Argoverse 2 logs and their ground-truth checks (A3): median relative depth disagreement "
                "of warped static LiDAR, and the grey-value error (0-255) of static LiDAR pixels with the exact warp and without it. "
                "Moving vehicles, pooled over the logs of each stage: " + (_vcs or "–") + ".", nd=3)
# T2 gate and oracle
t2 = GATE.assign(slope=_cis(GATE, "slope_median", "slope_q25", "slope_q75"), absrel=_cis(GATE, "absrel_median", "absrel_q25", "absrel_q75"))
write_table(t2[["testbed", "model", "n_images", "slope", "absrel", "gate"]], "T2_validity_gate",
            "Static-scene validity gate: median [IQR] log-depth slope and AbsRel on static pixels.")
_t2b = ORACLE_TABLE.assign(beta_s_ci=_cis(ORACLE_TABLE, "beta_s", "ci_lo", "ci_hi", 3))[["w_true", "n", "beta_s_ci", "beta_theory_identity",
                                                                                     "median_w_recovered", "verdict_P1"]]
if len(ORACLE_VK):
    _t2b = _t2b.merge(ORACLE_VK.assign(beta_geo_vkitti=_cis(ORACLE_VK, "beta_geo", "ci_lo", "ci_hi", 3))[["w_true", "beta_geo_vkitti",
                                                                                                       "beta_theory_identity"]]
                      .rename(columns={"beta_theory_identity": "identity_vkitti"}), on="w_true", how="left")
write_table(_t2b, "T2b_oracle_validation", "Estimator validation with predictors of known prior weight w (CONVOY-3D E1 estimator"
            + ("; Virtual KITTI 2 H9 estimator)." if len(ORACLE_VK) else ")."), nd=3)
# T3 E1
t3 = _finite(_interp(E1_TABLE), "beta_s")
t3 = t3.assign(beta_s_ci=_cis(t3, "beta_s", "ci_lo", "ci_hi"), beta_mono_ci=_cis(t3, "beta_mono", "beta_mono_lo", "beta_mono_hi"),
               beta_geo_ci=_cis(t3, "beta_geo", "beta_geo_lo", "beta_geo_hi"))
t3 = pd.concat([t3, pd.DataFrame([dict(model=k, training="control", n=v["n"], n_clips=v["n_clusters"],
                                       beta_s_ci=ci_str(v["beta"], v["ci_lo"], v["ci_hi"]), r2=v["r2"], verdict="control")
                                  for k, v in CONTROL_E1.items()])], ignore_index=True)
write_table(t3[["model", "training", "n", "n_clips", "beta_s_ci", "r2", "beta_mono_ci", "beta_geo_ci", "w_eff", "p_holm", "verdict"]],
            "T3_E1_geometry_share", "E1: geometry share beta_s (slope of log depth error on -log(1-s)) for purely parallel movers, "
            "CONVOY-3D; verdict under the locked Prediction 1." + _gate_note("MAIN"))
# T4 E2
t4 = _finite(_interp(E2_TABLE), "diff"); t4 = t4.assign(diff_ci=_cis(t4, "diff", "ci_lo", "ci_hi"))
write_table(t4[["model", "method", "n_par", "n_perp", "rec_par", "rec_perp", "diff_ci", "auroc_par", "auroc_perp", "verdict"]],
            "T4_E2_masks", "E2 (Prediction 2): dynamic-mask recall at 5% static FPR, parallel vs perpendicular movers (stratified by "
            "|s| and size)." + _gate_note("MAIN"))
# T5 E3
t5 = _interp(E3_TABLE, "verdict_P3a")
t5 = t5.assign(spearman_ci=_cis(t5, "spearman_w_eps", "sp_lo", "sp_hi"), class_diff_ci=_cis(t5, "class_diff", "cd_lo", "cd_hi"))
write_table(t5[["model", "n_w", "median_w_parallel", "median_w_perp", "spearman_ci", "verdict_P3a", "w_car", "w_crate",
                "w_crate_floating", "class_diff_ci", "verdict_P3b", "lock_condition_c"]],
            "T5_E3_prior_weight", "E3 (Prediction 3): prior weight w versus epipolar offset and object class." + _gate_note("MAIN"))
# T6 E4
if len(_finite(_interp(E4_SUMMARY), "drop_K8")):
    t6 = _finite(_interp(E4_SUMMARY), "drop_K8"); t6 = t6.assign(drop_ci=_cis(t6, "drop_K8", "drop_lo", "drop_hi"))
    write_table(t6[[c for c in t6.columns if c.startswith(("model", "straight", "curved"))] + ["n_K8", "drop_ci", "verdict"]],
                "T6_E4_multiview", "E4 (Prediction 4): beta_s for K = 2, 4, 8 frames on straight paths and their curved twins."
                + _gate_note("MAIN"))
# T7 A1
t7 = _finite(_interp(A1_TABLE, "A1"), "drop")
t7 = t7.assign(full=_cis(t7, "beta_full", "full_lo", "full_hi"), knockout=_cis(t7, "beta_knockout", "ko_lo", "ko_hi"),
               drop_ci=_cis(t7, "drop", "drop_lo", "drop_hi"))
write_table(t7[["model", "n", "n_clips", "full", "knockout", "drop_ci", "static_sensitivity", "A1"]], "T7_A1_knockout",
            "A1 correspondence knockout: beta_s with the mover present in / deleted from the second view." + _gate_note("MAIN"))
# T8 real replication
t8 = _finite(_interp(REAL_E1), "beta_s"); t8 = t8.assign(beta_s_ci=_cis(t8, "beta_s", "ci_lo", "ci_hi"))
write_table(t8[["model", "n", "n_pairs", "beta_s_ci", "r2", "beta_mono", "verdict"]], "T8_real_E1",
            "CONVOY-REAL: E1 replication on real TUM backgrounds." + _gate_note("REAL"))
if len(_finite(_interp(REAL_E2), "diff")):
    t8b = _finite(_interp(REAL_E2), "diff"); t8b = t8b.assign(diff_ci=_cis(t8b, "diff", "ci_lo", "ci_hi"))
    write_table(t8b[["model", "method", "n_par", "n_perp", "rec_par", "rec_perp", "diff_ci", "verdict"]], "T8b_real_E2",
                "CONVOY-REAL: E2 replication on real TUM backgrounds." + _gate_note("REAL"))
# T9 E5 ablation
_e5b = _finite(E5_BETA[E5_BETA["model"].isin(MODELS_OK["ABL"])], "beta") if len(E5_BETA) else E5_BETA
if len(_e5b):
    t9 = _e5b.assign(beta_ci=_cis(_e5b, "beta", "ci_lo", "ci_hi")).pivot_table(index="model", columns="condition",
                                                                                values="beta_ci", aggfunc="first")
    t9 = t9[[c for c in E5_CONDS if c in t9.columns]].reset_index()
    t9.columns = ["model"] + [ABLATION_DESIGN["labels"][c] for c in t9.columns[1:]]
    t9["n"] = [int(_e5b[_e5b["model"] == m]["n"].iloc[0]) for m in t9["model"]]
    write_table(t9, "T9a_E5_beta_by_condition", "E5: geometry share beta_s of the same movers in each ground-contact condition."
                + _gate_note("ABL"))
    t9b = _finite(_interp(E5_TESTS), "estimate"); t9b = t9b.assign(estimate_ci=_cis(t9b, "estimate", "ci_lo", "ci_hi", 3))
    write_table(t9b[["model", "test", "contrast", "n", "n_scenes", "estimate_ci", "p_holm", "verdict"]], "T9b_E5_contrasts",
                "E5: registered contrasts H5-H8 and exploratory contrasts (shadow, class), paired by mover, 95% cluster-bootstrap "
                "intervals." + _gate_note("ABL"), nd=3)
# T10 E6 driving
_e6t = _interp(E6_TABLE[E6_TABLE["verdict"] != "NOT TESTABLE"]) if len(E6_TABLE) else E6_TABLE
if len(_finite(_e6t, "beta_geo")) and (_e6t["verdict"] != "control").any():
    t10 = _finite(_e6t, "beta_geo")
    t10 = t10.assign(beta_geo_ci=_cis(t10, "beta_geo", "ci_lo", "ci_hi"))
    write_table(t10[["model", "trained_on_vkitti", "n", "n_pairs", "beta_geo_ci", "r2", "beta_mono", "w_eff", "verdict"]],
                "T10a_E6_driving", "E6 (H9, exploratory in v6.1: the sample rule was changed after the v6.0 count, D34): "
                "geometry share beta_geo for moving vehicles in Virtual KITTI 2 (parallel regime)." + _gate_note("VK"))
if len(VK_ATTRITION):
    write_table(VK_ATTRITION, "T10d_vkitti_sample", "Virtual KITTI 2: the H9 sample step by step (vehicles and image pairs left "
                "after each criterion); the last row applies the v6.0 rule for comparison.", nd=0)
if len(PREVALENCE_ALL):
    write_table(PREVALENCE_ALL[["dataset", "seq", "n_pairs", "n_moving_pixels", "frac_parallel", "frac_intermediate",
                                "frac_perpendicular", "median_eps"]],
                "T10b_prevalence_two_view", "Share of moving pixels per motion regime between two views ("
                + "; ".join(({"TUM fr3 walking (people)": "TUM walking people", "Virtual KITTI 2 (vehicles)":
                              "Virtual KITTI 2 moving vehicles"}).get(x, x) for x in PREVALENCE_ALL["dataset"].unique()) + ").", nd=3)
if len(KVIEW_SUMMARY):
    write_table(KVIEW_SUMMARY, "T10c_prevalence_k_view", "Virtual KITTI 2: share of moving vehicles whose K-frame track a static "
                "point explains within 1 px.", nd=3)
# T11 E5-REAL: the contact ablation on real images
_rb = _finite(RABL_BETA[RABL_BETA["model"].isin(MODELS_OK["RABL"])], "beta") if len(RABL_BETA) else RABL_BETA
if len(_rb):
    t11 = _rb.assign(beta_ci=_cis(_rb, "beta", "ci_lo", "ci_hi")).pivot_table(index="model", columns="condition",
                                                                              values="beta_ci", aggfunc="first")
    t11 = t11[[c for c in RABL_CONDS if c in t11.columns]].reset_index()
    t11.columns = ["model"] + [RABL_DESIGN["labels"][c] for c in t11.columns[1:]]
    t11["n"] = [int(_rb[_rb["model"] == m]["n"].iloc[0]) for m in t11["model"]]
    t11["n_pairs"] = [int(_rb[_rb["model"] == m]["n_pairs"].iloc[0]) for m in t11["model"]]
    _ctl = "; ".join(f"{RABL_DESIGN['labels'][c]} {ci_str(v['beta'], v['ci_lo'], v['ci_hi'])}" for c, v in RABL_CONTROL.items())
    write_table(t11, "T11a_E5real_beta_by_condition", "E5-REAL: geometry share beta_s of the same crates on real TUM backgrounds, "
                "grounded with a contact shadow or lifted without one. Classical triangulation with the true pose on the same "
                f"composites: {_ctl}." + _gate_note("RABL"))
    t11b = _finite(_interp(RABL_TESTS), "estimate"); t11b = t11b.assign(estimate_ci=_cis(t11b, "estimate", "ci_lo", "ci_hi", 3))
    write_table(t11b[["model", "test", "n", "n_pairs", "estimate_ci", "p_holm", "verdict"]], "T11b_E5real_contrasts",
                "E5-REAL: H11a, beta(lifted) - beta(grounded), and H11b, median single-image rho(lifted) - rho(grounded), "
                "paired by crate, 95% cluster-bootstrap intervals (cluster = image pair), Holm across models." + _gate_note("RABL"),
                nd=3)
# T12 summary, T13 robustness
_t12 = SUMMARY[~SUMMARY["verdict"].isin(["NOT TESTABLE", _NI])]
_t12_note = "".join(_gate_note(tb) for tb in ["MAIN", "ABL", "REAL", "RABL", "VK"])
write_table(_t12.assign(estimate_ci=_cis(_t12, "estimate", "ci_lo", "ci_hi", 3))
            [["family", "prediction", "model", "statistic", "estimate_ci", "n", "verdict"]], "T12_summary",
            "All registered tests (locked P1-P4 and A1; v6 H5-H8; v6.1 H11a-b) and the exploratory H9: estimate [95% CI] and "
            "verdict. H10 was withdrawn after the v6 run (D33)." + _t12_note, nd=3)
for _v in ["NOT TESTABLE", _NI]:
    if (SUMMARY["verdict"] == _v).any():
        print(f"{_v.capitalize()} in this run (left out of T12, kept in results/prediction_summary.csv):",
              SUMMARY[SUMMARY["verdict"] == _v].groupby("prediction").size().to_dict())
_rob = ROBUST[ROBUST["model"].isin(MODELS_OK["MAIN"])] if len(ROBUST) else ROBUST
if len(_finite(_rob, "beta_s")):
    t13 = _finite(_rob, "beta_s"); t13 = t13.assign(beta_ci=_cis(t13, "beta_s", "ci_lo", "ci_hi"))
    write_table(t13.pivot_table(index=["factor", "level"], columns="model", values="beta_ci", aggfunc="first").reset_index()
                .rename(columns=lambda c: MODEL_LABELS.get(c, c)),
                "T13_robustness", "beta_s by camera heading, mover class, depth and focal length (E1 movers).")
# ---- v6.2 exploratory tables ---------------------------------------------------------------------
if len(E5_MASK_TESTS):
    _t14 = E5_MASKS.assign(recall_ci=_cis(E5_MASKS, "recall_M2", "ci_lo", "ci_hi", 3)).pivot_table(
        index="model", columns="condition", values="recall_ci", aggfunc="first")[E5_CONDS].reset_index()
    _t14.columns = ["model"] + [ABLATION_DESIGN["labels"][c] for c in E5_CONDS]
    write_table(_t14, "T14a_masks_by_condition", "Exploratory: M2 recall (exact correspondences, 5% static FPR) on the same "
                "purely parallel movers in each ground-contact condition of E5." + _gate_note("ABL"), nd=3)
    _t14b = E5_MASK_TESTS.assign(change_ci=_cis(E5_MASK_TESTS, "estimate", "ci_lo", "ci_hi", 3))
    write_table(_t14b[["model", "contrast", "meaning", "n", "n_scenes", "change_ci", "direction"]], "T14b_masks_contrasts",
                "Exploratory: paired change of M2 recall when the contact is hidden or the mover lifted (95% cluster-bootstrap "
                "intervals over base scenes).", nd=3)
if len(X6_CONTACT_TEST):
    _dsm = {"Virtual KITTI 2": "VK", "Argoverse 2": "AV2"}
    _x6c = X6_CONTACT[[m in MODELS_OK.get(_dsm.get(d, ""), []) for d, m in zip(X6_CONTACT["dataset"], X6_CONTACT["model"])]]
    _t15 = _x6c.assign(g_ci=_cis(_x6c, "median_g", "ci_lo", "ci_hi", 3)).pivot_table(
        index=["dataset", "model"], columns="contact", values="g_ci", aggfunc="first").reset_index()
    _t15 = _t15.merge(X6_CONTACT_TEST.assign(diff_ci=_cis(X6_CONTACT_TEST, "diff_hidden_minus_visible", "ci_lo", "ci_hi", 3))
                      [["dataset", "model", "n_visible", "n_hidden", "n_hidden_tracks", "diff_ci", "direction"]],
                      on=["dataset", "model"], how="left")
    if len(X6_MATCHED):
        _t15 = _t15.merge(X6_MATCHED.assign(matched_ci=_cis(X6_MATCHED, "median_paired_diff", "ci_lo", "ci_hi", 3))
                          [["dataset", "model", "n_matched", "matched_ci", "direction"]].rename(columns={"direction": "direction_matched"}),
                          on=["dataset", "model"], how="left")
    _t15 = (_t15.assign(_o=_t15["dataset"].map({"Virtual KITTI 2": 0, "Argoverse 2": 1}).fillna(9))
            .sort_values(["_o", "model"]).drop(columns="_o"))
    write_table(_t15[[c for c in ["dataset", "model", "n_visible", "visible", "n_hidden", "n_hidden_tracks", "hidden", "diff_ci",
                                  "direction", "n_matched", "matched_ci", "direction_matched"] if c in _t15]],
                "T15a_driving_contact", "Exploratory: median per-object geometry share g of parallel vehicles whose wheel contact "
                "points are visible or hidden (Virtual KITTI 2: instance masks, D37; Argoverse 2: geometry and LiDAR, D50), the "
                "difference, and the median paired difference after matching every hidden-contact vehicle to the nearest "
                "visible-contact vehicle on log depth, log image area and x_geo (within 0.5 SD on each). 95% cluster-bootstrap "
                "intervals over vehicle tracks." + _gate_note("VK") + (_gate_note("AV2") if AV2_STATUS["available"] else ""), nd=3)
    if len(X6_MATCHED):
        _bal = X6_MATCHED[["dataset", "model", "n_hidden", "n_matched"] + [c for c in X6_MATCHED if c.startswith("smd_")]]
        write_table(_bal, "T15c_matching_balance", "Covariate balance of the matched hidden/visible comparison: standardized "
                    "mean difference (hidden minus visible, in units of the pooled SD of the unmatched groups) of log depth, log "
                    "image area and x_geo before and after matching (|SMD| < 0.1 is conventionally taken as balanced).", nd=2)
if len(CONTACT_RULE_CHECK) and "agreement" in CONTACT_RULE_CHECK:
    write_table(CONTACT_RULE_CHECK, "T15d_contact_rule_check", "Check of the Argoverse 2 contact rule (D50) on Virtual KITTI 2: "
                "the same geometric rule, with the exact depth of static pixels in place of the LiDAR, against the labels from the "
                "instance masks (D37), on the vehicles where both rules decide; agreement, Cohen's kappa, the number of hidden "
                "contacts under each rule and under both, and the vehicles the instance masks decide but the rule leaves "
                "undetermined.", nd=3)
if len(X6_PACE):
    _t15b = X6_PACE.assign(g_ci=_cis(X6_PACE, "median_g", "ci_lo", "ci_hi", 3))
    write_table(_t15b[["dataset", "model", "n_s_below_1", "n_tracks", "median_x_geo", "g_ci", "share_closer_to_single_image",
                       "n_s_at_least_1", "share_placed_farther_than_single_image"]], "T15b_convoy_case",
                "Exploratory: vehicles keeping pace with the camera (0.85 <= s < 1.15). For s < 1, the median per-object geometry "
                "share g (95% cluster-bootstrap intervals over vehicle tracks) and the share of vehicles whose depth is closer to "
                "the model's own single-image depth than to the geometric answer; for s >= 1 (no static solution), the share "
                "placed farther than the single-image depth.", nd=3)
if len(REAL_PREVALENCE):
    write_table(REAL_PREVALENCE, "T16a_real_driving_regimes", "Exploratory, real driving with LiDAR: share of moving vehicles (and "
                "of their LiDAR pixels) per motion regime, and the share keeping pace with the camera (0.85 <= s < 1.15); the "
                "parallel threshold is the label-noise floor measured on parked vehicles. The ego speed (camera baseline over the "
                "time between the two images) describes the traffic sampled: shares measured in urban traffic need not hold at "
                "highway speeds.", nd=3)
if len(_finite(REAL_BETA, "beta_geo")):
    _t16 = _finite(REAL_BETA, "beta_geo"); _t16 = _t16.assign(beta_geo_ci=_cis(_t16, "beta_geo", "ci_lo", "ci_hi", 3))
    write_table(_t16[["dataset", "model", "n", "n_tracks", "beta_geo_ci", "r2"]], "T16b_real_driving_beta",
                "Exploratory, real driving with LiDAR: slope beta_geo of the log depth error on x_geo for parallel moving vehicles "
                "(correspondences from labelled boxes and tracks, Sections 3.10-3.11; 95% cluster-bootstrap intervals over "
                "vehicle tracks)." + "".join(_gate_note(k) for k in REAL_BETA["set"].unique()), nd=3)
if len(PROTOCOL_CHECK):
    _t17 = PROTOCOL_CHECK.copy()
    if "H11b_one" in _t17:
        _t17 = _t17.assign(H11b_one_image=_cis(_t17, "H11b_one", "H11b_one_lo", "H11b_one_hi", 3),
                           H11b_image_twice=_cis(_t17, "H11b_twice", "H11b_twice_lo", "H11b_twice_hi", 3))
    write_table(_t17[[c for c in ["model", "n_crates", "H11b_one_image", "H11b_image_twice", "median_abs_diff", "spearman",
                                  "static_absrel_one", "static_absrel_twice", "real_mono_slope_one", "real_mono_slope_twice",
                                  "protocol"] if c in _t17]], "T17_single_image_protocol",
                "Single-image protocol check for the multi-view models: H11b and the CONVOY-REAL single-image slope with the "
                "image given once (the protocol of the paper) or twice.", nd=3)
# ---- v6.4: the synthesis by ground support (Section 6.9, descriptive) --------------------------------------
if len(SUPPORT_LEVELS):
    _s18 = SUPPORT_LEVELS.assign(value=_cis(SUPPORT_LEVELS, "estimate", "ci_lo", "ci_hi", 2),
                                 row=SUPPORT_LEVELS["testbed"] + ": " + SUPPORT_LEVELS["condition"])
    _ord = list(dict.fromkeys(_s18["row"]))
    _t18 = (_s18.pivot_table(index=["row", "support", "statistic"], columns="model", values="value", aggfunc="first")
            .reset_index().set_index("row").reindex(_ord).reset_index().rename(columns=lambda c: MODEL_LABELS.get(c, c)))
    write_table(_t18.rename(columns={"row": "testbed: objects"}), "T18a_support_levels",
                "Descriptive (Section 6.9): geometry share by ground support in every testbed, beta_s where s was designed "
                "(CONVOY-3D, CONVOY-REAL, E5, E5-REAL) and the median per-object share g in driving; grounded = ground contact in "
                "view. Every CONVOY-REAL mover is a floating crate (D52). 95% cluster-bootstrap intervals; only interpretable "
                "model x testbed results.")
    _f18 = SUPPORT_LEVELS[SUPPORT_LEVELS["statistic"] == "beta_s"]
    _f18 = _f18.assign(value=_cis(_f18, "implied_factor", "implied_lo", "implied_hi", 2), row=_f18["testbed"] + ": " + _f18["condition"])
    write_table(_f18.pivot_table(index="row", columns="model", values="value", aggfunc="first")
                .reindex([o for o in _ord if o in set(_f18["row"])]).reset_index()
                .rename(columns=lambda c: MODEL_LABELS.get(c, c)).rename(columns={"row": "testbed: objects"}),
                "T18c_implied_depth_error", f"Depth overestimate exp(beta_s x) implied by each slope for an object keeping pace "
                f"with the camera at s = {PREREG['implied_factor_s']} (x = {X_IMPLIED:.2f}; 1 = true depth, "
                f"{np.exp(X_IMPLIED):.1f} = the geometric answer), with the slope's interval mapped through the same function.")
if len(SUPPORT_CONTRASTS):
    _c18 = SUPPORT_CONTRASTS.assign(value=[f"{ci_str(e, l, h, 3)} {d_}" if np.isfinite(e) else "--" for e, l, h, d_ in
                                           SUPPORT_CONTRASTS[["estimate", "ci_lo", "ci_hi", "direction"]].itertuples(index=False)],
                                    row=SUPPORT_CONTRASTS["testbed"] + ": " + SUPPORT_CONTRASTS["contrast"])
    write_table(_c18.pivot_table(index=["row", "statistic"], columns="model", values="value", aggfunc="first")
                .reindex(list(dict.fromkeys(zip(_c18["row"], _c18["statistic"])))).reset_index()
                .rename(columns=lambda c: MODEL_LABELS.get(c, c)).rename(columns={"row": "testbed: contrast"}),
                "T18b_support_contrasts", "Descriptive (Section 6.9): unsupported or hidden-contact objects minus grounded ones within "
                "each testbed, estimate [95% CI] and the direction the interval gives; H5, H6 and H11a as registered, the driving "
                "contrasts as in T15a, and floating minus grounded CONVOY-3D movers (new in v6.4, unpaired).", nd=3)
# ---- v6.5: designed experiments, cue model, Bonn and the twin test ---------------------------------------------------
if len(BSW_BETA):
    _b19 = BSW_BETA[BSW_BETA["model"].isin(MODELS_OK["BSW"])]
    _b19 = _b19.assign(value=_cis(_b19, "beta", "ci_lo", "ci_hi", 2), level_label=_b19["level"].map(SWEEP_DESIGN["labels"]))
    _t19 = (_b19.pivot_table(index=["support", "level_label"], columns="model", values="value", aggfunc="first").reset_index()
            .rename(columns=lambda c: MODEL_LABELS.get(c, c)).rename(columns={"level_label": "baseline"}))
    _t19["_o"] = _t19["support"].map({"all": 0, "grounded": 1, "floating": 2}) * 10 + _t19["baseline"].map(
        {v: i for i, v in enumerate(SWEEP_DESIGN["labels"].values())})
    write_table(_t19.sort_values("_o").drop(columns="_o"), "T19a_baseline_sweep",
                "Baseline sweep (Section 6.10): beta_s [95% CI] of the same movers with the camera baseline scaled by 0.5, 1, 2 "
                "and 3 (only the parallax changes), for all movers and separately for grounded and floating ones; common sample: "
                "measurable at every baseline and identifiable at x1." + _gate_note("BSW"))
if len(BSW_TESTS):
    _t = _interp(BSW_TESTS)
    write_table(_t.assign(estimate_ci=_cis(_t, "estimate", "ci_lo", "ci_hi", 3))[["test", "model", "contrast", "n", "n_scenes",
                                                                                    "estimate_ci", "p_holm", "verdict"]],
                "T19b_baseline_sweep_tests", "H12 (registered for DUSt3R and MASt3R; no prediction for the other models) and "
                "the exploratory split by support: the rise of beta_s from x0.5 to x3, paired by mover, 95% cluster-bootstrap "
                "intervals over base scenes, Holm across the two registered models." + _gate_note("BSW"), nd=3)
if len(WDR_BETA):
    _b20 = WDR_BETA[WDR_BETA["model"].isin(MODELS_OK["WDR"])]
    _b20 = _b20.assign(value=_cis(_b20, "beta", "ci_lo", "ci_hi", 2), wall=_b20["level"].map(WALL_DOSE_DESIGN["labels"]))
    _hid20 = WDR_BETA.groupby("level")["median_hidden_share"].median()
    _t20 = (_b20.pivot_table(index="wall", columns="model", values="value", aggfunc="first")
            .reindex([WALL_DOSE_DESIGN["labels"][k] for k in WDR_LEVELS]).reset_index()
            .rename(columns=lambda c: MODEL_LABELS.get(c, c)))
    _t20.insert(1, "share hidden", [f"{_hid20.get(k, np.nan):.2f}" for k in WDR_LEVELS])
    write_table(_t20, "T20a_dose_response",
                "Contact dose-response (Section 6.10): beta_s [95% CI] of the same grounded movers with no wall and behind walls "
                "built to hide at least their lowest 15, 35 and 70 cm in both views; share hidden: median share of the mover's "
                "silhouette hidden in view 1 (a wall must cover the whole visible base, so it hides more of the near end); common "
                "sample: contact visible without a wall and hidden by every wall." + _gate_note("WDR"))
if len(WDR_TESTS):
    _t = _interp(WDR_TESTS)
    write_table(_t.assign(estimate_ci=_cis(_t, "estimate", "ci_lo", "ci_hi", 3))[["test", "model", "contrast", "n", "n_scenes",
                                                                                    "estimate_ci", "p_holm", "verdict"]],
                "T20b_dose_response_tests", "H13 (registered, every model, Holm across models) and the shape of the curve "
                "(exploratory): beta(70 cm) - beta(no wall), beta(70 cm) - beta(15 cm) and the share of the 70 cm effect reached by "
                "the 15 cm wall, reported only where the 70 cm effect is clearly positive (its interval excludes 0)."
                + _gate_note("WDR"), nd=3)
if len(CUE_FIT_TABLE):
    write_table(CUE_FIT_TABLE, "T21a_cue_model_fits", "Cue model (Section 6.11, exploratory): rho - rho_mono = w (x - rho_mono), "
                "w = c_support d^2 / (d^2 + kappa^2), fitted by least squares on CONVOY-3D and E5 (two channels), with kappa = 0 "
                "(contact only) and with one c for every support (parallax only); kappa in px, rmse of rho.", nd=3)
if len(CUE_CV):
    _t = CUE_CV.assign(delta_ci=_cis(CUE_CV, "delta_mse", "ci_lo", "ci_hi", 4))
    write_table(_t[["model", "comparison", "n", "mse_two_channels", "mse_other", "delta_ci", "direction"]], "T21b_cue_model_cv",
                "Cue model: 10-fold cross-validation over scenes; mean squared error of held-out rho and the difference to the "
                "two-channel model with a 95% cluster-bootstrap interval (positive: the two-channel model predicts better).", nd=4)
if len(CUE_PRED):
    _t = CUE_PRED[CUE_PRED["role"] == "predicted"]
    write_table(_t.assign(observed=_cis(_t, "beta_observed", "ci_lo", "ci_hi", 2))[
                    ["model", "testbed", "group", "n", "median_parallax", "observed", "beta_predicted", "inside_ci"]],
                "T21c_cue_model_predictions", "Cue model, out of sample: the slope each testbed shows (95% cluster-bootstrap "
                "interval) and the slope the model fitted on CONVOY-3D and E5 predicts from the testbed's own objects (x, rho_mono, "
                f"parallax, support), without refitting. Over all rows: Pearson r = {CUE_CALIBRATION['pearson_r']:.2f}, mean "
                f"absolute error {CUE_CALIBRATION['mean_abs_error']:.3f}.", nd=3)
if len(MONDUS) or len(REAL_PARALLAX):
    if len(MONDUS):
        write_table(MONDUS.assign(difference_ci=_cis(MONDUS, "difference", "ci_lo", "ci_hi", 3))[
                        ["contrast", "n", "n_scenes", "DUSt3R", "MonST3R", "difference_ci", "direction"]],
                    "T21d_monst3r_vs_dust3r", "What dynamic fine-tuning changed: MonST3R minus DUSt3R on the same movers, for the "
                    "parallax channel (slope of movers with more than 40 px of parallax; rise over the baseline sweep) and the "
                    "contact channel (the H6 contrast); 95% cluster-bootstrap intervals over scenes.", nd=3)
    if len(REAL_PARALLAX):
        _t = REAL_PARALLAX[REAL_PARALLAX["n"] > 0]
        write_table(_t.assign(beta_ci=_cis(_t, "beta", "ci_lo", "ci_hi", 2))[["model", "testbed", "bin", "n", "median_parallax",
                                                                              "beta_ci"]],
                    "T21e_convoy_real_by_parallax", "Correction of the v6.4 reading (D57): beta_s of CONVOY-REAL movers and of "
                    "CONVOY-3D floating crates (both unsupported) overall and within parallax bins (bins with >= 12 movers).", nd=2)
if len(BONN_PREV):
    write_table(BONN_PREV, "T22a_bonn_regimes", "Bonn RGB-D Dynamic (Section 6.12, exploratory): motion regimes of real moving "
                f"objects (parallel: median |eps| below the flow-noise floor, {BONN_TAU_PAR:.2f} px), the share without a static "
                "solution and the share on and off the floor, by kind of object.", nd=3)
if len(BONN_G):
    _t = BONN_G.assign(g_ci=_cis(BONN_G, "median_g", "ci_lo", "ci_hi", 3))
    write_table(_t[["model", "grouping", "group", "n", "n_pairs", "median_x_geo", "g_ci"]], "T22b_bonn_geometry_share",
                "Bonn RGB-D Dynamic: median per-object geometry share g of parallel, identifiable objects with |x - rho_mono| >= 0.3 "
                "(95% cluster-bootstrap intervals over image pairs), by kind and by measured floor contact; x_geo from optical flow "
                "and sensor depth, so no slope is fitted (D24)." + _gate_note("BONN"), nd=3)
if len(TWIN_RECALL):
    _t = TWIN_RECALL.copy()
    for c in ("det_epipolar", "det_twin", "det_either", "twin_given_epi_miss"):
        _t[c + "_ci"] = _cis(_t, c, c + "_lo", c + "_hi", 3)
    write_table(_t[["regime", "n", "n_tracks", "det_epipolar_ci", "det_twin_ci", "det_either_ci", "twin_below_road", "n_epi_miss",
                    "twin_given_epi_miss_ci"]],
                "T23a_twin_test", "The static twin test on Argoverse 2 (Section 6.13, exploratory): share of moving vehicles "
                f"flagged by the epipolar test (|eps| > {TWIN_THRESH.get('eps_px', np.nan):.2f} px), by the twin test (|twin height| "
                f"> {TWIN_THRESH.get('height_m', np.nan):.2f} m) and by either, both at 5% false positives on "
                f"{TWIN_THRESH.get('n_parked', 0)} parked vehicles, by motion regime (95% intervals over vehicle tracks; "
                "both tests use the same exact correspondences from labelled boxes); twin_below_road: flagged with the twin below the "
                "road; n_epi_miss: vehicles the epipolar test misses; twin_given_epi_miss: the share of those the twin test flags.", nd=3)
if len(TWIN_S):
    write_table(TWIN_S[["s_lo", "s_hi", "n", "frac_parallel", "det_epipolar", "det_twin", "det_either"]], "T23b_twin_test_by_s",
                "The static twin test by parallel fraction s (moving Argoverse 2 vehicles with a static solution).", nd=3)
if len(FTWIN_RECALL):
    _t = FTWIN_RECALL.copy()
    for c in ("det_epipolar", "det_twin", "det_either", "twin_given_epi_miss"):
        _t[c + "_ci"] = _cis(_t, c, c + "_lo", c + "_hi", 3)
    _h14 = H14_RESULT
    write_table(_t[["regime", "n", "n_tracks", "det_epipolar_ci", "det_twin_ci", "det_either_ci", "twin_below_road", "n_epi_miss",
                    "twin_given_epi_miss_ci"]],
                "T23c_twin_test_flow", f"The twin test with optical flow ({FTWIN_THRESH.get('flow', '')}; Section 6.13, D63): the same "
                "vehicles, with correspondences from forward-backward checked flow at their LiDAR pixels and the road plane fitted "
                "to the LiDAR returns (no labelled box). Epipolar test: |eps| > "
                f"{FTWIN_THRESH.get('eps_px', np.nan):.2f} px; twin test: the 10th percentile of the static reconstruction's "
                f"heights above the road outside [{FTWIN_THRESH.get('h_low_lo_m', np.nan):.2f}, "
                f"{FTWIN_THRESH.get('h_low_hi_m', np.nan):.2f}] m; both at 5% false positives on {FTWIN_THRESH.get('n_parked', 0)} "
                "parked vehicles. H14 (registered): in the parallel regime the twin test flags more vehicles than the epipolar test, "
                + (f"difference {_h14['estimate']:+.3f} [{_h14['ci_lo']:+.3f}, {_h14['ci_hi']:+.3f}] over {_h14['n']} vehicles "
                   f"({_h14['n_tracks']} tracks): {_h14['verdict']}." if np.isfinite(_h14.get("estimate", np.nan))
                   else f"{_h14.get('verdict', 'NOT TESTABLE')} ({_h14.get('n', 0)} vehicles, {_h14.get('n_tracks', 0)} tracks)."), nd=3)

In [ ]:
# Cell No: 089
# 12. Claim audit. Each claim the paper may make is listed with the evidence that decides it and a
#     status computed from this run's results (never written by hand). A claim whose evidence is not
#     part of this run is not assessable: it is left out of the table and named in one line below it,
#     so that nothing is claimed silently.
def _count(verdicts):
    v = list(verdicts)
    keys = ["SUPPORTED", "PARTIAL", "FALSIFIED", "NOT SUPPORTED", "CONTRADICTED", "NOT TESTABLE", "NOT INTERPRETABLE (gate)"]
    return ", ".join(f"{k}: {v.count(k)}" for k in keys if v.count(k)) or "none"

def _status(verdicts, strong="SUPPORTED"):
    v = [x for x in verdicts if x not in ("NOT TESTABLE", "NOT INTERPRETABLE (gate)")]
    if not v:
        return "NOT ASSESSABLE"
    if all(x == strong for x in v):
        return "ESTABLISHED (all interpretable models)"
    if any(x == strong for x in v):
        return "ESTABLISHED FOR A SUBSET"
    if any(x == "PARTIAL" for x in v):
        return "PARTIAL ONLY"
    return "NOT ESTABLISHED"

_e1i = E1_TABLE[E1_TABLE["gate"] == "PASS"]
_low = _e1i["ci_hi"] < PREREG["beta_support_range"][0]
_m2 = E2_TABLE[E2_TABLE["method_key"] == "M2_exact"]["verdict"]
_m1 = E2_TABLE[E2_TABLE["method_key"] == "M1_exact"]["verdict"]
_a1i = A1_TABLE[A1_TABLE["model"].isin(MODELS_OK["MAIN"]) & (A1_TABLE["A1"] != "NOT TESTABLE")] if len(A1_TABLE) else A1_TABLE
_ko = _a1i["A1"].str.startswith("HOLDS") if len(_a1i) else pd.Series(dtype=bool)
_e5 = lambda t: E5_TESTS[E5_TESTS["test"] == t]["verdict"] if len(E5_TESTS) else pd.Series(dtype=str)
_h9 = E6_TABLE[E6_TABLE["verdict"] != "control"]["verdict"] if len(E6_TABLE) else pd.Series(dtype=str)
_h11 = lambda t: RABL_TESTS[RABL_TESTS["test"] == t]["verdict"] if len(RABL_TESTS) else pd.Series(dtype=str)
_m2par = dict(zip(E2_TABLE.loc[E2_TABLE["method_key"] == "M2_exact", "model"], E2_TABLE.loc[E2_TABLE["method_key"] == "M2_exact", "rec_par"]))
_prev_vk = (np.average(PREVALENCE_VK["frac_parallel"], weights=PREVALENCE_VK["n_moving_pixels"]) if len(PREVALENCE_VK) else np.nan)
_prev_tum = (np.average(PREVALENCE["frac_parallel"], weights=PREVALENCE["n_moving_pixels"]) if len(PREVALENCE) else np.nan)
CLAIMS = [
    dict(id="C1", claim="The probe measures what it is meant to: planted prior weights are recovered through the slope-weight identity.",
         evidence="T2b, Fig. 2e", per_model=f"max |beta - beta(w)| = {np.nanmax(np.abs(ORACLE_TABLE['beta_s'] - ORACLE_TABLE['beta_theory_identity'])):.3f}",
         status="ESTABLISHED" if ORACLE_OK else "NOT ESTABLISHED"),
    dict(id="C2", claim="For purely parallel movers every interpretable model uses far less geometry than triangulation would "
                        "(beta_s interval below 0.8; locked P1 not supported).",
         evidence="T3, Fig. 3a", per_model=", ".join(f"{mlab(r.model)} {r.beta_s:.2f}" for r in _e1i.itertuples()) or "none",
         status=("NOT ASSESSABLE" if not len(_e1i) else "ESTABLISHED (all interpretable models)" if _low.all() else
                 "ESTABLISHED FOR A SUBSET" if _low.any() else "NOT ESTABLISHED")),
    dict(id="C3", claim="Whatever geometry is used comes from the cross-view correspondence (deleting the mover from view 2 lowers beta_s).",
         evidence="T7, Fig. 3b", per_model=f"holds for {int(_ko.sum())}/{len(_ko)}",
         status=("NOT ASSESSABLE" if not len(_ko) else "ESTABLISHED (all interpretable models)" if _ko.all() else
                 "ESTABLISHED FOR A SUBSET" if _ko.any() else "NOT ESTABLISHED")),
    dict(id="C4", claim="Removing an object's ground contact raises its geometry share (H5).", evidence="T9b, Fig. 4c",
         per_model=_count(_e5("H5")), status=_status(_e5("H5"))),
    dict(id="C5", claim="Hiding the ground contact raises the geometry share, and the effect acts through the contact (H6, H7).",
         evidence="T9b, Fig. 4c", per_model=f"H6 {_count(_e5('H6'))}; H7 {_count(_e5('H7'))}",
         status=_status(list(_e5("H6")) + list(_e5("H7")))),
    dict(id="C6", claim="Single-image depth places a lifted object farther away (H8): the prior reads the contact point.",
         evidence="T9b, Fig. 4d", per_model=_count(_e5("H8")), status=_status(_e5("H8"))),
    dict(id="C7", claim="In photo-realistic driving the geometry share for moving vehicles is below 0.3 (H9, exploratory: its "
                        "sample rule was changed after the v6.0 count, D34).",
         evidence="T10a, T10d", per_model=_count(_h9),
         status=("EXPLORATORY: " + _status(_h9).lower()) if len(_h9) and _status(_h9) != "NOT ASSESSABLE" else
                "NOT ASSESSABLE (Virtual KITTI 2 not in this run, or too few vehicles)"),
    dict(id="C8", claim="Motion that two views cannot see is common in driving (share of moving vehicle pixels, or of real moving "
                        "vehicles, within the parallel threshold of their epipolar line).",
         evidence="T10b, T10c, T16a, Fig. 7b",
         per_model="; ".join(x for x in [
             f"vehicle pixels {100 * _prev_vk:.0f}% (Virtual KITTI 2)" if np.isfinite(_prev_vk) else "",
             "; ".join(f"real vehicles {100 * r.frac_parallel:.0f}% ({r.dataset.split(' (')[0]})" for r in REAL_PREVALENCE.itertuples())
             if len(REAL_PREVALENCE) else "",
             f"people {100 * _prev_tum:.0f}% of pixels (TUM walking)" if np.isfinite(_prev_tum) else ""] if x) or "none",
         status="DESCRIPTIVE (no prediction)" if np.isfinite(_prev_vk) or np.isfinite(_prev_tum) or len(REAL_PREVALENCE)
         else "NOT ASSESSABLE"),
    dict(id="C9", claim="Geometric masks miss parallel movers far more often than perpendicular ones (P2: M1, M2).",
         evidence="T4, Fig. 6a", per_model=f"M1 {_count(_m1)}; M2 {_count(_m2)}", status=_status(list(_m1) + list(_m2))),
    dict(id="C10a", claim="On real images, lifting the same object off its support raises its geometry share (H11a).",
         evidence="T11a, T11b, Fig. 5b-c", per_model=_count(_h11("H11a")),
         status=_status(_h11("H11a")) if len(_h11("H11a")) else "NOT ASSESSABLE (no real-image contact ablation)"),
    dict(id="C10b", claim="On real images, single-image depth places the lifted object farther away (H11b).",
         evidence="T11b, Fig. 5d", per_model=_count(_h11("H11b")),
         status=_status(_h11("H11b")) if len(_h11("H11b")) else "NOT ASSESSABLE (no real-image contact ablation)"),
    dict(id="C11", claim="Multi-view models keep the degeneracy over 2-8 frames on straight paths and lose it on curved twins (P4).",
         evidence="T6, Fig. 8c", per_model=_count(E4_SUMMARY["verdict"]) if len(E4_SUMMARY) else "none",
         status=_status(E4_SUMMARY["verdict"]) if len(E4_SUMMARY) else "NOT ASSESSABLE"),
    dict(id="C12", claim="Remedy (exploratory): a model's own single-image depth exposes parallel movers that geometric tests miss (M5).",
         evidence="T4 (M5), Fig. S3",
         per_model=", ".join(f"{mlab(r.model)} {r.rec_par:.2f} vs M2 {_m2par.get(r.model, np.nan):.2f}"
                             for r in E2_TABLE[(E2_TABLE["method_key"] == "M5") & E2_TABLE["model"].isin(MODELS_OK["MAIN"])]
                             .itertuples()) or "none",
         status=("EXPLORATORY (no prediction); recall on parallel movers, M5 vs M2"
                 if ((E2_TABLE["method_key"] == "M5") & E2_TABLE["model"].isin(MODELS_OK["MAIN"])).any() else "NOT ASSESSABLE")),
    dict(id="C13", claim="Motion inside the epipolar plane (not only along b) is invisible to two views; the locked A2 fails.",
         evidence="Section 2.2-2.3, Fig. S1",
         per_model=f"in-plane max eps {PROP_CHECKS['P3_max_eps_inplane_perpendicular_px']:.1e} px; "
                   f"locked/exact median {A2_SUMMARY['locked_over_exact_p05_p50_p95'][1]:.2f}",
         status="ESTABLISHED (mathematical, verified numerically)"),
    dict(id="C14", claim="The previous 2-D testbed (LEGACY-FLAT) is rejected by the validity gate; CONVOY-3D is accepted.",
         evidence="T2, Fig. 2d",
         per_model=f"FLAT rejected for {int((GATE[GATE['set'] == 'FLAT']['gate'] == 'FAIL').sum())}/{int((GATE['set'] == 'FLAT').sum())}, "
                   f"CONVOY-3D accepted for {int((GATE[GATE['set'] == 'MAIN']['gate'] == 'PASS').sum())}/{int((GATE['set'] == 'MAIN').sum())}",
         status="ESTABLISHED" if (GATE[GATE["set"] == "FLAT"]["gate"] == "FAIL").all() and (GATE[GATE["set"] == "MAIN"]["gate"] == "PASS").all()
         else "PARTIAL"),
]
_m2drop = E5_MASK_TESTS[E5_MASK_TESTS["contrast"] == "CH - G"]["direction"] if len(E5_MASK_TESTS) else pd.Series(dtype=str)
_cnt = lambda v: ", ".join(f"{k}: {int(n)}" for k, n in pd.Series(list(v)).value_counts().items()) or "none"
_dsm = {"Virtual KITTI 2": "VK", "Argoverse 2": "AV2"}
def _by_ds(df):
    # directions of a driving contrast per dataset, for the models interpretable on that dataset
    out = {}
    for ds, g in (df.groupby("dataset", sort=False) if len(df) else []):
        out[ds] = g.loc[g["model"].isin(MODELS_OK.get(_dsm.get(ds, ""), [])), "direction"]
    return out
_xc_ds, _xm_ds = _by_ds(X6_CONTACT_TEST), _by_ds(X6_MATCHED)
def _c16_status():
    parts = []
    for ds, v in _xc_ds.items():
        t = v[v != "NOT TESTABLE"]
        if not len(t):
            continue
        m = _xm_ds.get(ds, pd.Series(dtype=str)); tm = m[m != "NOT TESTABLE"]
        parts.append(f"{ds}: higher for {int((t == 'higher').sum())}/{len(t)}"
                     + (f", after matching {int((tm == 'higher').sum())}/{len(tm)}" if len(tm) else ", matched comparison not testable"))
    return ("EXPLORATORY: " + "; ".join(parts)) if parts else "NOT ASSESSABLE (too few vehicles with a hidden or a visible contact)"
CLAIMS += [
    dict(id="C15", claim="Exploratory: M2 catches fewer of the same parallel movers once their contact is hidden (masks follow the "
                         "support cue).", evidence="T14, Fig. 6b", per_model=_cnt(_m2drop),
         status=("EXPLORATORY: lower recall for " + f"{int((_m2drop == 'lower').sum())}/{len(_m2drop)} models") if len(_m2drop)
         else "NOT ASSESSABLE"),
    dict(id="C16", claim="Exploratory: in traffic, vehicles whose ground contact is hidden get a larger geometry share (Virtual "
                         "KITTI 2; real Argoverse 2 vehicles from v6.4).",
         evidence="T15a, T15c, T15d, Fig. 10b",
         per_model=" | ".join(f"{ds}: {_cnt(v)}; matched on depth, size and lever: {_cnt(_xm_ds.get(ds, []))}" for ds, v in _xc_ds.items())
         or "none", status=_c16_status()),
    dict(id="C17", claim="Exploratory: vehicles keeping pace with the camera are placed by the prior, not at the geometric answer "
                         "(the convoy case in driving data).", evidence="T15b, Fig. 7c",
         per_model="; ".join(f"{r.dataset}: {mlab(r.model)} g = {r.median_g:.2f}" for r in X6_PACE.itertuples()
                             if np.isfinite(r.median_g)) if len(X6_PACE) else "none",
         status="EXPLORATORY (no prediction)" if len(X6_PACE) and np.isfinite(X6_PACE["median_g"]).any() else "NOT ASSESSABLE"),
    dict(id="C18", claim="Exploratory: on real vehicles with LiDAR depth (Argoverse 2; KITTI tracking if attached) the geometry "
                         "share is small.", evidence="T16b",
         per_model="; ".join(f"{r.dataset}: {mlab(r.model)} {r.beta_geo:.2f}" for r in REAL_BETA.itertuples() if np.isfinite(r.beta_geo))
         if len(REAL_BETA) else "none",
         status="EXPLORATORY (no prediction)" if len(_finite(REAL_BETA, "beta_geo")) else
         "NOT ASSESSABLE (no real-driving data in this run, or too few parallel vehicles)"),
    dict(id="C19", claim="Descriptive: the share of real moving vehicles (and of their LiDAR pixels) in the parallel regime, where "
                         "two views carry no motion signal.", evidence="T16a",
         per_model="; ".join(f"{r.dataset}: {100 * r.frac_parallel:.0f}% of vehicles, {100 * r.frac_parallel_lidar_px:.0f}% of their "
                             f"LiDAR pixels" for r in REAL_PREVALENCE.itertuples()) if len(REAL_PREVALENCE) else "none",
         status="DESCRIPTIVE (no prediction)" if len(REAL_PREVALENCE) else "NOT ASSESSABLE (no real-driving data in this run)"),
]
# ---- v6.4: the synthesis by ground support (descriptive; grouping chosen after the v6.3 results, AMENDMENT_V6_4) ----
_sc = SUPPORT_CONTRASTS[SUPPORT_CONTRASTS["direction"] != "NOT TESTABLE"] if len(SUPPORT_CONTRASTS) else pd.DataFrame(columns=["direction"])
_e5i = SUPPORT_LEVELS[(SUPPORT_LEVELS["testbed"] == "E5 ablation") & SUPPORT_LEVELS["condition"].isin(
    [ABLATION_DESIGN["labels"]["G"], ABLATION_DESIGN["labels"]["CH"]])] if len(SUPPORT_LEVELS) else pd.DataFrame()
_e5f = {}                                       # model -> (factor grounded, factor contact hidden)
for m_ in [m for m in EVALUATED_MODELS if len(_e5i) and (_e5i["model"] == m).any()]:
    g_ = _e5i[_e5i["model"] == m_].set_index("condition")["implied_factor"]
    _e5f[m_] = (float(g_.get(ABLATION_DESIGN["labels"]["G"], np.nan)), float(g_.get(ABLATION_DESIGN["labels"]["CH"], np.nan)))
def _factor_text():
    return ", ".join(f"{mlab(m_)} {a:.2f} -> {b:.2f}" for m_, (a, b) in _e5f.items()) or "none"
_e5f_ok = [(a, b) for a, b in _e5f.values() if np.isfinite(a) and np.isfinite(b)]
def _c21_rows():
    out = []
    for m_ in MODELS_OK["MAIN"]:
        if m_ not in MODELS_OK["REAL"] or not len(SUPPORT_LEVELS):
            continue
        r_ = REAL_E1[REAL_E1["model"] == m_]["beta_s"]
        f_ = SUPPORT_LEVELS[(SUPPORT_LEVELS["model"] == m_) & (SUPPORT_LEVELS["testbed"] == "CONVOY-3D")
                            & (SUPPORT_LEVELS["condition"] == "floating crates")]["estimate"]
        p_ = E1_TABLE[E1_TABLE["model"] == m_]["beta_s"]
        if len(r_) and len(f_) and len(p_) and np.isfinite([r_.iloc[0], f_.iloc[0], p_.iloc[0]]).all():
            out.append((m_, float(r_.iloc[0]), float(f_.iloc[0]), float(p_.iloc[0])))
    return out
_c21 = _c21_rows()
CLAIMS += [
    dict(id="C20", claim="Descriptive: within each testbed, objects whose ground contact is in view get a smaller geometry share "
                         "than floating or hidden-contact objects.",
         evidence="T18a, T18b, Fig. 10a-b",
         per_model="; ".join(f"{r.testbed}, {r.contrast}: higher {r.higher}, no clear change {r.no_clear_change}, lower {r.lower}"
                             for r in SUPPORT_COUNT.itertuples()) if len(SUPPORT_COUNT) else "none",
         status=(f"DESCRIPTIVE (grouping chosen after the v6.3 results): higher in {int((_sc['direction'] == 'higher').sum())} of "
                 f"{len(_sc)} testable model x contrast cells, lower in {int((_sc['direction'] == 'lower').sum())}")
         if len(_sc) else "NOT ASSESSABLE"),
    dict(id="C21", claim="Descriptive: CONVOY-REAL, whose movers are all floating crates (D52), is closer to the floating crates of "
                         "CONVOY-3D than to the pooled CONVOY-3D slope; the two also differ in parallax (D57, C27).",
         evidence="T3, T8, T18a, T21e",
         per_model=", ".join(f"{mlab(m_)} {r_:.2f} vs floating {f_:.2f} / pooled {p_:.2f}" for m_, r_, f_, p_ in _c21) or "none",
         status=(f"DESCRIPTIVE: closer to the floating crates for {sum(abs(r_ - f_) < abs(r_ - p_) for _, r_, f_, p_ in _c21)}/"
                 f"{len(_c21)} models") if _c21 else "NOT ASSESSABLE"),
    dict(id="C22", claim="Derived: for an object keeping pace with the camera (s = 0.85), the E5 slopes imply a larger depth "
                         "overestimate when its ground contact is hidden than when it is in view (exp(beta x); 1 = true depth, "
                         "6.7 = the geometric answer).",
         evidence="T18c, Fig. 10c", per_model="E5 grounded -> contact hidden: " + _factor_text(),
         status=(f"DERIVED from the slopes (not measured separately): larger with the contact hidden for "
                 f"{sum(b > a for a, b in _e5f_ok)}/{len(_e5f_ok)} models; the registered test of the slopes is H6")
         if _e5f_ok else "NOT ASSESSABLE"),
]
# ---- v6.5: designed experiments (registered), the static twin, the cue model and Bonn (exploratory) ----
_h12v = BSW_TESTS[(BSW_TESTS["test"] == "H12") & BSW_TESTS["model"].isin(PREREG["h12_models"])]["verdict"] if len(BSW_TESTS) else pd.Series(dtype=str)
_h12o = BSW_TESTS[(BSW_TESTS["test"] == "H12") & ~BSW_TESTS["model"].isin(PREREG["h12_models"])] if len(BSW_TESTS) else pd.DataFrame()
_h13v = WDR_TESTS[WDR_TESTS["test"] == "H13"]["verdict"] if len(WDR_TESTS) else pd.Series(dtype=str)
_share = WDR_TESTS[(WDR_TESTS["test"] == "dose share") & WDR_TESTS["model"].isin(MODELS_OK["WDR"])] if len(WDR_TESTS) else pd.DataFrame()
_cvd = CUE_CV[CUE_CV["comparison"].str.startswith("contact only")]["direction"] if len(CUE_CV) else pd.Series(dtype=str)
_cvc = CUE_CV[CUE_CV["comparison"].str.startswith("parallax only")]["direction"] if len(CUE_CV) else pd.Series(dtype=str)
_rp = CUE_PRED[(CUE_PRED["set"] == "REAL") & (CUE_PRED["role"] == "predicted")] if len(CUE_PRED) else pd.DataFrame()
_tw_par = TWIN_RECALL[TWIN_RECALL["regime"] == "parallel"] if len(TWIN_RECALL) else pd.DataFrame()
_bonn_d = BONN_TESTS[BONN_TESTS["direction"] != "NOT TESTABLE"]["direction"] if len(BONN_TESTS) else pd.Series(dtype=str)
# C28: both parts must be estimable; the rise must be lower for MonST3R and the H6 contrast not lower
_md = {r.contrast: r.direction for r in MONDUS.itertuples()} if len(MONDUS) else {}
_md_rise, _md_h6 = _md.get("rise of beta_s from x0.5 to x3, baseline sweep"), _md.get("H6 contrast beta(CH) - beta(G), E5 movers")
if _md_rise in (None, "NOT TESTABLE") or _md_h6 in (None, "NOT TESTABLE"):
    _md_status = "NOT ASSESSABLE (the sweep or the E5 contrast is not estimable for both models)"
else:
    _md_status = ("EXPLORATORY: consistent (paired, same movers)" if _md_rise == "lower" and _md_h6 != "lower" else
                  f"EXPLORATORY: not consistent (rise {_md_rise}, contact effect {_md_h6})")
CLAIMS += [
    dict(id="C23", claim="More parallax alone (a longer baseline, everything else fixed) raises the geometry share of DUSt3R and "
                         "MASt3R (H12).", evidence="T19a, T19b, Fig. 13b",
         per_model=_count(_h12v) + ("; no prediction for " + ", ".join(f"{mlab(r.model)} {r.estimate:+.2f}" for r in _h12o.itertuples()
                                                                        if np.isfinite(r.estimate)) if len(_h12o) else ""),
         status=_status(_h12v) if len(_h12v) else "NOT ASSESSABLE"),
    dict(id="C24", claim="Hiding the ground contact of a grounded object with the lowest wall (built to hide at least 15 cm) "
                         "raises its geometry share (H13).",
         evidence="T20a, T20b, Fig. 13d", per_model=_count(_h13v), status=_status(_h13v) if len(_h13v) else "NOT ASSESSABLE"),
    dict(id="C25", claim="Exploratory: most of the effect of a 70 cm wall is already reached by the 15 cm wall (the contact junction, "
                         "not the hidden extent, carries the cue).", evidence="T20b, Fig. 13d",
         per_model=", ".join(f"{mlab(r.model)} {r.estimate:.2f}" for r in _share.itertuples() if np.isfinite(r.estimate)) or "none",
         status=("EXPLORATORY: share of the 70 cm effect reached at 15 cm above 0.5 for "
                 f"{int((_share['estimate'] > 0.5).sum())}/{int(np.isfinite(_share['estimate']).sum())} models")
         if len(_share) and np.isfinite(_share["estimate"]).any() else "NOT ASSESSABLE"),
    dict(id="C26", claim="Exploratory: one cue-combination model with a contact channel and a parallax channel, fitted on CONVOY-3D "
                         "and E5, predicts the slopes of the other testbeds without refitting.",
         evidence="T21a-c, Fig. 12a-c",
         per_model=(f"r = {CUE_CALIBRATION['pearson_r']:.2f}, mean |error| {CUE_CALIBRATION['mean_abs_error']:.3f}, "
                    f"{100 * CUE_CALIBRATION['share_inside_ci']:.0f}% of predictions inside the observed interval; parallax channel "
                    f"better in CV for {int((_cvd == 'two channels predict better').sum())}/{len(_cvd)}, contact channel for "
                    f"{int((_cvc == 'two channels predict better').sum())}/{len(_cvc)}") if CUE_CALIBRATION["n"] else "none",
         status=f"EXPLORATORY ({CUE_CALIBRATION['n']} out-of-sample slopes)" if CUE_CALIBRATION["n"] else "NOT ASSESSABLE"),
    dict(id="C27", claim="Correction (D57): the higher CONVOY-REAL slope is not evidence of a real-image effect; CONVOY-REAL movers "
                         "have far more parallax than the CONVOY-3D floating crates, and the cue model's prediction for that "
                         "parallax is reported next to the observed slope.", evidence="T21c, T21e",
         per_model=", ".join(f"{mlab(r.model)} observed {r.beta_observed:.2f} vs predicted {r.beta_predicted:.2f}" for r in _rp.itertuples())
         or "none",
         status=("DESCRIPTIVE: prediction inside the observed interval for "
                 f"{int(_rp['inside_ci'].sum())}/{len(_rp)} models") if len(_rp) else "NOT ASSESSABLE"),
    dict(id="C28", claim="Exploratory: on the same movers, MonST3R (DUSt3R fine-tuned on dynamic scenes) rises less with parallax "
                         "than DUSt3R in the baseline sweep, and its contact effect (H6 contrast) is not smaller.", evidence="T21d",
         per_model="; ".join(f"{r.contrast}: {r.difference:+.2f} ({r.direction})" for r in MONDUS.itertuples()) if len(MONDUS) else "none",
         status=_md_status),
    dict(id="C29", claim="The static twin of a grounded mover with 0 < s < 1 lies H s/(1-s) below the road, e^x times larger "
                         "(Proposition 4).", evidence="Section 1, Section 2.2, Fig. 11a-b",
         per_model=f"max height error {PROP_CHECKS.get('P4_max_twin_height_error_m', np.nan):.1e} m, below the road in "
                   f"{100 * PROP_CHECKS.get('P4_fraction_twin_below_road_for_0_lt_s_lt_1', np.nan):.0f}% of trials",
         status="ESTABLISHED (mathematical, verified numerically)"),
    dict(id="C30", claim="Exploratory: on real vehicles, a twin test (the static explanation would sink into the road or float above "
                         "it) flags moving vehicles that the epipolar test misses.", evidence="T23a, T23b, Fig. 11c-d",
         per_model=(f"parallel regime: epipolar {100 * _tw_par['det_epipolar'].iloc[0]:.0f}%, twin {100 * _tw_par['det_twin'].iloc[0]:.0f}%, "
                    f"either {100 * _tw_par['det_either'].iloc[0]:.0f}% of {int(_tw_par['n'].iloc[0])} vehicles; the twin test flags "
                    + (f"{100 * _tw_par['twin_given_epi_miss'].iloc[0]:.0f}% of the {int(_tw_par['n_epi_miss'].iloc[0])} the epipolar "
                       "test misses" if _tw_par["n_epi_miss"].iloc[0] else "nothing extra (the epipolar test misses none)"))
         if len(_tw_par) and _tw_par["n"].iloc[0] else "none",
         status="EXPLORATORY (both tests on the same exact correspondences from labelled boxes, at 5% false positives on parked "
                "vehicles)" if len(_tw_par) and _tw_par["n"].iloc[0] else "NOT ASSESSABLE"),
    dict(id="C31", claim="Exploratory: real floating objects (Bonn balloons) get a larger geometry share than objects on the floor.",
         evidence="T22a, T22b, Fig. 14", per_model=_cnt(_bonn_d) if len(_bonn_d) else "none",
         status=(f"EXPLORATORY: higher for {int((_bonn_d == 'higher').sum())}/{len(_bonn_d)} models (v6.6 ground truth: relative "
                 "poses re-estimated, D62)") if len(_bonn_d)
         else ("NOT ASSESSABLE (the Bonn ground truth failed its quality gate: the median pair flags "
               f"{100 * BONN_QC.get('median_flag_frac', np.nan):.0f}% of its static scene as moving)") if BONN_STATUS.get("suspect")
         else "NOT ASSESSABLE (Bonn not in this run, or too few parallel objects on and off the floor)"),
    dict(id="C32", claim="Exploratory: wherever a condition raises the geometry share, M2 recall on the same movers falls (geometric "
                         "masks defeat themselves).", evidence="Fig. 12d",
         per_model=(f"Spearman {TRADEOFF_SUMMARY['spearman']:.2f} over {TRADEOFF_SUMMARY['n']} model x condition changes; opposite "
                    f"signs in {100 * TRADEOFF_SUMMARY['share_opposite_sign']:.0f}%") if TRADEOFF_SUMMARY["n"] else "none",
         status="EXPLORATORY (no prediction)" if TRADEOFF_SUMMARY["n"] >= 5 else "NOT ASSESSABLE"),
]
# ---- v6.6: the contact gates the parallax channel (exploratory) and the twin test with optical flow (registered H14) ----
_h12i = (BSW_TESTS[(BSW_TESTS["test"] == "H12-interaction") & np.isfinite(BSW_TESTS["ci_lo"])] if len(BSW_TESTS)
         else pd.DataFrame())
_ftp = FTWIN_RECALL[FTWIN_RECALL["regime"] == "parallel"] if len(FTWIN_RECALL) else pd.DataFrame()
CLAIMS += [
    dict(id="C33", claim="Exploratory: the contact gates the parallax channel; a longer baseline raises the geometry share of "
                         "floating movers more than that of grounded movers (same scenes, baseline sweep).",
         evidence="T19a, T19b, Fig. 13b",
         per_model="; ".join(f"{mlab(r.model)} {r.estimate:+.2f} [{r.ci_lo:+.2f}, {r.ci_hi:+.2f}]" for r in _h12i.itertuples()) or "none",
         status=(f"EXPLORATORY (chosen after the v6.5 results): interval above 0 for {int((_h12i['ci_lo'] > 0).sum())}/{len(_h12i)} "
                 f"models, below 0 for {int((_h12i['ci_hi'] < 0).sum())}") if len(_h12i) else "NOT ASSESSABLE"),
    dict(id="C34", claim="With optical flow instead of labelled correspondences and a LiDAR road plane instead of the box, the twin "
                         "test still flags more parallel-regime moving vehicles than the epipolar test (H14).",
         evidence="T23c, Fig. 11d",
         per_model=(f"parallel regime: epipolar {100 * _ftp['det_epipolar'].iloc[0]:.0f}%, twin {100 * _ftp['det_twin'].iloc[0]:.0f}%, "
                    f"either {100 * _ftp['det_either'].iloc[0]:.0f}% of {int(_ftp['n'].iloc[0])} vehicles; difference "
                    + (f"{H14_RESULT['estimate']:+.3f} [{H14_RESULT['ci_lo']:+.3f}, {H14_RESULT['ci_hi']:+.3f}]"
                       if np.isfinite(H14_RESULT.get("estimate", np.nan)) else "not estimable"))
         if len(_ftp) and _ftp["n"].iloc[0] else "none",
         status=(f"{H14_RESULT['verdict']} (registered in AMENDMENT_V6_6; {FTWIN_THRESH.get('flow', '')})" if H14_RESULT
                 else "NOT ASSESSABLE")),
]
if PROTOCOL_DEPENDENT:
    for c in CLAIMS:
        if c["id"] in ("C6", "C10b"):
            c["per_model"] += " | single-image results protocol-dependent for " + ", ".join(mlab(m) for m in sorted(PROTOCOL_DEPENDENT))
_all_claims = pd.DataFrame(CLAIMS)
_na = _all_claims["status"].str.startswith("NOT ASSESSABLE")
CLAIM_AUDIT = _all_claims[~_na].reset_index(drop=True)
with pd.option_context("display.max_colwidth", 95, "display.width", 250):
    print(CLAIM_AUDIT.to_string(index=False, na_rep="–"))
if _na.any():
    print("\nNot assessable in this run (their evidence is not part of it): "
          + "; ".join(f"{r.id} {r.status.replace('NOT ASSESSABLE', '').strip(' ()') or 'no estimable result'}"
                      for r in _all_claims[_na].itertuples()))
CLAIM_AUDIT.to_csv(os.path.join(RESULTS_DIR, "claim_audit.csv"), index=False)
print("\nScope: findings hold for the models in T1a as evaluated here; CONVOY-3D (mover depth "
      f"{MOVER_DESIGN['depth_range_m'][0]:g}-{MOVER_DESIGN['depth_range_m'][1]:g} m, s in "
      f"{MOVER_DESIGN['s_range_main']} plus an exploratory s in {MOVER_DESIGN['s_range_beyond']}); "
      + (f"CONVOY-REAL (TUM fr3 static backgrounds, mover depth {REAL_PLACEMENT['z_min']:g}-{REAL_PLACEMENT['z_max']:g} m); "
         if len(META_REAL) else "")
      + (f"E5-REAL (the same TUM backgrounds, crate depth {RABL_DESIGN['depth_m'][0]:g}-{RABL_DESIGN['depth_m'][1]:g} m); "
         if len(META_RABL) else "") + ("Virtual KITTI 2 (" + ", ".join(VK_SCENE_OBJS) + ", clone, Camera_0); " if VK_SCENE_OBJS else "")
      + (f"Argoverse 2 ({len(AV2_STATUS['logs'])} validation logs, front-centre camera; exploratory only); " if AV2_STATUS["available"] else "")
      + (f"KITTI tracking ({', '.join(KT_STATUS['sequences'])}; exploratory only); " if KT_STATUS["available"] else "")
      + ("TUM fr3 walking for prevalence only; " if len(PREVALENCE) else "")
      + (f"Bonn RGB-D Dynamic ({', '.join(BONN_STATUS['sequences'])}; exploratory only); " if BONN_STATUS["available"] else "")
      + f"designed sets: baseline sweep x0.5-x3, walls 15-70 cm. No state-of-the-art claim is made.")

In [ ]:
# Cell No: 090
# 13. Reproducibility record: manifests, environment, pinned code, model and data status, stage log,
#     timings, figure and table inventories, file checksums, and two archives: convoy_checkpoints.zip
#     (every finished stage; attached as a Kaggle dataset it restores them exactly in a later session)
#     and convoy_paper_outputs.zip (figures, tables, results and manifests for checking the paper).
import zipfile

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

ENV_LOG["end_time_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
ENV_LOG["total_runtime_min"] = round((time.time() - RUN_T0) / 60, 1)
STAGE_LOG_TABLE = pd.DataFrame(STAGE_LOG)
STAGE_LOG_TABLE.to_csv(os.path.join(MANIFEST_DIR, "stage_log.csv"), index=False)
manifest = {
    "idea_lock.json": IDEA_LOCK, "amendment_v6.json": AMENDMENT_V6, "amendment_v6_1.json": AMENDMENT_V6_1,
    "amendment_v6_2.json": AMENDMENT_V6_2, "amendment_v6_3.json": AMENDMENT_V6_3, "amendment_v6_4.json": AMENDMENT_V6_4,
    "amendment_v6_5.json": AMENDMENT_V6_5, "amendment_v6_6.json": AMENDMENT_V6_6,
    "decisions.json": DECISIONS, "env_log.json": ENV_LOG,
    "repos.json": REPO_STATUS, "models.json": MODEL_STATUS, "datasets.json": DATASET_STATUS,
    "a3_validation.json": A3_VALIDATION, "vkitti.json": dict(status=VK_STATUS, a3=VK_A3, selection=VK_SELECTION),
    "argoverse2.json": dict(status=AV2_STATUS, a3=AV2_A3, vehicle_check=AV2_VEHICLE_CHECK, extension=dict(
                                status=AV2X_STATUS, a3=AV2X_A3, vehicle_check=AV2X_VEHICLE_CHECK), contact=dict(
                                status=AV2C_STATUS, design=AV2X_DESIGN,
                                labels=(AV2_CONTACT["contact"].value_counts().to_dict() if len(AV2_CONTACT) else {}),
                                check_on_vkitti=(CONTACT_RULE_CHECK.to_dict("records") if len(CONTACT_RULE_CHECK) else [])),
                            ego_speed_mps=AV2_EGO_SPEED,
                            pairs=[{k: v for k, v in m.items() if k not in ("vehicles", "dir")} for m in AV2_PAIRS_ALL]),
    "kitti.json": dict(status={k: v for k, v in KT_STATUS.items() if k != "root"}, a3=KT_A3),
    "designed_experiments.json": dict(checks=DESIGN_CHECKS, sweep=_jsonable(SWEEP_DESIGN), dose=_jsonable(WALL_DOSE_DESIGN),
                                      stages={k: v.fp for k, v in DESIGN_STAGES.items()}),
    "bonn.json": dict(status=BONN_STATUS, a3=BONN_A3, parallel_threshold_px=BONN_TAU_PAR, quality=BONN_QC,
                      design={k: v for k, v in BONN_DESIGN.items() if k != "base_url"},
                      pairs=[{k: v for k, v in m_.items() if k not in ("objects", "dir", "static_patch_eps")} for m_ in BONN_PAIR_META]),
    "static_twin.json": dict(status=AV2T_STATUS, thresholds=TWIN_THRESH, design=TWIN_DESIGN),
    "static_twin_flow.json": dict(status=AV2F_STATUS, thresholds=FTWIN_THRESH, H14=H14_RESULT, design=FLOWTWIN_DESIGN),
    "cue_model.json": dict(fits=CUE_FITS, calibration=CUE_CALIBRATION, mask_tradeoff=TRADEOFF_SUMMARY, design=CUE_DESIGN),
    "stage_times_s.json": {k: round(v, 1) for k, v in STAGE_TIMES.items()}, "inference_times_s.json": RUN_TIMING,
    "flow.json": {k: v for k, v in FLOW.items() if k not in ("model", "transforms")},
    "testbed_checks.json": TESTBED_CHECKS, "proposition_checks.json": PROP_CHECKS,
    "figures.json": FIGURE_MANIFEST, "tables.json": TABLE_MANIFEST,
}
for name, obj in manifest.items():
    with open(os.path.join(MANIFEST_DIR, name), "w") as f:
        json.dump(obj, f, indent=2, default=str)

# figure captions for the paper (v6.3): bold lead-ins, LaTeX figure environments that include figures/<file>.pdf
with open(os.path.join(FIGURE_DIR, "captions.tex"), "w") as f:
    f.write("% Figure environments generated by the notebook; the PDFs are in figures/. Double-column figures use figure*.\n")
    for fm in FIGURE_MANIFEST:
        env = "figure*" if fm["placement"] == "double column" else "figure"
        f.write(f"\n% {fm['figure']}: {fm['question']}\n\\begin{{{env}}}[t]\n  \\centering\n"
                f"  \\includegraphics[width=\\linewidth]{{figures/{fm['figure']}.pdf}}\n  \\caption{{{fm['caption_tex']}}}\n"
                f"  \\label{{fig:{fm['figure']}}}\n\\end{{{env}}}\n")
with open(os.path.join(FIGURE_DIR, "captions.md"), "w") as f:
    for fm in FIGURE_MANIFEST:
        f.write(f"### {fm['figure']} ({fm['placement']})\n\n**{fm['title']}.** {fm['caption']}\n\n")

release_memory(); _ = mem_report("end of run")
pd.DataFrame(MEMORY_LOG).to_csv(os.path.join(MANIFEST_DIR, "memory_log.csv"), index=False)
inventory = []
for root, _, files in os.walk(OUTPUT_DIR):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        inventory.append(dict(path=os.path.relpath(p, OUTPUT_DIR), bytes=os.path.getsize(p), sha256=_sha256(p)))
pd.DataFrame(inventory).to_csv(os.path.join(MANIFEST_DIR, "file_inventory.csv"), index=False)

# Two archives (v6.2):
#   convoy_checkpoints.zip    every stage archive (stored as is), the resume point for a later session.
#                             Stages were appended as they finished (Section 0.5); missing ones are
#                             added here and the archive is rebuilt only if it holds stale copies.
#   convoy_paper_outputs.zip  figures, tables, results, manifests and logs (no per-model caches, which
#                             live in the checkpoints), i.e. what is needed to check the paper.
_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*.zip")))
_readme = os.path.join(SCRATCH_DIR, "RESUME_README.txt")
with open(_readme, "w") as f:
    f.write("convoy_checkpoints.zip: one archive per finished stage (checkpoints/<stage>.zip).\n"
            "To resume or extend: upload this zip as a Kaggle dataset, add it as an input and run all cells.\n"
            "Every stage whose fingerprint matches is restored instead of recomputed; only missing stages run.\n")
for p in _ckpts:
    archive_stage(p)
_names = _archive_names()
if _ARCHIVE["stale"] or any(not n.startswith("checkpoints/") and n != "RESUME_README.txt" for n in _names):
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH + ".part", "w", zipfile.ZIP_STORED, allowZip64=True) as zf:
        for p in _ckpts:
            zf.write(p, "checkpoints/" + os.path.basename(p))
    os.replace(ZIP_PATH + ".part", ZIP_PATH)
    _ARCHIVE.update(names=None, stale=set())
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "a", allowZip64=True) as zf:
        if "RESUME_README.txt" not in zf.namelist():
            zf.write(_readme, "RESUME_README.txt")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        _bad = zf.testzip()
        _n_ck = sum(1 for n in zf.namelist() if n.startswith("checkpoints/"))
    assert _bad is None, f"archive check failed at {_bad}"
    assert _n_ck == len(_ckpts), f"archive holds {_n_ck} of {len(_ckpts)} stage checkpoints"
    _zip_note = f"{os.path.basename(ZIP_PATH)} ({os.path.getsize(ZIP_PATH) / 2**30:.2f} GiB): {len(_ckpts)} stage checkpoints, verified"
else:
    _zip_note = (f"{os.path.basename(ZIP_PATH)} not written ({_ARCHIVE['note'] or 'archive disabled'}); "
                 f"checkpoints/ in {ROOT_WORK} holds the same files")
_n_paper = build_paper_zip()
with zipfile.ZipFile(PAPER_ZIP) as zf:
    assert zf.testzip() is None, "paper archive check failed"
_paper_note = f"{os.path.basename(PAPER_ZIP)} ({os.path.getsize(PAPER_ZIP) / 2**20:.0f} MiB): {_n_paper} files, verified"

print("Stage times (min):", {k: round(v / 60, 1) for k, v in STAGE_TIMES.items()})
print(f"Total runtime: {ENV_LOG['total_runtime_min']} min")
if len(STAGE_LOG_TABLE):
    _sl = STAGE_LOG_TABLE.drop_duplicates("stage", keep="first")         # how each stage was obtained in this session
    _rs, _cp, _kp = (_sl[_sl["status"] == k] for k in ("restored", "computed", "kept"))
    print(f"CHECKPOINT SUMMARY: {len(_sl)} stages; {len(_rs)} restored from checkpoints (not re-run; computing them took "
          f"{_fmt_seconds(_rs['original_seconds'].sum())}), {len(_cp)} computed in this session ({_fmt_seconds(_cp['seconds'].sum())})"
          + (f", {len(_kp)} already complete in this folder" if len(_kp) else "")
          + f"; checkpoints/: {len(_ckpts)} archives, {sum(os.path.getsize(p) for p in _ckpts) / 2**30:.2f} GiB")
print(f"Figures: {len(FIGURE_MANIFEST)} | tables: {len(TABLE_MANIFEST)} | files in outputs/: {len(inventory)} "
      f"({sum(r['bytes'] for r in inventory) / 2**20:.1f} MiB)")
_qa = [f for f in FIGURE_MANIFEST if f["text_outside"] or f["text_overlaps"] or f["legend_over_data"] or f["bold_text"]]
print(f"Figure layout QA: {len(FIGURE_MANIFEST) - len(_qa)}/{len(FIGURE_MANIFEST)} figures without layout issues"
      + (f" (check: {', '.join(f['figure'] for f in _qa)})" if _qa else ""))
print("Archives:\n  " + _zip_note + "\n  " + _paper_note)
print("\nModels evaluated:", ", ".join(EVALUATED_MODELS))
print("Data used:", ", ".join(["CONVOY-3D", "LEGACY-FLAT"] + [f"TUM {k}" for k in A3_VALIDATION]
                              + ([f"Virtual KITTI 2 ({', '.join(VK_SCENE_OBJS)})"] if VK_SCENE_OBJS else [])
                              + ([f"Argoverse 2 ({len(AV2_STATUS['logs'])} validation logs, {len(AV2_PAIRS_ALL)} pairs)"]
                                 if AV2_STATUS["available"] else [])
                              + ([f"KITTI tracking ({', '.join(KT_STATUS['sequences'])})"] if KT_STATUS["available"] else [])
                              + [f"CONVOY-3D designed sets (baseline sweep {N_BSW} scenes, dose-response {N_WDR} scenes)"]
                              + ([f"Bonn RGB-D Dynamic ({', '.join(BONN_STATUS['sequences'])})"] if BONN_STATUS["available"] else [])))
print("\nTo continue in a new session: upload convoy_checkpoints.zip as a Kaggle dataset (or attach this notebook's output), "
      "add it as an input and run all cells; every stage whose fingerprint matches is restored instead of recomputed (Section 0.5).")
print("Dataset copy:", backup_to_dataset([ZIP_PATH, PAPER_ZIP], f"end of run, {len(_ckpts)} stages"))
offer_downloads([ZIP_PATH, PAPER_ZIP], auto=True)